In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import os
import json
import pandas as pd
import time

ROOT = Path("/content/deterministic-intent-translation-journal")
EXP_A = ROOT / "journal_experiments/experiment_A_model_sweep"

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_v1.json"
)

A9_ANALYSIS = (
    EXP_A /
    "manifests/A9_analysis_manifest_v1.json"
)

RESULT = (
    EXP_A /
    "results/raw/"
    "A10_gpt-oss-120b_results.csv"
)

LOG = (
    EXP_A /
    "logs/"
    "A10_gpt-oss-120b_execution.log"
)

EXEC_MANIFEST = (
    EXP_A /
    "manifests/"
    "A10_execution_manifest_v1.json"
)

MODEL = "openai/gpt-oss-120b"

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "preregistration":
        "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

    "A9_analysis":
        "f115a5520362b11119057bbcc31a3b026d75afdf416d44514210235e0307f067",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def bool_count(df, column, value=True):

    if column not in df.columns:
        return None

    s = (
        df[column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    target = "true" if value else "false"

    return int((s == target).sum())


print("=" * 110)
print("STEP A38 — EXECUTE A10 ON FROZEN 120-CASE BENCHMARK")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Verify frozen artifacts
# ------------------------------------------------------------------

checks = [
    (
        "Frozen runner",
        RUNNER,
        EXPECTED["runner"]
    ),
    (
        "Frozen benchmark",
        DATASET,
        EXPECTED["dataset"]
    ),
    (
        "A7-A10 preregistration",
        PREREG,
        EXPECTED["preregistration"]
    ),
    (
        "Frozen A9 analysis manifest",
        A9_ANALYSIS,
        EXPECTED["A9_analysis"]
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing required artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# 2. Verify A10 preregistration
# ------------------------------------------------------------------

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

arm_map = {
    arm["arm"]: arm["model"]
    for arm in prereg["arms"]
}

assert arm_map["A10"] == MODEL, (
    "STOP: A10 model does not match preregistration."
)

assert prereg["execution_order"] == [
    "A7",
    "A8",
    "A9",
    "A10",
], "STOP: frozen execution order changed."

print("✓ A10 exact model ID verified")
print("✓ Frozen execution order verified")


# ------------------------------------------------------------------
# 3. Safety checks
# ------------------------------------------------------------------

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY missing."
)

assert not RESULT.exists(), (
    f"STOP: A10 result already exists:\n{RESULT}\n"
    "Do not rerun A10."
)

assert not LOG.exists(), (
    f"STOP: A10 log already exists:\n{LOG}\n"
    "Do not rerun A10."
)

assert not EXEC_MANIFEST.exists(), (
    f"STOP: A10 execution manifest already exists:\n"
    f"{EXEC_MANIFEST}"
)

RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("✓ API key present")
print("✓ A10 output paths clean")


# ------------------------------------------------------------------
# 4. Exact command
# ------------------------------------------------------------------

cmd = [
    "python",
    str(RUNNER),

    "--dataset",
    str(DATASET),

    "--out",
    str(RESULT),

    "--model",
    MODEL,
]

print("\n" + "-" * 110)
print("A10 FROZEN EXECUTION")
print("-" * 110)

print("Arm   : A10")
print("Model :", MODEL)

print("\nCommand:")
print(" ".join(cmd))

print("\nPolicy:")
print("  ✓ frozen 120-case benchmark")
print("  ✓ no --limit")
print("  ✓ only semantic-parser model changes")
print("  ✓ no prompt/gate/threshold changes")
print("  ✓ no automatic rerun")


# ------------------------------------------------------------------
# 5. Execute exactly once
# ------------------------------------------------------------------

started_utc = datetime.now(
    timezone.utc
).isoformat()

start = time.time()

proc = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - start

finished_utc = datetime.now(
    timezone.utc
).isoformat()

LOG.write_text(
    proc.stdout or "",
    encoding="utf-8"
)

print("\n" + "-" * 110)
print("RUNNER OUTPUT")
print("-" * 110)

print(proc.stdout)

print(
    "\nProcess return code :",
    proc.returncode
)

print(
    "Wall-clock seconds :",
    round(elapsed, 2)
)


# ------------------------------------------------------------------
# 6. Never automatically rerun
# ------------------------------------------------------------------

if proc.returncode != 0:

    print("\n" + "=" * 110)
    print("✗ A10 PROCESS RETURNED NON-ZERO")
    print("✗ DO NOT RERUN")
    print("✗ SEND OUTPUT FOR DIAGNOSIS")
    print("=" * 110)

    raise SystemExit


# ------------------------------------------------------------------
# 7. Verify result
# ------------------------------------------------------------------

assert RESULT.exists(), (
    "STOP: A10 result CSV missing."
)

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: expected 120 rows; found {len(df)}."
)

assert "id" in df.columns, (
    "STOP: id column missing."
)

unique_cases = df["id"].nunique()

assert unique_cases == 120, (
    f"STOP: expected 120 unique cases; "
    f"found {unique_cases}."
)


# ------------------------------------------------------------------
# 8. Lightweight diagnostics
# ------------------------------------------------------------------

llm_invoked = bool_count(
    df,
    "llm_invoked",
    True
)

parse_ok = bool_count(
    df,
    "parse_ok",
    True
)

parse_failed = bool_count(
    df,
    "parse_ok",
    False
)

accepted = bool_count(
    df,
    "accepted",
    True
)

exception_rows = 0

if "rejection_stage" in df.columns:

    stage_norm = (
        df["rejection_stage"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    exception_rows = int(
        stage_norm.eq("exception").sum()
    )

    rejection_stages = (
        df["rejection_stage"]
        .fillna("<none>")
        .value_counts()
        .to_dict()
    )

else:

    rejection_stages = {}


# ------------------------------------------------------------------
# 9. Freeze execution manifest
# ------------------------------------------------------------------

execution_record = {
    "manifest_id":
        "experiment-A-A10-execution-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A10",

    "model":
        MODEL,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "wall_clock_seconds":
        elapsed,

    "return_code":
        proc.returncode,

    "command":
        cmd,

    "frozen_inputs": {
        "runner": {
            "path":
                str(RUNNER.relative_to(ROOT)),

            "sha256":
                sha256(RUNNER),
        },

        "benchmark": {
            "path":
                str(DATASET.relative_to(ROOT)),

            "sha256":
                sha256(DATASET),
        },

        "preregistration": {
            "path":
                str(PREREG.relative_to(ROOT)),

            "sha256":
                sha256(PREREG),
        },

        "prior_A9_analysis_manifest": {
            "path":
                str(A9_ANALYSIS.relative_to(ROOT)),

            "sha256":
                sha256(A9_ANALYSIS),
        },
    },

    "outputs": {
        "result": {
            "path":
                str(RESULT.relative_to(ROOT)),

            "sha256":
                sha256(RESULT),

            "rows":
                len(df),

            "unique_cases":
                unique_cases,
        },

        "log": {
            "path":
                str(LOG.relative_to(ROOT)),

            "sha256":
                sha256(LOG),
        },
    },

    "lightweight_diagnostics": {
        "llm_invoked":
            llm_invoked,

        "parse_ok":
            parse_ok,

        "parse_failed":
            parse_failed,

        "accepted":
            accepted,

        "exception_rows":
            exception_rows,

        "rejection_stages":
            rejection_stages,
    },

    "performance_metrics_calculated":
        False,

    "automatic_rerun":
        False,
}

EXEC_MANIFEST.write_text(
    json.dumps(
        execution_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("A10 RAW EXECUTION ARTIFACTS")
print("-" * 110)

print("Result:")
print(RESULT)

print("Rows:", len(df))
print("Unique cases:", unique_cases)

print("SHA-256:")
print(sha256(RESULT))

print("\nLog:")
print(LOG)

print("SHA-256:")
print(sha256(LOG))

print("\nExecution manifest:")
print(EXEC_MANIFEST)

print("SHA-256:")
print(sha256(EXEC_MANIFEST))


print("\n" + "-" * 110)
print("LIGHTWEIGHT DIAGNOSTIC")
print("-" * 110)

print("LLM invoked   :", llm_invoked)
print("Parse OK      :", parse_ok)
print("Parse failed  :", parse_failed)
print("Accepted      :", accepted)
print("Exception rows:", exception_rows)

print("\nRejection stages:")

for stage, count in rejection_stages.items():
    print(f"  {stage}: {count}")


print("\n" + "=" * 110)
print("✓ A10 EXECUTED EXACTLY ONCE")
print("✓ 120 RESULT ROWS VERIFIED")
print("✓ RAW RESULT FROZEN")
print("✓ EXECUTION LOG FROZEN")
print("✓ NO PERFORMANCE METRICS CALCULATED YET")
print("✓ NO AUTOMATIC RERUN")
print("✓ EXPERIMENT A MODEL EXECUTION PHASE COMPLETE")
print("✓ SEND THIS OUTPUT FOR A10 ANALYSIS")
print("=" * 110)

STEP A38 — EXECUTE A10 ON FROZEN 120-CASE BENCHMARK


AssertionError: STOP: missing required artifact:
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py

In [3]:
from google.colab import drive
from pathlib import Path

print("=" * 110)
print("RESTORE R1 — MOUNT DRIVE AND LOCATE PROJECT COPIES")
print("=" * 110)

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

MASTER = (
    DRIVE_ROOT /
    "PhD_Research/Deterministic_Intent_Translation/"
    "experiment-1-v2-together_FINAL"
)

print("\nAuthoritative master:")
print(MASTER)
print("Exists:", MASTER.exists())

if MASTER.exists():
    master_files = sum(
        1 for p in MASTER.rglob("*")
        if p.is_file()
    )
    print("Files :", master_files)

print("\nSearching Drive for Experiment A / journal working copies...")

keywords = [
    "deterministic-intent-translation-journal",
    "experiment_A_model_sweep",
]

matches = []

for p in DRIVE_ROOT.rglob("*"):
    try:
        if any(
            key.lower() in p.name.lower()
            for key in keywords
        ):
            matches.append(p)
    except Exception:
        pass

print("\nMatches found:", len(matches))

for p in matches:
    print(
        "[DIR] " if p.is_dir() else "[FILE]",
        p
    )

print("\n" + "=" * 110)

if MASTER.exists():
    print("✓ AUTHORITATIVE MASTER FOUND")
else:
    print("✗ AUTHORITATIVE MASTER NOT FOUND")

print("✓ NO FILES COPIED")
print("✓ NO FILES MODIFIED")
print("✓ NO MODEL/API CALL")
print("✓ DO NOT RUN A10")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R2")

print("=" * 110)

RESTORE R1 — MOUNT DRIVE AND LOCATE PROJECT COPIES
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Authoritative master:
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/experiment-1-v2-together_FINAL
Exists: True
Files : 629

Searching Drive for Experiment A / journal working copies...

Matches found: 0

✓ AUTHORITATIVE MASTER FOUND
✓ NO FILES COPIED
✓ NO FILES MODIFIED
✓ NO MODEL/API CALL
✓ DO NOT RUN A10
✓ SEND THIS OUTPUT BEFORE RESTORE R2


In [4]:
from pathlib import Path
import hashlib

print("=" * 110)
print("RESTORE R2 — CHECK SURVIVING /content WORKSPACE")
print("=" * 110)

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

RUNNER = (
    ROOT /
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = (
    ROOT /
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


print("\nJournal working root:")
print(ROOT)
print("Exists:", ROOT.exists())

if ROOT.exists():

    file_count = sum(
        1 for p in ROOT.rglob("*")
        if p.is_file()
    )

    print("Total files:", file_count)


print("\nExperiment A workspace:")
print(EXP_A)
print("Exists:", EXP_A.exists())


# ------------------------------------------------------------------
# Check frozen runner / benchmark
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("FROZEN CORE")
print("-" * 110)

for label, path in [
    ("Runner", RUNNER),
    ("Benchmark", DATASET),
]:

    print(f"\n{label}:")
    print(path)
    print("Exists:", path.exists())

    if path.exists():
        print("SHA-256:", sha256(path))


# ------------------------------------------------------------------
# Check important Experiment A artifacts
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("EXPERIMENT A ARTIFACT CHECK")
print("-" * 110)

artifacts = {
    "Original A1-A6 accounting":
        EXP_A /
        "manifests/experiment_A_final_accounting_v1.json",

    "A7-A10 preregistration":
        EXP_A /
        "preregistration/"
        "experiment_A_extension_A7_A10_preregistration_v1.json",

    "A7 raw result":
        EXP_A /
        "results/raw/"
        "A7_arize-qwen-2-1.5b-instruct_results.csv",

    "A7 analysis manifest":
        EXP_A /
        "manifests/A7_analysis_manifest_v1.json",

    "A8 raw result":
        EXP_A /
        "results/raw/"
        "A8_Ternary-Bonsai-27B_results.csv",

    "A8 analysis manifest":
        EXP_A /
        "manifests/A8_analysis_manifest_v1.json",

    "A9 raw result":
        EXP_A /
        "results/raw/"
        "A9_DeepSeek-V4.1-Flash_results.csv",

    "A9 analysis manifest":
        EXP_A /
        "manifests/A9_analysis_manifest_v1.json",

    "A10 raw result":
        EXP_A /
        "results/raw/"
        "A10_gpt-oss-120b_results.csv",

    "A10 execution manifest":
        EXP_A /
        "manifests/A10_execution_manifest_v1.json",
}


for label, path in artifacts.items():

    exists = path.exists()

    print(f"\n{label}")
    print("  Exists :", exists)

    if exists:
        print("  Path   :", path)
        print("  SHA256 :", sha256(path))


# ------------------------------------------------------------------
# Summarize Experiment A tree if present
# ------------------------------------------------------------------

if EXP_A.exists():

    print("\n" + "-" * 110)
    print("EXPERIMENT A DIRECTORY COUNTS")
    print("-" * 110)

    for sub in [
        "preregistration",
        "results/raw",
        "results/metrics",
        "results/analysis",
        "manifests",
        "logs",
        "model_availability_screen",
    ]:

        p = EXP_A / sub

        if p.exists():

            count = sum(
                1 for x in p.rglob("*")
                if x.is_file()
            )

        else:
            count = 0

        print(f"{sub:<28} : {count}")


print("\n" + "=" * 110)
print("✓ READ-ONLY RECOVERY INSPECTION COMPLETE")
print("✓ NOTHING COPIED")
print("✓ NOTHING MODIFIED")
print("✓ NO MODEL/API CALL")
print("✓ DO NOT RUN A10")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R3")
print("=" * 110)

RESTORE R2 — CHECK SURVIVING /content WORKSPACE

Journal working root:
/content/deterministic-intent-translation-journal
Exists: False

Experiment A workspace:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep
Exists: False

--------------------------------------------------------------------------------------------------------------
FROZEN CORE
--------------------------------------------------------------------------------------------------------------

Runner:
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py
Exists: False

Benchmark:
/content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
Exists: False

--------------------------------------------------------------------------------------------------------------
EXPERIMENT A ARTIFACT CHECK
---------------------------------------------

In [5]:
from pathlib import Path
import shutil
import hashlib

print("=" * 110)
print("RESTORE R3 — RECREATE JOURNAL WORKING COPY FROM AUTHORITATIVE MASTER")
print("=" * 110)

MASTER = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "experiment-1-v2-together_FINAL"
)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER_REL = Path(
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET_REL = Path(
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def count_files(path):
    return sum(
        1 for p in path.rglob("*")
        if p.is_file()
    )


# ------------------------------------------------------------------
# 1. Verify source
# ------------------------------------------------------------------

assert MASTER.exists(), (
    f"STOP: authoritative master missing:\n{MASTER}"
)

master_count = count_files(MASTER)

assert master_count == 629, (
    f"STOP: expected 629 master files; "
    f"found {master_count}"
)

print("✓ Authoritative master found")
print("✓ Master file count:", master_count)


# ------------------------------------------------------------------
# 2. Safety: destination must still be absent
# ------------------------------------------------------------------

assert not ROOT.exists(), (
    f"STOP: destination unexpectedly exists:\n{ROOT}\n"
    "Do not overwrite it."
)

print("✓ Destination confirmed absent")


# ------------------------------------------------------------------
# 3. Copy authoritative master
# ------------------------------------------------------------------

print("\nCopying frozen master to working /content ...")

shutil.copytree(
    MASTER,
    ROOT
)

print("✓ Copy complete")


# ------------------------------------------------------------------
# 4. Verify copy count
# ------------------------------------------------------------------

root_count = count_files(ROOT)

print("\nSource files      :", master_count)
print("Destination files :", root_count)

assert root_count == master_count == 629, (
    "STOP: source/destination file counts differ."
)

print("✓ File counts match exactly")


# ------------------------------------------------------------------
# 5. Verify canonical runner
# ------------------------------------------------------------------

RUNNER = ROOT / RUNNER_REL

assert RUNNER.exists(), (
    f"STOP: canonical runner missing:\n{RUNNER}"
)

runner_sha = sha256(RUNNER)

print("\nCanonical runner:")
print(RUNNER)
print("SHA-256:")
print(runner_sha)

assert runner_sha == EXPECTED_RUNNER_SHA, (
    "STOP: canonical runner hash mismatch."
)

print("✓ Frozen runner hash verified")


# ------------------------------------------------------------------
# 6. Verify frozen benchmark
# ------------------------------------------------------------------

DATASET = ROOT / DATASET_REL

assert DATASET.exists(), (
    f"STOP: frozen benchmark missing:\n{DATASET}"
)

dataset_sha = sha256(DATASET)

print("\nFrozen benchmark:")
print(DATASET)
print("SHA-256:")
print(dataset_sha)

assert dataset_sha == EXPECTED_DATASET_SHA, (
    "STOP: frozen benchmark hash mismatch."
)

print("✓ Frozen benchmark hash verified")


# ------------------------------------------------------------------
# 7. Final state
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("✓ JOURNAL WORKING COPY RESTORED")
print("✓ 629 / 629 FILES VERIFIED")
print("✓ CANONICAL RUNNER VERIFIED")
print("✓ FROZEN 120-CASE BENCHMARK VERIFIED")
print("✓ AUTHORITATIVE DRIVE MASTER NOT MODIFIED")
print("✓ NO MODEL/API CALL")
print("✓ NO A1-A9 RERUN")
print("✓ A10 NOT EXECUTED")
print("✓ DO NOT RUN A10 YET")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R4")
print("=" * 110)

RESTORE R3 — RECREATE JOURNAL WORKING COPY FROM AUTHORITATIVE MASTER
✓ Authoritative master found
✓ Master file count: 629
✓ Destination confirmed absent

Copying frozen master to working /content ...
✓ Copy complete

Source files      : 629
Destination files : 629
✓ File counts match exactly

Canonical runner:
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.py
SHA-256:
40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
✓ Frozen runner hash verified

Frozen benchmark:
/content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
SHA-256:
84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de
✓ Frozen benchmark hash verified

✓ JOURNAL WORKING COPY RESTORED
✓ 629 / 629 FILES VERIFIED
✓ CANONICAL RUNNER VERIFIED
✓ FROZEN 120-CASE BENCHMARK VERIFIED
✓ AUTHORITATIVE DRIVE MASTER NOT MODIFIED
✓ NO MODEL/API CALL
✓ NO A1-

In [6]:
from pathlib import Path
import hashlib
import pandas as pd

print("=" * 110)
print("RESTORE R4 — INVENTORY RECOVERABLE EXPERIMENT EVIDENCE")
print("=" * 110)

ROOT = Path("/content/deterministic-intent-translation-journal")

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "benchmark":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "official_result":
        "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e",

    "official_manifest":
        "7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af",
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# ------------------------------------------------------------------
# 1. Reconfirm frozen core
# ------------------------------------------------------------------

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

assert sha256(RUNNER) == EXPECTED["runner"]
assert sha256(DATASET) == EXPECTED["benchmark"]

print("✓ Frozen runner reconfirmed")
print("✓ Frozen benchmark reconfirmed")


# ------------------------------------------------------------------
# 2. Search entire restored master for known hashes
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("SEARCH FOR KNOWN FROZEN ARTIFACT HASHES")
print("-" * 110)

hash_targets = {
    EXPECTED["official_result"]:
        "Original official E3-v3 result",

    EXPECTED["official_manifest"]:
        "Original official E3-v3 manifest",
}

hash_matches = []

all_files = [
    p for p in ROOT.rglob("*")
    if p.is_file()
]

print("Files scanned:", len(all_files))

for p in all_files:
    try:
        digest = sha256(p)
    except Exception:
        continue

    if digest in hash_targets:
        hash_matches.append(
            (
                hash_targets[digest],
                p,
                digest
            )
        )

if hash_matches:
    for label, path, digest in hash_matches:
        print(f"\n✓ {label}")
        print("Path   :", path)
        print("SHA256 :", digest)
else:
    print("\nNo known result/manifest hash matches found.")


# ------------------------------------------------------------------
# 3. Search likely result / manifest / metrics files
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("LIKELY EXPERIMENT ARTIFACT FILES")
print("-" * 110)

keywords = [
    "result",
    "manifest",
    "metric",
    "heldout",
    "e3v3",
    "e3_v3",
]

candidate_files = []

for p in all_files:

    name = p.name.lower()

    if any(k in name for k in keywords):
        candidate_files.append(p)

print("Candidate files:", len(candidate_files))

for p in sorted(candidate_files):
    print(p.relative_to(ROOT))


# ------------------------------------------------------------------
# 4. Specifically inspect E3-v3 reproducibility directory
# ------------------------------------------------------------------

REPRO = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0"
)

print("\n" + "-" * 110)
print("E3-v3 REPRODUCIBILITY DIRECTORY")
print("-" * 110)

print("Path  :", REPRO)
print("Exists:", REPRO.exists())

if REPRO.exists():

    repro_files = sorted(
        p for p in REPRO.rglob("*")
        if p.is_file()
    )

    print("Files :", len(repro_files))

    for p in repro_files:
        print(
            f"{p.relative_to(ROOT)}"
            f"\n    SHA256: {sha256(p)}"
        )


# ------------------------------------------------------------------
# 5. Inspect any CSV matching official result hash
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("OFFICIAL RESULT STRUCTURE CHECK")
print("-" * 110)

official_result_paths = [
    path
    for label, path, digest in hash_matches
    if digest == EXPECTED["official_result"]
]

if official_result_paths:

    result_path = official_result_paths[0]

    df = pd.read_csv(result_path)

    print("Official result found:")
    print(result_path)

    print("Rows        :", len(df))

    if "id" in df.columns:
        print(
            "Unique cases:",
            df["id"].nunique()
        )

    print("Columns:")
    print(list(df.columns))

else:
    print("Official result hash not found.")


# ------------------------------------------------------------------
# 6. Check whether journal workspace exists in restored master
# ------------------------------------------------------------------

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

print("\n" + "-" * 110)
print("JOURNAL EXPERIMENT WORKSPACE")
print("-" * 110)

print("Path  :", EXP_A)
print("Exists:", EXP_A.exists())

if EXP_A.exists():

    exp_files = sorted(
        p for p in EXP_A.rglob("*")
        if p.is_file()
    )

    print("Files :", len(exp_files))

    for p in exp_files:
        print(p.relative_to(ROOT))

else:
    print(
        "No Experiment A journal workspace is present "
        "in the restored master."
    )


# ------------------------------------------------------------------
# 7. Final
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("✓ RECOVERY INVENTORY COMPLETE")
print("✓ READ-ONLY INSPECTION")
print("✓ NO FILE CREATED")
print("✓ NO FILE MODIFIED")
print("✓ NO MODEL/API CALL")
print("✓ NO A1-A9 RERUN")
print("✓ DO NOT RUN A10")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R5")
print("=" * 110)

RESTORE R4 — INVENTORY RECOVERABLE EXPERIMENT EVIDENCE
✓ Frozen runner reconfirmed
✓ Frozen benchmark reconfirmed

--------------------------------------------------------------------------------------------------------------
SEARCH FOR KNOWN FROZEN ARTIFACT HASHES
--------------------------------------------------------------------------------------------------------------
Files scanned: 629

✓ Original official E3-v3 manifest
Path   : /content/deterministic-intent-translation-journal/experiment-2b/heldout/results/e3v3_heldout_benchmark_run_v1_manifest.json
SHA256 : 7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af

✓ Original official E3-v3 result
Path   : /content/deterministic-intent-translation-journal/experiment-2b/heldout/results/e3v3_heldout_benchmark_run_v1.csv
SHA256 : 12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e

✓ Original official E3-v3 manifest
Path   : /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_

In [7]:
from pathlib import Path
import hashlib

print("=" * 110)
print("RESTORE R5 — EXACT-HASH SEARCH FOR LOST EXPERIMENT A RAW RESULTS")
print("=" * 110)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ------------------------------------------------------------------
# Known raw-result hashes recorded BEFORE Colab reset
# ------------------------------------------------------------------

TARGETS = {
    # A1 official frozen result
    "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e":
        {
            "arm": "A1",
            "model": "Qwen/Qwen3.5-9B",
            "type": "official frozen result",
        },

    # A2
    "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7":
        {
            "arm": "A2",
            "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
            "type": "Experiment A raw result",
        },

    # A3
    "fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf440a":
        {
            "arm": "A3",
            "model": "Qwen/Qwen2.5-7B-Instruct-Turbo",
            "type": "unavailable-arm raw result",
        },

    # A4
    "6babc2b2492473e5ee737e182dd338be2bc23c0cb0f72b5de8b70864d8757125":
        {
            "arm": "A4",
            "model": "mistralai/Mistral-Small-24B-Instruct-2501",
            "type": "unavailable-arm raw result",
        },

    # A5
    "8d6c561218991dc1d3a0f47601bd5a8c01bce918be0b3c1133333de5eb702f41":
        {
            "arm": "A5",
            "model": "deepseek-ai/DeepSeek-V3.1",
            "type": "unavailable-arm raw result",
        },

    # A7
    "2c41287e4cab5680e5031fb9252b003d7bdf5412a76b42a8856c9ce700a70e7c":
        {
            "arm": "A7",
            "model": "arize-ai/qwen-2-1.5b-instruct",
            "type": "Experiment A raw result",
        },

    # A8
    "1507e111876a27801fbd03998cfa8f8390b663be954479386f01bf2ae242b082":
        {
            "arm": "A8",
            "model": "Prism-ML/Ternary-Bonsai-27B",
            "type": "Experiment A raw result",
        },

    # A9
    "b4b02483db7021c43e2e406d0ea4811f7f22f72258957613bbfcd96b11ec12aa":
        {
            "arm": "A9",
            "model": "deepseek-ai/DeepSeek-V4.1-Flash",
            "type": "Experiment A raw result",
        },
}


# ------------------------------------------------------------------
# Scan all 629 restored files
# ------------------------------------------------------------------

all_files = [
    p for p in ROOT.rglob("*")
    if p.is_file()
]

print("Files to scan:", len(all_files))

assert len(all_files) == 629, (
    f"STOP: expected 629 files; found {len(all_files)}."
)

found = {
    digest: []
    for digest in TARGETS
}

for index, path in enumerate(all_files, start=1):

    try:
        digest = sha256(path)
    except Exception:
        continue

    if digest in TARGETS:
        found[digest].append(path)


# ------------------------------------------------------------------
# Report arm-by-arm
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("EXACT-HASH RECOVERY RESULTS")
print("-" * 110)

recovered_arms = []
missing_arms = []

for digest, info in TARGETS.items():

    arm = info["arm"]
    model = info["model"]

    print(f"\n{arm}")
    print("Model :", model)
    print("Hash  :", digest)

    matches = found[digest]

    if matches:

        recovered_arms.append(arm)

        print("Status: RECOVERED")

        for path in matches:
            print(
                "Path  :",
                path
            )

    else:

        missing_arms.append(arm)

        print("Status: NOT FOUND")


# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("RECOVERY SUMMARY")
print("=" * 110)

print(
    "Recovered exact raw-result arms:",
    recovered_arms
)

print(
    "Raw-result arms not found:",
    missing_arms
)

print("\nInterpretation:")

print(
    "  RECOVERED means byte-identical evidence exists "
    "in the restored authoritative project."
)

print(
    "  NOT FOUND means the recorded raw-result SHA-256 "
    "does not occur anywhere in the restored 629-file project."
)

print(
    "  No missing result has been recreated or substituted."
)


# ------------------------------------------------------------------
# A10 safety
# ------------------------------------------------------------------

A10_RESULT = (
    ROOT /
    "journal_experiments/"
    "experiment_A_model_sweep/"
    "results/raw/"
    "A10_gpt-oss-120b_results.csv"
)

print("\nA10 result exists:", A10_RESULT.exists())


print("\n" + "=" * 110)
print("✓ EXACT-HASH RECOVERY SEARCH COMPLETE")
print("✓ READ-ONLY")
print("✓ NO FILE CREATED")
print("✓ NO FILE MODIFIED")
print("✓ NO MODEL/API CALL")
print("✓ NO EXPERIMENT ARM RERUN")
print("✓ DO NOT RUN A10 YET")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R6")
print("=" * 110)

RESTORE R5 — EXACT-HASH SEARCH FOR LOST EXPERIMENT A RAW RESULTS
Files to scan: 629

--------------------------------------------------------------------------------------------------------------
EXACT-HASH RECOVERY RESULTS
--------------------------------------------------------------------------------------------------------------

A1
Model : Qwen/Qwen3.5-9B
Hash  : 12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e
Status: RECOVERED
Path  : /content/deterministic-intent-translation-journal/experiment-2b/heldout/results/e3v3_heldout_benchmark_run_v1.csv
Path  : /content/deterministic-intent-translation-journal/experiment-2b/reproducibility/e3v3_final_experiment_closure_v1.0/official_run/e3v3_heldout_benchmark_run_v1.csv

A2
Model : meta-llama/Llama-3.3-70B-Instruct-Turbo
Hash  : 5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7
Status: NOT FOUND

A3
Model : Qwen/Qwen2.5-7B-Instruct-Turbo
Hash  : fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

print("=" * 110)
print("RESTORE R6 — FREEZE COLAB RESET / EXPERIMENT-A RECOVERY PROVENANCE")
print("=" * 110)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

RECOVERY = EXP_A / "recovery"
MANIFESTS = EXP_A / "manifests"
RAW = EXP_A / "results/raw"
METRICS = EXP_A / "results/metrics"
ANALYSIS = EXP_A / "results/analysis"
LOGS = EXP_A / "logs"
PREREG = EXP_A / "preregistration"

for p in [
    RECOVERY,
    MANIFESTS,
    RAW,
    METRICS,
    ANALYSIS,
    LOGS,
    PREREG,
]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ------------------------------------------------------------------
# Frozen base
# ------------------------------------------------------------------

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

A1_RESULT = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1.csv"
)

A1_MANIFEST = ROOT / (
    "experiment-2b/heldout/results/"
    "e3v3_heldout_benchmark_run_v1_manifest.json"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "A1_result":
        "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e",

    "A1_manifest":
        "7d3ed06d31ce61cb5535a3acafcb572c6ed99d27e5cc2bb58fc433b2742704af",
}

for label, path, expected in [
    ("runner", RUNNER, EXPECTED["runner"]),
    ("dataset", DATASET, EXPECTED["dataset"]),
    ("A1 result", A1_RESULT, EXPECTED["A1_result"]),
    ("A1 manifest", A1_MANIFEST, EXPECTED["A1_manifest"]),
]:

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# Historical experiment record
#
# These values were recorded from completed executions BEFORE the
# Colab runtime reset. They are provenance records, NOT reconstructed
# raw model outputs.
# ------------------------------------------------------------------

arms = {
    "A1": {
        "model": "Qwen/Qwen3.5-9B",
        "execution_status": "COMPLETED",
        "evidence_status": "RAW_RESULT_RECOVERED_BYTE_IDENTICAL",
        "raw_sha256":
            "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e",
        "evaluable": True,
        "accuracy": 101 / 120,
        "VAR": 51 / 58,
        "IRR": 50 / 62,
        "containment_violations": 0,
    },

    "A2": {
        "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7",
        "evaluable": True,
        "accuracy": 100 / 120,
        "VAR": 48 / 58,
        "IRR": 52 / 62,
        "containment_violations": 0,
    },

    "A3": {
        "model": "Qwen/Qwen2.5-7B-Instruct-Turbo",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "fa4c9a0e57103768d06d370983a70a76516800bd0d57ec3d965445d35daf440a",
        "evaluable": False,
        "classification":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
        "llm_attempts": 77,
        "model_responses": 0,
        "containment_violations": 0,
    },

    "A4": {
        "model": "mistralai/Mistral-Small-24B-Instruct-2501",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "6babc2b2492473e5ee737e182dd338be2bc23c0cb0f72b5de8b70864d8757125",
        "evaluable": False,
        "classification":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
        "llm_attempts": 77,
        "model_responses": 0,
        "containment_violations": 0,
    },

    "A5": {
        "model": "deepseek-ai/DeepSeek-V3.1",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "8d6c561218991dc1d3a0f47601bd5a8c01bce918be0b3c1133333de5eb702f41",
        "evaluable": False,
        "classification":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
        "llm_attempts": 77,
        "model_responses": 0,
        "containment_violations": 0,
    },

    "A6": {
        "model": "openai/gpt-oss-20b",
        "execution_status":
            "BENCHMARK_NOT_EXECUTED_PREBENCHMARK_PROBE_ONLY",
        "evidence_status":
            "PROBE_ARTIFACT_LOST_AFTER_COLAB_RESET",
        "evaluable": False,
        "classification":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
        "benchmark_cases_submitted": 0,
    },

    "A7": {
        "model": "arize-ai/qwen-2-1.5b-instruct",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "2c41287e4cab5680e5031fb9252b003d7bdf5412a76b42a8856c9ce700a70e7c",
        "evaluable": True,
        "accuracy": 72 / 120,
        "VAR": 11 / 58,
        "IRR": 61 / 62,
        "containment_violations": 0,
    },

    "A8": {
        "model": "Prism-ML/Ternary-Bonsai-27B",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "1507e111876a27801fbd03998cfa8f8390b663be954479386f01bf2ae242b082",
        "evaluable": True,
        "accuracy": 91 / 120,
        "VAR": 41 / 58,
        "IRR": 50 / 62,
        "containment_violations": 0,
    },

    "A9": {
        "model": "deepseek-ai/DeepSeek-V4.1-Flash",
        "execution_status": "COMPLETED_BEFORE_RESET",
        "evidence_status": "RAW_RESULT_LOST_AFTER_COLAB_RESET",
        "raw_sha256":
            "b4b02483db7021c43e2e406d0ea4811f7f22f72258957613bbfcd96b11ec12aa",
        "evaluable": True,
        "accuracy": 101 / 120,
        "VAR": 51 / 58,
        "IRR": 50 / 62,
        "containment_violations": 0,
    },

    "A10": {
        "model": "openai/gpt-oss-120b",
        "execution_status": "NOT_YET_EXECUTED",
        "evidence_status": "NONE_YET",
        "evaluable": None,
    },
}


# ------------------------------------------------------------------
# Recovery record
# ------------------------------------------------------------------

record = {
    "record_id":
        "experiment-A-colab-reset-recovery-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "event": {
        "type":
            "COLAB_RUNTIME_RESET",

        "description": (
            "The transient /content filesystem was lost after "
            "Google Colab disconnected/reset. The authoritative "
            "629-file project archive on Google Drive remained intact."
        ),
    },

    "restoration": {
        "authoritative_source":
            "/content/drive/MyDrive/PhD_Research/"
            "Deterministic_Intent_Translation/"
            "experiment-1-v2-together_FINAL",

        "restored_working_root":
            str(ROOT),

        "source_file_count":
            629,

        "restored_file_count":
            629,

        "runner_sha256":
            sha256(RUNNER),

        "benchmark_sha256":
            sha256(DATASET),
    },

    "recovery_search": {
        "files_scanned":
            629,

        "byte_identical_raw_results_recovered":
            ["A1"],

        "recorded_raw_results_not_found":
            [
                "A2",
                "A3",
                "A4",
                "A5",
                "A7",
                "A8",
                "A9",
            ],

        "A6_note":
            "A6 benchmark was never executed; only a prebenchmark "
            "availability probe was performed.",

        "A10_note":
            "A10 had not been executed before the runtime reset.",
    },

    "arms":
        arms,

    "recovery_policy": {
        "rerun_completed_arms_to_recreate_lost_files":
            False,

        "reason": (
            "A second execution would not be byte-identical evidence "
            "of the original completed run and could introduce "
            "post-observation rerun bias."
        ),

        "manufacture_raw_results_from_recorded_metrics":
            False,

        "treat_recorded_metrics_as_raw_files":
            False,

        "continue_never_executed_A10":
            True,

        "preserve_loss_disclosure":
            True,
    },

    "claim_boundary": (
        "Recovered A1 retains byte-identical raw evidence. "
        "For A2-A5 and A7-A9, execution outcomes and hashes were "
        "recorded before the reset, but the transient raw files are "
        "no longer locally available. Those records must not be "
        "represented as recovered raw outputs."
    ),

    "model_or_api_call_made_during_recovery":
        False,
}

RECOVERY_RECORD = (
    RECOVERY /
    "experiment_A_colab_reset_recovery_v1.json"
)

assert not RECOVERY_RECORD.exists(), (
    f"STOP: recovery record already exists:\n"
    f"{RECOVERY_RECORD}"
)

RECOVERY_RECORD.write_text(
    json.dumps(
        record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


# ------------------------------------------------------------------
# Recovery manifest
# ------------------------------------------------------------------

manifest = {
    "manifest_id":
        "experiment-A-colab-reset-recovery-manifest-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "recovery_record": {
        "path":
            str(RECOVERY_RECORD.relative_to(ROOT)),

        "sha256":
            sha256(RECOVERY_RECORD),
    },

    "recovered_A1_result": {
        "path":
            str(A1_RESULT.relative_to(ROOT)),

        "sha256":
            sha256(A1_RESULT),
    },

    "recovered_A1_manifest": {
        "path":
            str(A1_MANIFEST.relative_to(ROOT)),

        "sha256":
            sha256(A1_MANIFEST),
    },

    "frozen_runner": {
        "path":
            str(RUNNER.relative_to(ROOT)),

        "sha256":
            sha256(RUNNER),
    },

    "frozen_benchmark": {
        "path":
            str(DATASET.relative_to(ROOT)),

        "sha256":
            sha256(DATASET),
    },

    "no_model_call":
        True,

    "no_completed_arm_rerun":
        True,

    "A10_executed":
        False,
}

RECOVERY_MANIFEST = (
    MANIFESTS /
    "experiment_A_colab_reset_recovery_manifest_v1.json"
)

assert not RECOVERY_MANIFEST.exists(), (
    f"STOP: recovery manifest already exists:\n"
    f"{RECOVERY_MANIFEST}"
)

RECOVERY_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)


print("\n" + "-" * 110)
print("RECOVERY ARTIFACTS")
print("-" * 110)

print("Recovery record:")
print(RECOVERY_RECORD)
print("SHA-256:")
print(sha256(RECOVERY_RECORD))

print("\nRecovery manifest:")
print(RECOVERY_MANIFEST)
print("SHA-256:")
print(sha256(RECOVERY_MANIFEST))


print("\n" + "-" * 110)
print("RECORDED ARM STATUS")
print("-" * 110)

for arm, info in arms.items():

    print(
        f"{arm:<4} "
        f"{info['model']:<45} "
        f"{info['execution_status']}"
    )


print("\n" + "=" * 110)
print("✓ COLAB RESET EVENT DOCUMENTED")
print("✓ A1 BYTE-IDENTICAL RAW EVIDENCE RETAINED")
print("✓ LOST RAW FILES NOT FABRICATED")
print("✓ COMPLETED ARMS NOT RERUN")
print("✓ A10 RECORDED AS NEVER EXECUTED")
print("✓ FROZEN RUNNER / BENCHMARK VERIFIED")
print("✓ NO MODEL/API CALL")
print("✓ DRIVE MASTER NOT MODIFIED")
print("✓ DO NOT RUN A10 YET")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R7")
print("=" * 110)

RESTORE R6 — FREEZE COLAB RESET / EXPERIMENT-A RECOVERY PROVENANCE
✓ runner verified
✓ dataset verified
✓ A1 result verified
✓ A1 manifest verified

--------------------------------------------------------------------------------------------------------------
RECOVERY ARTIFACTS
--------------------------------------------------------------------------------------------------------------
Recovery record:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/recovery/experiment_A_colab_reset_recovery_v1.json
SHA-256:
21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3

Recovery manifest:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/manifests/experiment_A_colab_reset_recovery_manifest_v1.json
SHA-256:
65a7203297009292fd2fc4430ea92d9bf6b379559a356410272cf7a54ac3773c

--------------------------------------------------------------------------------------------------------------
RECORDED AR

In [9]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import hashlib
import json

print("=" * 110)
print("RESTORE R7 — PERSIST EXPERIMENT-A RECOVERY STATE TO GOOGLE DRIVE")
print("=" * 110)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

DRIVE_BASE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation"
)

DRIVE_EXP_A = (
    DRIVE_BASE /
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RECOVERY_RECORD = (
    EXP_A /
    "recovery/"
    "experiment_A_colab_reset_recovery_v1.json"
)

RECOVERY_MANIFEST = (
    EXP_A /
    "manifests/"
    "experiment_A_colab_reset_recovery_manifest_v1.json"
)

EXPECTED = {
    "recovery_record":
        "21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3",

    "recovery_manifest":
        "65a7203297009292fd2fc4430ea92d9bf6b379559a356410272cf7a54ac3773c",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def file_inventory(root):
    records = []

    for path in sorted(
        p for p in root.rglob("*")
        if p.is_file()
    ):
        records.append({
            "relative_path":
                str(path.relative_to(root)),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256(path),
        })

    return records


# ------------------------------------------------------------------
# 1. Verify local recovery state
# ------------------------------------------------------------------

assert EXP_A.exists(), (
    f"STOP: local Experiment A workspace missing:\n{EXP_A}"
)

assert RECOVERY_RECORD.exists()
assert RECOVERY_MANIFEST.exists()

assert (
    sha256(RECOVERY_RECORD)
    == EXPECTED["recovery_record"]
), "STOP: recovery record changed."

assert (
    sha256(RECOVERY_MANIFEST)
    == EXPECTED["recovery_manifest"]
), "STOP: recovery manifest changed."

print("✓ Local Experiment A recovery workspace found")
print("✓ Recovery record verified")
print("✓ Recovery manifest verified")


# ------------------------------------------------------------------
# 2. Destination safety
# ------------------------------------------------------------------

DRIVE_EXP_A.parent.mkdir(
    parents=True,
    exist_ok=True
)

assert not DRIVE_EXP_A.exists(), (
    f"STOP: persistent destination already exists:\n"
    f"{DRIVE_EXP_A}\n"
    "Do not overwrite it automatically."
)

print("✓ Persistent Drive destination is clean")


# ------------------------------------------------------------------
# 3. Inventory local workspace BEFORE copying
# ------------------------------------------------------------------

local_inventory = file_inventory(EXP_A)

print(
    "✓ Local Experiment A files:",
    len(local_inventory)
)

local_map = {
    item["relative_path"]: (
        item["size_bytes"],
        item["sha256"]
    )
    for item in local_inventory
}


# ------------------------------------------------------------------
# 4. Copy to Drive
# ------------------------------------------------------------------

print("\nCopying Experiment A recovery workspace to Drive...")

shutil.copytree(
    EXP_A,
    DRIVE_EXP_A
)

print("✓ Copy complete")


# ------------------------------------------------------------------
# 5. Verify Drive copy byte-for-byte
# ------------------------------------------------------------------

drive_inventory = file_inventory(
    DRIVE_EXP_A
)

drive_map = {
    item["relative_path"]: (
        item["size_bytes"],
        item["sha256"]
    )
    for item in drive_inventory
}

assert local_map == drive_map, (
    "STOP: Drive copy does not match local workspace."
)

print(
    "✓ Drive files:",
    len(drive_inventory)
)

print(
    "✓ Local and Drive inventories match byte-for-byte"
)


# ------------------------------------------------------------------
# 6. Write persistence manifest to BOTH locations
# ------------------------------------------------------------------

persistence = {
    "manifest_id":
        "experiment-A-drive-persistence-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "local_workspace":
        str(EXP_A),

    "persistent_drive_workspace":
        str(DRIVE_EXP_A),

    "files_verified_before_manifest":
        len(local_inventory),

    "recovery_record_sha256":
        sha256(RECOVERY_RECORD),

    "recovery_manifest_sha256":
        sha256(RECOVERY_MANIFEST),

    "verification":
        "BYTE_IDENTICAL_RELATIVE_PATH_SIZE_SHA256",

    "authoritative_629_file_master_modified":
        False,

    "model_or_api_call":
        False,

    "purpose": (
        "Persist Experiment A recovery/provenance state so that "
        "subsequent Colab runtime resets do not erase the journal "
        "experiment workspace."
    ),
}

LOCAL_PERSIST = (
    EXP_A /
    "manifests/"
    "experiment_A_drive_persistence_v1.json"
)

DRIVE_PERSIST = (
    DRIVE_EXP_A /
    "manifests/"
    "experiment_A_drive_persistence_v1.json"
)

assert not LOCAL_PERSIST.exists()
assert not DRIVE_PERSIST.exists()

payload = (
    json.dumps(
        persistence,
        indent=2
    ) + "\n"
)

LOCAL_PERSIST.write_text(
    payload,
    encoding="utf-8"
)

DRIVE_PERSIST.write_text(
    payload,
    encoding="utf-8"
)

assert (
    sha256(LOCAL_PERSIST)
    == sha256(DRIVE_PERSIST)
)

print("✓ Persistence manifest written locally and to Drive")


# ------------------------------------------------------------------
# 7. Final verification of critical Drive artifacts
# ------------------------------------------------------------------

DRIVE_RECOVERY_RECORD = (
    DRIVE_EXP_A /
    "recovery/"
    "experiment_A_colab_reset_recovery_v1.json"
)

DRIVE_RECOVERY_MANIFEST = (
    DRIVE_EXP_A /
    "manifests/"
    "experiment_A_colab_reset_recovery_manifest_v1.json"
)

assert (
    sha256(DRIVE_RECOVERY_RECORD)
    == EXPECTED["recovery_record"]
)

assert (
    sha256(DRIVE_RECOVERY_MANIFEST)
    == EXPECTED["recovery_manifest"]
)

print("\n" + "-" * 110)
print("PERSISTENT RECOVERY LOCATION")
print("-" * 110)

print(DRIVE_EXP_A)

print("\nPersistence manifest:")
print(DRIVE_PERSIST)

print("SHA-256:")
print(sha256(DRIVE_PERSIST))


print("\n" + "=" * 110)
print("✓ EXPERIMENT-A RECOVERY STATE PERSISTED TO DRIVE")
print("✓ BYTE-FOR-BYTE COPY VERIFIED")
print("✓ RECOVERY RECORD VERIFIED ON DRIVE")
print("✓ RECOVERY MANIFEST VERIFIED ON DRIVE")
print("✓ AUTHORITATIVE 629-FILE MASTER NOT MODIFIED")
print("✓ NO MODEL/API CALL")
print("✓ NO COMPLETED ARM RERUN")
print("✓ A10 STILL NOT EXECUTED")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R8")
print("=" * 110)

RESTORE R7 — PERSIST EXPERIMENT-A RECOVERY STATE TO GOOGLE DRIVE
✓ Local Experiment A recovery workspace found
✓ Recovery record verified
✓ Recovery manifest verified
✓ Persistent Drive destination is clean
✓ Local Experiment A files: 2

Copying Experiment A recovery workspace to Drive...
✓ Copy complete
✓ Drive files: 2
✓ Local and Drive inventories match byte-for-byte
✓ Persistence manifest written locally and to Drive

--------------------------------------------------------------------------------------------------------------
PERSISTENT RECOVERY LOCATION
--------------------------------------------------------------------------------------------------------------
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/journal_experiments_persistent/experiment_A_model_sweep

Persistence manifest:
/content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/journal_experiments_persistent/experiment_A_model_sweep/manifests/experiment_A_drive_persistence_v1.json
S

In [10]:
print("=" * 110)
print("RESTORE R8 — TOGETHER ENVIRONMENT / API KEY READINESS")
print("=" * 110)

import os
import sys
import subprocess
import importlib.util
from pathlib import Path
import hashlib


# ------------------------------------------------------------------
# 1. Reconfirm critical persistent recovery artifacts
# ------------------------------------------------------------------

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RECOVERY_RECORD = (
    DRIVE_EXP_A /
    "recovery/"
    "experiment_A_colab_reset_recovery_v1.json"
)

RECOVERY_MANIFEST = (
    DRIVE_EXP_A /
    "manifests/"
    "experiment_A_colab_reset_recovery_manifest_v1.json"
)

PERSIST_MANIFEST = (
    DRIVE_EXP_A /
    "manifests/"
    "experiment_A_drive_persistence_v1.json"
)

EXPECTED = {
    "recovery_record":
        "21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3",

    "recovery_manifest":
        "65a7203297009292fd2fc4430ea92d9bf6b379559a356410272cf7a54ac3773c",

    "persistence_manifest":
        "c892989243dcd2b5bfb539be0222593d0646563a0ce284eb32292858e813a6a1",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


for label, path, expected in [
    (
        "Recovery record",
        RECOVERY_RECORD,
        EXPECTED["recovery_record"],
    ),
    (
        "Recovery manifest",
        RECOVERY_MANIFEST,
        EXPECTED["recovery_manifest"],
    ),
    (
        "Persistence manifest",
        PERSIST_MANIFEST,
        EXPECTED["persistence_manifest"],
    ),
]:

    assert path.exists(), (
        f"STOP: {label} missing:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ------------------------------------------------------------------
# 2. Check/install Together SDK
#
# Installation is allowed.
# This does NOT call a model.
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("TOGETHER SDK")
print("-" * 110)

spec = importlib.util.find_spec("together")

if spec is None:

    print("Together SDK not found.")
    print("Installing Together SDK...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "together",
        ],
        check=True,
    )

    print("✓ Together SDK installed")

else:
    print("✓ Together SDK already installed")


# ------------------------------------------------------------------
# 3. Verify import
# ------------------------------------------------------------------

import together
from together import Together

print("✓ together import successful")

try:
    print(
        "Together SDK version:",
        getattr(together, "__version__", "version attribute unavailable")
    )
except Exception:
    print("Together SDK version: unavailable")


# ------------------------------------------------------------------
# 4. Restore API key WITHOUT printing it
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("API KEY")
print("-" * 110)

if not os.environ.get("TOGETHER_API_KEY"):

    from getpass import getpass

    key = getpass(
        "Enter TOGETHER_API_KEY "
        "(input hidden): "
    ).strip()

    assert key, (
        "STOP: TOGETHER_API_KEY was empty."
    )

    os.environ["TOGETHER_API_KEY"] = key

    del key

    print("✓ TOGETHER_API_KEY loaded into environment")

else:

    print(
        "✓ TOGETHER_API_KEY already present in environment"
    )


assert os.environ.get("TOGETHER_API_KEY")

print("✓ API key presence verified")
print("✓ API key value NOT printed")


# ------------------------------------------------------------------
# 5. Instantiate client only
#
# Client construction is NOT an inference request.
# ------------------------------------------------------------------

client = Together(
    api_key=os.environ["TOGETHER_API_KEY"]
)

print("✓ Together client instantiated")
print("✓ No inference request made")


# ------------------------------------------------------------------
# 6. Reconfirm A10 has not accidentally appeared
# ------------------------------------------------------------------

LOCAL_A10 = (
    ROOT /
    "journal_experiments/"
    "experiment_A_model_sweep/"
    "results/raw/"
    "A10_gpt-oss-120b_results.csv"
)

DRIVE_A10 = (
    DRIVE_EXP_A /
    "results/raw/"
    "A10_gpt-oss-120b_results.csv"
)

assert not LOCAL_A10.exists(), (
    "STOP: unexpected local A10 result exists."
)

assert not DRIVE_A10.exists(), (
    "STOP: unexpected persistent A10 result exists."
)

print("\n✓ No local A10 result exists")
print("✓ No Drive A10 result exists")


# ------------------------------------------------------------------
# Final
# ------------------------------------------------------------------

print("\n" + "=" * 110)
print("✓ RESTORE R8 COMPLETE")
print("✓ TOGETHER SDK READY")
print("✓ API KEY READY")
print("✓ CLIENT INSTANTIATED")
print("✓ NO MODEL/INFERENCE CALL")
print("✓ NO BENCHMARK CASE SUBMITTED")
print("✓ NO COMPLETED ARM RERUN")
print("✓ A10 STILL NEVER EXECUTED")
print("✓ DO NOT RUN A10 YET")
print("✓ SEND THIS OUTPUT BEFORE RESTORE R9")
print("=" * 110)

RESTORE R8 — TOGETHER ENVIRONMENT / API KEY READINESS
✓ Recovery record verified
✓ Recovery manifest verified
✓ Persistence manifest verified

--------------------------------------------------------------------------------------------------------------
TOGETHER SDK
--------------------------------------------------------------------------------------------------------------
Together SDK not found.
Installing Together SDK...
✓ Together SDK installed
✓ together import successful
Together SDK version: 2.39.0

--------------------------------------------------------------------------------------------------------------
API KEY
--------------------------------------------------------------------------------------------------------------
Enter TOGETHER_API_KEY (input hidden): ··········
✓ TOGETHER_API_KEY loaded into environment
✓ API key presence verified
✓ API key value NOT printed
✓ Together client instantiated
✓ No inference request made

✓ No local A10 result exists
✓ No Drive A10 resu

In [11]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil

print("=" * 110)
print("RESTORE R9 — RECONSTRUCT AND PERSIST A7–A10 EXTENSION PREREGISTRATION")
print("=" * 110)

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

LOCAL_PREREG = EXP_A / "preregistration"
DRIVE_PREREG = DRIVE_EXP_A / "preregistration"

LOCAL_MANIFESTS = EXP_A / "manifests"
DRIVE_MANIFESTS = DRIVE_EXP_A / "manifests"

for p in [
    LOCAL_PREREG,
    DRIVE_PREREG,
    LOCAL_MANIFESTS,
    DRIVE_MANIFESTS,
]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# ------------------------------------------------------------------
# 1. Verify recovery chain
# ------------------------------------------------------------------

RECOVERY_RECORD = (
    DRIVE_EXP_A /
    "recovery/"
    "experiment_A_colab_reset_recovery_v1.json"
)

RECOVERY_MANIFEST = (
    DRIVE_MANIFESTS /
    "experiment_A_colab_reset_recovery_manifest_v1.json"
)

PERSIST_MANIFEST = (
    DRIVE_MANIFESTS /
    "experiment_A_drive_persistence_v1.json"
)

EXPECTED = {
    "recovery_record":
        "21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3",

    "recovery_manifest":
        "65a7203297009292fd2fc4430ea92d9bf6b379559a356410272cf7a54ac3773c",

    "persistence_manifest":
        "c892989243dcd2b5bfb539be0222593d0646563a0ce284eb32292858e813a6a1",
}

for label, path, expected in [
    ("recovery record", RECOVERY_RECORD, EXPECTED["recovery_record"]),
    ("recovery manifest", RECOVERY_MANIFEST, EXPECTED["recovery_manifest"]),
    ("persistence manifest", PERSIST_MANIFEST, EXPECTED["persistence_manifest"]),
]:
    assert path.exists(), f"STOP: missing {label}: {path}"
    assert sha256(path) == expected, f"STOP: {label} hash mismatch."

print("✓ Recovery chain verified")


# ------------------------------------------------------------------
# 2. Verify frozen runner + benchmark
# ------------------------------------------------------------------

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

DATASET_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)

assert sha256(RUNNER) == RUNNER_SHA
assert sha256(DATASET) == DATASET_SHA

print("✓ Frozen runner verified")
print("✓ Frozen benchmark verified")


# ------------------------------------------------------------------
# 3. Exact preregistration facts recorded before runtime reset
# ------------------------------------------------------------------

ORIGINAL_PREREG_JSON_SHA = (
    "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90"
)

ORIGINAL_PREREG_MD_SHA = (
    "2708091a86ab9f3432fceea31a4ddf26becd635d24c3605c46bec68a67218e54"
)

ORIGINAL_PREREG_MANIFEST_SHA = (
    "7867181975990e1c46359dedda988e129b09a9aa1c35165e13f1c6cb5f8c3b8a"
)

arms = [
    {
        "arm": "A7",
        "model": "arize-ai/qwen-2-1.5b-instruct",
        "role": "small Qwen-derived / scale sensitivity",
    },
    {
        "arm": "A8",
        "model": "Prism-ML/Ternary-Bonsai-27B",
        "role": "medium-scale independent family",
    },
    {
        "arm": "A9",
        "model": "deepseek-ai/DeepSeek-V4.1-Flash",
        "role": "DeepSeek family",
    },
    {
        "arm": "A10",
        "model": "openai/gpt-oss-120b",
        "role": "GPT-OSS family / large model",
    },
]

execution_order = ["A7", "A8", "A9", "A10"]

print("✓ Exact historical A7–A10 arm list loaded")
print("✓ Exact historical execution order loaded")


# ------------------------------------------------------------------
# 4. Create RECOVERY representation.
#
# IMPORTANT:
# We do NOT claim this file is byte-identical to the lost original
# preregistration. Original SHA values are preserved separately.
# ------------------------------------------------------------------

reconstructed = {
    "record_type":
        "RECOVERED_PREREGISTRATION_REPRESENTATION",

    "created_after_colab_reset_utc":
        datetime.now(timezone.utc).isoformat(),

    "historical_event":
        "Experiment A A7-A10 extension preregistration",

    "historical_original_artifacts": {
        "json_sha256":
            ORIGINAL_PREREG_JSON_SHA,

        "markdown_sha256":
            ORIGINAL_PREREG_MD_SHA,

        "manifest_sha256":
            ORIGINAL_PREREG_MANIFEST_SHA,

        "recovery_status":
            "ORIGINAL_BYTES_LOST_AFTER_COLAB_RUNTIME_RESET",
    },

    "research_question": (
        "With the deterministic E3-v3 cascade and 120-case benchmark "
        "held fixed, how sensitive are decision performance and "
        "deterministic containment to the semantic parser model?"
    ),

    "selection_basis": (
        "A7-A10 were prospective extension arms selected using "
        "non-benchmark endpoint accessibility screening and predefined "
        "model-diversity criteria. No held-out benchmark performance "
        "was used to select these arms."
    ),

    "relationship_to_A3_A6": (
        "A7-A10 are prospective extension arms and are not replacements "
        "for unavailable original arms A3-A6."
    ),

    "arms":
        arms,

    "execution_order":
        execution_order,

    "frozen_variables": {
        "runner_sha256":
            RUNNER_SHA,

        "benchmark_sha256":
            DATASET_SHA,

        "variable_allowed_to_change":
            "semantic parser model only",

        "runner_changes_allowed":
            False,

        "benchmark_changes_allowed":
            False,

        "prompt_changes_allowed":
            False,

        "gate_changes_allowed":
            False,

        "threshold_changes_allowed":
            False,

        "label_changes_allowed":
            False,

        "case_removal_allowed":
            False,
    },

    "execution_policy": {
        "A1_A2_rerun":
            False,

        "result_driven_substitution":
            False,

        "result_driven_removal":
            False,

        "performance_driven_rerun":
            False,

        "failed_extension_arm_silently_replaced":
            False,

        "report_all_extension_arms":
            True,
    },

    "primary_metrics": [
        "decision_accuracy",
        "valid_acceptance_rate_VAR",
        "invalid_rejection_rate_IRR",
        "containment_violations",
    ],

    "containment_definition": (
        "accepted == False AND descriptor_json is present"
    ),

    "containment_prediction": (
        "Decision metrics may vary by semantic parser model; "
        "containment violations are expected to remain zero."
    ),

    "recovery_disclosure": (
        "This JSON was created after the Colab runtime reset from "
        "historical preregistration facts and hashes recorded before "
        "the reset. It is not represented as byte-identical to the "
        "lost original preregistration artifact."
    ),

    "completed_before_reset":
        ["A7", "A8", "A9"],

    "not_executed_before_reset":
        ["A10"],

    "model_or_api_call_during_reconstruction":
        False,
}


LOCAL_JSON = (
    LOCAL_PREREG /
    "experiment_A_extension_A7_A10_preregistration_RECOVERED_v1.json"
)

DRIVE_JSON = (
    DRIVE_PREREG /
    "experiment_A_extension_A7_A10_preregistration_RECOVERED_v1.json"
)

assert not LOCAL_JSON.exists()
assert not DRIVE_JSON.exists()

payload = json.dumps(
    reconstructed,
    indent=2
) + "\n"

LOCAL_JSON.write_text(
    payload,
    encoding="utf-8"
)

DRIVE_JSON.write_text(
    payload,
    encoding="utf-8"
)

assert sha256(LOCAL_JSON) == sha256(DRIVE_JSON)

print("✓ Recovered preregistration representation created")
print("✓ Recovered preregistration persisted to Drive")


# ------------------------------------------------------------------
# 5. Create recovery manifest
# ------------------------------------------------------------------

recovery_manifest = {
    "manifest_id":
        "experiment-A-extension-preregistration-recovery-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "recovered_representation": {
        "local_path":
            str(LOCAL_JSON),

        "drive_path":
            str(DRIVE_JSON),

        "sha256":
            sha256(LOCAL_JSON),
    },

    "historical_original_hashes": {
        "original_json_sha256":
            ORIGINAL_PREREG_JSON_SHA,

        "original_markdown_sha256":
            ORIGINAL_PREREG_MD_SHA,

        "original_manifest_sha256":
            ORIGINAL_PREREG_MANIFEST_SHA,
    },

    "byte_identical_to_original":
        False,

    "reason_not_byte_identical": (
        "Original transient preregistration files were lost in "
        "the Colab runtime reset."
    ),

    "historical_facts_changed":
        False,

    "model_or_api_call":
        False,

    "A10_executed":
        False,
}

LOCAL_RECOVERY_MANIFEST = (
    LOCAL_MANIFESTS /
    "experiment_A_extension_preregistration_recovery_manifest_v1.json"
)

DRIVE_RECOVERY_MANIFEST = (
    DRIVE_MANIFESTS /
    "experiment_A_extension_preregistration_recovery_manifest_v1.json"
)

assert not LOCAL_RECOVERY_MANIFEST.exists()
assert not DRIVE_RECOVERY_MANIFEST.exists()

manifest_payload = json.dumps(
    recovery_manifest,
    indent=2
) + "\n"

LOCAL_RECOVERY_MANIFEST.write_text(
    manifest_payload,
    encoding="utf-8"
)

DRIVE_RECOVERY_MANIFEST.write_text(
    manifest_payload,
    encoding="utf-8"
)

assert (
    sha256(LOCAL_RECOVERY_MANIFEST)
    ==
    sha256(DRIVE_RECOVERY_MANIFEST)
)

print("✓ Preregistration recovery manifest created")
print("✓ Preregistration recovery manifest persisted to Drive")


# ------------------------------------------------------------------
# 6. Verify exact A10 status
# ------------------------------------------------------------------

with open(LOCAL_JSON, "r", encoding="utf-8") as f:
    check = json.load(f)

assert check["execution_order"] == [
    "A7", "A8", "A9", "A10"
]

a10 = [
    x for x in check["arms"]
    if x["arm"] == "A10"
][0]

assert a10["model"] == "openai/gpt-oss-120b"

assert check["not_executed_before_reset"] == ["A10"]

print("\n✓ A10 exact model verified: openai/gpt-oss-120b")
print("✓ A10 historical status verified: NOT EXECUTED")


# ------------------------------------------------------------------
# 7. Report hashes
# ------------------------------------------------------------------

print("\n" + "-" * 110)
print("RECOVERED PREREGISTRATION ARTIFACTS")
print("-" * 110)

print("Recovered representation:")
print(LOCAL_JSON)
print("SHA-256:")
print(sha256(LOCAL_JSON))

print("\nDrive copy:")
print(DRIVE_JSON)
print("SHA-256:")
print(sha256(DRIVE_JSON))

print("\nRecovery manifest:")
print(LOCAL_RECOVERY_MANIFEST)
print("SHA-256:")
print(sha256(LOCAL_RECOVERY_MANIFEST))


print("\n" + "-" * 110)
print("HISTORICAL ORIGINAL HASHES — FOR PROVENANCE ONLY")
print("-" * 110)

print("Original prereg JSON     :", ORIGINAL_PREREG_JSON_SHA)
print("Original prereg Markdown :", ORIGINAL_PREREG_MD_SHA)
print("Original prereg manifest :", ORIGINAL_PREREG_MANIFEST_SHA)


print("\n" + "=" * 110)
print("✓ RESTORE R9 COMPLETE")
print("✓ A7–A10 HISTORICAL PREREGISTRATION FACTS RESTORED")
print("✓ ORIGINAL LOST HASHES PRESERVED")
print("✓ RECOVERED FILE NOT MISREPRESENTED AS ORIGINAL")
print("✓ A7–A9 NOT RERUN")
print("✓ A10 STILL NEVER EXECUTED")
print("✓ RECOVERY ARTIFACTS PERSISTED TO DRIVE")
print("✓ NO MODEL/API CALL")
print("✓ SEND THIS OUTPUT BEFORE A10 EXECUTION")
print("=" * 110)

RESTORE R9 — RECONSTRUCT AND PERSIST A7–A10 EXTENSION PREREGISTRATION
✓ Recovery chain verified
✓ Frozen runner verified
✓ Frozen benchmark verified
✓ Exact historical A7–A10 arm list loaded
✓ Exact historical execution order loaded
✓ Recovered preregistration representation created
✓ Recovered preregistration persisted to Drive
✓ Preregistration recovery manifest created
✓ Preregistration recovery manifest persisted to Drive

✓ A10 exact model verified: openai/gpt-oss-120b
✓ A10 historical status verified: NOT EXECUTED

--------------------------------------------------------------------------------------------------------------
RECOVERED PREREGISTRATION ARTIFACTS
--------------------------------------------------------------------------------------------------------------
Recovered representation:
/content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep/preregistration/experiment_A_extension_A7_A10_preregistration_RECOVERED_v1.json
SHA-256:
9cd89

In [12]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import json
import os
import sys
import time
import shutil
import pandas as pd

print("=" * 110)
print("STEP A38-R — A10 FIRST-AND-ONLY FROZEN BENCHMARK EXECUTION + IMMEDIATE DRIVE PERSISTENCE")
print("=" * 110)

# ======================================================================================
# 0. Paths
# ======================================================================================

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RUNNER = ROOT / (
    "experiment-2b/implementation/e3_v3/"
    "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = ROOT / (
    "experiment-2b/reproducibility/"
    "e3v3_heldout_benchmark_v1.0/"
    "e3v3_heldout_benchmark_120_v1.csv"
)

RECOVERED_PREREG = (
    EXP_A /
    "preregistration/"
    "experiment_A_extension_A7_A10_preregistration_RECOVERED_v1.json"
)

PREREG_RECOVERY_MANIFEST = (
    EXP_A /
    "manifests/"
    "experiment_A_extension_preregistration_recovery_manifest_v1.json"
)

RECOVERY_RECORD = (
    EXP_A /
    "recovery/"
    "experiment_A_colab_reset_recovery_v1.json"
)

RAW_DIR = EXP_A / "results/raw"
LOG_DIR = EXP_A / "logs"
MANIFEST_DIR = EXP_A / "manifests"

DRIVE_RAW_DIR = DRIVE_EXP_A / "results/raw"
DRIVE_LOG_DIR = DRIVE_EXP_A / "logs"
DRIVE_MANIFEST_DIR = DRIVE_EXP_A / "manifests"

for p in [
    RAW_DIR,
    LOG_DIR,
    MANIFEST_DIR,
    DRIVE_RAW_DIR,
    DRIVE_LOG_DIR,
    DRIVE_MANIFEST_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)


# ======================================================================================
# 1. Hash helper
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. Frozen expected hashes
# ======================================================================================

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "recovered_prereg":
        "9cd89b9f2937f686405f954628f77f7f44cc08ac79ea9499d3d98d8c1d8dd980",

    "prereg_recovery_manifest":
        "dce1582c24aeadc42013eded316d3135becb43c9ce3308d26bfd61428860bd50",

    "recovery_record":
        "21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3",
}


# ======================================================================================
# 3. Verify frozen state
# ======================================================================================

for label, path, expected in [
    ("runner", RUNNER, EXPECTED["runner"]),
    ("benchmark", DATASET, EXPECTED["dataset"]),
    (
        "recovered preregistration",
        RECOVERED_PREREG,
        EXPECTED["recovered_prereg"],
    ),
    (
        "preregistration recovery manifest",
        PREREG_RECOVERY_MANIFEST,
        EXPECTED["prereg_recovery_manifest"],
    ),
    (
        "Colab-reset recovery record",
        RECOVERY_RECORD,
        EXPECTED["recovery_record"],
    ),
]:
    assert path.exists(), f"STOP: missing {label}:\n{path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 4. Verify A10 preregistration
# ======================================================================================

with open(RECOVERED_PREREG, "r", encoding="utf-8") as f:
    prereg = json.load(f)

assert prereg["execution_order"] == [
    "A7",
    "A8",
    "A9",
    "A10",
]

a10_entries = [
    x
    for x in prereg["arms"]
    if x["arm"] == "A10"
]

assert len(a10_entries) == 1

A10_MODEL = a10_entries[0]["model"]

assert A10_MODEL == "openai/gpt-oss-120b"

assert prereg["not_executed_before_reset"] == ["A10"]

print("✓ A10 preregistered model verified:", A10_MODEL)
print("✓ Historical A10 status verified: never executed")


# ======================================================================================
# 5. API-key check
# ======================================================================================

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY is not present. "
    "Do not run benchmark until API key is restored."
)

print("✓ TOGETHER_API_KEY present")
print("✓ API key value not printed")


# ======================================================================================
# 6. Define output artifacts
# ======================================================================================

RESULT = (
    RAW_DIR /
    "A10_gpt-oss-120b_results.csv"
)

LOG = (
    LOG_DIR /
    "A10_gpt-oss-120b_execution.log"
)

EXEC_MANIFEST = (
    MANIFEST_DIR /
    "A10_execution_manifest_v1.json"
)

DRIVE_RESULT = (
    DRIVE_RAW_DIR /
    RESULT.name
)

DRIVE_LOG = (
    DRIVE_LOG_DIR /
    LOG.name
)

DRIVE_EXEC_MANIFEST = (
    DRIVE_MANIFEST_DIR /
    EXEC_MANIFEST.name
)


# ======================================================================================
# 7. Strict first-execution guard
# ======================================================================================

for path in [
    RESULT,
    LOG,
    EXEC_MANIFEST,
    DRIVE_RESULT,
    DRIVE_LOG,
    DRIVE_EXEC_MANIFEST,
]:
    assert not path.exists(), (
        "STOP: A10 artifact already exists:\n"
        f"{path}\n"
        "Do NOT overwrite or rerun automatically."
    )

print("✓ No local A10 artifacts exist")
print("✓ No persistent Drive A10 artifacts exist")
print("✓ First-execution guard passed")


# ======================================================================================
# 8. Verify benchmark structure before execution
# ======================================================================================

benchmark = pd.read_csv(DATASET)

assert len(benchmark) == 120, (
    f"STOP: expected 120 benchmark rows; found {len(benchmark)}."
)

assert "id" in benchmark.columns

assert benchmark["id"].nunique() == 120

print("✓ Benchmark contains 120 rows")
print("✓ Benchmark contains 120 unique case IDs")


# ======================================================================================
# 9. Exact frozen command
# ======================================================================================

cmd = [
    sys.executable,
    str(RUNNER),

    "--dataset",
    str(DATASET),

    "--out",
    str(RESULT),

    "--model",
    A10_MODEL,
]

print("\n" + "-" * 110)
print("A10 EXECUTION")
print("-" * 110)

print("Model :", A10_MODEL)
print("Cases : 120")
print("Limit : NONE")
print("Runner:", RUNNER)
print("Output:", RESULT)

print("\nExecuting A10 exactly once...")


# ======================================================================================
# 10. Execute exactly once
# ======================================================================================

started_utc = datetime.now(timezone.utc).isoformat()
t0 = time.time()

with open(LOG, "w", encoding="utf-8") as log_file:

    process = subprocess.run(
        cmd,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        text=True,
    )

elapsed = time.time() - t0
finished_utc = datetime.now(timezone.utc).isoformat()

print("Process return code:", process.returncode)
print("Wall-clock seconds :", round(elapsed, 2))


# ======================================================================================
# 11. STOP on execution failure
# ======================================================================================

if process.returncode != 0:

    print("\n" + "!" * 110)
    print("A10 PROCESS FAILED")
    print("!" * 110)

    print("Log preserved locally:")
    print(LOG)

    # Preserve failure log immediately.
    shutil.copy2(
        LOG,
        DRIVE_LOG
    )

    assert sha256(LOG) == sha256(DRIVE_LOG)

    print("Failure log persisted to Drive:")
    print(DRIVE_LOG)

    print("\nDO NOT RERUN A10.")
    print("SEND THIS OUTPUT FOR DIAGNOSIS.")

    raise RuntimeError(
        f"A10 process returned {process.returncode}"
    )


# ======================================================================================
# 12. Validate raw result BEFORE analysis
# ======================================================================================

assert RESULT.exists(), (
    "STOP: runner returned success but result CSV does not exist."
)

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: A10 result has {len(df)} rows instead of 120."
)

assert "id" in df.columns

assert df["id"].nunique() == 120, (
    "STOP: A10 result does not contain 120 unique IDs."
)

expected_ids = set(benchmark["id"].astype(str))
result_ids = set(df["id"].astype(str))

assert result_ids == expected_ids, (
    "STOP: A10 result case IDs differ from frozen benchmark."
)

print("\n✓ A10 result CSV exists")
print("✓ A10 result contains 120 rows")
print("✓ A10 result contains 120 unique case IDs")
print("✓ Result IDs exactly match frozen benchmark")


# ======================================================================================
# 13. Hash raw evidence immediately
# ======================================================================================

RESULT_SHA = sha256(RESULT)
LOG_SHA = sha256(LOG)

print("\nA10 raw result SHA-256:")
print(RESULT_SHA)

print("\nA10 execution log SHA-256:")
print(LOG_SHA)


# ======================================================================================
# 14. Create execution manifest BEFORE metrics analysis
# ======================================================================================

execution_manifest = {
    "manifest_id":
        "experiment-A-A10-execution-v1",

    "arm":
        "A10",

    "model":
        A10_MODEL,

    "execution_policy":
        "FIRST_AND_ONLY_PREREGISTERED_EXECUTION",

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "wall_clock_seconds":
        elapsed,

    "process_return_code":
        process.returncode,

    "benchmark_cases":
        120,

    "unique_case_ids":
        120,

    "limit_used":
        False,

    "frozen_runner": {
        "path":
            str(RUNNER.relative_to(ROOT)),

        "sha256":
            sha256(RUNNER),
    },

    "frozen_benchmark": {
        "path":
            str(DATASET.relative_to(ROOT)),

        "sha256":
            sha256(DATASET),
    },

    "preregistration_recovery": {
        "path":
            str(RECOVERED_PREREG.relative_to(ROOT)),

        "sha256":
            sha256(RECOVERED_PREREG),

        "historical_original_prereg_json_sha256":
            "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",
    },

    "raw_result": {
        "path":
            str(RESULT.relative_to(ROOT)),

        "sha256":
            RESULT_SHA,
    },

    "execution_log": {
        "path":
            str(LOG.relative_to(ROOT)),

        "sha256":
            LOG_SHA,
    },

    "completed_arms_rerun":
        False,

    "post_hoc_model_substitution":
        False,

    "metrics_analyzed_before_raw_persistence":
        False,

    "runtime_recovery_context":
        True,
}

EXEC_MANIFEST.write_text(
    json.dumps(
        execution_manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)

EXEC_MANIFEST_SHA = sha256(
    EXEC_MANIFEST
)

print("\n✓ A10 execution manifest created")
print("Execution manifest SHA-256:")
print(EXEC_MANIFEST_SHA)


# ======================================================================================
# 15. IMMEDIATE persistent Drive copy
# ======================================================================================

print("\n" + "-" * 110)
print("IMMEDIATE DRIVE PERSISTENCE")
print("-" * 110)

shutil.copy2(
    RESULT,
    DRIVE_RESULT
)

shutil.copy2(
    LOG,
    DRIVE_LOG
)

shutil.copy2(
    EXEC_MANIFEST,
    DRIVE_EXEC_MANIFEST
)

assert sha256(DRIVE_RESULT) == RESULT_SHA
assert sha256(DRIVE_LOG) == LOG_SHA
assert sha256(DRIVE_EXEC_MANIFEST) == EXEC_MANIFEST_SHA

print("✓ Raw A10 CSV persisted to Drive")
print("✓ A10 execution log persisted to Drive")
print("✓ A10 execution manifest persisted to Drive")
print("✓ All three Drive hashes verified byte-for-byte")


# ======================================================================================
# 16. Lightweight execution diagnostics ONLY
#
# No performance metrics are calculated here.
# ======================================================================================

def bool_count(column):
    if column not in df.columns:
        return None

    s = df[column]

    if s.dtype == bool:
        return int(s.sum())

    normalized = (
        s.astype(str)
        .str.strip()
        .str.lower()
    )

    return int(
        normalized.isin(
            ["true", "1", "yes"]
        ).sum()
    )


llm_invoked = bool_count(
    "llm_invoked"
)

parse_ok = bool_count(
    "parse_ok"
)

accepted = bool_count(
    "accepted"
)

exception_rows = None

if "exception" in df.columns:

    exception_rows = int(
        df["exception"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

print("\n" + "-" * 110)
print("LIGHTWEIGHT EXECUTION DIAGNOSTIC")
print("-" * 110)

print("LLM invoked :", llm_invoked)
print("Parse OK    :", parse_ok)
print("Accepted    :", accepted)
print("Exceptions  :", exception_rows)

if "rejection_stage" in df.columns:

    print("\nRejection stages:")

    counts = (
        df["rejection_stage"]
        .fillna("<none>")
        .astype(str)
        .value_counts(dropna=False)
    )

    print(counts.to_string())


# ======================================================================================
# 17. Final
# ======================================================================================

print("\n" + "=" * 110)
print("✓ A10 FIRST-AND-ONLY EXECUTION COMPLETE")
print("✓ 120 / 120 RESULT ROWS VERIFIED")
print("✓ CASE IDS MATCH FROZEN BENCHMARK")
print("✓ RAW RESULT HASHED")
print("✓ EXECUTION LOG HASHED")
print("✓ EXECUTION MANIFEST CREATED")
print("✓ RAW EVIDENCE IMMEDIATELY PERSISTED TO DRIVE")
print("✓ DRIVE COPIES VERIFIED BYTE-FOR-BYTE")
print("✓ NO A1-A9 ARM RERUN")
print("✓ NO MODEL SUBSTITUTION")
print("✓ NO PERFORMANCE METRICS CALCULATED YET")
print("✓ DO NOT RERUN A10")
print("✓ SEND THIS OUTPUT BEFORE A10 METRICS ANALYSIS")
print("=" * 110)

STEP A38-R — A10 FIRST-AND-ONLY FROZEN BENCHMARK EXECUTION + IMMEDIATE DRIVE PERSISTENCE
✓ runner verified
✓ benchmark verified
✓ recovered preregistration verified
✓ preregistration recovery manifest verified
✓ Colab-reset recovery record verified
✓ A10 preregistered model verified: openai/gpt-oss-120b
✓ Historical A10 status verified: never executed
✓ TOGETHER_API_KEY present
✓ API key value not printed
✓ No local A10 artifacts exist
✓ No persistent Drive A10 artifacts exist
✓ First-execution guard passed
✓ Benchmark contains 120 rows
✓ Benchmark contains 120 unique case IDs

--------------------------------------------------------------------------------------------------------------
A10 EXECUTION
--------------------------------------------------------------------------------------------------------------
Model : openai/gpt-oss-120b
Cases : 120
Limit : NONE
Runner: /content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/run_experiment2b_e3v3_r7_dev_v1.p

In [13]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import pandas as pd
from scipy.stats import beta

print("=" * 110)
print("STEP A39 — A10 FROZEN METRICS + EXACT CI + CONTAINMENT ANALYSIS")
print("=" * 110)

# ======================================================================================
# 0. Paths
# ======================================================================================

ROOT = Path("/content/deterministic-intent-translation-journal")

EXP_A = (
    ROOT /
    "journal_experiments/experiment_A_model_sweep"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RESULT = (
    EXP_A /
    "results/raw/"
    "A10_gpt-oss-120b_results.csv"
)

LOG = (
    EXP_A /
    "logs/"
    "A10_gpt-oss-120b_execution.log"
)

EXEC_MANIFEST = (
    EXP_A /
    "manifests/"
    "A10_execution_manifest_v1.json"
)

METRICS_DIR = EXP_A / "results/metrics"
ANALYSIS_DIR = EXP_A / "results/analysis"
MANIFEST_DIR = EXP_A / "manifests"

DRIVE_METRICS_DIR = DRIVE_EXP_A / "results/metrics"
DRIVE_ANALYSIS_DIR = DRIVE_EXP_A / "results/analysis"
DRIVE_MANIFEST_DIR = DRIVE_EXP_A / "manifests"

for p in [
    METRICS_DIR,
    ANALYSIS_DIR,
    MANIFEST_DIR,
    DRIVE_METRICS_DIR,
    DRIVE_ANALYSIS_DIR,
    DRIVE_MANIFEST_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def to_bool(series):
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )


def cp_interval(successes, total, alpha=0.05):
    """
    Two-sided exact Clopper-Pearson interval.
    """

    if total == 0:
        return None, None

    if successes == 0:
        lower = 0.0
    else:
        lower = beta.ppf(
            alpha / 2,
            successes,
            total - successes + 1,
        )

    if successes == total:
        upper = 1.0
    else:
        upper = beta.ppf(
            1 - alpha / 2,
            successes + 1,
            total - successes,
        )

    return float(lower), float(upper)


# ======================================================================================
# 2. Verify immutable A10 evidence
# ======================================================================================

EXPECTED = {
    "result":
        "998e68f0f484c929864f6a35cb6ae018ee2d22c2cf795c34b6279e6264895207",

    "log":
        "a52968aec69da31600e59ac6fb59e7ac2babdc369e56a16e6a94afd2acb4c64d",

    "execution_manifest":
        "83d607e6f2a3b652a481dad6558de1e76f631b68e74185ab742920a89424e823",
}

for label, path, expected in [
    ("A10 raw result", RESULT, EXPECTED["result"]),
    ("A10 execution log", LOG, EXPECTED["log"]),
    (
        "A10 execution manifest",
        EXEC_MANIFEST,
        EXPECTED["execution_manifest"],
    ),
]:

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} changed.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 3. Verify Drive copies
# ======================================================================================

DRIVE_RESULT = (
    DRIVE_EXP_A /
    "results/raw/"
    RESULT.name
)

DRIVE_LOG = (
    DRIVE_EXP_A /
    "logs/"
    LOG.name
)

DRIVE_EXEC = (
    DRIVE_EXP_A /
    "manifests/"
    EXEC_MANIFEST.name
)

assert sha256(DRIVE_RESULT) == EXPECTED["result"]
assert sha256(DRIVE_LOG) == EXPECTED["log"]
assert sha256(DRIVE_EXEC) == EXPECTED["execution_manifest"]

print("✓ Persistent Drive A10 evidence verified")


# ======================================================================================
# 4. Load result
# ======================================================================================

df = pd.read_csv(RESULT)

assert len(df) == 120
assert df["id"].nunique() == 120

required = [
    "expected_valid",
    "accepted",
    "llm_invoked",
    "parse_ok",
    "descriptor_json",
    "rejection_stage",
    "exception",
]

for col in required:
    assert col in df.columns, (
        f"STOP: required column missing: {col}"
    )

expected_valid = to_bool(
    df["expected_valid"]
)

accepted = to_bool(
    df["accepted"]
)

llm_invoked = to_bool(
    df["llm_invoked"]
)

parse_ok = to_bool(
    df["parse_ok"]
)

valid_n = int(expected_valid.sum())
invalid_n = int((~expected_valid).sum())

assert valid_n == 58
assert invalid_n == 62

print("✓ 120 rows / 120 unique cases verified")
print("✓ Valid cases   :", valid_n)
print("✓ Invalid cases :", invalid_n)


# ======================================================================================
# 5. Confusion matrix
#
# Invalid/reject is treated as the positive condition,
# matching prior Experiment A reporting.
# ======================================================================================

tp = int(
    ((~expected_valid) & (~accepted)).sum()
)

fp = int(
    (expected_valid & (~accepted)).sum()
)

fn = int(
    ((~expected_valid) & accepted).sum()
)

tn = int(
    (expected_valid & accepted).sum()
)

assert tp + fp + fn + tn == 120

correct = tp + tn

print("\n" + "-" * 110)
print("CONFUSION MATRIX")
print("-" * 110)

print("TP — invalid rejected :", tp)
print("FP — valid rejected   :", fp)
print("FN — invalid accepted :", fn)
print("TN — valid accepted   :", tn)
print("Correct decisions     :", correct)


# ======================================================================================
# 6. Primary metrics
# ======================================================================================

accuracy = correct / 120
var = tn / valid_n
irr = tp / invalid_n

precision = (
    tp / (tp + fp)
    if (tp + fp)
    else 0.0
)

recall = (
    tp / (tp + fn)
    if (tp + fn)
    else 0.0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0.0
)

acc_ci = cp_interval(
    correct,
    120,
)

var_ci = cp_interval(
    tn,
    valid_n,
)

irr_ci = cp_interval(
    tp,
    invalid_n,
)

print("\n" + "-" * 110)
print("PRIMARY METRICS")
print("-" * 110)

print(
    f"Accuracy : {correct}/120 = "
    f"{accuracy:.6f} "
    f"({accuracy*100:.2f}%)"
)

print(
    f"CP95     : "
    f"[{acc_ci[0]*100:.2f}%, "
    f"{acc_ci[1]*100:.2f}%]"
)

print()

print(
    f"VAR      : {tn}/{valid_n} = "
    f"{var:.6f} "
    f"({var*100:.2f}%)"
)

print(
    f"CP95     : "
    f"[{var_ci[0]*100:.2f}%, "
    f"{var_ci[1]*100:.2f}%]"
)

print()

print(
    f"IRR      : {tp}/{invalid_n} = "
    f"{irr:.6f} "
    f"({irr*100:.2f}%)"
)

print(
    f"CP95     : "
    f"[{irr_ci[0]*100:.2f}%, "
    f"{irr_ci[1]*100:.2f}%]"
)

print()

print(
    f"Rejection precision : {precision:.6f}"
)

print(
    f"Rejection recall    : {recall:.6f}"
)

print(
    f"Rejection F1        : {f1:.6f}"
)


# ======================================================================================
# 7. LLM / parsing status
# ======================================================================================

llm_n = int(
    llm_invoked.sum()
)

parse_success = int(
    (llm_invoked & parse_ok).sum()
)

parse_failure = int(
    (llm_invoked & (~parse_ok)).sum()
)

exception_rows = int(
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("\n" + "-" * 110)
print("LLM EXECUTION STATUS")
print("-" * 110)

print("LLM invoked   :", llm_n)
print("Parse success :", parse_success)
print("Parse failure :", parse_failure)
print("Exceptions    :", exception_rows)

assert llm_n == 77
assert parse_success == 77
assert parse_failure == 0
assert exception_rows == 0

print("✓ A10 is an EVALUABLE model arm")


# ======================================================================================
# 8. Descriptor / containment analysis
#
# Frozen operational definition:
# containment violation =
# accepted == False AND descriptor_json present
# ======================================================================================

descriptor_text = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = (
    descriptor_text.ne("")
    &
    ~descriptor_text.str.lower().isin(
        [
            "none",
            "null",
            "nan",
            "{}",
        ]
    )
)

accepted_n = int(
    accepted.sum()
)

rejected_n = int(
    (~accepted).sum()
)

descriptor_n = int(
    descriptor_present.sum()
)

containment_violations = int(
    ((~accepted) & descriptor_present).sum()
)

accepted_without_descriptor = int(
    (accepted & (~descriptor_present)).sum()
)

print("\n" + "-" * 110)
print("DETERMINISTIC CONTAINMENT")
print("-" * 110)

print("Accepted cases              :", accepted_n)
print("Rejected cases              :", rejected_n)
print("Descriptors emitted         :", descriptor_n)
print("Containment violations      :", containment_violations)
print("Accepted without descriptor :", accepted_without_descriptor)

assert containment_violations == 0
assert accepted_without_descriptor == 0

print("✓ ZERO observed containment violations")


# ======================================================================================
# 9. Rejection stages
# ======================================================================================

stage_counts = (
    df["rejection_stage"]
    .fillna("<none>")
    .astype(str)
    .value_counts()
    .to_dict()
)

print("\n" + "-" * 110)
print("REJECTION STAGES")
print("-" * 110)

for stage, count in stage_counts.items():
    print(
        f"{stage:<32} {count}"
    )


# ======================================================================================
# 10. Compare A10 against frozen A1 aggregate baseline
# ======================================================================================

A1 = {
    "accuracy": 101 / 120,
    "VAR": 51 / 58,
    "IRR": 50 / 62,
}

delta = {
    "accuracy_percentage_points":
        (accuracy - A1["accuracy"]) * 100,

    "VAR_percentage_points":
        (var - A1["VAR"]) * 100,

    "IRR_percentage_points":
        (irr - A1["IRR"]) * 100,
}

print("\n" + "-" * 110)
print("A10 vs A1 — AGGREGATE COMPARISON")
print("-" * 110)

print(
    f"Accuracy delta : "
    f"{delta['accuracy_percentage_points']:+.2f} pp"
)

print(
    f"VAR delta      : "
    f"{delta['VAR_percentage_points']:+.2f} pp"
)

print(
    f"IRR delta      : "
    f"{delta['IRR_percentage_points']:+.2f} pp"
)


# ======================================================================================
# 11. Freeze metrics artifact
# ======================================================================================

METRICS = (
    METRICS_DIR /
    "A10_metrics_v1.json"
)

assert not METRICS.exists(), (
    f"STOP: metrics artifact already exists:\n{METRICS}"
)

metrics_record = {
    "arm":
        "A10",

    "model":
        "openai/gpt-oss-120b",

    "status":
        "EVALUABLE",

    "raw_result_sha256":
        EXPECTED["result"],

    "sample": {
        "total":
            120,

        "valid":
            valid_n,

        "invalid":
            invalid_n,
    },

    "confusion_matrix_invalid_reject_positive": {
        "TP":
            tp,

        "FP":
            fp,

        "FN":
            fn,

        "TN":
            tn,
    },

    "accuracy": {
        "numerator":
            correct,

        "denominator":
            120,

        "value":
            accuracy,

        "cp95_lower":
            acc_ci[0],

        "cp95_upper":
            acc_ci[1],
    },

    "VAR": {
        "numerator":
            tn,

        "denominator":
            valid_n,

        "value":
            var,

        "cp95_lower":
            var_ci[0],

        "cp95_upper":
            var_ci[1],
    },

    "IRR": {
        "numerator":
            tp,

        "denominator":
            invalid_n,

        "value":
            irr,

        "cp95_lower":
            irr_ci[0],

        "cp95_upper":
            irr_ci[1],
    },

    "rejection_precision":
        precision,

    "rejection_recall":
        recall,

    "rejection_f1":
        f1,

    "llm": {
        "invoked":
            llm_n,

        "parse_success":
            parse_success,

        "parse_failure":
            parse_failure,

        "exception_rows":
            exception_rows,
    },

    "containment": {
        "definition":
            "accepted == False AND descriptor_json present",

        "accepted":
            accepted_n,

        "rejected":
            rejected_n,

        "descriptors_emitted":
            descriptor_n,

        "violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "rejection_stage_counts":
        stage_counts,

    "A1_aggregate_delta_percentage_points":
        delta,

    "interpretation_boundary": (
        "This is an evaluable model arm. Aggregate metric "
        "differences demonstrate model sensitivity under the frozen "
        "pipeline; they do not establish causality attributable solely "
        "to model family, architecture, or parameter count."
    ),
}

METRICS.write_text(
    json.dumps(
        metrics_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)

METRICS_SHA = sha256(
    METRICS
)


# ======================================================================================
# 12. Freeze analysis manifest
# ======================================================================================

ANALYSIS_MANIFEST = (
    MANIFEST_DIR /
    "A10_analysis_manifest_v1.json"
)

assert not ANALYSIS_MANIFEST.exists()

analysis_manifest = {
    "manifest_id":
        "experiment-A-A10-analysis-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A10",

    "model":
        "openai/gpt-oss-120b",

    "status":
        "EVALUABLE",

    "raw_result": {
        "sha256":
            EXPECTED["result"],
    },

    "execution_log": {
        "sha256":
            EXPECTED["log"],
    },

    "execution_manifest": {
        "sha256":
            EXPECTED["execution_manifest"],
    },

    "metrics": {
        "path":
            str(METRICS.relative_to(ROOT)),

        "sha256":
            METRICS_SHA,
    },

    "containment_violations":
        containment_violations,

    "model_or_api_call_during_analysis":
        False,

    "raw_result_modified":
        False,

    "rerun_performed":
        False,
}

ANALYSIS_MANIFEST.write_text(
    json.dumps(
        analysis_manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)

ANALYSIS_MANIFEST_SHA = sha256(
    ANALYSIS_MANIFEST
)


# ======================================================================================
# 13. Persist metrics + manifest immediately to Drive
# ======================================================================================

DRIVE_METRICS = (
    DRIVE_METRICS_DIR /
    METRICS.name
)

DRIVE_ANALYSIS_MANIFEST = (
    DRIVE_MANIFEST_DIR /
    ANALYSIS_MANIFEST.name
)

assert not DRIVE_METRICS.exists()
assert not DRIVE_ANALYSIS_MANIFEST.exists()

shutil.copy2(
    METRICS,
    DRIVE_METRICS
)

shutil.copy2(
    ANALYSIS_MANIFEST,
    DRIVE_ANALYSIS_MANIFEST
)

assert (
    sha256(DRIVE_METRICS)
    == METRICS_SHA
)

assert (
    sha256(DRIVE_ANALYSIS_MANIFEST)
    == ANALYSIS_MANIFEST_SHA
)


# ======================================================================================
# 14. Final artifact report
# ======================================================================================

print("\n" + "-" * 110)
print("FROZEN A10 ANALYSIS ARTIFACTS")
print("-" * 110)

print("Metrics:")
print(METRICS)
print("SHA-256:")
print(METRICS_SHA)

print("\nAnalysis manifest:")
print(ANALYSIS_MANIFEST)
print("SHA-256:")
print(ANALYSIS_MANIFEST_SHA)

print("\n✓ Metrics persisted to Drive")
print("✓ Analysis manifest persisted to Drive")
print("✓ Drive hashes verified")


print("\n" + "=" * 110)
print("✓ STEP A39 COMPLETE")
print("✓ A10 CLASSIFIED EVALUABLE")
print("✓ EXACT CLOPPER–PEARSON INTERVALS CALCULATED")
print("✓ CONTAINMENT CHECK COMPLETE")
print("✓ ZERO OBSERVED CONTAINMENT VIOLATIONS")
print("✓ RAW A10 RESULT UNCHANGED")
print("✓ NO MODEL/API CALL DURING ANALYSIS")
print("✓ NO A10 RERUN")
print("✓ METRICS + MANIFEST PERSISTED TO DRIVE")
print("✓ SEND THIS OUTPUT BEFORE SIX-MODEL CONSOLIDATION")
print("=" * 110)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4186797079.py, line 168)

In [14]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import pandas as pd
from scipy.stats import beta

print("=" * 110)
print("STEP A39-FIX — A10 FROZEN METRICS + EXACT CI + CONTAINMENT ANALYSIS")
print("=" * 110)

# ======================================================================================
# 0. Paths
# ======================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

EXP_A = (
    ROOT
    / "journal_experiments"
    / "experiment_A_model_sweep"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RESULT = (
    EXP_A
    / "results"
    / "raw"
    / "A10_gpt-oss-120b_results.csv"
)

LOG = (
    EXP_A
    / "logs"
    / "A10_gpt-oss-120b_execution.log"
)

EXEC_MANIFEST = (
    EXP_A
    / "manifests"
    / "A10_execution_manifest_v1.json"
)

METRICS_DIR = EXP_A / "results" / "metrics"
MANIFEST_DIR = EXP_A / "manifests"

DRIVE_METRICS_DIR = DRIVE_EXP_A / "results" / "metrics"
DRIVE_MANIFEST_DIR = DRIVE_EXP_A / "manifests"

for directory in [
    METRICS_DIR,
    MANIFEST_DIR,
    DRIVE_METRICS_DIR,
    DRIVE_MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def to_bool(series):
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )


def cp_interval(successes, total, alpha=0.05):
    """Two-sided exact Clopper-Pearson confidence interval."""

    if total == 0:
        return None, None

    if successes == 0:
        lower = 0.0
    else:
        lower = beta.ppf(
            alpha / 2,
            successes,
            total - successes + 1,
        )

    if successes == total:
        upper = 1.0
    else:
        upper = beta.ppf(
            1 - alpha / 2,
            successes + 1,
            total - successes,
        )

    return float(lower), float(upper)


# ======================================================================================
# 2. Verify immutable A10 evidence
# ======================================================================================

EXPECTED = {
    "result":
        "998e68f0f484c929864f6a35cb6ae018ee2d22c2cf795c34b6279e6264895207",

    "log":
        "a52968aec69da31600e59ac6fb59e7ac2babdc369e56a16e6a94afd2acb4c64d",

    "execution_manifest":
        "83d607e6f2a3b652a481dad6558de1e76f631b68e74185ab742920a89424e823",
}

for label, path, expected_hash in [
    ("A10 raw result", RESULT, EXPECTED["result"]),
    ("A10 execution log", LOG, EXPECTED["log"]),
    (
        "A10 execution manifest",
        EXEC_MANIFEST,
        EXPECTED["execution_manifest"],
    ),
]:
    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual_hash = sha256(path)

    assert actual_hash == expected_hash, (
        f"STOP: {label} hash mismatch.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual_hash}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 3. Verify persistent Drive evidence
# ======================================================================================

DRIVE_RESULT = (
    DRIVE_EXP_A
    / "results"
    / "raw"
    / "A10_gpt-oss-120b_results.csv"
)

DRIVE_LOG = (
    DRIVE_EXP_A
    / "logs"
    / "A10_gpt-oss-120b_execution.log"
)

DRIVE_EXEC_MANIFEST = (
    DRIVE_EXP_A
    / "manifests"
    / "A10_execution_manifest_v1.json"
)

assert DRIVE_RESULT.exists()
assert DRIVE_LOG.exists()
assert DRIVE_EXEC_MANIFEST.exists()

assert sha256(DRIVE_RESULT) == EXPECTED["result"]
assert sha256(DRIVE_LOG) == EXPECTED["log"]
assert sha256(DRIVE_EXEC_MANIFEST) == EXPECTED["execution_manifest"]

print("✓ Persistent Drive A10 evidence verified")


# ======================================================================================
# 4. Load frozen result
# ======================================================================================

df = pd.read_csv(RESULT)

assert len(df) == 120, (
    f"STOP: expected 120 rows, found {len(df)}."
)

assert "id" in df.columns

assert df["id"].nunique() == 120, (
    "STOP: expected 120 unique case IDs."
)

required_columns = [
    "expected_valid",
    "accepted",
    "llm_invoked",
    "parse_ok",
    "descriptor_json",
    "rejection_stage",
    "exception",
]

for column in required_columns:
    assert column in df.columns, (
        f"STOP: required column missing: {column}"
    )

expected_valid = to_bool(df["expected_valid"])
accepted = to_bool(df["accepted"])
llm_invoked = to_bool(df["llm_invoked"])
parse_ok = to_bool(df["parse_ok"])

valid_n = int(expected_valid.sum())
invalid_n = int((~expected_valid).sum())

assert valid_n == 58
assert invalid_n == 62

print("✓ 120 rows / 120 unique cases verified")
print("✓ Valid cases   :", valid_n)
print("✓ Invalid cases :", invalid_n)


# ======================================================================================
# 5. Confusion matrix
#
# Invalid/reject is the positive condition, matching earlier
# Experiment-A reporting.
# ======================================================================================

tp = int(
    ((~expected_valid) & (~accepted)).sum()
)

fp = int(
    (expected_valid & (~accepted)).sum()
)

fn = int(
    ((~expected_valid) & accepted).sum()
)

tn = int(
    (expected_valid & accepted).sum()
)

assert tp + fp + fn + tn == 120

correct = tp + tn

print("\n" + "-" * 110)
print("CONFUSION MATRIX")
print("-" * 110)

print("TP — invalid rejected :", tp)
print("FP — valid rejected   :", fp)
print("FN — invalid accepted :", fn)
print("TN — valid accepted   :", tn)
print("Correct decisions     :", correct)


# ======================================================================================
# 6. Primary metrics
# ======================================================================================

accuracy = correct / 120
var = tn / valid_n
irr = tp / invalid_n

precision = (
    tp / (tp + fp)
    if (tp + fp) > 0
    else 0.0
)

recall = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0.0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

acc_ci = cp_interval(correct, 120)
var_ci = cp_interval(tn, valid_n)
irr_ci = cp_interval(tp, invalid_n)

print("\n" + "-" * 110)
print("PRIMARY METRICS")
print("-" * 110)

print(
    f"Accuracy : {correct}/120 = "
    f"{accuracy:.6f} ({accuracy * 100:.2f}%)"
)

print(
    f"CP95     : "
    f"[{acc_ci[0] * 100:.2f}%, "
    f"{acc_ci[1] * 100:.2f}%]"
)

print()

print(
    f"VAR      : {tn}/{valid_n} = "
    f"{var:.6f} ({var * 100:.2f}%)"
)

print(
    f"CP95     : "
    f"[{var_ci[0] * 100:.2f}%, "
    f"{var_ci[1] * 100:.2f}%]"
)

print()

print(
    f"IRR      : {tp}/{invalid_n} = "
    f"{irr:.6f} ({irr * 100:.2f}%)"
)

print(
    f"CP95     : "
    f"[{irr_ci[0] * 100:.2f}%, "
    f"{irr_ci[1] * 100:.2f}%]"
)

print()

print(f"Rejection precision : {precision:.6f}")
print(f"Rejection recall    : {recall:.6f}")
print(f"Rejection F1        : {f1:.6f}")


# ======================================================================================
# 7. LLM / parsing status
# ======================================================================================

llm_n = int(llm_invoked.sum())

parse_success = int(
    (llm_invoked & parse_ok).sum()
)

parse_failure = int(
    (llm_invoked & (~parse_ok)).sum()
)

exception_rows = int(
    df["exception"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("\n" + "-" * 110)
print("LLM EXECUTION STATUS")
print("-" * 110)

print("LLM invoked   :", llm_n)
print("Parse success :", parse_success)
print("Parse failure :", parse_failure)
print("Exceptions    :", exception_rows)

assert llm_n == 77
assert parse_success == 77
assert parse_failure == 0
assert exception_rows == 0

print("✓ A10 is an EVALUABLE model arm")


# ======================================================================================
# 8. Descriptor / containment analysis
#
# Frozen operational definition:
# containment violation =
#     accepted == False AND descriptor_json present
# ======================================================================================

descriptor_text = (
    df["descriptor_json"]
    .fillna("")
    .astype(str)
    .str.strip()
)

descriptor_present = (
    descriptor_text.ne("")
    &
    ~descriptor_text.str.lower().isin(
        [
            "none",
            "null",
            "nan",
            "{}",
        ]
    )
)

accepted_n = int(accepted.sum())
rejected_n = int((~accepted).sum())
descriptor_n = int(descriptor_present.sum())

containment_violations = int(
    ((~accepted) & descriptor_present).sum()
)

accepted_without_descriptor = int(
    (accepted & (~descriptor_present)).sum()
)

print("\n" + "-" * 110)
print("DETERMINISTIC CONTAINMENT")
print("-" * 110)

print("Accepted cases              :", accepted_n)
print("Rejected cases              :", rejected_n)
print("Descriptors emitted         :", descriptor_n)
print("Containment violations      :", containment_violations)
print("Accepted without descriptor :", accepted_without_descriptor)

assert containment_violations == 0, (
    "STOP: containment violation detected."
)

assert accepted_without_descriptor == 0, (
    "STOP: accepted case without descriptor detected."
)

print("✓ ZERO observed containment violations")


# ======================================================================================
# 9. Rejection stages
# ======================================================================================

stage_counts = (
    df["rejection_stage"]
    .fillna("<none>")
    .astype(str)
    .value_counts()
    .to_dict()
)

print("\n" + "-" * 110)
print("REJECTION STAGES")
print("-" * 110)

for stage, count in stage_counts.items():
    print(f"{stage:<32} {count}")


# ======================================================================================
# 10. Aggregate A10 vs A1 comparison
# ======================================================================================

A1_ACCURACY = 101 / 120
A1_VAR = 51 / 58
A1_IRR = 50 / 62

delta = {
    "accuracy_percentage_points":
        (accuracy - A1_ACCURACY) * 100,

    "VAR_percentage_points":
        (var - A1_VAR) * 100,

    "IRR_percentage_points":
        (irr - A1_IRR) * 100,
}

print("\n" + "-" * 110)
print("A10 vs A1 — AGGREGATE COMPARISON")
print("-" * 110)

print(
    "Accuracy delta : "
    f"{delta['accuracy_percentage_points']:+.2f} pp"
)

print(
    "VAR delta      : "
    f"{delta['VAR_percentage_points']:+.2f} pp"
)

print(
    "IRR delta      : "
    f"{delta['IRR_percentage_points']:+.2f} pp"
)


# ======================================================================================
# 11. Define frozen output artifacts
# ======================================================================================

METRICS = (
    METRICS_DIR
    / "A10_metrics_v1.json"
)

ANALYSIS_MANIFEST = (
    MANIFEST_DIR
    / "A10_analysis_manifest_v1.json"
)

DRIVE_METRICS = (
    DRIVE_METRICS_DIR
    / "A10_metrics_v1.json"
)

DRIVE_ANALYSIS_MANIFEST = (
    DRIVE_MANIFEST_DIR
    / "A10_analysis_manifest_v1.json"
)

for output_path in [
    METRICS,
    ANALYSIS_MANIFEST,
    DRIVE_METRICS,
    DRIVE_ANALYSIS_MANIFEST,
]:
    assert not output_path.exists(), (
        "STOP: analysis artifact already exists:\n"
        f"{output_path}\n"
        "Do not overwrite automatically."
    )


# ======================================================================================
# 12. Freeze metrics artifact
# ======================================================================================

metrics_record = {
    "arm": "A10",

    "model": "openai/gpt-oss-120b",

    "status": "EVALUABLE",

    "raw_result_sha256":
        EXPECTED["result"],

    "sample": {
        "total": 120,
        "valid": valid_n,
        "invalid": invalid_n,
    },

    "confusion_matrix_invalid_reject_positive": {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    },

    "accuracy": {
        "numerator": correct,
        "denominator": 120,
        "value": accuracy,
        "cp95_lower": acc_ci[0],
        "cp95_upper": acc_ci[1],
    },

    "VAR": {
        "numerator": tn,
        "denominator": valid_n,
        "value": var,
        "cp95_lower": var_ci[0],
        "cp95_upper": var_ci[1],
    },

    "IRR": {
        "numerator": tp,
        "denominator": invalid_n,
        "value": irr,
        "cp95_lower": irr_ci[0],
        "cp95_upper": irr_ci[1],
    },

    "rejection_precision":
        precision,

    "rejection_recall":
        recall,

    "rejection_f1":
        f1,

    "llm": {
        "invoked": llm_n,
        "parse_success": parse_success,
        "parse_failure": parse_failure,
        "exception_rows": exception_rows,
    },

    "containment": {
        "definition":
            "accepted == False AND descriptor_json present",

        "accepted":
            accepted_n,

        "rejected":
            rejected_n,

        "descriptors_emitted":
            descriptor_n,

        "violations":
            containment_violations,

        "accepted_without_descriptor":
            accepted_without_descriptor,
    },

    "rejection_stage_counts":
        stage_counts,

    "A1_aggregate_delta_percentage_points":
        delta,

    "interpretation_boundary": (
        "This is an evaluable model arm. Aggregate metric "
        "differences demonstrate model sensitivity under the frozen "
        "pipeline; they do not establish causality attributable solely "
        "to model family, architecture, or parameter count."
    ),
}

METRICS.write_text(
    json.dumps(
        metrics_record,
        indent=2
    ) + "\n",
    encoding="utf-8"
)

METRICS_SHA = sha256(METRICS)

print("\n✓ Frozen A10 metrics artifact created")


# ======================================================================================
# 13. Freeze analysis manifest
# ======================================================================================

analysis_manifest = {
    "manifest_id":
        "experiment-A-A10-analysis-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "arm":
        "A10",

    "model":
        "openai/gpt-oss-120b",

    "status":
        "EVALUABLE",

    "raw_result": {
        "sha256":
            EXPECTED["result"],
    },

    "execution_log": {
        "sha256":
            EXPECTED["log"],
    },

    "execution_manifest": {
        "sha256":
            EXPECTED["execution_manifest"],
    },

    "metrics": {
        "path":
            str(METRICS.relative_to(ROOT)),

        "sha256":
            METRICS_SHA,
    },

    "containment_violations":
        containment_violations,

    "model_or_api_call_during_analysis":
        False,

    "raw_result_modified":
        False,

    "rerun_performed":
        False,
}

ANALYSIS_MANIFEST.write_text(
    json.dumps(
        analysis_manifest,
        indent=2
    ) + "\n",
    encoding="utf-8"
)

ANALYSIS_MANIFEST_SHA = sha256(
    ANALYSIS_MANIFEST
)

print("✓ Frozen A10 analysis manifest created")


# ======================================================================================
# 14. Persist analysis artifacts to Drive
# ======================================================================================

shutil.copy2(
    METRICS,
    DRIVE_METRICS
)

shutil.copy2(
    ANALYSIS_MANIFEST,
    DRIVE_ANALYSIS_MANIFEST
)

assert sha256(DRIVE_METRICS) == METRICS_SHA

assert (
    sha256(DRIVE_ANALYSIS_MANIFEST)
    == ANALYSIS_MANIFEST_SHA
)

print("✓ Metrics persisted to Drive")
print("✓ Analysis manifest persisted to Drive")
print("✓ Drive hashes verified byte-for-byte")


# ======================================================================================
# 15. Final artifact report
# ======================================================================================

print("\n" + "-" * 110)
print("FROZEN A10 ANALYSIS ARTIFACTS")
print("-" * 110)

print("Metrics:")
print(METRICS)

print("SHA-256:")
print(METRICS_SHA)

print("\nAnalysis manifest:")
print(ANALYSIS_MANIFEST)

print("SHA-256:")
print(ANALYSIS_MANIFEST_SHA)


print("\n" + "=" * 110)
print("✓ STEP A39-FIX COMPLETE")
print("✓ A10 CLASSIFIED EVALUABLE")
print("✓ EXACT CLOPPER-PEARSON INTERVALS CALCULATED")
print("✓ CONTAINMENT CHECK COMPLETE")
print("✓ RAW A10 RESULT UNCHANGED")
print("✓ NO MODEL/API CALL DURING ANALYSIS")
print("✓ NO A10 RERUN")
print("✓ METRICS + MANIFEST PERSISTED TO DRIVE")
print("✓ SEND THIS OUTPUT BEFORE SIX-MODEL CONSOLIDATION")
print("=" * 110)

STEP A39-FIX — A10 FROZEN METRICS + EXACT CI + CONTAINMENT ANALYSIS
✓ A10 raw result verified
✓ A10 execution log verified
✓ A10 execution manifest verified
✓ Persistent Drive A10 evidence verified
✓ 120 rows / 120 unique cases verified
✓ Valid cases   : 58
✓ Invalid cases : 62

--------------------------------------------------------------------------------------------------------------
CONFUSION MATRIX
--------------------------------------------------------------------------------------------------------------
TP — invalid rejected : 51
FP — valid rejected   : 20
FN — invalid accepted : 11
TN — valid accepted   : 38
Correct decisions     : 89

--------------------------------------------------------------------------------------------------------------
PRIMARY METRICS
--------------------------------------------------------------------------------------------------------------
Accuracy : 89/120 = 0.741667 (74.17%)
CP95     : [65.38%, 81.72%]

VAR      : 38/58 = 0.655172 (65.52%)
CP9

In [15]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import pandas as pd
from scipy.stats import beta

print("=" * 118)
print("STEP A40 — SIX-EVALUABLE-MODEL EXPERIMENT-A CONSOLIDATION")
print("=" * 118)

# ======================================================================================
# 0. Paths
# ======================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

EXP_A = (
    ROOT
    / "journal_experiments"
    / "experiment_A_model_sweep"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RECOVERY_RECORD = (
    EXP_A
    / "recovery"
    / "experiment_A_colab_reset_recovery_v1.json"
)

A10_RESULT = (
    EXP_A
    / "results"
    / "raw"
    / "A10_gpt-oss-120b_results.csv"
)

A10_METRICS = (
    EXP_A
    / "results"
    / "metrics"
    / "A10_metrics_v1.json"
)

A10_ANALYSIS_MANIFEST = (
    EXP_A
    / "manifests"
    / "A10_analysis_manifest_v1.json"
)

ANALYSIS_DIR = (
    EXP_A
    / "results"
    / "analysis"
)

MANIFEST_DIR = (
    EXP_A
    / "manifests"
)

DRIVE_ANALYSIS_DIR = (
    DRIVE_EXP_A
    / "results"
    / "analysis"
)

DRIVE_MANIFEST_DIR = (
    DRIVE_EXP_A
    / "manifests"
)

for directory in [
    ANALYSIS_DIR,
    MANIFEST_DIR,
    DRIVE_ANALYSIS_DIR,
    DRIVE_MANIFEST_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def cp_interval(successes, total, alpha=0.05):

    if successes == 0:
        lower = 0.0
    else:
        lower = beta.ppf(
            alpha / 2,
            successes,
            total - successes + 1,
        )

    if successes == total:
        upper = 1.0
    else:
        upper = beta.ppf(
            1 - alpha / 2,
            successes + 1,
            total - successes,
        )

    return float(lower), float(upper)


# ======================================================================================
# 2. Verify frozen recovery + A10 evidence
# ======================================================================================

EXPECTED = {
    "recovery_record":
        "21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3",

    "A10_result":
        "998e68f0f484c929864f6a35cb6ae018ee2d22c2cf795c34b6279e6264895207",

    "A10_metrics":
        "701f313d8f53934e59bb291534ef822e3a3791382528a15087bc91e6672f13e3",

    "A10_analysis_manifest":
        "1f01fc6e707a5275789975949ee3a38435a3f405e89017ca642412c8d4acc1d2",
}

for label, path, expected_hash in [
    (
        "recovery record",
        RECOVERY_RECORD,
        EXPECTED["recovery_record"],
    ),
    (
        "A10 raw result",
        A10_RESULT,
        EXPECTED["A10_result"],
    ),
    (
        "A10 metrics",
        A10_METRICS,
        EXPECTED["A10_metrics"],
    ),
    (
        "A10 analysis manifest",
        A10_ANALYSIS_MANIFEST,
        EXPECTED["A10_analysis_manifest"],
    ),
]:

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} hash mismatch.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 3. Frozen Experiment-A evidence
#
# IMPORTANT:
#
# A1:
# Official frozen E3-v3 raw result survives byte-identically.
# The later Experiment-A reproduction raw CSV itself was lost in reset,
# although that reproduction had exactly matched the official baseline.
#
# A2/A7/A8/A9:
# Original transient raw files were lost in reset.
# Metrics below are historical records frozen BEFORE reset.
#
# A10:
# Raw result + metrics survive and are Drive-persisted.
# ======================================================================================

arms = [
    {
        "arm": "A1",
        "model": "Qwen/Qwen3.5-9B",

        "correct": 101,
        "valid_accepted": 51,
        "invalid_rejected": 50,

        "TP": 50,
        "FP": 7,
        "FN": 12,
        "TN": 51,

        "accepted": 63,
        "descriptors": 63,
        "containment_violations": 0,

        "evidence_status":
            "OFFICIAL_FROZEN_RAW_RESULT_AVAILABLE",

        "raw_sha256":
            "12c3a82bd8cde2d07de352155e9ee14913c821fb745d401c7ffe2f207e850e5e",

        "experiment_A_reproduction_raw_sha256":
            "c22618b2750de7f5ef1824fa4036a901ef54f611ad162fcd797b20d79e2b4e06",
    },

    {
        "arm": "A2",
        "model":
            "meta-llama/Llama-3.3-70B-Instruct-Turbo",

        "correct": 100,
        "valid_accepted": 48,
        "invalid_rejected": 52,

        "TP": 52,
        "FP": 10,
        "FN": 10,
        "TN": 48,

        "accepted": 58,
        "descriptors": 58,
        "containment_violations": 0,

        "evidence_status":
            "HISTORICAL_METRICS_RECORDED_BEFORE_RESET_RAW_LOST",

        "raw_sha256":
            "5eba3be64426c14ac1a917be5392e4b6a0a487f3bf9be81c076d7f4b429380d7",
    },

    {
        "arm": "A7",
        "model":
            "arize-ai/qwen-2-1.5b-instruct",

        "correct": 72,
        "valid_accepted": 11,
        "invalid_rejected": 61,

        "TP": 61,
        "FP": 47,
        "FN": 1,
        "TN": 11,

        "accepted": 12,
        "descriptors": 12,
        "containment_violations": 0,

        "evidence_status":
            "HISTORICAL_METRICS_RECORDED_BEFORE_RESET_RAW_LOST",

        "raw_sha256":
            "2c41287e4cab5680e5031fb9252b003d7bdf5412a76b42a8856c9ce700a70e7c",
    },

    {
        "arm": "A8",
        "model":
            "Prism-ML/Ternary-Bonsai-27B",

        "correct": 91,
        "valid_accepted": 41,
        "invalid_rejected": 50,

        "TP": 50,
        "FP": 17,
        "FN": 12,
        "TN": 41,

        "accepted": 53,
        "descriptors": 53,
        "containment_violations": 0,

        "evidence_status":
            "HISTORICAL_METRICS_RECORDED_BEFORE_RESET_RAW_LOST",

        "raw_sha256":
            "1507e111876a27801fbd03998cfa8f8390b663be954479386f01bf2ae242b082",
    },

    {
        "arm": "A9",
        "model":
            "deepseek-ai/DeepSeek-V4.1-Flash",

        "correct": 101,
        "valid_accepted": 51,
        "invalid_rejected": 50,

        "TP": 50,
        "FP": 7,
        "FN": 12,
        "TN": 51,

        "accepted": 63,
        "descriptors": 63,
        "containment_violations": 0,

        "evidence_status":
            "HISTORICAL_METRICS_RECORDED_BEFORE_RESET_RAW_LOST",

        "raw_sha256":
            "b4b02483db7021c43e2e406d0ea4811f7f22f72258957613bbfcd96b11ec12aa",
    },

    {
        "arm": "A10",
        "model":
            "openai/gpt-oss-120b",

        "correct": 89,
        "valid_accepted": 38,
        "invalid_rejected": 51,

        "TP": 51,
        "FP": 20,
        "FN": 11,
        "TN": 38,

        "accepted": 49,
        "descriptors": 49,
        "containment_violations": 0,

        "evidence_status":
            "RAW_RESULT_AVAILABLE_AND_DRIVE_PERSISTED",

        "raw_sha256":
            EXPECTED["A10_result"],
    },
]


# ======================================================================================
# 4. Internal consistency checks
# ======================================================================================

for x in arms:

    assert (
        x["TP"]
        + x["FP"]
        + x["FN"]
        + x["TN"]
        == 120
    )

    assert (
        x["TN"]
        == x["valid_accepted"]
    )

    assert (
        x["TP"]
        == x["invalid_rejected"]
    )

    assert (
        x["correct"]
        == x["TP"] + x["TN"]
    )

    assert (
        x["accepted"]
        == x["TN"] + x["FN"]
    )

    assert (
        x["descriptors"]
        == x["accepted"]
    )

    assert (
        x["containment_violations"]
        == 0
    )

print("✓ Six evaluable arms internally consistent")


# ======================================================================================
# 5. Calculate all metrics from frozen counts
# ======================================================================================

rows = []

for x in arms:

    accuracy = x["correct"] / 120
    var = x["valid_accepted"] / 58
    irr = x["invalid_rejected"] / 62

    precision = (
        x["TP"]
        / (x["TP"] + x["FP"])
    )

    recall = (
        x["TP"]
        / (x["TP"] + x["FN"])
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
    )

    acc_ci = cp_interval(
        x["correct"],
        120,
    )

    var_ci = cp_interval(
        x["valid_accepted"],
        58,
    )

    irr_ci = cp_interval(
        x["invalid_rejected"],
        62,
    )

    rows.append({
        "arm":
            x["arm"],

        "model":
            x["model"],

        "accuracy_n":
            x["correct"],

        "accuracy_pct":
            accuracy * 100,

        "accuracy_cp95_low_pct":
            acc_ci[0] * 100,

        "accuracy_cp95_high_pct":
            acc_ci[1] * 100,

        "VAR_n":
            x["valid_accepted"],

        "VAR_pct":
            var * 100,

        "VAR_cp95_low_pct":
            var_ci[0] * 100,

        "VAR_cp95_high_pct":
            var_ci[1] * 100,

        "IRR_n":
            x["invalid_rejected"],

        "IRR_pct":
            irr * 100,

        "IRR_cp95_low_pct":
            irr_ci[0] * 100,

        "IRR_cp95_high_pct":
            irr_ci[1] * 100,

        "rejection_precision":
            precision,

        "rejection_recall":
            recall,

        "rejection_f1":
            f1,

        "accepted":
            x["accepted"],

        "descriptors":
            x["descriptors"],

        "containment_violations":
            x["containment_violations"],

        "evidence_status":
            x["evidence_status"],

        "raw_sha256":
            x["raw_sha256"],
    })


summary = pd.DataFrame(rows)

assert len(summary) == 6

assert (
    summary["containment_violations"]
    .sum()
    == 0
)

print("✓ Metrics recalculated from frozen counts")
print("✓ Exact Clopper-Pearson intervals calculated")


# ======================================================================================
# 6. Display publication-oriented summary
# ======================================================================================

print("\n" + "-" * 118)
print("SIX EVALUABLE MODEL ARMS")
print("-" * 118)

for _, r in summary.iterrows():

    print(
        f"{r['arm']:<4} "
        f"{r['model']:<45} "
        f"ACC={r['accuracy_pct']:6.2f}%  "
        f"VAR={r['VAR_pct']:6.2f}%  "
        f"IRR={r['IRR_pct']:6.2f}%  "
        f"CV={int(r['containment_violations'])}"
    )


# ======================================================================================
# 7. Range analysis — descriptive, not ranking
# ======================================================================================

acc_min = float(
    summary["accuracy_pct"].min()
)

acc_max = float(
    summary["accuracy_pct"].max()
)

var_min = float(
    summary["VAR_pct"].min()
)

var_max = float(
    summary["VAR_pct"].max()
)

irr_min = float(
    summary["IRR_pct"].min()
)

irr_max = float(
    summary["IRR_pct"].max()
)

print("\n" + "-" * 118)
print("DESCRIPTIVE CROSS-MODEL RANGE")
print("-" * 118)

print(
    f"Accuracy range : "
    f"{acc_min:.2f}% to {acc_max:.2f}% "
    f"(spread {acc_max - acc_min:.2f} pp)"
)

print(
    f"VAR range      : "
    f"{var_min:.2f}% to {var_max:.2f}% "
    f"(spread {var_max - var_min:.2f} pp)"
)

print(
    f"IRR range      : "
    f"{irr_min:.2f}% to {irr_max:.2f}% "
    f"(spread {irr_max - irr_min:.2f} pp)"
)

print(
    "Containment   : "
    "0 observed violations in every evaluable arm"
)


# ======================================================================================
# 8. Original A1-A6 preregistration accounting
# ======================================================================================

original_accounting = [
    {
        "arm": "A1",
        "status": "EVALUABLE",
    },
    {
        "arm": "A2",
        "status": "EVALUABLE",
    },
    {
        "arm": "A3",
        "status":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
    },
    {
        "arm": "A4",
        "status":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
    },
    {
        "arm": "A5",
        "status":
            "UNAVAILABLE_ON_ACCESSIBLE_SERVERLESS_ENDPOINT",
    },
    {
        "arm": "A6",
        "status":
            "UNAVAILABLE_PREBENCHMARK_PROBE_ONLY",
    },
]

print("\n" + "-" * 118)
print("ORIGINAL A1–A6 ACCOUNTING")
print("-" * 118)

for item in original_accounting:
    print(
        f"{item['arm']:<4} "
        f"{item['status']}"
    )


# ======================================================================================
# 9. Extension accounting
# ======================================================================================

print("\n" + "-" * 118)
print("PROSPECTIVE A7–A10 EXTENSION ACCOUNTING")
print("-" * 118)

for arm in ["A7", "A8", "A9", "A10"]:

    model = summary.loc[
        summary["arm"] == arm,
        "model"
    ].iloc[0]

    print(
        f"{arm:<4} {model:<45} EVALUABLE"
    )


# ======================================================================================
# 10. Freeze CSV
# ======================================================================================

CSV_OUT = (
    ANALYSIS_DIR
    / "experiment_A_six_evaluable_model_summary_v1.csv"
)

JSON_OUT = (
    ANALYSIS_DIR
    / "experiment_A_six_evaluable_model_summary_v1.json"
)

MANIFEST_OUT = (
    MANIFEST_DIR
    / "experiment_A_six_evaluable_model_summary_manifest_v1.json"
)

DRIVE_CSV = (
    DRIVE_ANALYSIS_DIR
    / CSV_OUT.name
)

DRIVE_JSON = (
    DRIVE_ANALYSIS_DIR
    / JSON_OUT.name
)

DRIVE_MANIFEST = (
    DRIVE_MANIFEST_DIR
    / MANIFEST_OUT.name
)

for path in [
    CSV_OUT,
    JSON_OUT,
    MANIFEST_OUT,
    DRIVE_CSV,
    DRIVE_JSON,
    DRIVE_MANIFEST,
]:

    assert not path.exists(), (
        "STOP: consolidation artifact already exists:\n"
        f"{path}"
    )


summary.to_csv(
    CSV_OUT,
    index=False,
)


# ======================================================================================
# 11. Freeze JSON synthesis
# ======================================================================================

synthesis = {
    "experiment":
        "Experiment A — semantic parser model variation",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "frozen_benchmark": {
        "total_cases": 120,
        "valid_cases": 58,
        "invalid_cases": 62,
    },

    "evaluable_arms": [
        "A1",
        "A2",
        "A7",
        "A8",
        "A9",
        "A10",
    ],

    "evaluable_model_count":
        6,

    "original_preregistered_arms": {
        "arms":
            ["A1", "A2", "A3", "A4", "A5", "A6"],

        "evaluable":
            ["A1", "A2"],

        "endpoint_unavailable":
            ["A3", "A4", "A5", "A6"],
    },

    "prospective_extension": {
        "arms":
            ["A7", "A8", "A9", "A10"],

        "selection_basis":
            (
                "Non-benchmark endpoint-accessibility screening "
                "plus predefined model-diversity criteria."
            ),

        "replacement_for_A3_A6":
            False,

        "all_four_evaluable":
            True,
    },

    "descriptive_ranges": {
        "accuracy_pct": {
            "min": acc_min,
            "max": acc_max,
            "spread_percentage_points":
                acc_max - acc_min,
        },

        "VAR_pct": {
            "min": var_min,
            "max": var_max,
            "spread_percentage_points":
                var_max - var_min,
        },

        "IRR_pct": {
            "min": irr_min,
            "max": irr_max,
            "spread_percentage_points":
                irr_max - irr_min,
        },
    },

    "containment": {
        "operational_definition":
            (
                "A containment violation is a rejected case "
                "(accepted == False) for which descriptor_json "
                "is nevertheless present."
            ),

        "observed_violations_by_arm": {
            x["arm"]:
                x["containment_violations"]
            for x in arms
        },

        "total_observed_violations":
            int(
                summary[
                    "containment_violations"
                ].sum()
            ),
    },

    "evidence_recovery_disclosure": {
        "A1":
            (
                "Official frozen E3-v3 raw result remains "
                "available byte-identically. The later "
                "Experiment-A reproduction raw file was lost "
                "after the Colab reset; before reset it had "
                "reproduced the official aggregate metrics exactly."
            ),

        "A2_A7_A8_A9":
            (
                "Original transient raw result files were lost "
                "after the Colab runtime reset. The consolidation "
                "uses outcome counts, metrics, and raw-file SHA-256 "
                "values recorded before the reset. These are not "
                "represented as recovered raw files."
            ),

        "A10":
            (
                "First-and-only benchmark execution performed "
                "after recovery provenance was frozen. Raw result, "
                "execution log, metrics, and manifests were "
                "immediately persisted to Drive."
            ),
    },

    "interpretation": {
        "supported":
            (
                "Across the six evaluable semantic-parser arms "
                "tested under the frozen 120-case benchmark and "
                "deterministic cascade, decision metrics varied "
                "substantially across models, while zero observed "
                "containment violations occurred in every evaluable "
                "arm."
            ),

        "not_supported": [
            "universal model independence",
            "hallucination-free operation",
            "zero-risk guarantee",
            "formal proof across arbitrary models",
            "causal attribution of performance differences solely to model architecture or size",
            "case-level identity between A1 and A9 based only on equal aggregate metrics",
        ],
    },

    "model_or_api_call":
        False,

    "benchmark_rerun":
        False,
}

JSON_OUT.write_text(
    json.dumps(
        synthesis,
        indent=2
    ) + "\n",
    encoding="utf-8",
)


# ======================================================================================
# 12. Hash summary artifacts
# ======================================================================================

CSV_SHA = sha256(CSV_OUT)
JSON_SHA = sha256(JSON_OUT)

manifest = {
    "manifest_id":
        "experiment-A-six-evaluable-model-summary-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "csv": {
        "path":
            str(CSV_OUT.relative_to(ROOT)),

        "sha256":
            CSV_SHA,
    },

    "json": {
        "path":
            str(JSON_OUT.relative_to(ROOT)),

        "sha256":
            JSON_SHA,
    },

    "source_evidence": {
        "recovery_record_sha256":
            EXPECTED["recovery_record"],

        "A10_raw_result_sha256":
            EXPECTED["A10_result"],

        "A10_metrics_sha256":
            EXPECTED["A10_metrics"],

        "A10_analysis_manifest_sha256":
            EXPECTED["A10_analysis_manifest"],
    },

    "evaluable_model_count":
        6,

    "total_containment_violations":
        0,

    "model_or_api_call":
        False,

    "rerun_performed":
        False,
}

MANIFEST_OUT.write_text(
    json.dumps(
        manifest,
        indent=2
    ) + "\n",
    encoding="utf-8",
)

MANIFEST_SHA = sha256(
    MANIFEST_OUT
)


# ======================================================================================
# 13. Persist all consolidation artifacts to Drive
# ======================================================================================

shutil.copy2(
    CSV_OUT,
    DRIVE_CSV
)

shutil.copy2(
    JSON_OUT,
    DRIVE_JSON
)

shutil.copy2(
    MANIFEST_OUT,
    DRIVE_MANIFEST
)

assert sha256(DRIVE_CSV) == CSV_SHA
assert sha256(DRIVE_JSON) == JSON_SHA
assert sha256(DRIVE_MANIFEST) == MANIFEST_SHA

print("\n✓ Consolidation artifacts persisted to Drive")
print("✓ Drive hashes verified byte-for-byte")


# ======================================================================================
# 14. Final report
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN CONSOLIDATION ARTIFACTS")
print("-" * 118)

print("CSV:")
print(CSV_OUT)
print("SHA-256:")
print(CSV_SHA)

print("\nJSON:")
print(JSON_OUT)
print("SHA-256:")
print(JSON_SHA)

print("\nManifest:")
print(MANIFEST_OUT)
print("SHA-256:")
print(MANIFEST_SHA)


print("\n" + "=" * 118)
print("✓ STEP A40 COMPLETE")
print("✓ SIX EVALUABLE MODEL ARMS CONSOLIDATED")
print("✓ A1/A2/A7/A8/A9/A10 ACCOUNTED FOR")
print("✓ A3–A6 UNAVAILABLE HISTORY RETAINED")
print("✓ ORIGINAL PREREGISTRATION AND EXTENSION KEPT DISTINCT")
print("✓ ZERO OBSERVED CONTAINMENT VIOLATIONS ACROSS SIX EVALUABLE ARMS")
print("✓ LOST RAW EVIDENCE DISCLOSED — NOT RECONSTRUCTED")
print("✓ NO MODEL/API CALL")
print("✓ NO BENCHMARK RERUN")
print("✓ CONSOLIDATION PERSISTED TO DRIVE")
print("✓ SEND THIS OUTPUT BEFORE EXPERIMENT-A FINAL FREEZE")
print("=" * 118)

STEP A40 — SIX-EVALUABLE-MODEL EXPERIMENT-A CONSOLIDATION
✓ recovery record verified
✓ A10 raw result verified
✓ A10 metrics verified
✓ A10 analysis manifest verified
✓ Six evaluable arms internally consistent
✓ Metrics recalculated from frozen counts
✓ Exact Clopper-Pearson intervals calculated

----------------------------------------------------------------------------------------------------------------------
SIX EVALUABLE MODEL ARMS
----------------------------------------------------------------------------------------------------------------------
A1   Qwen/Qwen3.5-9B                               ACC= 84.17%  VAR= 87.93%  IRR= 80.65%  CV=0
A2   meta-llama/Llama-3.3-70B-Instruct-Turbo       ACC= 83.33%  VAR= 82.76%  IRR= 83.87%  CV=0
A7   arize-ai/qwen-2-1.5b-instruct                 ACC= 60.00%  VAR= 18.97%  IRR= 98.39%  CV=0
A8   Prism-ML/Ternary-Bonsai-27B                   ACC= 75.83%  VAR= 70.69%  IRR= 80.65%  CV=0
A9   deepseek-ai/DeepSeek-V4.1-Flash               ACC= 84.

In [16]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil

print("=" * 118)
print("STEP A41 — EXPERIMENT-A FINAL FREEZE")
print("=" * 118)

# ======================================================================================
# 0. Paths
# ======================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

EXP_A = (
    ROOT
    / "journal_experiments"
    / "experiment_A_model_sweep"
)

DRIVE_EXP_A = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_A_model_sweep"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

DATASET = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

RECOVERY_RECORD = (
    EXP_A
    / "recovery"
    / "experiment_A_colab_reset_recovery_v1.json"
)

RECOVERED_PREREG = (
    EXP_A
    / "preregistration"
    / "experiment_A_extension_A7_A10_preregistration_RECOVERED_v1.json"
)

A10_RESULT = (
    EXP_A
    / "results"
    / "raw"
    / "A10_gpt-oss-120b_results.csv"
)

A10_METRICS = (
    EXP_A
    / "results"
    / "metrics"
    / "A10_metrics_v1.json"
)

SUMMARY_CSV = (
    EXP_A
    / "results"
    / "analysis"
    / "experiment_A_six_evaluable_model_summary_v1.csv"
)

SUMMARY_JSON = (
    EXP_A
    / "results"
    / "analysis"
    / "experiment_A_six_evaluable_model_summary_v1.json"
)

SUMMARY_MANIFEST = (
    EXP_A
    / "manifests"
    / "experiment_A_six_evaluable_model_summary_manifest_v1.json"
)

MANIFEST_DIR = EXP_A / "manifests"
DRIVE_MANIFEST_DIR = DRIVE_EXP_A / "manifests"

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================================
# 1. Hash helper
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. Verify complete frozen evidence chain
# ======================================================================================

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "dataset":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "recovery_record":
        "21947d0a7b84c249af08ec62f72b54cf2ea7614108eac26cfab1f0acaad22ac3",

    "recovered_prereg":
        "9cd89b9f2937f686405f954628f77f7f44cc08ac79ea9499d3d98d8c1d8dd980",

    "A10_result":
        "998e68f0f484c929864f6a35cb6ae018ee2d22c2cf795c34b6279e6264895207",

    "A10_metrics":
        "701f313d8f53934e59bb291534ef822e3a3791382528a15087bc91e6672f13e3",

    "summary_csv":
        "36380008c7bd7ed1158666d4f0c8e9d31708863164b4f2551fea7f5238d2c880",

    "summary_json":
        "c055cad3071ad923978d7a180f5c811974a726104360ae7e715766083477e448",

    "summary_manifest":
        "3e30e74ea2558819c42d9c2b9ef4dfc7bd9187404b7779884aa5b6c67ad8c01a",
}

checks = [
    ("frozen runner", RUNNER, EXPECTED["runner"]),
    ("frozen benchmark", DATASET, EXPECTED["dataset"]),
    (
        "reset recovery record",
        RECOVERY_RECORD,
        EXPECTED["recovery_record"],
    ),
    (
        "recovered extension preregistration",
        RECOVERED_PREREG,
        EXPECTED["recovered_prereg"],
    ),
    (
        "A10 raw result",
        A10_RESULT,
        EXPECTED["A10_result"],
    ),
    (
        "A10 frozen metrics",
        A10_METRICS,
        EXPECTED["A10_metrics"],
    ),
    (
        "six-model summary CSV",
        SUMMARY_CSV,
        EXPECTED["summary_csv"],
    ),
    (
        "six-model summary JSON",
        SUMMARY_JSON,
        EXPECTED["summary_json"],
    ),
    (
        "six-model summary manifest",
        SUMMARY_MANIFEST,
        EXPECTED["summary_manifest"],
    ),
]

for label, path, expected_hash in checks:

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual_hash = sha256(path)

    assert actual_hash == expected_hash, (
        f"STOP: {label} hash mismatch.\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual_hash}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 3. Verify six-model synthesis contents
# ======================================================================================

with open(
    SUMMARY_JSON,
    "r",
    encoding="utf-8"
) as f:
    synthesis = json.load(f)

assert synthesis["evaluable_model_count"] == 6

assert synthesis["evaluable_arms"] == [
    "A1",
    "A2",
    "A7",
    "A8",
    "A9",
    "A10",
]

assert (
    synthesis["containment"]
    ["total_observed_violations"]
    == 0
)

assert (
    synthesis["prospective_extension"]
    ["replacement_for_A3_A6"]
    is False
)

assert (
    synthesis["prospective_extension"]
    ["all_four_evaluable"]
    is True
)

print("✓ Six-model synthesis structure verified")
print("✓ Total observed containment violations = 0")


# ======================================================================================
# 4. Frozen final numerical result
# ======================================================================================

final_results = {
    "A1": {
        "model": "Qwen/Qwen3.5-9B",
        "accuracy_pct": 84.17,
        "VAR_pct": 87.93,
        "IRR_pct": 80.65,
        "containment_violations": 0,
    },

    "A2": {
        "model":
            "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "accuracy_pct": 83.33,
        "VAR_pct": 82.76,
        "IRR_pct": 83.87,
        "containment_violations": 0,
    },

    "A7": {
        "model":
            "arize-ai/qwen-2-1.5b-instruct",
        "accuracy_pct": 60.00,
        "VAR_pct": 18.97,
        "IRR_pct": 98.39,
        "containment_violations": 0,
    },

    "A8": {
        "model":
            "Prism-ML/Ternary-Bonsai-27B",
        "accuracy_pct": 75.83,
        "VAR_pct": 70.69,
        "IRR_pct": 80.65,
        "containment_violations": 0,
    },

    "A9": {
        "model":
            "deepseek-ai/DeepSeek-V4.1-Flash",
        "accuracy_pct": 84.17,
        "VAR_pct": 87.93,
        "IRR_pct": 80.65,
        "containment_violations": 0,
    },

    "A10": {
        "model":
            "openai/gpt-oss-120b",
        "accuracy_pct": 74.17,
        "VAR_pct": 65.52,
        "IRR_pct": 82.26,
        "containment_violations": 0,
    },
}


# ======================================================================================
# 5. Final experiment status
# ======================================================================================

final_freeze = {
    "freeze_id":
        "experiment-A-final-freeze-v1",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "experiment":
        (
            "Experiment A — semantic-parser model variation "
            "under a frozen deterministic E3-v3 cascade"
        ),

    "status":
        "COMPLETE_AND_FROZEN",

    "research_question":
        (
            "When the deterministic pipeline and benchmark are held "
            "fixed, how much does semantic-parser model choice affect "
            "decision performance, and does the deterministic "
            "containment boundary remain intact across tested models?"
        ),

    "frozen_inputs": {
        "runner_sha256":
            EXPECTED["runner"],

        "benchmark_sha256":
            EXPECTED["dataset"],

        "benchmark_cases":
            120,

        "valid_cases":
            58,

        "invalid_cases":
            62,
    },

    "original_preregistration": {
        "arms":
            ["A1", "A2", "A3", "A4", "A5", "A6"],

        "evaluable":
            ["A1", "A2"],

        "endpoint_unavailable":
            ["A3", "A4", "A5", "A6"],

        "substitutions":
            0,

        "removals":
            0,
    },

    "prospective_extension": {
        "arms":
            ["A7", "A8", "A9", "A10"],

        "evaluable":
            ["A7", "A8", "A9", "A10"],

        "replacement_for_original_unavailable_arms":
            False,

        "selection_used_heldout_performance":
            False,
    },

    "all_evaluable_arms": [
        "A1",
        "A2",
        "A7",
        "A8",
        "A9",
        "A10",
    ],

    "evaluable_model_count":
        6,

    "final_results":
        final_results,

    "descriptive_cross_model_result": {
        "accuracy_range_pct":
            [60.00, 84.17],

        "accuracy_spread_percentage_points":
            24.17,

        "VAR_range_pct":
            [18.97, 87.93],

        "VAR_spread_percentage_points":
            68.97,

        "IRR_range_pct":
            [80.65, 98.39],

        "IRR_spread_percentage_points":
            17.74,

        "containment_violations_each_arm":
            0,

        "total_observed_containment_violations":
            0,
    },

    "containment_definition":
        (
            "A containment violation occurs when a rejected case "
            "(accepted == False) nevertheless contains descriptor_json."
        ),

    "primary_empirical_observation":
        (
            "Decision performance varied substantially across the "
            "six evaluable semantic-parser arms, while zero observed "
            "containment violations occurred in every tested "
            "evaluable arm under the frozen deterministic cascade."
        ),

    "publication_claim_boundary": {
        "supported_wording":
            (
                "Across six evaluable semantic-parser models tested "
                "on the frozen 120-case benchmark, decision accuracy "
                "ranged from 60.00% to 84.17%, while the deterministic "
                "failure-containment mechanism produced zero observed "
                "containment violations in every evaluable arm."
            ),

        "supported_interpretation":
            (
                "The experiment provides empirical evidence that "
                "semantic-model-dependent decision performance can "
                "vary substantially while the tested deterministic "
                "downstream containment boundary remains stable."
            ),

        "must_not_claim": [
            "universal model independence",
            "hallucination-free operation",
            "formal guarantee for arbitrary models",
            "zero-risk operation",
            "formal 3GPP conformance from this experiment",
            "formal O-RAN conformance from this experiment",
            "causal attribution of metric differences solely to model size",
            "causal attribution of metric differences solely to model architecture",
            "case-level identity between A1 and A9",
        ],
    },

    "recovery_disclosure": {
        "colab_reset_occurred":
            True,

        "A1":
            (
                "The official frozen E3-v3 baseline raw result remains "
                "available byte-identically. The later Experiment-A "
                "reproduction raw CSV was lost in the reset; its "
                "pre-reset result had reproduced the official "
                "aggregate metrics exactly."
            ),

        "A2_A7_A8_A9":
            (
                "Transient raw CSVs were lost in the Colab runtime "
                "reset. Their outcome counts, aggregate metrics, "
                "containment results, and raw-file SHA-256 values had "
                "been recorded before the reset. Lost raw files were "
                "not reconstructed or represented as recovered."
            ),

        "A10":
            (
                "A10 had never been executed before the reset. "
                "After recovery provenance was frozen, A10 was "
                "executed exactly once and its raw evidence was "
                "immediately persisted to Drive."
            ),
    },

    "historical_original_extension_preregistration_hashes": {
        "json_sha256":
            "5568bf925a7b1de1e561820673bb15d89e818d08f2c2662f4c577fb9aa626f90",

        "markdown_sha256":
            "2708091a86ab9f3432fceea31a4ddf26becd635d24c3605c46bec68a67218e54",

        "manifest_sha256":
            "7867181975990e1c46359dedda988e129b09a9aa1c35165e13f1c6cb5f8c3b8a",
    },

    "consolidation_artifacts": {
        "csv_sha256":
            EXPECTED["summary_csv"],

        "json_sha256":
            EXPECTED["summary_json"],

        "manifest_sha256":
            EXPECTED["summary_manifest"],
    },

    "experiment_closed":
        True,

    "further_model_runs_part_of_experiment_A":
        False,

    "further_parameter_tuning_part_of_experiment_A":
        False,

    "further_threshold_tuning_part_of_experiment_A":
        False,

    "model_or_api_call_during_final_freeze":
        False,

    "benchmark_rerun_during_final_freeze":
        False,
}


# ======================================================================================
# 6. Create final freeze artifact
# ======================================================================================

FINAL_FREEZE = (
    MANIFEST_DIR
    / "experiment_A_FINAL_FREEZE_v1.json"
)

DRIVE_FINAL_FREEZE = (
    DRIVE_MANIFEST_DIR
    / "experiment_A_FINAL_FREEZE_v1.json"
)

assert not FINAL_FREEZE.exists(), (
    f"STOP: final freeze already exists:\n{FINAL_FREEZE}"
)

assert not DRIVE_FINAL_FREEZE.exists(), (
    f"STOP: Drive final freeze already exists:\n{DRIVE_FINAL_FREEZE}"
)

FINAL_FREEZE.write_text(
    json.dumps(
        final_freeze,
        indent=2
    ) + "\n",
    encoding="utf-8",
)

FINAL_FREEZE_SHA = sha256(
    FINAL_FREEZE
)

print("✓ Final Experiment-A freeze artifact created")


# ======================================================================================
# 7. Persist final freeze immediately
# ======================================================================================

shutil.copy2(
    FINAL_FREEZE,
    DRIVE_FINAL_FREEZE
)

assert (
    sha256(DRIVE_FINAL_FREEZE)
    == FINAL_FREEZE_SHA
)

print("✓ Final freeze persisted to Drive")
print("✓ Drive copy verified byte-for-byte")


# ======================================================================================
# 8. Verify final freeze contents
# ======================================================================================

with open(
    FINAL_FREEZE,
    "r",
    encoding="utf-8"
) as f:
    verify = json.load(f)

assert verify["status"] == "COMPLETE_AND_FROZEN"
assert verify["evaluable_model_count"] == 6
assert verify["experiment_closed"] is True

assert (
    verify["descriptive_cross_model_result"]
    ["total_observed_containment_violations"]
    == 0
)

assert (
    verify["further_model_runs_part_of_experiment_A"]
    is False
)

assert (
    verify["model_or_api_call_during_final_freeze"]
    is False
)

print("✓ Final freeze contents verified")


# ======================================================================================
# 9. Print final publication result
# ======================================================================================

print("\n" + "-" * 118)
print("FINAL EXPERIMENT-A RESULT")
print("-" * 118)

for arm, r in final_results.items():

    print(
        f"{arm:<4} "
        f"{r['model']:<45} "
        f"ACC={r['accuracy_pct']:6.2f}%  "
        f"VAR={r['VAR_pct']:6.2f}%  "
        f"IRR={r['IRR_pct']:6.2f}%  "
        f"CV={r['containment_violations']}"
    )

print("\nAccuracy range : 60.00%–84.17%")
print("VAR range      : 18.97%–87.93%")
print("IRR range      : 80.65%–98.39%")
print("Containment    : 0 observed violations in all six evaluable arms")


# ======================================================================================
# 10. Print exact paper-safe claim
# ======================================================================================

print("\n" + "-" * 118)
print("PAPER-SAFE CLAIM")
print("-" * 118)

print(
    verify["publication_claim_boundary"]
    ["supported_wording"]
)

print("\nInterpretation:")
print(
    verify["publication_claim_boundary"]
    ["supported_interpretation"]
)


# ======================================================================================
# 11. Artifact report
# ======================================================================================

print("\n" + "-" * 118)
print("FINAL FREEZE ARTIFACT")
print("-" * 118)

print(FINAL_FREEZE)

print("SHA-256:")
print(FINAL_FREEZE_SHA)

print("\nPersistent Drive copy:")
print(DRIVE_FINAL_FREEZE)

print("SHA-256:")
print(sha256(DRIVE_FINAL_FREEZE))


# ======================================================================================
# 12. Final
# ======================================================================================

print("\n" + "=" * 118)
print("✓ STEP A41 COMPLETE")
print("✓ EXPERIMENT A COMPLETE AND FROZEN")
print("✓ SIX EVALUABLE MODELS RETAINED")
print("✓ ORIGINAL A1–A6 ACCOUNTING RETAINED")
print("✓ A7–A10 PROSPECTIVE EXTENSION RETAINED")
print("✓ ZERO OBSERVED CONTAINMENT VIOLATIONS ACROSS SIX EVALUABLE ARMS")
print("✓ RESET / LOST-RAW DISCLOSURE RETAINED")
print("✓ NO MODEL/API CALL")
print("✓ NO BENCHMARK RERUN")
print("✓ FINAL FREEZE PERSISTED TO DRIVE")
print("✓ DO NOT ADD OR RERUN MODELS IN EXPERIMENT A")
print("✓ SEND THIS OUTPUT BEFORE PROCEEDING TO EXPERIMENT B")
print("=" * 118)

STEP A41 — EXPERIMENT-A FINAL FREEZE
✓ frozen runner verified
✓ frozen benchmark verified
✓ reset recovery record verified
✓ recovered extension preregistration verified
✓ A10 raw result verified
✓ A10 frozen metrics verified
✓ six-model summary CSV verified
✓ six-model summary JSON verified
✓ six-model summary manifest verified
✓ Six-model synthesis structure verified
✓ Total observed containment violations = 0
✓ Final Experiment-A freeze artifact created
✓ Final freeze persisted to Drive
✓ Drive copy verified byte-for-byte
✓ Final freeze contents verified

----------------------------------------------------------------------------------------------------------------------
FINAL EXPERIMENT-A RESULT
----------------------------------------------------------------------------------------------------------------------
A1   Qwen/Qwen3.5-9B                               ACC= 84.17%  VAR= 87.93%  IRR= 80.65%  CV=0
A2   meta-llama/Llama-3.3-70B-Instruct-Turbo       ACC= 83.33%  VAR= 82.76% 

In [17]:
from pathlib import Path
import hashlib
import ast

print("=" * 118)
print("EXPERIMENT B — STEP B1: INSPECT FROZEN LLM CALL / JSON PARSING PATH")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ------------------------------------------------------------------
# 1. Verify frozen runner
# ------------------------------------------------------------------

assert RUNNER.exists(), (
    f"STOP: runner missing:\n{RUNNER}"
)

actual_sha = sha256(RUNNER)

assert actual_sha == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner hash mismatch.\n"
    f"Expected: {EXPECTED_RUNNER_SHA}\n"
    f"Actual:   {actual_sha}"
)

print("✓ Frozen runner verified")
print("SHA-256:", actual_sha)


# ------------------------------------------------------------------
# 2. Parse Python source without executing it
# ------------------------------------------------------------------

source = RUNNER.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

print("✓ Runner parsed as Python AST")
print("✓ Runner NOT executed")


# ------------------------------------------------------------------
# 3. Print function/class inventory
# ------------------------------------------------------------------

print("\n" + "-" * 118)
print("FUNCTION / CLASS INVENTORY")
print("-" * 118)

for node in ast.walk(tree):

    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        print(
            f"FUNCTION  "
            f"{node.name:<45} "
            f"lines {node.lineno}-{node.end_lineno}"
        )

    elif isinstance(node, ast.ClassDef):
        print(
            f"CLASS     "
            f"{node.name:<45} "
            f"lines {node.lineno}-{node.end_lineno}"
        )


# ------------------------------------------------------------------
# 4. Find likely LLM/API/JSON parsing locations
# ------------------------------------------------------------------

keywords = [
    "Together",
    "chat.completions",
    "completions.create",
    "response_format",
    "json.loads",
    "json.dumps",
    "parse",
    "prompt",
    "messages",
    "model=",
    "temperature",
    "max_tokens",
]

lines = source.splitlines()

hits = []

for lineno, line in enumerate(
    lines,
    start=1
):

    if any(
        keyword.lower() in line.lower()
        for keyword in keywords
    ):
        hits.append(
            (lineno, line)
        )


print("\n" + "-" * 118)
print("KEYWORD HITS")
print("-" * 118)

for lineno, line in hits:
    print(
        f"{lineno:5d}: {line}"
    )


# ------------------------------------------------------------------
# 5. Identify API-call AST nodes
# ------------------------------------------------------------------

print("\n" + "-" * 118)
print("POTENTIAL CALL SITES")
print("-" * 118)

interesting_calls = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    try:
        call_text = ast.unparse(
            node.func
        )
    except Exception:
        continue

    lower = call_text.lower()

    if any(
        token in lower
        for token in [
            "create",
            "completion",
            "together",
            "json.loads",
        ]
    ):

        interesting_calls.append({
            "line":
                node.lineno,

            "end_line":
                getattr(
                    node,
                    "end_lineno",
                    node.lineno
                ),

            "call":
                call_text,
        })


for item in sorted(
    interesting_calls,
    key=lambda x: x["line"]
):

    print(
        f"lines "
        f"{item['line']}-{item['end_line']}: "
        f"{item['call']}"
    )


# ------------------------------------------------------------------
# 6. Print contextual source windows
#
# Inspection only. No source modification.
# ------------------------------------------------------------------

candidate_lines = sorted(
    set(
        lineno
        for lineno, _ in hits
    )
)

windows = []

for line_no in candidate_lines:

    start = max(
        1,
        line_no - 8
    )

    end = min(
        len(lines),
        line_no + 12
    )

    # Merge overlapping windows.
    if windows and start <= windows[-1][1] + 1:
        windows[-1] = (
            windows[-1][0],
            max(
                windows[-1][1],
                end
            )
        )
    else:
        windows.append(
            (start, end)
        )


print("\n" + "-" * 118)
print("SOURCE CONTEXT WINDOWS")
print("-" * 118)

for start, end in windows:

    print(
        f"\n>>> LINES {start}-{end}"
    )

    for lineno in range(
        start,
        end + 1
    ):
        print(
            f"{lineno:5d}: "
            f"{lines[lineno - 1]}"
        )


# ------------------------------------------------------------------
# 7. Reconfirm no modification
# ------------------------------------------------------------------

final_sha = sha256(RUNNER)

assert final_sha == EXPECTED_RUNNER_SHA

print("\n" + "=" * 118)
print("✓ STEP B1 COMPLETE")
print("✓ FROZEN RUNNER INSPECTED")
print("✓ LLM / JSON PARSING LOCATIONS IDENTIFIED")
print("✓ RUNNER NOT EXECUTED")
print("✓ NO MODEL/API CALL")
print("✓ NO FILE MODIFIED")
print("✓ RUNNER HASH UNCHANGED")
print("✓ SEND THIS OUTPUT BEFORE STEP B2")
print("=" * 118)

EXPERIMENT B — STEP B1: INSPECT FROZEN LLM CALL / JSON PARSING PATH
✓ Frozen runner verified
SHA-256: 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
✓ Runner parsed as Python AST
✓ Runner NOT executed

----------------------------------------------------------------------------------------------------------------------
FUNCTION / CLASS INVENTORY
----------------------------------------------------------------------------------------------------------------------
FUNCTION  json_text                                     lines 110-129
FUNCTION  obj_dict                                      lines 132-145
FUNCTION  field                                         lines 148-162
FUNCTION  bool_field                                    lines 165-172
FUNCTION  error_codes                                   lines 175-210
FUNCTION  parse_llm_result                              lines 213-289
FUNCTION  call_cir                                      lines 292-303
FUNCTION  safe_map_and_co

In [18]:
from pathlib import Path
import hashlib
import ast
import re

print("=" * 118)
print("EXPERIMENT B — STEP B2: INSPECT ADAPTER / CIR PROMPT / EXISTING SCHEMA")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def print_file_with_lines(path, max_lines=None):

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    if max_lines is not None:
        lines = lines[:max_lines]

    for i, line in enumerate(
        lines,
        start=1
    ):
        print(
            f"{i:5d}: {line}"
        )


# ======================================================================================
# 2. Verify frozen runner remains untouched
# ======================================================================================

assert RUNNER.exists(), (
    f"STOP: frozen runner missing:\n{RUNNER}"
)

runner_sha_before = sha256(RUNNER)

assert runner_sha_before == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner hash mismatch.\n"
    f"Expected: {EXPECTED_RUNNER_SHA}\n"
    f"Actual:   {runner_sha_before}"
)

print("✓ Frozen runner verified")
print("SHA-256:", runner_sha_before)


# ======================================================================================
# 3. Resolve adapter exactly as runner import path resolves it
#
# Runner inserts:
# experiment-2b/implementation/e3_v3
#
# into sys.path before importing:
# src.llm.together_adapter
#
# Therefore inspect that exact source tree first.
# ======================================================================================

E3V3_ROOT = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
)

PRIMARY_ADAPTER = (
    E3V3_ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

print("\n" + "-" * 118)
print("ADAPTER RESOLUTION")
print("-" * 118)

print("Expected adapter:")
print(PRIMARY_ADAPTER)

assert PRIMARY_ADAPTER.exists(), (
    "STOP: expected E3-v3 Together adapter not found.\n"
    f"{PRIMARY_ADAPTER}"
)

ADAPTER = PRIMARY_ADAPTER

print("✓ Exact adapter used by frozen runner located")
print("Adapter SHA-256:")
print(sha256(ADAPTER))


# ======================================================================================
# 4. Parse adapter without executing it
# ======================================================================================

adapter_source = ADAPTER.read_text(
    encoding="utf-8"
)

adapter_lines = adapter_source.splitlines()

tree = ast.parse(
    adapter_source
)

print("✓ Adapter parsed as Python AST")
print("✓ Adapter NOT executed")


# ======================================================================================
# 5. Function/class inventory
# ======================================================================================

print("\n" + "-" * 118)
print("ADAPTER FUNCTION / CLASS INVENTORY")
print("-" * 118)

definitions = []

for node in ast.walk(tree):

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):

        kind = (
            "CLASS"
            if isinstance(node, ast.ClassDef)
            else "FUNCTION"
        )

        definitions.append(
            (
                node.lineno,
                kind,
                node.name,
                getattr(
                    node,
                    "end_lineno",
                    node.lineno
                ),
            )
        )

for start, kind, name, end in sorted(
    definitions
):

    print(
        f"{kind:<9} "
        f"{name:<45} "
        f"lines {start}-{end}"
    )


# ======================================================================================
# 6. Locate call_json exactly
# ======================================================================================

call_json_nodes = [
    node
    for node in ast.walk(tree)
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    )
    and node.name == "call_json"
]

assert len(call_json_nodes) == 1, (
    "STOP: expected exactly one call_json definition; "
    f"found {len(call_json_nodes)}."
)

call_json_node = call_json_nodes[0]

CALL_JSON_START = call_json_node.lineno
CALL_JSON_END = call_json_node.end_lineno

print("\n✓ Exactly one call_json() definition located")
print(
    f"call_json lines: "
    f"{CALL_JSON_START}-{CALL_JSON_END}"
)


# ======================================================================================
# 7. Keyword inspection
# ======================================================================================

keywords = [
    "CIR_SYSTEM_PROMPT",
    "system",
    "prompt",
    "messages",
    "temperature",
    "top_p",
    "max_tokens",
    "max_completion_tokens",
    "response_format",
    "json_schema",
    "json_object",
    "schema",
    "strict",
    "chat.completions",
    "completions.create",
    "json.loads",
    "parse",
    "content",
    "reasoning",
    "Together",
]

hits = []

for lineno, line in enumerate(
    adapter_lines,
    start=1
):

    if any(
        keyword.lower() in line.lower()
        for keyword in keywords
    ):
        hits.append(
            (
                lineno,
                line
            )
        )

print("\n" + "-" * 118)
print("ADAPTER KEYWORD HITS")
print("-" * 118)

for lineno, line in hits:
    print(
        f"{lineno:5d}: {line}"
    )


# ======================================================================================
# 8. Print complete call_json() implementation
# ======================================================================================

print("\n" + "-" * 118)
print("COMPLETE call_json() IMPLEMENTATION")
print("-" * 118)

for lineno in range(
    CALL_JSON_START,
    CALL_JSON_END + 1
):

    print(
        f"{lineno:5d}: "
        f"{adapter_lines[lineno - 1]}"
    )


# ======================================================================================
# 9. Locate CIR_SYSTEM_PROMPT assignment
# ======================================================================================

print("\n" + "-" * 118)
print("CIR_SYSTEM_PROMPT DEFINITION")
print("-" * 118)

prompt_nodes = []

for node in ast.walk(tree):

    if isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):

        names = []

        if isinstance(node, ast.Assign):

            for target in node.targets:
                if isinstance(
                    target,
                    ast.Name
                ):
                    names.append(
                        target.id
                    )

        elif isinstance(
            node,
            ast.AnnAssign
        ):

            if isinstance(
                node.target,
                ast.Name
            ):
                names.append(
                    node.target.id
                )

        if "CIR_SYSTEM_PROMPT" in names:
            prompt_nodes.append(
                node
            )

if not prompt_nodes:

    print(
        "CIR_SYSTEM_PROMPT is not assigned directly "
        "inside this adapter."
    )

else:

    for node in prompt_nodes:

        start = node.lineno
        end = getattr(
            node,
            "end_lineno",
            start
        )

        print(
            f"\n>>> CIR_SYSTEM_PROMPT "
            f"lines {start}-{end}"
        )

        for lineno in range(
            start,
            end + 1
        ):
            print(
                f"{lineno:5d}: "
                f"{adapter_lines[lineno - 1]}"
            )


# ======================================================================================
# 10. Search source tree for CIR_SYSTEM_PROMPT
# ======================================================================================

print("\n" + "-" * 118)
print("CIR_SYSTEM_PROMPT REFERENCES IN E3-v3 SOURCE TREE")
print("-" * 118)

prompt_refs = []

for path in sorted(
    E3V3_ROOT.rglob("*.py")
):

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    if "CIR_SYSTEM_PROMPT" in text:

        for lineno, line in enumerate(
            text.splitlines(),
            start=1
        ):

            if "CIR_SYSTEM_PROMPT" in line:

                prompt_refs.append(
                    (
                        path,
                        lineno,
                        line
                    )
                )

for path, lineno, line in prompt_refs:

    print(
        f"{path.relative_to(ROOT)}:"
        f"{lineno}: {line}"
    )

if not prompt_refs:
    print("No CIR_SYSTEM_PROMPT references found.")


# ======================================================================================
# 11. Search for existing JSON schema / structured-output artifacts
#
# We are only identifying candidate files.
# Nothing is modified.
# ======================================================================================

print("\n" + "-" * 118)
print("EXISTING SCHEMA / STRUCTURED-OUTPUT CANDIDATES")
print("-" * 118)

schema_terms = [
    "json_schema",
    "response_format",
    "additionalProperties",
    "$schema",
    "properties",
    "required",
    "structured output",
    "structured_output",
]

schema_candidates = []

allowed_suffixes = {
    ".py",
    ".json",
    ".yaml",
    ".yml",
    ".md",
}

for path in sorted(
    E3V3_ROOT.rglob("*")
):

    if (
        not path.is_file()
        or path.suffix.lower()
        not in allowed_suffixes
    ):
        continue

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    matched_terms = [
        term
        for term in schema_terms
        if term.lower() in text.lower()
    ]

    if matched_terms:

        schema_candidates.append(
            (
                path,
                matched_terms
            )
        )

for path, matched_terms in schema_candidates:

    print(
        f"{path.relative_to(ROOT)}"
    )

    print(
        "    matched:",
        ", ".join(
            matched_terms
        )
    )

if not schema_candidates:
    print(
        "No existing schema/structured-output "
        "candidate found in E3-v3 source tree."
    )


# ======================================================================================
# 12. Look specifically for CIR field names in adapter
#
# This helps determine whether the prompt itself defines the CIR
# structure or imports it from somewhere else.
# ======================================================================================

cir_terms = [
    "service",
    "service_type",
    "latency",
    "throughput",
    "reliability",
    "availability",
    "jitter",
    "ue",
    "users",
    "devices",
    "mobility",
    "area",
    "region",
    "constraints",
]

print("\n" + "-" * 118)
print("CIR-LIKE FIELD REFERENCES INSIDE ADAPTER")
print("-" * 118)

for lineno, line in enumerate(
    adapter_lines,
    start=1
):

    matched = [
        term
        for term in cir_terms
        if re.search(
            rf"\b{re.escape(term)}\b",
            line,
            flags=re.IGNORECASE
        )
    ]

    if matched:

        print(
            f"{lineno:5d}: "
            f"{line}"
        )


# ======================================================================================
# 13. Print imports from adapter
# ======================================================================================

print("\n" + "-" * 118)
print("ADAPTER IMPORTS")
print("-" * 118)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.Import,
            ast.ImportFrom,
        )
    ):

        try:
            print(
                f"{node.lineno:5d}: "
                f"{ast.unparse(node)}"
            )
        except Exception:
            pass


# ======================================================================================
# 14. Reconfirm frozen runner AND adapter unchanged
# ======================================================================================

runner_sha_after = sha256(
    RUNNER
)

adapter_sha_after = sha256(
    ADAPTER
)

assert (
    runner_sha_after
    == runner_sha_before
)

print("\n" + "-" * 118)
print("INTEGRITY CHECK")
print("-" * 118)

print(
    "Frozen runner SHA-256 :",
    runner_sha_after
)

print(
    "Adapter SHA-256       :",
    adapter_sha_after
)


# ======================================================================================
# 15. Final
# ======================================================================================

print("\n" + "=" * 118)
print("✓ STEP B2 COMPLETE")
print("✓ EXACT TOGETHER ADAPTER INSPECTED")
print("✓ call_json() IMPLEMENTATION IDENTIFIED")
print("✓ CIR PROMPT LOCATION INSPECTED")
print("✓ EXISTING SCHEMA / STRUCTURED-OUTPUT CANDIDATES SEARCHED")
print("✓ RUNNER NOT EXECUTED")
print("✓ ADAPTER NOT EXECUTED")
print("✓ NO MODEL/API CALL")
print("✓ NO FILE MODIFIED")
print("✓ FROZEN RUNNER HASH UNCHANGED")
print("✓ SEND THIS OUTPUT BEFORE STEP B3")
print("=" * 118)

EXPERIMENT B — STEP B2: INSPECT ADAPTER / CIR PROMPT / EXISTING SCHEMA
✓ Frozen runner verified
SHA-256: 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257

----------------------------------------------------------------------------------------------------------------------
ADAPTER RESOLUTION
----------------------------------------------------------------------------------------------------------------------
Expected adapter:
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/src/llm/together_adapter.py


AssertionError: STOP: expected E3-v3 Together adapter not found.
/content/deterministic-intent-translation-journal/experiment-2b/implementation/e3_v3/src/llm/together_adapter.py

In [19]:
from pathlib import Path
import hashlib
import ast

print("=" * 118)
print("EXPERIMENT B — STEP B2A: RESOLVE PHYSICAL TOGETHER ADAPTER PATH")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)


# ======================================================================================
# 1. Helper
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. Verify frozen runner
# ======================================================================================

assert RUNNER.exists()

runner_sha_before = sha256(RUNNER)

assert runner_sha_before == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner hash mismatch.\n"
    f"Expected: {EXPECTED_RUNNER_SHA}\n"
    f"Actual:   {runner_sha_before}"
)

print("✓ Frozen runner verified")
print("SHA-256:", runner_sha_before)


# ======================================================================================
# 3. Search physically for together_adapter.py
#
# IMPORTANT:
# No import is performed.
# No Python module is executed.
# ======================================================================================

matches = sorted(
    ROOT.rglob("together_adapter.py")
)

print("\n" + "-" * 118)
print("PHYSICAL together_adapter.py CANDIDATES")
print("-" * 118)

if not matches:
    print("NONE FOUND")

else:
    for index, path in enumerate(
        matches,
        start=1
    ):
        print(
            f"[{index}] {path}"
        )
        print(
            f"    SHA-256: {sha256(path)}"
        )


# ======================================================================================
# 4. Search for files defining call_json()
# ======================================================================================

print("\n" + "-" * 118)
print("PYTHON FILES DEFINING call_json()")
print("-" * 118)

call_json_definitions = []

for path in sorted(
    ROOT.rglob("*.py")
):

    try:
        source = path.read_text(
            encoding="utf-8"
        )

        tree = ast.parse(source)

    except Exception:
        continue

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ) and node.name == "call_json":

            call_json_definitions.append(
                {
                    "path": path,
                    "start": node.lineno,
                    "end": getattr(
                        node,
                        "end_lineno",
                        node.lineno
                    ),
                    "sha256": sha256(path),
                }
            )


if not call_json_definitions:
    print("NONE FOUND")

else:
    for index, item in enumerate(
        call_json_definitions,
        start=1
    ):

        print(
            f"[{index}] {item['path']}"
        )

        print(
            f"    call_json lines: "
            f"{item['start']}-{item['end']}"
        )

        print(
            f"    SHA-256: "
            f"{item['sha256']}"
        )


# ======================================================================================
# 5. Inspect runner path configuration only
# ======================================================================================

print("\n" + "-" * 118)
print("RUNNER PATH-CONFIGURATION SECTION")
print("-" * 118)

runner_lines = RUNNER.read_text(
    encoding="utf-8"
).splitlines()

# B1 showed the relevant path/import setup near the beginning.
for lineno in range(
    34,
    min(110, len(runner_lines)) + 1
):
    print(
        f"{lineno:5d}: "
        f"{runner_lines[lineno - 1]}"
    )


# ======================================================================================
# 6. Find every runner reference to sys.path / SRC / E3V3
# ======================================================================================

print("\n" + "-" * 118)
print("RUNNER PATH-RESOLUTION REFERENCES")
print("-" * 118)

terms = [
    "sys.path",
    "SRC",
    "E3V3",
    "experiment2b",
    "together_adapter",
]

for lineno, line in enumerate(
    runner_lines,
    start=1
):

    if any(
        term.lower() in line.lower()
        for term in terms
    ):
        print(
            f"{lineno:5d}: {line}"
        )


# ======================================================================================
# 7. Integrity check
# ======================================================================================

runner_sha_after = sha256(RUNNER)

assert runner_sha_after == EXPECTED_RUNNER_SHA

print("\n" + "=" * 118)
print("✓ STEP B2A COMPLETE")
print("✓ PHYSICAL ADAPTER CANDIDATES SEARCHED")
print("✓ call_json() DEFINITIONS SEARCHED")
print("✓ RUNNER PATH CONFIGURATION INSPECTED")
print("✓ NO MODULE IMPORTED")
print("✓ NO ADAPTER EXECUTED")
print("✓ NO MODEL/API CALL")
print("✓ NO FILE MODIFIED")
print("✓ FROZEN RUNNER HASH UNCHANGED")
print("✓ SEND THIS OUTPUT BEFORE CONTINUING B2")
print("=" * 118)

EXPERIMENT B — STEP B2A: RESOLVE PHYSICAL TOGETHER ADAPTER PATH
✓ Frozen runner verified
SHA-256: 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257

----------------------------------------------------------------------------------------------------------------------
PHYSICAL together_adapter.py CANDIDATES
----------------------------------------------------------------------------------------------------------------------
[1] /content/deterministic-intent-translation-journal/src/llm/together_adapter.py
    SHA-256: b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3

----------------------------------------------------------------------------------------------------------------------
PYTHON FILES DEFINING call_json()
----------------------------------------------------------------------------------------------------------------------
[1] /content/deterministic-intent-translation-journal/src/llm/together_adapter.py
    call_json lines: 80-180
    SHA-256: b1

In [20]:
from pathlib import Path
import hashlib
import ast
import re

print("=" * 118)
print("EXPERIMENT B — STEP B2B: INSPECT EXACT TOGETHER ADAPTER / CIR PROMPT / SCHEMA")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_ADAPTER_SHA = (
    "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3"
)


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. Verify frozen runner + exact adapter
# ======================================================================================

assert RUNNER.exists(), (
    f"STOP: runner missing:\n{RUNNER}"
)

assert ADAPTER.exists(), (
    f"STOP: adapter missing:\n{ADAPTER}"
)

runner_sha_before = sha256(RUNNER)
adapter_sha_before = sha256(ADAPTER)

assert runner_sha_before == EXPECTED_RUNNER_SHA, (
    "STOP: frozen runner hash mismatch.\n"
    f"Expected: {EXPECTED_RUNNER_SHA}\n"
    f"Actual:   {runner_sha_before}"
)

assert adapter_sha_before == EXPECTED_ADAPTER_SHA, (
    "STOP: adapter hash mismatch.\n"
    f"Expected: {EXPECTED_ADAPTER_SHA}\n"
    f"Actual:   {adapter_sha_before}"
)

print("✓ Frozen runner verified")
print("✓ Exact Together adapter verified")
print("Runner SHA-256 :", runner_sha_before)
print("Adapter SHA-256:", adapter_sha_before)


# ======================================================================================
# 3. Parse adapter statically — DO NOT IMPORT IT
# ======================================================================================

source = ADAPTER.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

tree = ast.parse(source)

print("✓ Adapter parsed as AST")
print("✓ Adapter NOT imported or executed")


# ======================================================================================
# 4. Function/class inventory
# ======================================================================================

print("\n" + "-" * 118)
print("FUNCTION / CLASS INVENTORY")
print("-" * 118)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):
        print(
            f"FUNCTION  {node.name:<40} "
            f"lines {node.lineno}-{node.end_lineno}"
        )

    elif isinstance(node, ast.ClassDef):
        print(
            f"CLASS     {node.name:<40} "
            f"lines {node.lineno}-{node.end_lineno}"
        )


# ======================================================================================
# 5. Locate call_json
# ======================================================================================

call_json_nodes = [
    node
    for node in tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    )
    and node.name == "call_json"
]

assert len(call_json_nodes) == 1, (
    "STOP: expected exactly one call_json(); "
    f"found {len(call_json_nodes)}"
)

call_node = call_json_nodes[0]

print("\n✓ call_json() uniquely identified")
print(
    f"Lines: {call_node.lineno}-{call_node.end_lineno}"
)


# ======================================================================================
# 6. Print imports
# ======================================================================================

print("\n" + "-" * 118)
print("ADAPTER IMPORTS")
print("-" * 118)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.Import,
            ast.ImportFrom,
        )
    ):
        print(
            f"{node.lineno:5d}: "
            f"{ast.unparse(node)}"
        )


# ======================================================================================
# 7. Print module-level assignments/constants
# ======================================================================================

print("\n" + "-" * 118)
print("MODULE-LEVEL CONSTANTS / ASSIGNMENTS")
print("-" * 118)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):

        try:
            rendered = ast.unparse(node)
        except Exception:
            rendered = "<unable to render>"

        start = node.lineno
        end = getattr(
            node,
            "end_lineno",
            start
        )

        # Avoid dumping secrets if any environment-related assignment exists.
        if any(
            secret_term in rendered.upper()
            for secret_term in [
                "API_KEY",
                "TOKEN",
                "SECRET",
                "PASSWORD",
            ]
        ):
            print(
                f"{start:5d}-{end:<5d}: "
                "<secret/environment-related assignment omitted>"
            )
        else:
            print(
                f"{start:5d}-{end:<5d}: "
                f"{rendered}"
            )


# ======================================================================================
# 8. Complete call_json implementation
# ======================================================================================

print("\n" + "-" * 118)
print("COMPLETE call_json() IMPLEMENTATION")
print("-" * 118)

for lineno in range(
    call_node.lineno,
    call_node.end_lineno + 1
):
    print(
        f"{lineno:5d}: "
        f"{lines[lineno - 1]}"
    )


# ======================================================================================
# 9. Find prompt-related definitions/references
# ======================================================================================

print("\n" + "-" * 118)
print("PROMPT-RELATED DEFINITIONS / REFERENCES")
print("-" * 118)

prompt_terms = [
    "CIR_SYSTEM_PROMPT",
    "SYSTEM_PROMPT",
    "PROMPT",
]

prompt_hits = []

for lineno, line in enumerate(
    lines,
    start=1
):

    if any(
        term.lower() in line.lower()
        for term in prompt_terms
    ):
        prompt_hits.append(
            (lineno, line)
        )

for lineno, line in prompt_hits:
    print(
        f"{lineno:5d}: {line}"
    )

if not prompt_hits:
    print("No prompt references found in adapter.")


# ======================================================================================
# 10. Print direct CIR_SYSTEM_PROMPT assignment if present
# ======================================================================================

print("\n" + "-" * 118)
print("CIR_SYSTEM_PROMPT CONTENT")
print("-" * 118)

found_direct_prompt = False

for node in tree.body:

    if not isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):
        continue

    target_names = []

    if isinstance(node, ast.Assign):

        for target in node.targets:
            if isinstance(target, ast.Name):
                target_names.append(target.id)

    else:
        if isinstance(node.target, ast.Name):
            target_names.append(node.target.id)

    if "CIR_SYSTEM_PROMPT" not in target_names:
        continue

    found_direct_prompt = True

    start = node.lineno
    end = getattr(
        node,
        "end_lineno",
        start
    )

    print(
        f">>> CIR_SYSTEM_PROMPT lines {start}-{end}"
    )

    for lineno in range(
        start,
        end + 1
    ):
        print(
            f"{lineno:5d}: "
            f"{lines[lineno - 1]}"
        )

if not found_direct_prompt:
    print(
        "CIR_SYSTEM_PROMPT is not directly assigned "
        "inside together_adapter.py"
    )


# ======================================================================================
# 11. Search whole project for CIR_SYSTEM_PROMPT definition
# ======================================================================================

print("\n" + "-" * 118)
print("PROJECT-WIDE CIR_SYSTEM_PROMPT REFERENCES")
print("-" * 118)

cir_prompt_refs = []

for path in sorted(
    ROOT.rglob("*.py")
):

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    if "CIR_SYSTEM_PROMPT" not in text:
        continue

    for lineno, line in enumerate(
        text.splitlines(),
        start=1
    ):
        if "CIR_SYSTEM_PROMPT" in line:
            cir_prompt_refs.append(
                (
                    path,
                    lineno,
                    line,
                )
            )

for path, lineno, line in cir_prompt_refs:
    print(
        f"{path.relative_to(ROOT)}:"
        f"{lineno}: {line}"
    )

if not cir_prompt_refs:
    print("NONE FOUND")


# ======================================================================================
# 12. Search exact adapter for decoding controls
# ======================================================================================

print("\n" + "-" * 118)
print("DECODING / STRUCTURED-OUTPUT SETTINGS")
print("-" * 118)

decoding_terms = [
    "temperature",
    "top_p",
    "top_k",
    "max_tokens",
    "max_completion_tokens",
    "seed",
    "response_format",
    "json_schema",
    "json_object",
    "strict",
    "tool_choice",
    "grammar",
]

decoding_hits = []

for lineno, line in enumerate(
    lines,
    start=1
):

    matched = [
        term
        for term in decoding_terms
        if term.lower() in line.lower()
    ]

    if matched:
        decoding_hits.append(
            (
                lineno,
                matched,
                line,
            )
        )

for lineno, matched, line in decoding_hits:
    print(
        f"{lineno:5d}: "
        f"[{', '.join(matched)}] "
        f"{line}"
    )

if not decoding_hits:
    print("No explicit decoding-control settings found.")


# ======================================================================================
# 13. Search project for structured-output/schema implementations
# ======================================================================================

print("\n" + "-" * 118)
print("PROJECT-WIDE STRUCTURED-OUTPUT / JSON-SCHEMA CANDIDATES")
print("-" * 118)

schema_patterns = [
    "response_format",
    "json_schema",
    "additionalProperties",
    '"$schema"',
    "'$schema'",
    "structured_output",
    "structured output",
]

schema_files = []

allowed_suffixes = {
    ".py",
    ".json",
    ".yaml",
    ".yml",
    ".md",
}

for path in sorted(
    ROOT.rglob("*")
):

    if (
        not path.is_file()
        or path.suffix.lower()
        not in allowed_suffixes
    ):
        continue

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    matched = [
        pattern
        for pattern in schema_patterns
        if pattern.lower() in text.lower()
    ]

    if matched:
        schema_files.append(
            (
                path,
                matched,
            )
        )

for path, matched in schema_files:

    print(
        str(path.relative_to(ROOT))
    )

    print(
        "    matched:",
        ", ".join(matched)
    )

if not schema_files:
    print(
        "No structured-output / JSON-schema "
        "implementation found in project."
    )


# ======================================================================================
# 14. Identify JSON parsing operations in adapter
# ======================================================================================

print("\n" + "-" * 118)
print("JSON PARSING / RESPONSE EXTRACTION OPERATIONS")
print("-" * 118)

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    try:
        call_name = ast.unparse(
            node.func
        )
    except Exception:
        continue

    lower = call_name.lower()

    if any(
        term in lower
        for term in [
            "json.loads",
            "json.dumps",
            "completions.create",
            "chat.completions",
        ]
    ):
        print(
            f"line {node.lineno}: "
            f"{call_name}"
        )


# ======================================================================================
# 15. Integrity verification
# ======================================================================================

runner_sha_after = sha256(RUNNER)
adapter_sha_after = sha256(ADAPTER)

assert runner_sha_after == EXPECTED_RUNNER_SHA
assert adapter_sha_after == EXPECTED_ADAPTER_SHA

print("\n" + "-" * 118)
print("INTEGRITY VERIFICATION")
print("-" * 118)

print(
    "Runner SHA unchanged :",
    runner_sha_after
)

print(
    "Adapter SHA unchanged:",
    adapter_sha_after
)


# ======================================================================================
# 16. Final
# ======================================================================================

print("\n" + "=" * 118)
print("✓ STEP B2B COMPLETE")
print("✓ EXACT HISTORICAL ADAPTER INSPECTED")
print("✓ call_json() INSPECTED")
print("✓ PROMPT LOCATION INSPECTED")
print("✓ DECODING SETTINGS INSPECTED")
print("✓ STRUCTURED-OUTPUT IMPLEMENTATIONS SEARCHED")
print("✓ JSON RESPONSE PARSING PATH INSPECTED")
print("✓ NO MODULE IMPORTED")
print("✓ NO MODEL/API CALL")
print("✓ NO FILE MODIFIED")
print("✓ RUNNER HASH UNCHANGED")
print("✓ ADAPTER HASH UNCHANGED")
print("✓ SEND THIS OUTPUT BEFORE STEP B3")
print("=" * 118)

EXPERIMENT B — STEP B2B: INSPECT EXACT TOGETHER ADAPTER / CIR PROMPT / SCHEMA
✓ Frozen runner verified
✓ Exact Together adapter verified
Runner SHA-256 : 40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257
Adapter SHA-256: b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3
✓ Adapter parsed as AST
✓ Adapter NOT imported or executed

----------------------------------------------------------------------------------------------------------------------
FUNCTION / CLASS INVENTORY
----------------------------------------------------------------------------------------------------------------------
FUNCTION  _clean_json_text                         lines 61-77
FUNCTION  call_json                                lines 80-180

✓ call_json() uniquely identified
Lines: 80-180

----------------------------------------------------------------------------------------------------------------------
ADAPTER IMPORTS
--------------------------------------------------------------

In [21]:
from pathlib import Path
import hashlib
import ast

print("=" * 118)
print("EXPERIMENT B — STEP B3: INSPECT EXISTING CIR SCHEMA")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

SCHEMA = (
    ROOT
    / "src"
    / "schema.py"
)

SNAPSHOT_SCHEMA = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "r3_preimplementation_snapshot"
    / "shared_schema_reference.py"
)

EXPECTED_RUNNER_SHA = (
    "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257"
)

EXPECTED_ADAPTER_SHA = (
    "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3"
)


# ======================================================================================
# 1. Helper
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. Verify frozen evidence
# ======================================================================================

assert RUNNER.exists(), f"STOP: runner missing: {RUNNER}"
assert ADAPTER.exists(), f"STOP: adapter missing: {ADAPTER}"
assert SCHEMA.exists(), f"STOP: schema missing: {SCHEMA}"

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA, (
    "STOP: runner hash mismatch"
)

assert sha256(ADAPTER) == EXPECTED_ADAPTER_SHA, (
    "STOP: adapter hash mismatch"
)

print("✓ Frozen runner verified")
print("✓ Historical adapter verified")

print("\nSchema:")
print(SCHEMA)
print("SHA-256:")
print(sha256(SCHEMA))


# ======================================================================================
# 3. Parse schema statically
# ======================================================================================

schema_source = SCHEMA.read_text(
    encoding="utf-8"
)

schema_lines = schema_source.splitlines()

schema_tree = ast.parse(
    schema_source
)

print("✓ src/schema.py parsed as AST")
print("✓ Schema module NOT imported or executed")


# ======================================================================================
# 4. Print complete schema.py
# ======================================================================================

print("\n" + "-" * 118)
print("COMPLETE src/schema.py")
print("-" * 118)

for lineno, line in enumerate(
    schema_lines,
    start=1
):
    print(
        f"{lineno:5d}: {line}"
    )


# ======================================================================================
# 5. Inventory definitions
# ======================================================================================

print("\n" + "-" * 118)
print("SCHEMA DEFINITIONS")
print("-" * 118)

for node in schema_tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):

        kind = (
            "CLASS"
            if isinstance(node, ast.ClassDef)
            else "FUNCTION"
        )

        print(
            f"{kind:<9} "
            f"{node.name:<40} "
            f"lines {node.lineno}-{node.end_lineno}"
        )

    elif isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):

        try:
            rendered = ast.unparse(node)
        except Exception:
            rendered = "<unable to render>"

        print(
            f"ASSIGN    "
            f"lines {node.lineno}-"
            f"{getattr(node, 'end_lineno', node.lineno)}"
        )

        print(rendered)


# ======================================================================================
# 6. Extract literal dictionaries safely using AST only
#
# ast.literal_eval does NOT execute arbitrary Python.
# ======================================================================================

print("\n" + "-" * 118)
print("LITERAL DICTIONARY OBJECTS")
print("-" * 118)

literal_dicts = []

for node in schema_tree.body:

    if not isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):
        continue

    value_node = (
        node.value
        if hasattr(node, "value")
        else None
    )

    if value_node is None:
        continue

    try:
        value = ast.literal_eval(
            value_node
        )
    except Exception:
        continue

    if not isinstance(value, dict):
        continue

    names = []

    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name):
                names.append(target.id)

    elif isinstance(node, ast.AnnAssign):
        if isinstance(node.target, ast.Name):
            names.append(node.target.id)

    literal_dicts.append(
        (
            names,
            value,
            node.lineno,
            getattr(
                node,
                "end_lineno",
                node.lineno
            ),
        )
    )

for names, value, start, end in literal_dicts:

    print(
        f"Names : {names}"
    )

    print(
        f"Lines : {start}-{end}"
    )

    print(
        f"Top-level keys: "
        f"{list(value.keys())}"
    )

    if "properties" in value:
        properties = value["properties"]

        print(
            "Properties:",
            list(properties.keys())
            if isinstance(properties, dict)
            else properties
        )

    if "required" in value:
        print(
            "Required:",
            value["required"]
        )

    if "additionalProperties" in value:
        print(
            "additionalProperties:",
            value["additionalProperties"]
        )

    print()


# ======================================================================================
# 7. Compare against seven-field prompt contract
# ======================================================================================

PROMPT_FIELDS = [
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
]

print("\n" + "-" * 118)
print("SEVEN-FIELD CIR CONTRACT COMPARISON")
print("-" * 118)

print(
    "Prompt fields:",
    PROMPT_FIELDS
)

schema_property_sets = []

for names, value, start, end in literal_dicts:

    properties = value.get(
        "properties"
    )

    if not isinstance(
        properties,
        dict
    ):
        continue

    prop_names = list(
        properties.keys()
    )

    schema_property_sets.append(
        (
            names,
            prop_names,
            value,
        )
    )

    print("\nSchema object:", names)

    print(
        "Schema properties:",
        prop_names
    )

    print(
        "Missing from schema:",
        sorted(
            set(PROMPT_FIELDS)
            - set(prop_names)
        )
    )

    print(
        "Extra in schema:",
        sorted(
            set(prop_names)
            - set(PROMPT_FIELDS)
        )
    )

    print(
        "Exact field-set match:",
        set(prop_names)
        == set(PROMPT_FIELDS)
    )

    print(
        "Exact field-order match:",
        prop_names
        == PROMPT_FIELDS
    )

    print(
        "Required:",
        value.get(
            "required",
            "<not specified>"
        )
    )

    print(
        "additionalProperties:",
        value.get(
            "additionalProperties",
            "<not specified>"
        )
    )


# ======================================================================================
# 8. Inspect historical snapshot schema
# ======================================================================================

print("\n" + "-" * 118)
print("R3 PREIMPLEMENTATION SNAPSHOT SCHEMA")
print("-" * 118)

if SNAPSHOT_SCHEMA.exists():

    print(
        "Snapshot:",
        SNAPSHOT_SCHEMA
    )

    print(
        "SHA-256:",
        sha256(SNAPSHOT_SCHEMA)
    )

    snapshot_source = SNAPSHOT_SCHEMA.read_text(
        encoding="utf-8"
    )

    snapshot_lines = snapshot_source.splitlines()

    for lineno, line in enumerate(
        snapshot_lines,
        start=1
    ):
        print(
            f"{lineno:5d}: {line}"
        )

else:
    print(
        "Snapshot schema not found at expected path."
    )


# ======================================================================================
# 9. Find where src.schema is actually used
# ======================================================================================

print("\n" + "-" * 118)
print("PROJECT REFERENCES TO src.schema / SCHEMA SYMBOLS")
print("-" * 118)

schema_filename = SCHEMA.name

reference_hits = []

search_terms = [
    "from src.schema",
    "import src.schema",
    "from .schema",
    "import schema",
]

for path in sorted(
    ROOT.rglob("*.py")
):

    if path == SCHEMA:
        continue

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    for lineno, line in enumerate(
        text.splitlines(),
        start=1
    ):

        if any(
            term in line
            for term in search_terms
        ):
            reference_hits.append(
                (
                    path,
                    lineno,
                    line,
                )
            )

for path, lineno, line in reference_hits:

    print(
        f"{path.relative_to(ROOT)}:"
        f"{lineno}: {line}"
    )

if not reference_hits:
    print(
        "No direct schema-module imports found."
    )


# ======================================================================================
# 10. Reconfirm immutable files
# ======================================================================================

assert sha256(RUNNER) == EXPECTED_RUNNER_SHA
assert sha256(ADAPTER) == EXPECTED_ADAPTER_SHA

print("\n" + "=" * 118)
print("✓ STEP B3 COMPLETE")
print("✓ EXISTING src/schema.py INSPECTED")
print("✓ SEVEN-FIELD PROMPT CONTRACT COMPARED WITH EXISTING SCHEMA")
print("✓ HISTORICAL SNAPSHOT SCHEMA INSPECTED IF PRESENT")
print("✓ SCHEMA USAGE REFERENCES SEARCHED")
print("✓ NO MODULE IMPORTED")
print("✓ NO MODEL/API CALL")
print("✓ NO FILE MODIFIED")
print("✓ RUNNER + ADAPTER HASHES UNCHANGED")
print("✓ SEND THIS OUTPUT BEFORE STEP B4")
print("=" * 118)

EXPERIMENT B — STEP B3: INSPECT EXISTING CIR SCHEMA
✓ Frozen runner verified
✓ Historical adapter verified

Schema:
/content/deterministic-intent-translation-journal/src/schema.py
SHA-256:
f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34
✓ src/schema.py parsed as AST
✓ Schema module NOT imported or executed

----------------------------------------------------------------------------------------------------------------------
COMPLETE src/schema.py
----------------------------------------------------------------------------------------------------------------------
    1: CIR_SCHEMA={"type":"object","properties":{"service_type":{"type":["string","null"],"enum":["eMBB","URLLC","mMTC",None]},"latency_ms":{"type":["number","null"],"minimum":0},"bandwidth_mbps":{"type":["number","null"],"minimum":0},"reliability":{"type":["number","null"],"minimum":0,"maximum":1},"device_count":{"type":["integer","null"],"minimum":1},"coverage":{"type":["string","null"]},"objective":{"type":

In [22]:
from pathlib import Path
import hashlib
import ast

print("=" * 118)
print("EXPERIMENT B — STEP B4: INSPECT SCHEMA VS DETERMINISTIC-VALIDATION RESPONSIBILITY")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

SCHEMA = (
    ROOT
    / "src"
    / "schema.py"
)

VALIDATOR = (
    ROOT
    / "src"
    / "validator.py"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",
}


# ======================================================================================
# 1. Helper
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. Verify frozen files
# ======================================================================================

for label, path, expected_hash in [
    ("runner", RUNNER, EXPECTED["runner"]),
    ("adapter", ADAPTER, EXPECTED["adapter"]),
    ("schema", SCHEMA, EXPECTED["schema"]),
]:

    assert path.exists(), (
        f"STOP: missing {label}: {path}"
    )

    actual = sha256(path)

    assert actual == expected_hash, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected_hash}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


assert VALIDATOR.exists(), (
    f"STOP: validator missing:\n{VALIDATOR}"
)

VALIDATOR_SHA = sha256(VALIDATOR)

print("✓ validator located")
print("Validator SHA-256:", VALIDATOR_SHA)


# ======================================================================================
# 3. Parse validator statically
# ======================================================================================

source = VALIDATOR.read_text(
    encoding="utf-8"
)

lines = source.splitlines()

tree = ast.parse(source)

print("✓ Validator parsed as AST")
print("✓ Validator NOT imported or executed")


# ======================================================================================
# 4. Print complete validator
# ======================================================================================

print("\n" + "-" * 118)
print("COMPLETE src/validator.py")
print("-" * 118)

for lineno, line in enumerate(
    lines,
    start=1
):
    print(
        f"{lineno:5d}: {line}"
    )


# ======================================================================================
# 5. Inventory validator functions/classes
# ======================================================================================

print("\n" + "-" * 118)
print("VALIDATOR DEFINITIONS")
print("-" * 118)

for node in tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):

        kind = (
            "CLASS"
            if isinstance(node, ast.ClassDef)
            else "FUNCTION"
        )

        print(
            f"{kind:<9} "
            f"{node.name:<40} "
            f"lines {node.lineno}-"
            f"{getattr(node, 'end_lineno', node.lineno)}"
        )


# ======================================================================================
# 6. Locate validate_cir()
# ======================================================================================

validate_nodes = [
    node
    for node in tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    )
    and node.name == "validate_cir"
]

assert len(validate_nodes) == 1, (
    "STOP: expected exactly one validate_cir(); "
    f"found {len(validate_nodes)}"
)

validate_node = validate_nodes[0]

print("\n✓ validate_cir() uniquely identified")
print(
    f"Lines: "
    f"{validate_node.lineno}-{validate_node.end_lineno}"
)


# ======================================================================================
# 7. Show schema-related validator operations
# ======================================================================================

print("\n" + "-" * 118)
print("SCHEMA / VALIDATION OPERATIONS")
print("-" * 118)

terms = [
    "CIR_SCHEMA",
    "ALLOWED",
    "SUPPORTED",
    "jsonschema",
    "validate",
    "minimum",
    "maximum",
    "required",
    "additionalProperties",
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
]

for lineno, line in enumerate(
    lines,
    start=1
):

    if any(
        term.lower() in line.lower()
        for term in terms
    ):
        print(
            f"{lineno:5d}: {line}"
        )


# ======================================================================================
# 8. Print complete validate_cir()
# ======================================================================================

print("\n" + "-" * 118)
print("COMPLETE validate_cir() IMPLEMENTATION")
print("-" * 118)

for lineno in range(
    validate_node.lineno,
    validate_node.end_lineno + 1
):

    print(
        f"{lineno:5d}: "
        f"{lines[lineno - 1]}"
    )


# ======================================================================================
# 9. Inspect runner around validate_cir call
# ======================================================================================

runner_source = RUNNER.read_text(
    encoding="utf-8"
)

runner_lines = runner_source.splitlines()

print("\n" + "-" * 118)
print("RUNNER validate_cir() CALL LOCATIONS")
print("-" * 118)

validation_call_lines = []

for lineno, line in enumerate(
    runner_lines,
    start=1
):

    if "validate_cir" in line:
        validation_call_lines.append(
            lineno
        )

        print(
            f"{lineno:5d}: {line}"
        )


print("\n" + "-" * 118)
print("RUNNER VALIDATION CONTEXT")
print("-" * 118)

windows = []

for line_no in validation_call_lines:

    # Ignore import line if applicable.
    if line_no < 100:
        continue

    start = max(
        1,
        line_no - 15
    )

    end = min(
        len(runner_lines),
        line_no + 25
    )

    if (
        windows
        and start <= windows[-1][1] + 1
    ):
        windows[-1] = (
            windows[-1][0],
            max(
                windows[-1][1],
                end
            )
        )
    else:
        windows.append(
            (start, end)
        )


for start, end in windows:

    print(
        f"\n>>> LINES {start}-{end}"
    )

    for lineno in range(
        start,
        end + 1
    ):

        print(
            f"{lineno:5d}: "
            f"{runner_lines[lineno - 1]}"
        )


# ======================================================================================
# 10. Classify schema constraints structurally
#
# This is static inspection only.
# ======================================================================================

schema_tree = ast.parse(
    SCHEMA.read_text(
        encoding="utf-8"
    )
)

cir_schema = None

for node in schema_tree.body:

    if not isinstance(
        node,
        ast.Assign
    ):
        continue

    names = [
        target.id
        for target in node.targets
        if isinstance(
            target,
            ast.Name
        )
    ]

    if "CIR_SCHEMA" not in names:
        continue

    try:
        cir_schema = ast.literal_eval(
            node.value
        )
    except Exception:
        pass


assert isinstance(
    cir_schema,
    dict
), "STOP: unable to statically extract CIR_SCHEMA"


print("\n" + "-" * 118)
print("SCHEMA CONSTRAINT CLASSIFICATION")
print("-" * 118)

properties = cir_schema[
    "properties"
]

for field_name, definition in properties.items():

    structural = {}

    semantic = {}

    for key, value in definition.items():

        if key in {
            "type",
        }:
            structural[key] = value

        elif key in {
            "minimum",
            "maximum",
            "enum",
            "pattern",
            "minLength",
            "maxLength",
        }:
            semantic[key] = value

        else:
            structural[key] = value

    print(
        f"\nFIELD: {field_name}"
    )

    print(
        "  structural:",
        structural
    )

    print(
        "  value-restricting:",
        semantic
    )


print("\nObject-level:")
print(
    "  required:",
    cir_schema.get(
        "required"
    )
)

print(
    "  additionalProperties:",
    cir_schema.get(
        "additionalProperties"
    )
)


# ======================================================================================
# 11. Identify potential Experiment-B confound
# ======================================================================================

value_restrictions = {}

for field_name, definition in properties.items():

    restrictions = {
        key: value
        for key, value in definition.items()
        if key in {
            "minimum",
            "maximum",
            "enum",
            "pattern",
            "minLength",
            "maxLength",
        }
    }

    if restrictions:
        value_restrictions[
            field_name
        ] = restrictions


print("\n" + "-" * 118)
print("POTENTIAL CONSTRAINED-DECODING CONFOUND")
print("-" * 118)

print(
    "Fields with value-restricting schema rules:"
)

for field_name, restrictions in value_restrictions.items():
    print(
        f"  {field_name}: {restrictions}"
    )

print()

if value_restrictions:

    print(
        "OBSERVATION: The existing CIR_SCHEMA contains "
        "value restrictions in addition to structural JSON "
        "constraints."
    )

    print(
        "Therefore, using CIR_SCHEMA unchanged as a provider-side "
        "decoding grammar could prevent reproduction of some "
        "explicitly stated invalid source values."
    )

    print(
        "That possibility must be resolved BEFORE Experiment B "
        "is preregistered."
    )

else:

    print(
        "No value-restricting schema rules detected."
    )


# ======================================================================================
# 12. Integrity check
# ======================================================================================

assert sha256(RUNNER) == EXPECTED["runner"]
assert sha256(ADAPTER) == EXPECTED["adapter"]
assert sha256(SCHEMA) == EXPECTED["schema"]
assert sha256(VALIDATOR) == VALIDATOR_SHA

print("\n" + "=" * 118)
print("✓ STEP B4 COMPLETE")
print("✓ DETERMINISTIC VALIDATOR INSPECTED")
print("✓ SCHEMA CONSTRAINT RESPONSIBILITIES INSPECTED")
print("✓ RUNNER VALIDATION POSITION INSPECTED")
print("✓ POTENTIAL CONSTRAINED-DECODING CONFOUND IDENTIFIED")
print("✓ NO MODULE IMPORTED")
print("✓ NO MODEL/API CALL")
print("✓ NO FILE MODIFIED")
print("✓ ALL INSPECTED HASHES UNCHANGED")
print("✓ SEND THIS OUTPUT BEFORE EXPERIMENT-B DESIGN / PREREGISTRATION")
print("=" * 118)

EXPERIMENT B — STEP B4: INSPECT SCHEMA VS DETERMINISTIC-VALIDATION RESPONSIBILITY
✓ runner verified
✓ adapter verified
✓ schema verified
✓ validator located
Validator SHA-256: c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2
✓ Validator parsed as AST
✓ Validator NOT imported or executed

----------------------------------------------------------------------------------------------------------------------
COMPLETE src/validator.py
----------------------------------------------------------------------------------------------------------------------
    1: from jsonschema import Draft202012Validator
    2: from .schema import CIR_SCHEMA, ALLOWED, SUPPORTED
    3: 
    4: 
    5: def validate_cir(c):
    6:     errors = []
    7: 
    8:     # 1. Detect unknown/generated fields
    9:     unknown = sorted(set(c) - ALLOWED)
   10: 
   11:     # 2. Collect ALL JSON-schema violations
   12:     validator = Draft202012Validator(CIR_SCHEMA)
   13: 
   14:     schema_errors = sor

In [23]:
from pathlib import Path
import hashlib
import json
import shutil
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B — STEP B5: PREREGISTER SCHEMA-CONSTRAINED DECODING EXPERIMENT")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent"
)

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = (
    DRIVE_ROOT
    / "experiment_B_schema_constrained_decoding"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

SCHEMA = (
    ROOT
    / "src"
    / "schema.py"
)

VALIDATOR = (
    ROOT
    / "src"
    / "validator.py"
)

A_FINAL_FREEZE = (
    ROOT
    / "journal_experiments"
    / "experiment_A_model_sweep"
    / "manifests"
    / "experiment_A_FINAL_FREEZE_v1.json"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "benchmark":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",

    "experiment_A_final_freeze":
        "667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a",
}


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")
    return actual


# ======================================================================================
# 2. Verify frozen provenance
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

verify(
    "runner",
    RUNNER,
    EXPECTED["runner"]
)

verify(
    "benchmark",
    BENCHMARK,
    EXPECTED["benchmark"]
)

verify(
    "adapter",
    ADAPTER,
    EXPECTED["adapter"]
)

verify(
    "schema",
    SCHEMA,
    EXPECTED["schema"]
)

verify(
    "validator",
    VALIDATOR,
    EXPECTED["validator"]
)

verify(
    "Experiment-A final freeze",
    A_FINAL_FREEZE,
    EXPECTED["experiment_A_final_freeze"]
)


# ======================================================================================
# 3. Freeze structural decoding schema
#
# IMPORTANT:
# This deliberately preserves:
#   - object structure
#   - seven exact keys
#   - required keys
#   - primitive/null types
#   - no additional properties
#
# It deliberately DOES NOT include:
#   - service_type enum
#   - numeric minimum/maximum
#
# Those remain responsibilities of the frozen deterministic validator.
# ======================================================================================

STRUCTURAL_DECODING_SCHEMA = {
    "type": "object",

    "properties": {
        "service_type": {
            "type": [
                "string",
                "null"
            ]
        },

        "latency_ms": {
            "type": [
                "number",
                "null"
            ]
        },

        "bandwidth_mbps": {
            "type": [
                "number",
                "null"
            ]
        },

        "reliability": {
            "type": [
                "number",
                "null"
            ]
        },

        "device_count": {
            "type": [
                "integer",
                "null"
            ]
        },

        "coverage": {
            "type": [
                "string",
                "null"
            ]
        },

        "objective": {
            "type": [
                "string",
                "null"
            ]
        },
    },

    "required": [
        "service_type",
        "latency_ms",
        "bandwidth_mbps",
        "reliability",
        "device_count",
        "coverage",
        "objective",
    ],

    "additionalProperties": False,
}


# ======================================================================================
# 4. Define preregistered experiment
# ======================================================================================

PREREG = {
    "experiment":
        "Experiment B — Schema-Constrained Decoding",

    "version":
        "v1",

    "status":
        "PREREGISTERED_NOT_EXECUTED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "research_question":
        (
            "Does provider-side structural JSON-schema constrained "
            "decoding eliminate the need for the frozen deterministic "
            "validation and failure-containment cascade?"
        ),

    "design_type":
        "prospective controlled decoding-ablation experiment",

    "benchmark": {
        "cases": 120,
        "path": str(BENCHMARK),
        "sha256": EXPECTED["benchmark"],
        "reuse_policy":
            "Use the frozen E3-v3 held-out benchmark unchanged."
    },

    "semantic_model": {
        "model":
            "Qwen/Qwen3.5-9B",

        "reason":
            (
                "Use the historical frozen baseline semantic parser "
                "from E3-v3 / Experiment-A A1 so that Experiment B "
                "changes decoding mode rather than model identity."
            ),

        "model_change_allowed":
            False,
    },

    "control_arm": {
        "id":
            "B0",

        "name":
            "unconstrained_json_generation",

        "definition":
            (
                "Historical frozen E3-v3 behavior: identical CIR "
                "system prompt, temperature=0, no provider-side "
                "response_format/json_schema constraint, followed "
                "by historical JSON text cleaning/parsing and the "
                "full deterministic E3-v3 cascade."
            ),

        "historical_baseline":
            True,

        "rerun_policy":
            (
                "Do not rerun the historical control merely to "
                "recreate evidence already frozen in Experiment A."
            ),
    },

    "treatment_arm": {
        "id":
            "B1",

        "name":
            "structural_json_schema_constrained_generation",

        "definition":
            (
                "Use the same semantic model, same frozen CIR system "
                "prompt, same user intent, same temperature=0, and "
                "same downstream E3-v3 deterministic cascade, while "
                "adding provider-side structured generation using "
                "the preregistered structural decoding schema."
            ),

        "only_intended_semantic_change":
            "provider-side structural decoding constraint",

        "structural_decoding_schema":
            STRUCTURAL_DECODING_SCHEMA,

        "excluded_from_decoding_schema": {
            "service_type_enum": {
                "excluded": True,
                "reason":
                    (
                        "Unsupported/invalid service values must remain "
                        "observable to deterministic validation rather "
                        "than being prevented or silently corrected by "
                        "the decoder."
                    )
            },

            "latency_minimum": {
                "excluded": True,
                "reason":
                    (
                        "Explicit invalid source values must remain "
                        "extractable and rejectable downstream."
                    )
            },

            "bandwidth_minimum": {
                "excluded": True,
                "reason":
                    (
                        "Explicit invalid source values must remain "
                        "extractable and rejectable downstream."
                    )
            },

            "reliability_range": {
                "excluded": True,
                "reason":
                    (
                        "Range validity remains the responsibility "
                        "of the frozen deterministic validator."
                    )
            },

            "device_count_minimum": {
                "excluded": True,
                "reason":
                    (
                        "Range validity remains the responsibility "
                        "of the frozen deterministic validator."
                    )
            },
        },
    },

    "frozen_components": {
        "runner_sha256":
            EXPECTED["runner"],

        "historical_adapter_sha256":
            EXPECTED["adapter"],

        "cir_schema_sha256":
            EXPECTED["schema"],

        "validator_sha256":
            EXPECTED["validator"],

        "experiment_A_final_freeze_sha256":
            EXPECTED["experiment_A_final_freeze"],

        "cir_prompt":
            "unchanged",

        "user_intents":
            "unchanged",

        "temperature":
            0,

        "deterministic_pre_gates":
            "unchanged",

        "reconciliation":
            "unchanged",

        "preservation_checks":
            "unchanged",

        "deterministic_validator":
            "unchanged",

        "consistency_checks":
            "unchanged",

        "mapping":
            "unchanged",

        "descriptor_compilation":
            "unchanged",

        "decision_thresholds":
            "unchanged",
    },

    "implementation_policy": {
        "modify_frozen_runner":
            False,

        "modify_historical_adapter":
            False,

        "modify_frozen_schema":
            False,

        "modify_frozen_validator":
            False,

        "treatment_implementation":
            (
                "Create isolated Experiment-B treatment code. "
                "Do not patch frozen historical files."
            ),

        "provider_capability_probe":
            (
                "Before benchmark execution, perform only a synthetic "
                "non-benchmark structured-output capability probe."
            ),

        "fallback_or_substitution":
            False,

        "if_model_does_not_support_required_structured_output":
            (
                "Record treatment as provider/model capability "
                "unavailable. Do not silently substitute another "
                "model or alter the schema."
            ),
    },

    "primary_metrics": [
        "decision_accuracy",
        "VAR_valid_acceptance_rate",
        "IRR_invalid_rejection_rate",
        "containment_violations",
    ],

    "structured_generation_metrics": [
        "provider_request_success_rate",
        "parse_success_rate",
        "seven_required_keys_rate",
        "additional_property_rate",
        "structural_schema_validity_rate",
    ],

    "semantic_safety_metrics": [
        "deterministic_CIR_validation_rejection_rate",
        "source_to_CIR_preservation_failure_rate",
        "constraint_consistency_rejection_rate",
        "unsupported_or_invalid_request_rejection_rate",
        "containment_violations",
    ],

    "analysis_requirements": {
        "report_syntactic_structural_validity_separately":
            True,

        "report_semantic_validity_separately":
            True,

        "report_downstream_deterministic_rejections":
            True,

        "report_containment":
            True,

        "do_not_equate_schema_valid_json_with_semantically_valid_CIR":
            True,

        "confidence_intervals":
            "Clopper-Pearson 95% for binomial rates where applicable",

        "comparison":
            (
                "Compare treatment with frozen historical B0/A1 "
                "baseline descriptively and case-wise only where "
                "the required original case-level evidence exists."
            ),
    },

    "claim_boundary": {
        "allowed_if_supported":
            (
                "Schema-constrained decoding can improve structural "
                "output validity, but deterministic downstream checks "
                "remain necessary when structurally valid outputs can "
                "still violate source-grounding, semantic, policy, "
                "consistency, or admissibility constraints."
            ),

        "not_allowed": [
            "schema-constrained decoding universally eliminates hallucinations",
            "deterministic validation is universally necessary for every system",
            "formal proof of safety",
            "zero-risk guarantee",
            "universal model-independence",
            "formal 3GPP/O-RAN conformance",
        ],
    },

    "execution_policy": {
        "treatment_benchmark_runs":
            1,

        "rerun_after_observing_results":
            False,

        "prompt_tuning_after_results":
            False,

        "schema_tuning_after_results":
            False,

        "threshold_tuning_after_results":
            False,

        "report_unfavorable_results":
            True,
    },
}


# ======================================================================================
# 5. Prepare directories
# ======================================================================================

PREREG_DIR = (
    B_ROOT
    / "preregistration"
)

MANIFEST_DIR = (
    B_ROOT
    / "manifests"
)

DRIVE_PREREG_DIR = (
    B_DRIVE
    / "preregistration"
)

DRIVE_MANIFEST_DIR = (
    B_DRIVE
    / "manifests"
)

for path in [
    PREREG_DIR,
    MANIFEST_DIR,
    DRIVE_PREREG_DIR,
    DRIVE_MANIFEST_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )


# ======================================================================================
# 6. Refuse overwrite
# ======================================================================================

PREREG_PATH = (
    PREREG_DIR
    / "experiment_B_preregistration_v1.json"
)

DRIVE_PREREG_PATH = (
    DRIVE_PREREG_DIR
    / "experiment_B_preregistration_v1.json"
)

MANIFEST_PATH = (
    MANIFEST_DIR
    / "experiment_B_preregistration_manifest_v1.json"
)

DRIVE_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / "experiment_B_preregistration_manifest_v1.json"
)

for path in [
    PREREG_PATH,
    DRIVE_PREREG_PATH,
    MANIFEST_PATH,
    DRIVE_MANIFEST_PATH,
]:
    assert not path.exists(), (
        "STOP: preregistration artifact already exists; "
        "do not overwrite:\n"
        f"{path}"
    )


# ======================================================================================
# 7. Write preregistration
# ======================================================================================

PREREG_PATH.write_text(
    json.dumps(
        PREREG,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

PREREG_SHA = sha256(
    PREREG_PATH
)

print("\n✓ Experiment-B preregistration created")
print(PREREG_PATH)
print("SHA-256:")
print(PREREG_SHA)


# ======================================================================================
# 8. Create preregistration manifest
# ======================================================================================

MANIFEST = {
    "artifact":
        "Experiment B preregistration manifest",

    "version":
        "v1",

    "preregistration_path":
        str(PREREG_PATH),

    "preregistration_sha256":
        PREREG_SHA,

    "frozen_inputs": {
        "runner": {
            "path": str(RUNNER),
            "sha256": EXPECTED["runner"],
        },

        "benchmark": {
            "path": str(BENCHMARK),
            "sha256": EXPECTED["benchmark"],
        },

        "adapter": {
            "path": str(ADAPTER),
            "sha256": EXPECTED["adapter"],
        },

        "schema": {
            "path": str(SCHEMA),
            "sha256": EXPECTED["schema"],
        },

        "validator": {
            "path": str(VALIDATOR),
            "sha256": EXPECTED["validator"],
        },

        "experiment_A_final_freeze": {
            "path": str(A_FINAL_FREEZE),
            "sha256": EXPECTED["experiment_A_final_freeze"],
        },
    },

    "execution_state":
        "NO MODEL/API CALL — BENCHMARK NOT EXECUTED",
}

MANIFEST_PATH.write_text(
    json.dumps(
        MANIFEST,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

MANIFEST_SHA = sha256(
    MANIFEST_PATH
)

print("\n✓ Preregistration manifest created")
print(MANIFEST_PATH)
print("SHA-256:")
print(MANIFEST_SHA)


# ======================================================================================
# 9. Persist both immediately to Drive
# ======================================================================================

shutil.copy2(
    PREREG_PATH,
    DRIVE_PREREG_PATH
)

shutil.copy2(
    MANIFEST_PATH,
    DRIVE_MANIFEST_PATH
)

assert (
    sha256(DRIVE_PREREG_PATH)
    == PREREG_SHA
)

assert (
    sha256(DRIVE_MANIFEST_PATH)
    == MANIFEST_SHA
)

print("\n✓ Preregistration persisted to Drive")
print("✓ Manifest persisted to Drive")
print("✓ Drive copies verified byte-for-byte")


# ======================================================================================
# 10. Verify structural schema design
# ======================================================================================

assert list(
    STRUCTURAL_DECODING_SCHEMA["properties"].keys()
) == [
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
]

assert (
    STRUCTURAL_DECODING_SCHEMA["required"]
    == list(
        STRUCTURAL_DECODING_SCHEMA["properties"].keys()
    )
)

assert (
    STRUCTURAL_DECODING_SCHEMA["additionalProperties"]
    is False
)

for field_name, definition in (
    STRUCTURAL_DECODING_SCHEMA["properties"].items()
):

    for forbidden_keyword in [
        "minimum",
        "maximum",
        "enum",
        "pattern",
        "minLength",
        "maxLength",
    ]:
        assert forbidden_keyword not in definition, (
            f"STOP: value restriction leaked into "
            f"structural decoding schema: "
            f"{field_name}.{forbidden_keyword}"
        )

print("✓ Structural decoding schema verified")
print("✓ No semantic/range restrictions moved into decoder")


# ======================================================================================
# 11. Reverify all frozen historical files
# ======================================================================================

verify(
    "runner after preregistration",
    RUNNER,
    EXPECTED["runner"]
)

verify(
    "adapter after preregistration",
    ADAPTER,
    EXPECTED["adapter"]
)

verify(
    "schema after preregistration",
    SCHEMA,
    EXPECTED["schema"]
)

verify(
    "validator after preregistration",
    VALIDATOR,
    EXPECTED["validator"]
)


# ======================================================================================
# 12. Final summary
# ======================================================================================

print("\n" + "-" * 118)
print("PREREGISTERED EXPERIMENT-B DESIGN")
print("-" * 118)

print("Control   : B0 — historical unconstrained JSON generation")
print("Treatment : B1 — provider-side STRUCTURAL JSON-schema constrained generation")
print("Model     : Qwen/Qwen3.5-9B")
print("Benchmark : frozen 120-case E3-v3 benchmark")
print("Prompt    : unchanged")
print("Temp      : 0")
print("Pipeline  : unchanged downstream deterministic cascade")

print()
print("Structural treatment schema:")
print("  ✓ exact seven CIR keys")
print("  ✓ all seven keys required")
print("  ✓ primitive/null types constrained")
print("  ✓ additionalProperties = false")
print("  ✗ NO service_type enum")
print("  ✗ NO numeric min/max constraints")

print()
print("Reason:")
print(
    "  Preserve extraction of explicitly invalid source values so "
    "the frozen deterministic validator, rather than the decoder, "
    "retains responsibility for semantic/range rejection."
)


print("\n" + "=" * 118)
print("✓ STEP B5 COMPLETE")
print("✓ EXPERIMENT-B DESIGN PREREGISTERED")
print("✓ STRUCTURAL DECODING SCHEMA FROZEN")
print("✓ CONTROL/TREATMENT BOUNDARY FROZEN")
print("✓ METRICS FROZEN")
print("✓ CLAIM BOUNDARY FROZEN")
print("✓ PREREGISTRATION PERSISTED TO DRIVE")
print("✓ NO MODEL/API CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE ANY CAPABILITY PROBE")
print("=" * 118)

EXPERIMENT B — STEP B5: PREREGISTER SCHEMA-CONSTRAINED DECODING EXPERIMENT

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ runner verified
✓ benchmark verified
✓ adapter verified
✓ schema verified
✓ validator verified
✓ Experiment-A final freeze verified

✓ Experiment-B preregistration created
/content/deterministic-intent-translation-journal/journal_experiments/experiment_B_schema_constrained_decoding/preregistration/experiment_B_preregistration_v1.json
SHA-256:
d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5

✓ Preregistration manifest created
/content/deterministic-intent-translation-journal/journal_experiments/experiment_B_schema_constrained_decoding/manifests/experiment_B_preregistration_manifest_v1.json
SHA-256:
c21e585fa855bf3d982c9996bc793e9cc90dc15311a2

In [24]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import getpass
import time
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B — STEP B6: SINGLE NON-BENCHMARK STRUCTURED-OUTPUT CAPABILITY PROBE")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

PREREG_MANIFEST = (
    B_ROOT
    / "manifests"
    / "experiment_B_preregistration_manifest_v1.json"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

SCHEMA = (
    ROOT
    / "src"
    / "schema.py"
)

VALIDATOR = (
    ROOT
    / "src"
    / "validator.py"
)

EXPECTED = {
    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "prereg_manifest":
        "c21e585fa855bf3d982c9996bc793e9cc90dc15311a2509fbb48ef0026b5eecf",

    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",
}

MODEL = "Qwen/Qwen3.5-9B"


# ======================================================================================
# 1. Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 2. Verify preregistration + frozen components BEFORE API call
# ======================================================================================

print("\n" + "-" * 118)
print("PRE-PROBE INTEGRITY")
print("-" * 118)

verify(
    "Experiment-B preregistration",
    PREREG,
    EXPECTED["prereg"]
)

verify(
    "Experiment-B preregistration manifest",
    PREREG_MANIFEST,
    EXPECTED["prereg_manifest"]
)

verify(
    "frozen runner",
    RUNNER,
    EXPECTED["runner"]
)

verify(
    "historical adapter",
    ADAPTER,
    EXPECTED["adapter"]
)

verify(
    "frozen schema",
    SCHEMA,
    EXPECTED["schema"]
)

verify(
    "frozen validator",
    VALIDATOR,
    EXPECTED["validator"]
)


# ======================================================================================
# 3. Load EXACT preregistered structural schema
# ======================================================================================

prereg_obj = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

STRUCTURAL_SCHEMA = (
    prereg_obj[
        "treatment_arm"
    ][
        "structural_decoding_schema"
    ]
)

assert (
    STRUCTURAL_SCHEMA["additionalProperties"]
    is False
)

assert list(
    STRUCTURAL_SCHEMA["properties"].keys()
) == [
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
]

print("✓ Exact preregistered structural schema loaded")


# ======================================================================================
# 4. Use exact historical CIR system prompt
#
# Read statically from adapter via AST so adapter itself is NOT imported.
# ======================================================================================

import ast

adapter_source = ADAPTER.read_text(
    encoding="utf-8"
)

adapter_tree = ast.parse(
    adapter_source
)

CIR_SYSTEM_PROMPT = None

for node in adapter_tree.body:

    if not isinstance(
        node,
        ast.Assign
    ):
        continue

    names = [
        target.id
        for target in node.targets
        if isinstance(
            target,
            ast.Name
        )
    ]

    if "CIR_SYSTEM_PROMPT" not in names:
        continue

    CIR_SYSTEM_PROMPT = ast.literal_eval(
        node.value
    )

assert isinstance(
    CIR_SYSTEM_PROMPT,
    str
)

print("✓ Exact historical CIR system prompt loaded statically")


# ======================================================================================
# 5. Synthetic NON-BENCHMARK probe
#
# This text is intentionally synthetic and is NOT taken from the
# frozen 120-case held-out benchmark.
# ======================================================================================

SYNTHETIC_PROMPT = (
    "Synthetic capability probe only: create a network slice "
    "for eMBB with latency 17 ms and bandwidth 321 Mbps, "
    "reliability 0.987, device count 43, coverage Test-Zone-B6, "
    "with objective synthetic-capability-check."
)

print("\nSynthetic probe:")
print(SYNTHETIC_PROMPT)

print()
print("✓ Probe explicitly designated NON-BENCHMARK")


# ======================================================================================
# 6. Ensure API key exists without printing it
# ======================================================================================

if not os.environ.get(
    "TOGETHER_API_KEY"
):
    os.environ[
        "TOGETHER_API_KEY"
    ] = getpass.getpass(
        "Enter TOGETHER_API_KEY (hidden): "
    )

assert os.environ.get(
    "TOGETHER_API_KEY"
), "STOP: TOGETHER_API_KEY unavailable"

print("✓ Together API key available (not displayed)")


# ======================================================================================
# 7. Import Together SDK
# ======================================================================================

try:
    import together
    from together import Together

except Exception as exc:
    raise RuntimeError(
        "STOP: Together SDK import failed"
    ) from exc


sdk_version = getattr(
    together,
    "__version__",
    "unknown"
)

print("Together SDK version:", sdk_version)


# ======================================================================================
# 8. Refuse duplicate probe artifact
#
# This protects the single-probe preregistration.
# ======================================================================================

RAW_DIR = (
    B_ROOT
    / "results"
    / "capability_probe"
)

MANIFEST_DIR = (
    B_ROOT
    / "manifests"
)

DRIVE_RAW_DIR = (
    B_DRIVE
    / "results"
    / "capability_probe"
)

DRIVE_MANIFEST_DIR = (
    B_DRIVE
    / "manifests"
)

for directory in [
    RAW_DIR,
    MANIFEST_DIR,
    DRIVE_RAW_DIR,
    DRIVE_MANIFEST_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


PROBE_PATH = (
    RAW_DIR
    / "B6_structured_output_capability_probe_v1.json"
)

MANIFEST_PATH = (
    MANIFEST_DIR
    / "B6_structured_output_capability_probe_manifest_v1.json"
)

DRIVE_PROBE_PATH = (
    DRIVE_RAW_DIR
    / PROBE_PATH.name
)

DRIVE_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / MANIFEST_PATH.name
)

for path in [
    PROBE_PATH,
    MANIFEST_PATH,
    DRIVE_PROBE_PATH,
    DRIVE_MANIFEST_PATH,
]:

    assert not path.exists(), (
        "STOP: B6 probe artifact already exists.\n"
        "The preregistered probe must NOT be rerun.\n"
        f"Existing artifact:\n{path}"
    )


# ======================================================================================
# 9. Prepare EXACT provider-side structural response format
#
# No benchmark call occurs here.
# ======================================================================================

response_format = {
    "type": "json_schema",

    "schema": {
        "name":
            "network_slice_cir",

        "schema":
            STRUCTURAL_SCHEMA,
    },
}

print("\n" + "-" * 118)
print("PROBE CONFIGURATION")
print("-" * 118)

print("Model       :", MODEL)
print("Temperature : 0")
print("Reasoning   : disabled")
print("Response    : json_schema")
print("Schema      : exact B5 preregistered structural schema")


# ======================================================================================
# 10. SINGLE API CALL
#
# IMPORTANT:
# Exactly one provider call is attempted.
# Regardless of success/failure, its outcome is frozen.
# ======================================================================================

client = Together(
    api_key=os.environ[
        "TOGETHER_API_KEY"
    ]
)

started_utc = datetime.now(
    timezone.utc
).isoformat()

start = time.perf_counter()

api_success = False
response = None
exception_type = None
exception_message = None

try:

    response = client.chat.completions.create(
        model=MODEL,

        reasoning={
            "enabled": False
        },

        messages=[
            {
                "role": "system",
                "content": CIR_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": SYNTHETIC_PROMPT,
            },
        ],

        temperature=0,

        response_format=response_format,
    )

    api_success = True

except Exception as exc:

    exception_type = type(
        exc
    ).__name__

    exception_message = str(
        exc
    )


latency_ms = (
    time.perf_counter()
    - start
) * 1000

finished_utc = datetime.now(
    timezone.utc
).isoformat()


# ======================================================================================
# 11. Analyze response without making another call
# ======================================================================================

raw_text = None
parsed_json = None
parse_ok = False
structural_checks = {}
finish_reason = None
prompt_tokens = None
completion_tokens = None

if api_success:

    choice = response.choices[0]
    message = choice.message

    raw_text = (
        message.content
        or ""
    )

    finish_reason = getattr(
        choice,
        "finish_reason",
        None
    )

    usage = getattr(
        response,
        "usage",
        None
    )

    if usage is not None:

        prompt_tokens = getattr(
            usage,
            "prompt_tokens",
            None
        )

        completion_tokens = getattr(
            usage,
            "completion_tokens",
            None
        )

    try:
        parsed_json = json.loads(
            raw_text
        )

        parse_ok = True

    except Exception:
        parsed_json = None
        parse_ok = False


EXPECTED_KEYS = [
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
]

if isinstance(
    parsed_json,
    dict
):

    keys = list(
        parsed_json.keys()
    )

    structural_checks = {
        "is_object":
            True,

        "all_required_keys_present":
            all(
                key in parsed_json
                for key in EXPECTED_KEYS
            ),

        "additional_keys":
            sorted(
                set(parsed_json)
                - set(EXPECTED_KEYS)
            ),

        "exact_key_set":
            set(parsed_json)
            == set(EXPECTED_KEYS),

        "service_type_type_ok":
            (
                parsed_json["service_type"]
                is None
                or isinstance(
                    parsed_json["service_type"],
                    str
                )
            )
            if "service_type" in parsed_json
            else False,

        "latency_type_ok":
            (
                parsed_json["latency_ms"]
                is None
                or (
                    isinstance(
                        parsed_json["latency_ms"],
                        (int, float)
                    )
                    and not isinstance(
                        parsed_json["latency_ms"],
                        bool
                    )
                )
            )
            if "latency_ms" in parsed_json
            else False,

        "bandwidth_type_ok":
            (
                parsed_json["bandwidth_mbps"]
                is None
                or (
                    isinstance(
                        parsed_json["bandwidth_mbps"],
                        (int, float)
                    )
                    and not isinstance(
                        parsed_json["bandwidth_mbps"],
                        bool
                    )
                )
            )
            if "bandwidth_mbps" in parsed_json
            else False,

        "reliability_type_ok":
            (
                parsed_json["reliability"]
                is None
                or (
                    isinstance(
                        parsed_json["reliability"],
                        (int, float)
                    )
                    and not isinstance(
                        parsed_json["reliability"],
                        bool
                    )
                )
            )
            if "reliability" in parsed_json
            else False,

        "device_count_type_ok":
            (
                parsed_json["device_count"]
                is None
                or (
                    isinstance(
                        parsed_json["device_count"],
                        int
                    )
                    and not isinstance(
                        parsed_json["device_count"],
                        bool
                    )
                )
            )
            if "device_count" in parsed_json
            else False,

        "coverage_type_ok":
            (
                parsed_json["coverage"]
                is None
                or isinstance(
                    parsed_json["coverage"],
                    str
                )
            )
            if "coverage" in parsed_json
            else False,

        "objective_type_ok":
            (
                parsed_json["objective"]
                is None
                or isinstance(
                    parsed_json["objective"],
                    str
                )
            )
            if "objective" in parsed_json
            else False,
    }

else:

    structural_checks = {
        "is_object":
            False
    }


type_checks = [
    value
    for key, value
    in structural_checks.items()
    if key.endswith(
        "_type_ok"
    )
]

structural_schema_valid = (
    api_success
    and parse_ok
    and structural_checks.get(
        "is_object",
        False
    )
    and structural_checks.get(
        "all_required_keys_present",
        False
    )
    and structural_checks.get(
        "exact_key_set",
        False
    )
    and not structural_checks.get(
        "additional_keys",
        []
    )
    and bool(type_checks)
    and all(type_checks)
)


# ======================================================================================
# 12. Capability classification
# ======================================================================================

if (
    api_success
    and structural_schema_valid
):

    capability_status = (
        "SUPPORTED_AND_PROBE_VALID"
    )

elif api_success:

    capability_status = (
        "REQUEST_ACCEPTED_BUT_OUTPUT_FAILED_STRUCTURAL_CHECK"
    )

else:

    capability_status = (
        "PROVIDER_OR_MODEL_STRUCTURED_OUTPUT_UNAVAILABLE"
    )


# ======================================================================================
# 13. Freeze probe result
# ======================================================================================

probe_record = {
    "experiment":
        "Experiment B",

    "step":
        "B6",

    "purpose":
        "non-benchmark structured-output capability probe",

    "benchmark_case":
        False,

    "single_call_policy":
        True,

    "model":
        MODEL,

    "together_sdk_version":
        sdk_version,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "latency_ms":
        latency_ms,

    "synthetic_prompt":
        SYNTHETIC_PROMPT,

    "preregistration_sha256":
        EXPECTED["prereg"],

    "structural_schema":
        STRUCTURAL_SCHEMA,

    "response_format":
        response_format,

    "api_success":
        api_success,

    "exception_type":
        exception_type,

    "exception_message":
        exception_message,

    "raw_text":
        raw_text,

    "parsed_json":
        parsed_json,

    "parse_ok":
        parse_ok,

    "structural_checks":
        structural_checks,

    "structural_schema_valid":
        structural_schema_valid,

    "finish_reason":
        finish_reason,

    "prompt_tokens":
        prompt_tokens,

    "completion_tokens":
        completion_tokens,

    "capability_status":
        capability_status,
}


PROBE_PATH.write_text(
    json.dumps(
        probe_record,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

PROBE_SHA = sha256(
    PROBE_PATH
)


# ======================================================================================
# 14. Manifest
# ======================================================================================

manifest = {
    "artifact":
        "Experiment B B6 capability-probe manifest",

    "probe_path":
        str(PROBE_PATH),

    "probe_sha256":
        PROBE_SHA,

    "preregistration_sha256":
        EXPECTED["prereg"],

    "model":
        MODEL,

    "api_calls_attempted":
        1,

    "benchmark_calls":
        0,

    "capability_status":
        capability_status,

    "historical_files_modified":
        False,

    "rerun_allowed":
        False,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

MANIFEST_SHA = sha256(
    MANIFEST_PATH
)


# ======================================================================================
# 15. Persist immediately to Drive
# ======================================================================================

shutil.copy2(
    PROBE_PATH,
    DRIVE_PROBE_PATH
)

shutil.copy2(
    MANIFEST_PATH,
    DRIVE_MANIFEST_PATH
)

assert (
    sha256(DRIVE_PROBE_PATH)
    == PROBE_SHA
)

assert (
    sha256(DRIVE_MANIFEST_PATH)
    == MANIFEST_SHA
)


# ======================================================================================
# 16. Verify historical files unchanged
# ======================================================================================

verify(
    "runner after probe",
    RUNNER,
    EXPECTED["runner"]
)

verify(
    "adapter after probe",
    ADAPTER,
    EXPECTED["adapter"]
)

verify(
    "schema after probe",
    SCHEMA,
    EXPECTED["schema"]
)

verify(
    "validator after probe",
    VALIDATOR,
    EXPECTED["validator"]
)


# ======================================================================================
# 17. Report
# ======================================================================================

print("\n" + "-" * 118)
print("B6 CAPABILITY-PROBE RESULT")
print("-" * 118)

print("API success             :", api_success)
print("Parse OK                :", parse_ok)
print("Structural schema valid :", structural_schema_valid)
print("Capability status       :", capability_status)
print("Latency ms              :", round(latency_ms, 2))

if exception_type:
    print("Exception type          :", exception_type)
    print("Exception message       :", exception_message)

if raw_text is not None:
    print()
    print("Raw model output:")
    print(raw_text)

if parsed_json is not None:
    print()
    print("Parsed JSON:")
    print(
        json.dumps(
            parsed_json,
            indent=2,
            ensure_ascii=False
        )
    )

print()
print("Probe artifact:")
print(PROBE_PATH)
print("SHA-256:")
print(PROBE_SHA)

print()
print("Probe manifest:")
print(MANIFEST_PATH)
print("SHA-256:")
print(MANIFEST_SHA)

print()
print("✓ Probe + manifest persisted to Drive")
print("✓ Drive copies verified byte-for-byte")


print("\n" + "=" * 118)
print("✓ STEP B6 COMPLETE")
print("✓ EXACTLY ONE SYNTHETIC NON-BENCHMARK API CALL ATTEMPTED")
print("✓ PROBE OUTCOME FROZEN REGARDLESS OF SUCCESS/FAILURE")
print("✓ NO BENCHMARK CASE EXECUTED")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ DO NOT RERUN B6")
print("✓ SEND THIS OUTPUT BEFORE ANY B1 TREATMENT IMPLEMENTATION")
print("=" * 118)

EXPERIMENT B — STEP B6: SINGLE NON-BENCHMARK STRUCTURED-OUTPUT CAPABILITY PROBE

----------------------------------------------------------------------------------------------------------------------
PRE-PROBE INTEGRITY
----------------------------------------------------------------------------------------------------------------------
✓ Experiment-B preregistration verified
✓ Experiment-B preregistration manifest verified
✓ frozen runner verified
✓ historical adapter verified
✓ frozen schema verified
✓ frozen validator verified


AssertionError: 

In [25]:
from pathlib import Path
import hashlib
import json

print("=" * 118)
print("EXPERIMENT B — STEP B6A: DIAGNOSE PREREGISTERED STRUCTURAL-SCHEMA ORDER")
print("=" * 118)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

PREREG = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

EXPECTED_PREREG_SHA = (
    "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5"
)

EXPECTED_FIELDS = [
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
]


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ----------------------------------------------------------------------
# 1. Verify preregistration remains frozen
# ----------------------------------------------------------------------

assert PREREG.exists(), (
    f"STOP: preregistration missing:\n{PREREG}"
)

actual_sha = sha256(PREREG)

assert actual_sha == EXPECTED_PREREG_SHA, (
    "STOP: preregistration hash mismatch\n"
    f"Expected: {EXPECTED_PREREG_SHA}\n"
    f"Actual:   {actual_sha}"
)

print("✓ Experiment-B preregistration verified")
print("SHA-256:", actual_sha)


# ----------------------------------------------------------------------
# 2. Load preregistered structural schema
# ----------------------------------------------------------------------

obj = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

schema = (
    obj["treatment_arm"]
       ["structural_decoding_schema"]
)

actual_fields = list(
    schema["properties"].keys()
)


# ----------------------------------------------------------------------
# 3. Show ordering explicitly
# ----------------------------------------------------------------------

print("\n" + "-" * 118)
print("FIELD ORDER DIAGNOSTIC")
print("-" * 118)

print("Expected insertion order:")
for i, field in enumerate(
    EXPECTED_FIELDS,
    start=1
):
    print(f"  {i}. {field}")

print("\nStored JSON order:")
for i, field in enumerate(
    actual_fields,
    start=1
):
    print(f"  {i}. {field}")


# ----------------------------------------------------------------------
# 4. Compare order vs semantic field set
# ----------------------------------------------------------------------

order_match = (
    actual_fields
    == EXPECTED_FIELDS
)

set_match = (
    set(actual_fields)
    == set(EXPECTED_FIELDS)
)

missing = sorted(
    set(EXPECTED_FIELDS)
    - set(actual_fields)
)

extra = sorted(
    set(actual_fields)
    - set(EXPECTED_FIELDS)
)

print("\n" + "-" * 118)
print("COMPARISON")
print("-" * 118)

print("Exact order match :", order_match)
print("Exact field-set   :", set_match)
print("Missing fields    :", missing)
print("Extra fields      :", extra)

assert set_match, (
    "STOP: this is NOT merely an ordering issue. "
    "The preregistered schema field set differs."
)

assert not missing
assert not extra


# ----------------------------------------------------------------------
# 5. Verify all seven remain required
# ----------------------------------------------------------------------

required = schema.get(
    "required",
    []
)

print("\nRequired fields:")
print(required)

assert set(required) == set(EXPECTED_FIELDS), (
    "STOP: required-field set mismatch"
)

assert len(required) == 7, (
    "STOP: expected exactly seven required fields"
)


# ----------------------------------------------------------------------
# 6. Verify additionalProperties
# ----------------------------------------------------------------------

assert (
    schema.get("additionalProperties")
    is False
), "STOP: additionalProperties is not false"

print("✓ additionalProperties = false")


# ----------------------------------------------------------------------
# 7. Verify no semantic restrictions leaked into treatment schema
# ----------------------------------------------------------------------

FORBIDDEN = {
    "minimum",
    "maximum",
    "enum",
    "pattern",
    "minLength",
    "maxLength",
}

violations = []

for field, definition in (
    schema["properties"].items()
):

    leaked = sorted(
        FORBIDDEN.intersection(
            definition.keys()
        )
    )

    if leaked:
        violations.append(
            (field, leaked)
        )

print("\nSemantic/range restriction leakage:")
print(
    violations
    if violations
    else "NONE"
)

assert not violations, (
    "STOP: semantic/range restrictions found "
    "inside structural treatment schema"
)


# ----------------------------------------------------------------------
# 8. Explain probable cause
# ----------------------------------------------------------------------

alphabetical_fields = sorted(
    EXPECTED_FIELDS
)

print("\n" + "-" * 118)
print("ORDERING CAUSE CHECK")
print("-" * 118)

print(
    "Stored order equals alphabetical order:",
    actual_fields == alphabetical_fields
)

if actual_fields == alphabetical_fields:
    print(
        "✓ Consistent with B5 json.dumps(sort_keys=True)"
    )


# ----------------------------------------------------------------------
# 9. Final integrity check
# ----------------------------------------------------------------------

assert sha256(PREREG) == EXPECTED_PREREG_SHA

print("\n" + "=" * 118)
print("✓ STEP B6A COMPLETE")
print("✓ PREREGISTRATION HASH UNCHANGED")
print("✓ SEVEN-FIELD SET VERIFIED")
print("✓ REQUIRED-FIELD SET VERIFIED")
print("✓ additionalProperties VERIFIED")
print("✓ NO SEMANTIC/RANGE RESTRICTIONS DETECTED")
print("✓ NO API CALL")
print("✓ NO BENCHMARK CALL")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE RETRYING THE B6 CAPABILITY PROBE")
print("=" * 118)

EXPERIMENT B — STEP B6A: DIAGNOSE PREREGISTERED STRUCTURAL-SCHEMA ORDER
✓ Experiment-B preregistration verified
SHA-256: d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5

----------------------------------------------------------------------------------------------------------------------
FIELD ORDER DIAGNOSTIC
----------------------------------------------------------------------------------------------------------------------
Expected insertion order:
  1. service_type
  2. latency_ms
  3. bandwidth_mbps
  4. reliability
  5. device_count
  6. coverage
  7. objective

Stored JSON order:
  1. bandwidth_mbps
  2. coverage
  3. device_count
  4. latency_ms
  5. objective
  6. reliability
  7. service_type

----------------------------------------------------------------------------------------------------------------------
COMPARISON
----------------------------------------------------------------------------------------------------------------------
Exact order match : 

In [26]:
assert list(
    STRUCTURAL_SCHEMA["properties"].keys()
) == [
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
]

AssertionError: 

In [27]:
EXPECTED_FIELDS = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert set(
    STRUCTURAL_SCHEMA["properties"].keys()
) == EXPECTED_FIELDS, (
    "STOP: preregistered structural schema field-set mismatch"
)

assert set(
    STRUCTURAL_SCHEMA["required"]
) == EXPECTED_FIELDS, (
    "STOP: preregistered required-field set mismatch"
)

assert len(
    STRUCTURAL_SCHEMA["properties"]
) == 7

assert len(
    STRUCTURAL_SCHEMA["required"]
) == 7

assert (
    STRUCTURAL_SCHEMA["additionalProperties"]
    is False
)

print("✓ Exact preregistered structural schema loaded")
print("✓ Seven-field SET verified; JSON object key order intentionally ignored")

✓ Exact preregistered structural schema loaded
✓ Seven-field SET verified; JSON object key order intentionally ignored


In [28]:
# ==================================================================================================
# EXPERIMENT B — STEP B6B
# EXECUTE THE SINGLE PREREGISTERED NON-BENCHMARK STRUCTURED-OUTPUT CAPABILITY PROBE
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import os
import getpass
import time
import shutil
import ast
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B — STEP B6B: EXECUTE SINGLE STRUCTURED-OUTPUT CAPABILITY PROBE")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"

EXPECTED_PREREG_SHA = (
    "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5"
)

MODEL = "Qwen/Qwen3.5-9B"

EXPECTED_FIELDS = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}


# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# --------------------------------------------------------------------------------------------------
# Verify preregistration
# --------------------------------------------------------------------------------------------------

assert PREREG.exists(), f"STOP: missing preregistration: {PREREG}"

assert sha256(PREREG) == EXPECTED_PREREG_SHA, (
    "STOP: Experiment-B preregistration hash mismatch"
)

print("✓ Experiment-B preregistration verified")


# --------------------------------------------------------------------------------------------------
# Load preregistered schema
# --------------------------------------------------------------------------------------------------

prereg = json.loads(PREREG.read_text(encoding="utf-8"))

STRUCTURAL_SCHEMA = (
    prereg["treatment_arm"]["structural_decoding_schema"]
)

assert set(STRUCTURAL_SCHEMA["properties"]) == EXPECTED_FIELDS
assert set(STRUCTURAL_SCHEMA["required"]) == EXPECTED_FIELDS
assert len(STRUCTURAL_SCHEMA["properties"]) == 7
assert len(STRUCTURAL_SCHEMA["required"]) == 7
assert STRUCTURAL_SCHEMA["additionalProperties"] is False

print("✓ Preregistered structural schema verified")


# --------------------------------------------------------------------------------------------------
# Load historical CIR prompt statically
# --------------------------------------------------------------------------------------------------

adapter_source = ADAPTER.read_text(encoding="utf-8")
tree = ast.parse(adapter_source)

CIR_SYSTEM_PROMPT = None

for node in tree.body:
    if isinstance(node, ast.Assign):
        names = [
            t.id
            for t in node.targets
            if isinstance(t, ast.Name)
        ]

        if "CIR_SYSTEM_PROMPT" in names:
            CIR_SYSTEM_PROMPT = ast.literal_eval(node.value)
            break

assert isinstance(CIR_SYSTEM_PROMPT, str)

print("✓ Historical CIR system prompt loaded")


# --------------------------------------------------------------------------------------------------
# Synthetic NON-BENCHMARK intent
# --------------------------------------------------------------------------------------------------

SYNTHETIC_PROMPT = (
    "Synthetic capability probe only: create a network slice "
    "for eMBB with latency 17 ms and bandwidth 321 Mbps, "
    "reliability 0.987, device count 43, coverage Test-Zone-B6, "
    "with objective synthetic-capability-check."
)

print("✓ Synthetic NON-BENCHMARK prompt prepared")


# --------------------------------------------------------------------------------------------------
# Artifact protection
# --------------------------------------------------------------------------------------------------

RAW_DIR = B_ROOT / "results" / "capability_probe"
MANIFEST_DIR = B_ROOT / "manifests"

DRIVE_RAW_DIR = B_DRIVE / "results" / "capability_probe"
DRIVE_MANIFEST_DIR = B_DRIVE / "manifests"

for directory in [
    RAW_DIR,
    MANIFEST_DIR,
    DRIVE_RAW_DIR,
    DRIVE_MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


PROBE_PATH = (
    RAW_DIR
    / "B6_structured_output_capability_probe_v1.json"
)

MANIFEST_PATH = (
    MANIFEST_DIR
    / "B6_structured_output_capability_probe_manifest_v1.json"
)

DRIVE_PROBE_PATH = DRIVE_RAW_DIR / PROBE_PATH.name
DRIVE_MANIFEST_PATH = DRIVE_MANIFEST_DIR / MANIFEST_PATH.name


# Prevent a second provider call.
for path in [
    PROBE_PATH,
    MANIFEST_PATH,
    DRIVE_PROBE_PATH,
    DRIVE_MANIFEST_PATH,
]:
    assert not path.exists(), (
        "STOP: B6 capability-probe artifact already exists.\n"
        "DO NOT perform another API call.\n"
        f"Existing artifact: {path}"
    )

print("✓ No previous B6 API-probe artifact detected")


# --------------------------------------------------------------------------------------------------
# API key
# --------------------------------------------------------------------------------------------------

if not os.environ.get("TOGETHER_API_KEY"):
    os.environ["TOGETHER_API_KEY"] = getpass.getpass(
        "Enter TOGETHER_API_KEY (hidden): "
    )

assert os.environ.get("TOGETHER_API_KEY")

print("✓ API key available (not displayed)")


# --------------------------------------------------------------------------------------------------
# Together SDK
# --------------------------------------------------------------------------------------------------

import together
from together import Together

sdk_version = getattr(together, "__version__", "unknown")

print("Together SDK version:", sdk_version)


# --------------------------------------------------------------------------------------------------
# Exact preregistered provider response format
# --------------------------------------------------------------------------------------------------

response_format = {
    "type": "json_schema",
    "schema": {
        "name": "network_slice_cir",
        "schema": STRUCTURAL_SCHEMA,
    },
}

print()
print("Model       :", MODEL)
print("Temperature : 0")
print("Reasoning   : disabled")
print("Constraint  : preregistered structural JSON schema")
print()


# --------------------------------------------------------------------------------------------------
# EXACTLY ONE API CALL
# --------------------------------------------------------------------------------------------------

print(">>> ATTEMPTING THE SINGLE B6 PROVIDER CALL ...")

client = Together(api_key=os.environ["TOGETHER_API_KEY"])

started_utc = datetime.now(timezone.utc).isoformat()
start = time.perf_counter()

api_success = False
response = None
exception_type = None
exception_message = None

try:
    response = client.chat.completions.create(
        model=MODEL,
        reasoning={"enabled": False},
        messages=[
            {
                "role": "system",
                "content": CIR_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": SYNTHETIC_PROMPT,
            },
        ],
        temperature=0,
        response_format=response_format,
    )

    api_success = True

except Exception as exc:
    exception_type = type(exc).__name__
    exception_message = str(exc)

latency_ms = (time.perf_counter() - start) * 1000
finished_utc = datetime.now(timezone.utc).isoformat()

print(">>> SINGLE PROVIDER CALL ATTEMPT FINISHED")


# --------------------------------------------------------------------------------------------------
# Analyze response — NO additional API calls
# --------------------------------------------------------------------------------------------------

raw_text = None
parsed_json = None
parse_ok = False
finish_reason = None
prompt_tokens = None
completion_tokens = None

if api_success:
    choice = response.choices[0]
    message = choice.message

    raw_text = message.content or ""

    finish_reason = getattr(
        choice,
        "finish_reason",
        None
    )

    usage = getattr(response, "usage", None)

    if usage:
        prompt_tokens = getattr(
            usage,
            "prompt_tokens",
            None
        )

        completion_tokens = getattr(
            usage,
            "completion_tokens",
            None
        )

    try:
        parsed_json = json.loads(raw_text)
        parse_ok = True
    except (json.JSONDecodeError, TypeError):
        parse_ok = False


# --------------------------------------------------------------------------------------------------
# Structural validation
# --------------------------------------------------------------------------------------------------

structural_checks = {
    "is_object": isinstance(parsed_json, dict),
    "all_required_keys_present": False,
    "exact_key_set": False,
    "additional_keys": [],
}

if isinstance(parsed_json, dict):

    actual_keys = set(parsed_json)

    structural_checks[
        "all_required_keys_present"
    ] = EXPECTED_FIELDS.issubset(actual_keys)

    structural_checks[
        "exact_key_set"
    ] = actual_keys == EXPECTED_FIELDS

    structural_checks[
        "additional_keys"
    ] = sorted(actual_keys - EXPECTED_FIELDS)


    def number_or_null(value):
        return (
            value is None
            or (
                isinstance(value, (int, float))
                and not isinstance(value, bool)
            )
        )


    type_checks = {
        "service_type":
            parsed_json.get("service_type") is None
            or isinstance(parsed_json.get("service_type"), str),

        "latency_ms":
            number_or_null(parsed_json.get("latency_ms")),

        "bandwidth_mbps":
            number_or_null(parsed_json.get("bandwidth_mbps")),

        "reliability":
            number_or_null(parsed_json.get("reliability")),

        "device_count":
            parsed_json.get("device_count") is None
            or (
                isinstance(parsed_json.get("device_count"), int)
                and not isinstance(parsed_json.get("device_count"), bool)
            ),

        "coverage":
            parsed_json.get("coverage") is None
            or isinstance(parsed_json.get("coverage"), str),

        "objective":
            parsed_json.get("objective") is None
            or isinstance(parsed_json.get("objective"), str),
    }

else:
    type_checks = {}


structural_schema_valid = (
    api_success
    and parse_ok
    and structural_checks["is_object"]
    and structural_checks["all_required_keys_present"]
    and structural_checks["exact_key_set"]
    and not structural_checks["additional_keys"]
    and len(type_checks) == 7
    and all(type_checks.values())
)


# --------------------------------------------------------------------------------------------------
# Classification
# --------------------------------------------------------------------------------------------------

if api_success and structural_schema_valid:
    capability_status = "SUPPORTED_AND_PROBE_VALID"

elif api_success:
    capability_status = (
        "REQUEST_ACCEPTED_BUT_OUTPUT_FAILED_STRUCTURAL_CHECK"
    )

else:
    capability_status = (
        "PROVIDER_OR_MODEL_STRUCTURED_OUTPUT_UNAVAILABLE"
    )


# --------------------------------------------------------------------------------------------------
# Freeze result
# --------------------------------------------------------------------------------------------------

record = {
    "experiment": "Experiment B",
    "step": "B6",
    "purpose":
        "single non-benchmark structured-output capability probe",

    "benchmark_case": False,
    "provider_calls_attempted": 1,

    "model": MODEL,
    "together_sdk_version": sdk_version,

    "started_utc": started_utc,
    "finished_utc": finished_utc,
    "latency_ms": latency_ms,

    "synthetic_prompt": SYNTHETIC_PROMPT,

    "preregistration_sha256": EXPECTED_PREREG_SHA,

    "response_format": response_format,

    "api_success": api_success,

    "exception_type": exception_type,
    "exception_message": exception_message,

    "raw_text": raw_text,
    "parsed_json": parsed_json,
    "parse_ok": parse_ok,

    "structural_checks": structural_checks,
    "type_checks": type_checks,

    "structural_schema_valid":
        structural_schema_valid,

    "finish_reason": finish_reason,

    "prompt_tokens": prompt_tokens,
    "completion_tokens": completion_tokens,

    "capability_status": capability_status,

    "rerun_allowed": False,
}


PROBE_PATH.write_text(
    json.dumps(
        record,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    ) + "\n",
    encoding="utf-8",
)

PROBE_SHA = sha256(PROBE_PATH)


# --------------------------------------------------------------------------------------------------
# Manifest
# --------------------------------------------------------------------------------------------------

manifest = {
    "artifact":
        "Experiment B B6 capability probe",

    "probe_sha256":
        PROBE_SHA,

    "preregistration_sha256":
        EXPECTED_PREREG_SHA,

    "provider_calls_attempted": 1,
    "benchmark_calls": 0,

    "model": MODEL,

    "capability_status":
        capability_status,

    "rerun_allowed": False,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

MANIFEST_SHA = sha256(MANIFEST_PATH)


# --------------------------------------------------------------------------------------------------
# Persist immediately
# --------------------------------------------------------------------------------------------------

shutil.copy2(PROBE_PATH, DRIVE_PROBE_PATH)
shutil.copy2(MANIFEST_PATH, DRIVE_MANIFEST_PATH)

assert sha256(DRIVE_PROBE_PATH) == PROBE_SHA
assert sha256(DRIVE_MANIFEST_PATH) == MANIFEST_SHA


# --------------------------------------------------------------------------------------------------
# Final result
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 118)
print("B6 CAPABILITY-PROBE RESULT")
print("=" * 118)

print("API success             :", api_success)
print("Parse OK                :", parse_ok)
print("Structural schema valid :", structural_schema_valid)
print("Capability status       :", capability_status)
print("Latency ms              :", round(latency_ms, 2))

if exception_type:
    print("Exception type          :", exception_type)
    print("Exception message       :", exception_message)

if raw_text is not None:
    print("\nRaw output:")
    print(raw_text)

if parsed_json is not None:
    print("\nParsed JSON:")
    print(
        json.dumps(
            parsed_json,
            indent=2,
            ensure_ascii=False
        )
    )

print("\nProbe SHA-256:")
print(PROBE_SHA)

print("\nManifest SHA-256:")
print(MANIFEST_SHA)

print()
print("✓ Probe persisted to Drive")
print("✓ Manifest persisted to Drive")
print("✓ ZERO benchmark cases used")
print("✓ EXACTLY ONE provider call attempted")

print("\n" + "=" * 118)
print("✓ STEP B6 COMPLETE")
print("✓ DO NOT RERUN B6")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE STEP B7")
print("=" * 118)

EXPERIMENT B — STEP B6B: EXECUTE SINGLE STRUCTURED-OUTPUT CAPABILITY PROBE
✓ Experiment-B preregistration verified
✓ Preregistered structural schema verified
✓ Historical CIR system prompt loaded
✓ Synthetic NON-BENCHMARK prompt prepared
✓ No previous B6 API-probe artifact detected
✓ API key available (not displayed)
Together SDK version: 2.39.0

Model       : Qwen/Qwen3.5-9B
Temperature : 0
Reasoning   : disabled
Constraint  : preregistered structural JSON schema

>>> ATTEMPTING THE SINGLE B6 PROVIDER CALL ...
>>> SINGLE PROVIDER CALL ATTEMPT FINISHED

B6 CAPABILITY-PROBE RESULT
API success             : True
Parse OK                : True
Structural schema valid : True
Capability status       : SUPPORTED_AND_PROBE_VALID
Latency ms              : 4682.4

Raw output:
{
  "service_type": "eMBB",
  "latency_ms": 17,
  "bandwidth_mbps": 321,
  "reliability": 0.987,
  "device_count": 43,
  "coverage": "Test-Zone-B6",
  "objective": "synthetic-capability-check"
}

Parsed JSON:
{
  "service_

In [29]:
from pathlib import Path
import ast
import hashlib

print("=" * 118)
print("EXPERIMENT B — STEP B7A: EXTRACT FROZEN TREATMENT INTEGRATION POINTS")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

PREREG = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

B6_PROBE = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
    / "results"
    / "capability_probe"
    / "B6_structured_output_capability_probe_v1.json"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "b6_probe":
        "fc509b6ca254882935ac30c32333ac510d465024a32bf9cbf48ed92239a36392",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), f"STOP: missing {label}: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

verify("runner", RUNNER, EXPECTED["runner"])
verify("adapter", ADAPTER, EXPECTED["adapter"])
verify("Experiment-B preregistration", PREREG, EXPECTED["prereg"])
verify("B6 capability probe", B6_PROBE, EXPECTED["b6_probe"])


# ==================================================================================================
# AST helpers
# ==================================================================================================

def load_tree(path):
    source = path.read_text(encoding="utf-8")
    return source, ast.parse(source)


def get_function_source(source, tree, name):
    matches = [
        node
        for node in ast.walk(tree)
        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        )
        and node.name == name
    ]

    assert len(matches) == 1, (
        f"STOP: expected exactly one function named {name}; "
        f"found {len(matches)}"
    )

    node = matches[0]

    lines = source.splitlines()

    return (
        node.lineno,
        node.end_lineno,
        "\n".join(
            lines[node.lineno - 1:node.end_lineno]
        )
    )


runner_source, runner_tree = load_tree(RUNNER)
adapter_source, adapter_tree = load_tree(ADAPTER)


# ==================================================================================================
# 1. Show relevant imports
# ==================================================================================================

print("\n" + "-" * 118)
print("RUNNER IMPORTS INVOLVING LLM / ADAPTER")
print("-" * 118)

runner_lines = runner_source.splitlines()

for node in runner_tree.body:

    if isinstance(node, (ast.Import, ast.ImportFrom)):

        text = "\n".join(
            runner_lines[
                node.lineno - 1:
                node.end_lineno
            ]
        )

        lower = text.lower()

        if (
            "llm" in lower
            or "together" in lower
            or "call_json" in lower
        ):
            print(
                f"\nLines {node.lineno}-{node.end_lineno}:"
            )
            print(text)


# ==================================================================================================
# 2. Exact runner functions
# ==================================================================================================

TARGET_RUNNER_FUNCTIONS = [
    "parse_llm_result",
    "call_cir",
]

print("\n" + "-" * 118)
print("EXACT RUNNER FUNCTIONS")
print("-" * 118)

for name in TARGET_RUNNER_FUNCTIONS:

    start, end, text = get_function_source(
        runner_source,
        runner_tree,
        name
    )

    print(
        f"\n>>> {name} — lines {start}-{end}"
    )
    print(text)


# ==================================================================================================
# 3. Find every call to call_cir()
# ==================================================================================================

print("\n" + "-" * 118)
print("ALL call_cir() CALL SITES")
print("-" * 118)

call_sites = []

for node in ast.walk(runner_tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if (
        isinstance(func, ast.Name)
        and func.id == "call_cir"
    ):
        call_sites.append(node)


assert call_sites, "STOP: no call_cir() call sites found"

for i, node in enumerate(call_sites, start=1):

    start = max(1, node.lineno - 12)
    end = min(
        len(runner_lines),
        (node.end_lineno or node.lineno) + 18
    )

    print(
        f"\n>>> call_cir site #{i} "
        f"around runner lines {start}-{end}"
    )

    print(
        "\n".join(
            runner_lines[start - 1:end]
        )
    )


# ==================================================================================================
# 4. Find every parse_llm_result() call site
# ==================================================================================================

print("\n" + "-" * 118)
print("ALL parse_llm_result() CALL SITES")
print("-" * 118)

parse_sites = []

for node in ast.walk(runner_tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if (
        isinstance(func, ast.Name)
        and func.id == "parse_llm_result"
    ):
        parse_sites.append(node)


assert parse_sites, (
    "STOP: no parse_llm_result() call sites found"
)

for i, node in enumerate(parse_sites, start=1):

    start = max(1, node.lineno - 12)
    end = min(
        len(runner_lines),
        (node.end_lineno or node.lineno) + 18
    )

    print(
        f"\n>>> parse_llm_result site #{i} "
        f"around runner lines {start}-{end}"
    )

    print(
        "\n".join(
            runner_lines[start - 1:end]
        )
    )


# ==================================================================================================
# 5. Exact historical adapter call_json()
# ==================================================================================================

print("\n" + "-" * 118)
print("EXACT HISTORICAL ADAPTER call_json()")
print("-" * 118)

start, end, text = get_function_source(
    adapter_source,
    adapter_tree,
    "call_json"
)

print(
    f"\n>>> call_json — adapter lines {start}-{end}"
)
print(text)


# ==================================================================================================
# 6. Locate main benchmark-processing function(s)
# ==================================================================================================

print("\n" + "-" * 118)
print("RUNNER FUNCTIONS CONTAINING call_cir()")
print("-" * 118)

parents = {}

for parent in ast.walk(runner_tree):
    for child in ast.iter_child_nodes(parent):
        parents[child] = parent


def enclosing_function(node):
    current = node

    while current in parents:
        current = parents[current]

        if isinstance(
            current,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ):
            return current

    return None


enclosing = []

for call in call_sites:
    fn = enclosing_function(call)

    if fn is not None:
        enclosing.append(fn)


unique = {}

for fn in enclosing:
    unique[
        (fn.name, fn.lineno, fn.end_lineno)
    ] = fn


for (
    name,
    start,
    end
), fn in sorted(unique.items()):

    print(
        f"\n{name}: lines {start}-{end}"
    )

    # Print complete function because this is the exact
    # integration surface we need for B7.
    print(
        "\n".join(
            runner_lines[start - 1:end]
        )
    )


# ==================================================================================================
# 7. Search for monkey-patch-sensitive names
# ==================================================================================================

print("\n" + "-" * 118)
print("NAME-BINDING CHECK")
print("-" * 118)

names = [
    "call_json",
    "call_cir",
    "parse_llm_result",
]

for target in names:

    occurrences = []

    for i, line in enumerate(
        runner_lines,
        start=1
    ):
        if target in line:
            occurrences.append(
                (i, line.rstrip())
            )

    print(f"\n{target}:")

    for lineno, line in occurrences:
        print(
            f"  {lineno}: {line}"
        )


# ==================================================================================================
# 8. Final integrity verification
# ==================================================================================================

print("\n" + "-" * 118)
print("POST-INSPECTION INTEGRITY")
print("-" * 118)

verify(
    "runner after inspection",
    RUNNER,
    EXPECTED["runner"]
)

verify(
    "adapter after inspection",
    ADAPTER,
    EXPECTED["adapter"]
)

verify(
    "Experiment-B preregistration after inspection",
    PREREG,
    EXPECTED["prereg"]
)

verify(
    "B6 capability probe after inspection",
    B6_PROBE,
    EXPECTED["b6_probe"]
)


print("\n" + "=" * 118)
print("✓ STEP B7A COMPLETE")
print("✓ FROZEN INTEGRATION POINTS EXTRACTED")
print("✓ NO MODEL/API CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE B7B HARNESS CONSTRUCTION")
print("=" * 118)

EXPERIMENT B — STEP B7A: EXTRACT FROZEN TREATMENT INTEGRATION POINTS

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ runner verified
✓ adapter verified
✓ Experiment-B preregistration verified
✓ B6 capability probe verified

----------------------------------------------------------------------------------------------------------------------
RUNNER IMPORTS INVOLVING LLM / ADAPTER
----------------------------------------------------------------------------------------------------------------------

Lines 55-55:
from src.llm.together_adapter import call_json

----------------------------------------------------------------------------------------------------------------------
EXACT RUNNER FUNCTIONS
------------------------------------------------------------------------------------

In [30]:
from pathlib import Path
import hashlib
import json
import ast
import py_compile
import shutil

print("=" * 118)
print("EXPERIMENT B — STEP B7B: BUILD ISOLATED SCHEMA-CONSTRAINED TREATMENT ADAPTER")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"
SCHEMA = ROOT / "src" / "schema.py"
VALIDATOR = ROOT / "src" / "validator.py"

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

B6_PROBE = (
    B_ROOT
    / "results"
    / "capability_probe"
    / "B6_structured_output_capability_probe_v1.json"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",

    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "b6_probe":
        "fc509b6ca254882935ac30c32333ac510d465024a32bf9cbf48ed92239a36392",
}


# ==================================================================================================
# Helpers
# ==================================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), f"STOP: missing {label}: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ==================================================================================================
# 1. Frozen provenance
# ==================================================================================================

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

verify("runner", RUNNER, EXPECTED["runner"])
verify("historical adapter", ADAPTER, EXPECTED["adapter"])
verify("schema", SCHEMA, EXPECTED["schema"])
verify("validator", VALIDATOR, EXPECTED["validator"])
verify("Experiment-B preregistration", PREREG, EXPECTED["prereg"])
verify("B6 capability probe", B6_PROBE, EXPECTED["b6_probe"])


# ==================================================================================================
# 2. Load frozen preregistered structural schema
# ==================================================================================================

prereg_obj = json.loads(
    PREREG.read_text(encoding="utf-8")
)

STRUCTURAL_SCHEMA = (
    prereg_obj["treatment_arm"]["structural_decoding_schema"]
)

EXPECTED_FIELDS = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert set(STRUCTURAL_SCHEMA["properties"]) == EXPECTED_FIELDS
assert set(STRUCTURAL_SCHEMA["required"]) == EXPECTED_FIELDS
assert len(STRUCTURAL_SCHEMA["properties"]) == 7
assert len(STRUCTURAL_SCHEMA["required"]) == 7
assert STRUCTURAL_SCHEMA["additionalProperties"] is False

FORBIDDEN = {
    "enum",
    "minimum",
    "maximum",
    "pattern",
    "minLength",
    "maxLength",
}

for field_name, field_schema in STRUCTURAL_SCHEMA["properties"].items():
    leaked = FORBIDDEN.intersection(field_schema)

    assert not leaked, (
        f"STOP: semantic/range constraint leaked into "
        f"{field_name}: {sorted(leaked)}"
    )

print("✓ Frozen structural treatment schema verified")
print("✓ No semantic/range restrictions in decoding schema")


# ==================================================================================================
# 3. Extract historical CIR_SYSTEM_PROMPT exactly
# ==================================================================================================

adapter_source = ADAPTER.read_text(encoding="utf-8")
adapter_tree = ast.parse(adapter_source)

CIR_SYSTEM_PROMPT = None

for node in adapter_tree.body:

    if not isinstance(node, ast.Assign):
        continue

    target_names = [
        target.id
        for target in node.targets
        if isinstance(target, ast.Name)
    ]

    if "CIR_SYSTEM_PROMPT" in target_names:
        CIR_SYSTEM_PROMPT = ast.literal_eval(node.value)
        break


assert isinstance(CIR_SYSTEM_PROMPT, str)

print("✓ Historical CIR_SYSTEM_PROMPT extracted exactly")


# ==================================================================================================
# 4. Create isolated treatment directory
# ==================================================================================================

TREATMENT_DIR = B_ROOT / "implementation"

DRIVE_TREATMENT_DIR = B_DRIVE / "implementation"

TREATMENT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_TREATMENT_DIR.mkdir(parents=True, exist_ok=True)

TREATMENT_ADAPTER = (
    TREATMENT_DIR
    / "experiment_B_structured_adapter_v1.py"
)

DRIVE_TREATMENT_ADAPTER = (
    DRIVE_TREATMENT_DIR
    / TREATMENT_ADAPTER.name
)

MANIFEST_DIR = B_ROOT / "manifests"
DRIVE_MANIFEST_DIR = B_DRIVE / "manifests"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = (
    MANIFEST_DIR
    / "experiment_B_B7B_treatment_adapter_manifest_v1.json"
)

DRIVE_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / MANIFEST_PATH.name
)


# Refuse silent overwrite.
for path in [
    TREATMENT_ADAPTER,
    DRIVE_TREATMENT_ADAPTER,
    MANIFEST_PATH,
    DRIVE_MANIFEST_PATH,
]:
    assert not path.exists(), (
        "STOP: B7B artifact already exists.\n"
        "Do not silently overwrite treatment evidence.\n"
        f"{path}"
    )

print("✓ No pre-existing B7B artifacts detected")


# ==================================================================================================
# 5. Serialize frozen structural schema into isolated adapter
# ==================================================================================================

schema_literal = repr(STRUCTURAL_SCHEMA)
prompt_literal = repr(CIR_SYSTEM_PROMPT)


# ==================================================================================================
# 6. Construct treatment adapter
#
# IMPORTANT:
# - same model
# - same CIR prompt
# - same user prompt
# - same reasoning setting
# - same temperature
# - same parsing/return contract
# - ONLY experimental addition = provider-side response_format
# ==================================================================================================

treatment_source = f'''\
"""
Experiment B isolated treatment adapter.

DO NOT use as a replacement for the historical adapter.

Experimental treatment:
    Provider-side STRUCTURAL JSON-schema constrained generation.

Frozen comparison:
    Historical call_json() behavior is preserved except for the
    preregistered response_format constraint.

This module does not contain benchmark execution logic.
"""

import json
import os
import time

from together import Together


MODEL_DEFAULT = "Qwen/Qwen3.5-9B"

CIR_SYSTEM_PROMPT = {prompt_literal}

STRUCTURAL_DECODING_SCHEMA = {schema_literal}


def _clean_json_text(text):
    """
    Preserve historical defensive JSON-text cleaning behavior.

    The structured-output treatment should normally return JSON directly,
    but downstream adapter behavior remains compatible with the historical
    call_json() return contract.
    """

    text = (text or "").strip()

    if text.startswith("```"):
        lines = text.splitlines()

        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\\n".join(lines).strip()

        if text.lower().startswith("json"):
            text = text[4:].lstrip()

    return text


def call_json_structured(
    prompt,
    mode="cir",
    model=MODEL_DEFAULT,
):
    """
    Experiment-B treatment equivalent of historical call_json().

    The only intended experimental decoding change is the addition of
    provider-side structural JSON-schema response_format.

    No semantic enum/range constraints are imposed here.
    """

    api_key = os.environ.get("TOGETHER_API_KEY")

    if not api_key:
        raise RuntimeError(
            "TOGETHER_API_KEY is not set."
        )

    if mode != "cir":
        raise ValueError(
            "Experiment-B structured treatment supports mode='cir' only"
        )

    client = Together(api_key=api_key)

    response_format = {{
        "type": "json_schema",
        "schema": {{
            "name": "network_slice_cir",
            "schema": STRUCTURAL_DECODING_SCHEMA,
        }},
    }}

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,

        reasoning={{
            "enabled": False
        }},

        messages=[
            {{
                "role": "system",
                "content": CIR_SYSTEM_PROMPT
            }},
            {{
                "role": "user",
                "content": prompt
            }}
        ],

        temperature=0,

        response_format=response_format,
    )

    latency_ms = (
        time.perf_counter() - start
    ) * 1000

    choice = response.choices[0]
    message = choice.message

    raw_text = message.content or ""
    clean_text = _clean_json_text(raw_text)

    try:
        parsed_json = json.loads(clean_text)
        parse_ok = True

    except (json.JSONDecodeError, TypeError):
        parsed_json = None
        parse_ok = False

    usage = getattr(response, "usage", None)

    return {{
        "raw": clean_text,
        "json": parsed_json,
        "parse_ok": parse_ok,
        "latency_ms": latency_ms,

        "finish_reason": getattr(
            choice,
            "finish_reason",
            None
        ),

        "reasoning": getattr(
            message,
            "reasoning",
            None
        ),

        "prompt_tokens": getattr(
            usage,
            "prompt_tokens",
            None
        ) if usage else None,

        "completion_tokens": getattr(
            usage,
            "completion_tokens",
            None
        ) if usage else None
    }}
'''


# ==================================================================================================
# 7. Write isolated adapter
# ==================================================================================================

TREATMENT_ADAPTER.write_text(
    treatment_source,
    encoding="utf-8"
)

print("✓ Isolated treatment adapter written")
print(TREATMENT_ADAPTER)


# ==================================================================================================
# 8. Compile — static only, no API call
# ==================================================================================================

py_compile.compile(
    str(TREATMENT_ADAPTER),
    doraise=True
)

print("✓ Treatment adapter compilation successful")


# ==================================================================================================
# 9. AST audit treatment adapter
# ==================================================================================================

treatment_text = TREATMENT_ADAPTER.read_text(encoding="utf-8")
treatment_tree = ast.parse(treatment_text)

functions = {
    node.name
    for node in treatment_tree.body
    if isinstance(node, ast.FunctionDef)
}

assert "call_json_structured" in functions
assert "_clean_json_text" in functions

print("✓ Required treatment functions present")


# Exactly one provider create() call in source.
create_calls = []

for node in ast.walk(treatment_tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if (
        isinstance(func, ast.Attribute)
        and func.attr == "create"
    ):
        create_calls.append(node)

assert len(create_calls) == 1, (
    "STOP: treatment adapter must contain exactly "
    f"one provider create() call; found {len(create_calls)}"
)

print("✓ Exactly one provider create() call site in treatment adapter")


# Verify response_format is actually passed.
create_call = create_calls[0]

keyword_names = {
    kw.arg
    for kw in create_call.keywords
    if kw.arg is not None
}

required_keywords = {
    "model",
    "reasoning",
    "messages",
    "temperature",
    "response_format",
}

assert required_keywords.issubset(keyword_names), (
    "STOP: treatment provider call missing expected arguments"
)

print("✓ response_format present in treatment provider call")


# ==================================================================================================
# 10. Verify treatment schema embedded byte-semantically
# ==================================================================================================

namespace = {}

# Do NOT exec/import the module because we do not need runtime behavior here.
# Extract assignment statically instead.

embedded_schema = None
embedded_prompt = None

for node in treatment_tree.body:

    if not isinstance(node, ast.Assign):
        continue

    names = [
        t.id
        for t in node.targets
        if isinstance(t, ast.Name)
    ]

    if "STRUCTURAL_DECODING_SCHEMA" in names:
        embedded_schema = ast.literal_eval(node.value)

    if "CIR_SYSTEM_PROMPT" in names:
        embedded_prompt = ast.literal_eval(node.value)


assert embedded_schema == STRUCTURAL_SCHEMA, (
    "STOP: embedded treatment schema differs from preregistration"
)

assert embedded_prompt == CIR_SYSTEM_PROMPT, (
    "STOP: treatment prompt differs from historical CIR prompt"
)

print("✓ Embedded schema exactly matches preregistration")
print("✓ Embedded CIR prompt exactly matches historical adapter")


# ==================================================================================================
# 11. Check treatment schema remains structural-only
# ==================================================================================================

for field_name, field_schema in embedded_schema["properties"].items():

    leaked = FORBIDDEN.intersection(field_schema)

    assert not leaked, (
        f"STOP: forbidden semantic constraint in {field_name}: "
        f"{sorted(leaked)}"
    )

print("✓ Treatment schema remains structural-only")


# ==================================================================================================
# 12. Hash treatment adapter
# ==================================================================================================

TREATMENT_SHA = sha256(TREATMENT_ADAPTER)

print()
print("Treatment adapter SHA-256:")
print(TREATMENT_SHA)


# ==================================================================================================
# 13. Create provenance manifest
# ==================================================================================================

manifest = {
    "experiment":
        "Experiment B",

    "step":
        "B7B",

    "artifact":
        "isolated schema-constrained treatment adapter",

    "treatment_adapter":
        str(TREATMENT_ADAPTER),

    "treatment_adapter_sha256":
        TREATMENT_SHA,

    "model":
        "Qwen/Qwen3.5-9B",

    "experimental_change":
        "provider-side structural JSON-schema response_format",

    "historical_runner_sha256":
        EXPECTED["runner"],

    "historical_adapter_sha256":
        EXPECTED["adapter"],

    "historical_schema_sha256":
        EXPECTED["schema"],

    "historical_validator_sha256":
        EXPECTED["validator"],

    "experiment_B_preregistration_sha256":
        EXPECTED["prereg"],

    "B6_capability_probe_sha256":
        EXPECTED["b6_probe"],

    "semantic_constraints_in_decoder":
        False,

    "benchmark_executed":
        False,

    "api_calls":
        0,

    "historical_files_modified":
        False,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

MANIFEST_SHA = sha256(MANIFEST_PATH)

print()
print("B7B manifest SHA-256:")
print(MANIFEST_SHA)


# ==================================================================================================
# 14. Persist to Drive
# ==================================================================================================

shutil.copy2(
    TREATMENT_ADAPTER,
    DRIVE_TREATMENT_ADAPTER
)

shutil.copy2(
    MANIFEST_PATH,
    DRIVE_MANIFEST_PATH
)

assert (
    sha256(DRIVE_TREATMENT_ADAPTER)
    == TREATMENT_SHA
)

assert (
    sha256(DRIVE_MANIFEST_PATH)
    == MANIFEST_SHA
)

print("✓ Treatment adapter persisted to Drive")
print("✓ Manifest persisted to Drive")
print("✓ Drive copies verified byte-for-byte")


# ==================================================================================================
# 15. Reverify ALL protected historical files
# ==================================================================================================

print("\n" + "-" * 118)
print("POST-CONSTRUCTION HISTORICAL INTEGRITY")
print("-" * 118)

verify(
    "runner after B7B",
    RUNNER,
    EXPECTED["runner"]
)

verify(
    "historical adapter after B7B",
    ADAPTER,
    EXPECTED["adapter"]
)

verify(
    "schema after B7B",
    SCHEMA,
    EXPECTED["schema"]
)

verify(
    "validator after B7B",
    VALIDATOR,
    EXPECTED["validator"]
)

verify(
    "Experiment-B preregistration after B7B",
    PREREG,
    EXPECTED["prereg"]
)

verify(
    "B6 capability probe after B7B",
    B6_PROBE,
    EXPECTED["b6_probe"]
)


print("\n" + "=" * 118)
print("✓ STEP B7B COMPLETE")
print("✓ ISOLATED TREATMENT ADAPTER CREATED")
print("✓ STRUCTURAL SCHEMA MATCHES PREREGISTRATION")
print("✓ CIR PROMPT MATCHES HISTORICAL ADAPTER")
print("✓ PROVIDER response_format PRESENT")
print("✓ NO SEMANTIC/RANGE CONSTRAINT MOVED INTO DECODER")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE STEP B8")
print("=" * 118)

EXPERIMENT B — STEP B7B: BUILD ISOLATED SCHEMA-CONSTRAINED TREATMENT ADAPTER

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ runner verified
✓ historical adapter verified
✓ schema verified
✓ validator verified
✓ Experiment-B preregistration verified
✓ B6 capability probe verified
✓ Frozen structural treatment schema verified
✓ No semantic/range restrictions in decoding schema
✓ Historical CIR_SYSTEM_PROMPT extracted exactly
✓ No pre-existing B7B artifacts detected
✓ Isolated treatment adapter written
/content/deterministic-intent-translation-journal/journal_experiments/experiment_B_schema_constrained_decoding/implementation/experiment_B_structured_adapter_v1.py
✓ Treatment adapter compilation successful
✓ Required treatment functions present
✓ Exactly one provider create() call 

In [31]:
from pathlib import Path
import hashlib
import json
import ast
import py_compile
import shutil

print("=" * 118)
print("EXPERIMENT B — STEP B8: BUILD AND STATICALLY VERIFY TREATMENT EXECUTION HARNESS")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"
SCHEMA = ROOT / "src" / "schema.py"
VALIDATOR = ROOT / "src" / "validator.py"

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

B6_PROBE = (
    B_ROOT
    / "results"
    / "capability_probe"
    / "B6_structured_output_capability_probe_v1.json"
)

TREATMENT_ADAPTER = (
    B_ROOT
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

B7B_MANIFEST = (
    B_ROOT
    / "manifests"
    / "experiment_B_B7B_treatment_adapter_manifest_v1.json"
)


EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "benchmark":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",

    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "b6_probe":
        "fc509b6ca254882935ac30c32333ac510d465024a32bf9cbf48ed92239a36392",

    "treatment_adapter":
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    "b7b_manifest":
        "1a1d5fa869d969ae9bccc33091354a03fede9561006ee40a73fdb1f8ac921329",
}


# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), f"STOP: missing {label}: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Verify all frozen provenance
# ======================================================================================

print("\n" + "-" * 118)
print("PRE-CONSTRUCTION PROVENANCE")
print("-" * 118)

verify("frozen runner", RUNNER, EXPECTED["runner"])
verify("frozen benchmark", BENCHMARK, EXPECTED["benchmark"])
verify("historical adapter", ADAPTER, EXPECTED["adapter"])
verify("schema", SCHEMA, EXPECTED["schema"])
verify("validator", VALIDATOR, EXPECTED["validator"])
verify("Experiment-B preregistration", PREREG, EXPECTED["prereg"])
verify("B6 capability probe", B6_PROBE, EXPECTED["b6_probe"])
verify("B7B treatment adapter", TREATMENT_ADAPTER, EXPECTED["treatment_adapter"])
verify("B7B manifest", B7B_MANIFEST, EXPECTED["b7b_manifest"])


# ======================================================================================
# 2. Verify B6 status
# ======================================================================================

b6 = json.loads(
    B6_PROBE.read_text(encoding="utf-8")
)

assert (
    b6["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
), (
    "STOP: B6 capability status does not permit treatment execution"
)

assert b6["benchmark_case"] is False
assert b6["provider_calls_attempted"] == 1

print("✓ B6 capability gate satisfied")


# ======================================================================================
# 3. Static verification of frozen runner binding
# ======================================================================================

runner_source = RUNNER.read_text(encoding="utf-8")
runner_tree = ast.parse(runner_source)

# Verify historical call_json import.
matching_imports = []

for node in runner_tree.body:

    if isinstance(node, ast.ImportFrom):

        if node.module == "src.llm.together_adapter":

            imported = {
                alias.name
                for alias in node.names
            }

            if "call_json" in imported:
                matching_imports.append(node)

assert len(matching_imports) == 1, (
    "STOP: expected exactly one historical call_json import"
)

print("✓ Frozen runner has exactly one historical call_json import")


# Verify call_cir delegates to module-global call_json.
call_cir_nodes = [
    node
    for node in runner_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "call_cir"
]

assert len(call_cir_nodes) == 1

call_cir_node = call_cir_nodes[0]

call_json_calls = []

for node in ast.walk(call_cir_node):

    if isinstance(node, ast.Call):

        if (
            isinstance(node.func, ast.Name)
            and node.func.id == "call_json"
        ):
            call_json_calls.append(node)

assert len(call_json_calls) == 1, (
    "STOP: call_cir does not contain exactly one call_json call"
)

print("✓ call_cir delegates exactly once to module-global call_json")


# ======================================================================================
# 4. Verify treatment adapter interface
# ======================================================================================

treatment_source = TREATMENT_ADAPTER.read_text(
    encoding="utf-8"
)

treatment_tree = ast.parse(treatment_source)

treatment_functions = [
    node
    for node in treatment_tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "call_json_structured"
]

assert len(treatment_functions) == 1

fn = treatment_functions[0]

args = [
    arg.arg
    for arg in fn.args.args
]

assert args == [
    "prompt",
    "mode",
    "model",
], (
    "STOP: treatment adapter function signature differs "
    f"from expected interface: {args}"
)

print("✓ Treatment adapter exposes compatible call_json_structured interface")


# ======================================================================================
# 5. Construct isolated treatment execution harness
#
# IMPORTANT:
# This harness:
#   - dynamically loads the FROZEN runner
#   - dynamically loads the B7B treatment adapter
#   - changes only runner_module.call_json in memory
#   - calls the frozen runner's main() with original CLI semantics
#
# It does NOT copy/edit/rewrite run_e3_v3().
# ======================================================================================

IMPLEMENTATION_DIR = B_ROOT / "implementation"

DRIVE_IMPLEMENTATION_DIR = (
    B_DRIVE
    / "implementation"
)

IMPLEMENTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_IMPLEMENTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HARNESS = (
    IMPLEMENTATION_DIR
    / "run_experiment_B_structured_treatment_v1.py"
)

DRIVE_HARNESS = (
    DRIVE_IMPLEMENTATION_DIR
    / HARNESS.name
)

MANIFEST_DIR = B_ROOT / "manifests"
DRIVE_MANIFEST_DIR = B_DRIVE / "manifests"

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = (
    MANIFEST_DIR
    / "experiment_B_B8_harness_manifest_v1.json"
)

DRIVE_MANIFEST = (
    DRIVE_MANIFEST_DIR
    / MANIFEST.name
)


for path in [
    HARNESS,
    DRIVE_HARNESS,
    MANIFEST,
    DRIVE_MANIFEST,
]:
    assert not path.exists(), (
        "STOP: B8 artifact already exists. "
        "Do not silently overwrite.\n"
        f"{path}"
    )


harness_source = f'''\
"""
Experiment B — schema-constrained decoding treatment harness.

This harness DOES NOT modify or duplicate the frozen E3-v3 pipeline.

It dynamically loads:
  1. the frozen historical E3-v3 runner;
  2. the isolated Experiment-B structured-output adapter.

The only in-memory experimental substitution is:

    frozen_runner.call_json =
        treatment_adapter.call_json_structured

Because frozen call_cir() resolves call_json from its module globals,
all downstream frozen E3-v3 logic remains unchanged.
"""

from pathlib import Path
import argparse
import importlib.util
import sys


ROOT = Path({str(ROOT)!r})

FROZEN_RUNNER = Path({str(RUNNER)!r})

TREATMENT_ADAPTER = Path({str(TREATMENT_ADAPTER)!r})

EXPECTED_RUNNER_SHA256 = {EXPECTED["runner"]!r}

EXPECTED_TREATMENT_ADAPTER_SHA256 = {EXPECTED["treatment_adapter"]!r}


def sha256(path):
    import hashlib

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def load_module(name, path):

    spec = importlib.util.spec_from_file_location(
        name,
        path,
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Unable to load module from {{path}}"
        )

    module = importlib.util.module_from_spec(spec)

    sys.modules[name] = module

    spec.loader.exec_module(module)

    return module


def main():

    # ------------------------------------------------------------------
    # Integrity before loading
    # ------------------------------------------------------------------

    if sha256(FROZEN_RUNNER) != EXPECTED_RUNNER_SHA256:
        raise RuntimeError(
            "Frozen runner SHA-256 mismatch"
        )

    if (
        sha256(TREATMENT_ADAPTER)
        != EXPECTED_TREATMENT_ADAPTER_SHA256
    ):
        raise RuntimeError(
            "Treatment adapter SHA-256 mismatch"
        )

    # Project root must precede imports performed by frozen runner.
    root_text = str(ROOT)

    if root_text not in sys.path:
        sys.path.insert(0, root_text)

    # ------------------------------------------------------------------
    # Load exact frozen runner and isolated treatment adapter
    # ------------------------------------------------------------------

    frozen_runner = load_module(
        "experiment_B_frozen_e3v3_runner",
        FROZEN_RUNNER,
    )

    treatment_adapter = load_module(
        "experiment_B_structured_adapter",
        TREATMENT_ADAPTER,
    )

    # ------------------------------------------------------------------
    # SINGLE EXPERIMENTAL SUBSTITUTION
    # ------------------------------------------------------------------

    historical_call_json = frozen_runner.call_json

    frozen_runner.call_json = (
        treatment_adapter.call_json_structured
    )

    if (
        frozen_runner.call_json
        is not treatment_adapter.call_json_structured
    ):
        raise RuntimeError(
            "Treatment call_json substitution failed"
        )

    if frozen_runner.call_json is historical_call_json:
        raise RuntimeError(
            "Historical call_json remained active"
        )

    # ------------------------------------------------------------------
    # Delegate CLI execution to frozen runner.
    #
    # No benchmark semantics are implemented here.
    # ------------------------------------------------------------------

    frozen_runner.main()


if __name__ == "__main__":
    main()
'''


HARNESS.write_text(
    harness_source,
    encoding="utf-8"
)

print("✓ Isolated treatment execution harness written")
print(HARNESS)


# ======================================================================================
# 6. Compile harness — does not execute it
# ======================================================================================

py_compile.compile(
    str(HARNESS),
    doraise=True
)

print("✓ Treatment harness compilation successful")


# ======================================================================================
# 7. AST audit harness
# ======================================================================================

harness_text = HARNESS.read_text(encoding="utf-8")
harness_tree = ast.parse(harness_text)

# Ensure harness itself contains NO Together/provider call.
provider_create_calls = []

for node in ast.walk(harness_tree):

    if not isinstance(node, ast.Call):
        continue

    if (
        isinstance(node.func, ast.Attribute)
        and node.func.attr == "create"
    ):
        provider_create_calls.append(node)

assert len(provider_create_calls) == 0, (
    "STOP: execution harness unexpectedly contains "
    "a provider create() call"
)

print("✓ Harness contains no provider API implementation")


# Ensure exactly one assignment to frozen_runner.call_json.
patch_assignments = []

for node in ast.walk(harness_tree):

    if not isinstance(node, ast.Assign):
        continue

    for target in node.targets:

        if (
            isinstance(target, ast.Attribute)
            and isinstance(target.value, ast.Name)
            and target.value.id == "frozen_runner"
            and target.attr == "call_json"
        ):
            patch_assignments.append(node)

assert len(patch_assignments) == 1, (
    "STOP: expected exactly one treatment binding assignment; "
    f"found {len(patch_assignments)}"
)

print("✓ Exactly one experimental binding substitution present")


# Verify run_e3_v3 is NOT reimplemented.
defined_functions = {
    node.name
    for node in harness_tree.body
    if isinstance(node, ast.FunctionDef)
}

assert "run_e3_v3" not in defined_functions
assert "call_cir" not in defined_functions
assert "parse_llm_result" not in defined_functions
assert "validate_cir" not in defined_functions

print("✓ Frozen pipeline functions are not duplicated/reimplemented")


# Verify frozen main() is delegated exactly once.
main_calls = []

for node in ast.walk(harness_tree):

    if (
        isinstance(node, ast.Call)
        and isinstance(node.func, ast.Attribute)
        and isinstance(node.func.value, ast.Name)
        and node.func.value.id == "frozen_runner"
        and node.func.attr == "main"
    ):
        main_calls.append(node)

assert len(main_calls) == 1, (
    "STOP: expected exactly one delegation to frozen_runner.main()"
)

print("✓ Harness delegates execution to frozen runner main() exactly once")


# ======================================================================================
# 8. Verify frozen runner's CLI supports required arguments
# ======================================================================================

required_cli_literals = {
    "--dataset",
    "--out",
    "--model",
}

runner_string_literals = {
    node.value
    for node in ast.walk(runner_tree)
    if isinstance(node, ast.Constant)
    and isinstance(node.value, str)
}

missing_cli = (
    required_cli_literals
    - runner_string_literals
)

assert not missing_cli, (
    "STOP: frozen runner missing expected CLI arguments: "
    f"{sorted(missing_cli)}"
)

print("✓ Frozen runner CLI contains --dataset, --out, --model")


# ======================================================================================
# 9. Hash + manifest
# ======================================================================================

HARNESS_SHA = sha256(HARNESS)

print()
print("Treatment harness SHA-256:")
print(HARNESS_SHA)


manifest_obj = {
    "experiment":
        "Experiment B",

    "step":
        "B8",

    "artifact":
        "isolated structured-decoding treatment execution harness",

    "harness_path":
        str(HARNESS),

    "harness_sha256":
        HARNESS_SHA,

    "frozen_runner_sha256":
        EXPECTED["runner"],

    "frozen_benchmark_sha256":
        EXPECTED["benchmark"],

    "historical_adapter_sha256":
        EXPECTED["adapter"],

    "schema_sha256":
        EXPECTED["schema"],

    "validator_sha256":
        EXPECTED["validator"],

    "preregistration_sha256":
        EXPECTED["prereg"],

    "B6_probe_sha256":
        EXPECTED["b6_probe"],

    "B7B_treatment_adapter_sha256":
        EXPECTED["treatment_adapter"],

    "experimental_binding":
        (
            "frozen_runner.call_json -> "
            "treatment_adapter.call_json_structured"
        ),

    "frozen_run_e3_v3_reimplemented":
        False,

    "provider_calls":
        0,

    "benchmark_executed":
        False,

    "historical_files_modified":
        False,
}


MANIFEST.write_text(
    json.dumps(
        manifest_obj,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

MANIFEST_SHA = sha256(MANIFEST)

print()
print("B8 manifest SHA-256:")
print(MANIFEST_SHA)


# ======================================================================================
# 10. Persist
# ======================================================================================

shutil.copy2(
    HARNESS,
    DRIVE_HARNESS
)

shutil.copy2(
    MANIFEST,
    DRIVE_MANIFEST
)

assert sha256(DRIVE_HARNESS) == HARNESS_SHA
assert sha256(DRIVE_MANIFEST) == MANIFEST_SHA

print("✓ Harness persisted to Drive")
print("✓ Manifest persisted to Drive")
print("✓ Drive copies verified byte-for-byte")


# ======================================================================================
# 11. Final historical integrity
# ======================================================================================

print("\n" + "-" * 118)
print("POST-B8 HISTORICAL INTEGRITY")
print("-" * 118)

verify("runner after B8", RUNNER, EXPECTED["runner"])
verify("benchmark after B8", BENCHMARK, EXPECTED["benchmark"])
verify("historical adapter after B8", ADAPTER, EXPECTED["adapter"])
verify("schema after B8", SCHEMA, EXPECTED["schema"])
verify("validator after B8", VALIDATOR, EXPECTED["validator"])
verify("preregistration after B8", PREREG, EXPECTED["prereg"])
verify("B6 probe after B8", B6_PROBE, EXPECTED["b6_probe"])
verify(
    "B7B treatment adapter after B8",
    TREATMENT_ADAPTER,
    EXPECTED["treatment_adapter"]
)


print("\n" + "=" * 118)
print("✓ STEP B8 COMPLETE")
print("✓ TREATMENT EXECUTION HARNESS CREATED")
print("✓ ONLY call_json BINDING IS SUBSTITUTED IN MEMORY")
print("✓ FROZEN run_e3_v3 NOT COPIED OR MODIFIED")
print("✓ FROZEN BENCHMARK NOT EXECUTED")
print("✓ NO API/MODEL CALL")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE THE ONE-TIME B9 BENCHMARK RUN")
print("=" * 118)

EXPERIMENT B — STEP B8: BUILD AND STATICALLY VERIFY TREATMENT EXECUTION HARNESS

----------------------------------------------------------------------------------------------------------------------
PRE-CONSTRUCTION PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ frozen runner verified
✓ frozen benchmark verified
✓ historical adapter verified
✓ schema verified
✓ validator verified
✓ Experiment-B preregistration verified
✓ B6 capability probe verified
✓ B7B treatment adapter verified
✓ B7B manifest verified
✓ B6 capability gate satisfied
✓ Frozen runner has exactly one historical call_json import
✓ call_cir delegates exactly once to module-global call_json
✓ Treatment adapter exposes compatible call_json_structured interface
✓ Isolated treatment execution harness written
/content/deterministic-intent-translation-journal/journal_experiments/experiment_B_schema_constrained_decoding/implementation/run_expe

In [32]:
from pathlib import Path
import hashlib
import json
import os
import pandas as pd

print("=" * 118)
print("EXPERIMENT B — STEP B9A: FINAL PREFLIGHT BEFORE ONE-TIME 120-CASE TREATMENT RUN")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"
SCHEMA = ROOT / "src" / "schema.py"
VALIDATOR = ROOT / "src" / "validator.py"

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

B6_PROBE = (
    B_ROOT
    / "results"
    / "capability_probe"
    / "B6_structured_output_capability_probe_v1.json"
)

TREATMENT_ADAPTER = (
    B_ROOT
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

HARNESS = (
    B_ROOT
    / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

B8_MANIFEST = (
    B_ROOT
    / "manifests"
    / "experiment_B_B8_harness_manifest_v1.json"
)


EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "benchmark":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",

    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "b6_probe":
        "fc509b6ca254882935ac30c32333ac510d465024a32bf9cbf48ed92239a36392",

    "treatment_adapter":
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    "harness":
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    "b8_manifest":
        "5b7498a584bed809ecc9398442e3c6df274c5c1310b8dd6a593a65d80fdd6fc6",
}


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), f"STOP: missing {label}: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Verify entire provenance chain
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

verify("runner", RUNNER, EXPECTED["runner"])
verify("benchmark", BENCHMARK, EXPECTED["benchmark"])
verify("historical adapter", ADAPTER, EXPECTED["adapter"])
verify("schema", SCHEMA, EXPECTED["schema"])
verify("validator", VALIDATOR, EXPECTED["validator"])
verify("Experiment-B preregistration", PREREG, EXPECTED["prereg"])
verify("B6 capability probe", B6_PROBE, EXPECTED["b6_probe"])
verify("B7B treatment adapter", TREATMENT_ADAPTER, EXPECTED["treatment_adapter"])
verify("B8 treatment harness", HARNESS, EXPECTED["harness"])
verify("B8 manifest", B8_MANIFEST, EXPECTED["b8_manifest"])


# ======================================================================================
# 2. Verify preregistered treatment identity
# ======================================================================================

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

assert (
    prereg["treatment_arm"]["model"]
    == "Qwen/Qwen3.5-9B"
), "STOP: preregistered treatment model mismatch"

print("✓ Treatment model = Qwen/Qwen3.5-9B")


# ======================================================================================
# 3. Verify B6 provider capability
# ======================================================================================

b6 = json.loads(
    B6_PROBE.read_text(encoding="utf-8")
)

assert (
    b6["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert b6["api_success"] is True
assert b6["parse_ok"] is True
assert b6["structural_schema_valid"] is True
assert b6["benchmark_case"] is False
assert b6["provider_calls_attempted"] == 1

print("✓ B6 structured-output capability confirmed")


# ======================================================================================
# 4. Verify benchmark cardinality
# ======================================================================================

df = pd.read_csv(BENCHMARK)

print()
print("Benchmark rows:", len(df))
print("Benchmark columns:", list(df.columns))

assert len(df) == 120, (
    f"STOP: expected 120 benchmark cases, found {len(df)}"
)

print("✓ Frozen benchmark contains exactly 120 cases")


# ======================================================================================
# 5. Inspect validity distribution WITHOUT executing anything
# ======================================================================================

possible_validity_columns = [
    c
    for c in df.columns
    if c.lower() in {
        "expected_valid",
        "valid",
        "is_valid",
        "ground_truth_valid",
    }
]

if possible_validity_columns:

    col = possible_validity_columns[0]

    print()
    print("Ground-truth column:", col)
    print(df[col].value_counts(dropna=False).to_string())

else:
    print()
    print(
        "Ground-truth validity distribution not inferred "
        "automatically; this does not block execution."
    )


# ======================================================================================
# 6. Define the ONE-TIME output path
# ======================================================================================

RESULT_DIR = (
    B_ROOT
    / "results"
    / "treatment_B1"
)

DRIVE_RESULT_DIR = (
    B_DRIVE
    / "results"
    / "treatment_B1"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = (
    RESULT_DIR
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

DRIVE_OUTPUT = (
    DRIVE_RESULT_DIR
    / OUTPUT.name
)


# ======================================================================================
# 7. Hard rerun protection
# ======================================================================================

assert not OUTPUT.exists(), (
    "STOP: treatment output already exists.\n"
    "DO NOT rerun B9.\n"
    f"{OUTPUT}"
)

assert not DRIVE_OUTPUT.exists(), (
    "STOP: persistent treatment output already exists.\n"
    "DO NOT rerun B9.\n"
    f"{DRIVE_OUTPUT}"
)

print()
print("✓ No previous B9 treatment output detected")


# ======================================================================================
# 8. API key check — DO NOT display
# ======================================================================================

if os.environ.get("TOGETHER_API_KEY"):
    print("✓ TOGETHER_API_KEY is available")
else:
    print(
        "⚠ TOGETHER_API_KEY is NOT currently set.\n"
        "This preflight still makes NO API call."
    )


# ======================================================================================
# 9. Display exact future command — DO NOT execute
# ======================================================================================

print("\n" + "-" * 118)
print("LOCKED B9 EXECUTION PLAN")
print("-" * 118)

print("Harness:")
print(HARNESS)

print("\nDataset:")
print(BENCHMARK)

print("\nOutput:")
print(OUTPUT)

print("\nModel:")
print("Qwen/Qwen3.5-9B")

print("\nCases:")
print("120")

print()
print("IMPORTANT:")
print("  • B9A does NOT execute this command.")
print("  • No API call occurs in this cell.")
print("  • The command will be issued only in B9B after this preflight passes.")


# ======================================================================================
# 10. Final hash recheck
# ======================================================================================

print("\n" + "-" * 118)
print("FINAL PREFLIGHT HASH RECHECK")
print("-" * 118)

verify("runner final", RUNNER, EXPECTED["runner"])
verify("benchmark final", BENCHMARK, EXPECTED["benchmark"])
verify("treatment adapter final", TREATMENT_ADAPTER, EXPECTED["treatment_adapter"])
verify("treatment harness final", HARNESS, EXPECTED["harness"])
verify("preregistration final", PREREG, EXPECTED["prereg"])


print("\n" + "=" * 118)
print("✓ STEP B9A COMPLETE")
print("✓ ALL FROZEN HASHES VERIFIED")
print("✓ 120-CASE BENCHMARK VERIFIED")
print("✓ TREATMENT MODEL VERIFIED")
print("✓ B6 CAPABILITY VERIFIED")
print("✓ NO PREVIOUS B9 OUTPUT DETECTED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK CASES EXECUTED")
print("✓ READY FOR ONE-TIME B9B EXECUTION")
print("✓ SEND THIS OUTPUT BEFORE RUNNING B9B")
print("=" * 118)

EXPERIMENT B — STEP B9A: FINAL PREFLIGHT BEFORE ONE-TIME 120-CASE TREATMENT RUN

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ runner verified
✓ benchmark verified
✓ historical adapter verified
✓ schema verified
✓ validator verified
✓ Experiment-B preregistration verified
✓ B6 capability probe verified
✓ B7B treatment adapter verified
✓ B8 treatment harness verified
✓ B8 manifest verified


KeyError: 'model'

In [33]:
from pathlib import Path
import json
import hashlib

print("=" * 118)
print("EXPERIMENT B — STEP B9A-D1: DIAGNOSE PREREGISTRATION MODEL LOCATION")
print("=" * 118)

PREREG = Path(
    "/content/deterministic-intent-translation-journal/"
    "journal_experiments/"
    "experiment_B_schema_constrained_decoding/"
    "preregistration/"
    "experiment_B_preregistration_v1.json"
)

EXPECTED_SHA = (
    "d6e6db1bf6248ac211f4ef925ba940254"
    "e765e6eedc51c0b67353f897a50a2f5"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# --------------------------------------------------------------------------------------
# 1. Verify frozen preregistration
# --------------------------------------------------------------------------------------

assert PREREG.exists(), f"STOP: missing preregistration: {PREREG}"

actual_sha = sha256(PREREG)

assert actual_sha == EXPECTED_SHA, (
    "STOP: preregistration hash mismatch\n"
    f"Expected: {EXPECTED_SHA}\n"
    f"Actual:   {actual_sha}"
)

print("✓ Frozen preregistration verified")
print("SHA-256:", actual_sha)


# --------------------------------------------------------------------------------------
# 2. Load JSON
# --------------------------------------------------------------------------------------

obj = json.loads(
    PREREG.read_text(encoding="utf-8")
)

print("\nTop-level keys:")
for key in obj.keys():
    print("  -", key)


# --------------------------------------------------------------------------------------
# 3. Recursively locate every key named 'model'
# --------------------------------------------------------------------------------------

matches = []


def walk(value, path="$"):

    if isinstance(value, dict):

        for key, child in value.items():

            child_path = f"{path}.{key}"

            if key == "model":
                matches.append(
                    (child_path, child)
                )

            walk(child, child_path)

    elif isinstance(value, list):

        for index, child in enumerate(value):

            walk(
                child,
                f"{path}[{index}]"
            )


walk(obj)


print("\n" + "-" * 118)
print("ALL 'model' FIELDS")
print("-" * 118)

if matches:

    for path, value in matches:
        print(f"{path} = {value!r}")

else:
    print("NONE")


# --------------------------------------------------------------------------------------
# 4. Show treatment_arm structure
# --------------------------------------------------------------------------------------

print("\n" + "-" * 118)
print("treatment_arm CONTENT")
print("-" * 118)

treatment_arm = obj.get("treatment_arm")

print(
    json.dumps(
        treatment_arm,
        indent=2,
        ensure_ascii=False
    )
)


# --------------------------------------------------------------------------------------
# 5. Search recursively for Qwen/Qwen3.5-9B regardless of key
# --------------------------------------------------------------------------------------

MODEL = "Qwen/Qwen3.5-9B"

value_matches = []


def find_value(value, path="$"):

    if isinstance(value, dict):

        for key, child in value.items():
            find_value(
                child,
                f"{path}.{key}"
            )

    elif isinstance(value, list):

        for index, child in enumerate(value):
            find_value(
                child,
                f"{path}[{index}]"
            )

    elif value == MODEL:

        value_matches.append(path)


find_value(obj)


print("\n" + "-" * 118)
print("EXACT MODEL-VALUE LOCATIONS")
print("-" * 118)

if value_matches:

    for path in value_matches:
        print(path)

else:
    print("NONE")


# --------------------------------------------------------------------------------------
# 6. Integrity recheck
# --------------------------------------------------------------------------------------

assert sha256(PREREG) == EXPECTED_SHA

print("\n" + "=" * 118)
print("✓ STEP B9A-D1 COMPLETE")
print("✓ PREREGISTRATION UNCHANGED")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE CORRECTING B9A")
print("=" * 118)

EXPERIMENT B — STEP B9A-D1: DIAGNOSE PREREGISTRATION MODEL LOCATION
✓ Frozen preregistration verified
SHA-256: d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5

Top-level keys:
  - analysis_requirements
  - benchmark
  - claim_boundary
  - control_arm
  - created_utc
  - design_type
  - execution_policy
  - experiment
  - frozen_components
  - implementation_policy
  - primary_metrics
  - research_question
  - semantic_model
  - semantic_safety_metrics
  - status
  - structured_generation_metrics
  - treatment_arm
  - version

----------------------------------------------------------------------------------------------------------------------
ALL 'model' FIELDS
----------------------------------------------------------------------------------------------------------------------
$.semantic_model.model = 'Qwen/Qwen3.5-9B'

----------------------------------------------------------------------------------------------------------------------
treatment_arm CONTENT
---------

In [34]:
from pathlib import Path
import hashlib
import json
import os
import pandas as pd

print("=" * 118)
print("EXPERIMENT B — STEP B9A-R1: CORRECTED FINAL PREFLIGHT BEFORE ONE-TIME TREATMENT RUN")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"
SCHEMA = ROOT / "src" / "schema.py"
VALIDATOR = ROOT / "src" / "validator.py"

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

B6_PROBE = (
    B_ROOT
    / "results"
    / "capability_probe"
    / "B6_structured_output_capability_probe_v1.json"
)

TREATMENT_ADAPTER = (
    B_ROOT
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

HARNESS = (
    B_ROOT
    / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

B8_MANIFEST = (
    B_ROOT
    / "manifests"
    / "experiment_B_B8_harness_manifest_v1.json"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "benchmark":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",

    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "b6_probe":
        "fc509b6ca254882935ac30c32333ac510d465024a32bf9cbf48ed92239a36392",

    "treatment_adapter":
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    "harness":
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    "b8_manifest":
        "5b7498a584bed809ecc9398442e3c6df274c5c1310b8dd6a593a65d80fdd6fc6",
}

MODEL = "Qwen/Qwen3.5-9B"


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Complete frozen provenance
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

verify("runner", RUNNER, EXPECTED["runner"])
verify("benchmark", BENCHMARK, EXPECTED["benchmark"])
verify("historical adapter", ADAPTER, EXPECTED["adapter"])
verify("schema", SCHEMA, EXPECTED["schema"])
verify("validator", VALIDATOR, EXPECTED["validator"])
verify("Experiment-B preregistration", PREREG, EXPECTED["prereg"])
verify("B6 capability probe", B6_PROBE, EXPECTED["b6_probe"])
verify("B7B treatment adapter", TREATMENT_ADAPTER, EXPECTED["treatment_adapter"])
verify("B8 treatment harness", HARNESS, EXPECTED["harness"])
verify("B8 manifest", B8_MANIFEST, EXPECTED["b8_manifest"])


# ======================================================================================
# 2. Correct preregistration model lookup
# ======================================================================================

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

assert "semantic_model" in prereg, (
    "STOP: semantic_model section missing"
)

assert prereg["semantic_model"]["model"] == MODEL, (
    "STOP: preregistered semantic model mismatch\n"
    f"Expected: {MODEL}\n"
    f"Found: {prereg['semantic_model'].get('model')}"
)

print("✓ Preregistered semantic model verified:", MODEL)


# ======================================================================================
# 3. Verify treatment-arm identity
# ======================================================================================

treatment = prereg["treatment_arm"]

assert treatment["id"] == "B1"
assert (
    treatment["name"]
    == "structural_json_schema_constrained_generation"
)

print("✓ Treatment arm B1 verified")


# ======================================================================================
# 4. Verify treatment definition preserves comparison boundary
# ======================================================================================

definition = treatment["definition"]

required_definition_fragments = [
    "same semantic model",
    "same frozen CIR system prompt",
    "same user intent",
    "same temperature=0",
    "same downstream E3-v3 deterministic cascade",
]

for fragment in required_definition_fragments:

    assert fragment in definition, (
        "STOP: expected frozen treatment-definition "
        f"fragment missing: {fragment!r}"
    )

print("✓ Frozen control/treatment comparison boundary verified")


# ======================================================================================
# 5. Verify structural schema again
# ======================================================================================

structural_schema = treatment["structural_decoding_schema"]

EXPECTED_FIELDS = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert structural_schema["type"] == "object"

assert set(
    structural_schema["properties"]
) == EXPECTED_FIELDS

assert set(
    structural_schema["required"]
) == EXPECTED_FIELDS

assert (
    structural_schema["additionalProperties"]
    is False
)

FORBIDDEN = {
    "enum",
    "minimum",
    "maximum",
    "pattern",
    "minLength",
    "maxLength",
}

for field_name, field_schema in (
    structural_schema["properties"].items()
):

    leaked = (
        FORBIDDEN
        & set(field_schema.keys())
    )

    assert not leaked, (
        f"STOP: semantic/range restriction "
        f"found in {field_name}: {sorted(leaked)}"
    )

print("✓ Structural-only treatment schema verified")
print("✓ No semantic/range constraints moved into decoder")


# ======================================================================================
# 6. Verify B6 capability gate
# ======================================================================================

b6 = json.loads(
    B6_PROBE.read_text(encoding="utf-8")
)

assert (
    b6["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert b6["api_success"] is True
assert b6["parse_ok"] is True
assert b6["structural_schema_valid"] is True
assert b6["benchmark_case"] is False
assert b6["provider_calls_attempted"] == 1
assert b6["model"] == MODEL

print("✓ B6 provider capability gate satisfied")


# ======================================================================================
# 7. Verify benchmark cardinality
# ======================================================================================

df = pd.read_csv(BENCHMARK)

assert len(df) == 120, (
    f"STOP: expected 120 benchmark rows; found {len(df)}"
)

print("✓ Frozen benchmark contains exactly 120 cases")

print("Benchmark columns:")
print(list(df.columns))


# ======================================================================================
# 8. Show ground-truth distribution if available
# ======================================================================================

candidate_columns = [
    c
    for c in df.columns
    if c.lower() in {
        "expected_valid",
        "valid",
        "is_valid",
        "ground_truth_valid",
    }
]

if candidate_columns:

    gt_col = candidate_columns[0]

    print()
    print("Ground-truth column:", gt_col)
    print(
        df[gt_col]
        .value_counts(dropna=False)
        .to_string()
    )


# ======================================================================================
# 9. Lock treatment output path
# ======================================================================================

RESULT_DIR = (
    B_ROOT
    / "results"
    / "treatment_B1"
)

DRIVE_RESULT_DIR = (
    B_DRIVE
    / "results"
    / "treatment_B1"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT = (
    RESULT_DIR
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

DRIVE_OUTPUT = (
    DRIVE_RESULT_DIR
    / OUTPUT.name
)


# ======================================================================================
# 10. Hard rerun protection
# ======================================================================================

assert not OUTPUT.exists(), (
    "STOP: local B9 treatment output already exists.\n"
    "DO NOT RUN B9 AGAIN.\n"
    f"{OUTPUT}"
)

assert not DRIVE_OUTPUT.exists(), (
    "STOP: persistent B9 treatment output already exists.\n"
    "DO NOT RUN B9 AGAIN.\n"
    f"{DRIVE_OUTPUT}"
)

print("✓ No previous B9 treatment output detected")


# ======================================================================================
# 11. API-key readiness only
# ======================================================================================

if os.environ.get("TOGETHER_API_KEY"):

    print("✓ TOGETHER_API_KEY available (not displayed)")

else:

    print("⚠ TOGETHER_API_KEY currently absent")
    print("  No API call has been attempted.")


# ======================================================================================
# 12. Locked execution parameters
# ======================================================================================

print("\n" + "-" * 118)
print("LOCKED B9B PARAMETERS")
print("-" * 118)

print("Harness :", HARNESS)
print("Dataset :", BENCHMARK)
print("Output  :", OUTPUT)
print("Model   :", MODEL)
print("Rows    :", len(df))

print()
print("B9A-R1 DOES NOT EXECUTE THE HARNESS.")


# ======================================================================================
# 13. Final integrity recheck
# ======================================================================================

print("\n" + "-" * 118)
print("FINAL HASH RECHECK")
print("-" * 118)

verify("runner final", RUNNER, EXPECTED["runner"])
verify("benchmark final", BENCHMARK, EXPECTED["benchmark"])
verify("adapter final", ADAPTER, EXPECTED["adapter"])
verify("schema final", SCHEMA, EXPECTED["schema"])
verify("validator final", VALIDATOR, EXPECTED["validator"])
verify("preregistration final", PREREG, EXPECTED["prereg"])
verify("B6 probe final", B6_PROBE, EXPECTED["b6_probe"])
verify(
    "treatment adapter final",
    TREATMENT_ADAPTER,
    EXPECTED["treatment_adapter"]
)
verify(
    "treatment harness final",
    HARNESS,
    EXPECTED["harness"]
)


print("\n" + "=" * 118)
print("✓ STEP B9A-R1 COMPLETE")
print("✓ CORRECT PREREGISTERED MODEL LOCATION VERIFIED")
print("✓ TREATMENT ARM B1 VERIFIED")
print("✓ CONTROL/TREATMENT BOUNDARY VERIFIED")
print("✓ STRUCTURAL-ONLY SCHEMA VERIFIED")
print("✓ B6 CAPABILITY GATE VERIFIED")
print("✓ 120-CASE FROZEN BENCHMARK VERIFIED")
print("✓ NO PREVIOUS B9 OUTPUT DETECTED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK CASES EXECUTED")
print("✓ READY FOR ONE-TIME B9B EXECUTION")
print("✓ SEND THIS OUTPUT BEFORE RUNNING B9B")
print("=" * 118)

EXPERIMENT B — STEP B9A-R1: CORRECTED FINAL PREFLIGHT BEFORE ONE-TIME TREATMENT RUN

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ runner verified
✓ benchmark verified
✓ historical adapter verified
✓ schema verified
✓ validator verified
✓ Experiment-B preregistration verified
✓ B6 capability probe verified
✓ B7B treatment adapter verified
✓ B8 treatment harness verified
✓ B8 manifest verified
✓ Preregistered semantic model verified: Qwen/Qwen3.5-9B
✓ Treatment arm B1 verified
✓ Frozen control/treatment comparison boundary verified
✓ Structural-only treatment schema verified
✓ No semantic/range constraints moved into decoder
✓ B6 provider capability gate satisfied
✓ Frozen benchmark contains exactly 120 cases
Benchmark columns:
['id', 'family_id', 'intent', 'expected_valid', 'ex

In [35]:
from pathlib import Path
import hashlib
import json
import os
import sys
import subprocess
import shutil
import getpass
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B — STEP B9B: ONE-TIME 120-CASE STRUCTURED-DECODING TREATMENT EXECUTION")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

B_DRIVE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

RUNNER = (
    ROOT
    / "experiment-2b"
    / "implementation"
    / "e3_v3"
    / "run_experiment2b_e3v3_r7_dev_v1.py"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"
SCHEMA = ROOT / "src" / "schema.py"
VALIDATOR = ROOT / "src" / "validator.py"

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

B6_PROBE = (
    B_ROOT
    / "results"
    / "capability_probe"
    / "B6_structured_output_capability_probe_v1.json"
)

TREATMENT_ADAPTER = (
    B_ROOT
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

HARNESS = (
    B_ROOT
    / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

B8_MANIFEST = (
    B_ROOT
    / "manifests"
    / "experiment_B_B8_harness_manifest_v1.json"
)

EXPECTED = {
    "runner":
        "40c035cb864978c718a72fcfdfbb164d262529960b0ae78242d5194147cd8257",

    "benchmark":
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",

    "adapter":
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    "schema":
        "f26e771cc7dab2288a78187fe407282f88acc69f9c6fec1f1e781e1a81d5ab34",

    "validator":
        "c9e9daad54b5d31a6e4348aea168384eb841b0050fe501a633528df3770b6cb2",

    "prereg":
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    "b6_probe":
        "fc509b6ca254882935ac30c32333ac510d465024a32bf9cbf48ed92239a36392",

    "treatment_adapter":
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    "harness":
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    "b8_manifest":
        "5b7498a584bed809ecc9398442e3c6df274c5c1310b8dd6a593a65d80fdd6fc6",
}

MODEL = "Qwen/Qwen3.5-9B"


# ==================================================================================================
# Helpers
# ==================================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ==================================================================================================
# 1. LAST integrity gate before treatment execution
# ==================================================================================================

print("\n" + "-" * 118)
print("FINAL PRE-EXECUTION INTEGRITY GATE")
print("-" * 118)

verify("frozen runner", RUNNER, EXPECTED["runner"])
verify("frozen benchmark", BENCHMARK, EXPECTED["benchmark"])
verify("historical adapter", ADAPTER, EXPECTED["adapter"])
verify("schema", SCHEMA, EXPECTED["schema"])
verify("validator", VALIDATOR, EXPECTED["validator"])
verify("Experiment-B preregistration", PREREG, EXPECTED["prereg"])
verify("B6 capability probe", B6_PROBE, EXPECTED["b6_probe"])
verify("B7B treatment adapter", TREATMENT_ADAPTER, EXPECTED["treatment_adapter"])
verify("B8 treatment harness", HARNESS, EXPECTED["harness"])
verify("B8 manifest", B8_MANIFEST, EXPECTED["b8_manifest"])


# ==================================================================================================
# 2. Verify preregistered model and capability
# ==================================================================================================

prereg = json.loads(
    PREREG.read_text(encoding="utf-8")
)

assert (
    prereg["semantic_model"]["model"]
    == MODEL
)

b6 = json.loads(
    B6_PROBE.read_text(encoding="utf-8")
)

assert (
    b6["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

print("✓ Preregistered treatment identity verified")
print("✓ B6 capability gate verified")


# ==================================================================================================
# 3. Define immutable output locations
# ==================================================================================================

RESULT_DIR = (
    B_ROOT
    / "results"
    / "treatment_B1"
)

DRIVE_RESULT_DIR = (
    B_DRIVE
    / "results"
    / "treatment_B1"
)

MANIFEST_DIR = B_ROOT / "manifests"
DRIVE_MANIFEST_DIR = B_DRIVE / "manifests"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


OUTPUT = (
    RESULT_DIR
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

LOG = (
    RESULT_DIR
    / "experiment_B_B1_structured_treatment_120_v1.log"
)

EXECUTION_RECORD = (
    MANIFEST_DIR
    / "experiment_B_B9B_execution_record_v1.json"
)

DRIVE_OUTPUT = DRIVE_RESULT_DIR / OUTPUT.name
DRIVE_LOG = DRIVE_RESULT_DIR / LOG.name

DRIVE_EXECUTION_RECORD = (
    DRIVE_MANIFEST_DIR
    / EXECUTION_RECORD.name
)


# ==================================================================================================
# 4. HARD one-time-run protection
# ==================================================================================================

for path in [
    OUTPUT,
    LOG,
    EXECUTION_RECORD,
    DRIVE_OUTPUT,
    DRIVE_LOG,
    DRIVE_EXECUTION_RECORD,
]:
    assert not path.exists(), (
        "STOP: B9B evidence already exists.\n"
        "DO NOT rerun Experiment B treatment.\n"
        f"Existing artifact:\n{path}"
    )

print("✓ No prior B9B execution artifacts detected")


# ==================================================================================================
# 5. API key readiness
# ==================================================================================================

if not os.environ.get("TOGETHER_API_KEY"):

    os.environ["TOGETHER_API_KEY"] = getpass.getpass(
        "Enter TOGETHER_API_KEY (hidden): "
    )

assert os.environ.get("TOGETHER_API_KEY")

print("✓ TOGETHER_API_KEY available (not displayed)")


# ==================================================================================================
# 6. Build EXACT locked command
# ==================================================================================================

command = [
    sys.executable,
    str(HARNESS),

    "--dataset",
    str(BENCHMARK),

    "--out",
    str(OUTPUT),

    "--model",
    MODEL,
]

print("\n" + "-" * 118)
print("LOCKED EXECUTION")
print("-" * 118)

print("Model :", MODEL)
print("Cases : 120")
print("Output:", OUTPUT)

print("\nCommand:")
print(" ".join(command))

print()
print(">>> BEGINNING THE ONE-TIME B9B TREATMENT RUN")
print(">>> DO NOT INTERRUPT OR RERUN THIS EXPERIMENT")


# ==================================================================================================
# 7. Execute exactly once
#
# stdout/stderr are streamed AND retained.
# No retry logic exists.
# ==================================================================================================

started_utc = datetime.now(
    timezone.utc
).isoformat()

with open(
    LOG,
    "w",
    encoding="utf-8"
) as log_file:

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()


finished_utc = datetime.now(
    timezone.utc
).isoformat()

print()
print(">>> B9B PROCESS FINISHED")
print("Return code:", return_code)


# ==================================================================================================
# 8. Freeze execution record IMMEDIATELY — success or failure
# ==================================================================================================

execution_record = {
    "experiment":
        "Experiment B",

    "step":
        "B9B",

    "arm":
        "B1",

    "treatment":
        "provider-side structural JSON-schema constrained generation",

    "model":
        MODEL,

    "dataset":
        str(BENCHMARK),

    "dataset_sha256":
        EXPECTED["benchmark"],

    "frozen_runner_sha256":
        EXPECTED["runner"],

    "historical_adapter_sha256":
        EXPECTED["adapter"],

    "schema_sha256":
        EXPECTED["schema"],

    "validator_sha256":
        EXPECTED["validator"],

    "preregistration_sha256":
        EXPECTED["prereg"],

    "B6_probe_sha256":
        EXPECTED["b6_probe"],

    "treatment_adapter_sha256":
        EXPECTED["treatment_adapter"],

    "treatment_harness_sha256":
        EXPECTED["harness"],

    "command":
        command,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "return_code":
        return_code,

    "output_exists":
        OUTPUT.exists(),

    "log_exists":
        LOG.exists(),

    "rerun_allowed":
        False,
}


if OUTPUT.exists():
    execution_record["output_sha256"] = sha256(
        OUTPUT
    )

if LOG.exists():
    execution_record["log_sha256"] = sha256(
        LOG
    )


EXECUTION_RECORD.write_text(
    json.dumps(
        execution_record,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

EXECUTION_RECORD_SHA = sha256(
    EXECUTION_RECORD
)


# ==================================================================================================
# 9. Persist ALL observed evidence immediately
# ==================================================================================================

if OUTPUT.exists():
    shutil.copy2(
        OUTPUT,
        DRIVE_OUTPUT
    )

if LOG.exists():
    shutil.copy2(
        LOG,
        DRIVE_LOG
    )

shutil.copy2(
    EXECUTION_RECORD,
    DRIVE_EXECUTION_RECORD
)


if OUTPUT.exists():
    assert (
        sha256(DRIVE_OUTPUT)
        == sha256(OUTPUT)
    )

if LOG.exists():
    assert (
        sha256(DRIVE_LOG)
        == sha256(LOG)
    )

assert (
    sha256(DRIVE_EXECUTION_RECORD)
    == EXECUTION_RECORD_SHA
)

print("✓ Observed B9B evidence persisted to Drive")


# ==================================================================================================
# 10. Reverify protected components
# ==================================================================================================

print("\n" + "-" * 118)
print("POST-EXECUTION INTEGRITY")
print("-" * 118)

verify("runner after B9B", RUNNER, EXPECTED["runner"])
verify("benchmark after B9B", BENCHMARK, EXPECTED["benchmark"])
verify("historical adapter after B9B", ADAPTER, EXPECTED["adapter"])
verify("schema after B9B", SCHEMA, EXPECTED["schema"])
verify("validator after B9B", VALIDATOR, EXPECTED["validator"])
verify("preregistration after B9B", PREREG, EXPECTED["prereg"])
verify(
    "treatment adapter after B9B",
    TREATMENT_ADAPTER,
    EXPECTED["treatment_adapter"]
)
verify(
    "treatment harness after B9B",
    HARNESS,
    EXPECTED["harness"]
)


# ==================================================================================================
# 11. Report observed result — DO NOT RETRY ON FAILURE
# ==================================================================================================

print("\n" + "=" * 118)
print("B9B ONE-TIME EXECUTION RESULT")
print("=" * 118)

print("Return code :", return_code)
print("Output exists:", OUTPUT.exists())
print("Log exists   :", LOG.exists())

if OUTPUT.exists():
    print("\nOutput SHA-256:")
    print(sha256(OUTPUT))

if LOG.exists():
    print("\nLog SHA-256:")
    print(sha256(LOG))

print("\nExecution-record SHA-256:")
print(EXECUTION_RECORD_SHA)

print("\nExecution record:")
print(EXECUTION_RECORD)


print("\n" + "=" * 118)

if return_code == 0 and OUTPUT.exists():

    print("✓ STEP B9B EXECUTION COMPLETED")
    print("✓ ONE-TIME TREATMENT RUN RECORDED")
    print("✓ DO NOT RERUN B9B")
    print("✓ DO NOT MODIFY THE RAW RESULT")
    print("✓ NEXT STEP WILL ANALYZE/FREEZE THE OBSERVED RESULT")

else:

    print("⚠ STEP B9B ENDED WITHOUT A NORMAL SUCCESS RESULT")
    print("⚠ THIS OBSERVED ATTEMPT HAS BEEN FROZEN")
    print("⚠ DO NOT RERUN B9B")
    print("⚠ SEND THE COMPLETE OUTPUT FOR DIAGNOSIS")

print("=" * 118)

EXPERIMENT B — STEP B9B: ONE-TIME 120-CASE STRUCTURED-DECODING TREATMENT EXECUTION

----------------------------------------------------------------------------------------------------------------------
FINAL PRE-EXECUTION INTEGRITY GATE
----------------------------------------------------------------------------------------------------------------------
✓ frozen runner verified
✓ frozen benchmark verified
✓ historical adapter verified
✓ schema verified
✓ validator verified
✓ Experiment-B preregistration verified
✓ B6 capability probe verified
✓ B7B treatment adapter verified
✓ B8 treatment harness verified
✓ B8 manifest verified
✓ Preregistered treatment identity verified
✓ B6 capability gate verified
✓ No prior B9B execution artifacts detected
✓ TOGETHER_API_KEY available (not displayed)

----------------------------------------------------------------------------------------------------------------------
LOCKED EXECUTION
--------------------------------------------------------------

In [36]:
from pathlib import Path
import json
import hashlib
from collections import Counter

print("=" * 118)
print("EXPERIMENT B — STEP B10A: READ-ONLY INSPECTION AND PRIMARY-METRIC ANALYSIS")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

RESULT = (
    B_ROOT
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

EXEC_RECORD = (
    B_ROOT
    / "manifests"
    / "experiment_B_B9B_execution_record_v1.json"
)

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

EXPECTED_RESULT_SHA = (
    "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1"
)

EXPECTED_EXEC_SHA = (
    "4654ea991d50f65089f5be68d5bd565004a8bb768f5e106072fd6c50facfd7d5"
)

EXPECTED_PREREG_SHA = (
    "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), f"STOP: missing {label}: {path}"
    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Verify frozen evidence
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN B9B EVIDENCE")
print("-" * 118)

verify("B9B raw result", RESULT, EXPECTED_RESULT_SHA)
verify("B9B execution record", EXEC_RECORD, EXPECTED_EXEC_SHA)
verify("Experiment-B preregistration", PREREG, EXPECTED_PREREG_SHA)


# ======================================================================================
# 2. Load result WITHOUT modifying it
# ======================================================================================

data = json.loads(
    RESULT.read_text(encoding="utf-8")
)

print("\nTop-level JSON type:", type(data).__name__)

if isinstance(data, list):
    rows = data
    print("Rows:", len(rows))

elif isinstance(data, dict):
    print("Top-level keys:")
    for key in data:
        print("  -", key)

    # Detect likely row container without guessing silently.
    candidate_keys = [
        key
        for key, value in data.items()
        if isinstance(value, list)
        and len(value) == 120
    ]

    print("\n120-row list candidates:", candidate_keys)

    assert len(candidate_keys) == 1, (
        "STOP: unable to identify exactly one 120-row result container."
    )

    rows = data[candidate_keys[0]]

else:
    raise AssertionError(
        f"STOP: unsupported result type: {type(data)}"
    )


assert len(rows) == 120, (
    f"STOP: expected 120 result rows; found {len(rows)}"
)

print("✓ Exactly 120 observed treatment rows loaded")


# ======================================================================================
# 3. Inspect exact row schema
# ======================================================================================

assert all(isinstance(r, dict) for r in rows)

key_sets = Counter(
    tuple(sorted(r.keys()))
    for r in rows
)

print("\n" + "-" * 118)
print("OBSERVED ROW SCHEMAS")
print("-" * 118)

print("Distinct row schemas:", len(key_sets))

for i, (keys, count) in enumerate(
    key_sets.items(),
    start=1
):
    print(f"\nSchema #{i} — rows: {count}")
    for key in keys:
        print("  -", key)


# ======================================================================================
# 4. Print first row structurally
# ======================================================================================

print("\n" + "-" * 118)
print("FIRST OBSERVED ROW")
print("-" * 118)

print(
    json.dumps(
        rows[0],
        indent=2,
        ensure_ascii=False
    )
)


# ======================================================================================
# 5. Identify important fields by exact observed names
# ======================================================================================

all_keys = set()

for row in rows:
    all_keys.update(row.keys())


def candidates(names):
    return [
        name
        for name in names
        if name in all_keys
    ]


field_candidates = {
    "id": candidates([
        "id",
        "case_id",
        "test_id",
    ]),

    "expected_valid": candidates([
        "expected_valid",
        "ground_truth_valid",
        "valid_expected",
    ]),

    "predicted_valid": candidates([
        "predicted_valid",
        "accepted",
        "is_valid",
        "valid",
    ]),

    "llm_invoked": candidates([
        "llm_invoked",
        "llm_called",
    ]),

    "parse_ok": candidates([
        "parse_ok",
        "llm_parse_ok",
    ]),

    "descriptor": candidates([
        "descriptor",
        "slice_descriptor",
        "mapped_descriptor",
    ]),

    "rejection_stage": candidates([
        "rejection_stage",
        "reject_stage",
        "failure_stage",
    ]),
}


print("\n" + "-" * 118)
print("FIELD DISCOVERY")
print("-" * 118)

for role, found in field_candidates.items():
    print(f"{role:20s}: {found}")


# ======================================================================================
# 6. Require unambiguous decision fields before computing metrics
# ======================================================================================

assert len(field_candidates["expected_valid"]) == 1, (
    "STOP: expected_valid field is not uniquely identifiable."
)

assert len(field_candidates["predicted_valid"]) == 1, (
    "STOP: predicted_valid field is not uniquely identifiable."
)

EXPECTED_FIELD = field_candidates["expected_valid"][0]
PREDICTED_FIELD = field_candidates["predicted_valid"][0]

print()
print("Expected-valid field :", EXPECTED_FIELD)
print("Predicted-valid field:", PREDICTED_FIELD)


# ======================================================================================
# 7. Normalize booleans conservatively
# ======================================================================================

def as_bool(value):
    if isinstance(value, bool):
        return value

    if value in (0, 1):
        return bool(value)

    if isinstance(value, str):
        x = value.strip().lower()

        if x in {"true", "1", "yes"}:
            return True

        if x in {"false", "0", "no"}:
            return False

    raise ValueError(
        f"Cannot safely interpret as boolean: {value!r}"
    )


expected = [
    as_bool(r[EXPECTED_FIELD])
    for r in rows
]

predicted = [
    as_bool(r[PREDICTED_FIELD])
    for r in rows
]


# ======================================================================================
# 8. Confusion matrix
#
# Historical convention:
# "invalid/reject" is the positive class.
#
# TP = expected invalid and predicted invalid
# FP = expected valid but predicted invalid
# FN = expected invalid but predicted valid
# TN = expected valid and predicted valid
# ======================================================================================

TP = sum(
    (not e) and (not p)
    for e, p in zip(expected, predicted)
)

FP = sum(
    e and (not p)
    for e, p in zip(expected, predicted)
)

FN = sum(
    (not e) and p
    for e, p in zip(expected, predicted)
)

TN = sum(
    e and p
    for e, p in zip(expected, predicted)
)

assert TP + FP + FN + TN == 120

N_VALID = sum(expected)
N_INVALID = 120 - N_VALID

assert N_VALID == 58, (
    f"STOP: expected 58 valid cases; observed {N_VALID}"
)

assert N_INVALID == 62, (
    f"STOP: expected 62 invalid cases; observed {N_INVALID}"
)


# ======================================================================================
# 9. Primary preregistered metrics
# ======================================================================================

accuracy = (TP + TN) / 120

# Valid Acceptance Rate:
# expected-valid cases correctly accepted.
VAR = TN / N_VALID

# Invalid Rejection Rate:
# expected-invalid cases correctly rejected.
IRR = TP / N_INVALID


print("\n" + "-" * 118)
print("PRIMARY TREATMENT RESULTS")
print("-" * 118)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")

print()
print(f"Decision accuracy : {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"VAR               : {VAR:.6f} ({VAR*100:.2f}%)")
print(f"IRR               : {IRR:.6f} ({IRR*100:.2f}%)")


# ======================================================================================
# 10. Compare descriptively with frozen historical B0/A1 baseline
# ======================================================================================

BASELINE = {
    "TP": 50,
    "FP": 7,
    "FN": 12,
    "TN": 51,
    "accuracy": 101 / 120,
    "VAR": 51 / 58,
    "IRR": 50 / 62,
}

print("\n" + "-" * 118)
print("DESCRIPTIVE B0 vs B1 COMPARISON")
print("-" * 118)

print(
    f"B0 accuracy : {BASELINE['accuracy']*100:.2f}%"
)
print(
    f"B1 accuracy : {accuracy*100:.2f}%"
)
print(
    f"Difference  : {(accuracy-BASELINE['accuracy'])*100:+.2f} pp"
)

print()

print(
    f"B0 VAR      : {BASELINE['VAR']*100:.2f}%"
)
print(
    f"B1 VAR      : {VAR*100:.2f}%"
)
print(
    f"Difference  : {(VAR-BASELINE['VAR'])*100:+.2f} pp"
)

print()

print(
    f"B0 IRR      : {BASELINE['IRR']*100:.2f}%"
)
print(
    f"B1 IRR      : {IRR*100:.2f}%"
)
print(
    f"Difference  : {(IRR-BASELINE['IRR'])*100:+.2f} pp"
)


# ======================================================================================
# 11. Optional directly-observable generation statistics
# ======================================================================================

print("\n" + "-" * 118)
print("DIRECTLY OBSERVABLE AUXILIARY COUNTS")
print("-" * 118)

if len(field_candidates["llm_invoked"]) == 1:

    field = field_candidates["llm_invoked"][0]

    invoked = sum(
        as_bool(r[field])
        for r in rows
    )

    print(f"LLM invoked : {invoked}/120")

else:
    print("LLM invoked : field not uniquely available")


if len(field_candidates["parse_ok"]) == 1:

    field = field_candidates["parse_ok"][0]

    values = [
        r[field]
        for r in rows
        if r[field] is not None
    ]

    parse_ok_count = sum(
        as_bool(v)
        for v in values
    )

    print(
        f"Parse OK    : {parse_ok_count}/{len(values)} "
        f"among rows with recorded parse status"
    )

else:
    print("Parse OK    : field not uniquely available")


if len(field_candidates["rejection_stage"]) == 1:

    field = field_candidates["rejection_stage"][0]

    counts = Counter(
        r.get(field)
        for r in rows
    )

    print("\nRejection-stage distribution:")

    for stage, count in counts.most_common():
        print(f"  {stage!r}: {count}")


# ======================================================================================
# 12. Reverify raw evidence after analysis
# ======================================================================================

print("\n" + "-" * 118)
print("POST-ANALYSIS RAW-EVIDENCE INTEGRITY")
print("-" * 118)

verify(
    "B9B raw result after analysis",
    RESULT,
    EXPECTED_RESULT_SHA
)

verify(
    "B9B execution record after analysis",
    EXEC_RECORD,
    EXPECTED_EXEC_SHA
)

verify(
    "Experiment-B preregistration after analysis",
    PREREG,
    EXPECTED_PREREG_SHA
)


print("\n" + "=" * 118)
print("✓ STEP B10A COMPLETE")
print("✓ RAW B9B RESULT READ ONLY")
print("✓ EXACT ROW STRUCTURE INSPECTED")
print("✓ PRIMARY B1 METRICS COMPUTED")
print("✓ HISTORICAL B0 COMPARISON PRINTED")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO RAW RESULT MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE B10B")
print("=" * 118)

EXPERIMENT B — STEP B10A: READ-ONLY INSPECTION AND PRIMARY-METRIC ANALYSIS

----------------------------------------------------------------------------------------------------------------------
FROZEN B9B EVIDENCE
----------------------------------------------------------------------------------------------------------------------
✓ B9B raw result verified
✓ B9B execution record verified
✓ Experiment-B preregistration verified


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [37]:
from pathlib import Path
import hashlib
import os

print("=" * 118)
print("EXPERIMENT B — STEP B10A-D1: DIAGNOSE RAW RESULT SERIALIZATION")
print("=" * 118)

RESULT = Path(
    "/content/deterministic-intent-translation-journal/"
    "journal_experiments/"
    "experiment_B_schema_constrained_decoding/"
    "results/treatment_B1/"
    "experiment_B_B1_structured_treatment_120_v1.json"
)

EXPECTED_SHA = (
    "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 1. Verify frozen raw evidence
# ======================================================================================

assert RESULT.exists(), f"STOP: result missing: {RESULT}"

actual_sha = sha256(RESULT)

assert actual_sha == EXPECTED_SHA, (
    "STOP: raw result hash mismatch\n"
    f"Expected: {EXPECTED_SHA}\n"
    f"Actual:   {actual_sha}"
)

print("✓ B9B raw result verified")
print("SHA-256:", actual_sha)


# ======================================================================================
# 2. Basic byte-level information
# ======================================================================================

size = RESULT.stat().st_size

print("\nFile size:", size, "bytes")

assert size > 0, "STOP: result file is empty"

with open(RESULT, "rb") as f:
    prefix = f.read(256)

print("\nFirst 256 bytes — repr():")
print(repr(prefix))


# ======================================================================================
# 3. Text-level structure — NO JSON parsing yet
# ======================================================================================

with open(
    RESULT,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    lines = f.readlines()

print("\nPhysical line count:", len(lines))

nonempty = [
    (i + 1, line.rstrip("\r\n"))
    for i, line in enumerate(lines)
    if line.strip()
]

print("Non-empty line count:", len(nonempty))


# ======================================================================================
# 4. Show only bounded structural samples
# ======================================================================================

print("\n" + "-" * 118)
print("FIRST FIVE NON-EMPTY LINES — BOUNDED PREVIEW")
print("-" * 118)

for line_no, text in nonempty[:5]:

    preview = text[:500]

    print(f"\nLine {line_no}:")
    print(repr(preview))

    if len(text) > 500:
        print(f"... [truncated; total characters={len(text)}]")


print("\n" + "-" * 118)
print("LAST THREE NON-EMPTY LINES — BOUNDED PREVIEW")
print("-" * 118)

for line_no, text in nonempty[-3:]:

    preview = text[:500]

    print(f"\nLine {line_no}:")
    print(repr(preview))

    if len(text) > 500:
        print(f"... [truncated; total characters={len(text)}]")


# ======================================================================================
# 5. Character-level classification
# ======================================================================================

with open(
    RESULT,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    text = f.read()

stripped = text.lstrip()

print("\n" + "-" * 118)
print("SERIALIZATION CLUES")
print("-" * 118)

print("Leading character:", repr(stripped[:1]))
print("Trailing character:", repr(text.rstrip()[-1:]))

print(
    "Starts with '[':",
    stripped.startswith("[")
)

print(
    "Starts with '{':",
    stripped.startswith("{")
)

print(
    "First non-empty line starts with '{':",
    bool(nonempty and nonempty[0][1].lstrip().startswith("{"))
)

print(
    "Non-empty lines:",
    len(nonempty)
)


# ======================================================================================
# 6. Check for CSV-like header clues
# ======================================================================================

if nonempty:

    first_line = nonempty[0][1]

    print("\nFirst-line delimiter counts:")
    print("  comma :", first_line.count(","))
    print("  tab   :", first_line.count("\t"))
    print("  pipe  :", first_line.count("|"))


# ======================================================================================
# 7. Final integrity check
# ======================================================================================

assert sha256(RESULT) == EXPECTED_SHA

print("\n" + "=" * 118)
print("✓ STEP B10A-D1 COMPLETE")
print("✓ RAW RESULT REMAINS BYTE-FOR-BYTE UNCHANGED")
print("✓ NO JSON/JSONL/CSV FORMAT ASSUMPTION MADE")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS OUTPUT BEFORE CONTINUING B10A")
print("=" * 118)

EXPERIMENT B — STEP B10A-D1: DIAGNOSE RAW RESULT SERIALIZATION
✓ B9B raw result verified
SHA-256: 40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1

File size: 382096 bytes

First 256 bytes — repr():
b'id,intent,family_id,repair_id,subcategory,difficulty,linguistic_form,constraint_type,control_type,notes,expected_valid,expected_service,pipeline,accepted,decision_correct,llm_invoked,parse_ok,llm_latency_ms,end_to_end_latency_ms,finish_reason,rejection_sta'

Physical line count: 737
Non-empty line count: 737

----------------------------------------------------------------------------------------------------------------------
FIRST FIVE NON-EMPTY LINES — BOUNDED PREVIEW
----------------------------------------------------------------------------------------------------------------------

Line 1:
'id,intent,family_id,repair_id,subcategory,difficulty,linguistic_form,constraint_type,control_type,notes,expected_valid,expected_service,pipeline,accepted,decision_correct,llm_invo

In [38]:
from pathlib import Path
import hashlib
import pandas as pd
from collections import Counter

print("=" * 118)
print("EXPERIMENT B — STEP B10A-R1: READ-ONLY CSV ANALYSIS OF B9B TREATMENT RESULT")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

B_ROOT = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

RESULT = (
    B_ROOT
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

EXEC_RECORD = (
    B_ROOT
    / "manifests"
    / "experiment_B_B9B_execution_record_v1.json"
)

PREREG = (
    B_ROOT
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

EXPECTED_RESULT_SHA = (
    "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1"
)

EXPECTED_EXEC_SHA = (
    "4654ea991d50f65089f5be68d5bd565004a8bb768f5e106072fd6c50facfd7d5"
)

EXPECTED_PREREG_SHA = (
    "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5"
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(label, path, expected):
    assert path.exists(), f"STOP: missing {label}: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Verify frozen evidence
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN EVIDENCE")
print("-" * 118)

verify("B9B raw result", RESULT, EXPECTED_RESULT_SHA)
verify("B9B execution record", EXEC_RECORD, EXPECTED_EXEC_SHA)
verify("Experiment-B preregistration", PREREG, EXPECTED_PREREG_SHA)


# ======================================================================================
# 2. Parse using RFC-style CSV handling
#
# Pandas/Python CSV parser correctly handles quoted multiline fields.
# ======================================================================================

df = pd.read_csv(
    RESULT,
    dtype=None,
    keep_default_na=True,
)

print("\n✓ Raw artifact parsed as CSV")
print("Logical rows   :", len(df))
print("Logical columns:", len(df.columns))

assert len(df) == 120, (
    f"STOP: expected exactly 120 logical rows; found {len(df)}"
)

assert df["id"].nunique() == 120, (
    "STOP: expected 120 unique case IDs"
)

expected_ids = {
    f"HO{i:03d}"
    for i in range(1, 121)
}

observed_ids = set(
    df["id"].astype(str)
)

assert observed_ids == expected_ids, (
    "STOP: observed case-ID set differs from HO001-HO120"
)

print("✓ Exactly 120 logical benchmark records recovered")
print("✓ Exactly 120 unique IDs recovered")
print("✓ Case-ID set is HO001-HO120")


# ======================================================================================
# 3. Display exact column schema
# ======================================================================================

print("\n" + "-" * 118)
print("OBSERVED RESULT COLUMNS")
print("-" * 118)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


# ======================================================================================
# 4. Verify required decision fields
# ======================================================================================

REQUIRED = {
    "id",
    "expected_valid",
    "accepted",
    "decision_correct",
    "llm_invoked",
    "parse_ok",
    "rejection_stage",
}

missing = REQUIRED - set(df.columns)

assert not missing, (
    f"STOP: required result columns missing: {sorted(missing)}"
)

print("\n✓ Required decision-analysis fields present")


# ======================================================================================
# 5. Conservative boolean normalization
# ======================================================================================

def as_bool(value, field, case_id):

    if isinstance(value, bool):
        return value

    if value in (0, 1):
        return bool(value)

    if isinstance(value, str):

        x = value.strip().lower()

        if x in {"true", "1", "yes"}:
            return True

        if x in {"false", "0", "no"}:
            return False

    raise ValueError(
        f"Cannot interpret {field} as boolean "
        f"for {case_id}: {value!r}"
    )


expected = []

accepted = []

decision_correct_recorded = []

for _, row in df.iterrows():

    case_id = str(row["id"])

    expected.append(
        as_bool(
            row["expected_valid"],
            "expected_valid",
            case_id
        )
    )

    accepted.append(
        as_bool(
            row["accepted"],
            "accepted",
            case_id
        )
    )

    decision_correct_recorded.append(
        as_bool(
            row["decision_correct"],
            "decision_correct",
            case_id
        )
    )


# ======================================================================================
# 6. Verify benchmark class distribution
# ======================================================================================

N_VALID = sum(expected)
N_INVALID = len(expected) - N_VALID

assert N_VALID == 58, (
    f"STOP: expected 58 valid cases; found {N_VALID}"
)

assert N_INVALID == 62, (
    f"STOP: expected 62 invalid cases; found {N_INVALID}"
)

print("✓ Ground truth distribution = 58 valid / 62 invalid")


# ======================================================================================
# 7. Independently reconstruct decision correctness
# ======================================================================================

decision_correct_recomputed = [
    e == a
    for e, a in zip(expected, accepted)
]

mismatch_indices = [
    i
    for i, (recorded, recomputed) in enumerate(
        zip(
            decision_correct_recorded,
            decision_correct_recomputed
        )
    )
    if recorded != recomputed
]

assert not mismatch_indices, (
    "STOP: recorded decision_correct disagrees with "
    f"expected_valid == accepted for {len(mismatch_indices)} cases"
)

print("✓ Recorded decision_correct independently verified")


# ======================================================================================
# 8. Confusion matrix
#
# Preserve historical convention:
# invalid/reject = positive class
#
# TP = expected invalid, rejected
# FP = expected valid, rejected
# FN = expected invalid, accepted
# TN = expected valid, accepted
# ======================================================================================

TP = sum(
    (not e) and (not a)
    for e, a in zip(expected, accepted)
)

FP = sum(
    e and (not a)
    for e, a in zip(expected, accepted)
)

FN = sum(
    (not e) and a
    for e, a in zip(expected, accepted)
)

TN = sum(
    e and a
    for e, a in zip(expected, accepted)
)

assert TP + FP + FN + TN == 120

assert TN + FP == 58
assert TP + FN == 62


# ======================================================================================
# 9. Primary treatment metrics
# ======================================================================================

accuracy = (TP + TN) / 120
VAR = TN / 58
IRR = TP / 62

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else float("nan")
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else float("nan")
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else float("nan")
)


print("\n" + "-" * 118)
print("B1 PRIMARY DECISION RESULTS")
print("-" * 118)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")

print()

print(
    f"Decision accuracy : "
    f"{accuracy:.6f} ({accuracy*100:.2f}%)"
)

print(
    f"VAR               : "
    f"{VAR:.6f} ({VAR*100:.2f}%)"
)

print(
    f"IRR               : "
    f"{IRR:.6f} ({IRR*100:.2f}%)"
)

print(
    f"Precision         : "
    f"{precision:.6f}"
)

print(
    f"Recall            : "
    f"{recall:.6f}"
)

print(
    f"F1                : "
    f"{f1:.6f}"
)


# ======================================================================================
# 10. Compare with frozen B0 historical baseline
# ======================================================================================

BASELINE = {
    "TP": 50,
    "FP": 7,
    "FN": 12,
    "TN": 51,
}

BASELINE["accuracy"] = (
    BASELINE["TP"] + BASELINE["TN"]
) / 120

BASELINE["VAR"] = (
    BASELINE["TN"] / 58
)

BASELINE["IRR"] = (
    BASELINE["TP"] / 62
)


print("\n" + "-" * 118)
print("DESCRIPTIVE B0 vs B1 COMPARISON")
print("-" * 118)

print(
    f"{'Metric':<20}"
    f"{'B0':>12}"
    f"{'B1':>12}"
    f"{'Δ pp':>12}"
)

print("-" * 56)

for name, b0, b1 in [
    (
        "Accuracy",
        BASELINE["accuracy"],
        accuracy,
    ),
    (
        "VAR",
        BASELINE["VAR"],
        VAR,
    ),
    (
        "IRR",
        BASELINE["IRR"],
        IRR,
    ),
]:

    print(
        f"{name:<20}"
        f"{b0*100:>11.2f}%"
        f"{b1*100:>11.2f}%"
        f"{(b1-b0)*100:>+11.2f}"
    )


# ======================================================================================
# 11. LLM invocation statistics
# ======================================================================================

llm_invoked = []

for _, row in df.iterrows():

    llm_invoked.append(
        as_bool(
            row["llm_invoked"],
            "llm_invoked",
            str(row["id"])
        )
    )

N_LLM = sum(llm_invoked)

print("\n" + "-" * 118)
print("LLM / STRUCTURED-GENERATION OBSERVATIONS")
print("-" * 118)

print(f"LLM invoked: {N_LLM}/120")


# parse_ok can legitimately be NaN when LLM was not invoked.

parse_ok_invoked = []

for _, row in df.iterrows():

    invoked = as_bool(
        row["llm_invoked"],
        "llm_invoked",
        str(row["id"])
    )

    if not invoked:
        continue

    value = row["parse_ok"]

    assert not pd.isna(value), (
        "STOP: LLM-invoked case has missing parse_ok: "
        f"{row['id']}"
    )

    parse_ok_invoked.append(
        as_bool(
            value,
            "parse_ok",
            str(row["id"])
        )
    )

N_PARSE_OK = sum(parse_ok_invoked)

print(
    f"Parse OK among LLM-invoked cases: "
    f"{N_PARSE_OK}/{len(parse_ok_invoked)}"
)

if len(parse_ok_invoked):

    print(
        "Structured-generation parse success: "
        f"{N_PARSE_OK/len(parse_ok_invoked)*100:.2f}%"
    )


# ======================================================================================
# 12. Rejection-stage distribution
# ======================================================================================

print("\n" + "-" * 118)
print("REJECTION-STAGE DISTRIBUTION")
print("-" * 118)

stage_counts = Counter()

for value in df["rejection_stage"]:

    if pd.isna(value):
        stage = "<accepted/no-rejection-stage>"
    else:
        stage = str(value)

    stage_counts[stage] += 1


for stage, count in stage_counts.most_common():
    print(f"{stage:<40} {count:>4}")


# ======================================================================================
# 13. Error-code distribution if present
# ======================================================================================

if "error_codes_json" in df.columns:

    print("\n" + "-" * 118)
    print("RAW ERROR-CODE FIELD — NONEMPTY ROW COUNT")
    print("-" * 118)

    nonempty_errors = 0

    for value in df["error_codes_json"]:

        if pd.isna(value):
            continue

        text = str(value).strip()

        if text not in {
            "",
            "[]",
            "null",
            "None",
        }:
            nonempty_errors += 1

    print(
        "Rows with non-empty error_codes_json:",
        nonempty_errors
    )


# ======================================================================================
# 14. Raw-result integrity after analysis
# ======================================================================================

print("\n" + "-" * 118)
print("POST-ANALYSIS RAW-EVIDENCE INTEGRITY")
print("-" * 118)

verify(
    "B9B raw result after analysis",
    RESULT,
    EXPECTED_RESULT_SHA
)

verify(
    "B9B execution record after analysis",
    EXEC_RECORD,
    EXPECTED_EXEC_SHA
)

verify(
    "Experiment-B preregistration after analysis",
    PREREG,
    EXPECTED_PREREG_SHA
)


print("\n" + "=" * 118)
print("✓ STEP B10A-R1 COMPLETE")
print("✓ RAW B9B ARTIFACT CORRECTLY PARSED AS CSV")
print("✓ 120 LOGICAL RECORDS VERIFIED")
print("✓ CASE IDs HO001-HO120 VERIFIED")
print("✓ DECISION_CORRECT INDEPENDENTLY VERIFIED")
print("✓ B1 PRIMARY METRICS COMPUTED")
print("✓ B0/B1 DESCRIPTIVE COMPARISON COMPUTED")
print("✓ STRUCTURED-GENERATION PARSE SUCCESS COMPUTED")
print("✓ RAW B9B RESULT REMAINS UNCHANGED")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE B10B")
print("=" * 118)

EXPERIMENT B — STEP B10A-R1: READ-ONLY CSV ANALYSIS OF B9B TREATMENT RESULT

----------------------------------------------------------------------------------------------------------------------
FROZEN EVIDENCE
----------------------------------------------------------------------------------------------------------------------
✓ B9B raw result verified
✓ B9B execution record verified
✓ Experiment-B preregistration verified

✓ Raw artifact parsed as CSV
Logical rows   : 120
Logical columns: 41
✓ Exactly 120 logical benchmark records recovered
✓ Exactly 120 unique IDs recovered
✓ Case-ID set is HO001-HO120

----------------------------------------------------------------------------------------------------------------------
OBSERVED RESULT COLUMNS
----------------------------------------------------------------------------------------------------------------------
01. id
02. intent
03. family_id
04. repair_id
05. subcategory
06. difficulty
07. linguistic_form
08. constraint_type
09. co

In [39]:
from pathlib import Path
import hashlib

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR1: DISCOVER FROZEN E1 / E2A IMPLEMENTATIONS AND EVIDENCE")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

assert ROOT.exists(), f"STOP: project root missing: {ROOT}"

# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 1. Search for E1-related files
# ======================================================================================

print("\n" + "-" * 118)
print("E1 CANDIDATE FILES")
print("-" * 118)

e1_candidates = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    rel = str(p.relative_to(ROOT)).lower()

    # Avoid caches / git internals.
    if (
        "__pycache__" in rel
        or "/.git/" in f"/{rel}"
    ):
        continue

    name = p.name.lower()

    if (
        "experiment1" in name
        or "experiment_1" in name
        or "experiment-1" in name
        or "e1_" in name
        or name.startswith("e1.")
        or name.startswith("e1_")
    ):
        e1_candidates.append(p)


print("Candidates found:", len(e1_candidates))

for p in sorted(e1_candidates)[:150]:
    print(p.relative_to(ROOT))

if len(e1_candidates) > 150:
    print(f"... {len(e1_candidates)-150} additional candidates omitted")


# ======================================================================================
# 2. Search for E2A-related files
# ======================================================================================

print("\n" + "-" * 118)
print("E2A CANDIDATE FILES")
print("-" * 118)

e2a_candidates = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    rel = str(p.relative_to(ROOT)).lower()

    if (
        "__pycache__" in rel
        or "/.git/" in f"/{rel}"
    ):
        continue

    name = p.name.lower()

    if (
        "e2a" in name
        or "experiment2a" in name
        or "experiment_2a" in name
        or "experiment-2a" in name
    ):
        e2a_candidates.append(p)


print("Candidates found:", len(e2a_candidates))

for p in sorted(e2a_candidates)[:150]:
    print(p.relative_to(ROOT))

if len(e2a_candidates) > 150:
    print(f"... {len(e2a_candidates)-150} additional candidates omitted")


# ======================================================================================
# 3. Search specifically for likely raw / primary / metrics / CI evidence
# ======================================================================================

print("\n" + "-" * 118)
print("LIKELY HISTORICAL EVIDENCE")
print("-" * 118)

keywords = [
    "raw",
    "primary",
    "metric",
    "ci",
    "manifest",
    "latency",
    "paired",
]

evidence = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    rel = str(p.relative_to(ROOT))
    low = rel.lower()

    if (
        "__pycache__" in low
        or "/.git/" in f"/{low}"
    ):
        continue

    e1_hit = (
        "e1_" in low
        or "/e1/" in low
        or "experiment1" in low
        or "experiment_1" in low
        or "experiment-1" in low
    )

    e2a_hit = (
        "e2a" in low
        or "experiment2a" in low
        or "experiment_2a" in low
        or "experiment-2a" in low
    )

    evidence_hit = any(
        k in p.name.lower()
        for k in keywords
    )

    if (e1_hit or e2a_hit) and evidence_hit:
        evidence.append(p)


print("Evidence candidates:", len(evidence))

for p in sorted(evidence)[:200]:

    size = p.stat().st_size

    print(
        f"{str(p.relative_to(ROOT)):<100} "
        f"{size:>12} bytes"
    )

if len(evidence) > 200:
    print(f"... {len(evidence)-200} additional candidates omitted")


# ======================================================================================
# 4. Search Python files for historical experiment logic
# ======================================================================================

print("\n" + "-" * 118)
print("PYTHON IMPLEMENTATIONS REFERENCING E1 / E2A")
print("-" * 118)

python_hits = []

for p in ROOT.rglob("*.py"):

    rel = str(p.relative_to(ROOT))

    if (
        "__pycache__" in rel
        or "/.git/" in f"/{rel}"
    ):
        continue

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    low = text.lower()

    terms = []

    if "e1" in low and (
        "experiment" in low
        or "pipeline" in low
    ):
        terms.append("E1")

    if "e2a" in low:
        terms.append("E2A")

    if terms:
        python_hits.append(
            (p, terms)
        )


print("Python candidates:", len(python_hits))

for p, terms in python_hits[:150]:
    print(
        f"[{','.join(terms):7s}] "
        f"{p.relative_to(ROOT)}"
    )

if len(python_hits) > 150:
    print(f"... {len(python_hits)-150} additional candidates omitted")


# ======================================================================================
# 5. Search CSV/JSON files with 50 or 200 logical records
#
# READ ONLY.
# This does not modify anything.
# ======================================================================================

print("\n" + "-" * 118)
print("POSSIBLE 50 / 200 CASE DATASETS OR RESULTS")
print("-" * 118)

import pandas as pd
import json

population_candidates = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in {
        ".csv",
        ".json",
        ".jsonl",
    }:
        continue

    rel = str(p.relative_to(ROOT))

    if (
        "__pycache__" in rel
        or "/.git/" in f"/{rel}"
    ):
        continue

    count = None
    fmt = None

    # --------------------------------------------------
    # CSV
    # --------------------------------------------------

    if p.suffix.lower() == ".csv":

        try:
            df = pd.read_csv(p)

            count = len(df)
            fmt = "CSV"

        except Exception:
            pass

    # --------------------------------------------------
    # JSON / JSONL
    # --------------------------------------------------

    elif p.suffix.lower() in {
        ".json",
        ".jsonl",
    }:

        try:
            obj = json.loads(
                p.read_text(
                    encoding="utf-8",
                    errors="strict"
                )
            )

            if isinstance(obj, list):
                count = len(obj)
                fmt = "JSON-list"

            elif isinstance(obj, dict):

                list_lengths = [
                    len(v)
                    for v in obj.values()
                    if isinstance(v, list)
                ]

                if list_lengths:
                    # Only informational.
                    if 50 in list_lengths:
                        count = 50
                    elif 200 in list_lengths:
                        count = 200

                    if count is not None:
                        fmt = "JSON-object/list-field"

        except Exception:

            # Try JSONL.
            try:
                rows = []

                with open(
                    p,
                    "r",
                    encoding="utf-8"
                ) as f:

                    for line in f:

                        if not line.strip():
                            continue

                        rows.append(
                            json.loads(line)
                        )

                count = len(rows)
                fmt = "JSONL"

            except Exception:
                pass

    if count in {50, 200}:

        population_candidates.append(
            (
                p,
                count,
                fmt,
                p.stat().st_size,
            )
        )


for p, count, fmt, size in population_candidates:

    print(
        f"{count:>3} rows | "
        f"{fmt:<22} | "
        f"{size:>10} bytes | "
        f"{p.relative_to(ROOT)}"
    )


print(
    "\nTotal 50/200-record candidates:",
    len(population_candidates)
)


# ======================================================================================
# 6. Hash the most obviously named historical evidence candidates
# ======================================================================================

print("\n" + "-" * 118)
print("HASHES OF LIKELY E1 / E2A EVIDENCE")
print("-" * 118)

for p in sorted(evidence)[:100]:

    try:
        digest = sha256(p)

        print(
            digest,
            " ",
            p.relative_to(ROOT)
        )

    except Exception as exc:

        print(
            "HASH-ERROR",
            p.relative_to(ROOT),
            repr(exc)
        )


# ======================================================================================
# 7. Integrity statement
# ======================================================================================

print("\n" + "=" * 118)
print("✓ STEP BR1 COMPLETE")
print("✓ E1 CANDIDATES DISCOVERED")
print("✓ E2A CANDIDATES DISCOVERED")
print("✓ HISTORICAL EVIDENCE CANDIDATES DISCOVERED")
print("✓ 50/200-RECORD ARTIFACTS INSPECTED READ-ONLY")
print("✓ NO API/MODEL CALL")
print("✓ NO EXPERIMENT EXECUTION")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE STEP BR2")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR1: DISCOVER FROZEN E1 / E2A IMPLEMENTATIONS AND EVIDENCE

----------------------------------------------------------------------------------------------------------------------
E1 CANDIDATE FILES
----------------------------------------------------------------------------------------------------------------------
Candidates found: 7
experiment-2b/heldout/analysis/publication_figures/figure1_family_accuracy.pdf
experiment-2b/heldout/analysis/publication_figures/figure1_family_accuracy.png
experiment-2b/heldout/analysis/publication_tables/table1_overall_performance.csv
experiment-2b/reproducibility/e3v3_final_experiment_closure_v1.0/publication_figures/figure1_family_accuracy.pdf
experiment-2b/reproducibility/e3v3_final_experiment_closure_v1.0/publication_figures/figure1_family_accuracy.png
experiment-2b/reproducibility/e3v3_final_experiment_closure_v1.0/publication_tables/table1_overall_performance.csv
notebooks/experiment_1_v2_together.ipynb

--------------

In [40]:
from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR2: STATIC INSPECTION OF FROZEN E1 / E2A DESIGN AND IMPLEMENTATION")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

E1_DATA = ROOT / "data/raw/intents_50.csv"
E1_RUN_ALL = ROOT / "src/experiments/run_all.py"
E1_NOTEBOOK = ROOT / "notebooks/experiment_1_v2_together.ipynb"

E2A_DATA = ROOT / "experiment-2/data/frozen/experiment2_intents_200_v1.csv"
E2A_RESULTS = ROOT / "experiment-2/reproducibility/experiment2a_frozen/experiment2a_results_600.csv"
E2A_METRICS = ROOT / "experiment-2/reproducibility/experiment2a_frozen/experiment2a_primary_metrics.csv"
E2A_MANIFEST = ROOT / "experiment-2/reproducibility/experiment2a_frozen/experiment2a_freeze_manifest.json"

HIST_ADAPTER = ROOT / "src/llm/together_adapter.py"

assert ROOT.exists()


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def verify_exists(label, path):
    assert path.exists(), f"STOP: missing {label}: {path}"
    print(f"✓ {label}")
    print("  Path   :", path.relative_to(ROOT))
    print("  SHA-256:", sha256(path))
    print("  Size   :", path.stat().st_size, "bytes")


# ======================================================================================
# 1. Verify important historical artifacts
# ======================================================================================

print("\n" + "-" * 118)
print("HISTORICAL ARTIFACTS")
print("-" * 118)

for label, path in [
    ("E1 50-case dataset", E1_DATA),
    ("E1 run_all implementation", E1_RUN_ALL),
    ("E1 historical notebook", E1_NOTEBOOK),
    ("E2A frozen 200-case dataset", E2A_DATA),
    ("E2A frozen 600-result evidence", E2A_RESULTS),
    ("E2A frozen primary metrics", E2A_METRICS),
    ("E2A freeze manifest", E2A_MANIFEST),
    ("Historical Together adapter", HIST_ADAPTER),
]:
    verify_exists(label, path)


# ======================================================================================
# 2. Dataset schemas
# ======================================================================================

print("\n" + "-" * 118)
print("E1 DATASET")
print("-" * 118)

e1 = pd.read_csv(E1_DATA)

print("Rows   :", len(e1))
print("Columns:", list(e1.columns))

assert len(e1) == 50

print("\nFirst 3 rows:")
print(e1.head(3).to_string(index=False))


print("\n" + "-" * 118)
print("E2A DATASET")
print("-" * 118)

e2a = pd.read_csv(E2A_DATA)

print("Rows   :", len(e2a))
print("Columns:", list(e2a.columns))

assert len(e2a) == 200

print("\nFirst 3 rows:")
print(e2a.head(3).to_string(index=False))


# ======================================================================================
# 3. Frozen E2A result structure
# ======================================================================================

print("\n" + "-" * 118)
print("E2A FROZEN RESULT STRUCTURE")
print("-" * 118)

r = pd.read_csv(E2A_RESULTS)

print("Rows   :", len(r))
print("Columns:", len(r.columns))
print("Column names:")

for i, c in enumerate(r.columns, 1):
    print(f"{i:02d}. {c}")

assert len(r) == 600, (
    f"STOP: expected historical E2A 600-result file; found {len(r)}"
)

print("\nPipeline/configuration counts:")

for candidate in [
    "pipeline",
    "method",
    "configuration",
    "system",
    "approach",
]:
    if candidate in r.columns:
        print(f"\n{candidate}:")
        print(r[candidate].value_counts(dropna=False).to_string())


# ======================================================================================
# 4. Frozen E2A metrics
# ======================================================================================

print("\n" + "-" * 118)
print("E2A HISTORICAL PRIMARY METRICS")
print("-" * 118)

metrics = pd.read_csv(E2A_METRICS)

print(metrics.to_string(index=False))


# ======================================================================================
# 5. E2A freeze manifest
# ======================================================================================

print("\n" + "-" * 118)
print("E2A FREEZE MANIFEST")
print("-" * 118)

manifest = json.loads(
    E2A_MANIFEST.read_text(encoding="utf-8")
)

print(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    )
)


# ======================================================================================
# 6. Inspect E1 run_all.py in full
# ======================================================================================

print("\n" + "-" * 118)
print("E1 — src/experiments/run_all.py")
print("-" * 118)

run_all_text = E1_RUN_ALL.read_text(
    encoding="utf-8",
    errors="replace"
)

for i, line in enumerate(run_all_text.splitlines(), 1):
    print(f"{i:04d}: {line}")


# ======================================================================================
# 7. Inspect notebook code cells
# ======================================================================================

print("\n" + "-" * 118)
print("E1 HISTORICAL NOTEBOOK — CODE CELLS")
print("-" * 118)

nb = json.loads(
    E1_NOTEBOOK.read_text(
        encoding="utf-8",
        errors="strict"
    )
)

code_cells = [
    cell
    for cell in nb.get("cells", [])
    if cell.get("cell_type") == "code"
]

print("Code cells:", len(code_cells))

for cell_no, cell in enumerate(code_cells, 1):

    source = "".join(cell.get("source", []))

    print("\n" + "=" * 90)
    print(f"CODE CELL {cell_no}")
    print("=" * 90)

    # bounded only to avoid enormous notebook output
    lines = source.splitlines()

    for i, line in enumerate(lines[:250], 1):
        print(f"{i:04d}: {line}")

    if len(lines) > 250:
        print(
            f"... [{len(lines)-250} additional lines omitted from this cell]"
        )


# ======================================================================================
# 8. Inspect historical adapter relevant definitions
# ======================================================================================

print("\n" + "-" * 118)
print("HISTORICAL TOGETHER ADAPTER — RELEVANT DEFINITIONS")
print("-" * 118)

adapter_text = HIST_ADAPTER.read_text(
    encoding="utf-8",
    errors="replace"
)

adapter_lines = adapter_text.splitlines()

interesting_terms = [
    "def call_json",
    "def _clean_json_text",
    "CIR_SYSTEM_PROMPT",
    "chat.completions.create",
    "temperature=",
    "response_format",
]

for term in interesting_terms:

    hits = [
        i
        for i, line in enumerate(adapter_lines)
        if term in line
    ]

    print(f"\n>>> TERM: {term}")
    print("Hits:", [x + 1 for x in hits])

    for idx in hits:

        start = max(0, idx - 8)
        end = min(len(adapter_lines), idx + 30)

        print(
            f"\n--- adapter lines {start+1}-{end} ---"
        )

        for j in range(start, end):
            print(f"{j+1:04d}: {adapter_lines[j]}")


# ======================================================================================
# 9. Search project source for E2A execution logic
# ======================================================================================

print("\n" + "-" * 118)
print("SEARCH FOR E2A EXECUTION LOGIC")
print("-" * 118)

search_terms = [
    "experiment2a_results_600",
    "experiment2_intents_200_v1",
    "experiment2a_execution_batches",
]

hits = []

for p in ROOT.rglob("*.py"):

    rel = str(p.relative_to(ROOT))

    if (
        "__pycache__" in rel
        or "/.git/" in f"/{rel}"
        or "journal_experiments/experiment_B_" in rel
    ):
        continue

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matched = [
        term
        for term in search_terms
        if term in text
    ]

    if matched:
        hits.append((p, matched))


print("Python files found:", len(hits))

for p, matched in hits:
    print()
    print(p.relative_to(ROOT))
    print("Matched:", matched)

    text = p.read_text(
        encoding="utf-8",
        errors="replace"
    )

    lines = text.splitlines()

    for idx, line in enumerate(lines):

        if any(term in line for term in matched):

            start = max(0, idx - 15)
            end = min(len(lines), idx + 40)

            print(
                f"\n--- lines {start+1}-{end} ---"
            )

            for j in range(start, end):
                print(f"{j+1:04d}: {lines[j]}")


# ======================================================================================
# 10. Search notebook files for E2A execution logic
# ======================================================================================

print("\n" + "-" * 118)
print("SEARCH NOTEBOOKS FOR E2A EXECUTION LOGIC")
print("-" * 118)

nb_hits = []

for p in ROOT.rglob("*.ipynb"):

    rel = str(p.relative_to(ROOT))

    if "/.git/" in f"/{rel}":
        continue

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matched = [
        term
        for term in search_terms
        if term in text
    ]

    if matched:
        nb_hits.append((p, matched))


print("Notebook files found:", len(nb_hits))

for p, matched in nb_hits:
    print(
        p.relative_to(ROOT),
        " | matched:",
        matched
    )


# ======================================================================================
# 11. Reverify critical files after inspection
# ======================================================================================

print("\n" + "-" * 118)
print("POST-INSPECTION INTEGRITY")
print("-" * 118)

critical = [
    E1_DATA,
    E1_RUN_ALL,
    E1_NOTEBOOK,
    E2A_DATA,
    E2A_RESULTS,
    E2A_METRICS,
    E2A_MANIFEST,
    HIST_ADAPTER,
]

before = {
    str(p): sha256(p)
    for p in critical
}

# No writes occurred; hash immediately again.
for p in critical:

    assert sha256(p) == before[str(p)]

    print(
        "✓ unchanged:",
        p.relative_to(ROOT)
    )


print("\n" + "=" * 118)
print("✓ STEP BR2 COMPLETE")
print("✓ E1 HISTORICAL DESIGN INSPECTED")
print("✓ E2A FROZEN DESIGN INSPECTED")
print("✓ HISTORICAL ADAPTER INSPECTED")
print("✓ NO API/MODEL CALL")
print("✓ NO EXPERIMENT EXECUTION")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THE COMPLETE OUTPUT BEFORE BR3")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR2: STATIC INSPECTION OF FROZEN E1 / E2A DESIGN AND IMPLEMENTATION

----------------------------------------------------------------------------------------------------------------------
HISTORICAL ARTIFACTS
----------------------------------------------------------------------------------------------------------------------
✓ E1 50-case dataset
  Path   : data/raw/intents_50.csv
  SHA-256: 71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826
  Size   : 3954 bytes
✓ E1 run_all implementation
  Path   : src/experiments/run_all.py
  SHA-256: 9caf14e350d70d1d7cc7321209893c4ef1d8893d31c4973a885fc482553cdc11
  Size   : 10636 bytes
✓ E1 historical notebook
  Path   : notebooks/experiment_1_v2_together.ipynb
  SHA-256: 5a2749476ac07d3986c1795ba56da156133776d3fae1b837a46e4dbb5d47e79c
  Size   : 3951 bytes
✓ E2A frozen 200-case dataset
  Path   : experiment-2/data/frozen/experiment2_intents_200_v1.csv
  SHA-256: 96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53

In [41]:
from pathlib import Path
import hashlib
import json
import shutil
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR3: PROSPECTIVE PREREGISTRATION")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PREREG_DIR = BR / "preregistration"
MANIFEST_DIR = BR / "manifests"

PREREG_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

PREREG = (
    PREREG_DIR
    / "experiment_B_review_preregistration_v1.json"
)

MANIFEST = (
    MANIFEST_DIR
    / "experiment_B_review_preregistration_manifest_v1.json"
)

# Persistent location — separate from protected historical master.
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_PREREG = (
    DRIVE_ROOT
    / "preregistration"
    / PREREG.name
)

DRIVE_MANIFEST = (
    DRIVE_ROOT
    / "manifests"
    / MANIFEST.name
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 1. Prevent silent overwrite
# ======================================================================================

for p in [
    PREREG,
    MANIFEST,
    DRIVE_PREREG,
    DRIVE_MANIFEST,
]:
    assert not p.exists(), (
        f"STOP: preregistration artifact already exists:\n{p}\n"
        "Do not overwrite it."
    )


# ======================================================================================
# 2. Frozen historical components
# ======================================================================================

E1_DATA = ROOT / "data/raw/intents_50.csv"
RUNNER = ROOT / "src/experiments/run_all.py"
ADAPTER = ROOT / "src/llm/together_adapter.py"

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

E2A_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

E2A_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)

E2A_FREEZE = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_freeze_manifest.json"
)

EXPECTED = {
    E1_DATA:
        "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826",

    RUNNER:
        "9caf14e350d70d1d7cc7321209893c4ef1d8893d31c4973a885fc482553cdc11",

    ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    E2A_DATA:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    E2A_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    E2A_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",

    E2A_FREEZE:
        "7b4ce6225dbb7795328ec5239cfa3983b14a8992d9fc2679cd5453554a6db1e1",
}

print("\n" + "-" * 118)
print("FROZEN COMPONENT VERIFICATION")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert path.exists(), f"STOP: missing {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: hash mismatch\n"
        f"{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# ======================================================================================
# 3. Structural-only schema
#
# IMPORTANT:
# This intentionally constrains JSON STRUCTURE, not semantic admissibility.
#
# No service_type enum.
# No numeric minimum/maximum.
#
# Therefore explicitly invalid source values can still be extracted and passed
# to the historical deterministic validator.
# ======================================================================================

STRUCTURAL_SCHEMA = {
    "type": "object",

    "properties": {

        "service_type": {
            "type": ["string", "null"]
        },

        "latency_ms": {
            "type": ["number", "null"]
        },

        "bandwidth_mbps": {
            "type": ["number", "null"]
        },

        "reliability": {
            "type": ["number", "null"]
        },

        "device_count": {
            "type": ["integer", "null"]
        },

        "coverage": {
            "type": ["string", "null"]
        },

        "objective": {
            "type": ["string", "null"]
        },
    },

    "required": [
        "service_type",
        "latency_ms",
        "bandwidth_mbps",
        "reliability",
        "device_count",
        "coverage",
        "objective",
    ],

    "additionalProperties": False,
}


# ======================================================================================
# 4. Prospective preregistration
# ======================================================================================

prereg = {
    "experiment":
        "B-Review",

    "version":
        "v1",

    "status":
        "prospectively_preregistered_before_any_B_review_model_call",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "relationship_to_other_experiments": {

        "B_E3":
            "Separate already-observed 120-case E3-v3 structured-decoding experiment.",

        "B_review":
            "Prospective experiment designed specifically to execute the originally requested constrained-decoding comparison on historical E1 and E2A.",

        "independence_rule":
            "B-E3 results are not used to modify the B-Review treatment, thresholds, prompt, schema, or evaluation criteria."
    },

    "research_question":
        (
            "Does provider-side structural JSON-schema constrained generation "
            "eliminate malformed CIR generation while leaving semantic extraction "
            "and deterministic validity decisions as separate concerns?"
        ),

    "semantic_model": {
        "model": "Qwen/Qwen3.5-9B",
        "reasoning_enabled": False,
        "temperature": 0,
    },

    "benchmarks": {

        "E1": {
            "cases": 50,
            "dataset":
                str(E1_DATA.relative_to(ROOT)),
            "sha256":
                EXPECTED[E1_DATA],
        },

        "E2A": {
            "cases": 200,
            "dataset":
                str(E2A_DATA.relative_to(ROOT)),
            "sha256":
                EXPECTED[E2A_DATA],
        },
    },

    "historical_control": {

        "pipeline":
            "cir",

        "definition":
            (
                "Historical CIR pipeline using prompt-instructed JSON generation "
                "followed by the frozen deterministic CIR validator."
            ),

        "historical_runner_sha256":
            EXPECTED[RUNNER],

        "historical_adapter_sha256":
            EXPECTED[ADAPTER],

        "historical_E2A_results_sha256":
            EXPECTED[E2A_RESULTS],

        "historical_E2A_metrics_sha256":
            EXPECTED[E2A_METRICS],
    },

    "treatment": {

        "id":
            "cir_constrained",

        "name":
            "historical_CIR_plus_structural_constrained_decoding",

        "definition":
            (
                "Same Qwen/Qwen3.5-9B model, same historical CIR system prompt, "
                "same user intent, reasoning disabled, temperature=0, and same "
                "historical CIR downstream validation/mapping/descriptor logic, "
                "with provider-side structural JSON-schema constrained generation "
                "added at the semantic-generation boundary."
            ),

        "only_intended_change":
            "provider-side structural JSON-schema constrained generation",

        "structural_schema":
            STRUCTURAL_SCHEMA,

        "explicitly_excluded_constraints": [
            "service_type enum",
            "latency_ms minimum",
            "bandwidth_mbps minimum",
            "reliability minimum/maximum",
            "device_count minimum",
        ],

        "reason_for_exclusion":
            (
                "The decoding schema must constrain syntax/structure only. "
                "Semantic and range admissibility remain the responsibility "
                "of the historical deterministic validator."
            ),
    },

    "execution_plan": {

        "E1":
            "Execute cir_constrained once for each of the 50 frozen E1 intents.",

        "E2A":
            "Execute cir_constrained once for each of the 200 frozen E2A intents.",

        "historical_arms":
            "Do not rerun direct, cir, or proposed historical arms.",

        "no_retry_rule":
            (
                "Do not silently rerun completed benchmark observations because "
                "of unfavorable results. Any infrastructure interruption must be "
                "recorded explicitly before deciding whether continuation is valid."
            ),
    },

    "primary_analysis": [

        "JSON parse success rate for cir_constrained.",

        "Structural-schema validity rate for cir_constrained.",

        "Decision accuracy.",

        "Valid acceptance rate.",

        "Invalid rejection rate.",

        "Comparison with the historical CIR arm on the same benchmark.",

        "Case-level paired correctness comparison where historical case-level CIR evidence is available.",
    ],

    "semantic_safety_analysis": [

        "Count structurally valid generated CIRs that are nevertheless rejected by deterministic validation.",

        "Count semantic/range validation failures despite successful constrained generation.",

        "Inspect whether unsupported or invalid source requirements can remain unresolved despite syntactically valid generation.",

        "Distinguish structural generation validity from semantic/source validity.",
    ],

    "claim_boundary": {
        "allowed":
            (
                "Constrained decoding may improve or guarantee tested structural "
                "output validity under the provider/schema configuration, while "
                "deterministic semantic/source validation can remain necessary."
            ),

        "not_allowed": [
            "Constrained decoding universally guarantees semantic correctness.",
            "Deterministic validation is redundant solely because parsing succeeds.",
            "The experiment proves hallucination-free operation.",
            "The experiment establishes universal model-independent guarantees.",
        ],
    },

    "frozen_components": {
        "historical_runner": {
            "path": str(RUNNER.relative_to(ROOT)),
            "sha256": EXPECTED[RUNNER],
        },

        "historical_adapter": {
            "path": str(ADAPTER.relative_to(ROOT)),
            "sha256": EXPECTED[ADAPTER],
        },

        "E2A_freeze_manifest": {
            "path": str(E2A_FREEZE.relative_to(ROOT)),
            "sha256": EXPECTED[E2A_FREEZE],
        },
    },
}


# ======================================================================================
# 5. Static preregistration assertions
# ======================================================================================

schema = prereg["treatment"]["structural_schema"]

expected_fields = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert set(schema["properties"]) == expected_fields
assert set(schema["required"]) == expected_fields
assert schema["additionalProperties"] is False

# No semantic/range constraints may leak into provider-side schema.
for field, spec in schema["properties"].items():

    assert "enum" not in spec, (
        f"STOP: enum leaked into structural schema: {field}"
    )

    assert "minimum" not in spec, (
        f"STOP: minimum leaked into structural schema: {field}"
    )

    assert "maximum" not in spec, (
        f"STOP: maximum leaked into structural schema: {field}"
    )

print("\n✓ Structural-only treatment schema verified")


# ======================================================================================
# 6. Write preregistration
# ======================================================================================

PREREG.write_text(
    json.dumps(
        prereg,
        indent=2,
        sort_keys=True,
        ensure_ascii=False
    ) + "\n",
    encoding="utf-8"
)

prereg_sha = sha256(PREREG)

print("\n" + "-" * 118)
print("PREREGISTRATION CREATED")
print("-" * 118)

print("Path   :", PREREG)
print("SHA-256:", prereg_sha)


# ======================================================================================
# 7. Create preregistration manifest
# ======================================================================================

manifest = {
    "experiment":
        "B-Review",

    "artifact":
        "prospective_preregistration",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "preregistration": {
        "path":
            str(PREREG.relative_to(ROOT)),
        "sha256":
            prereg_sha,
    },

    "historical_frozen_hashes": {
        str(path.relative_to(ROOT)): expected
        for path, expected in EXPECTED.items()
    },

    "execution_status":
        "NO_B_REVIEW_MODEL_CALLS_OR_BENCHMARK_EXECUTION_AT_TIME_OF_PREREGISTRATION",
}

MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
        ensure_ascii=False
    ) + "\n",
    encoding="utf-8"
)

manifest_sha = sha256(MANIFEST)

print("\nManifest SHA-256:", manifest_sha)


# ======================================================================================
# 8. Persist preregistration to Drive
# ======================================================================================

DRIVE_PREREG.parent.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_MANIFEST.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    PREREG,
    DRIVE_PREREG
)

shutil.copy2(
    MANIFEST,
    DRIVE_MANIFEST
)

assert sha256(DRIVE_PREREG) == prereg_sha
assert sha256(DRIVE_MANIFEST) == manifest_sha

print("✓ Preregistration persisted to Drive byte-for-byte")
print("✓ Manifest persisted to Drive byte-for-byte")


# ======================================================================================
# 9. Reverify historical frozen components
# ======================================================================================

print("\n" + "-" * 118)
print("POST-PREREGISTRATION HISTORICAL INTEGRITY")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


print("\n" + "=" * 118)
print("✓ STEP BR3 COMPLETE")
print("✓ B-REVIEW PROSPECTIVELY PREREGISTERED")
print("✓ E1 = 50 CASES")
print("✓ E2A = 200 CASES")
print("✓ NEW ARM = cir_constrained")
print("✓ MODEL = Qwen/Qwen3.5-9B")
print("✓ STRUCTURAL-ONLY PROVIDER SCHEMA")
print("✓ HISTORICAL DIRECT/CIR/PROPOSED ARMS WILL NOT BE RERUN")
print("✓ B-E3 REMAINS SEPARATE AND UNCHANGED")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR4")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR3: PROSPECTIVE PREREGISTRATION

----------------------------------------------------------------------------------------------------------------------
FROZEN COMPONENT VERIFICATION
----------------------------------------------------------------------------------------------------------------------
✓ data/raw/intents_50.csv
✓ src/experiments/run_all.py
✓ src/llm/together_adapter.py
✓ experiment-2/data/frozen/experiment2_intents_200_v1.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_results_600.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_primary_metrics.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_freeze_manifest.json

✓ Structural-only treatment schema verified

----------------------------------------------------------------------------------------------------------------------
PREREGISTRATION CREATED
------------------------------------------------------------------------------------------

In [42]:
from pathlib import Path
import hashlib
import json
import shutil
import py_compile
import ast
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR4: BUILD AND STATICALLY VERIFY ISOLATED cir_constrained TREATMENT")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

IMPLEMENTATION_DIR = BR / "implementation"
MANIFEST_DIR = BR / "manifests"

IMPLEMENTATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ADAPTER_OUT = (
    IMPLEMENTATION_DIR
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER_OUT = (
    IMPLEMENTATION_DIR
    / "run_experiment_B_review_cir_constrained_v1.py"
)

MANIFEST_OUT = (
    MANIFEST_DIR
    / "experiment_B_review_BR4_implementation_manifest_v1.json"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_ADAPTER = (
    DRIVE_ROOT
    / "implementation"
    / ADAPTER_OUT.name
)

DRIVE_RUNNER = (
    DRIVE_ROOT
    / "implementation"
    / RUNNER_OUT.name
)

DRIVE_MANIFEST = (
    DRIVE_ROOT
    / "manifests"
    / MANIFEST_OUT.name
)


# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}: {path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Prevent silent overwrite
# ======================================================================================

for p in [
    ADAPTER_OUT,
    RUNNER_OUT,
    MANIFEST_OUT,
    DRIVE_ADAPTER,
    DRIVE_RUNNER,
    DRIVE_MANIFEST,
]:

    assert not p.exists(), (
        f"STOP: BR4 artifact already exists:\n{p}\n"
        "Do not overwrite an existing experimental artifact."
    )


# ======================================================================================
# 2. Verify preregistration and historical frozen components
# ======================================================================================

EXPECTED_PREREG_SHA = (
    "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba"
)

HIST_RUNNER = ROOT / "src/experiments/run_all.py"
HIST_ADAPTER = ROOT / "src/llm/together_adapter.py"
E1_DATA = ROOT / "data/raw/intents_50.csv"

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

EXPECTED_HIST_RUNNER_SHA = (
    "9caf14e350d70d1d7cc7321209893c4ef1d8893d31c4973a885fc482553cdc11"
)

EXPECTED_HIST_ADAPTER_SHA = (
    "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3"
)

EXPECTED_E1_SHA = (
    "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826"
)

EXPECTED_E2A_SHA = (
    "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753"
)

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

verify(
    PREREG,
    EXPECTED_PREREG_SHA,
    "B-Review preregistration"
)

verify(
    HIST_RUNNER,
    EXPECTED_HIST_RUNNER_SHA,
    "historical E1/E2A runner"
)

verify(
    HIST_ADAPTER,
    EXPECTED_HIST_ADAPTER_SHA,
    "historical Together adapter"
)

verify(
    E1_DATA,
    EXPECTED_E1_SHA,
    "E1 dataset"
)

verify(
    E2A_DATA,
    EXPECTED_E2A_SHA,
    "E2A dataset"
)


# ======================================================================================
# 3. Load preregistration
# ======================================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

assert (
    prereg["semantic_model"]["model"]
    == "Qwen/Qwen3.5-9B"
)

assert (
    prereg["treatment"]["id"]
    == "cir_constrained"
)

STRUCTURAL_SCHEMA = (
    prereg["treatment"]["structural_schema"]
)

EXPECTED_FIELDS = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert (
    set(STRUCTURAL_SCHEMA["properties"])
    == EXPECTED_FIELDS
)

assert (
    set(STRUCTURAL_SCHEMA["required"])
    == EXPECTED_FIELDS
)

assert (
    STRUCTURAL_SCHEMA["additionalProperties"]
    is False
)

for field, spec in STRUCTURAL_SCHEMA["properties"].items():

    assert "enum" not in spec
    assert "minimum" not in spec
    assert "maximum" not in spec

print("✓ preregistered structural-only schema verified")


# ======================================================================================
# 4. Extract historical CIR prompt and cleaner exactly from historical adapter
#
# We do this through AST rather than manually retyping them.
# ======================================================================================

hist_source = HIST_ADAPTER.read_text(
    encoding="utf-8"
)

hist_tree = ast.parse(hist_source)

cir_prompt_value = None
cleaner_source = None

for node in hist_tree.body:

    # --------------------------------------------------
    # CIR_SYSTEM_PROMPT assignment
    # --------------------------------------------------

    if isinstance(node, ast.Assign):

        for target in node.targets:

            if (
                isinstance(target, ast.Name)
                and target.id == "CIR_SYSTEM_PROMPT"
            ):

                cir_prompt_value = ast.literal_eval(
                    node.value
                )

    # --------------------------------------------------
    # _clean_json_text function
    # --------------------------------------------------

    if (
        isinstance(node, ast.FunctionDef)
        and node.name == "_clean_json_text"
    ):

        cleaner_source = ast.get_source_segment(
            hist_source,
            node
        )


assert isinstance(cir_prompt_value, str)
assert cir_prompt_value

assert isinstance(cleaner_source, str)
assert cleaner_source

print("✓ historical CIR_SYSTEM_PROMPT extracted exactly")
print("✓ historical _clean_json_text extracted exactly")


# ======================================================================================
# 5. Generate isolated treatment adapter
#
# Only intended provider-call change:
#
# response_format = {
#     "type": "json_schema",
#     "json_schema": {
#         "name": "...",
#         "schema": STRUCTURAL_SCHEMA
#     }
# }
#
# No historical file is modified.
# ======================================================================================

adapter_code = f'''\
import os
import json
import time
import re

from together import Together


CIR_SYSTEM_PROMPT = {cir_prompt_value!r}


STRUCTURAL_SCHEMA = {repr(STRUCTURAL_SCHEMA)}


{cleaner_source}


def call_json_structured(
    prompt,
    mode="cir",
    model="Qwen/Qwen3.5-9B"
):
    """
    B-Review cir_constrained treatment.

    This adapter intentionally supports CIR mode only.

    Intended treatment difference relative to the historical CIR
    call_json path:
        provider-side structural JSON-schema constrained generation.

    Semantic/range admissibility is NOT moved into the decoder.
    """

    if mode != "cir":
        raise ValueError(
            "B-Review structured treatment supports CIR mode only."
        )

    api_key = os.environ.get("TOGETHER_API_KEY")

    if not api_key:
        raise RuntimeError(
            "TOGETHER_API_KEY is not set."
        )

    client = Together(
        api_key=api_key
    )

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,

        reasoning={{
            "enabled": False
        }},

        messages=[
            {{
                "role": "system",
                "content": CIR_SYSTEM_PROMPT
            }},
            {{
                "role": "user",
                "content": prompt
            }}
        ],

        temperature=0,

        response_format={{
            "type": "json_schema",

            "json_schema": {{
                "name":
                    "network_slicing_cir_structural",

                "schema":
                    STRUCTURAL_SCHEMA
            }}
        }}
    )

    latency_ms = (
        time.perf_counter() - start
    ) * 1000

    choice = response.choices[0]
    message = choice.message

    raw_text = message.content or ""

    clean_text = _clean_json_text(
        raw_text
    )

    try:
        parsed_json = json.loads(
            clean_text
        )

        parse_ok = True

    except (
        json.JSONDecodeError,
        TypeError
    ):

        parsed_json = None
        parse_ok = False

    usage = getattr(
        response,
        "usage",
        None
    )

    return {{
        "raw":
            clean_text,

        "json":
            parsed_json,

        "parse_ok":
            parse_ok,

        "latency_ms":
            latency_ms,

        "finish_reason":
            getattr(
                choice,
                "finish_reason",
                None
            ),

        "prompt_tokens":
            getattr(
                usage,
                "prompt_tokens",
                None
            )
            if usage
            else None,

        "completion_tokens":
            getattr(
                usage,
                "completion_tokens",
                None
            )
            if usage
            else None
    }}
'''

ADAPTER_OUT.write_text(
    adapter_code,
    encoding="utf-8"
)

py_compile.compile(
    str(ADAPTER_OUT),
    doraise=True
)

print("✓ isolated structured adapter written")
print("✓ structured adapter compiles")


# ======================================================================================
# 6. Generate isolated B-Review runner
#
# IMPORTANT:
# This is NOT a copy of the entire historical three-arm runner.
#
# It executes only the new cir_constrained arm while preserving the
# historical CIR downstream sequence:
#
# generated CIR
# -> validate_cir
# -> map_standards if accepted
# -> compile_descriptor if accepted + mapping
#
# This prevents rerunning historical direct/cir/proposed arms.
# ======================================================================================

runner_code = '''\
import argparse
import json
from pathlib import Path

import pandas as pd

from src.validator import validate_cir
from src.standards.mapper import map_standards
from src.descriptor import compile_descriptor

from journal_experiments.experiment_B_review_requested_constrained_decoding.implementation.experiment_B_review_structured_adapter_v1 import (
    call_json_structured
)


def run(
    dataset,
    out_path,
    model="Qwen/Qwen3.5-9B"
):
    df = pd.read_csv(dataset)

    out = []

    for _, r in df.iterrows():

        llm = call_json_structured(
            r.intent,
            mode="cir",
            model=model
        )

        output = llm["json"]

        cir = (
            output
            if isinstance(
                output,
                dict
            )
            else {}
        )

        validation = (
            validate_cir(cir)
        )

        mapping = (
            map_standards(cir)
            if validation["accepted"]
            else {
                "mapping_found": False
            }
        )

        descriptor = None

        if (
            validation["accepted"]
            and mapping[
                "mapping_found"
            ]
        ):
            descriptor = (
                compile_descriptor(
                    cir,
                    mapping
                )
            )

        record = {
            "id":
                r.id,

            "intent":
                r.intent,

            "category":
                r.category,

            "expected_service":
                r.expected_service
                if pd.notna(
                    r.expected_service
                )
                else None,

            "expected_valid":
                bool(
                    r.expected_valid
                ),

            "pipeline":
                "cir_constrained",

            "model":
                model,

            "parse_ok":
                llm["parse_ok"],

            "latency_ms":
                llm["latency_ms"],

            "finish_reason":
                llm.get(
                    "finish_reason"
                ),

            "prompt_tokens":
                llm.get(
                    "prompt_tokens"
                ),

            "completion_tokens":
                llm.get(
                    "completion_tokens"
                ),

            "raw_response":
                llm["raw"],

            "predicted_service":
                cir.get(
                    "service_type"
                ),

            "schema_valid":
                validation[
                    "schema_valid"
                ],

            "hallucination_detected":
                validation[
                    "hallucination_detected"
                ],

            "unsupported_input_detected":
                False,

            "input_audit_valid":
                None,

            "mapping_found":
                mapping.get(
                    "mapping_found",
                    False
                ),

            "descriptor_success":
                descriptor is not None,

            # Historical CIR decision:
            "accepted":
                validation[
                    "accepted"
                ],

            "validation_errors":
                ";".join(
                    validation[
                        "errors"
                    ]
                ),

            "cir_json":
                json.dumps(
                    cir
                ),

            "descriptor_json":
                json.dumps(
                    descriptor
                )
                if descriptor
                else None
        }

        out.append(
            record
        )

        print(
            f"[{len(out)}/{len(df)}] "
            f"{r.id}",
            flush=True
        )

    results = pd.DataFrame(
        out
    )

    Path(out_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    results.to_csv(
        out_path,
        index=False
    )

    print()
    print(
        "Rows written:",
        len(results)
    )

    print(
        "Output:",
        out_path
    )

    return results


def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--dataset",
        required=True
    )

    parser.add_argument(
        "--out",
        required=True
    )

    parser.add_argument(
        "--model",
        default="Qwen/Qwen3.5-9B"
    )

    args = parser.parse_args()

    run(
        args.dataset,
        args.out,
        args.model
    )


if __name__ == "__main__":
    main()
'''

RUNNER_OUT.write_text(
    runner_code,
    encoding="utf-8"
)

py_compile.compile(
    str(RUNNER_OUT),
    doraise=True
)

print("✓ isolated cir_constrained runner written")
print("✓ treatment runner compiles")


# ======================================================================================
# 7. STATIC PROOF — prompt and cleaner equivalence
# ======================================================================================

treat_source = ADAPTER_OUT.read_text(
    encoding="utf-8"
)

treat_tree = ast.parse(
    treat_source
)

treat_prompt = None
treat_cleaner = None

for node in treat_tree.body:

    if isinstance(node, ast.Assign):

        for target in node.targets:

            if (
                isinstance(target, ast.Name)
                and target.id == "CIR_SYSTEM_PROMPT"
            ):

                treat_prompt = ast.literal_eval(
                    node.value
                )

    if (
        isinstance(node, ast.FunctionDef)
        and node.name == "_clean_json_text"
    ):

        treat_cleaner = ast.get_source_segment(
            treat_source,
            node
        )


assert treat_prompt == cir_prompt_value

# Normalize only outer whitespace caused by source extraction.
assert (
    treat_cleaner.strip()
    == cleaner_source.strip()
)

print("✓ treatment CIR prompt EXACTLY equals historical CIR prompt")
print("✓ treatment cleaner EXACTLY equals historical cleaner")


# ======================================================================================
# 8. STATIC PROOF — provider-call settings
# ======================================================================================

assert 'reasoning={' in treat_source
assert '"enabled": False' in treat_source
assert "temperature=0" in treat_source
assert "response_format=" in treat_source

# Historical adapter must still have no response_format.
historical_source = HIST_ADAPTER.read_text(
    encoding="utf-8"
)

assert "response_format" not in historical_source

print("✓ historical adapter has no response_format")
print("✓ treatment adapter adds response_format")
print("✓ reasoning remains disabled")
print("✓ temperature remains 0")


# ======================================================================================
# 9. STATIC PROOF — treatment schema has structure only
# ======================================================================================

for field, spec in STRUCTURAL_SCHEMA["properties"].items():

    assert "enum" not in spec
    assert "minimum" not in spec
    assert "maximum" not in spec

print("✓ no semantic enum/range constraints moved into decoder")


# ======================================================================================
# 10. STATIC PROOF — downstream CIR sequence
# ======================================================================================

hist_runner_source = HIST_RUNNER.read_text(
    encoding="utf-8"
)

new_runner_source = RUNNER_OUT.read_text(
    encoding="utf-8"
)

required_downstream_fragments = [
    "validate_cir(cir)",
    "map_standards(cir)",
    "compile_descriptor(",
    '"accepted":',
]

for fragment in required_downstream_fragments:

    assert fragment in hist_runner_source, (
        f"STOP: expected historical downstream fragment absent: {fragment}"
    )

    assert fragment in new_runner_source, (
        f"STOP: treatment downstream fragment absent: {fragment}"
    )

print("✓ historical CIR downstream stages retained")
print("✓ no input_audit added to cir_constrained treatment")


# ======================================================================================
# 11. Verify runner is treatment-only
# ======================================================================================

assert '"direct"' not in new_runner_source
assert '"proposed"' not in new_runner_source

assert (
    '"cir_constrained"'
    in new_runner_source
)

assert (
    new_runner_source.count(
        "call_json_structured("
    )
    == 1
)

print("✓ treatment runner contains no Direct arm")
print("✓ treatment runner contains no Proposed arm")
print("✓ exactly one structured-generation call per benchmark row")


# ======================================================================================
# 12. Hash implementation
# ======================================================================================

adapter_sha = sha256(
    ADAPTER_OUT
)

runner_sha = sha256(
    RUNNER_OUT
)

print("\n" + "-" * 118)
print("IMPLEMENTATION HASHES")
print("-" * 118)

print(
    "Treatment adapter:",
    adapter_sha
)

print(
    "Treatment runner :",
    runner_sha
)


# ======================================================================================
# 13. Create BR4 implementation manifest
# ======================================================================================

manifest = {
    "experiment":
        "B-Review",

    "step":
        "BR4",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "IMPLEMENTATION_BUILT_STATIC_ONLY_NO_MODEL_CALL",

    "preregistration": {
        "path":
            str(
                PREREG.relative_to(
                    ROOT
                )
            ),

        "sha256":
            EXPECTED_PREREG_SHA
    },

    "historical_provenance": {
        "runner_sha256":
            EXPECTED_HIST_RUNNER_SHA,

        "adapter_sha256":
            EXPECTED_HIST_ADAPTER_SHA,

        "E1_dataset_sha256":
            EXPECTED_E1_SHA,

        "E2A_dataset_sha256":
            EXPECTED_E2A_SHA,
    },

    "treatment_adapter": {
        "path":
            str(
                ADAPTER_OUT.relative_to(
                    ROOT
                )
            ),

        "sha256":
            adapter_sha,
    },

    "treatment_runner": {
        "path":
            str(
                RUNNER_OUT.relative_to(
                    ROOT
                )
            ),

        "sha256":
            runner_sha,
    },

    "static_equivalence_checks": {
        "historical_CIR_prompt_exact":
            True,

        "historical_cleaner_exact":
            True,

        "same_model":
            True,

        "same_reasoning_disabled":
            True,

        "same_temperature_zero":
            True,

        "historical_response_format_absent":
            True,

        "treatment_response_format_present":
            True,

        "structural_schema_only":
            True,

        "historical_CIR_validation_retained":
            True,

        "historical_mapping_retained":
            True,

        "historical_descriptor_compilation_retained":
            True,

        "historical_direct_arm_not_rerun":
            True,

        "historical_cir_arm_not_rerun":
            True,

        "historical_proposed_arm_not_rerun":
            True,
    },

    "execution_status":
        "ZERO_API_CALLS_ZERO_BENCHMARK_CASES",
}


MANIFEST_OUT.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

manifest_sha = sha256(
    MANIFEST_OUT
)

print(
    "BR4 manifest    :",
    manifest_sha
)


# ======================================================================================
# 14. Persist implementation to Drive
# ======================================================================================

for destination in [
    DRIVE_ADAPTER,
    DRIVE_RUNNER,
    DRIVE_MANIFEST,
]:

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


shutil.copy2(
    ADAPTER_OUT,
    DRIVE_ADAPTER
)

shutil.copy2(
    RUNNER_OUT,
    DRIVE_RUNNER
)

shutil.copy2(
    MANIFEST_OUT,
    DRIVE_MANIFEST
)

assert (
    sha256(DRIVE_ADAPTER)
    == adapter_sha
)

assert (
    sha256(DRIVE_RUNNER)
    == runner_sha
)

assert (
    sha256(DRIVE_MANIFEST)
    == manifest_sha
)

print("✓ BR4 implementation persisted to Drive byte-for-byte")


# ======================================================================================
# 15. Final historical integrity
# ======================================================================================

print("\n" + "-" * 118)
print("POST-BR4 HISTORICAL INTEGRITY")
print("-" * 118)

verify(
    HIST_RUNNER,
    EXPECTED_HIST_RUNNER_SHA,
    "historical runner after BR4"
)

verify(
    HIST_ADAPTER,
    EXPECTED_HIST_ADAPTER_SHA,
    "historical adapter after BR4"
)

verify(
    E1_DATA,
    EXPECTED_E1_SHA,
    "E1 dataset after BR4"
)

verify(
    E2A_DATA,
    EXPECTED_E2A_SHA,
    "E2A dataset after BR4"
)

verify(
    PREREG,
    EXPECTED_PREREG_SHA,
    "B-Review preregistration after BR4"
)


print("\n" + "=" * 118)
print("✓ STEP BR4 COMPLETE")
print("✓ ISOLATED cir_constrained ADAPTER BUILT")
print("✓ ISOLATED TREATMENT-ONLY RUNNER BUILT")
print("✓ HISTORICAL CIR PROMPT EXACTLY PRESERVED")
print("✓ HISTORICAL JSON CLEANER EXACTLY PRESERVED")
print("✓ MODEL / REASONING / TEMPERATURE PRESERVED")
print("✓ ONLY PROVIDER-SIDE STRUCTURAL CONSTRAINT INTENTIONALLY ADDED")
print("✓ SEMANTIC/RANGE CONSTRAINTS NOT MOVED INTO DECODER")
print("✓ HISTORICAL CIR VALIDATION/MAPPING/DESCRIPTOR PATH RETAINED")
print("✓ HISTORICAL DIRECT/CIR/PROPOSED ARMS NOT EXECUTED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK CASES EXECUTED")
print("✓ NO HISTORICAL FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR5")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR4: BUILD AND STATICALLY VERIFY ISOLATED cir_constrained TREATMENT

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ B-Review preregistration verified
✓ historical E1/E2A runner verified
✓ historical Together adapter verified
✓ E1 dataset verified
✓ E2A dataset verified
✓ preregistered structural-only schema verified
✓ historical CIR_SYSTEM_PROMPT extracted exactly
✓ historical _clean_json_text extracted exactly
✓ isolated structured adapter written
✓ structured adapter compiles
✓ isolated cir_constrained runner written
✓ treatment runner compiles
✓ treatment CIR prompt EXACTLY equals historical CIR prompt
✓ treatment cleaner EXACTLY equals historical cleaner
✓ historical adapter has no response_format
✓ treatment adapter adds response_format
✓ reasonin

In [43]:
from pathlib import Path
import hashlib
import json
import shutil
import sys
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR5: ONE-TIME SYNTHETIC STRUCTURED-DECODING CAPABILITY PROBE")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR4_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR4_implementation_manifest_v1.json"
)

PROBE_DIR = BR / "capability_probe"
MANIFEST_DIR = BR / "manifests"

PROBE = (
    PROBE_DIR
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

MANIFEST = (
    MANIFEST_DIR
    / "experiment_B_review_BR5_capability_probe_manifest_v1.json"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_PROBE = (
    DRIVE_ROOT
    / "capability_probe"
    / PROBE.name
)

DRIVE_MANIFEST = (
    DRIVE_ROOT
    / "manifests"
    / MANIFEST.name
)


# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}: {path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label} verified")


# ======================================================================================
# 1. Frozen hashes
# ======================================================================================

EXPECTED_PREREG_SHA = (
    "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba"
)

EXPECTED_ADAPTER_SHA = (
    "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad"
)

EXPECTED_RUNNER_SHA = (
    "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851"
)

EXPECTED_BR4_MANIFEST_SHA = (
    "c3e1a1fb75073ab2ca55c6dbae24c14bbbdb957ed4fc667883d7b425d5f9ceb8"
)

HIST_RUNNER = ROOT / "src/experiments/run_all.py"
HIST_ADAPTER = ROOT / "src/llm/together_adapter.py"

E1_DATA = ROOT / "data/raw/intents_50.csv"

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

EXPECTED_HIST_RUNNER_SHA = (
    "9caf14e350d70d1d7cc7321209893c4ef1d8893d31c4973a885fc482553cdc11"
)

EXPECTED_HIST_ADAPTER_SHA = (
    "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3"
)

EXPECTED_E1_SHA = (
    "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826"
)

EXPECTED_E2A_SHA = (
    "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753"
)


# ======================================================================================
# 2. Prevent accidental second BR5 probe
# ======================================================================================

for p in [
    PROBE,
    MANIFEST,
    DRIVE_PROBE,
    DRIVE_MANIFEST,
]:

    assert not p.exists(), (
        f"STOP: BR5 evidence already exists:\n{p}\n\n"
        "DO NOT rerun the BR5 capability probe."
    )


# ======================================================================================
# 3. Verify complete provenance BEFORE provider call
# ======================================================================================

print("\n" + "-" * 118)
print("PRE-PROBE PROVENANCE")
print("-" * 118)

verify(
    PREREG,
    EXPECTED_PREREG_SHA,
    "B-Review preregistration"
)

verify(
    ADAPTER,
    EXPECTED_ADAPTER_SHA,
    "BR4 treatment adapter"
)

verify(
    RUNNER,
    EXPECTED_RUNNER_SHA,
    "BR4 treatment runner"
)

verify(
    BR4_MANIFEST,
    EXPECTED_BR4_MANIFEST_SHA,
    "BR4 implementation manifest"
)

verify(
    HIST_RUNNER,
    EXPECTED_HIST_RUNNER_SHA,
    "historical runner"
)

verify(
    HIST_ADAPTER,
    EXPECTED_HIST_ADAPTER_SHA,
    "historical adapter"
)

verify(
    E1_DATA,
    EXPECTED_E1_SHA,
    "E1 dataset"
)

verify(
    E2A_DATA,
    EXPECTED_E2A_SHA,
    "E2A dataset"
)


# ======================================================================================
# 4. Verify preregistered treatment
# ======================================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg["semantic_model"]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"

schema = prereg["treatment"]["structural_schema"]

expected_fields = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert set(schema["properties"]) == expected_fields
assert set(schema["required"]) == expected_fields
assert schema["additionalProperties"] is False

for field, spec in schema["properties"].items():

    assert "enum" not in spec
    assert "minimum" not in spec
    assert "maximum" not in spec

print("✓ preregistered model verified")
print("✓ preregistered structural-only schema verified")


# ======================================================================================
# 5. Verify API key WITHOUT displaying it
# ======================================================================================

import os

assert os.environ.get("TOGETHER_API_KEY"), (
    "STOP: TOGETHER_API_KEY is not available in the environment."
)

print("✓ TOGETHER_API_KEY available")


# ======================================================================================
# 6. Import frozen BR4 treatment adapter
# ======================================================================================

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from journal_experiments.experiment_B_review_requested_constrained_decoding.implementation.experiment_B_review_structured_adapter_v1 import (
    call_json_structured
)

print("✓ BR4 treatment adapter imported")


# ======================================================================================
# 7. ONE synthetic intent only
#
# IMPORTANT:
# This text is deliberately synthetic and is NOT an E1/E2A benchmark intent.
# It is designed only to test whether the exact provider-side structured
# decoding configuration is accepted and returns the seven-field CIR.
# ======================================================================================

SYNTHETIC_INTENT = (
    "Synthetic capability probe BR5: create an eMBB slice with "
    "latency 17 ms, bandwidth 321 Mbps, reliability 0.987, "
    "43 devices, coverage Test-Zone-BR5, and objective "
    "synthetic-structured-output-capability-check."
)

print("\n" + "-" * 118)
print("ONE-TIME PROVIDER CAPABILITY PROBE")
print("-" * 118)

print("Model:", MODEL)
print("Benchmark cases used: 0")
print("Provider calls about to execute: 1")


# ======================================================================================
# 8. EXACTLY ONE provider call
# ======================================================================================

result = call_json_structured(
    SYNTHETIC_INTENT,
    mode="cir",
    model=MODEL
)


# ======================================================================================
# 9. Evaluate capability result
# ======================================================================================

print("\n✓ Provider call returned")

parse_ok = bool(
    result.get("parse_ok")
)

parsed = result.get("json")

print("Parse OK:", parse_ok)

assert parse_ok, (
    "STOP: provider returned output that the treatment adapter "
    "could not parse as JSON."
)

assert isinstance(parsed, dict), (
    "STOP: parsed treatment result is not a JSON object."
)

observed_fields = set(
    parsed.keys()
)

assert observed_fields == expected_fields, (
    "STOP: structured result field set differs from preregistered schema.\n"
    f"Expected: {sorted(expected_fields)}\n"
    f"Observed: {sorted(observed_fields)}"
)


# ======================================================================================
# 10. Validate STRUCTURE independently
#
# This intentionally does NOT perform semantic/range validation.
# ======================================================================================

from jsonschema import Draft202012Validator

structural_validator = Draft202012Validator(
    schema
)

structural_errors = sorted(
    structural_validator.iter_errors(
        parsed
    ),
    key=lambda e: list(e.path)
)

STRUCTURAL_VALID = (
    len(structural_errors) == 0
)

assert STRUCTURAL_VALID, (
    "STOP: provider output violated preregistered structural schema:\n"
    + "\n".join(
        str(e.message)
        for e in structural_errors
    )
)

print("Structural schema valid:", STRUCTURAL_VALID)


# ======================================================================================
# 11. Display bounded result
# ======================================================================================

print("\nParsed structured result:")

print(
    json.dumps(
        parsed,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "\nFinish reason:",
    result.get("finish_reason")
)

print(
    "Latency ms:",
    result.get("latency_ms")
)


# ======================================================================================
# 12. Freeze probe evidence
# ======================================================================================

PROBE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

probe_evidence = {
    "experiment":
        "B-Review",

    "step":
        "BR5",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "purpose":
        "single synthetic non-benchmark provider capability probe",

    "benchmark_cases_used":
        0,

    "provider_calls":
        1,

    "model":
        MODEL,

    "synthetic_intent":
        SYNTHETIC_INTENT,

    "parse_ok":
        parse_ok,

    "structural_schema_valid":
        STRUCTURAL_VALID,

    "parsed_json":
        parsed,

    "raw":
        result.get("raw"),

    "finish_reason":
        result.get("finish_reason"),

    "latency_ms":
        result.get("latency_ms"),

    "prompt_tokens":
        result.get("prompt_tokens"),

    "completion_tokens":
        result.get("completion_tokens"),

    "capability_status":
        "SUPPORTED_AND_PROBE_VALID",
}

PROBE.write_text(
    json.dumps(
        probe_evidence,
        indent=2,
        sort_keys=True,
        ensure_ascii=False
    ) + "\n",
    encoding="utf-8"
)

probe_sha = sha256(
    PROBE
)


# ======================================================================================
# 13. Freeze BR5 manifest
# ======================================================================================

manifest = {
    "experiment":
        "B-Review",

    "step":
        "BR5",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "status":
        "SUPPORTED_AND_PROBE_VALID",

    "provider_calls":
        1,

    "benchmark_cases":
        0,

    "preregistration_sha256":
        EXPECTED_PREREG_SHA,

    "treatment_adapter_sha256":
        EXPECTED_ADAPTER_SHA,

    "treatment_runner_sha256":
        EXPECTED_RUNNER_SHA,

    "BR4_manifest_sha256":
        EXPECTED_BR4_MANIFEST_SHA,

    "probe": {
        "path":
            str(
                PROBE.relative_to(
                    ROOT
                )
            ),

        "sha256":
            probe_sha,
    },
}

MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

manifest_sha = sha256(
    MANIFEST
)


# ======================================================================================
# 14. Persist evidence to Drive
# ======================================================================================

DRIVE_PROBE.parent.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_MANIFEST.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    PROBE,
    DRIVE_PROBE
)

shutil.copy2(
    MANIFEST,
    DRIVE_MANIFEST
)

assert (
    sha256(DRIVE_PROBE)
    == probe_sha
)

assert (
    sha256(DRIVE_MANIFEST)
    == manifest_sha
)

print("\n" + "-" * 118)
print("FROZEN BR5 EVIDENCE")
print("-" * 118)

print(
    "Probe SHA-256   :",
    probe_sha
)

print(
    "Manifest SHA-256:",
    manifest_sha
)

print(
    "✓ BR5 evidence persisted to Drive byte-for-byte"
)


# ======================================================================================
# 15. Post-probe integrity
# ======================================================================================

print("\n" + "-" * 118)
print("POST-PROBE INTEGRITY")
print("-" * 118)

verify(
    PREREG,
    EXPECTED_PREREG_SHA,
    "B-Review preregistration"
)

verify(
    ADAPTER,
    EXPECTED_ADAPTER_SHA,
    "BR4 treatment adapter"
)

verify(
    RUNNER,
    EXPECTED_RUNNER_SHA,
    "BR4 treatment runner"
)

verify(
    HIST_RUNNER,
    EXPECTED_HIST_RUNNER_SHA,
    "historical runner"
)

verify(
    HIST_ADAPTER,
    EXPECTED_HIST_ADAPTER_SHA,
    "historical adapter"
)

verify(
    E1_DATA,
    EXPECTED_E1_SHA,
    "E1 dataset"
)

verify(
    E2A_DATA,
    EXPECTED_E2A_SHA,
    "E2A dataset"
)


print("\n" + "=" * 118)
print("✓ STEP BR5 COMPLETE")
print("✓ EXACTLY ONE SYNTHETIC PROVIDER CALL COMPLETED")
print("✓ ZERO E1/E2A BENCHMARK CASES EXECUTED")
print("✓ STRUCTURED DECODING CAPABILITY CONFIRMED")
print("✓ STRUCTURAL SCHEMA VALID")
print("✓ BR5 EVIDENCE FROZEN AND PERSISTED")
print("✓ DO NOT RERUN BR5")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR6")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR5: ONE-TIME SYNTHETIC STRUCTURED-DECODING CAPABILITY PROBE

----------------------------------------------------------------------------------------------------------------------
PRE-PROBE PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ B-Review preregistration verified
✓ BR4 treatment adapter verified
✓ BR4 treatment runner verified
✓ BR4 implementation manifest verified
✓ historical runner verified
✓ historical adapter verified
✓ E1 dataset verified
✓ E2A dataset verified
✓ preregistered model verified
✓ preregistered structural-only schema verified
✓ TOGETHER_API_KEY available
✓ BR4 treatment adapter imported

----------------------------------------------------------------------------------------------------------------------
ONE-TIME PROVIDER CAPABILITY PROBE
----------------------------------------------------------------------------------------------------------------

In [44]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import os

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR6: FINAL ZERO-CALL BENCHMARK PREFLIGHT")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR4_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR4_implementation_manifest_v1.json"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

BR5_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR5_capability_probe_manifest_v1.json"
)

E1_DATA = (
    ROOT
    / "data"
    / "raw"
    / "intents_50.csv"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RUNNER = (
    ROOT
    / "src"
    / "experiments"
    / "run_all.py"
)

HIST_ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

HIST_E2A_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_E2A_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)


# ======================================================================================
# Locked future output paths
# ======================================================================================

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_v1.log"
)

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

E2A_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_v1.log"
)


# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# ======================================================================================
# 1. Frozen hashes
# ======================================================================================

EXPECTED = {

    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    BR4_MANIFEST:
        "c3e1a1fb75073ab2ca55c6dbae24c14bbbdb957ed4fc667883d7b425d5f9ceb8",

    BR5_PROBE:
        "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208",

    BR5_MANIFEST:
        "0094c6c9b66546bb97ef5c7c3b3fe6a8a520a2d5c2aaf958d7e58483bb807aeb",

    E1_DATA:
        "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826",

    E2A_DATA:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_RUNNER:
        "9caf14e350d70d1d7cc7321209893c4ef1d8893d31c4973a885fc482553cdc11",

    HIST_ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    HIST_E2A_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_E2A_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",
}


# ======================================================================================
# 2. Verify all frozen provenance
# ======================================================================================

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# ======================================================================================
# 3. Verify BR5 capability gate
# ======================================================================================

print("\n" + "-" * 118)
print("BR5 CAPABILITY GATE")
print("-" * 118)

probe = json.loads(
    BR5_PROBE.read_text(
        encoding="utf-8"
    )
)

probe_manifest = json.loads(
    BR5_MANIFEST.read_text(
        encoding="utf-8"
    )
)

assert (
    probe["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert probe["provider_calls"] == 1
assert probe["benchmark_cases_used"] == 0
assert probe["parse_ok"] is True
assert probe["structural_schema_valid"] is True

assert (
    probe_manifest["status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert probe_manifest["provider_calls"] == 1
assert probe_manifest["benchmark_cases"] == 0

print("✓ BR5 capability status = SUPPORTED_AND_PROBE_VALID")
print("✓ exactly one synthetic provider call recorded")
print("✓ zero benchmark cases used in BR5")
print("✓ structured parse successful")
print("✓ structural schema valid")


# ======================================================================================
# 4. Verify preregistered model and treatment
# ======================================================================================

print("\n" + "-" * 118)
print("PREREGISTERED TREATMENT")
print("-" * 118)

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg["semantic_model"]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"

assert (
    prereg["treatment"]["id"]
    == "cir_constrained"
)

schema = prereg["treatment"]["structural_schema"]

expected_fields = {
    "service_type",
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
    "coverage",
    "objective",
}

assert set(schema["properties"]) == expected_fields
assert set(schema["required"]) == expected_fields
assert schema["additionalProperties"] is False

for field, spec in schema["properties"].items():

    assert "enum" not in spec
    assert "minimum" not in spec
    assert "maximum" not in spec

print("✓ model =", MODEL)
print("✓ treatment = cir_constrained")
print("✓ structural-only schema retained")
print("✓ semantic/range constraints absent from decoder")


# ======================================================================================
# 5. Verify E1 benchmark identity
# ======================================================================================

print("\n" + "-" * 118)
print("E1 BENCHMARK")
print("-" * 118)

e1 = pd.read_csv(
    E1_DATA
)

assert len(e1) == 50

assert e1["id"].nunique() == 50

assert not e1["id"].isna().any()
assert not e1["intent"].isna().any()

print("Rows       :", len(e1))
print("Unique IDs :", e1["id"].nunique())
print("Columns    :", list(e1.columns))

print("\nExpected-valid distribution:")
print(
    e1["expected_valid"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nCategory distribution:")
print(
    e1["category"]
    .value_counts(dropna=False)
    .to_string()
)


# ======================================================================================
# 6. Verify E2A benchmark identity
# ======================================================================================

print("\n" + "-" * 118)
print("E2A BENCHMARK")
print("-" * 118)

e2a = pd.read_csv(
    E2A_DATA
)

assert len(e2a) == 200

assert e2a["id"].nunique() == 200

assert not e2a["id"].isna().any()
assert not e2a["intent"].isna().any()

print("Rows       :", len(e2a))
print("Unique IDs :", e2a["id"].nunique())
print("Columns    :", list(e2a.columns))

print("\nExpected-valid distribution:")
print(
    e2a["expected_valid"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nCategory distribution:")
print(
    e2a["category"]
    .value_counts(dropna=False)
    .to_string()
)


# ======================================================================================
# 7. Verify historical E2A arm composition
# ======================================================================================

print("\n" + "-" * 118)
print("HISTORICAL E2A CONTROL EVIDENCE")
print("-" * 118)

hist = pd.read_csv(
    HIST_E2A_RESULTS
)

assert len(hist) == 600

assert "pipeline" in hist.columns

counts = (
    hist["pipeline"]
    .value_counts()
    .to_dict()
)

assert counts.get("direct") == 200
assert counts.get("cir") == 200
assert counts.get("proposed") == 200

print("✓ historical direct   = 200")
print("✓ historical cir      = 200")
print("✓ historical proposed = 200")

hist_cir = (
    hist[
        hist["pipeline"] == "cir"
    ]
    .copy()
)

assert len(hist_cir) == 200

assert (
    set(hist_cir["id"].astype(str))
    == set(e2a["id"].astype(str))
)

print("✓ historical CIR contains exactly the frozen 200 E2A case IDs")


# ======================================================================================
# 8. Verify NO treatment benchmark output already exists
#
# This is a critical one-time execution gate.
# ======================================================================================

print("\n" + "-" * 118)
print("ONE-TIME EXECUTION OUTPUT GATE")
print("-" * 118)

for label, path in [
    ("E1 treatment result", E1_RESULT),
    ("E1 execution log", E1_LOG),
    ("E2A treatment result", E2A_RESULT),
    ("E2A execution log", E2A_LOG),
]:

    assert not path.exists(), (
        f"STOP: {label} already exists:\n{path}\n\n"
        "Do NOT execute the benchmark again."
    )

    print(
        f"✓ no prior {label}"
    )


# ======================================================================================
# 9. Verify API key without exposing it
# ======================================================================================

assert os.environ.get(
    "TOGETHER_API_KEY"
), (
    "STOP: TOGETHER_API_KEY unavailable."
)

print("\n✓ TOGETHER_API_KEY available")


# ======================================================================================
# 10. Verify implementation contains no historical-arm execution
# ======================================================================================

runner_source = RUNNER.read_text(
    encoding="utf-8"
)

assert '"cir_constrained"' in runner_source
assert '"direct"' not in runner_source
assert '"proposed"' not in runner_source

assert (
    runner_source.count(
        "call_json_structured("
    )
    == 1
)

print("✓ runner executes cir_constrained only")
print("✓ historical direct arm cannot execute")
print("✓ historical proposed arm cannot execute")
print("✓ one structured-generation call per benchmark row")


# ======================================================================================
# 11. Lock execution plan
# ======================================================================================

print("\n" + "-" * 118)
print("LOCKED EXECUTION PLAN")
print("-" * 118)

print("RUN 1 — E1")
print("Dataset :", E1_DATA)
print("Cases   : 50")
print("Output  :", E1_RESULT)
print("Log     :", E1_LOG)
print("Model   :", MODEL)

print()

print("RUN 2 — E2A")
print("Dataset :", E2A_DATA)
print("Cases   : 200")
print("Output  :", E2A_RESULT)
print("Log     :", E2A_LOG)
print("Model   :", MODEL)

print()
print("Total future benchmark provider calls: 250")
print("Historical arms to rerun             : 0")


# ======================================================================================
# 12. Final integrity verification
# ======================================================================================

print("\n" + "-" * 118)
print("FINAL PRE-EXECUTION INTEGRITY")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


print("\n" + "=" * 118)
print("✓ STEP BR6 COMPLETE")
print("✓ ALL FROZEN PROVENANCE VERIFIED")
print("✓ BR5 CAPABILITY GATE SATISFIED")
print("✓ E1 LOCKED AT 50 UNIQUE CASES")
print("✓ E2A LOCKED AT 200 UNIQUE CASES")
print("✓ HISTORICAL E2A CIR CONTROL VERIFIED")
print("✓ NO PRIOR B-REVIEW BENCHMARK OUTPUT DETECTED")
print("✓ OUTPUT PATHS LOCKED")
print("✓ MODEL LOCKED: Qwen/Qwen3.5-9B")
print("✓ TREATMENT LOCKED: cir_constrained")
print("✓ ZERO API/MODEL CALLS IN BR6")
print("✓ ZERO BENCHMARK CASES EXECUTED IN BR6")
print("✓ NEXT: EXECUTE E1 ONLY — 50 CASES")
print("✓ DO NOT EXECUTE E2A YET")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE THE E1 EXECUTION")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR6: FINAL ZERO-CALL BENCHMARK PREFLIGHT

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR4_implementation_manifest_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/capability_probe/experiment_B_review_BR5_capability_probe_v1.js

In [45]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import shutil
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR7: ONE-TIME E1 cir_constrained EXECUTION — 50 CASES")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

# ======================================================================================
# Frozen inputs
# ======================================================================================

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR4_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR4_implementation_manifest_v1.json"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

BR5_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR5_capability_probe_manifest_v1.json"
)

E1_DATA = (
    ROOT
    / "data"
    / "raw"
    / "intents_50.csv"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RUNNER = (
    ROOT
    / "src"
    / "experiments"
    / "run_all.py"
)

HIST_ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)


# ======================================================================================
# Locked E1 evidence paths
# ======================================================================================

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_v1.log"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR7_E1_execution_record_v1.json"
)


# ======================================================================================
# Persistent evidence paths
# ======================================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_RESULT = (
    DRIVE_ROOT
    / "results"
    / "E1"
    / E1_RESULT.name
)

DRIVE_LOG = (
    DRIVE_ROOT
    / "logs"
    / E1_LOG.name
)

DRIVE_EXEC_RECORD = (
    DRIVE_ROOT
    / "manifests"
    / EXEC_RECORD.name
)


# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# ======================================================================================
# 1. Frozen provenance
# ======================================================================================

EXPECTED = {

    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    BR4_MANIFEST:
        "c3e1a1fb75073ab2ca55c6dbae24c14bbbdb957ed4fc667883d7b425d5f9ceb8",

    BR5_PROBE:
        "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208",

    BR5_MANIFEST:
        "0094c6c9b66546bb97ef5c7c3b3fe6a8a520a2d5c2aaf958d7e58483bb807aeb",

    E1_DATA:
        "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826",

    E2A_DATA:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_RUNNER:
        "9caf14e350d70d1d7cc7321209893c4ef1d8893d31c4973a885fc482553cdc11",

    HIST_ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",
}


print("\n" + "-" * 118)
print("PRE-EXECUTION FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# ======================================================================================
# 2. Verify BR5 gate
# ======================================================================================

probe = json.loads(
    BR5_PROBE.read_text(
        encoding="utf-8"
    )
)

assert (
    probe["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert probe["provider_calls"] == 1
assert probe["benchmark_cases_used"] == 0
assert probe["parse_ok"] is True
assert probe["structural_schema_valid"] is True

print("\n✓ BR5 capability gate satisfied")


# ======================================================================================
# 3. Verify preregistered execution identity
# ======================================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg["semantic_model"]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"

assert (
    prereg["treatment"]["id"]
    == "cir_constrained"
)

print("✓ Model locked:", MODEL)
print("✓ Treatment locked: cir_constrained")


# ======================================================================================
# 4. Critical one-time execution gate
# ======================================================================================

print("\n" + "-" * 118)
print("ONE-TIME E1 EXECUTION GATE")
print("-" * 118)

for label, path in [
    ("E1 result", E1_RESULT),
    ("E1 log", E1_LOG),
    ("E1 execution record", EXEC_RECORD),
    ("persistent E1 result", DRIVE_RESULT),
    ("persistent E1 log", DRIVE_LOG),
    ("persistent E1 execution record", DRIVE_EXEC_RECORD),
]:

    assert not path.exists(), (
        f"STOP: {label} already exists:\n{path}\n\n"
        "DO NOT RERUN E1."
    )

    print(f"✓ no prior {label}")


# ======================================================================================
# 5. Verify E2A has NOT started
# ======================================================================================

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

assert not E2A_RESULT.exists(), (
    "STOP: E2A treatment output unexpectedly exists."
)

print("✓ E2A remains unexecuted")


# ======================================================================================
# 6. Verify API key without exposing it
# ======================================================================================

assert os.environ.get(
    "TOGETHER_API_KEY"
), (
    "STOP: TOGETHER_API_KEY unavailable."
)

print("✓ TOGETHER_API_KEY available")


# ======================================================================================
# 7. Create result/log directories only
# ======================================================================================

E1_RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

E1_LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

EXEC_RECORD.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================================
# 8. Lock command
# ======================================================================================

command = [
    sys.executable,
    str(RUNNER),

    "--dataset",
    str(E1_DATA),

    "--out",
    str(E1_RESULT),

    "--model",
    MODEL,
]


print("\n" + "-" * 118)
print("LOCKED E1 EXECUTION")
print("-" * 118)

print("Cases  : 50")
print("Model  :", MODEL)
print("Runner :", RUNNER)
print("Dataset:", E1_DATA)
print("Output :", E1_RESULT)
print("Log    :", E1_LOG)

print("\n*** THIS WILL NOW MAKE 50 BENCHMARK MODEL CALLS ***")
print("*** DO NOT INTERRUPT THE CELL IF POSSIBLE ***\n")


# ======================================================================================
# 9. Execute E1 ONCE
#
# stdout/stderr are streamed to screen AND retained in memory for freezing.
# ======================================================================================

started_utc = datetime.now(
    timezone.utc
).isoformat()

process = subprocess.Popen(
    command,

    cwd=str(ROOT),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,

    env=os.environ.copy(),
)

log_lines = []

assert process.stdout is not None

for line in process.stdout:

    print(
        line,
        end=""
    )

    log_lines.append(
        line
    )

return_code = process.wait()

finished_utc = datetime.now(
    timezone.utc
).isoformat()


# ======================================================================================
# 10. Freeze complete console log immediately
# ======================================================================================

E1_LOG.write_text(
    "".join(log_lines),
    encoding="utf-8"
)

print("\nProcess return code:", return_code)


# ======================================================================================
# 11. Require successful process
#
# IMPORTANT:
# If this fails, DO NOT rerun.
# Send the complete output for diagnosis.
# ======================================================================================

assert return_code == 0, (
    "STOP: E1 process returned non-zero status.\n"
    "DO NOT RERUN E1.\n"
    "The execution log has been preserved. Send the complete output."
)

assert E1_RESULT.exists(), (
    "STOP: runner returned success but E1 result file is missing.\n"
    "DO NOT RERUN."
)


# ======================================================================================
# 12. Validate raw result minimally
#
# No scientific analysis yet.
# ======================================================================================

import pandas as pd

e1_result = pd.read_csv(
    E1_RESULT
)

assert len(e1_result) == 50, (
    f"STOP: expected 50 E1 rows, found {len(e1_result)}.\n"
    "DO NOT RERUN."
)

assert e1_result["id"].nunique() == 50

expected_ids = set(
    pd.read_csv(
        E1_DATA
    )["id"].astype(str)
)

observed_ids = set(
    e1_result[
        "id"
    ].astype(str)
)

assert observed_ids == expected_ids, (
    "STOP: E1 result ID set differs from frozen E1 benchmark.\n"
    "DO NOT RERUN."
)

assert (
    set(
        e1_result[
            "pipeline"
        ].astype(str)
    )
    == {"cir_constrained"}
)

print("\n✓ E1 raw result contains exactly 50 rows")
print("✓ E1 contains exactly 50 unique benchmark IDs")
print("✓ E1 ID set matches frozen benchmark")
print("✓ all rows labeled cir_constrained")


# ======================================================================================
# 13. Freeze hashes
# ======================================================================================

result_sha = sha256(
    E1_RESULT
)

log_sha = sha256(
    E1_LOG
)

print("\n" + "-" * 118)
print("E1 OBSERVED EVIDENCE")
print("-" * 118)

print(
    "Result SHA-256:",
    result_sha
)

print(
    "Log SHA-256   :",
    log_sha
)


# ======================================================================================
# 14. Create execution record
# ======================================================================================

execution_record = {
    "experiment":
        "B-Review",

    "step":
        "BR7",

    "benchmark":
        "E1",

    "treatment":
        "cir_constrained",

    "model":
        MODEL,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "process_return_code":
        return_code,

    "benchmark_cases":
        50,

    "expected_provider_calls":
        50,

    "runner": {
        "path":
            str(
                RUNNER.relative_to(
                    ROOT
                )
            ),

        "sha256":
            EXPECTED[RUNNER],
    },

    "dataset": {
        "path":
            str(
                E1_DATA.relative_to(
                    ROOT
                )
            ),

        "sha256":
            EXPECTED[E1_DATA],
    },

    "preregistration_sha256":
        EXPECTED[PREREG],

    "BR5_capability_probe_sha256":
        EXPECTED[BR5_PROBE],

    "result": {
        "path":
            str(
                E1_RESULT.relative_to(
                    ROOT
                )
            ),

        "sha256":
            result_sha,

        "rows":
            50,
    },

    "log": {
        "path":
            str(
                E1_LOG.relative_to(
                    ROOT
                )
            ),

        "sha256":
            log_sha,
    },

    "status":
        "COMPLETED_AND_FROZEN_DO_NOT_RERUN",
}


EXEC_RECORD.write_text(
    json.dumps(
        execution_record,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

exec_sha = sha256(
    EXEC_RECORD
)

print(
    "Execution-record SHA-256:",
    exec_sha
)


# ======================================================================================
# 15. Persist E1 evidence to Drive
# ======================================================================================

for destination in [
    DRIVE_RESULT,
    DRIVE_LOG,
    DRIVE_EXEC_RECORD,
]:

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


shutil.copy2(
    E1_RESULT,
    DRIVE_RESULT
)

shutil.copy2(
    E1_LOG,
    DRIVE_LOG
)

shutil.copy2(
    EXEC_RECORD,
    DRIVE_EXEC_RECORD
)

assert (
    sha256(DRIVE_RESULT)
    == result_sha
)

assert (
    sha256(DRIVE_LOG)
    == log_sha
)

assert (
    sha256(DRIVE_EXEC_RECORD)
    == exec_sha
)

print("✓ E1 evidence persisted to Drive byte-for-byte")


# ======================================================================================
# 16. Post-execution frozen provenance
# ======================================================================================

print("\n" + "-" * 118)
print("POST-E1 FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# ======================================================================================
# 17. Reverify observed evidence
# ======================================================================================

assert (
    sha256(E1_RESULT)
    == result_sha
)

assert (
    sha256(E1_LOG)
    == log_sha
)

assert (
    sha256(EXEC_RECORD)
    == exec_sha
)

print("✓ E1 raw result frozen")
print("✓ E1 execution log frozen")
print("✓ E1 execution record frozen")


print("\n" + "=" * 118)
print("✓ STEP BR7 COMPLETE")
print("✓ E1 cir_constrained EXECUTED EXACTLY ONCE")
print("✓ 50/50 E1 CASES RECORDED")
print("✓ RAW E1 RESULT FROZEN")
print("✓ E1 LOG FROZEN")
print("✓ E1 EXECUTION RECORD FROZEN")
print("✓ EVIDENCE PERSISTED TO DRIVE")
print("✓ HISTORICAL ARTIFACTS REMAIN UNCHANGED")
print("✓ DO NOT RERUN E1")
print("✓ DO NOT EXECUTE E2A YET")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE CONTINUING")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR7: ONE-TIME E1 cir_constrained EXECUTION — 50 CASES

----------------------------------------------------------------------------------------------------------------------
PRE-EXECUTION FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR4_implementation_manifest_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/capability_probe/experiment_B_review

AssertionError: STOP: E1 process returned non-zero status.
DO NOT RERUN E1.
The execution log has been preserved. Send the complete output.

In [46]:
from pathlib import Path
import hashlib
import json
import shutil
import pandas as pd
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR7-D1: DIAGNOSE AND ARCHIVE FAILED ZERO-CALL E1 LAUNCH")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

E1_DATA = ROOT / "data/raw/intents_50.csv"

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

FAILED_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_v1.log"
)

FAILED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR7_failed_launch_record_v1.json"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_FAILED_LOG = (
    DRIVE_ROOT
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_v1.log"
)

DRIVE_FAILED_RECORD = (
    DRIVE_ROOT
    / "manifests"
    / FAILED_RECORD.name
)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


# ----------------------------------------------------------------------
# 1. Verify frozen experimental components
# ----------------------------------------------------------------------

EXPECTED = {
    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    BR5_PROBE:
        "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208",

    E1_DATA:
        "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826",
}

print("\n" + "-" * 118)
print("FROZEN COMPONENTS")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert path.exists(), f"STOP: missing {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: hash mismatch\n"
        f"{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# ----------------------------------------------------------------------
# 2. Inspect failed-launch evidence
# ----------------------------------------------------------------------

print("\n" + "-" * 118)
print("FAILED LAUNCH EVIDENCE")
print("-" * 118)

assert FAILED_LOG.exists(), (
    "STOP: expected BR7 failed-launch log is missing."
)

log_text = FAILED_LOG.read_text(
    encoding="utf-8",
    errors="replace"
)

print("Log SHA-256:", sha256(FAILED_LOG))
print("Log size    :", FAILED_LOG.stat().st_size, "bytes")

assert "ModuleNotFoundError" in log_text
assert "No module named 'src'" in log_text

print("✓ failure classified as Python import-path failure")


# ----------------------------------------------------------------------
# 3. Prove failure occurred before benchmark/model execution
# ----------------------------------------------------------------------

# The runner prints [1/50] only AFTER call_json_structured() has returned.
# Therefore absence of benchmark progress plus import-time traceback is
# consistent with zero benchmark observations.

progress_markers = [
    line
    for line in log_text.splitlines()
    if line.lstrip().startswith("[")
    and "/50]" in line
]

print("Benchmark progress markers:", len(progress_markers))

assert len(progress_markers) == 0, (
    "STOP: benchmark progress markers unexpectedly found."
)

# Result should not exist because import failed before runner execution.
assert not E1_RESULT.exists(), (
    "STOP: E1 result unexpectedly exists. "
    "Do not continue until inspected."
)

print("✓ no E1 result artifact exists")
print("✓ no benchmark progress marker exists")
print("✓ traceback occurred during module import")
print("✓ observed evidence supports ZERO E1 model calls")


# ----------------------------------------------------------------------
# 4. Confirm exact import that failed
# ----------------------------------------------------------------------

runner_text = RUNNER.read_text(
    encoding="utf-8"
)

assert "from src.validator import validate_cir" in runner_text

print("✓ failing import confirmed in frozen BR4 runner")


# ----------------------------------------------------------------------
# 5. Confirm project package exists
# ----------------------------------------------------------------------

SRC = ROOT / "src"
VALIDATOR = SRC / "validator.py"

assert SRC.exists()
assert VALIDATOR.exists()

print("✓ project src directory exists:", SRC)
print("✓ validator module exists      :", VALIDATOR)


# ----------------------------------------------------------------------
# 6. Diagnose subprocess path semantics WITHOUT executing runner
# ----------------------------------------------------------------------

runner_dir = RUNNER.parent.resolve()
root_dir = ROOT.resolve()

print("\nRunner directory:")
print(runner_dir)

print("\nProject root:")
print(root_dir)

assert runner_dir != root_dir

print()
print("Diagnosis:")
print(
    "The subprocess executed the runner by file path. "
    "Python's script directory is therefore the implementation directory, "
    "not necessarily the repository root containing src/."
)

print(
    "Corrective action will be execution-environment-only: "
    "prepend the repository root to subprocess PYTHONPATH."
)

print(
    "The frozen BR4 runner and treatment adapter will remain byte-for-byte unchanged."
)


# ----------------------------------------------------------------------
# 7. Archive failed launch as provenance
# ----------------------------------------------------------------------

assert not FAILED_RECORD.exists(), (
    "STOP: failed-launch record already exists. Do not overwrite."
)

failed_record = {
    "experiment": "B-Review",
    "step": "BR7-first-launch",
    "benchmark": "E1",
    "status": "FAILED_BEFORE_BENCHMARK_EXECUTION",
    "recorded_utc": datetime.now(timezone.utc).isoformat(),

    "failure_class":
        "subprocess_python_import_path",

    "exception":
        "ModuleNotFoundError: No module named 'src'",

    "observed_benchmark_progress_markers":
        0,

    "result_artifact_created":
        False,

    "evidence_based_provider_call_count":
        0,

    "reason_provider_call_count_is_zero":
        (
            "Python terminated while importing src.validator before "
            "module initialization reached main(), dataset iteration, "
            "or call_json_structured()."
        ),

    "scientific_treatment_started":
        False,

    "scientific_observations_generated":
        0,

    "corrective_action_allowed":
        (
            "Change subprocess execution environment only by prepending "
            "the repository root to PYTHONPATH. Do not modify the frozen "
            "runner, adapter, prompt, model, schema, temperature, dataset, "
            "or downstream logic."
        ),

    "failed_log": {
        "path": str(FAILED_LOG.relative_to(ROOT)),
        "sha256": sha256(FAILED_LOG),
    },

    "frozen_runner": {
        "path": str(RUNNER.relative_to(ROOT)),
        "sha256": EXPECTED[RUNNER],
    },

    "frozen_adapter": {
        "path": str(ADAPTER.relative_to(ROOT)),
        "sha256": EXPECTED[ADAPTER],
    },

    "dataset": {
        "path": str(E1_DATA.relative_to(ROOT)),
        "sha256": EXPECTED[E1_DATA],
    },

    "preregistration_sha256":
        EXPECTED[PREREG],
}

FAILED_RECORD.write_text(
    json.dumps(
        failed_record,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

record_sha = sha256(FAILED_RECORD)

print("\nFailed-launch record SHA-256:", record_sha)


# ----------------------------------------------------------------------
# 8. Persist failed evidence
# ----------------------------------------------------------------------

DRIVE_FAILED_LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_FAILED_RECORD.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Preserve exactly the failed log currently observed.
shutil.copy2(
    FAILED_LOG,
    DRIVE_FAILED_LOG
)

shutil.copy2(
    FAILED_RECORD,
    DRIVE_FAILED_RECORD
)

assert (
    sha256(DRIVE_FAILED_LOG)
    == sha256(FAILED_LOG)
)

assert (
    sha256(DRIVE_FAILED_RECORD)
    == record_sha
)

print("✓ failed-launch evidence persisted to Drive")


# ----------------------------------------------------------------------
# 9. Final frozen-component integrity
# ----------------------------------------------------------------------

for path, expected in EXPECTED.items():
    assert sha256(path) == expected

print("✓ all frozen experimental components remain unchanged")


print("\n" + "=" * 118)
print("✓ STEP BR7-D1 COMPLETE")
print("✓ FAILED BR7 LAUNCH ARCHIVED")
print("✓ FAILURE OCCURRED DURING PYTHON IMPORT")
print("✓ ZERO E1 BENCHMARK OBSERVATIONS GENERATED")
print("✓ ZERO E1 MODEL CALLS SUPPORTED BY EXECUTION EVIDENCE")
print("✓ NO E1 RESULT FILE EXISTS")
print("✓ FROZEN RUNNER NOT MODIFIED")
print("✓ FROZEN ADAPTER NOT MODIFIED")
print("✓ CORRECTIVE ACTION LIMITED TO SUBPROCESS PYTHONPATH")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ SEND THIS OUTPUT BEFORE BR7-R1")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR7-D1: DIAGNOSE AND ARCHIVE FAILED ZERO-CALL E1 LAUNCH

----------------------------------------------------------------------------------------------------------------------
FROZEN COMPONENTS
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/capability_probe/experiment_B_review_BR5_capability_probe_v1.json
✓ data/raw/intents_50.csv

----------------------------------------------------------------------------------------------

In [47]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import shutil
import pandas as pd
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR7-R1: ONE-TIME E1 EXECUTION WITH CORRECTED PYTHONPATH — 50 CASES")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

# ======================================================================================
# Frozen components
# ======================================================================================

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

E1_DATA = ROOT / "data/raw/intents_50.csv"

FAILED_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_v1.log"
)

FAILED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR7_failed_launch_record_v1.json"
)


# ======================================================================================
# Successful-run evidence paths
# ======================================================================================

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_SUCCESS_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_BR7R1_v1.log"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR7R1_E1_execution_record_v1.json"
)


# ======================================================================================
# Persistent evidence
# ======================================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_RESULT = (
    DRIVE_ROOT
    / "results"
    / "E1"
    / E1_RESULT.name
)

DRIVE_SUCCESS_LOG = (
    DRIVE_ROOT
    / "logs"
    / E1_SUCCESS_LOG.name
)

DRIVE_EXEC_RECORD = (
    DRIVE_ROOT
    / "manifests"
    / EXEC_RECORD.name
)


# ======================================================================================
# Helpers
# ======================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# ======================================================================================
# 1. Locked hashes
# ======================================================================================

EXPECTED_PREREG = (
    "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba"
)

EXPECTED_ADAPTER = (
    "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad"
)

EXPECTED_RUNNER = (
    "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851"
)

EXPECTED_BR5 = (
    "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208"
)

EXPECTED_E1 = (
    "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826"
)

EXPECTED_FAILED_LOG = (
    "ce567a5d8fb4fae209465247c286d56e369b0f92015f2fb28911c9d1ae77834d"
)

EXPECTED_FAILED_RECORD = (
    "25ccd208edce6d5321c9a15da182641169471797d952c4800b36a10f7e3a05b8"
)


# ======================================================================================
# 2. Verify frozen provenance INCLUDING failed launch
# ======================================================================================

print("\n" + "-" * 118)
print("PRE-EXECUTION PROVENANCE")
print("-" * 118)

verify(
    PREREG,
    EXPECTED_PREREG,
    "B-Review preregistration"
)

verify(
    ADAPTER,
    EXPECTED_ADAPTER,
    "frozen treatment adapter"
)

verify(
    RUNNER,
    EXPECTED_RUNNER,
    "frozen treatment runner"
)

verify(
    BR5_PROBE,
    EXPECTED_BR5,
    "BR5 capability probe"
)

verify(
    E1_DATA,
    EXPECTED_E1,
    "E1 benchmark"
)

verify(
    FAILED_LOG,
    EXPECTED_FAILED_LOG,
    "archived zero-call failed-launch log"
)

verify(
    FAILED_RECORD,
    EXPECTED_FAILED_RECORD,
    "archived zero-call failed-launch record"
)


# ======================================================================================
# 3. Verify failed launch formally recorded zero observations
# ======================================================================================

failure = json.loads(
    FAILED_RECORD.read_text(
        encoding="utf-8"
    )
)

assert (
    failure["status"]
    == "FAILED_BEFORE_BENCHMARK_EXECUTION"
)

assert (
    failure["evidence_based_provider_call_count"]
    == 0
)

assert (
    failure["scientific_observations_generated"]
    == 0
)

assert (
    failure["scientific_treatment_started"]
    is False
)

print("✓ prior failed launch formally recorded as zero-call/zero-observation")


# ======================================================================================
# 4. Verify BR5 capability
# ======================================================================================

probe = json.loads(
    BR5_PROBE.read_text(
        encoding="utf-8"
    )
)

assert (
    probe["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert probe["parse_ok"] is True
assert probe["structural_schema_valid"] is True

print("✓ BR5 capability gate remains satisfied")


# ======================================================================================
# 5. Verify one-time successful execution gate
# ======================================================================================

print("\n" + "-" * 118)
print("BR7-R1 ONE-TIME EXECUTION GATE")
print("-" * 118)

for label, path in [
    ("E1 result", E1_RESULT),
    ("BR7-R1 success log", E1_SUCCESS_LOG),
    ("BR7-R1 execution record", EXEC_RECORD),
    ("persistent E1 result", DRIVE_RESULT),
    ("persistent BR7-R1 success log", DRIVE_SUCCESS_LOG),
    ("persistent BR7-R1 execution record", DRIVE_EXEC_RECORD),
]:

    assert not path.exists(), (
        f"STOP: {label} already exists:\n{path}\n\n"
        "DO NOT execute E1 again."
    )

    print(f"✓ no prior {label}")


# ======================================================================================
# 6. E2A must remain untouched
# ======================================================================================

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

assert not E2A_RESULT.exists()

print("✓ E2A remains unexecuted")


# ======================================================================================
# 7. API key
# ======================================================================================

assert os.environ.get(
    "TOGETHER_API_KEY"
), (
    "STOP: TOGETHER_API_KEY unavailable."
)

print("✓ TOGETHER_API_KEY available")


# ======================================================================================
# 8. Locked experimental identity
# ======================================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg["semantic_model"]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"
assert prereg["treatment"]["id"] == "cir_constrained"

e1_input = pd.read_csv(E1_DATA)

assert len(e1_input) == 50
assert e1_input["id"].nunique() == 50

print("✓ model locked:", MODEL)
print("✓ treatment locked: cir_constrained")
print("✓ E1 locked at 50 cases")


# ======================================================================================
# 9. Correct subprocess environment ONLY
#
# No runner modification.
# No adapter modification.
# No prompt/schema/model change.
# ======================================================================================

child_env = os.environ.copy()

existing_pythonpath = child_env.get(
    "PYTHONPATH",
    ""
)

if existing_pythonpath:

    child_env["PYTHONPATH"] = (
        str(ROOT)
        + os.pathsep
        + existing_pythonpath
    )

else:

    child_env["PYTHONPATH"] = str(ROOT)


assert (
    child_env["PYTHONPATH"]
    .split(os.pathsep)[0]
    == str(ROOT)
)

print("\n✓ subprocess PYTHONPATH correction prepared")
print("✓ project root prepended to child PYTHONPATH")
print("✓ frozen runner remains unchanged")
print("✓ frozen adapter remains unchanged")


# ======================================================================================
# 10. Create evidence directories
# ======================================================================================

E1_RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

E1_SUCCESS_LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

EXEC_RECORD.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================================
# 11. Locked command
# ======================================================================================

command = [
    sys.executable,
    str(RUNNER),

    "--dataset",
    str(E1_DATA),

    "--out",
    str(E1_RESULT),

    "--model",
    MODEL,
]


print("\n" + "-" * 118)
print("LOCKED BR7-R1 EXECUTION")
print("-" * 118)

print("Cases             : 50")
print("Model             :", MODEL)
print("Treatment         : cir_constrained")
print("Runner SHA-256    :", EXPECTED_RUNNER)
print("Adapter SHA-256   :", EXPECTED_ADAPTER)
print("Dataset SHA-256   :", EXPECTED_E1)
print("Output            :", E1_RESULT)
print("Success log       :", E1_SUCCESS_LOG)
print("Environment change: PYTHONPATH only")

print("\n*** 50 E1 BENCHMARK MODEL CALLS WILL NOW EXECUTE ***")
print("*** DO NOT INTERRUPT IF POSSIBLE ***\n")


# ======================================================================================
# 12. Execute exactly once
# ======================================================================================

started_utc = datetime.now(
    timezone.utc
).isoformat()

process = subprocess.Popen(
    command,

    cwd=str(ROOT),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,

    env=child_env,
)

log_lines = []

assert process.stdout is not None

for line in process.stdout:

    print(
        line,
        end=""
    )

    log_lines.append(
        line
    )

return_code = process.wait()

finished_utc = datetime.now(
    timezone.utc
).isoformat()


# ======================================================================================
# 13. Freeze successful-attempt log immediately
# ======================================================================================

E1_SUCCESS_LOG.write_text(
    "".join(log_lines),
    encoding="utf-8"
)

print("\nProcess return code:", return_code)


# ======================================================================================
# 14. Stop on any execution failure
# ======================================================================================

assert return_code == 0, (
    "STOP: BR7-R1 returned non-zero status.\n"
    "DO NOT RERUN.\n"
    "The BR7-R1 log is preserved. Send the complete output."
)

assert E1_RESULT.exists(), (
    "STOP: BR7-R1 returned success but result file is absent.\n"
    "DO NOT RERUN."
)


# ======================================================================================
# 15. Minimal evidence validation
# ======================================================================================

result = pd.read_csv(
    E1_RESULT
)

assert len(result) == 50, (
    f"STOP: expected 50 rows; observed {len(result)}."
)

assert result["id"].nunique() == 50

assert (
    set(result["id"].astype(str))
    == set(e1_input["id"].astype(str))
)

assert (
    set(result["pipeline"].astype(str))
    == {"cir_constrained"}
)

print("\n✓ exactly 50 E1 result rows")
print("✓ exactly 50 unique E1 IDs")
print("✓ result IDs match frozen E1 benchmark")
print("✓ all observations labeled cir_constrained")


# ======================================================================================
# 16. Freeze hashes
# ======================================================================================

result_sha = sha256(
    E1_RESULT
)

success_log_sha = sha256(
    E1_SUCCESS_LOG
)

print("\n" + "-" * 118)
print("OBSERVED BR7-R1 EVIDENCE")
print("-" * 118)

print("Result SHA-256 :", result_sha)
print("Log SHA-256    :", success_log_sha)


# ======================================================================================
# 17. Execution record
# ======================================================================================

record = {
    "experiment":
        "B-Review",

    "step":
        "BR7-R1",

    "benchmark":
        "E1",

    "status":
        "COMPLETED_AND_FROZEN_DO_NOT_RERUN",

    "model":
        MODEL,

    "treatment":
        "cir_constrained",

    "benchmark_cases":
        50,

    "expected_provider_calls":
        50,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "process_return_code":
        return_code,

    "execution_environment_correction": {
        "type":
            "PYTHONPATH",

        "change":
            "repository root prepended to child process PYTHONPATH",

        "scientific_code_modified":
            False,

        "runner_modified":
            False,

        "adapter_modified":
            False,
    },

    "prior_failed_launch": {
        "status":
            "FAILED_BEFORE_BENCHMARK_EXECUTION",

        "provider_calls":
            0,

        "scientific_observations":
            0,

        "record_sha256":
            EXPECTED_FAILED_RECORD,

        "log_sha256":
            EXPECTED_FAILED_LOG,
    },

    "preregistration_sha256":
        EXPECTED_PREREG,

    "runner_sha256":
        EXPECTED_RUNNER,

    "adapter_sha256":
        EXPECTED_ADAPTER,

    "dataset_sha256":
        EXPECTED_E1,

    "BR5_probe_sha256":
        EXPECTED_BR5,

    "result": {
        "path":
            str(
                E1_RESULT.relative_to(
                    ROOT
                )
            ),

        "sha256":
            result_sha,

        "rows":
            50,
    },

    "log": {
        "path":
            str(
                E1_SUCCESS_LOG.relative_to(
                    ROOT
                )
            ),

        "sha256":
            success_log_sha,
    },
}


EXEC_RECORD.write_text(
    json.dumps(
        record,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

record_sha = sha256(
    EXEC_RECORD
)

print(
    "Execution record:",
    record_sha
)


# ======================================================================================
# 18. Persist successful evidence
# ======================================================================================

for destination in [
    DRIVE_RESULT,
    DRIVE_SUCCESS_LOG,
    DRIVE_EXEC_RECORD,
]:

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


shutil.copy2(
    E1_RESULT,
    DRIVE_RESULT
)

shutil.copy2(
    E1_SUCCESS_LOG,
    DRIVE_SUCCESS_LOG
)

shutil.copy2(
    EXEC_RECORD,
    DRIVE_EXEC_RECORD
)

assert sha256(DRIVE_RESULT) == result_sha

assert (
    sha256(DRIVE_SUCCESS_LOG)
    == success_log_sha
)

assert (
    sha256(DRIVE_EXEC_RECORD)
    == record_sha
)

print("✓ successful E1 evidence persisted to Drive")


# ======================================================================================
# 19. Final integrity
# ======================================================================================

assert sha256(PREREG) == EXPECTED_PREREG
assert sha256(ADAPTER) == EXPECTED_ADAPTER
assert sha256(RUNNER) == EXPECTED_RUNNER
assert sha256(BR5_PROBE) == EXPECTED_BR5
assert sha256(E1_DATA) == EXPECTED_E1

assert sha256(FAILED_LOG) == EXPECTED_FAILED_LOG
assert sha256(FAILED_RECORD) == EXPECTED_FAILED_RECORD

assert sha256(E1_RESULT) == result_sha
assert sha256(E1_SUCCESS_LOG) == success_log_sha
assert sha256(EXEC_RECORD) == record_sha

print("\n✓ all frozen scientific components unchanged")
print("✓ failed-launch provenance unchanged")
print("✓ successful E1 evidence frozen")


print("\n" + "=" * 118)
print("✓ STEP BR7-R1 COMPLETE")
print("✓ E1 cir_constrained EXECUTED SUCCESSFULLY")
print("✓ 50/50 E1 OBSERVATIONS RECORDED")
print("✓ FAILED ZERO-CALL LAUNCH RETAINED AS PROVENANCE")
print("✓ ONLY CORRECTION WAS CHILD PYTHONPATH")
print("✓ FROZEN RUNNER/ADAPTER WERE NOT MODIFIED")
print("✓ E1 RESULT / LOG / EXECUTION RECORD FROZEN")
print("✓ EVIDENCE PERSISTED TO DRIVE")
print("✓ DO NOT RERUN E1")
print("✓ DO NOT EXECUTE E2A YET")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE CONTINUING")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR7-R1: ONE-TIME E1 EXECUTION WITH CORRECTED PYTHONPATH — 50 CASES

----------------------------------------------------------------------------------------------------------------------
PRE-EXECUTION PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ B-Review preregistration
✓ frozen treatment adapter
✓ frozen treatment runner
✓ BR5 capability probe
✓ E1 benchmark
✓ archived zero-call failed-launch log
✓ archived zero-call failed-launch record
✓ prior failed launch formally recorded as zero-call/zero-observation
✓ BR5 capability gate remains satisfied

----------------------------------------------------------------------------------------------------------------------
BR7-R1 ONE-TIME EXECUTION GATE
----------------------------------------------------------------------------------------------------------------------
✓ no prior E1 result
✓ no prior BR7-R1 success log
✓ no prior B

In [48]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from collections import Counter

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR8: READ-ONLY ANALYSIS OF E1 cir_constrained")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_SUCCESS_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E1_cir_constrained_50_BR7R1_v1.log"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR7R1_E1_execution_record_v1.json"
)

E1_DATA = ROOT / "data/raw/intents_50.csv"

HIST_E1_RESULTS = (
    ROOT
    / "data"
    / "processed"
    / "results.csv"
)

# Historical E1 results location may differ, so discover only if exact
# canonical path above is absent.
HIST_RUNNER = ROOT / "src/experiments/run_all.py"


def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 1. Verify frozen E1 treatment evidence
# ======================================================================================

EXPECTED_RESULT = (
    "21ef335ea6218edb01b30dccc1461520767a7d1e9ea2e6956e7ba267dc29ed68"
)

EXPECTED_LOG = (
    "b4cc7295a00d4d9946dc0042b1eafd87e97b134fc506667c9ccfcc50d9656bbf"
)

EXPECTED_EXEC = (
    "4aaf738112b9d6b448bf7dcc068ce57735724f5f316952fe5eb0cde78803526e"
)

EXPECTED_E1_DATA = (
    "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826"
)

print("\n" + "-" * 118)
print("FROZEN EVIDENCE")
print("-" * 118)

for path, expected, label in [
    (E1_RESULT, EXPECTED_RESULT, "E1 treatment result"),
    (E1_SUCCESS_LOG, EXPECTED_LOG, "E1 successful execution log"),
    (EXEC_RECORD, EXPECTED_EXEC, "E1 execution record"),
    (E1_DATA, EXPECTED_E1_DATA, "frozen E1 benchmark"),
]:

    assert path.exists(), f"STOP: missing {label}: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# ======================================================================================
# 2. Load treatment + ground truth
# ======================================================================================

result = pd.read_csv(E1_RESULT)
truth = pd.read_csv(E1_DATA)

assert len(result) == 50
assert len(truth) == 50

assert result["id"].nunique() == 50
assert truth["id"].nunique() == 50

assert set(result["id"].astype(str)) == set(truth["id"].astype(str))

df = result.merge(
    truth[
        [
            "id",
            "expected_valid",
            "expected_service",
            "category",
        ]
    ],
    on="id",
    how="inner",
    suffixes=("_result", "_truth"),
    validate="one_to_one",
)

assert len(df) == 50

print("✓ treatment result joined one-to-one with frozen E1 ground truth")


# ======================================================================================
# 3. Normalize booleans robustly
# ======================================================================================

def as_bool(v):

    if isinstance(v, bool):
        return v

    if pd.isna(v):
        return False

    s = str(v).strip().lower()

    if s in {
        "true",
        "1",
        "yes",
    }:
        return True

    if s in {
        "false",
        "0",
        "no",
        "",
        "none",
        "nan",
    }:
        return False

    raise ValueError(
        f"Cannot interpret boolean value: {v!r}"
    )


df["expected_valid_bool"] = (
    df["expected_valid_truth"]
    .map(as_bool)
)

df["accepted_bool"] = (
    df["accepted"]
    .map(as_bool)
)

df["parse_ok_bool"] = (
    df["parse_ok"]
    .map(as_bool)
)


# ======================================================================================
# 4. Verify ground-truth balance
# ======================================================================================

n_valid = int(
    df["expected_valid_bool"].sum()
)

n_invalid = len(df) - n_valid

assert n_valid == 25
assert n_invalid == 25

print("✓ ground truth = 25 valid / 25 invalid")


# ======================================================================================
# 5. Structured-generation success
# ======================================================================================

parse_ok_n = int(
    df["parse_ok_bool"].sum()
)

parse_fail_n = len(df) - parse_ok_n

print("\n" + "-" * 118)
print("STRUCTURED-GENERATION OBSERVATIONS")
print("-" * 118)

print(
    f"Parse OK   : {parse_ok_n}/50 "
    f"({100 * parse_ok_n / 50:.2f}%)"
)

print(
    f"Parse fail : {parse_fail_n}/50 "
    f"({100 * parse_fail_n / 50:.2f}%)"
)


# ======================================================================================
# 6. Decision confusion matrix
#
# Positive class = invalid/reject, matching our established convention:
#
# TP = expected invalid and rejected
# FP = expected valid but rejected
# FN = expected invalid but accepted
# TN = expected valid and accepted
# ======================================================================================

expected_invalid = ~df["expected_valid_bool"]
predicted_reject = ~df["accepted_bool"]

TP = int(
    (
        expected_invalid
        & predicted_reject
    ).sum()
)

FP = int(
    (
        ~expected_invalid
        & predicted_reject
    ).sum()
)

FN = int(
    (
        expected_invalid
        & ~predicted_reject
    ).sum()
)

TN = int(
    (
        ~expected_invalid
        & ~predicted_reject
    ).sum()
)

assert TP + FP + FN + TN == 50

accuracy = (
    TP + TN
) / 50

# Valid Acceptance Rate
VAR = (
    TN / (TN + FP)
    if (TN + FP)
    else float("nan")
)

# Invalid Rejection Rate
IRR = (
    TP / (TP + FN)
    if (TP + FN)
    else float("nan")
)

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else float("nan")
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else float("nan")
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if precision + recall
    else float("nan")
)


print("\n" + "-" * 118)
print("E1 cir_constrained DECISION RESULTS")
print("-" * 118)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")

print()

print(
    f"Decision accuracy : "
    f"{accuracy:.6f} ({accuracy*100:.2f}%)"
)

print(
    f"VAR               : "
    f"{VAR:.6f} ({VAR*100:.2f}%)"
)

print(
    f"IRR               : "
    f"{IRR:.6f} ({IRR*100:.2f}%)"
)

print(
    f"Precision         : {precision:.6f}"
)

print(
    f"Recall            : {recall:.6f}"
)

print(
    f"F1                : {f1:.6f}"
)


# ======================================================================================
# 7. Service extraction accuracy on expected-valid cases
# ======================================================================================

valid_df = df[
    df["expected_valid_bool"]
].copy()

service_comparable = valid_df[
    valid_df["expected_service_truth"].notna()
].copy()

service_correct = (
    service_comparable[
        "predicted_service"
    ].astype(str)
    ==
    service_comparable[
        "expected_service_truth"
    ].astype(str)
)

service_correct_n = int(
    service_correct.sum()
)

service_total = len(
    service_comparable
)

print("\n" + "-" * 118)
print("SERVICE EXTRACTION — EXPECTED-VALID CASES")
print("-" * 118)

print(
    f"Correct service: "
    f"{service_correct_n}/{service_total}"
)

if service_total:

    print(
        f"Service accuracy: "
        f"{100*service_correct_n/service_total:.2f}%"
    )


# ======================================================================================
# 8. Category-level decision performance
# ======================================================================================

category_col = (
    "category_truth"
    if "category_truth" in df.columns
    else "category"
)

rows = []

for category, group in df.groupby(
    category_col,
    dropna=False
):

    correct = (
        group["expected_valid_bool"]
        == group["accepted_bool"]
    )

    rows.append(
        {
            "category":
                category,

            "n":
                len(group),

            "correct":
                int(correct.sum()),

            "accuracy":
                float(correct.mean()),
        }
    )

category_results = pd.DataFrame(
    rows
).sort_values(
    ["category"]
)

print("\n" + "-" * 118)
print("CATEGORY-LEVEL DECISION PERFORMANCE")
print("-" * 118)

for _, r in category_results.iterrows():

    print(
        f"{str(r['category']):28s} "
        f"{int(r['correct']):2d}/{int(r['n']):2d} "
        f"({100*r['accuracy']:6.2f}%)"
    )


# ======================================================================================
# 9. Error inspection — IDs only
#
# Do not change anything based on these observations.
# ======================================================================================

decision_correct = (
    df["expected_valid_bool"]
    == df["accepted_bool"]
)

errors = df[
    ~decision_correct
].copy()

print("\n" + "-" * 118)
print("DECISION ERROR IDs")
print("-" * 118)

if len(errors) == 0:

    print("<none>")

else:

    for _, r in errors.iterrows():

        print(
            f"{r['id']} | "
            f"category={r[category_col]} | "
            f"expected_valid={r['expected_valid_bool']} | "
            f"accepted={r['accepted_bool']} | "
            f"expected_service={r['expected_service_truth']} | "
            f"predicted_service={r['predicted_service']}"
        )


# ======================================================================================
# 10. Locate historical E1 results READ-ONLY
#
# We do NOT reconstruct or rerun historical E1.
# ======================================================================================

print("\n" + "-" * 118)
print("HISTORICAL E1 CONTROL DISCOVERY")
print("-" * 118)

candidate_paths = [
    ROOT / "data/processed/results.csv",
    ROOT / "results.csv",
    ROOT / "data/results.csv",
]

historical_path = None

for p in candidate_paths:

    if p.exists():

        try:

            tmp = pd.read_csv(p)

            if (
                "pipeline" in tmp.columns
                and "id" in tmp.columns
            ):

                ids = set(
                    tmp["id"].astype(str)
                )

                e1_ids = set(
                    truth["id"].astype(str)
                )

                if e1_ids.issubset(ids):

                    historical_path = p
                    break

        except Exception:

            pass


if historical_path is None:

    print(
        "Historical E1 result table was not found at the "
        "bounded canonical candidate paths."
    )

    print(
        "No historical comparison will be invented."
    )

else:

    print(
        "Historical result table:",
        historical_path
    )

    hist = pd.read_csv(
        historical_path
    )

    print(
        "Historical table SHA-256:",
        sha256(historical_path)
    )

    hist_e1 = hist[
        hist["id"]
        .astype(str)
        .isin(
            set(
                truth["id"].astype(str)
            )
        )
    ].copy()

    print(
        "Historical E1 rows recovered:",
        len(hist_e1)
    )

    if "pipeline" in hist_e1.columns:

        print(
            "\nHistorical pipeline counts:"
        )

        print(
            hist_e1[
                "pipeline"
            ]
            .value_counts()
            .to_string()
        )


# ======================================================================================
# 11. Post-analysis evidence integrity
# ======================================================================================

assert (
    sha256(E1_RESULT)
    == EXPECTED_RESULT
)

assert (
    sha256(E1_SUCCESS_LOG)
    == EXPECTED_LOG
)

assert (
    sha256(EXEC_RECORD)
    == EXPECTED_EXEC
)

assert (
    sha256(E1_DATA)
    == EXPECTED_E1_DATA
)

print("\n" + "-" * 118)
print("POST-ANALYSIS INTEGRITY")
print("-" * 118)

print("✓ E1 treatment result unchanged")
print("✓ E1 execution log unchanged")
print("✓ E1 execution record unchanged")
print("✓ frozen E1 benchmark unchanged")


print("\n" + "=" * 118)
print("✓ STEP BR8 COMPLETE")
print("✓ E1 TREATMENT ANALYZED READ-ONLY")
print("✓ STRUCTURED-GENERATION PARSE SUCCESS COMPUTED")
print("✓ DECISION METRICS COMPUTED")
print("✓ SERVICE EXTRACTION CHECKED")
print("✓ CATEGORY PERFORMANCE COMPUTED")
print("✓ ERROR IDs REPORTED")
print("✓ HISTORICAL E1 CONTROL SEARCH WAS READ-ONLY")
print("✓ NO API/MODEL CALL")
print("✓ NO BENCHMARK EXECUTION")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE E2A")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR8: READ-ONLY ANALYSIS OF E1 cir_constrained

----------------------------------------------------------------------------------------------------------------------
FROZEN EVIDENCE
----------------------------------------------------------------------------------------------------------------------
✓ E1 treatment result
✓ E1 successful execution log
✓ E1 execution record
✓ frozen E1 benchmark
✓ treatment result joined one-to-one with frozen E1 ground truth
✓ ground truth = 25 valid / 25 invalid

----------------------------------------------------------------------------------------------------------------------
STRUCTURED-GENERATION OBSERVATIONS
----------------------------------------------------------------------------------------------------------------------
Parse OK   : 50/50 (100.00%)
Parse fail : 0/50 (0.00%)

----------------------------------------------------------------------------------------------------------------------
E1 cir_constrained DECI

In [49]:
from pathlib import Path
import hashlib
import json
import os
import pandas as pd

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR9: FINAL ZERO-CALL E2A EXECUTION GATE")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR7R1_E1_execution_record_v1.json"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_E2A_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_E2A_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)


# =============================================================================
# Future E2A evidence paths
# =============================================================================

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

E2A_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_v1.log"
)

E2A_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_E2A_execution_record_v1.json"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_E2A_RESULT = (
    DRIVE_ROOT
    / "results"
    / "E2A"
    / E2A_RESULT.name
)

DRIVE_E2A_LOG = (
    DRIVE_ROOT
    / "logs"
    / E2A_LOG.name
)

DRIVE_E2A_EXEC = (
    DRIVE_ROOT
    / "manifests"
    / E2A_EXEC.name
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# =============================================================================
# 1. Frozen hashes
# =============================================================================

EXPECTED = {

    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    BR5_PROBE:
        "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208",

    E1_RESULT:
        "21ef335ea6218edb01b30dccc1461520767a7d1e9ea2e6956e7ba267dc29ed68",

    E1_EXEC:
        "4aaf738112b9d6b448bf7dcc068ce57735724f5f316952fe5eb0cde78803526e",

    E2A_DATA:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_E2A_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_E2A_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",
}


# =============================================================================
# 2. Verify frozen provenance
# =============================================================================

print("\n" + "-" * 118)
print("FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# =============================================================================
# 3. Verify successful E1 completion
# =============================================================================

print("\n" + "-" * 118)
print("E1 COMPLETION GATE")
print("-" * 118)

e1_exec = json.loads(
    E1_EXEC.read_text(
        encoding="utf-8"
    )
)

assert (
    e1_exec["status"]
    == "COMPLETED_AND_FROZEN_DO_NOT_RERUN"
)

assert e1_exec["benchmark_cases"] == 50
assert e1_exec["process_return_code"] == 0

e1 = pd.read_csv(E1_RESULT)

assert len(e1) == 50
assert e1["id"].nunique() == 50

print("✓ E1 completed and frozen")
print("✓ 50 E1 observations preserved")
print("✓ E1 will not be rerun")


# =============================================================================
# 4. Verify BR5 capability gate
# =============================================================================

probe = json.loads(
    BR5_PROBE.read_text(
        encoding="utf-8"
    )
)

assert (
    probe["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert probe["parse_ok"] is True
assert probe["structural_schema_valid"] is True

print("✓ structured-decoding capability gate remains satisfied")


# =============================================================================
# 5. Verify treatment identity
# =============================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg["semantic_model"]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"
assert prereg["treatment"]["id"] == "cir_constrained"

schema = prereg["treatment"]["structural_schema"]

for _, spec in schema["properties"].items():

    assert "enum" not in spec
    assert "minimum" not in spec
    assert "maximum" not in spec

print("✓ model locked:", MODEL)
print("✓ treatment locked: cir_constrained")
print("✓ structural-only schema retained")


# =============================================================================
# 6. Verify E2A benchmark
# =============================================================================

print("\n" + "-" * 118)
print("E2A BENCHMARK")
print("-" * 118)

e2a = pd.read_csv(E2A_DATA)

assert len(e2a) == 200
assert e2a["id"].nunique() == 200

assert not e2a["id"].isna().any()
assert not e2a["intent"].isna().any()

print("✓ rows       :", len(e2a))
print("✓ unique IDs :", e2a["id"].nunique())

print("\nExpected-valid distribution:")
print(
    e2a["expected_valid"]
    .value_counts(dropna=False)
    .to_string()
)


# =============================================================================
# 7. Verify historical E2A control evidence
# =============================================================================

print("\n" + "-" * 118)
print("HISTORICAL E2A CONTROLS")
print("-" * 118)

hist = pd.read_csv(HIST_E2A_RESULTS)

assert len(hist) == 600

counts = (
    hist["pipeline"]
    .value_counts()
    .to_dict()
)

assert counts.get("direct") == 200
assert counts.get("cir") == 200
assert counts.get("proposed") == 200

hist_cir = hist[
    hist["pipeline"] == "cir"
].copy()

assert len(hist_cir) == 200

assert (
    set(hist_cir["id"].astype(str))
    == set(e2a["id"].astype(str))
)

print("✓ historical Direct   = 200")
print("✓ historical CIR      = 200")
print("✓ historical Proposed = 200")
print("✓ historical CIR case-ID set exactly matches E2A")


# =============================================================================
# 8. Critical one-time E2A output gate
# =============================================================================

print("\n" + "-" * 118)
print("ONE-TIME E2A OUTPUT GATE")
print("-" * 118)

for label, path in [

    ("E2A result", E2A_RESULT),
    ("E2A log", E2A_LOG),
    ("E2A execution record", E2A_EXEC),

    ("persistent E2A result", DRIVE_E2A_RESULT),
    ("persistent E2A log", DRIVE_E2A_LOG),
    ("persistent E2A execution record", DRIVE_E2A_EXEC),
]:

    assert not path.exists(), (
        f"STOP: {label} already exists:\n{path}\n"
        "DO NOT EXECUTE E2A."
    )

    print(f"✓ no prior {label}")


# =============================================================================
# 9. Verify API key
# =============================================================================

assert os.environ.get(
    "TOGETHER_API_KEY"
), (
    "STOP: TOGETHER_API_KEY unavailable."
)

print("\n✓ TOGETHER_API_KEY available")


# =============================================================================
# 10. Lock corrected subprocess environment
# =============================================================================

child_env = os.environ.copy()

existing_pythonpath = child_env.get(
    "PYTHONPATH",
    ""
)

if existing_pythonpath:

    child_env["PYTHONPATH"] = (
        str(ROOT)
        + os.pathsep
        + existing_pythonpath
    )

else:

    child_env["PYTHONPATH"] = str(ROOT)

assert (
    child_env["PYTHONPATH"]
    .split(os.pathsep)[0]
    == str(ROOT)
)

print("✓ project root will be prepended to child PYTHONPATH")
print("✓ same execution-environment correction used for successful E1")


# =============================================================================
# 11. Verify runner itself remains frozen
# =============================================================================

runner_text = RUNNER.read_text(
    encoding="utf-8"
)

assert "call_json_structured(" in runner_text
assert '"cir_constrained"' in runner_text
assert '"direct"' not in runner_text
assert '"proposed"' not in runner_text

print("✓ frozen treatment runner verified")
print("✓ runner contains cir_constrained only")
print("✓ historical arms cannot execute")


# =============================================================================
# 12. Locked execution plan
# =============================================================================

print("\n" + "-" * 118)
print("LOCKED E2A EXECUTION PLAN")
print("-" * 118)

print("Benchmark          : E2A")
print("Cases              : 200")
print("Model              :", MODEL)
print("Treatment          : cir_constrained")
print("Dataset            :", E2A_DATA)
print("Runner             :", RUNNER)
print("Future output      :", E2A_RESULT)
print("Future log         :", E2A_LOG)
print("Environment change : PYTHONPATH only")
print("Expected model calls: 200")


# =============================================================================
# 13. Final integrity
# =============================================================================

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

print("\n✓ all frozen evidence remains byte-for-byte unchanged")


print("\n" + "=" * 118)
print("✓ STEP BR9 COMPLETE")
print("✓ E1 COMPLETE AND FROZEN")
print("✓ E2A FROZEN AT 200 UNIQUE CASES")
print("✓ HISTORICAL E2A CONTROLS VERIFIED")
print("✓ TREATMENT/MODEL/SCHEMA LOCKED")
print("✓ CORRECTED PYTHONPATH LOCKED")
print("✓ NO PRIOR E2A TREATMENT OUTPUT EXISTS")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO E2A CASES EXECUTED")
print("✓ NEXT STEP: ONE-TIME 200-CASE E2A EXECUTION")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR10")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR9: FINAL ZERO-CALL E2A EXECUTION GATE

----------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/capability_probe/experiment_B_review_BR5_capability_probe_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/results/E1/experiment_B_review_E1_cir_constrained_50_v1.csv
✓ jo

In [50]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import shutil
import pandas as pd
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR10: ONE-TIME E2A cir_constrained EXECUTION — 200 CASES")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

# =============================================================================
# Frozen components
# =============================================================================

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR7R1_E1_execution_record_v1.json"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_E2A_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_E2A_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)


# =============================================================================
# New BR10 evidence
# =============================================================================

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

E2A_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_v1.log"
)

E2A_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_E2A_execution_record_v1.json"
)


# =============================================================================
# Persistent evidence
# =============================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_RESULT = (
    DRIVE_ROOT
    / "results"
    / "E2A"
    / E2A_RESULT.name
)

DRIVE_LOG = (
    DRIVE_ROOT
    / "logs"
    / E2A_LOG.name
)

DRIVE_EXEC = (
    DRIVE_ROOT
    / "manifests"
    / E2A_EXEC.name
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# =============================================================================
# 1. Frozen hashes
# =============================================================================

EXPECTED = {

    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    BR5_PROBE:
        "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208",

    E1_RESULT:
        "21ef335ea6218edb01b30dccc1461520767a7d1e9ea2e6956e7ba267dc29ed68",

    E1_EXEC:
        "4aaf738112b9d6b448bf7dcc068ce57735724f5f316952fe5eb0cde78803526e",

    E2A_DATA:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_E2A_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_E2A_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",
}


# =============================================================================
# 2. Verify complete provenance BEFORE calls
# =============================================================================

print("\n" + "-" * 118)
print("PRE-EXECUTION FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# =============================================================================
# 3. Verify E1 is already closed
# =============================================================================

e1_record = json.loads(
    E1_EXEC.read_text(
        encoding="utf-8"
    )
)

assert (
    e1_record["status"]
    == "COMPLETED_AND_FROZEN_DO_NOT_RERUN"
)

assert e1_record["benchmark_cases"] == 50
assert e1_record["process_return_code"] == 0

print("\n✓ E1 remains completed and frozen")


# =============================================================================
# 4. Verify BR5 capability
# =============================================================================

probe = json.loads(
    BR5_PROBE.read_text(
        encoding="utf-8"
    )
)

assert (
    probe["capability_status"]
    == "SUPPORTED_AND_PROBE_VALID"
)

assert probe["parse_ok"] is True
assert probe["structural_schema_valid"] is True

print("✓ BR5 capability gate satisfied")


# =============================================================================
# 5. Verify model/treatment
# =============================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg["semantic_model"]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"
assert prereg["treatment"]["id"] == "cir_constrained"

print("✓ model locked:", MODEL)
print("✓ treatment locked: cir_constrained")


# =============================================================================
# 6. Verify E2A benchmark identity
# =============================================================================

e2a_input = pd.read_csv(
    E2A_DATA
)

assert len(e2a_input) == 200
assert e2a_input["id"].nunique() == 200

assert (
    e2a_input["expected_valid"].sum()
    == 100
)

print("✓ E2A = 200 unique cases")
print("✓ E2A = 100 expected-valid / 100 expected-invalid")


# =============================================================================
# 7. Verify historical control identity
# =============================================================================

historical = pd.read_csv(
    HIST_E2A_RESULTS
)

assert len(historical) == 600

counts = historical[
    "pipeline"
].value_counts().to_dict()

assert counts.get("direct") == 200
assert counts.get("cir") == 200
assert counts.get("proposed") == 200

historical_cir = historical[
    historical["pipeline"] == "cir"
]

assert (
    set(historical_cir["id"].astype(str))
    == set(e2a_input["id"].astype(str))
)

print("✓ historical Direct/CIR/Proposed controls preserved")
print("✓ historical CIR IDs exactly match E2A")


# =============================================================================
# 8. CRITICAL one-time execution gate
# =============================================================================

print("\n" + "-" * 118)
print("ONE-TIME BR10 EXECUTION GATE")
print("-" * 118)

for label, path in [
    ("E2A result", E2A_RESULT),
    ("E2A log", E2A_LOG),
    ("E2A execution record", E2A_EXEC),

    ("persistent E2A result", DRIVE_RESULT),
    ("persistent E2A log", DRIVE_LOG),
    ("persistent E2A execution record", DRIVE_EXEC),
]:

    assert not path.exists(), (
        f"STOP: {label} already exists:\n{path}\n\n"
        "DO NOT RERUN E2A."
    )

    print(f"✓ no prior {label}")


# =============================================================================
# 9. API key
# =============================================================================

assert os.environ.get(
    "TOGETHER_API_KEY"
), (
    "STOP: TOGETHER_API_KEY unavailable."
)

print("✓ TOGETHER_API_KEY available")


# =============================================================================
# 10. Corrected subprocess environment
#
# Same environment-only correction that produced successful E1.
# =============================================================================

child_env = os.environ.copy()

existing_pythonpath = child_env.get(
    "PYTHONPATH",
    ""
)

if existing_pythonpath:

    child_env["PYTHONPATH"] = (
        str(ROOT)
        + os.pathsep
        + existing_pythonpath
    )

else:

    child_env["PYTHONPATH"] = str(ROOT)

assert (
    child_env["PYTHONPATH"]
    .split(os.pathsep)[0]
    == str(ROOT)
)

print("✓ repository root prepended to child PYTHONPATH")
print("✓ scientific runner/adapter remain unchanged")


# =============================================================================
# 11. Create evidence directories
# =============================================================================

E2A_RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

E2A_LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

E2A_EXEC.parent.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 12. Lock command
# =============================================================================

command = [
    sys.executable,
    str(RUNNER),

    "--dataset",
    str(E2A_DATA),

    "--out",
    str(E2A_RESULT),

    "--model",
    MODEL,
]


print("\n" + "-" * 118)
print("LOCKED FINAL BENCHMARK EXECUTION")
print("-" * 118)

print("Benchmark          : E2A")
print("Cases              : 200")
print("Model              :", MODEL)
print("Treatment          : cir_constrained")
print("Runner SHA-256     :", EXPECTED[RUNNER])
print("Adapter SHA-256    :", EXPECTED[ADAPTER])
print("Dataset SHA-256    :", EXPECTED[E2A_DATA])
print("Output             :", E2A_RESULT)
print("Log                :", E2A_LOG)
print("Environment change : PYTHONPATH only")

print()
print("*** FINAL 200 BENCHMARK MODEL CALLS WILL NOW EXECUTE ***")
print("*** DO NOT INTERRUPT THE CELL IF POSSIBLE ***")
print()


# =============================================================================
# 13. Execute E2A exactly once
# =============================================================================

started_utc = datetime.now(
    timezone.utc
).isoformat()

process = subprocess.Popen(
    command,

    cwd=str(ROOT),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,

    env=child_env,
)

log_lines = []

assert process.stdout is not None

for line in process.stdout:

    print(
        line,
        end=""
    )

    log_lines.append(
        line
    )

return_code = process.wait()

finished_utc = datetime.now(
    timezone.utc
).isoformat()


# =============================================================================
# 14. Freeze execution log immediately
# =============================================================================

E2A_LOG.write_text(
    "".join(log_lines),
    encoding="utf-8"
)

print("\nProcess return code:", return_code)


# =============================================================================
# 15. Stop on ANY execution failure
# =============================================================================

assert return_code == 0, (
    "STOP: BR10 returned non-zero status.\n"
    "DO NOT RERUN E2A.\n"
    "The execution log has been preserved. Send the complete output."
)

assert E2A_RESULT.exists(), (
    "STOP: process returned success but E2A result is absent.\n"
    "DO NOT RERUN."
)


# =============================================================================
# 16. Minimal evidence validation
# =============================================================================

result = pd.read_csv(
    E2A_RESULT
)

assert len(result) == 200, (
    f"STOP: expected 200 rows, observed {len(result)}."
)

assert result["id"].nunique() == 200

assert (
    set(result["id"].astype(str))
    == set(e2a_input["id"].astype(str))
)

assert (
    set(result["pipeline"].astype(str))
    == {"cir_constrained"}
)

print("\n✓ exactly 200 E2A result rows")
print("✓ exactly 200 unique E2A IDs")
print("✓ result IDs exactly match frozen E2A benchmark")
print("✓ all observations labeled cir_constrained")


# =============================================================================
# 17. Freeze observed hashes
# =============================================================================

result_sha = sha256(
    E2A_RESULT
)

log_sha = sha256(
    E2A_LOG
)

print("\n" + "-" * 118)
print("OBSERVED BR10 EVIDENCE")
print("-" * 118)

print(
    "Result SHA-256:",
    result_sha
)

print(
    "Log SHA-256   :",
    log_sha
)


# =============================================================================
# 18. Create execution record
# =============================================================================

record = {
    "experiment":
        "B-Review",

    "step":
        "BR10",

    "benchmark":
        "E2A",

    "status":
        "COMPLETED_AND_FROZEN_DO_NOT_RERUN",

    "model":
        MODEL,

    "treatment":
        "cir_constrained",

    "benchmark_cases":
        200,

    "expected_provider_calls":
        200,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "process_return_code":
        return_code,

    "execution_environment": {
        "PYTHONPATH_repository_root_prepended":
            True,

        "runner_modified":
            False,

        "adapter_modified":
            False,

        "scientific_code_modified":
            False,
    },

    "preregistration_sha256":
        EXPECTED[PREREG],

    "BR5_capability_probe_sha256":
        EXPECTED[BR5_PROBE],

    "runner_sha256":
        EXPECTED[RUNNER],

    "adapter_sha256":
        EXPECTED[ADAPTER],

    "dataset_sha256":
        EXPECTED[E2A_DATA],

    "historical_E2A_results_sha256":
        EXPECTED[HIST_E2A_RESULTS],

    "historical_E2A_metrics_sha256":
        EXPECTED[HIST_E2A_METRICS],

    "result": {
        "path":
            str(
                E2A_RESULT.relative_to(
                    ROOT
                )
            ),

        "sha256":
            result_sha,

        "rows":
            200,
    },

    "log": {
        "path":
            str(
                E2A_LOG.relative_to(
                    ROOT
                )
            ),

        "sha256":
            log_sha,
    },
}


E2A_EXEC.write_text(
    json.dumps(
        record,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

exec_sha = sha256(
    E2A_EXEC
)

print(
    "Execution-record SHA-256:",
    exec_sha
)


# =============================================================================
# 19. Persist observed evidence immediately
# =============================================================================

for destination in [
    DRIVE_RESULT,
    DRIVE_LOG,
    DRIVE_EXEC,
]:

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


shutil.copy2(
    E2A_RESULT,
    DRIVE_RESULT
)

shutil.copy2(
    E2A_LOG,
    DRIVE_LOG
)

shutil.copy2(
    E2A_EXEC,
    DRIVE_EXEC
)

assert (
    sha256(DRIVE_RESULT)
    == result_sha
)

assert (
    sha256(DRIVE_LOG)
    == log_sha
)

assert (
    sha256(DRIVE_EXEC)
    == exec_sha
)

print("✓ E2A evidence persisted to Drive byte-for-byte")


# =============================================================================
# 20. Verify frozen scientific provenance after execution
# =============================================================================

print("\n" + "-" * 118)
print("POST-EXECUTION FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# =============================================================================
# 21. Reverify observed evidence
# =============================================================================

assert (
    sha256(E2A_RESULT)
    == result_sha
)

assert (
    sha256(E2A_LOG)
    == log_sha
)

assert (
    sha256(E2A_EXEC)
    == exec_sha
)

print("✓ E2A raw result frozen")
print("✓ E2A execution log frozen")
print("✓ E2A execution record frozen")


print("\n" + "=" * 118)
print("✓ STEP BR10 COMPLETE")
print("✓ E2A cir_constrained EXECUTED EXACTLY ONCE")
print("✓ 200/200 E2A OBSERVATIONS RECORDED")
print("✓ RAW E2A RESULT FROZEN")
print("✓ E2A LOG FROZEN")
print("✓ E2A EXECUTION RECORD FROZEN")
print("✓ EVIDENCE PERSISTED TO DRIVE")
print("✓ HISTORICAL CONTROLS REMAIN UNCHANGED")
print("✓ DO NOT RERUN E2A")
print("✓ ALL PLANNED MODEL-RUNNING EXPERIMENTS ARE NOW COMPLETE")
print("✓ NEXT: READ-ONLY E2A ANALYSIS")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE CONTINUING")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR10: ONE-TIME E2A cir_constrained EXECUTION — 200 CASES

----------------------------------------------------------------------------------------------------------------------
PRE-EXECUTION FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/capability_probe/experiment_B_review_BR5_capability_probe_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/results/E1/experiment_B_review_E1

AssertionError: STOP: BR10 returned non-zero status.
DO NOT RERUN E2A.
The execution log has been preserved. Send the complete output.

In [51]:
from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR10-D1: READ-ONLY DIAGNOSIS OF INTERRUPTED E2A EXECUTION")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

E2A_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_v1.log"
)

E2A_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_E2A_execution_record_v1.json"
)


def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


# =============================================================================
# 1. Verify frozen inputs
# =============================================================================

EXPECTED_RUNNER = (
    "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851"
)

EXPECTED_ADAPTER = (
    "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad"
)

EXPECTED_DATA = (
    "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753"
)

print("\n" + "-" * 118)
print("FROZEN INPUTS")
print("-" * 118)

assert sha256(RUNNER) == EXPECTED_RUNNER
assert sha256(ADAPTER) == EXPECTED_ADAPTER
assert sha256(E2A_DATA) == EXPECTED_DATA

print("✓ frozen runner unchanged")
print("✓ frozen adapter unchanged")
print("✓ frozen E2A dataset unchanged")


# =============================================================================
# 2. Inspect preserved BR10 log
# =============================================================================

print("\n" + "-" * 118)
print("INTERRUPTED BR10 LOG")
print("-" * 118)

assert E2A_LOG.exists(), (
    "STOP: expected BR10 execution log does not exist."
)

log_text = E2A_LOG.read_text(
    encoding="utf-8",
    errors="replace"
)

log_sha = sha256(E2A_LOG)

print("Log SHA-256:", log_sha)
print("Log bytes   :", E2A_LOG.stat().st_size)

assert "InternalServerError" in log_text
assert "503" in log_text
assert "Service unavailable" in log_text

print("✓ provider-side HTTP 503 recorded")
print("✓ failure classified as external service-unavailable interruption")


# =============================================================================
# 3. Recover progress markers from log
# =============================================================================

progress = []

for line in log_text.splitlines():

    s = line.strip()

    if (
        s.startswith("[")
        and "/200]" in s
    ):

        try:

            prefix, case_id = s.split("] ", 1)

            ordinal = int(
                prefix[1:].split("/")[0]
            )

            progress.append(
                (
                    ordinal,
                    case_id.strip()
                )
            )

        except Exception:

            pass


print("\n" + "-" * 118)
print("OBSERVED EXECUTION PROGRESS")
print("-" * 118)

print("Progress markers:", len(progress))

if progress:

    print(
        "First marker:",
        progress[0]
    )

    print(
        "Last marker :",
        progress[-1]
    )

    ordinals = [
        x[0]
        for x in progress
    ]

    assert ordinals == list(
        range(
            1,
            len(progress) + 1
        )
    )

    print("✓ progress markers form a contiguous prefix")


# =============================================================================
# 4. Compare markers with frozen dataset order
# =============================================================================

dataset = pd.read_csv(
    E2A_DATA
)

assert len(dataset) == 200

dataset_ids = (
    dataset["id"]
    .astype(str)
    .tolist()
)

if progress:

    progress_ids = [
        case_id
        for _, case_id in progress
    ]

    assert (
        progress_ids
        == dataset_ids[
            :len(progress_ids)
        ]
    )

    print("✓ observed progress IDs exactly match frozen dataset prefix")


# =============================================================================
# 5. Determine failed case boundary
# =============================================================================

completed_marker_count = len(progress)

next_index = completed_marker_count

if next_index < len(dataset_ids):

    next_case = dataset_ids[
        next_index
    ]

else:

    next_case = None


print()
print(
    "Last case with completion marker:",
    progress[-1][1] if progress else "<none>"
)

print(
    "Next frozen case after markers:",
    next_case
)


# =============================================================================
# 6. Inspect whether runner wrote a partial result
# =============================================================================

print("\n" + "-" * 118)
print("PARTIAL RESULT ARTIFACT")
print("-" * 118)

if not E2A_RESULT.exists():

    print("Partial result file exists: NO")

else:

    print("Partial result file exists: YES")

    result_sha = sha256(
        E2A_RESULT
    )

    print(
        "Partial result SHA-256:",
        result_sha
    )

    print(
        "Partial result bytes:",
        E2A_RESULT.stat().st_size
    )

    try:

        partial = pd.read_csv(
            E2A_RESULT
        )

        print(
            "Readable CSV rows:",
            len(partial)
        )

        print(
            "Columns:",
            list(partial.columns)
        )

        if "id" in partial.columns:

            print(
                "Unique IDs:",
                partial["id"].nunique()
            )

            partial_ids = (
                partial["id"]
                .astype(str)
                .tolist()
            )

            print(
                "First ID:",
                partial_ids[0]
                if partial_ids
                else "<none>"
            )

            print(
                "Last ID :",
                partial_ids[-1]
                if partial_ids
                else "<none>"
            )

            if partial_ids:

                assert (
                    partial_ids
                    == dataset_ids[
                        :len(partial_ids)
                    ]
                )

                print(
                    "✓ partial result IDs form exact frozen-dataset prefix"
                )

    except Exception as exc:

        print(
            "Partial result could not be parsed as CSV:"
        )

        print(
            repr(exc)
        )


# =============================================================================
# 7. Determine whether normal BR10 execution record exists
#
# It should normally NOT exist because BR10 stopped before that section.
# =============================================================================

print("\n" + "-" * 118)
print("EXECUTION RECORD STATUS")
print("-" * 118)

if E2A_EXEC.exists():

    print(
        "BR10 execution record exists: YES"
    )

    print(
        "SHA-256:",
        sha256(E2A_EXEC)
    )

    print(
        E2A_EXEC.read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

else:

    print(
        "BR10 execution record exists: NO"
    )

    print(
        "✓ expected because BR10 stopped before successful freeze section"
    )


# =============================================================================
# 8. Inspect runner write semantics
#
# We need to know whether results are accumulated in memory and written only
# at the end, or incrementally persisted.
# =============================================================================

print("\n" + "-" * 118)
print("FROZEN RUNNER WRITE-SEMANTICS EXCERPT")
print("-" * 118)

runner_lines = RUNNER.read_text(
    encoding="utf-8"
).splitlines()

interesting_terms = [
    "rows =",
    "rows.append",
    "DataFrame",
    "to_csv",
    "csv.",
    "open(",
]

for i, line in enumerate(
    runner_lines,
    start=1
):

    if any(
        term in line
        for term in interesting_terms
    ):

        lo = max(
            1,
            i - 3
        )

        hi = min(
            len(runner_lines),
            i + 5
        )

        print(
            f"\n--- lines {lo}-{hi} ---"
        )

        for j in range(
            lo,
            hi + 1
        ):

            print(
                f"{j:4d}: "
                f"{runner_lines[j-1]}"
            )


# =============================================================================
# 9. Critical diagnostic summary
# =============================================================================

print("\n" + "-" * 118)
print("DIAGNOSTIC SUMMARY")
print("-" * 118)

print(
    "Provider failure        : HTTP 503 Service unavailable"
)

print(
    "Progress markers        :",
    completed_marker_count
)

print(
    "Last marked case        :",
    progress[-1][1] if progress else None
)

print(
    "Next frozen case        :",
    next_case
)

print(
    "Partial result exists   :",
    E2A_RESULT.exists()
)

print(
    "Successful exec record  :",
    E2A_EXEC.exists()
)


# =============================================================================
# 10. Final integrity
# =============================================================================

assert sha256(RUNNER) == EXPECTED_RUNNER
assert sha256(ADAPTER) == EXPECTED_ADAPTER
assert sha256(E2A_DATA) == EXPECTED_DATA
assert sha256(E2A_LOG) == log_sha

print("\n✓ frozen runner unchanged")
print("✓ frozen adapter unchanged")
print("✓ frozen dataset unchanged")
print("✓ interrupted execution log unchanged")


print("\n" + "=" * 118)
print("✓ STEP BR10-D1 COMPLETE")
print("✓ INTERRUPTED BR10 DIAGNOSED READ-ONLY")
print("✓ PROVIDER 503 EVIDENCE PRESERVED")
print("✓ EXECUTED PREFIX IDENTIFIED FROM LOG")
print("✓ PARTIAL-RESULT STATUS INSPECTED")
print("✓ RUNNER WRITE SEMANTICS DISPLAYED")
print("✓ ZERO API/MODEL CALLS")
print("✓ NO BENCHMARK CASE EXECUTED")
print("✓ DO NOT RERUN BR10")
print("✓ DO NOT MANUALLY RESUME YET")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE RECOVERY")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR10-D1: READ-ONLY DIAGNOSIS OF INTERRUPTED E2A EXECUTION

----------------------------------------------------------------------------------------------------------------------
FROZEN INPUTS
----------------------------------------------------------------------------------------------------------------------
✓ frozen runner unchanged
✓ frozen adapter unchanged
✓ frozen E2A dataset unchanged

----------------------------------------------------------------------------------------------------------------------
INTERRUPTED BR10 LOG
----------------------------------------------------------------------------------------------------------------------
Log SHA-256: 1aefd5022a637bb57775794c248e3d757a7086889d7c8caf0d383715464c85da
Log bytes   : 3791
✓ provider-side HTTP 503 recorded
✓ failure classified as external service-unavailable interruption

----------------------------------------------------------------------------------------------------------------------
O

In [52]:
from pathlib import Path
import hashlib
import json
import shutil
import ast
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR10-R0: ARCHIVE INTERRUPTION AND PREPARE ROBUST RECOVERY")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

FAILED_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_v1.log"
)

FAILED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_interrupted_execution_record_v1.json"
)

RECOVERY_RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_recovery_v1.py"
)

RECOVERY_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R0_recovery_manifest_v1.json"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_FAILED_LOG = (
    DRIVE_ROOT
    / "logs"
    / FAILED_LOG.name
)

DRIVE_FAILED_RECORD = (
    DRIVE_ROOT
    / "manifests"
    / FAILED_RECORD.name
)

DRIVE_RECOVERY_RUNNER = (
    DRIVE_ROOT
    / "implementation"
    / RECOVERY_RUNNER.name
)

DRIVE_RECOVERY_MANIFEST = (
    DRIVE_ROOT
    / "manifests"
    / RECOVERY_MANIFEST.name
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


EXPECTED_RUNNER = (
    "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851"
)

EXPECTED_ADAPTER = (
    "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad"
)

EXPECTED_DATA = (
    "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753"
)

EXPECTED_FAILED_LOG = (
    "1aefd5022a637bb57775794c248e3d757a7086889d7c8caf0d383715464c85da"
)


# =============================================================================
# 1. Verify interrupted evidence and frozen scientific components
# =============================================================================

print("\n" + "-" * 118)
print("INTERRUPTED ATTEMPT PROVENANCE")
print("-" * 118)

assert sha256(RUNNER) == EXPECTED_RUNNER
assert sha256(ADAPTER) == EXPECTED_ADAPTER
assert sha256(E2A_DATA) == EXPECTED_DATA
assert sha256(FAILED_LOG) == EXPECTED_FAILED_LOG

print("✓ frozen original runner verified")
print("✓ frozen treatment adapter verified")
print("✓ frozen E2A benchmark verified")
print("✓ interrupted BR10 log verified")


# =============================================================================
# 2. Confirm interruption evidence
# =============================================================================

log_text = FAILED_LOG.read_text(
    encoding="utf-8",
    errors="replace"
)

assert "InternalServerError" in log_text
assert "503" in log_text
assert "Service unavailable" in log_text

markers = []

for line in log_text.splitlines():
    s = line.strip()

    if s.startswith("[") and "/200]" in s:
        prefix, case_id = s.split("] ", 1)
        ordinal = int(prefix[1:].split("/")[0])
        markers.append((ordinal, case_id.strip()))

assert len(markers) == 79
assert markers[0] == (1, "EXP2-001")
assert markers[-1] == (79, "EXP2-079")

print("✓ 79 completed progress markers verified")
print("✓ failure boundary = call associated with EXP2-080")
print("✓ provider failure = HTTP 503 Service unavailable")


# =============================================================================
# 3. Confirm no partial scientific result survived
# =============================================================================

ORIGINAL_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

assert not ORIGINAL_RESULT.exists(), (
    "STOP: a result artifact now exists unexpectedly."
)

print("✓ no partial E2A result artifact survived")


# =============================================================================
# 4. Record interrupted attempt permanently
# =============================================================================

assert not FAILED_RECORD.exists(), (
    "STOP: interrupted-execution record already exists."
)

failed_record = {
    "experiment": "B-Review",
    "step": "BR10",
    "benchmark": "E2A",
    "status": "INTERRUPTED_PROVIDER_503_NO_RESULT_ARTIFACT",

    "recorded_utc":
        datetime.now(timezone.utc).isoformat(),

    "provider_failure": {
        "http_status": 503,
        "type": "service_unavailable",
        "failed_case_boundary": "EXP2-080",
    },

    "completed_progress_markers":
        79,

    "completed_marker_range":
        ["EXP2-001", "EXP2-079"],

    "partial_result_artifact":
        False,

    "scientific_observations_available_for_analysis":
        0,

    "important_interpretation":
        (
            "The first 79 provider calls completed sufficiently for the "
            "runner to emit progress markers, but their generated outputs "
            "were retained only in process memory. Because the frozen runner "
            "writes the result CSV only after the full loop, those outputs "
            "were lost when the provider returned HTTP 503 during EXP2-080."
        ),

    "recovery_policy":
        (
            "Do not reconstruct or fabricate the lost outputs. "
            "A complete replacement E2A treatment run is required. "
            "The interrupted attempt remains part of provenance and is "
            "excluded from scientific outcome analysis."
        ),

    "original_runner_sha256":
        EXPECTED_RUNNER,

    "adapter_sha256":
        EXPECTED_ADAPTER,

    "dataset_sha256":
        EXPECTED_DATA,

    "interrupted_log": {
        "path":
            str(FAILED_LOG.relative_to(ROOT)),

        "sha256":
            EXPECTED_FAILED_LOG,
    },
}

FAILED_RECORD.write_text(
    json.dumps(
        failed_record,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

failed_record_sha = sha256(
    FAILED_RECORD
)

print(
    "Interrupted-record SHA-256:",
    failed_record_sha
)


# =============================================================================
# 5. Create recovery runner
#
# Scientific treatment remains unchanged.
#
# Execution-only changes:
#   - transient provider retry
#   - checkpoint each completed case
#
# IMPORTANT:
# This runner is a recovery implementation and is NOT the original BR4 runner.
# =============================================================================

assert not RECOVERY_RUNNER.exists(), (
    "STOP: recovery runner already exists."
)

original = RUNNER.read_text(
    encoding="utf-8"
)

# Preserve original imports/logic as the basis.
# Add only standard-library imports required for execution robustness.
original = original.replace(
    "import argparse",
    "import argparse\nimport time\nimport random"
)

# Replace the single provider invocation with bounded transient retry.
old_call = '''        llm = call_json_structured(
            r.intent,
            mode="cir",
            model=model
        )'''

new_call = '''        # Execution-resilience wrapper only.
        # Scientific treatment arguments are unchanged.
        max_attempts = 5
        llm = None

        for attempt in range(1, max_attempts + 1):
            try:
                llm = call_json_structured(
                    r.intent,
                    mode="cir",
                    model=model
                )
                break

            except Exception as exc:
                message = str(exc).lower()

                transient = (
                    "503" in message
                    or "service unavailable" in message
                    or "timeout" in message
                    or "temporarily unavailable" in message
                    or "rate limit" in message
                    or "429" in message
                )

                if (not transient) or attempt == max_attempts:
                    raise

                wait_seconds = min(30, 2 ** attempt)

                print(
                    f"TRANSIENT_PROVIDER_RETRY "
                    f"case={r.id} "
                    f"attempt={attempt}/{max_attempts} "
                    f"wait={wait_seconds}s",
                    flush=True
                )

                time.sleep(wait_seconds)

        assert llm is not None'''

assert old_call in original, (
    "STOP: expected frozen provider-call block not found. "
    "Do not create recovery runner."
)

recovery_source = original.replace(
    old_call,
    new_call,
    1
)

# Syntax check before writing.
ast.parse(recovery_source)

RECOVERY_RUNNER.write_text(
    recovery_source,
    encoding="utf-8"
)

recovery_runner_sha = sha256(
    RECOVERY_RUNNER
)

print(
    "Recovery-runner SHA-256:",
    recovery_runner_sha
)


# =============================================================================
# 6. Static equivalence checks
# =============================================================================

print("\n" + "-" * 118)
print("RECOVERY RUNNER STATIC CHECK")
print("-" * 118)

recovery_text = RECOVERY_RUNNER.read_text(
    encoding="utf-8"
)

assert 'mode="cir"' in recovery_text
assert "call_json_structured(" in recovery_text
assert '"cir_constrained"' in recovery_text

# Ensure no scientific arms introduced.
assert '"direct"' not in recovery_text
assert '"proposed"' not in recovery_text

# Ensure original treatment adapter untouched.
assert sha256(ADAPTER) == EXPECTED_ADAPTER

# Ensure original runner untouched.
assert sha256(RUNNER) == EXPECTED_RUNNER

print("✓ original runner remains byte-for-byte unchanged")
print("✓ treatment adapter remains byte-for-byte unchanged")
print("✓ same call_json_structured treatment retained")
print("✓ same mode='cir' retained")
print("✓ same model supplied by CLI retained")
print("✓ no Direct arm introduced")
print("✓ no Proposed arm introduced")
print("✓ recovery addition limited to transient-call retry")


# =============================================================================
# 7. Recovery manifest
# =============================================================================

assert not RECOVERY_MANIFEST.exists(), (
    "STOP: recovery manifest already exists."
)

manifest = {
    "experiment": "B-Review",
    "step": "BR10-R0",
    "status": "RECOVERY_IMPLEMENTATION_FROZEN",

    "purpose":
        "Recover from provider HTTP 503 without changing scientific treatment.",

    "original_runner": {
        "path":
            str(RUNNER.relative_to(ROOT)),
        "sha256":
            EXPECTED_RUNNER,
    },

    "recovery_runner": {
        "path":
            str(RECOVERY_RUNNER.relative_to(ROOT)),
        "sha256":
            recovery_runner_sha,
    },

    "treatment_adapter_sha256":
        EXPECTED_ADAPTER,

    "dataset_sha256":
        EXPECTED_DATA,

    "interrupted_attempt_record_sha256":
        failed_record_sha,

    "allowed_execution_changes": [
        "bounded retry of transient provider failures",
        "retry wait between transient failures",
    ],

    "scientific_changes": {
        "prompt": False,
        "model": False,
        "temperature": False,
        "reasoning_setting": False,
        "structural_schema": False,
        "validation": False,
        "mapping": False,
        "descriptor_logic": False,
        "benchmark": False,
    },

    "retry_policy": {
        "maximum_attempts_per_case": 5,
        "transient_indicators": [
            "HTTP 503",
            "service unavailable",
            "timeout",
            "temporarily unavailable",
            "rate limit",
            "HTTP 429",
        ],
    },
}

RECOVERY_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

manifest_sha = sha256(
    RECOVERY_MANIFEST
)

print(
    "Recovery-manifest SHA-256:",
    manifest_sha
)


# =============================================================================
# 8. Persist provenance/recovery implementation
# =============================================================================

for destination in [
    DRIVE_FAILED_LOG,
    DRIVE_FAILED_RECORD,
    DRIVE_RECOVERY_RUNNER,
    DRIVE_RECOVERY_MANIFEST,
]:

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

shutil.copy2(
    FAILED_LOG,
    DRIVE_FAILED_LOG
)

shutil.copy2(
    FAILED_RECORD,
    DRIVE_FAILED_RECORD
)

shutil.copy2(
    RECOVERY_RUNNER,
    DRIVE_RECOVERY_RUNNER
)

shutil.copy2(
    RECOVERY_MANIFEST,
    DRIVE_RECOVERY_MANIFEST
)

assert (
    sha256(DRIVE_FAILED_LOG)
    == EXPECTED_FAILED_LOG
)

assert (
    sha256(DRIVE_FAILED_RECORD)
    == failed_record_sha
)

assert (
    sha256(DRIVE_RECOVERY_RUNNER)
    == recovery_runner_sha
)

assert (
    sha256(DRIVE_RECOVERY_MANIFEST)
    == manifest_sha
)

print("✓ recovery provenance persisted to Drive byte-for-byte")


# =============================================================================
# 9. Final integrity
# =============================================================================

assert sha256(RUNNER) == EXPECTED_RUNNER
assert sha256(ADAPTER) == EXPECTED_ADAPTER
assert sha256(E2A_DATA) == EXPECTED_DATA
assert sha256(FAILED_LOG) == EXPECTED_FAILED_LOG

print("\n✓ frozen original scientific artifacts remain unchanged")


print("\n" + "=" * 118)
print("✓ STEP BR10-R0 COMPLETE")
print("✓ INTERRUPTED BR10 ATTEMPT ARCHIVED")
print("✓ 79 LOST IN-MEMORY OUTPUTS NOT RECONSTRUCTED")
print("✓ PROVIDER 503 RETAINED IN PROVENANCE")
print("✓ RECOVERY RUNNER CREATED AND FROZEN")
print("✓ SCIENTIFIC TREATMENT UNCHANGED")
print("✓ ORIGINAL BR4 RUNNER UNCHANGED")
print("✓ TREATMENT ADAPTER UNCHANGED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK CASES EXECUTED")
print("✓ DO NOT RUN ORIGINAL BR10 AGAIN")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE RECOVERY EXECUTION")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR10-R0: ARCHIVE INTERRUPTION AND PREPARE ROBUST RECOVERY

----------------------------------------------------------------------------------------------------------------------
INTERRUPTED ATTEMPT PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ frozen original runner verified
✓ frozen treatment adapter verified
✓ frozen E2A benchmark verified
✓ interrupted BR10 log verified
✓ 79 completed progress markers verified
✓ failure boundary = call associated with EXP2-080
✓ provider failure = HTTP 503 Service unavailable
✓ no partial E2A result artifact survived
Interrupted-record SHA-256: 21514eb08dcf63d6742acbfb4e80bb0ec7416b332b4f6b5d681fe57912cf091d
Recovery-runner SHA-256: 8b1acb6862cf967c4dd7ad75802caac9c2219efcbe42d3f6431a41c2275d0b16

----------------------------------------------------------------------------------------------------------------------
RECOVERY RUNNER STATIC C

In [53]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import shutil
import pandas as pd
from datetime import datetime, timezone

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR10-R1: REPLACEMENT E2A EXECUTION WITH TRANSIENT RETRY — 200 CASES")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

# =============================================================================
# Frozen provenance
# =============================================================================

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

ORIGINAL_RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

RECOVERY_RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_recovery_v1.py"
)

RECOVERY_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R0_recovery_manifest_v1.json"
)

INTERRUPTED_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_v1.log"
)

INTERRUPTED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_interrupted_execution_record_v1.json"
)

E2A_DATA = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_E2A_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_E2A_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)


# =============================================================================
# Replacement-run evidence paths
# =============================================================================

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

E2A_LOG = (
    BR
    / "logs"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.log"
)

E2A_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R1_E2A_execution_record_v1.json"
)


# =============================================================================
# Persistent evidence
# =============================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

DRIVE_RESULT = (
    DRIVE_ROOT
    / "results"
    / "E2A"
    / E2A_RESULT.name
)

DRIVE_LOG = (
    DRIVE_ROOT
    / "logs"
    / E2A_LOG.name
)

DRIVE_EXEC = (
    DRIVE_ROOT
    / "manifests"
    / E2A_EXEC.name
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def verify(path, expected, label):

    assert path.exists(), (
        f"STOP: missing {label}:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: {label} hash mismatch\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(f"✓ {label}")


# =============================================================================
# 1. Locked hashes
# =============================================================================

EXPECTED = {

    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    ORIGINAL_RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    RECOVERY_RUNNER:
        "8b1acb6862cf967c4dd7ad75802caac9c2219efcbe42d3f6431a41c2275d0b16",

    RECOVERY_MANIFEST:
        "6923681077b6d47a3e7862989ef660dcdef2efdefb2829336de2fcef20ae2398",

    INTERRUPTED_LOG:
        "1aefd5022a637bb57775794c248e3d757a7086889d7c8caf0d383715464c85da",

    INTERRUPTED_RECORD:
        "21514eb08dcf63d6742acbfb4e80bb0ec7416b332b4f6b5d681fe57912cf091d",

    E2A_DATA:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_E2A_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_E2A_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",
}


# =============================================================================
# 2. Verify complete provenance
# =============================================================================

print("\n" + "-" * 118)
print("PRE-RECOVERY FROZEN PROVENANCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    verify(
        path,
        expected,
        str(path.relative_to(ROOT))
    )


# =============================================================================
# 3. Verify interrupted attempt classification
# =============================================================================

interrupted = json.loads(
    INTERRUPTED_RECORD.read_text(
        encoding="utf-8"
    )
)

assert (
    interrupted["status"]
    == "INTERRUPTED_PROVIDER_503_NO_RESULT_ARTIFACT"
)

assert (
    interrupted["completed_progress_markers"]
    == 79
)

assert (
    interrupted["scientific_observations_available_for_analysis"]
    == 0
)

assert (
    interrupted["partial_result_artifact"]
    is False
)

print("\n✓ interrupted BR10 attempt retained")
print("✓ 79 completed calls documented")
print("✓ zero interrupted-run observations available for analysis")
print("✓ no lost output reconstructed")


# =============================================================================
# 4. Verify recovery implementation policy
# =============================================================================

recovery_manifest = json.loads(
    RECOVERY_MANIFEST.read_text(
        encoding="utf-8"
    )
)

assert (
    recovery_manifest["status"]
    == "RECOVERY_IMPLEMENTATION_FROZEN"
)

assert (
    recovery_manifest["retry_policy"]["maximum_attempts_per_case"]
    == 5
)

scientific_changes = recovery_manifest[
    "scientific_changes"
]

assert not any(
    scientific_changes.values()
)

print("✓ recovery runner frozen")
print("✓ maximum transient attempts per case = 5")
print("✓ no preregistered scientific treatment component changed")


# =============================================================================
# 5. Verify model/treatment identity
# =============================================================================

prereg = json.loads(
    PREREG.read_text(
        encoding="utf-8"
    )
)

MODEL = prereg[
    "semantic_model"
]["model"]

assert MODEL == "Qwen/Qwen3.5-9B"

assert (
    prereg["treatment"]["id"]
    == "cir_constrained"
)

print("✓ model locked:", MODEL)
print("✓ treatment locked: cir_constrained")


# =============================================================================
# 6. Verify benchmark
# =============================================================================

dataset = pd.read_csv(
    E2A_DATA
)

assert len(dataset) == 200
assert dataset["id"].nunique() == 200

assert (
    dataset["expected_valid"].sum()
    == 100
)

print("✓ E2A benchmark = 200 unique cases")
print("✓ ground truth = 100 valid / 100 invalid")


# =============================================================================
# 7. Critical replacement-run output gate
# =============================================================================

print("\n" + "-" * 118)
print("ONE-TIME BR10-R1 OUTPUT GATE")
print("-" * 118)

for label, path in [

    ("replacement E2A result", E2A_RESULT),
    ("replacement E2A log", E2A_LOG),
    ("replacement E2A execution record", E2A_EXEC),

    ("persistent replacement result", DRIVE_RESULT),
    ("persistent replacement log", DRIVE_LOG),
    ("persistent replacement execution record", DRIVE_EXEC),
]:

    assert not path.exists(), (
        f"STOP: {label} already exists:\n{path}\n\n"
        "DO NOT EXECUTE BR10-R1."
    )

    print(f"✓ no prior {label}")


# =============================================================================
# 8. Ensure original failed result path still does not exist
# =============================================================================

ORIGINAL_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_v1.csv"
)

assert not ORIGINAL_RESULT.exists(), (
    "STOP: original BR10 result unexpectedly exists."
)

print("✓ original interrupted BR10 produced no result artifact")


# =============================================================================
# 9. API key
# =============================================================================

assert os.environ.get(
    "TOGETHER_API_KEY"
), (
    "STOP: TOGETHER_API_KEY unavailable."
)

print("✓ TOGETHER_API_KEY available")


# =============================================================================
# 10. Same PYTHONPATH correction used for successful E1
# =============================================================================

child_env = os.environ.copy()

existing_pythonpath = child_env.get(
    "PYTHONPATH",
    ""
)

if existing_pythonpath:

    child_env["PYTHONPATH"] = (
        str(ROOT)
        + os.pathsep
        + existing_pythonpath
    )

else:

    child_env["PYTHONPATH"] = str(ROOT)

assert (
    child_env["PYTHONPATH"]
    .split(os.pathsep)[0]
    == str(ROOT)
)

print("✓ repository root prepended to child PYTHONPATH")


# =============================================================================
# 11. Create evidence directories
# =============================================================================

E2A_RESULT.parent.mkdir(
    parents=True,
    exist_ok=True
)

E2A_LOG.parent.mkdir(
    parents=True,
    exist_ok=True
)

E2A_EXEC.parent.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 12. Locked replacement command
# =============================================================================

command = [
    sys.executable,
    str(RECOVERY_RUNNER),

    "--dataset",
    str(E2A_DATA),

    "--out",
    str(E2A_RESULT),

    "--model",
    MODEL,
]


print("\n" + "-" * 118)
print("LOCKED REPLACEMENT E2A EXECUTION")
print("-" * 118)

print("Benchmark            : E2A")
print("Cases                : 200")
print("Model                :", MODEL)
print("Treatment            : cir_constrained")
print("Recovery runner SHA  :", EXPECTED[RECOVERY_RUNNER])
print("Treatment adapter SHA:", EXPECTED[ADAPTER])
print("Dataset SHA          :", EXPECTED[E2A_DATA])
print("Output               :", E2A_RESULT)
print("Log                  :", E2A_LOG)
print("Transient retry      : maximum 5 attempts/case")
print("Scientific changes   : NONE")

print()
print("*** REPLACEMENT 200-CASE E2A RUN WILL NOW EXECUTE ***")
print("*** DO NOT INTERRUPT IF POSSIBLE ***")
print()


# =============================================================================
# 13. Execute replacement run exactly once
# =============================================================================

started_utc = datetime.now(
    timezone.utc
).isoformat()

process = subprocess.Popen(
    command,

    cwd=str(ROOT),

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,

    env=child_env,
)

log_lines = []

assert process.stdout is not None

for line in process.stdout:

    print(
        line,
        end=""
    )

    log_lines.append(
        line
    )

return_code = process.wait()

finished_utc = datetime.now(
    timezone.utc
).isoformat()


# =============================================================================
# 14. Freeze replacement log immediately
# =============================================================================

E2A_LOG.write_text(
    "".join(log_lines),
    encoding="utf-8"
)

print("\nProcess return code:", return_code)


# =============================================================================
# 15. Stop on any failure
# =============================================================================

assert return_code == 0, (
    "STOP: BR10-R1 returned non-zero status.\n"
    "DO NOT RERUN.\n"
    "The replacement log is preserved. Send the complete output."
)

assert E2A_RESULT.exists(), (
    "STOP: replacement process returned success but result is absent.\n"
    "DO NOT RERUN."
)


# =============================================================================
# 16. Minimal raw-evidence validation
# =============================================================================

result = pd.read_csv(
    E2A_RESULT
)

assert len(result) == 200

assert (
    result["id"].nunique()
    == 200
)

assert (
    set(result["id"].astype(str))
    == set(dataset["id"].astype(str))
)

assert (
    result["id"].astype(str).tolist()
    == dataset["id"].astype(str).tolist()
)

assert (
    set(result["pipeline"].astype(str))
    == {"cir_constrained"}
)

print("\n✓ exactly 200 result rows")
print("✓ exactly 200 unique E2A IDs")
print("✓ result IDs match frozen benchmark")
print("✓ result order matches frozen benchmark")
print("✓ all rows labeled cir_constrained")


# =============================================================================
# 17. Count retry events from frozen execution log
# =============================================================================

retry_lines = [
    line
    for line in log_lines
    if "TRANSIENT_PROVIDER_RETRY" in line
]

print(
    "Transient retry events:",
    len(retry_lines)
)

if retry_lines:

    for line in retry_lines:
        print(
            "  ",
            line.strip()
        )


# =============================================================================
# 18. Freeze hashes
# =============================================================================

result_sha = sha256(
    E2A_RESULT
)

log_sha = sha256(
    E2A_LOG
)

print("\n" + "-" * 118)
print("OBSERVED BR10-R1 EVIDENCE")
print("-" * 118)

print(
    "Result SHA-256:",
    result_sha
)

print(
    "Log SHA-256   :",
    log_sha
)


# =============================================================================
# 19. Execution record
# =============================================================================

record = {
    "experiment":
        "B-Review",

    "step":
        "BR10-R1",

    "benchmark":
        "E2A",

    "status":
        "COMPLETED_AND_FROZEN_DO_NOT_RERUN",

    "role":
        "complete replacement for interrupted BR10 run",

    "model":
        MODEL,

    "treatment":
        "cir_constrained",

    "benchmark_cases":
        200,

    "result_rows":
        200,

    "started_utc":
        started_utc,

    "finished_utc":
        finished_utc,

    "process_return_code":
        return_code,

    "transient_retry_events":
        len(retry_lines),

    "scientific_observations_from_interrupted_BR10_used":
        0,

    "interrupted_BR10_record_sha256":
        EXPECTED[INTERRUPTED_RECORD],

    "recovery_manifest_sha256":
        EXPECTED[RECOVERY_MANIFEST],

    "original_runner_sha256":
        EXPECTED[ORIGINAL_RUNNER],

    "recovery_runner_sha256":
        EXPECTED[RECOVERY_RUNNER],

    "adapter_sha256":
        EXPECTED[ADAPTER],

    "dataset_sha256":
        EXPECTED[E2A_DATA],

    "scientific_treatment_changed":
        False,

    "execution_resilience_change":
        "bounded transient-provider retry only",

    "result": {
        "path":
            str(E2A_RESULT.relative_to(ROOT)),
        "sha256":
            result_sha,
    },

    "log": {
        "path":
            str(E2A_LOG.relative_to(ROOT)),
        "sha256":
            log_sha,
    },
}

E2A_EXEC.write_text(
    json.dumps(
        record,
        indent=2,
        sort_keys=True
    ) + "\n",
    encoding="utf-8"
)

exec_sha = sha256(
    E2A_EXEC
)

print(
    "Execution-record SHA-256:",
    exec_sha
)


# =============================================================================
# 20. Persist evidence
# =============================================================================

for destination in [
    DRIVE_RESULT,
    DRIVE_LOG,
    DRIVE_EXEC,
]:

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

shutil.copy2(
    E2A_RESULT,
    DRIVE_RESULT
)

shutil.copy2(
    E2A_LOG,
    DRIVE_LOG
)

shutil.copy2(
    E2A_EXEC,
    DRIVE_EXEC
)

assert (
    sha256(DRIVE_RESULT)
    == result_sha
)

assert (
    sha256(DRIVE_LOG)
    == log_sha
)

assert (
    sha256(DRIVE_EXEC)
    == exec_sha
)

print("✓ replacement E2A evidence persisted to Drive byte-for-byte")


# =============================================================================
# 21. Final integrity
# =============================================================================

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

assert sha256(E2A_RESULT) == result_sha
assert sha256(E2A_LOG) == log_sha
assert sha256(E2A_EXEC) == exec_sha

print("\n✓ all prior frozen provenance unchanged")
print("✓ replacement result frozen")
print("✓ replacement log frozen")
print("✓ replacement execution record frozen")


print("\n" + "=" * 118)
print("✓ STEP BR10-R1 COMPLETE")
print("✓ REPLACEMENT E2A RUN COMPLETED SUCCESSFULLY")
print("✓ 200/200 E2A OBSERVATIONS RECORDED")
print("✓ INTERRUPTED BR10 OBSERVATIONS NOT USED")
print("✓ SCIENTIFIC TREATMENT UNCHANGED")
print("✓ RESULT / LOG / EXECUTION RECORD FROZEN")
print("✓ EVIDENCE PERSISTED TO DRIVE")
print("✓ DO NOT RERUN BR10-R1")
print("✓ ALL PLANNED MODEL-RUNNING EXPERIMENTS ARE COMPLETE")
print("✓ NEXT: READ-ONLY E2A STATISTICAL ANALYSIS")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE CONTINUING")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR10-R1: REPLACEMENT E2A EXECUTION WITH TRANSIENT RETRY — 200 CASES

----------------------------------------------------------------------------------------------------------------------
PRE-RECOVERY FROZEN PROVENANCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_recovery_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experimen

In [54]:
from pathlib import Path
import hashlib
import json
import math
import pandas as pd
from collections import Counter

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR11: READ-ONLY E2A STATISTICAL AND PAIRED ANALYSIS")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

TREATMENT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R1_E2A_execution_record_v1.json"
)

DATASET = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)

INTERRUPTED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_interrupted_execution_record_v1.json"
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def cp_interval(k, n, alpha=0.05):
    """
    Exact Clopper-Pearson interval.
    Uses scipy only for read-only statistical analysis.
    """
    from scipy.stats import beta

    if n == 0:
        return (float("nan"), float("nan"))

    lo = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)

    return float(lo), float(hi)


def fmt_ci(k, n):
    lo, hi = cp_interval(k, n)
    return f"{k}/{n} = {k/n:.4f} ({100*k/n:.2f}%), 95% CI [{lo:.4f}, {hi:.4f}]"


def boolify(v):
    if isinstance(v, bool):
        return v

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(f"Cannot convert to bool: {v!r}")


# =============================================================================
# 1. Verify frozen evidence
# =============================================================================

EXPECTED = {
    TREATMENT:
        "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6",

    EXEC_RECORD:
        "418b45e0ad487fed94442f7da2b8cd1987006a6727cb3b757218923c5c8322e5",

    DATASET:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",

    INTERRUPTED_RECORD:
        "21514eb08dcf63d6742acbfb4e80bb0ec7416b332b4f6b5d681fe57912cf091d",
}

print("\n" + "-" * 118)
print("FROZEN EVIDENCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert path.exists(), f"Missing: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"Hash mismatch:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# 2. Verify execution provenance
# =============================================================================

record = json.loads(
    EXEC_RECORD.read_text(
        encoding="utf-8"
    )
)

assert record["status"] == "COMPLETED_AND_FROZEN_DO_NOT_RERUN"
assert record["benchmark_cases"] == 200
assert record["result_rows"] == 200
assert record["process_return_code"] == 0
assert record["transient_retry_events"] == 0
assert record["scientific_treatment_changed"] is False

print("\n✓ successful replacement execution verified")
print("✓ transient retry events in successful run = 0")
print("✓ interrupted BR10 observations excluded")


# =============================================================================
# 3. Load and join treatment with ground truth
# =============================================================================

gt = pd.read_csv(DATASET)
tr = pd.read_csv(TREATMENT)

assert len(gt) == 200
assert len(tr) == 200

assert gt["id"].nunique() == 200
assert tr["id"].nunique() == 200

gt["id"] = gt["id"].astype(str)
tr["id"] = tr["id"].astype(str)

assert set(gt["id"]) == set(tr["id"])

joined = gt.merge(
    tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 200

print("\n✓ treatment joined one-to-one with frozen E2A ground truth")


# =============================================================================
# 4. Resolve expected-valid and accepted columns
# =============================================================================

joined["expected_valid_bool"] = joined[
    "expected_valid"
].map(boolify)

assert joined["expected_valid_bool"].sum() == 100


accepted_candidates = [
    "accepted",
    "is_accepted",
    "validation_accepted",
]

accepted_col = next(
    (
        c
        for c in accepted_candidates
        if c in joined.columns
    ),
    None
)

assert accepted_col is not None, (
    "STOP: could not identify treatment acceptance column.\n"
    f"Columns: {list(joined.columns)}"
)

joined["accepted_bool"] = joined[
    accepted_col
].map(boolify)

print("✓ treatment acceptance column:", accepted_col)


# =============================================================================
# 5. Parse success
# =============================================================================

parse_candidates = [
    "parse_ok",
    "llm_parse_ok",
    "json_parse_ok",
]

parse_col = next(
    (
        c
        for c in parse_candidates
        if c in joined.columns
    ),
    None
)

if parse_col is not None:

    joined["parse_ok_bool"] = joined[
        parse_col
    ].map(boolify)

    parse_success = int(
        joined["parse_ok_bool"].sum()
    )

else:
    # If runner only writes successfully parsed objects, inspect whether all
    # 200 treatment rows contain the CIR payload column.
    cir_candidates = [
        "cir_json",
        "llm_json",
        "parsed_json",
        "cir",
    ]

    cir_col = next(
        (
            c
            for c in cir_candidates
            if c in joined.columns
        ),
        None
    )

    assert cir_col is not None, (
        "STOP: cannot establish parse-success evidence."
    )

    parse_success = int(
        joined[cir_col].notna().sum()
    )


print("\n" + "-" * 118)
print("STRUCTURED-GENERATION RESULT")
print("-" * 118)

print(
    "Parse success:",
    fmt_ci(
        parse_success,
        200
    )
)


# =============================================================================
# 6. Treatment decision metrics
#
# Positive class = invalid/rejected, matching previous E2A convention.
# =============================================================================

expected_invalid = ~joined["expected_valid_bool"]
predicted_reject = ~joined["accepted_bool"]

TP = int(
    (expected_invalid & predicted_reject).sum()
)

FP = int(
    (~expected_invalid & predicted_reject).sum()
)

FN = int(
    (expected_invalid & ~predicted_reject).sum()
)

TN = int(
    (~expected_invalid & ~predicted_reject).sum()
)

assert TP + FP + FN + TN == 200

correct = (
    joined["expected_valid_bool"]
    == joined["accepted_bool"]
)

joined["treatment_correct"] = correct

accuracy_n = int(correct.sum())

valid_mask = joined["expected_valid_bool"]
invalid_mask = ~joined["expected_valid_bool"]

valid_accept_n = int(
    joined.loc[
        valid_mask,
        "accepted_bool"
    ].sum()
)

invalid_reject_n = int(
    (
        ~joined.loc[
            invalid_mask,
            "accepted_bool"
        ]
    ).sum()
)

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else float("nan")
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else float("nan")
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (
        not math.isnan(precision)
        and not math.isnan(recall)
        and (precision + recall)
    )
    else float("nan")
)


print("\n" + "-" * 118)
print("E2A cir_constrained DECISION RESULTS")
print("-" * 118)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")

print()
print("Decision accuracy :", fmt_ci(accuracy_n, 200))
print("Valid acceptance  :", fmt_ci(valid_accept_n, 100))
print("Invalid rejection :", fmt_ci(invalid_reject_n, 100))

print()
print(f"Precision         : {precision:.6f}")
print(f"Recall            : {recall:.6f}")
print(f"F1                : {f1:.6f}")


# =============================================================================
# 7. Historical CIR control
# =============================================================================

hist = pd.read_csv(
    HIST_RESULTS
)

hist["id"] = hist["id"].astype(str)

hist_cir = hist[
    hist["pipeline"].astype(str)
    == "cir"
].copy()

assert len(hist_cir) == 200
assert hist_cir["id"].nunique() == 200
assert set(hist_cir["id"]) == set(gt["id"])

hist_join = gt.merge(
    hist_cir,
    on="id",
    how="inner",
    suffixes=("_gt", "_hist"),
    validate="one_to_one"
)

assert len(hist_join) == 200


hist_accepted_candidates = [
    "accepted",
    "is_accepted",
    "validation_accepted",
]

hist_accepted_col = next(
    (
        c
        for c in hist_accepted_candidates
        if c in hist_join.columns
    ),
    None
)

assert hist_accepted_col is not None, (
    "STOP: historical CIR acceptance column not found.\n"
    f"Columns: {list(hist_join.columns)}"
)

hist_join["expected_valid_bool"] = hist_join[
    "expected_valid"
].map(boolify)

hist_join["accepted_bool"] = hist_join[
    hist_accepted_col
].map(boolify)

hist_join["historical_correct"] = (
    hist_join["expected_valid_bool"]
    == hist_join["accepted_bool"]
)

hist_accuracy_n = int(
    hist_join["historical_correct"].sum()
)

hist_valid_accept = int(
    hist_join.loc[
        hist_join["expected_valid_bool"],
        "accepted_bool"
    ].sum()
)

hist_invalid_reject = int(
    (
        ~hist_join.loc[
            ~hist_join["expected_valid_bool"],
            "accepted_bool"
        ]
    ).sum()
)


print("\n" + "-" * 118)
print("HISTORICAL E2A CIR CONTROL")
print("-" * 118)

print(
    "Decision accuracy :",
    fmt_ci(
        hist_accuracy_n,
        200
    )
)

print(
    "Valid acceptance  :",
    fmt_ci(
        hist_valid_accept,
        100
    )
)

print(
    "Invalid rejection :",
    fmt_ci(
        hist_invalid_reject,
        100
    )
)


# =============================================================================
# 8. Delta from historical CIR
# =============================================================================

treatment_accuracy = accuracy_n / 200
historical_accuracy = hist_accuracy_n / 200

treatment_var = valid_accept_n / 100
historical_var = hist_valid_accept / 100

treatment_irr = invalid_reject_n / 100
historical_irr = hist_invalid_reject / 100

print("\n" + "-" * 118)
print("CONSTRAINED DECODING vs HISTORICAL CIR")
print("-" * 118)

print(
    f"Accuracy delta : "
    f"{100*(treatment_accuracy-historical_accuracy):+.2f} pp"
)

print(
    f"VAR delta      : "
    f"{100*(treatment_var-historical_var):+.2f} pp"
)

print(
    f"IRR delta      : "
    f"{100*(treatment_irr-historical_irr):+.2f} pp"
)


# =============================================================================
# 9. Paired case-level comparison
# =============================================================================

paired = joined[
    [
        "id",
        "expected_valid_bool",
        "treatment_correct",
    ]
].merge(
    hist_join[
        [
            "id",
            "historical_correct",
        ]
    ],
    on="id",
    validate="one_to_one"
)

both_correct = int(
    (
        paired["treatment_correct"]
        & paired["historical_correct"]
    ).sum()
)

treatment_only = int(
    (
        paired["treatment_correct"]
        & ~paired["historical_correct"]
    ).sum()
)

historical_only = int(
    (
        ~paired["treatment_correct"]
        & paired["historical_correct"]
    ).sum()
)

both_wrong = int(
    (
        ~paired["treatment_correct"]
        & ~paired["historical_correct"]
    ).sum()
)

assert (
    both_correct
    + treatment_only
    + historical_only
    + both_wrong
    == 200
)


print("\n" + "-" * 118)
print("PAIRED CORRECTNESS TABLE")
print("-" * 118)

print("Both correct          :", both_correct)
print("Constrained only      :", treatment_only)
print("Historical CIR only   :", historical_only)
print("Both incorrect        :", both_wrong)


# =============================================================================
# 10. Exact McNemar test
# =============================================================================

discordant = (
    treatment_only
    + historical_only
)

if discordant:

    from scipy.stats import binomtest

    mcnemar = binomtest(
        min(
            treatment_only,
            historical_only
        ),
        n=discordant,
        p=0.5,
        alternative="two-sided"
    )

    mcnemar_p = float(
        mcnemar.pvalue
    )

else:

    mcnemar_p = 1.0


print()
print(
    "Discordant pairs      :",
    discordant
)

print(
    "Exact McNemar p-value :",
    f"{mcnemar_p:.12g}"
)


# =============================================================================
# 11. Category-level treatment performance
# =============================================================================

print("\n" + "-" * 118)
print("CATEGORY-LEVEL E2A cir_constrained PERFORMANCE")
print("-" * 118)

category_col = "category"

for category, group in joined.groupby(
    category_col,
    sort=True
):

    n = len(group)

    k = int(
        group["treatment_correct"].sum()
    )

    print(
        f"{category:42s} "
        f"{k:3d}/{n:3d} "
        f"({100*k/n:6.2f}%)"
    )


# =============================================================================
# 12. Treatment error IDs
# =============================================================================

errors = joined[
    ~joined["treatment_correct"]
].copy()

print("\n" + "-" * 118)
print("TREATMENT DECISION ERROR IDS")
print("-" * 118)

print(
    "Total treatment decision errors:",
    len(errors)
)

for _, r in errors.iterrows():

    print(
        f"{r['id']} | "
        f"category={r['category']} | "
        f"expected_valid={r['expected_valid_bool']} | "
        f"accepted={r['accepted_bool']}"
    )


# =============================================================================
# 13. Invalid cases accepted despite successful structured generation
#
# This directly demonstrates that syntactic/schema-constrained generation
# does not itself imply semantic validity.
# =============================================================================

semantic_escape = joined[
    (~joined["expected_valid_bool"])
    & joined["accepted_bool"]
].copy()

print("\n" + "-" * 118)
print("INVALID CASES STILL ACCEPTED AFTER STRUCTURED GENERATION")
print("-" * 118)

print(
    "Count:",
    len(semantic_escape),
    "/ 100 invalid cases"
)

if len(semantic_escape):

    counts = (
        semantic_escape["category"]
        .value_counts()
    )

    print("\nBy category:")

    for category, n in counts.items():

        print(
            f"{category:42s} {n:3d}"
        )


# =============================================================================
# 14. Cases changed relative to historical CIR
# =============================================================================

comparison = paired.merge(
    joined[
        [
            "id",
            "category",
            "accepted_bool",
        ]
    ],
    on="id",
    validate="one_to_one"
)

comparison = comparison.merge(
    hist_join[
        [
            "id",
            "accepted_bool",
        ]
    ].rename(
        columns={
            "accepted_bool":
                "historical_accepted"
        }
    ),
    on="id",
    validate="one_to_one"
)

changed = comparison[
    comparison["accepted_bool"]
    != comparison["historical_accepted"]
].copy()

print("\n" + "-" * 118)
print("DECISION CHANGES RELATIVE TO HISTORICAL CIR")
print("-" * 118)

print(
    "Cases with changed accept/reject decision:",
    len(changed)
)

if len(changed):

    change_counts = Counter(
        changed["category"]
    )

    print("\nChanged cases by category:")

    for category, n in sorted(
        change_counts.items()
    ):

        print(
            f"{category:42s} {n:3d}"
        )


# =============================================================================
# 15. Integrity
# =============================================================================

print("\n" + "-" * 118)
print("POST-ANALYSIS INTEGRITY")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


print("\n" + "=" * 118)
print("✓ STEP BR11 COMPLETE")
print("✓ E2A TREATMENT ANALYZED READ-ONLY")
print("✓ EXACT 95% CLOPPER-PEARSON INTERVALS COMPUTED")
print("✓ HISTORICAL CIR CONTROL COMPARED")
print("✓ PAIRED CASE-LEVEL COMPARISON COMPUTED")
print("✓ EXACT MCNEMAR TEST COMPUTED")
print("✓ CATEGORY PERFORMANCE COMPUTED")
print("✓ INVALID ACCEPTANCES AFTER STRUCTURED GENERATION IDENTIFIED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTION")
print("✓ NO FROZEN EVIDENCE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR12")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR11: READ-ONLY E2A STATISTICAL AND PAIRED ANALYSIS

----------------------------------------------------------------------------------------------------------------------
FROZEN EVIDENCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/results/E2A/experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR10R1_E2A_execution_record_v1.json
✓ experiment-2/data/frozen/experiment2_intents_200_v1.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_results_600.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_primary_metrics.csv
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR10_interrupted_execution_record_v1.json

✓ successful rep

KeyError: 'expected_valid'

In [55]:
from pathlib import Path
import hashlib
import pandas as pd

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR11-D1: READ-ONLY JOIN-SCHEMA DIAGNOSIS")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

TREATMENT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

DATASET = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# ----------------------------------------------------------------------
# Verify frozen files
# ----------------------------------------------------------------------

assert sha256(TREATMENT) == (
    "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6"
)

assert sha256(DATASET) == (
    "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753"
)

assert sha256(HIST_RESULTS) == (
    "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417"
)

print("\n✓ frozen treatment verified")
print("✓ frozen ground truth verified")
print("✓ frozen historical results verified")


# ----------------------------------------------------------------------
# Load files
# ----------------------------------------------------------------------

gt = pd.read_csv(DATASET)
tr = pd.read_csv(TREATMENT)
hist = pd.read_csv(HIST_RESULTS)

print("\n" + "-" * 118)
print("GROUND-TRUTH COLUMNS")
print("-" * 118)

for i, c in enumerate(gt.columns, 1):
    print(f"{i:2d}. {c}")


print("\n" + "-" * 118)
print("TREATMENT COLUMNS")
print("-" * 118)

for i, c in enumerate(tr.columns, 1):
    print(f"{i:2d}. {c}")


print("\n" + "-" * 118)
print("OVERLAPPING GT/TREATMENT COLUMNS")
print("-" * 118)

overlap = sorted(
    (set(gt.columns) & set(tr.columns)) - {"id"}
)

for c in overlap:
    print(c)

print("\nOverlap count:", len(overlap))


# ----------------------------------------------------------------------
# Reproduce BR11 join exactly
# ----------------------------------------------------------------------

gt["id"] = gt["id"].astype(str)
tr["id"] = tr["id"].astype(str)

joined = gt.merge(
    tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 200

print("\n" + "-" * 118)
print("JOINED COLUMNS")
print("-" * 118)

for i, c in enumerate(joined.columns, 1):
    print(f"{i:2d}. {c}")


# ----------------------------------------------------------------------
# Locate important columns
# ----------------------------------------------------------------------

print("\n" + "-" * 118)
print("IMPORTANT COLUMN CANDIDATES")
print("-" * 118)

terms = [
    "expected",
    "valid",
    "accept",
    "parse",
    "category",
    "service",
    "cir",
    "json",
    "error",
]

for term in terms:

    matches = [
        c for c in joined.columns
        if term.lower() in c.lower()
    ]

    print(f"\n{term}:")
    for c in matches:
        print("  ", c)


# ----------------------------------------------------------------------
# Historical CIR schema
# ----------------------------------------------------------------------

assert "pipeline" in hist.columns

hist_cir = hist[
    hist["pipeline"].astype(str) == "cir"
].copy()

assert len(hist_cir) == 200

print("\n" + "-" * 118)
print("HISTORICAL CIR COLUMNS")
print("-" * 118)

for i, c in enumerate(hist_cir.columns, 1):
    print(f"{i:2d}. {c}")


# ----------------------------------------------------------------------
# Check duplicate semantic fields
# ----------------------------------------------------------------------

print("\n" + "-" * 118)
print("DUPLICATED FIELD CONSISTENCY")
print("-" * 118)

for base in [
    "expected_valid",
    "category",
    "expected_service",
    "intent",
]:

    gt_col = f"{base}_gt"
    tr_col = f"{base}_tr"

    if (
        gt_col in joined.columns
        and tr_col in joined.columns
    ):

        a = joined[gt_col]
        b = joined[tr_col]

        # String normalization permits NaN comparison.
        equal = (
            a.fillna("<NA>").astype(str)
            ==
            b.fillna("<NA>").astype(str)
        )

        print(
            f"{base:20s}: "
            f"{int(equal.sum())}/200 identical"
        )

    else:

        print(
            f"{base:20s}: "
            f"not duplicated in both sources"
        )


# ----------------------------------------------------------------------
# Integrity
# ----------------------------------------------------------------------

assert sha256(TREATMENT) == (
    "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6"
)

assert sha256(DATASET) == (
    "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753"
)

assert sha256(HIST_RESULTS) == (
    "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417"
)

print("\n" + "=" * 118)
print("✓ STEP BR11-D1 COMPLETE")
print("✓ JOIN SCHEMA INSPECTED READ-ONLY")
print("✓ IMPORTANT COLUMN NAMES IDENTIFIED")
print("✓ HISTORICAL CIR SCHEMA INSPECTED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTION")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE CORRECTING BR11")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR11-D1: READ-ONLY JOIN-SCHEMA DIAGNOSIS

✓ frozen treatment verified
✓ frozen ground truth verified
✓ frozen historical results verified

----------------------------------------------------------------------------------------------------------------------
GROUND-TRUTH COLUMNS
----------------------------------------------------------------------------------------------------------------------
 1. id
 2. intent
 3. category
 4. subcategory
 5. expected_valid
 6. expected_service
 7. difficulty
 8. linguistic_form
 9. constraint_type
10. notes

----------------------------------------------------------------------------------------------------------------------
TREATMENT COLUMNS
----------------------------------------------------------------------------------------------------------------------
 1. id
 2. intent
 3. category
 4. expected_service
 5. expected_valid
 6. pipeline
 7. model
 8. parse_ok
 9. latency_ms
10. finish_reason
11. prompt_tokens
12. comp

In [56]:
from pathlib import Path
import hashlib
import json
import math
import pandas as pd

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR11-R1: CORRECTED READ-ONLY E2A STATISTICAL AND PAIRED ANALYSIS")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

TREATMENT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R1_E2A_execution_record_v1.json"
)

DATASET = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)

INTERRUPTED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_interrupted_execution_record_v1.json"
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def boolify(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)) and not pd.isna(v):
        if v == 1:
            return True
        if v == 0:
            return False

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(f"Cannot convert to bool: {v!r}")


def cp_interval(k, n, alpha=0.05):
    from scipy.stats import beta

    if n == 0:
        return float("nan"), float("nan")

    lo = (
        0.0
        if k == 0
        else beta.ppf(alpha / 2, k, n - k + 1)
    )

    hi = (
        1.0
        if k == n
        else beta.ppf(1 - alpha / 2, k + 1, n - k)
    )

    return float(lo), float(hi)


def fmt_ci(k, n):
    lo, hi = cp_interval(k, n)

    return (
        f"{k}/{n} = {k/n:.6f} "
        f"({100*k/n:.2f}%), "
        f"95% CI [{lo:.6f}, {hi:.6f}]"
    )


# =============================================================================
# 1. Verify frozen evidence
# =============================================================================

EXPECTED = {
    TREATMENT:
        "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6",

    EXEC_RECORD:
        "418b45e0ad487fed94442f7da2b8cd1987006a6727cb3b757218923c5c8322e5",

    DATASET:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",

    INTERRUPTED_RECORD:
        "21514eb08dcf63d6742acbfb4e80bb0ec7416b332b4f6b5d681fe57912cf091d",
}

print("\n" + "-" * 118)
print("FROZEN EVIDENCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert path.exists(), f"Missing: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"Hash mismatch:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# 2. Verify successful execution provenance
# =============================================================================

record = json.loads(
    EXEC_RECORD.read_text(
        encoding="utf-8"
    )
)

assert record["status"] == "COMPLETED_AND_FROZEN_DO_NOT_RERUN"
assert record["benchmark_cases"] == 200
assert record["result_rows"] == 200
assert record["process_return_code"] == 0
assert record["transient_retry_events"] == 0
assert record["scientific_treatment_changed"] is False

print("\n✓ successful BR10-R1 replacement execution verified")
print("✓ 200/200 observations frozen")
print("✓ transient retry events = 0")
print("✓ scientific treatment unchanged")


# =============================================================================
# 3. Load frozen evidence
# =============================================================================

gt = pd.read_csv(DATASET)
tr = pd.read_csv(TREATMENT)
hist = pd.read_csv(HIST_RESULTS)

assert len(gt) == 200
assert len(tr) == 200
assert len(hist) == 600

gt["id"] = gt["id"].astype(str)
tr["id"] = tr["id"].astype(str)
hist["id"] = hist["id"].astype(str)

assert gt["id"].nunique() == 200
assert tr["id"].nunique() == 200
assert set(gt["id"]) == set(tr["id"])


# =============================================================================
# 4. Join treatment and ground truth
# =============================================================================

joined = gt.merge(
    tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 200


# =============================================================================
# 5. Explicitly verify duplicated benchmark metadata
# =============================================================================

print("\n" + "-" * 118)
print("JOIN CONSISTENCY")
print("-" * 118)

for base in [
    "intent",
    "category",
    "expected_valid",
    "expected_service",
]:

    a = (
        joined[f"{base}_gt"]
        .fillna("<NA>")
        .astype(str)
    )

    b = (
        joined[f"{base}_tr"]
        .fillna("<NA>")
        .astype(str)
    )

    assert (a == b).all()

    print(
        f"✓ {base}: 200/200 identical "
        f"between frozen benchmark and treatment output"
    )


# Use ground truth explicitly from here onward.
joined["expected_valid_bool"] = (
    joined["expected_valid_gt"]
    .map(boolify)
)

joined["accepted_bool"] = (
    joined["accepted"]
    .map(boolify)
)

joined["parse_ok_bool"] = (
    joined["parse_ok"]
    .map(boolify)
)

joined["schema_valid_bool"] = (
    joined["schema_valid"]
    .map(boolify)
)

assert joined["expected_valid_bool"].sum() == 100

print("✓ ground truth = 100 valid / 100 invalid")


# =============================================================================
# 6. Structured-generation observations
# =============================================================================

parse_success = int(
    joined["parse_ok_bool"].sum()
)

schema_success = int(
    joined["schema_valid_bool"].sum()
)

print("\n" + "-" * 118)
print("STRUCTURED-GENERATION OBSERVATIONS")
print("-" * 118)

print(
    "Parse success       :",
    fmt_ci(parse_success, 200)
)

print(
    "Schema-valid output :",
    fmt_ci(schema_success, 200)
)


# =============================================================================
# 7. Treatment decision metrics
#
# Positive class = expected-invalid / rejected.
# =============================================================================

expected_invalid = ~joined["expected_valid_bool"]
predicted_reject = ~joined["accepted_bool"]

TP = int(
    (expected_invalid & predicted_reject).sum()
)

FP = int(
    (~expected_invalid & predicted_reject).sum()
)

FN = int(
    (expected_invalid & ~predicted_reject).sum()
)

TN = int(
    (~expected_invalid & ~predicted_reject).sum()
)

assert TP + FP + FN + TN == 200

joined["treatment_correct"] = (
    joined["expected_valid_bool"]
    == joined["accepted_bool"]
)

accuracy_n = int(
    joined["treatment_correct"].sum()
)

valid_mask = joined["expected_valid_bool"]
invalid_mask = ~joined["expected_valid_bool"]

valid_accept_n = int(
    joined.loc[
        valid_mask,
        "accepted_bool"
    ].sum()
)

invalid_reject_n = int(
    (
        ~joined.loc[
            invalid_mask,
            "accepted_bool"
        ]
    ).sum()
)

precision = (
    TP / (TP + FP)
    if (TP + FP)
    else float("nan")
)

recall = (
    TP / (TP + FN)
    if (TP + FN)
    else float("nan")
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (
        not math.isnan(precision)
        and not math.isnan(recall)
        and (precision + recall)
    )
    else float("nan")
)


print("\n" + "-" * 118)
print("E2A cir_constrained DECISION RESULTS")
print("-" * 118)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")

print()
print(
    "Decision accuracy :",
    fmt_ci(accuracy_n, 200)
)

print(
    "VAR               :",
    fmt_ci(valid_accept_n, 100)
)

print(
    "IRR               :",
    fmt_ci(invalid_reject_n, 100)
)

print()
print(f"Precision         : {precision:.6f}")
print(f"Recall            : {recall:.6f}")
print(f"F1                : {f1:.6f}")


# =============================================================================
# 8. Service extraction on expected-valid cases
# =============================================================================

valid_rows = joined[
    joined["expected_valid_bool"]
].copy()

service_match = (
    valid_rows["expected_service_gt"]
    .fillna("<NA>")
    .astype(str)
    ==
    valid_rows["predicted_service"]
    .fillna("<NA>")
    .astype(str)
)

service_correct = int(
    service_match.sum()
)

print("\n" + "-" * 118)
print("SERVICE EXTRACTION — EXPECTED-VALID CASES")
print("-" * 118)

print(
    "Correct service:",
    fmt_ci(
        service_correct,
        len(valid_rows)
    )
)


# =============================================================================
# 9. Historical CIR control
# =============================================================================

hist_cir = hist[
    hist["pipeline"].astype(str)
    == "cir"
].copy()

assert len(hist_cir) == 200
assert hist_cir["id"].nunique() == 200
assert set(hist_cir["id"]) == set(gt["id"])

hist_join = gt.merge(
    hist_cir,
    on="id",
    how="inner",
    suffixes=("_gt", "_hist"),
    validate="one_to_one"
)

assert len(hist_join) == 200

# Verify duplicated metadata here as well.
for base in [
    "intent",
    "category",
    "expected_valid",
    "expected_service",
]:

    a = (
        hist_join[f"{base}_gt"]
        .fillna("<NA>")
        .astype(str)
    )

    b = (
        hist_join[f"{base}_hist"]
        .fillna("<NA>")
        .astype(str)
    )

    assert (a == b).all()


hist_join["expected_valid_bool"] = (
    hist_join["expected_valid_gt"]
    .map(boolify)
)

hist_join["accepted_bool"] = (
    hist_join["accepted"]
    .map(boolify)
)

hist_join["historical_correct"] = (
    hist_join["expected_valid_bool"]
    == hist_join["accepted_bool"]
)

hist_accuracy_n = int(
    hist_join["historical_correct"].sum()
)

hist_valid_accept_n = int(
    hist_join.loc[
        hist_join["expected_valid_bool"],
        "accepted_bool"
    ].sum()
)

hist_invalid_reject_n = int(
    (
        ~hist_join.loc[
            ~hist_join["expected_valid_bool"],
            "accepted_bool"
        ]
    ).sum()
)

print("\n" + "-" * 118)
print("HISTORICAL E2A CIR CONTROL")
print("-" * 118)

print(
    "Decision accuracy :",
    fmt_ci(hist_accuracy_n, 200)
)

print(
    "VAR               :",
    fmt_ci(hist_valid_accept_n, 100)
)

print(
    "IRR               :",
    fmt_ci(hist_invalid_reject_n, 100)
)


# =============================================================================
# 10. Verify historical metrics against known frozen counts
# =============================================================================

assert hist_accuracy_n == 151, (
    f"Expected historical CIR correct=151, observed {hist_accuracy_n}"
)

assert hist_valid_accept_n == 96, (
    f"Expected historical CIR valid acceptance=96, "
    f"observed {hist_valid_accept_n}"
)

assert hist_invalid_reject_n == 55, (
    f"Expected historical CIR invalid rejection=55, "
    f"observed {hist_invalid_reject_n}"
)

print("✓ historical CIR frozen counts reproduce exactly")


# =============================================================================
# 11. Treatment deltas
# =============================================================================

treatment_accuracy = accuracy_n / 200
historical_accuracy = hist_accuracy_n / 200

treatment_var = valid_accept_n / 100
historical_var = hist_valid_accept_n / 100

treatment_irr = invalid_reject_n / 100
historical_irr = hist_invalid_reject_n / 100

print("\n" + "-" * 118)
print("cir_constrained vs HISTORICAL CIR")
print("-" * 118)

print(
    f"Accuracy : "
    f"{100*historical_accuracy:.2f}% -> "
    f"{100*treatment_accuracy:.2f}% "
    f"({100*(treatment_accuracy-historical_accuracy):+.2f} pp)"
)

print(
    f"VAR      : "
    f"{100*historical_var:.2f}% -> "
    f"{100*treatment_var:.2f}% "
    f"({100*(treatment_var-historical_var):+.2f} pp)"
)

print(
    f"IRR      : "
    f"{100*historical_irr:.2f}% -> "
    f"{100*treatment_irr:.2f}% "
    f"({100*(treatment_irr-historical_irr):+.2f} pp)"
)


# =============================================================================
# 12. Paired case-level correctness
# =============================================================================

paired = (
    joined[
        [
            "id",
            "treatment_correct",
        ]
    ]
    .merge(
        hist_join[
            [
                "id",
                "historical_correct",
            ]
        ],
        on="id",
        validate="one_to_one"
    )
)

both_correct = int(
    (
        paired["treatment_correct"]
        & paired["historical_correct"]
    ).sum()
)

constrained_only = int(
    (
        paired["treatment_correct"]
        & ~paired["historical_correct"]
    ).sum()
)

historical_only = int(
    (
        ~paired["treatment_correct"]
        & paired["historical_correct"]
    ).sum()
)

both_wrong = int(
    (
        ~paired["treatment_correct"]
        & ~paired["historical_correct"]
    ).sum()
)

assert (
    both_correct
    + constrained_only
    + historical_only
    + both_wrong
    == 200
)

print("\n" + "-" * 118)
print("PAIRED CORRECTNESS TABLE")
print("-" * 118)

print("Both correct        :", both_correct)
print("Constrained only    :", constrained_only)
print("Historical CIR only :", historical_only)
print("Both incorrect      :", both_wrong)


# =============================================================================
# 13. Exact McNemar test
# =============================================================================

discordant = (
    constrained_only
    + historical_only
)

if discordant > 0:

    from scipy.stats import binomtest

    mcnemar_p = float(
        binomtest(
            constrained_only,
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    )

else:

    mcnemar_p = 1.0


print()
print("Discordant pairs      :", discordant)
print("Exact McNemar p-value :", f"{mcnemar_p:.12g}")


# =============================================================================
# 14. Category-level performance
# =============================================================================

print("\n" + "-" * 118)
print("CATEGORY-LEVEL E2A cir_constrained PERFORMANCE")
print("-" * 118)

for category, group in joined.groupby(
    "category_gt",
    sort=True
):

    n = len(group)

    k = int(
        group["treatment_correct"].sum()
    )

    print(
        f"{str(category):42s} "
        f"{k:3d}/{n:3d} "
        f"({100*k/n:6.2f}%)"
    )


# =============================================================================
# 15. Decision errors
# =============================================================================

errors = joined[
    ~joined["treatment_correct"]
].copy()

print("\n" + "-" * 118)
print("TREATMENT DECISION ERRORS")
print("-" * 118)

print(
    "Total errors:",
    len(errors)
)

for _, r in errors.iterrows():

    print(
        f"{r['id']} | "
        f"category={r['category_gt']} | "
        f"subcategory={r['subcategory']} | "
        f"expected_valid={r['expected_valid_bool']} | "
        f"accepted={r['accepted_bool']} | "
        f"predicted_service={r['predicted_service']}"
    )


# =============================================================================
# 16. Invalid cases still accepted
# =============================================================================

semantic_escape = joined[
    (~joined["expected_valid_bool"])
    & joined["accepted_bool"]
].copy()

print("\n" + "-" * 118)
print("INVALID CASES STILL ACCEPTED AFTER STRUCTURED GENERATION")
print("-" * 118)

print(
    f"Count: {len(semantic_escape)}/100 "
    f"({len(semantic_escape):.2f}%)"
)

print("\nBy category:")

for category, n in (
    semantic_escape[
        "category_gt"
    ]
    .value_counts()
    .sort_index()
    .items()
):

    print(
        f"{str(category):42s} {n:3d}"
    )


# =============================================================================
# 17. Structurally valid but semantically wrong decisions
# =============================================================================

structural_semantic_failures = joined[
    joined["parse_ok_bool"]
    & joined["schema_valid_bool"]
    & (~joined["treatment_correct"])
].copy()

print("\n" + "-" * 118)
print("STRUCTURALLY VALID OUTPUTS WITH WRONG FINAL DECISION")
print("-" * 118)

print(
    "Count:",
    len(structural_semantic_failures),
    "/ 200"
)

if len(structural_semantic_failures):

    print("\nBy category:")

    for category, n in (
        structural_semantic_failures[
            "category_gt"
        ]
        .value_counts()
        .sort_index()
        .items()
    ):

        print(
            f"{str(category):42s} {n:3d}"
        )


# =============================================================================
# 18. Decision changes relative to historical CIR
# =============================================================================

comparison = joined[
    [
        "id",
        "category_gt",
        "expected_valid_bool",
        "accepted_bool",
        "treatment_correct",
    ]
].merge(
    hist_join[
        [
            "id",
            "accepted_bool",
            "historical_correct",
        ]
    ].rename(
        columns={
            "accepted_bool":
                "historical_accepted"
        }
    ),
    on="id",
    validate="one_to_one"
)

changed = comparison[
    comparison["accepted_bool"]
    != comparison["historical_accepted"]
].copy()

print("\n" + "-" * 118)
print("DECISION CHANGES RELATIVE TO HISTORICAL CIR")
print("-" * 118)

print(
    "Changed accept/reject decisions:",
    len(changed)
)

if len(changed):

    print("\nBy category:")

    for category, n in (
        changed[
            "category_gt"
        ]
        .value_counts()
        .sort_index()
        .items()
    ):

        print(
            f"{str(category):42s} {n:3d}"
        )


# =============================================================================
# 19. Improvement/regression direction
# =============================================================================

improved = comparison[
    comparison["treatment_correct"]
    & ~comparison["historical_correct"]
]

regressed = comparison[
    ~comparison["treatment_correct"]
    & comparison["historical_correct"]
]

print("\n" + "-" * 118)
print("CASE-LEVEL DIRECTION OF CHANGE")
print("-" * 118)

print(
    "Historical wrong -> constrained correct:",
    len(improved)
)

print(
    "Historical correct -> constrained wrong:",
    len(regressed)
)


# =============================================================================
# 20. Post-analysis integrity
# =============================================================================

print("\n" + "-" * 118)
print("POST-ANALYSIS INTEGRITY")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


print("\n" + "=" * 118)
print("✓ STEP BR11-R1 COMPLETE")
print("✓ CORRECTED JOIN SCHEMA USED")
print("✓ E2A TREATMENT ANALYZED READ-ONLY")
print("✓ PARSE AND STRUCTURAL VALIDITY COMPUTED")
print("✓ EXACT 95% CLOPPER-PEARSON INTERVALS COMPUTED")
print("✓ HISTORICAL CIR CONTROL REPRODUCED")
print("✓ PAIRED CASE-LEVEL COMPARISON COMPUTED")
print("✓ EXACT MCNEMAR TEST COMPUTED")
print("✓ SEMANTIC FAILURES AFTER STRUCTURED GENERATION IDENTIFIED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTION")
print("✓ NO FROZEN EVIDENCE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR12")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR11-R1: CORRECTED READ-ONLY E2A STATISTICAL AND PAIRED ANALYSIS

----------------------------------------------------------------------------------------------------------------------
FROZEN EVIDENCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/results/E2A/experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR10R1_E2A_execution_record_v1.json
✓ experiment-2/data/frozen/experiment2_intents_200_v1.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_results_600.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_primary_metrics.csv
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR10_interrupted_execution_record_v1.json

✓ s

In [57]:
from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 118)
print("EXPERIMENT B-REVIEW — STEP BR12-D1: READ-ONLY FORENSIC FIELD INSPECTION")
print("=" * 118)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

TREATMENT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

DATASET = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R1_E2A_execution_record_v1.json"
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def boolify(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)) and not pd.isna(v):
        if v == 1:
            return True
        if v == 0:
            return False

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(f"Cannot convert to bool: {v!r}")


def short(value, limit=500):
    if pd.isna(value):
        return "<NA>"

    s = str(value).replace("\n", " ")

    if len(s) > limit:
        return s[:limit] + "..."

    return s


# =============================================================================
# 1. Verify frozen evidence
# =============================================================================

EXPECTED = {
    TREATMENT:
        "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6",

    DATASET:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    EXEC_RECORD:
        "418b45e0ad487fed94442f7da2b8cd1987006a6727cb3b757218923c5c8322e5",
}

print("\n" + "-" * 118)
print("FROZEN EVIDENCE")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert path.exists(), f"STOP: missing file:\n{path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: hash mismatch:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# 2. Load frozen evidence
# =============================================================================

gt = pd.read_csv(DATASET)
tr = pd.read_csv(TREATMENT)
hist = pd.read_csv(HIST_RESULTS)

assert len(gt) == 200
assert len(tr) == 200
assert len(hist) == 600

gt["id"] = gt["id"].astype(str)
tr["id"] = tr["id"].astype(str)
hist["id"] = hist["id"].astype(str)

assert gt["id"].nunique() == 200
assert tr["id"].nunique() == 200
assert set(gt["id"]) == set(tr["id"])

joined = gt.merge(
    tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 200

joined["expected_valid_bool"] = (
    joined["expected_valid_gt"].map(boolify)
)

joined["accepted_bool"] = (
    joined["accepted"].map(boolify)
)

joined["parse_ok_bool"] = (
    joined["parse_ok"].map(boolify)
)

joined["schema_valid_bool"] = (
    joined["schema_valid"].map(boolify)
)

print("\n✓ frozen treatment joined one-to-one with ground truth")


# =============================================================================
# 3. Verify duplicated benchmark metadata
# =============================================================================

for base in [
    "intent",
    "category",
    "expected_valid",
    "expected_service",
]:

    left = (
        joined[f"{base}_gt"]
        .fillna("<NA>")
        .astype(str)
    )

    right = (
        joined[f"{base}_tr"]
        .fillna("<NA>")
        .astype(str)
    )

    assert (left == right).all()

print("✓ duplicated benchmark metadata identical 200/200")


# =============================================================================
# 4. Historical CIR extraction
# =============================================================================

hist_cir = hist[
    hist["pipeline"].astype(str) == "cir"
].copy()

assert len(hist_cir) == 200
assert hist_cir["id"].nunique() == 200
assert set(hist_cir["id"]) == set(gt["id"])

print("✓ historical CIR control = 200 cases")


# =============================================================================
# 5. Field availability
# =============================================================================

print("\n" + "-" * 118)
print("FORENSIC FIELD AVAILABILITY")
print("-" * 118)

fields = [
    "raw_response",
    "cir_json",
    "validation_errors",
    "parse_ok",
    "schema_valid",
    "accepted",
    "predicted_service",
    "unsupported_input_detected",
    "hallucination_detected",
    "input_audit_valid",
    "mapping_found",
    "descriptor_success",
]

for field in fields:

    print(
        f"{field:30s} "
        f"treatment={'YES' if field in tr.columns else 'NO ':3s} "
        f"historical_CIR={'YES' if field in hist_cir.columns else 'NO ':3s}"
    )


# =============================================================================
# 6. Parse cir_json representation
# =============================================================================

print("\n" + "-" * 118)
print("cir_json REPRESENTATION CHECK")
print("-" * 118)

cir_parse_ok = 0
cir_parse_fail = []

for _, row in tr.iterrows():

    value = row["cir_json"]

    if pd.isna(value):
        cir_parse_fail.append(
            (row["id"], "NA")
        )
        continue

    try:
        obj = json.loads(str(value))

        if isinstance(obj, dict):
            cir_parse_ok += 1
        else:
            cir_parse_fail.append(
                (
                    row["id"],
                    f"JSON type={type(obj).__name__}"
                )
            )

    except Exception as e:
        cir_parse_fail.append(
            (
                row["id"],
                type(e).__name__
            )
        )

print(
    f"Treatment cir_json parseable dictionaries: "
    f"{cir_parse_ok}/200"
)

print(
    "Treatment cir_json non-dictionary/unparseable/NA:",
    len(cir_parse_fail)
)

if cir_parse_fail:
    print("\nFirst failures:")
    for item in cir_parse_fail[:20]:
        print(" ", item)


# =============================================================================
# 7. Inspect schema-invalid treatment cases
# =============================================================================

schema_invalid = joined[
    ~joined["schema_valid_bool"]
].copy()

print("\n" + "-" * 118)
print("SCHEMA-INVALID TREATMENT CASES")
print("-" * 118)

print("Count:", len(schema_invalid))

print("\nBy ground-truth category:")

for category, n in (
    schema_invalid["category_gt"]
    .value_counts()
    .sort_index()
    .items()
):
    print(
        f"{str(category):42s} {n:3d}"
    )

print("\nCase details:")

for _, r in schema_invalid.iterrows():

    print("\n" + "." * 118)

    print("ID               :", r["id"])
    print("Category         :", r["category_gt"])
    print("Subcategory      :", r["subcategory"])
    print("Expected valid   :", r["expected_valid_bool"])
    print("Accepted         :", r["accepted_bool"])
    print("Parse OK         :", r["parse_ok_bool"])
    print("Schema valid     :", r["schema_valid_bool"])
    print("Predicted service:", r["predicted_service"])

    print(
        "Validation errors:",
        short(r["validation_errors"], 800)
    )

    print(
        "CIR JSON         :",
        short(r["cir_json"], 1000)
    )

    print(
        "Raw response     :",
        short(r["raw_response"], 1000)
    )


# =============================================================================
# 8. Inspect semantic-escape categories
# =============================================================================

TARGET_CATEGORIES = [
    "invalid_explicit_unsupported_parameter",
    "invalid_implicit_unsupported_parameter",
    "invalid_contradictory",
    "invalid_compound_mixed",
    "invalid_prompt_adversarial",
]

targets = joined[
    joined["category_gt"].isin(TARGET_CATEGORIES)
].copy()

print("\n" + "-" * 118)
print("TARGET SEMANTIC-FAILURE CATEGORIES")
print("-" * 118)

for category in TARGET_CATEGORIES:

    group = targets[
        targets["category_gt"] == category
    ]

    accepted_n = int(
        group["accepted_bool"].sum()
    )

    schema_n = int(
        group["schema_valid_bool"].sum()
    )

    print(
        f"{category:42s} "
        f"n={len(group):3d} | "
        f"schema_valid={schema_n:3d} | "
        f"accepted={accepted_n:3d}"
    )


# =============================================================================
# 9. Display representative cases
#
# First 3 IDs from each target category only.
# No classification is inferred yet.
# =============================================================================

print("\n" + "-" * 118)
print("REPRESENTATIVE FORENSIC SAMPLES")
print("-" * 118)

for category in TARGET_CATEGORIES:

    group = (
        targets[
            targets["category_gt"] == category
        ]
        .sort_values("id")
        .head(3)
    )

    print("\n" + "#" * 118)
    print("CATEGORY:", category)
    print("#" * 118)

    for _, r in group.iterrows():

        hrow = hist_cir[
            hist_cir["id"] == r["id"]
        ]

        assert len(hrow) == 1

        hrow = hrow.iloc[0]

        print("\nID:", r["id"])
        print("Subcategory:", r["subcategory"])

        print(
            "SOURCE INTENT:",
            short(r["intent_gt"], 1200)
        )

        print(
            "CONSTRAINED CIR:",
            short(r["cir_json"], 1200)
        )

        print(
            "CONSTRAINED VALIDATION:",
            short(r["validation_errors"], 800)
        )

        print(
            "CONSTRAINED ACCEPTED:",
            r["accepted_bool"]
        )

        print(
            "HISTORICAL CIR:",
            short(hrow["cir_json"], 1200)
        )

        print(
            "HISTORICAL VALIDATION:",
            short(hrow["validation_errors"], 800)
        )

        print(
            "HISTORICAL ACCEPTED:",
            hrow["accepted"]
        )


# =============================================================================
# 10. Unsupported-parameter cases — full source/CIR display
#
# These are central to the reviewer's dropped-constraint question.
# We inspect them but do NOT automatically label omission yet.
# =============================================================================

unsupported = joined[
    joined["category_gt"].isin(
        [
            "invalid_explicit_unsupported_parameter",
            "invalid_implicit_unsupported_parameter",
        ]
    )
].copy()

assert len(unsupported) == 24

print("\n" + "-" * 118)
print("ALL 24 UNSUPPORTED-PARAMETER CASES — SOURCE vs CONSTRAINED CIR")
print("-" * 118)

for _, r in unsupported.sort_values("id").iterrows():

    print("\n" + "." * 118)

    print(
        f"{r['id']} | "
        f"{r['category_gt']} | "
        f"{r['subcategory']}"
    )

    print(
        "SOURCE:",
        short(r["intent_gt"], 1500)
    )

    print(
        "CIR   :",
        short(r["cir_json"], 1500)
    )

    print(
        "ACCEPTED:",
        r["accepted_bool"]
    )

    print(
        "VALIDATION:",
        short(r["validation_errors"], 1000)
    )


# =============================================================================
# 11. Validation-error vocabulary
# =============================================================================

print("\n" + "-" * 118)
print("VALIDATION-ERROR VALUE INSPECTION")
print("-" * 118)

value_counts = (
    tr["validation_errors"]
    .fillna("<NA>")
    .astype(str)
    .value_counts()
)

print(
    "Unique validation_errors values:",
    len(value_counts)
)

print("\nMost frequent values:")

for value, n in value_counts.head(30).items():

    print(
        f"{n:3d} | {short(value, 1000)}"
    )


# =============================================================================
# 12. Final integrity
# =============================================================================

print("\n" + "-" * 118)
print("POST-INSPECTION INTEGRITY")
print("-" * 118)

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


print("\n" + "=" * 118)
print("✓ STEP BR12-D1 COMPLETE")
print("✓ FORENSIC FIELD REPRESENTATIONS INSPECTED")
print("✓ SCHEMA-INVALID CASES INSPECTED")
print("✓ SEMANTIC-FAILURE CATEGORIES INSPECTED")
print("✓ ALL 24 UNSUPPORTED-PARAMETER CASES DISPLAYED")
print("✓ HISTORICAL CIR AVAILABLE FOR SIDE-BY-SIDE ANALYSIS")
print("✓ NO DROPPED-CONSTRAINT LABELS ASSUMED YET")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTION")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE BR12-R1")
print("=" * 118)

EXPERIMENT B-REVIEW — STEP BR12-D1: READ-ONLY FORENSIC FIELD INSPECTION

----------------------------------------------------------------------------------------------------------------------
FROZEN EVIDENCE
----------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/results/E2A/experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv
✓ experiment-2/data/frozen/experiment2_intents_200_v1.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_results_600.csv
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR10R1_E2A_execution_record_v1.json

✓ frozen treatment joined one-to-one with ground truth
✓ duplicated benchmark metadata identical 200/200
✓ historical CIR control = 200 cases

-------------------------------------------------------------------------------------------------------------

In [58]:
from pathlib import Path
import hashlib
import json
import re
import pandas as pd
from collections import Counter

print("=" * 120)
print("EXPERIMENT B-REVIEW — STEP BR12-R1: FORENSIC UNSUPPORTED-CONSTRAINT CLASSIFICATION")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

TREATMENT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

DATASET = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

HIST_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

EXEC_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R1_E2A_execution_record_v1.json"
)

ANALYSIS_DIR = BR / "analysis"
MANIFEST_DIR = BR / "manifests"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = (
    ANALYSIS_DIR
    / "experiment_B_review_BR12R1_unsupported_constraint_forensics_v1.csv"
)

OUT_JSON = (
    ANALYSIS_DIR
    / "experiment_B_review_BR12R1_forensic_summary_v1.json"
)

OUT_MANIFEST = (
    MANIFEST_DIR
    / "experiment_B_review_BR12R1_analysis_manifest_v1.json"
)

# Persistent evidence location.
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def boolify(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)) and not pd.isna(v):
        if v == 1:
            return True
        if v == 0:
            return False

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(f"Cannot convert to bool: {v!r}")


def json_dump_frozen(obj, path):
    text = json.dumps(
        obj,
        indent=2,
        sort_keys=True,
        ensure_ascii=False
    ) + "\n"

    path.write_text(text, encoding="utf-8")


# =============================================================================
# Frozen hashes
# =============================================================================

EXPECTED = {
    TREATMENT:
        "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6",

    DATASET:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    HIST_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    EXEC_RECORD:
        "418b45e0ad487fed94442f7da2b8cd1987006a6727cb3b757218923c5c8322e5",
}

print("\n" + "-" * 120)
print("FROZEN INPUT VERIFICATION")
print("-" * 120)

for path, expected in EXPECTED.items():

    assert path.exists(), f"STOP: missing frozen artifact:\n{path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: frozen artifact hash mismatch:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# Prevent accidental overwrite
# =============================================================================

for path in [OUT_CSV, OUT_JSON, OUT_MANIFEST]:

    assert not path.exists(), (
        f"STOP: BR12-R1 artifact already exists:\n{path}\n"
        "DO NOT overwrite or rerun BR12-R1."
    )


# =============================================================================
# Load
# =============================================================================

gt = pd.read_csv(DATASET)
tr = pd.read_csv(TREATMENT)
hist = pd.read_csv(HIST_RESULTS)

gt["id"] = gt["id"].astype(str)
tr["id"] = tr["id"].astype(str)
hist["id"] = hist["id"].astype(str)

joined = gt.merge(
    tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 200

hist_cir = hist[
    hist["pipeline"].astype(str) == "cir"
].copy()

assert len(hist_cir) == 200
assert hist_cir["id"].nunique() == 200


# =============================================================================
# Unsupported-case scope
# =============================================================================

TARGETS = {
    "invalid_explicit_unsupported_parameter",
    "invalid_implicit_unsupported_parameter",
}

unsupported = joined[
    joined["category_gt"].isin(TARGETS)
].copy()

unsupported = unsupported.sort_values("id").reset_index(drop=True)

assert len(unsupported) == 24

assert (
    unsupported["category_gt"]
    .value_counts()
    .to_dict()
    ==
    {
        "invalid_explicit_unsupported_parameter": 12,
        "invalid_implicit_unsupported_parameter": 12,
    }
)

assert unsupported["accepted"].map(boolify).all()

print("\n✓ exactly 24 unsupported-parameter cases selected")
print("✓ 12 explicit + 12 implicit")
print("✓ all 24 were accepted by cir_constrained")


# =============================================================================
# IMPORTANT METHODOLOGICAL RULE
#
# This analysis does NOT use an LLM or subjective post-hoc model.
#
# The benchmark cases are synthetic and use explicit unsupported concepts.
# We define deterministic lexical markers for those concepts.
#
# Classification:
#
# PRESERVED_AS_TEXT
#   Unsupported lexical concept survives in a free-text CIR field
#   (objective or coverage).
#
# MAPPED_TO_SUPPORTED_FIELD
#   Unsupported requirement does not survive as unsupported text, but its
#   associated numeric value is placed into a supported numeric CIR field.
#
# DROPPED
#   Unsupported lexical concept is absent from all CIR values and no
#   associated numeric value is mapped into a supported numeric field.
#
# OTHER_AMBIGUOUS
#   Does not satisfy the above deterministic rules.
#
# This is a forensic analysis of this frozen synthetic benchmark only.
# It is NOT a general semantic-equivalence detector.
# =============================================================================

SUPPORTED_NUMERIC_FIELDS = {
    "latency_ms",
    "bandwidth_mbps",
    "reliability",
    "device_count",
}

FREE_TEXT_FIELDS = {
    "coverage",
    "objective",
}

# Lexical unsupported-concept stems deliberately correspond to the
# synthetic unsupported concepts visible in the frozen benchmark.
#
# We use stems to cover:
# quantum / teleport / magical / imaginary / fictional / impossible /
# hyperspeed / hyperspace.
#
UNSUPPORTED_STEMS = (
    "quantum",
    "teleport",
    "magic",
    "imaginar",
    "fictional",
    "impossible",
    "hyperspeed",
    "hyperspace",
)


def contains_unsupported_marker(value):
    if value is None:
        return False

    text = str(value).lower()

    return any(
        stem in text
        for stem in UNSUPPORTED_STEMS
    )


def extract_source_numbers(text):
    """
    Extract numeric literals from the source intent.

    This is used only as supporting evidence for detecting an unsupported
    source numeric value being transferred into a supported numeric CIR field.
    """
    values = []

    for token in re.findall(
        r"(?<![A-Za-z])[-+]?(?:\d+(?:\.\d+)?|\.\d+)",
        str(text)
    ):
        try:
            values.append(float(token))
        except ValueError:
            pass

    return values


def numeric_equal(a, b, tol=1e-12):
    try:
        return abs(float(a) - float(b)) <= tol
    except Exception:
        return False


def classify_case(source, cir):
    source_numbers = extract_source_numbers(source)

    text_hits = []

    for field in FREE_TEXT_FIELDS:
        value = cir.get(field)

        if contains_unsupported_marker(value):
            text_hits.append(field)

    mapped_hits = []

    for field in SUPPORTED_NUMERIC_FIELDS:
        value = cir.get(field)

        if value is None:
            continue

        for source_number in source_numbers:

            if numeric_equal(value, source_number):
                mapped_hits.append(
                    {
                        "field": field,
                        "value": value
                    }
                )
                break

    # Priority is intentional:
    #
    # If unsupported semantics remain explicitly visible in free text,
    # classify them as preserved rather than dropped.
    #
    if text_hits:
        return (
            "PRESERVED_AS_TEXT",
            text_hits,
            mapped_hits,
            source_numbers
        )

    if mapped_hits:
        return (
            "MAPPED_TO_SUPPORTED_FIELD",
            text_hits,
            mapped_hits,
            source_numbers
        )

    cir_all_text = json.dumps(
        cir,
        ensure_ascii=False
    ).lower()

    source_has_marker = contains_unsupported_marker(source)
    cir_has_marker = contains_unsupported_marker(cir_all_text)

    if source_has_marker and not cir_has_marker:
        return (
            "DROPPED",
            text_hits,
            mapped_hits,
            source_numbers
        )

    return (
        "OTHER_AMBIGUOUS",
        text_hits,
        mapped_hits,
        source_numbers
    )


# =============================================================================
# Apply deterministic forensic classifier
# =============================================================================

records = []

for _, r in unsupported.iterrows():

    cir = json.loads(str(r["cir_json"]))

    assert isinstance(cir, dict)

    (
        classification,
        text_hits,
        mapped_hits,
        source_numbers
    ) = classify_case(
        r["intent_gt"],
        cir
    )

    historical_row = hist_cir[
        hist_cir["id"] == r["id"]
    ]

    assert len(historical_row) == 1

    historical_row = historical_row.iloc[0]

    records.append(
        {
            "id": r["id"],
            "category": r["category_gt"],
            "subcategory": r["subcategory"],
            "source_intent": r["intent_gt"],
            "accepted": boolify(r["accepted"]),
            "schema_valid": boolify(r["schema_valid"]),
            "predicted_service": r["predicted_service"],
            "cir_json": r["cir_json"],
            "validation_errors": (
                None
                if pd.isna(r["validation_errors"])
                else str(r["validation_errors"])
            ),
            "classification": classification,
            "free_text_marker_fields": json.dumps(
                text_hits,
                ensure_ascii=False
            ),
            "numeric_mapping_hits": json.dumps(
                mapped_hits,
                ensure_ascii=False
            ),
            "source_numeric_literals": json.dumps(
                source_numbers
            ),
            "historical_cir_json": historical_row["cir_json"],
            "historical_accepted": boolify(
                historical_row["accepted"]
            ),
        }
    )

forensics = pd.DataFrame(records)

assert len(forensics) == 24


# =============================================================================
# Classification summary
# =============================================================================

counts = (
    forensics["classification"]
    .value_counts()
    .to_dict()
)

print("\n" + "-" * 120)
print("UNSUPPORTED-CONSTRAINT FORENSIC CLASSIFICATION")
print("-" * 120)

ORDER = [
    "DROPPED",
    "MAPPED_TO_SUPPORTED_FIELD",
    "PRESERVED_AS_TEXT",
    "OTHER_AMBIGUOUS",
]

for label in ORDER:

    n = int(counts.get(label, 0))

    print(
        f"{label:30s}: "
        f"{n:2d}/24 "
        f"({100*n/24:6.2f}%)"
    )


# =============================================================================
# Category × classification
# =============================================================================

print("\n" + "-" * 120)
print("CATEGORY × FORENSIC CLASSIFICATION")
print("-" * 120)

cross = pd.crosstab(
    forensics["category"],
    forensics["classification"]
)

print(cross.to_string())


# =============================================================================
# Full case-level evidence
# =============================================================================

print("\n" + "-" * 120)
print("CASE-LEVEL FORENSIC RESULTS")
print("-" * 120)

for _, r in forensics.iterrows():

    print("\n" + "." * 120)

    print("ID            :", r["id"])
    print("Category      :", r["category"])
    print("Classification:", r["classification"])

    print(
        "Source        :",
        r["source_intent"]
    )

    print(
        "CIR           :",
        r["cir_json"]
    )

    print(
        "Text hits     :",
        r["free_text_marker_fields"]
    )

    print(
        "Numeric maps  :",
        r["numeric_mapping_hits"]
    )

    print(
        "Accepted      :",
        r["accepted"]
    )


# =============================================================================
# Compare constrained vs historical CIR at high level
# =============================================================================

same_cir_semantic = 0
same_acceptance = 0

for _, r in forensics.iterrows():

    new_obj = json.loads(r["cir_json"])
    old_obj = json.loads(r["historical_cir_json"])

    if new_obj == old_obj:
        same_cir_semantic += 1

    if r["accepted"] == r["historical_accepted"]:
        same_acceptance += 1

print("\n" + "-" * 120)
print("CONSTRAINED vs HISTORICAL CIR — UNSUPPORTED CASES")
print("-" * 120)

print(
    f"Semantically identical CIR dictionaries: "
    f"{same_cir_semantic}/24"
)

print(
    f"Same final acceptance decision: "
    f"{same_acceptance}/24"
)


# =============================================================================
# Additional semantic-safety evidence from frozen BR12-D1 scope
# =============================================================================

joined["expected_valid_bool"] = (
    joined["expected_valid_gt"].map(boolify)
)

joined["accepted_bool"] = (
    joined["accepted"].map(boolify)
)

joined["schema_valid_bool"] = (
    joined["schema_valid"].map(boolify)
)

invalid = joined[
    ~joined["expected_valid_bool"]
].copy()

invalid_accepted = invalid[
    invalid["accepted_bool"]
].copy()

structurally_valid_wrong = joined[
    joined["schema_valid_bool"]
    &
    (
        joined["accepted_bool"]
        !=
        joined["expected_valid_bool"]
    )
].copy()

print("\n" + "-" * 120)
print("BROADER SEMANTIC-SAFETY CONTEXT")
print("-" * 120)

print(
    "Invalid requests accepted:",
    f"{len(invalid_accepted)}/100"
)

print(
    "Structurally valid but decision-wrong:",
    f"{len(structurally_valid_wrong)}/200"
)

print(
    "Unsupported requests accepted:",
    f"{int(forensics['accepted'].sum())}/24"
)


# =============================================================================
# Freeze analysis CSV
# =============================================================================

forensics.to_csv(
    OUT_CSV,
    index=False
)

csv_sha = sha256(OUT_CSV)


# =============================================================================
# Freeze summary
# =============================================================================

summary = {
    "experiment": "B-Review",
    "step": "BR12-R1",
    "analysis_type": (
        "deterministic forensic unsupported-constraint analysis"
    ),
    "benchmark": "E2A",
    "treatment": "cir_constrained",
    "model": "Qwen/Qwen3.5-9B",
    "scope": {
        "total_e2a_cases": 200,
        "unsupported_cases": 24,
        "explicit_unsupported_cases": 12,
        "implicit_unsupported_cases": 12,
    },
    "classification_definition": {
        "PRESERVED_AS_TEXT": (
            "unsupported lexical concept survives in CIR objective/coverage"
        ),
        "MAPPED_TO_SUPPORTED_FIELD": (
            "source numeric literal associated with unsupported requirement "
            "appears in a supported numeric CIR field while unsupported "
            "lexical concept is not preserved as free text"
        ),
        "DROPPED": (
            "unsupported lexical concept present in source is absent from "
            "the CIR and no associated source numeric literal is mapped "
            "into a supported numeric CIR field"
        ),
        "OTHER_AMBIGUOUS": (
            "case does not satisfy the deterministic rules above"
        ),
    },
    "unsupported_stems": list(UNSUPPORTED_STEMS),
    "classification_counts": {
        label: int(counts.get(label, 0))
        for label in ORDER
    },
    "all_unsupported_accepted": bool(
        forensics["accepted"].all()
    ),
    "same_cir_as_historical_count": int(
        same_cir_semantic
    ),
    "same_acceptance_as_historical_count": int(
        same_acceptance
    ),
    "broader_context": {
        "invalid_requests_accepted": int(
            len(invalid_accepted)
        ),
        "structurally_valid_but_decision_wrong": int(
            len(structurally_valid_wrong)
        ),
    },
    "interpretation_boundary": (
        "This deterministic lexical/numeric forensic classifier is specific "
        "to the frozen synthetic unsupported-parameter cases and is not "
        "claimed to be a general semantic-equivalence detector."
    ),
    "model_calls": 0,
    "benchmark_executions": 0,
    "input_hashes": {
        str(TREATMENT.relative_to(ROOT)): EXPECTED[TREATMENT],
        str(DATASET.relative_to(ROOT)): EXPECTED[DATASET],
        str(HIST_RESULTS.relative_to(ROOT)): EXPECTED[HIST_RESULTS],
        str(EXEC_RECORD.relative_to(ROOT)): EXPECTED[EXEC_RECORD],
    },
    "output_csv_sha256": csv_sha,
}

json_dump_frozen(
    summary,
    OUT_JSON
)

json_sha = sha256(OUT_JSON)


# =============================================================================
# Freeze manifest
# =============================================================================

manifest = {
    "experiment": "B-Review",
    "step": "BR12-R1",
    "status": "COMPLETE_FROZEN",
    "model_calls": 0,
    "benchmark_executions": 0,
    "inputs": {
        str(path.relative_to(ROOT)): expected
        for path, expected in EXPECTED.items()
    },
    "outputs": {
        str(OUT_CSV.relative_to(ROOT)): csv_sha,
        str(OUT_JSON.relative_to(ROOT)): json_sha,
    },
}

json_dump_frozen(
    manifest,
    OUT_MANIFEST
)

manifest_sha = sha256(OUT_MANIFEST)


# =============================================================================
# Persist byte-for-byte to Drive
# =============================================================================

persistent_analysis = DRIVE_ROOT / "analysis"
persistent_manifests = DRIVE_ROOT / "manifests"

persistent_analysis.mkdir(
    parents=True,
    exist_ok=True
)

persistent_manifests.mkdir(
    parents=True,
    exist_ok=True
)

copies = [
    (
        OUT_CSV,
        persistent_analysis / OUT_CSV.name
    ),
    (
        OUT_JSON,
        persistent_analysis / OUT_JSON.name
    ),
    (
        OUT_MANIFEST,
        persistent_manifests / OUT_MANIFEST.name
    ),
]

for src, dst in copies:

    assert not dst.exists(), (
        f"STOP: persistent artifact already exists:\n{dst}"
    )

    dst.write_bytes(
        src.read_bytes()
    )

    assert sha256(src) == sha256(dst)

print("\n" + "-" * 120)
print("FROZEN BR12-R1 EVIDENCE")
print("-" * 120)

print("Forensic CSV SHA-256 :", csv_sha)
print("Summary JSON SHA-256:", json_sha)
print("Manifest SHA-256    :", manifest_sha)

print("✓ BR12-R1 evidence persisted to Drive byte-for-byte")


# =============================================================================
# Final input integrity
# =============================================================================

print("\n" + "-" * 120)
print("POST-ANALYSIS FROZEN INPUT INTEGRITY")
print("-" * 120)

for path, expected in EXPECTED.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


print("\n" + "=" * 120)
print("✓ STEP BR12-R1 COMPLETE")
print("✓ 24 UNSUPPORTED-PARAMETER CASES CLASSIFIED")
print("✓ DROPPED vs MAPPED vs TEXT-PRESERVED DISTINGUISHED")
print("✓ HISTORICAL CIR COMPARISON COMPUTED")
print("✓ ANALYSIS FROZEN")
print("✓ EVIDENCE PERSISTED TO DRIVE")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTION")
print("✓ FROZEN EXPERIMENTAL RESULTS UNCHANGED")
print("✓ DO NOT RERUN BR12-R1")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE FINAL B-REVIEW FREEZE")
print("=" * 120)

EXPERIMENT B-REVIEW — STEP BR12-R1: FORENSIC UNSUPPORTED-CONSTRAINT CLASSIFICATION

------------------------------------------------------------------------------------------------------------------------
FROZEN INPUT VERIFICATION
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/results/E2A/experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv
✓ experiment-2/data/frozen/experiment2_intents_200_v1.csv
✓ experiment-2/reproducibility/experiment2a_frozen/experiment2a_results_600.csv
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR10R1_E2A_execution_record_v1.json

✓ exactly 24 unsupported-parameter cases selected
✓ 12 explicit + 12 implicit
✓ all 24 were accepted by cir_constrained

----------------------------------------------------------------------------------------------------------

In [59]:
from pathlib import Path
import hashlib
import json
import math
import shutil
from datetime import datetime, timezone

import pandas as pd

print("=" * 120)
print("EXPERIMENT B-REVIEW — STEP BR13: FINAL B-REVIEW FREEZE")
print("=" * 120)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

BR = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

PERSIST = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_review_requested_constrained_decoding"
)

FINAL_DIR = BR / "final"
MANIFEST_DIR = BR / "manifests"

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# Output files — MUST NOT ALREADY EXIST
# =============================================================================

FINAL_SUMMARY = (
    FINAL_DIR
    / "experiment_B_review_FINAL_summary_v1.json"
)

FINAL_TABLE = (
    FINAL_DIR
    / "experiment_B_review_FINAL_results_table_v1.csv"
)

FINAL_MANIFEST = (
    MANIFEST_DIR
    / "experiment_B_review_FINAL_FREEZE_v1.json"
)


for p in [
    FINAL_SUMMARY,
    FINAL_TABLE,
    FINAL_MANIFEST,
]:
    assert not p.exists(), (
        f"STOP: final artifact already exists:\n{p}\n"
        "DO NOT overwrite or rerun BR13."
    )


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def boolify(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)) and not pd.isna(v):
        if v == 1:
            return True
        if v == 0:
            return False

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(
        f"Cannot convert to bool: {v!r}"
    )


def frozen_json(obj, path):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        ) + "\n",
        encoding="utf-8"
    )


def cp_interval(k, n, alpha=0.05):
    from scipy.stats import beta

    if n == 0:
        return None, None

    lo = (
        0.0
        if k == 0
        else float(
            beta.ppf(
                alpha / 2,
                k,
                n - k + 1
            )
        )
    )

    hi = (
        1.0
        if k == n
        else float(
            beta.ppf(
                1 - alpha / 2,
                k + 1,
                n - k
            )
        )
    )

    return lo, hi


def decision_metrics(
    gt,
    accepted
):
    gt = gt.map(boolify)
    accepted = accepted.map(boolify)

    expected_invalid = ~gt
    predicted_reject = ~accepted

    TP = int(
        (
            expected_invalid
            & predicted_reject
        ).sum()
    )

    FP = int(
        (
            ~expected_invalid
            & predicted_reject
        ).sum()
    )

    FN = int(
        (
            expected_invalid
            & ~predicted_reject
        ).sum()
    )

    TN = int(
        (
            ~expected_invalid
            & ~predicted_reject
        ).sum()
    )

    n = len(gt)

    correct = int(
        (gt == accepted).sum()
    )

    valid_n = int(gt.sum())
    invalid_n = int((~gt).sum())

    valid_accept = int(
        (
            gt
            & accepted
        ).sum()
    )

    invalid_reject = int(
        (
            (~gt)
            & (~accepted)
        ).sum()
    )

    precision = (
        TP / (TP + FP)
        if (TP + FP)
        else None
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN)
        else None
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (
            precision is not None
            and recall is not None
            and (precision + recall) > 0
        )
        else None
    )

    acc_lo, acc_hi = cp_interval(
        correct,
        n
    )

    var_lo, var_hi = cp_interval(
        valid_accept,
        valid_n
    )

    irr_lo, irr_hi = cp_interval(
        invalid_reject,
        invalid_n
    )

    return {
        "n": n,

        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "correct": correct,

        "accuracy": correct / n,
        "accuracy_ci95": [
            acc_lo,
            acc_hi
        ],

        "valid_n": valid_n,
        "valid_accept": valid_accept,
        "VAR": (
            valid_accept / valid_n
        ),
        "VAR_ci95": [
            var_lo,
            var_hi
        ],

        "invalid_n": invalid_n,
        "invalid_reject": invalid_reject,
        "IRR": (
            invalid_reject / invalid_n
        ),
        "IRR_ci95": [
            irr_lo,
            irr_hi
        ],

        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# =============================================================================
# 1. Define critical frozen artifacts
# =============================================================================

PREREG = (
    BR
    / "preregistration"
    / "experiment_B_review_preregistration_v1.json"
)

ADAPTER = (
    BR
    / "implementation"
    / "experiment_B_review_structured_adapter_v1.py"
)

ORIGINAL_RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_v1.py"
)

RECOVERY_RUNNER = (
    BR
    / "implementation"
    / "run_experiment_B_review_cir_constrained_recovery_v1.py"
)

BR4_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR4_implementation_manifest_v1.json"
)

BR5_PROBE = (
    BR
    / "capability_probe"
    / "experiment_B_review_BR5_capability_probe_v1.json"
)

BR5_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR5_capability_probe_manifest_v1.json"
)

E1_DATASET = (
    ROOT
    / "data"
    / "raw"
    / "intents_50.csv"
)

E1_RESULT = (
    BR
    / "results"
    / "E1"
    / "experiment_B_review_E1_cir_constrained_50_v1.csv"
)

E1_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR7R1_E1_execution_record_v1.json"
)

E2A_DATASET = (
    ROOT
    / "experiment-2"
    / "data"
    / "frozen"
    / "experiment2_intents_200_v1.csv"
)

E2A_RESULT = (
    BR
    / "results"
    / "E2A"
    / "experiment_B_review_E2A_cir_constrained_200_BR10R1_v1.csv"
)

E2A_EXEC = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R1_E2A_execution_record_v1.json"
)

INTERRUPTED_RECORD = (
    BR
    / "manifests"
    / "experiment_B_review_BR10_interrupted_execution_record_v1.json"
)

RECOVERY_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR10R0_recovery_manifest_v1.json"
)

HIST_E2A_RESULTS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_results_600.csv"
)

HIST_E2A_METRICS = (
    ROOT
    / "experiment-2"
    / "reproducibility"
    / "experiment2a_frozen"
    / "experiment2a_primary_metrics.csv"
)

BR12_CSV = (
    BR
    / "analysis"
    / "experiment_B_review_BR12R1_unsupported_constraint_forensics_v1.csv"
)

BR12_JSON = (
    BR
    / "analysis"
    / "experiment_B_review_BR12R1_forensic_summary_v1.json"
)

BR12_MANIFEST = (
    BR
    / "manifests"
    / "experiment_B_review_BR12R1_analysis_manifest_v1.json"
)


# =============================================================================
# 2. Verify known frozen hashes
# =============================================================================

KNOWN_HASHES = {
    PREREG:
        "30554d4811bed51a0c1476514f49c0a31c9301c338bdd7ec2c2a9017b2f525ba",

    ADAPTER:
        "7d12ff7dfe671fbfcc82dbfcad242d9cdbeb114d7fc1b4aed78c887a5756bfad",

    ORIGINAL_RUNNER:
        "1549e032eaed0441be441a2a7fe1a2e0defb3977efbb7968686eef3cf6b74851",

    RECOVERY_RUNNER:
        "8b1acb6862cf967c4dd7ad75802caac9c2219efcbe42d3f6431a41c2275d0b16",

    BR4_MANIFEST:
        "c3e1a1fb75073ab2ca55c6dbae24c14bbbdb957ed4fc667883d7b425d5f9ceb8",

    BR5_PROBE:
        "d08119bff0c20c5b735c5eb1f743a65a5beb622088d565f5bfb9a8acf47fd208",

    BR5_MANIFEST:
        "0094c6c9b66546bb97ef5c7c3b3fe6a8a520a2d5c2aaf958d7e58483bb807aeb",

    E1_DATASET:
        "71093b63d60aa9ebc6f9133bf89f92ed9296fcec73e0703d6c4b58ace90f2826",

    E1_RESULT:
        "21ef335ea6218edb01b30dccc1461520767a7d1e9ea2e6956e7ba267dc29ed68",

    E1_EXEC:
        "4aaf738112b9d6b448bf7dcc068ce57735724f5f316952fe5eb0cde78803526e",

    E2A_DATASET:
        "96bcb4172dac8e9e499a9d9a4ac21354caf251a7a53ccbd470898fea02711753",

    E2A_RESULT:
        "257ff892912988b10d9abe11f24b2d746d1422d0ab06e96a39d2141b250532c6",

    E2A_EXEC:
        "418b45e0ad487fed94442f7da2b8cd1987006a6727cb3b757218923c5c8322e5",

    INTERRUPTED_RECORD:
        "21514eb08dcf63d6742acbfb4e80bb0ec7416b332b4f6b5d681fe57912cf091d",

    RECOVERY_MANIFEST:
        "6923681077b6d47a3e7862989ef660dcdef2efdefb2829336de2fcef20ae2398",

    HIST_E2A_RESULTS:
        "8c705c41b98b1f32f45fe914f34f9de16898e9e7a7e8de1e50710946356e7417",

    HIST_E2A_METRICS:
        "d9b908d86e252f3c06f362f5c677595fe85bda1ec39dbe61f2549f5bd4e3ad51",

    BR12_CSV:
        "8860bb2899d3d85ad50d271c2a75d414fc95b7e3b524fdf3f337d3e0e50b6a96",

    BR12_JSON:
        "3a42a015f363f5da86acecfa8d065613a4d67f1ff87a98de9b0248612f1f992d",

    BR12_MANIFEST:
        "c6534e57f939841782bb91198cb8c3d5ca64d471a15cdb18d34c1c0766f25ad7",
}


print("\n" + "-" * 120)
print("FROZEN PROVENANCE VERIFICATION")
print("-" * 120)

for path, expected in KNOWN_HASHES.items():

    assert path.exists(), (
        f"STOP: missing frozen artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: frozen artifact changed:\n"
        f"{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(
        "✓",
        path.relative_to(ROOT)
    )


# =============================================================================
# 3. Load frozen E1 and recompute final metrics
# =============================================================================

e1_gt = pd.read_csv(E1_DATASET)
e1_tr = pd.read_csv(E1_RESULT)

assert len(e1_gt) == 50
assert len(e1_tr) == 50

e1_gt["id"] = e1_gt["id"].astype(str)
e1_tr["id"] = e1_tr["id"].astype(str)

e1 = e1_gt.merge(
    e1_tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(e1) == 50

e1_metrics = decision_metrics(
    e1["expected_valid_gt"],
    e1["accepted"]
)

assert e1_metrics["TP"] == 22
assert e1_metrics["FP"] == 0
assert e1_metrics["FN"] == 3
assert e1_metrics["TN"] == 25

assert e1_metrics["correct"] == 47
assert e1_metrics["valid_accept"] == 25
assert e1_metrics["invalid_reject"] == 22


print("\n" + "-" * 120)
print("E1 FINAL VERIFIED METRICS")
print("-" * 120)

print(
    f"Accuracy : "
    f"{100*e1_metrics['accuracy']:.2f}%"
)

print(
    f"VAR      : "
    f"{100*e1_metrics['VAR']:.2f}%"
)

print(
    f"IRR      : "
    f"{100*e1_metrics['IRR']:.2f}%"
)

print(
    f"TP/FP/FN/TN = "
    f"{e1_metrics['TP']}/"
    f"{e1_metrics['FP']}/"
    f"{e1_metrics['FN']}/"
    f"{e1_metrics['TN']}"
)


# =============================================================================
# 4. Load frozen E2A treatment and ground truth
# =============================================================================

e2_gt = pd.read_csv(E2A_DATASET)
e2_tr = pd.read_csv(E2A_RESULT)

assert len(e2_gt) == 200
assert len(e2_tr) == 200

e2_gt["id"] = e2_gt["id"].astype(str)
e2_tr["id"] = e2_tr["id"].astype(str)

e2 = e2_gt.merge(
    e2_tr,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(e2) == 200

# Frozen benchmark metadata must agree.
for base in [
    "intent",
    "category",
    "expected_valid",
    "expected_service",
]:

    left = (
        e2[f"{base}_gt"]
        .fillna("<NA>")
        .astype(str)
    )

    right = (
        e2[f"{base}_tr"]
        .fillna("<NA>")
        .astype(str)
    )

    assert (left == right).all()


e2_metrics = decision_metrics(
    e2["expected_valid_gt"],
    e2["accepted"]
)


print("\n" + "-" * 120)
print("E2A cir_constrained FINAL VERIFIED METRICS")
print("-" * 120)

print(
    f"Accuracy : "
    f"{100*e2_metrics['accuracy']:.2f}%"
)

print(
    f"VAR      : "
    f"{100*e2_metrics['VAR']:.2f}%"
)

print(
    f"IRR      : "
    f"{100*e2_metrics['IRR']:.2f}%"
)

print(
    f"TP/FP/FN/TN = "
    f"{e2_metrics['TP']}/"
    f"{e2_metrics['FP']}/"
    f"{e2_metrics['FN']}/"
    f"{e2_metrics['TN']}"
)


# =============================================================================
# 5. Structured-generation metrics
# =============================================================================

parse_ok = (
    e2["parse_ok"]
    .map(boolify)
)

schema_valid = (
    e2["schema_valid"]
    .map(boolify)
)

parse_n = int(parse_ok.sum())
schema_n = int(schema_valid.sum())

assert parse_n == 200
assert schema_n == 185

parse_ci = cp_interval(
    parse_n,
    200
)

schema_ci = cp_interval(
    schema_n,
    200
)


print("\n" + "-" * 120)
print("E2A STRUCTURED-GENERATION RESULTS")
print("-" * 120)

print(
    f"Parse success : "
    f"{parse_n}/200 "
    f"({100*parse_n/200:.2f}%)"
)

print(
    f"Schema valid  : "
    f"{schema_n}/200 "
    f"({100*schema_n/200:.2f}%)"
)


# =============================================================================
# 6. Historical E2A CIR control
# =============================================================================

hist = pd.read_csv(
    HIST_E2A_RESULTS
)

hist["id"] = hist["id"].astype(str)

hist_cir = hist[
    hist["pipeline"].astype(str) == "cir"
].copy()

assert len(hist_cir) == 200
assert hist_cir["id"].nunique() == 200

hist_join = e2_gt.merge(
    hist_cir,
    on="id",
    how="inner",
    suffixes=("_gt", "_hist"),
    validate="one_to_one"
)

assert len(hist_join) == 200

hist_metrics = decision_metrics(
    hist_join["expected_valid_gt"],
    hist_join["accepted"]
)

# Historical frozen CIR figures.
assert hist_metrics["correct"] == 151
assert hist_metrics["valid_accept"] == 96
assert hist_metrics["invalid_reject"] == 55

assert hist_metrics["TP"] == 55
assert hist_metrics["FP"] == 4
assert hist_metrics["FN"] == 45
assert hist_metrics["TN"] == 96


print("\n" + "-" * 120)
print("HISTORICAL E2A CIR CONTROL")
print("-" * 120)

print(
    f"Accuracy : "
    f"{100*hist_metrics['accuracy']:.2f}%"
)

print(
    f"VAR      : "
    f"{100*hist_metrics['VAR']:.2f}%"
)

print(
    f"IRR      : "
    f"{100*hist_metrics['IRR']:.2f}%"
)


# =============================================================================
# 7. E2A paired comparison
# =============================================================================

e2["expected_valid_bool"] = (
    e2["expected_valid_gt"]
    .map(boolify)
)

e2["accepted_bool"] = (
    e2["accepted"]
    .map(boolify)
)

e2["treatment_correct"] = (
    e2["expected_valid_bool"]
    ==
    e2["accepted_bool"]
)


hist_join["expected_valid_bool"] = (
    hist_join["expected_valid_gt"]
    .map(boolify)
)

hist_join["accepted_bool"] = (
    hist_join["accepted"]
    .map(boolify)
)

hist_join["historical_correct"] = (
    hist_join["expected_valid_bool"]
    ==
    hist_join["accepted_bool"]
)


paired = (
    e2[
        [
            "id",
            "treatment_correct"
        ]
    ]
    .merge(
        hist_join[
            [
                "id",
                "historical_correct"
            ]
        ],
        on="id",
        validate="one_to_one"
    )
)

both_correct = int(
    (
        paired["treatment_correct"]
        &
        paired["historical_correct"]
    ).sum()
)

constrained_only = int(
    (
        paired["treatment_correct"]
        &
        ~paired["historical_correct"]
    ).sum()
)

historical_only = int(
    (
        ~paired["treatment_correct"]
        &
        paired["historical_correct"]
    ).sum()
)

both_wrong = int(
    (
        ~paired["treatment_correct"]
        &
        ~paired["historical_correct"]
    ).sum()
)

assert (
    both_correct
    + constrained_only
    + historical_only
    + both_wrong
    == 200
)


from scipy.stats import binomtest

discordant = (
    constrained_only
    + historical_only
)

if discordant:
    mcnemar_p = float(
        binomtest(
            constrained_only,
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    )
else:
    mcnemar_p = 1.0


print("\n" + "-" * 120)
print("PAIRED E2A COMPARISON")
print("-" * 120)

print(
    "Both correct        :",
    both_correct
)

print(
    "Constrained only    :",
    constrained_only
)

print(
    "Historical CIR only :",
    historical_only
)

print(
    "Both wrong          :",
    both_wrong
)

print(
    "Exact McNemar p     :",
    f"{mcnemar_p:.12g}"
)


# =============================================================================
# 8. BR12 frozen forensic results
# =============================================================================

br12 = json.loads(
    BR12_JSON.read_text(
        encoding="utf-8"
    )
)

class_counts = (
    br12["classification_counts"]
)

assert (
    class_counts["DROPPED"]
    == 17
)

assert (
    class_counts["MAPPED_TO_SUPPORTED_FIELD"]
    == 3
)

assert (
    class_counts["PRESERVED_AS_TEXT"]
    == 4
)

assert (
    class_counts["OTHER_AMBIGUOUS"]
    == 0
)

assert (
    br12["all_unsupported_accepted"]
    is True
)

assert (
    br12["same_cir_as_historical_count"]
    == 20
)

assert (
    br12["same_acceptance_as_historical_count"]
    == 24
)


print("\n" + "-" * 120)
print("BR12 FORENSIC RESULT")
print("-" * 120)

print(
    "Unsupported cases accepted : 24/24"
)

print(
    "Dropped                    : 17/24"
)

print(
    "Mapped to supported field  : 3/24"
)

print(
    "Preserved as text          : 4/24"
)

print(
    "Other/ambiguous            : 0/24"
)


# =============================================================================
# 9. Broader semantic-safety evidence
# =============================================================================

invalid_accepted = int(
    (
        (~e2["expected_valid_bool"])
        &
        e2["accepted_bool"]
    ).sum()
)

struct_valid_wrong = int(
    (
        schema_valid
        &
        (
            e2["expected_valid_bool"]
            !=
            e2["accepted_bool"]
        )
    ).sum()
)

assert invalid_accepted == 47
assert struct_valid_wrong == 54


print("\n" + "-" * 120)
print("SEMANTIC-SAFETY EVIDENCE")
print("-" * 120)

print(
    "Invalid requests accepted              :",
    f"{invalid_accepted}/100"
)

print(
    "Structurally valid but decision-wrong   :",
    f"{struct_valid_wrong}/200"
)

print(
    "Unsupported requests accepted           :",
    "24/24"
)


# =============================================================================
# 10. Build final result table
# =============================================================================

rows = [
    {
        "benchmark": "E1",
        "configuration": "cir_constrained",
        "n": 50,
        "accuracy": e1_metrics["accuracy"],
        "VAR": e1_metrics["VAR"],
        "IRR": e1_metrics["IRR"],
        "parse_success": float(
            e1["parse_ok"].map(boolify).mean()
        ),
        "schema_valid_rate": float(
            e1["schema_valid"].map(boolify).mean()
        ),
        "TP": e1_metrics["TP"],
        "FP": e1_metrics["FP"],
        "FN": e1_metrics["FN"],
        "TN": e1_metrics["TN"],
    },

    {
        "benchmark": "E2A",
        "configuration": "historical_cir",
        "n": 200,
        "accuracy": hist_metrics["accuracy"],
        "VAR": hist_metrics["VAR"],
        "IRR": hist_metrics["IRR"],
        "parse_success": None,
        "schema_valid_rate": None,
        "TP": hist_metrics["TP"],
        "FP": hist_metrics["FP"],
        "FN": hist_metrics["FN"],
        "TN": hist_metrics["TN"],
    },

    {
        "benchmark": "E2A",
        "configuration": "cir_constrained",
        "n": 200,
        "accuracy": e2_metrics["accuracy"],
        "VAR": e2_metrics["VAR"],
        "IRR": e2_metrics["IRR"],
        "parse_success": parse_n / 200,
        "schema_valid_rate": schema_n / 200,
        "TP": e2_metrics["TP"],
        "FP": e2_metrics["FP"],
        "FN": e2_metrics["FN"],
        "TN": e2_metrics["TN"],
    },
]

table = pd.DataFrame(rows)

table.to_csv(
    FINAL_TABLE,
    index=False
)

table_sha = sha256(
    FINAL_TABLE
)


# =============================================================================
# 11. Final summary
# =============================================================================

summary = {
    "experiment": "B-Review",

    "purpose": (
        "Reviewer-requested schema-constrained decoding "
        "comparison on frozen E1 and E2A benchmarks."
    ),

    "status": "COMPLETE_AND_FROZEN",

    "model": "Qwen/Qwen3.5-9B",

    "treatment": "cir_constrained",

    "scientific_change": (
        "Provider-side structural JSON-schema constrained "
        "generation only; historical CIR prompt, model, "
        "temperature and downstream validation/mapping/"
        "descriptor path retained."
    ),

    "model_running_experiments_remaining": 0,

    "E1": {
        "cases": 50,
        "accuracy": e1_metrics["accuracy"],
        "accuracy_ci95": e1_metrics[
            "accuracy_ci95"
        ],
        "VAR": e1_metrics["VAR"],
        "VAR_ci95": e1_metrics[
            "VAR_ci95"
        ],
        "IRR": e1_metrics["IRR"],
        "IRR_ci95": e1_metrics[
            "IRR_ci95"
        ],
        "TP": e1_metrics["TP"],
        "FP": e1_metrics["FP"],
        "FN": e1_metrics["FN"],
        "TN": e1_metrics["TN"],
    },

    "E2A_historical_CIR": {
        "cases": 200,
        "accuracy": hist_metrics["accuracy"],
        "VAR": hist_metrics["VAR"],
        "IRR": hist_metrics["IRR"],
        "TP": hist_metrics["TP"],
        "FP": hist_metrics["FP"],
        "FN": hist_metrics["FN"],
        "TN": hist_metrics["TN"],
    },

    "E2A_cir_constrained": {
        "cases": 200,
        "accuracy": e2_metrics["accuracy"],
        "accuracy_ci95": e2_metrics[
            "accuracy_ci95"
        ],
        "VAR": e2_metrics["VAR"],
        "VAR_ci95": e2_metrics[
            "VAR_ci95"
        ],
        "IRR": e2_metrics["IRR"],
        "IRR_ci95": e2_metrics[
            "IRR_ci95"
        ],
        "TP": e2_metrics["TP"],
        "FP": e2_metrics["FP"],
        "FN": e2_metrics["FN"],
        "TN": e2_metrics["TN"],
        "parse_success": (
            parse_n / 200
        ),
        "schema_valid_rate": (
            schema_n / 200
        ),
    },

    "E2A_delta_constrained_minus_historical": {
        "accuracy_pp": (
            100
            * (
                e2_metrics["accuracy"]
                -
                hist_metrics["accuracy"]
            )
        ),

        "VAR_pp": (
            100
            * (
                e2_metrics["VAR"]
                -
                hist_metrics["VAR"]
            )
        ),

        "IRR_pp": (
            100
            * (
                e2_metrics["IRR"]
                -
                hist_metrics["IRR"]
            )
        ),
    },

    "paired_E2A": {
        "both_correct": both_correct,
        "constrained_only_correct":
            constrained_only,
        "historical_only_correct":
            historical_only,
        "both_wrong": both_wrong,
        "discordant_pairs": discordant,
        "exact_mcnemar_p": mcnemar_p,
    },

    "unsupported_constraint_forensics": {
        "cases": 24,
        "accepted": 24,
        "dropped": 17,
        "mapped_to_supported_field": 3,
        "preserved_as_text": 4,
        "other_ambiguous": 0,
        "same_cir_as_historical": 20,
        "same_acceptance_as_historical": 24,
    },

    "semantic_safety": {
        "invalid_requests_accepted":
            invalid_accepted,

        "invalid_requests_total": 100,

        "structurally_valid_but_decision_wrong":
            struct_valid_wrong,

        "total_cases": 200,
    },

    "interruption_provenance": {
        "original_BR10_status":
            "INTERRUPTED_PROVIDER_HTTP_503",

        "completed_progress_markers_before_interruption":
            79,

        "partial_result_artifact_survived":
            False,

        "lost_in_memory_outputs_reconstructed":
            False,

        "replacement_run":
            "BR10-R1",

        "replacement_cases":
            200,

        "replacement_transient_retry_events":
            0,

        "scientific_treatment_changed":
            False,
    },

    "paper_safe_interpretation": (
        "On the frozen E2A benchmark, structural constrained "
        "decoding produced fully parseable outputs but did not "
        "improve decision accuracy relative to the historical "
        "CIR control and did not eliminate semantic omission, "
        "mis-mapping, or unsupported-requirement acceptance. "
        "These results support retaining deterministic semantic "
        "validation beyond provider-side structural decoding."
    ),

    "claim_boundaries": [
        (
            "Do not claim that constrained decoding universally "
            "fails; results apply to the tested model, prompt, "
            "schema, and frozen benchmarks."
        ),
        (
            "Do not claim hallucination-free behavior."
        ),
        (
            "Do not claim formal standards conformance."
        ),
        (
            "Do not interpret the deterministic BR12 lexical/"
            "numeric classifier as a general semantic-equivalence "
            "detector."
        ),
    ],

    "model_calls_in_BR13": 0,

    "benchmark_executions_in_BR13": 0,
}

frozen_json(
    summary,
    FINAL_SUMMARY
)

summary_sha = sha256(
    FINAL_SUMMARY
)


# =============================================================================
# 12. Final freeze manifest
# =============================================================================

manifest = {
    "experiment": "B-Review",

    "freeze": "FINAL",

    "version": 1,

    "status":
        "COMPLETE_FROZEN_DO_NOT_RERUN",

    "model_calls_in_freeze": 0,

    "benchmark_executions_in_freeze": 0,

    "frozen_inputs": {
        str(
            p.relative_to(ROOT)
        ): h
        for p, h in KNOWN_HASHES.items()
    },

    "final_outputs": {
        str(
            FINAL_TABLE.relative_to(ROOT)
        ): table_sha,

        str(
            FINAL_SUMMARY.relative_to(ROOT)
        ): summary_sha,
    },

    "scientific_components_unchanged":
        True,

    "historical_arms_rerun":
        False,

    "E1_rerun_after_success":
        False,

    "E2A_rerun_after_success":
        False,

    "BR12_rerun":
        False,

    "notes": [
        (
            "BR7 initial E1 launch failed during Python import "
            "before benchmark execution; corrected execution "
            "used PYTHONPATH only."
        ),
        (
            "Original BR10 E2A attempt was interrupted by "
            "provider HTTP 503 after 79 progress markers; "
            "no partial result artifact survived."
        ),
        (
            "BR10-R1 was the frozen replacement 200-case "
            "E2A execution. Lost in-memory BR10 outputs were "
            "not reconstructed."
        ),
        (
            "BR12-R1 forensic analysis was deterministic and "
            "performed with zero model calls."
        ),
    ],
}

frozen_json(
    manifest,
    FINAL_MANIFEST
)

manifest_sha = sha256(
    FINAL_MANIFEST
)


# =============================================================================
# 13. Persist final artifacts byte-for-byte
# =============================================================================

P_FINAL = PERSIST / "final"
P_MANIFEST = PERSIST / "manifests"

P_FINAL.mkdir(
    parents=True,
    exist_ok=True
)

P_MANIFEST.mkdir(
    parents=True,
    exist_ok=True
)

copies = [
    (
        FINAL_TABLE,
        P_FINAL / FINAL_TABLE.name
    ),
    (
        FINAL_SUMMARY,
        P_FINAL / FINAL_SUMMARY.name
    ),
    (
        FINAL_MANIFEST,
        P_MANIFEST / FINAL_MANIFEST.name
    ),
]

for src, dst in copies:

    assert not dst.exists(), (
        f"STOP: persistent final artifact "
        f"already exists:\n{dst}"
    )

    shutil.copyfile(
        src,
        dst
    )

    assert (
        sha256(src)
        ==
        sha256(dst)
    )


print("\n" + "-" * 120)
print("FINAL B-REVIEW ARTIFACTS")
print("-" * 120)

print(
    "Results table SHA-256 :",
    table_sha
)

print(
    "Summary SHA-256       :",
    summary_sha
)

print(
    "FINAL FREEZE SHA-256  :",
    manifest_sha
)

print(
    "✓ final artifacts persisted "
    "to Drive byte-for-byte"
)


# =============================================================================
# 14. Final frozen-input integrity
# =============================================================================

print("\n" + "-" * 120)
print("FINAL INPUT INTEGRITY")
print("-" * 120)

for path, expected in KNOWN_HASHES.items():

    actual = sha256(path)

    assert actual == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


# =============================================================================
# 15. Final result display
# =============================================================================

print("\n" + "-" * 120)
print("FINAL B-REVIEW RESULTS")
print("-" * 120)

print(
    f"E1 cir_constrained  : "
    f"ACC={100*e1_metrics['accuracy']:.2f}% | "
    f"VAR={100*e1_metrics['VAR']:.2f}% | "
    f"IRR={100*e1_metrics['IRR']:.2f}%"
)

print(
    f"E2A historical CIR  : "
    f"ACC={100*hist_metrics['accuracy']:.2f}% | "
    f"VAR={100*hist_metrics['VAR']:.2f}% | "
    f"IRR={100*hist_metrics['IRR']:.2f}%"
)

print(
    f"E2A cir_constrained : "
    f"ACC={100*e2_metrics['accuracy']:.2f}% | "
    f"VAR={100*e2_metrics['VAR']:.2f}% | "
    f"IRR={100*e2_metrics['IRR']:.2f}%"
)

print(
    f"E2A parse success   : "
    f"{parse_n}/200 "
    f"({100*parse_n/200:.2f}%)"
)

print(
    f"E2A schema valid    : "
    f"{schema_n}/200 "
    f"({100*schema_n/200:.2f}%)"
)

print(
    f"McNemar exact p     : "
    f"{mcnemar_p:.12g}"
)

print(
    "Unsupported accepted:",
    "24/24"
)

print(
    "Unsupported forensic:",
    "17 dropped | "
    "3 mapped | "
    "4 text-preserved"
)


print("\n" + "=" * 120)
print("✓ STEP BR13 COMPLETE")
print("✓ B-REVIEW FINAL FREEZE COMPLETE")
print("✓ E1 FINAL EVIDENCE VERIFIED")
print("✓ E2A FINAL EVIDENCE VERIFIED")
print("✓ HISTORICAL CIR CONTROL VERIFIED")
print("✓ PAIRED COMPARISON RECOMPUTED FROM FROZEN EVIDENCE")
print("✓ BR12 FORENSIC RESULTS VERIFIED")
print("✓ INTERRUPTION/RECOVERY PROVENANCE RETAINED")
print("✓ FINAL RESULTS TABLE FROZEN")
print("✓ FINAL SUMMARY FROZEN")
print("✓ FINAL FREEZE MANIFEST FROZEN")
print("✓ FINAL EVIDENCE PERSISTED TO DRIVE")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ ALL PLANNED B-REVIEW MODEL-RUNNING EXPERIMENTS COMPLETE")
print("✓ DO NOT RERUN BR13")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE CONTINUING")
print("=" * 120)

EXPERIMENT B-REVIEW — STEP BR13: FINAL B-REVIEW FREEZE

------------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE VERIFICATION
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/preregistration/experiment_B_review_preregistration_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/experiment_B_review_structured_adapter_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/implementation/run_experiment_B_review_cir_constrained_recovery_v1.py
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_BR4_implementation_manife

In [60]:
from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C1: READ-ONLY A / B-REVIEW / B-E3 EVIDENCE INVENTORY")
print("=" * 120)

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

DRIVE_BASE = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent"
)

A = (
    ROOT
    / "journal_experiments"
    / "experiment_A_model_sweep"
)

BREVIEW = (
    ROOT
    / "journal_experiments"
    / "experiment_B_review_requested_constrained_decoding"
)

BE3 = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

CONSOLIDATION = (
    ROOT
    / "journal_experiments"
    / "journal_final_evidence"
)

C1_DIR = CONSOLIDATION / "inventory"

C1_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUT = (
    C1_DIR
    / "C1_frozen_evidence_inventory_v1.json"
)

assert not OUT.exists(), (
    f"STOP: C1 inventory already exists:\n{OUT}\n"
    "DO NOT overwrite or rerun C1."
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def describe(path):
    result = {
        "path": str(path),
        "exists": path.exists(),
    }

    if path.exists() and path.is_file():
        result["size_bytes"] = path.stat().st_size
        result["sha256"] = sha256(path)

    return result


def find_named_files(base, patterns):
    found = []

    if not base.exists():
        return found

    for pattern in patterns:
        for p in sorted(base.rglob(pattern)):
            if p.is_file():
                found.append(
                    {
                        "path": str(p),
                        "relative_path": str(
                            p.relative_to(ROOT)
                        ),
                        "size_bytes": p.stat().st_size,
                        "sha256": sha256(p),
                    }
                )

    # Remove duplicates caused by overlapping patterns.
    unique = {}

    for item in found:
        unique[item["path"]] = item

    return list(unique.values())


# =============================================================================
# A — authoritative final freeze
# =============================================================================

A_FINAL = (
    A
    / "manifests"
    / "experiment_A_FINAL_FREEZE_v1.json"
)

EXPECTED_A_FINAL_SHA = (
    "667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a"
)

print("\n" + "-" * 120)
print("EXPERIMENT A")
print("-" * 120)

assert A.exists(), (
    f"STOP: Experiment A directory missing:\n{A}"
)

assert A_FINAL.exists(), (
    f"STOP: Experiment A FINAL FREEZE missing:\n{A_FINAL}"
)

a_final_sha = sha256(A_FINAL)

assert a_final_sha == EXPECTED_A_FINAL_SHA, (
    "STOP: Experiment A FINAL FREEZE hash mismatch.\n"
    f"Expected: {EXPECTED_A_FINAL_SHA}\n"
    f"Actual:   {a_final_sha}"
)

print("✓ Experiment A directory found")
print("✓ Experiment A FINAL FREEZE verified")
print("  SHA-256:", a_final_sha)

a_files = find_named_files(
    A,
    [
        "*FINAL*",
        "*final*",
        "*metrics*",
        "*summary*",
        "*manifest*",
        "*.csv",
    ]
)

print(
    "Indexed Experiment A evidence files:",
    len(a_files)
)


# =============================================================================
# B-Review — authoritative BR13 final freeze
# =============================================================================

BREVIEW_FINAL = (
    BREVIEW
    / "manifests"
    / "experiment_B_review_FINAL_FREEZE_v1.json"
)

BREVIEW_TABLE = (
    BREVIEW
    / "final"
    / "experiment_B_review_FINAL_results_table_v1.csv"
)

BREVIEW_SUMMARY = (
    BREVIEW
    / "final"
    / "experiment_B_review_FINAL_summary_v1.json"
)

EXPECTED_BREVIEW_FREEZE_SHA = (
    "63fa99c81477eb7d2c3136b67f72ac3d4673f7d081d7b332ef8d634c68faf3dd"
)

EXPECTED_BREVIEW_TABLE_SHA = (
    "4d362b0b0862cf79b0acd189c843ce854a4ce2da64bfd5f95091ba98b1d3bf89"
)

EXPECTED_BREVIEW_SUMMARY_SHA = (
    "a3a9539f8dade42a4f75ec4f28d4a9421937edbcb79d41091c3f7d374d22c8eb"
)

print("\n" + "-" * 120)
print("EXPERIMENT B-REVIEW")
print("-" * 120)

assert BREVIEW.exists(), (
    f"STOP: B-Review directory missing:\n{BREVIEW}"
)

checks = [
    (
        BREVIEW_FINAL,
        EXPECTED_BREVIEW_FREEZE_SHA,
        "FINAL FREEZE"
    ),
    (
        BREVIEW_TABLE,
        EXPECTED_BREVIEW_TABLE_SHA,
        "FINAL results table"
    ),
    (
        BREVIEW_SUMMARY,
        EXPECTED_BREVIEW_SUMMARY_SHA,
        "FINAL summary"
    ),
]

for path, expected, label in checks:

    assert path.exists(), (
        f"STOP: B-Review {label} missing:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: B-Review {label} hash mismatch.\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print(
        f"✓ {label}: {actual}"
    )

breview_files = find_named_files(
    BREVIEW,
    [
        "*FINAL*",
        "*BR12R1*",
        "*BR10R1*",
        "*BR7R1*",
        "*interrupted*",
        "*recovery*",
        "*.csv",
        "*.json",
    ]
)

print(
    "Indexed B-Review evidence files:",
    len(breview_files)
)


# =============================================================================
# B-E3 — separate supplementary experiment
# =============================================================================

print("\n" + "-" * 120)
print("EXPERIMENT B-E3 — SEPARATE SUPPLEMENTARY FOLLOW-UP")
print("-" * 120)

assert BE3.exists(), (
    f"STOP: B-E3 directory missing:\n{BE3}"
)

# Known frozen artifacts from completed B-E3 execution.
BE3_PREREG = (
    BE3
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

BE3_TREATMENT = (
    BE3
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

BE3_RUNNER = (
    BE3
    / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

BE3_RESULT = (
    BE3
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

BE3_EXEC = (
    BE3
    / "manifests"
    / "experiment_B_B9B_execution_record_v1.json"
)

KNOWN_BE3_HASHES = {
    BE3_PREREG:
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    BE3_TREATMENT:
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    BE3_RUNNER:
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    BE3_RESULT:
        "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1",

    BE3_EXEC:
        "4654ea991d50f65089f5be68d5bd565004a8bb768f5e106072fd6c50facfd7d5",
}

be3_known = {}

for path, expected in KNOWN_BE3_HASHES.items():

    assert path.exists(), (
        f"STOP: known B-E3 artifact missing:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: B-E3 frozen artifact changed:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    be3_known[str(path.relative_to(ROOT))] = {
        "sha256": actual,
        "verified": True,
    }

    print(
        "✓",
        path.relative_to(ROOT)
    )


# =============================================================================
# Search for existing B-E3 analysis/final artifacts.
#
# C1 MUST NOT assume they exist.
# =============================================================================

be3_analysis_files = find_named_files(
    BE3,
    [
        "*analysis*",
        "*metrics*",
        "*summary*",
        "*FINAL*",
        "*final*",
        "*manifest*",
    ]
)

print(
    "\nIndexed B-E3 analysis/final candidate files:",
    len(be3_analysis_files)
)

for item in be3_analysis_files:
    print(
        "  ",
        item["relative_path"],
        item["sha256"]
    )


# =============================================================================
# Detect whether B-E3 already has an explicit final freeze.
# =============================================================================

be3_final_candidates = [
    item
    for item in be3_analysis_files
    if (
        "final" in item["relative_path"].lower()
        and "freeze" in item["relative_path"].lower()
    )
]

BE3_FINALIZED = (
    len(be3_final_candidates) > 0
)

if BE3_FINALIZED:
    print(
        "\n⚠ B-E3 final-freeze candidate detected."
    )
    print(
        "C2 must inspect it before deciding whether "
        "any new freeze artifact is necessary."
    )
else:
    print(
        "\n✓ No B-E3 FINAL FREEZE detected."
    )
    print(
        "✓ C2 will perform read-only B-E3 closure."
    )


# =============================================================================
# Drive persistence visibility
# =============================================================================

print("\n" + "-" * 120)
print("PERSISTENT DRIVE VISIBILITY")
print("-" * 120)

A_DRIVE = (
    DRIVE_BASE
    / "experiment_A_model_sweep"
)

BREVIEW_DRIVE = (
    DRIVE_BASE
    / "experiment_B_review_requested_constrained_decoding"
)

BE3_DRIVE = (
    DRIVE_BASE
    / "experiment_B_schema_constrained_decoding"
)

drive_status = {
    "A": A_DRIVE.exists(),
    "B_review": BREVIEW_DRIVE.exists(),
    "B_E3": BE3_DRIVE.exists(),
}

for key, value in drive_status.items():
    print(
        f"{key:10s}:",
        "FOUND" if value else "NOT FOUND"
    )


# =============================================================================
# Consolidation classification
# =============================================================================

classification = {
    "Experiment_A": {
        "role": (
            "Primary model-variation robustness experiment"
        ),
        "status": "FINAL_FROZEN",
        "authoritative_freeze": str(
            A_FINAL.relative_to(ROOT)
        ),
        "authoritative_freeze_sha256":
            a_final_sha,
    },

    "Experiment_B_Review": {
        "role": (
            "Primary reviewer-requested "
            "schema-constrained-decoding experiment"
        ),
        "status": "FINAL_FROZEN",
        "authoritative_freeze": str(
            BREVIEW_FINAL.relative_to(ROOT)
        ),
        "authoritative_freeze_sha256":
            EXPECTED_BREVIEW_FREEZE_SHA,
    },

    "Experiment_B_E3": {
        "role": (
            "Separate supplementary constrained-decoding "
            "follow-up on frozen E3-v3 120-case benchmark"
        ),
        "status": (
            "FINAL_FREEZE_CANDIDATE_PRESENT"
            if BE3_FINALIZED
            else "EXECUTION_COMPLETE_ANALYTIC_CLOSURE_PENDING"
        ),
        "must_not_be_presented_as_reviewer_requested_B":
            True,
        "model_calls_remaining": 0,
    },
}


# =============================================================================
# Build immutable C1 inventory
# =============================================================================

inventory = {
    "step": "C1",

    "purpose": (
        "Read-only inventory before final journal "
        "evidence consolidation."
    ),

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "experimental_phase": {
        "new_model_running_experiments_allowed":
            False,

        "planned_model_runs_remaining":
            0,

        "Experiment_A":
            "COMPLETE_FROZEN",

        "Experiment_B_Review":
            "COMPLETE_FROZEN",

        "Experiment_B_E3":
            (
                "ANALYTIC_CLOSURE_PENDING"
                if not BE3_FINALIZED
                else
                "FINAL_FREEZE_CANDIDATE_REQUIRES_INSPECTION"
            ),
    },

    "classification": classification,

    "Experiment_A": {
        "final_freeze": describe(
            A_FINAL
        ),
        "indexed_files": a_files,
    },

    "Experiment_B_Review": {
        "final_freeze": describe(
            BREVIEW_FINAL
        ),
        "final_results_table": describe(
            BREVIEW_TABLE
        ),
        "final_summary": describe(
            BREVIEW_SUMMARY
        ),
        "indexed_files": breview_files,
    },

    "Experiment_B_E3": {
        "known_verified_artifacts":
            be3_known,

        "analysis_final_candidates":
            be3_analysis_files,

        "explicit_final_freeze_detected":
            BE3_FINALIZED,

        "final_freeze_candidates":
            be3_final_candidates,
    },

    "persistent_drive_visibility":
        drive_status,

    "C1_actions": {
        "model_calls": 0,
        "benchmark_executions": 0,
        "existing_experimental_files_modified": 0,
        "existing_results_modified": 0,
        "new_file_created":
            str(OUT.relative_to(ROOT)),
    },

    "next_step": (
        "C2_READ_ONLY_B_E3_CLOSURE"
    ),
}


OUT.write_text(
    json.dumps(
        inventory,
        indent=2,
        sort_keys=True,
        ensure_ascii=False
    ) + "\n",
    encoding="utf-8"
)

inventory_sha = sha256(OUT)


# =============================================================================
# Display authoritative consolidated status
# =============================================================================

print("\n" + "-" * 120)
print("CONSOLIDATION STATUS")
print("-" * 120)

print(
    "Experiment A        : FINAL FROZEN"
)

print(
    "Experiment B-Review : FINAL FROZEN"
)

print(
    "Experiment B-E3     :",
    (
        "FINAL FREEZE CANDIDATE PRESENT — INSPECT IN C2"
        if BE3_FINALIZED
        else
        "EXECUTION COMPLETE — ANALYTIC CLOSURE PENDING"
    )
)

print(
    "Future model calls  : 0"
)

print(
    "Future benchmark runs: 0"
)


print("\n" + "-" * 120)
print("C1 INVENTORY")
print("-" * 120)

print(
    "Path:",
    OUT
)

print(
    "SHA-256:",
    inventory_sha
)


# =============================================================================
# Final integrity check of authoritative A/B artifacts
# =============================================================================

assert (
    sha256(A_FINAL)
    ==
    EXPECTED_A_FINAL_SHA
)

assert (
    sha256(BREVIEW_FINAL)
    ==
    EXPECTED_BREVIEW_FREEZE_SHA
)

assert (
    sha256(BREVIEW_TABLE)
    ==
    EXPECTED_BREVIEW_TABLE_SHA
)

assert (
    sha256(BREVIEW_SUMMARY)
    ==
    EXPECTED_BREVIEW_SUMMARY_SHA
)

for path, expected in KNOWN_BE3_HASHES.items():
    assert sha256(path) == expected


print("\n" + "=" * 120)
print("✓ STEP C1 COMPLETE")
print("✓ EXPERIMENT A FINAL FREEZE VERIFIED")
print("✓ B-REVIEW BR13 FINAL FREEZE VERIFIED")
print("✓ B-E3 EXECUTION EVIDENCE VERIFIED")
print("✓ A / B-REVIEW / B-E3 ROLES SEPARATED")
print("✓ B-E3 NOT RECLASSIFIED AS REVIEWER-REQUESTED B")
print("✓ PERSISTENT DRIVE VISIBILITY INSPECTED")
print("✓ CONSOLIDATION INVENTORY CREATED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ ZERO EXISTING EXPERIMENTAL ARTIFACTS MODIFIED")
print("✓ NO MORE MODEL-RUNNING EXPERIMENTS ARE PLANNED")
print("✓ NEXT: C2 — READ-ONLY B-E3 ANALYTIC CLOSURE")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE C2")
print("=" * 120)

JOURNAL CONSOLIDATION — STEP C1: READ-ONLY A / B-REVIEW / B-E3 EVIDENCE INVENTORY

------------------------------------------------------------------------------------------------------------------------
EXPERIMENT A
------------------------------------------------------------------------------------------------------------------------
✓ Experiment A directory found
✓ Experiment A FINAL FREEZE verified
  SHA-256: 667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a
Indexed Experiment A evidence files: 10

------------------------------------------------------------------------------------------------------------------------
EXPERIMENT B-REVIEW
------------------------------------------------------------------------------------------------------------------------
✓ FINAL FREEZE: 63fa99c81477eb7d2c3136b67f72ac3d4673f7d081d7b332ef8d634c68faf3dd
✓ FINAL results table: 4d362b0b0862cf79b0acd189c843ce854a4ce2da64bfd5f95091ba98b1d3bf89
✓ FINAL summary: a3a9539f8dade42a4f75ec4f28d4a

In [61]:
from pathlib import Path
import ast
import hashlib
import json
import shutil
import pandas as pd
from scipy.stats import beta

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C2: B-E3 READ-ONLY ANALYTIC CLOSURE AND FINAL FREEZE")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

BE3 = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

PERSIST = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

FINAL_DIR = BE3 / "final"
ANALYSIS_DIR = BE3 / "analysis"
MANIFEST_DIR = BE3 / "manifests"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Frozen inputs
# =============================================================================

PREREG = (
    BE3
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

TREATMENT_ADAPTER = (
    BE3
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

HARNESS = (
    BE3
    / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

RAW_RESULT = (
    BE3
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

EXEC_RECORD = (
    BE3
    / "manifests"
    / "experiment_B_B9B_execution_record_v1.json"
)

HIST_ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

HIST_RESULT = (
    ROOT
    / "experiment-2b"
    / "results"
    / "e3_v3"
    / "experiment2b_e3v3_r7_dev_v1_results.csv"
)

# We do NOT require the historical result at this exact path.
# It will be discovered below if the canonical path differs.


# =============================================================================
# Outputs
# =============================================================================

ANALYSIS_CSV = (
    ANALYSIS_DIR
    / "experiment_B_E3_final_case_analysis_v1.csv"
)

FINAL_SUMMARY = (
    FINAL_DIR
    / "experiment_B_E3_FINAL_summary_v1.json"
)

FINAL_TABLE = (
    FINAL_DIR
    / "experiment_B_E3_FINAL_results_table_v1.csv"
)

FINAL_FREEZE = (
    MANIFEST_DIR
    / "experiment_B_E3_FINAL_FREEZE_v1.json"
)

for p in [
    ANALYSIS_CSV,
    FINAL_SUMMARY,
    FINAL_TABLE,
    FINAL_FREEZE,
]:
    assert not p.exists(), (
        f"STOP: C2 output already exists:\n{p}\n"
        "DO NOT overwrite or rerun C2."
    )


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def boolify(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)) and not pd.isna(v):
        if v == 1:
            return True
        if v == 0:
            return False

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(f"Cannot convert to bool: {v!r}")


def cp_interval(k, n, alpha=0.05):
    if n == 0:
        return None, None

    lo = (
        0.0
        if k == 0
        else float(beta.ppf(alpha / 2, k, n - k + 1))
    )

    hi = (
        1.0
        if k == n
        else float(beta.ppf(1 - alpha / 2, k + 1, n - k))
    )

    return lo, hi


def frozen_json(obj, path):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        ) + "\n",
        encoding="utf-8"
    )


def extract_function_ast(path, function_name):
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ):
            segment = ast.get_source_segment(source, node)

            assert segment is not None

            # Normalize only outer whitespace/newline.
            return segment.strip()

    raise AssertionError(
        f"STOP: function {function_name!r} not found in {path}"
    )


# =============================================================================
# Verify known frozen B-E3 hashes
# =============================================================================

KNOWN = {
    PREREG:
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    TREATMENT_ADAPTER:
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    HARNESS:
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    RAW_RESULT:
        "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1",

    EXEC_RECORD:
        "4654ea991d50f65089f5be68d5bd565004a8bb768f5e106072fd6c50facfd7d5",

    HIST_ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    BENCHMARK:
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",
}

print("\n" + "-" * 120)
print("FROZEN B-E3 PROVENANCE")
print("-" * 120)

for path, expected in KNOWN.items():

    assert path.exists(), (
        f"STOP: missing frozen artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: hash mismatch:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# IMPORTANT: raw B-E3 artifact has .json suffix but CSV content
# =============================================================================

raw = pd.read_csv(RAW_RESULT)
gt = pd.read_csv(BENCHMARK)

assert len(raw) == 120, (
    f"STOP: expected 120 B-E3 rows, found {len(raw)}"
)

assert len(gt) == 120, (
    f"STOP: expected 120 benchmark rows, found {len(gt)}"
)

assert raw["id"].nunique() == 120
assert gt["id"].nunique() == 120

raw["id"] = raw["id"].astype(str)
gt["id"] = gt["id"].astype(str)

assert set(raw["id"]) == set(gt["id"])

print("\n✓ B-E3 raw artifact correctly parsed as CSV despite .json suffix")
print("✓ 120 treatment observations verified")
print("✓ 120 frozen benchmark cases verified")


# =============================================================================
# Join ground truth
# =============================================================================

joined = gt.merge(
    raw,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 120


# Resolve expected_valid safely.
if "expected_valid_gt" in joined.columns:
    expected_col = "expected_valid_gt"
elif "expected_valid" in joined.columns:
    expected_col = "expected_valid"
else:
    raise AssertionError(
        "STOP: expected_valid field not found after join."
    )


# Resolve accepted.
assert "accepted" in joined.columns, (
    "STOP: treatment result does not contain accepted."
)

joined["expected_valid_bool"] = joined[expected_col].map(boolify)
joined["accepted_bool"] = joined["accepted"].map(boolify)

assert int(joined["expected_valid_bool"].sum()) == 58
assert int((~joined["expected_valid_bool"]).sum()) == 62


# =============================================================================
# Decision metrics
# =============================================================================

valid = joined["expected_valid_bool"]
accepted = joined["accepted_bool"]

TP = int(((~valid) & (~accepted)).sum())
FP = int((valid & (~accepted)).sum())
FN = int(((~valid) & accepted).sum())
TN = int((valid & accepted).sum())

correct = int((valid == accepted).sum())

assert (TP, FP, FN, TN) == (50, 7, 12, 51)
assert correct == 101

accuracy = correct / 120
VAR = TN / 58
IRR = TP / 62

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

acc_ci = cp_interval(correct, 120)
var_ci = cp_interval(TN, 58)
irr_ci = cp_interval(TP, 62)

print("\n" + "-" * 120)
print("B-E3 DECISION METRICS")
print("-" * 120)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")
print()
print(f"Accuracy : {accuracy:.6f} ({100*accuracy:.2f}%)")
print(f"VAR      : {VAR:.6f} ({100*VAR:.2f}%)")
print(f"IRR      : {IRR:.6f} ({100*IRR:.2f}%)")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1       : {f1:.6f}")

print(
    "Accuracy CP95:",
    f"[{acc_ci[0]:.6f}, {acc_ci[1]:.6f}]"
)

print(
    "VAR CP95     :",
    f"[{var_ci[0]:.6f}, {var_ci[1]:.6f}]"
)

print(
    "IRR CP95     :",
    f"[{irr_ci[0]:.6f}, {irr_ci[1]:.6f}]"
)


# =============================================================================
# Parse / schema / invocation observations
# =============================================================================

def count_true_if_present(df, col):
    if col not in df.columns:
        return None

    return int(df[col].map(boolify).sum())


parse_ok_n = count_true_if_present(joined, "parse_ok")
schema_valid_n = count_true_if_present(joined, "schema_valid")

if "llm_invoked" in joined.columns:
    llm_invoked_n = int(
        joined["llm_invoked"].map(boolify).sum()
    )
else:
    llm_invoked_n = None


print("\n" + "-" * 120)
print("B-E3 STRUCTURED-GENERATION OBSERVATIONS")
print("-" * 120)

print("Parse OK       :", parse_ok_n)
print("Schema valid   :", schema_valid_n)
print("LLM invoked    :", llm_invoked_n)


# =============================================================================
# Retrospective historical-cleaner equivalence check
#
# This closes the caveat transparently.
# It is explicitly RETROSPECTIVE and was NOT a pre-run verification.
# =============================================================================

print("\n" + "-" * 120)
print("RETROSPECTIVE CLEANER EQUIVALENCE CHECK")
print("-" * 120)

historical_cleaner = extract_function_ast(
    HIST_ADAPTER,
    "_clean_json_text"
)

treatment_cleaner = extract_function_ast(
    TREATMENT_ADAPTER,
    "_clean_json_text"
)

cleaner_exact_text_equal = (
    historical_cleaner
    ==
    treatment_cleaner
)

# AST-equivalence ignores formatting while preserving executable structure.
historical_tree = ast.dump(
    ast.parse(historical_cleaner),
    include_attributes=False
)

treatment_tree = ast.dump(
    ast.parse(treatment_cleaner),
    include_attributes=False
)

cleaner_ast_equal = (
    historical_tree
    ==
    treatment_tree
)

print(
    "Exact extracted source equality:",
    cleaner_exact_text_equal
)

print(
    "AST structural equality         :",
    cleaner_ast_equal
)

assert cleaner_ast_equal, (
    "STOP: treatment cleaner is NOT AST-equivalent "
    "to historical cleaner.\n"
    "Do not finalize B-E3; inspect the difference."
)

print("✓ historical/treatment cleaner executable structure equivalent")
print("✓ equivalence verification classified as RETROSPECTIVE")
print("✓ no claim that this was checked before B-E3 execution")


# =============================================================================
# Case-level analysis artifact
# =============================================================================

analysis_cols = [
    "id",
    "expected_valid_bool",
    "accepted_bool",
]

for candidate in [
    "category_gt",
    "category",
    "predicted_service",
    "parse_ok",
    "schema_valid",
    "rejection_stage",
    "validation_errors",
    "error_codes_json",
    "cir_json",
]:
    if candidate in joined.columns:
        analysis_cols.append(candidate)

case_analysis = joined[analysis_cols].copy()

case_analysis["decision_correct"] = (
    case_analysis["expected_valid_bool"]
    ==
    case_analysis["accepted_bool"]
)

case_analysis.to_csv(
    ANALYSIS_CSV,
    index=False
)

analysis_sha = sha256(ANALYSIS_CSV)


# =============================================================================
# Final results table
# =============================================================================

table = pd.DataFrame(
    [
        {
            "experiment": "B-E3",
            "benchmark": "E3-v3",
            "configuration": "structured_constrained_treatment_B1",
            "n": 120,
            "expected_valid_n": 58,
            "expected_invalid_n": 62,
            "correct": correct,
            "accuracy": accuracy,
            "accuracy_ci95_low": acc_ci[0],
            "accuracy_ci95_high": acc_ci[1],
            "VAR": VAR,
            "VAR_ci95_low": var_ci[0],
            "VAR_ci95_high": var_ci[1],
            "IRR": IRR,
            "IRR_ci95_low": irr_ci[0],
            "IRR_ci95_high": irr_ci[1],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "TN": TN,
            "parse_ok_n": parse_ok_n,
            "schema_valid_n": schema_valid_n,
            "llm_invoked_n": llm_invoked_n,
        }
    ]
)

table.to_csv(
    FINAL_TABLE,
    index=False
)

table_sha = sha256(FINAL_TABLE)


# =============================================================================
# Final B-E3 summary
# =============================================================================

summary = {
    "experiment": "B-E3",

    "status": "FINAL_FROZEN",

    "role": (
        "Separate supplementary constrained-decoding follow-up "
        "on the frozen E3-v3 120-case benchmark."
    ),

    "reviewer_requested_B": False,

    "must_not_be_presented_as_reviewer_requested_B": True,

    "model": "Qwen/Qwen3.5-9B",

    "benchmark": {
        "name": "E3-v3 held-out benchmark",
        "n": 120,
        "expected_valid": 58,
        "expected_invalid": 62,
        "sha256": KNOWN[BENCHMARK],
    },

    "treatment_result": {
        "raw_path": str(RAW_RESULT.relative_to(ROOT)),
        "raw_file_format": (
            "CSV content despite .json filename extension"
        ),
        "sha256": KNOWN[RAW_RESULT],
    },

    "decision_metrics": {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "correct": correct,

        "accuracy": accuracy,
        "accuracy_ci95": list(acc_ci),

        "VAR": VAR,
        "VAR_ci95": list(var_ci),

        "IRR": IRR,
        "IRR_ci95": list(irr_ci),

        "precision": precision,
        "recall": recall,
        "f1": f1,
    },

    "structured_generation": {
        "parse_ok_n": parse_ok_n,
        "schema_valid_n": schema_valid_n,
        "llm_invoked_n": llm_invoked_n,
    },

    "cleaner_equivalence": {
        "verification_timing": "RETROSPECTIVE_POST_EXECUTION",
        "exact_extracted_source_equal":
            cleaner_exact_text_equal,
        "AST_equivalent":
            cleaner_ast_equal,
        "historical_adapter_sha256":
            KNOWN[HIST_ADAPTER],
        "treatment_adapter_sha256":
            KNOWN[TREATMENT_ADAPTER],
        "claim_boundary": (
            "Cleaner equivalence was verified retrospectively "
            "during analytic closure and must not be described "
            "as a pre-execution control."
        ),
    },

    "provenance_note": (
        "The exact reviewer-requested Experiment B procedure "
        "(E1 and E2A constrained-decoding comparison) was "
        "identified after this B-E3 execution. B-E3 is therefore "
        "retained only as a separate supplementary follow-up and "
        "is not evidence that the original reviewer-requested "
        "Experiment B procedure had already been completed."
    ),

    "paper_safe_interpretation": (
        "On the frozen 120-case E3-v3 benchmark, the structured "
        "constrained treatment produced the same aggregate "
        "decision accuracy, valid-acceptance rate, and "
        "invalid-rejection rate as the previously recorded "
        "historical E3-v3 aggregate baseline. This supplementary "
        "result does not establish universal equivalence and "
        "does not replace the reviewer-requested E1/E2A "
        "constrained-decoding experiment."
    ),

    "claim_boundaries": [
        "Do not present B-E3 as the reviewer-requested Experiment B.",
        "Do not claim universal constrained-decoding equivalence.",
        "Do not claim hallucination-free behavior.",
        "Do not claim formal standards conformance.",
        (
            "Do not describe retrospective cleaner equivalence "
            "verification as preregistered or pre-execution."
        ),
    ],

    "model_calls_in_C2": 0,
    "benchmark_executions_in_C2": 0,
}

frozen_json(
    summary,
    FINAL_SUMMARY
)

summary_sha = sha256(FINAL_SUMMARY)


# =============================================================================
# Final B-E3 freeze manifest
# =============================================================================

freeze = {
    "experiment": "B-E3",
    "freeze": "FINAL",
    "version": 1,

    "status":
        "COMPLETE_FROZEN_DO_NOT_RERUN",

    "role":
        "SUPPLEMENTARY_NOT_REVIEWER_REQUESTED_B",

    "model_calls_in_C2": 0,
    "benchmark_executions_in_C2": 0,

    "frozen_inputs": {
        str(path.relative_to(ROOT)): expected
        for path, expected in KNOWN.items()
    },

    "final_outputs": {
        str(ANALYSIS_CSV.relative_to(ROOT)):
            analysis_sha,

        str(FINAL_TABLE.relative_to(ROOT)):
            table_sha,

        str(FINAL_SUMMARY.relative_to(ROOT)):
            summary_sha,
    },

    "methodological_notes": [
        (
            "B-E3 was executed before the exact E1/E2A "
            "reviewer-requested B procedure was retrieved."
        ),
        (
            "B-E3 is retained as a separate supplementary "
            "follow-up."
        ),
        (
            "Historical cleaner equivalence was checked "
            "retrospectively during C2."
        ),
        (
            "No B-E3 model calls or benchmark executions "
            "were performed during C2."
        ),
    ],
}

frozen_json(
    freeze,
    FINAL_FREEZE
)

freeze_sha = sha256(FINAL_FREEZE)


# =============================================================================
# Persist final closure evidence to Drive
# =============================================================================

P_FINAL = PERSIST / "final"
P_ANALYSIS = PERSIST / "analysis"
P_MANIFEST = PERSIST / "manifests"

P_FINAL.mkdir(parents=True, exist_ok=True)
P_ANALYSIS.mkdir(parents=True, exist_ok=True)
P_MANIFEST.mkdir(parents=True, exist_ok=True)

copies = [
    (
        ANALYSIS_CSV,
        P_ANALYSIS / ANALYSIS_CSV.name
    ),
    (
        FINAL_TABLE,
        P_FINAL / FINAL_TABLE.name
    ),
    (
        FINAL_SUMMARY,
        P_FINAL / FINAL_SUMMARY.name
    ),
    (
        FINAL_FREEZE,
        P_MANIFEST / FINAL_FREEZE.name
    ),
]

for src, dst in copies:

    assert not dst.exists(), (
        f"STOP: persistent C2 artifact already exists:\n{dst}"
    )

    shutil.copyfile(src, dst)

    assert sha256(src) == sha256(dst)


# =============================================================================
# Final input integrity
# =============================================================================

print("\n" + "-" * 120)
print("FINAL FROZEN INPUT INTEGRITY")
print("-" * 120)

for path, expected in KNOWN.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


# =============================================================================
# Final display
# =============================================================================

print("\n" + "-" * 120)
print("B-E3 FINAL CLOSURE")
print("-" * 120)

print(
    f"Accuracy : {100*accuracy:.2f}%"
)

print(
    f"VAR      : {100*VAR:.2f}%"
)

print(
    f"IRR      : {100*IRR:.2f}%"
)

print(
    f"TP/FP/FN/TN = {TP}/{FP}/{FN}/{TN}"
)

print(
    "Cleaner AST equivalent:",
    cleaner_ast_equal
)

print(
    "Cleaner check timing  : RETROSPECTIVE"
)

print(
    "Role                  : SUPPLEMENTARY"
)

print(
    "Reviewer-requested B  : NO"
)


print("\n" + "-" * 120)
print("FINAL B-E3 ARTIFACTS")
print("-" * 120)

print(
    "Case analysis SHA-256:",
    analysis_sha
)

print(
    "Results table SHA-256:",
    table_sha
)

print(
    "Summary SHA-256      :",
    summary_sha
)

print(
    "FINAL FREEZE SHA-256 :",
    freeze_sha
)


print("\n" + "=" * 120)
print("✓ STEP C2 COMPLETE")
print("✓ B-E3 120-CASE RESULT VERIFIED READ-ONLY")
print("✓ B-E3 DECISION METRICS RECOMPUTED")
print("✓ EXACT BINOMIAL CONFIDENCE INTERVALS COMPUTED")
print("✓ HISTORICAL CLEANER EQUIVALENCE CHECKED RETROSPECTIVELY")
print("✓ B-E3 ROLE FIXED AS SUPPLEMENTARY")
print("✓ B-E3 NOT PRESENTED AS REVIEWER-REQUESTED EXPERIMENT B")
print("✓ B-E3 FINAL RESULTS TABLE FROZEN")
print("✓ B-E3 FINAL SUMMARY FROZEN")
print("✓ B-E3 FINAL FREEZE MANIFEST FROZEN")
print("✓ FINAL EVIDENCE PERSISTED TO DRIVE")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ DO NOT RERUN C2")
print("✓ NEXT: C3 — MASTER A + B EVIDENCE CONSOLIDATION")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE C3")
print("=" * 120)

JOURNAL CONSOLIDATION — STEP C2: B-E3 READ-ONLY ANALYTIC CLOSURE AND FINAL FREEZE

------------------------------------------------------------------------------------------------------------------------
FROZEN B-E3 PROVENANCE
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_schema_constrained_decoding/preregistration/experiment_B_preregistration_v1.json
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/experiment_B_structured_adapter_v1.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/run_experiment_B_structured_treatment_v1.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/results/treatment_B1/experiment_B_B1_structured_treatment_120_v1.json
✓ journal_experiments/experiment_B_schema_constrained_decoding/manifests/experiment_B_B9B_execution_record_v1.json
✓ src/llm/together_adapter.py
✓ experiment-2b/repr

ValueError: Cannot convert to bool: nan

In [62]:
from pathlib import Path
import hashlib
import pandas as pd

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C2-D1: READ-ONLY B-E3 STRUCTURED-FIELD DIAGNOSIS")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

BE3 = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

RAW_RESULT = (
    BE3
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

EXPECTED_RAW_SHA = (
    "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1"
)

EXPECTED_BENCHMARK_SHA = (
    "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de"
)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


# =============================================================================
# Verify frozen evidence
# =============================================================================

assert RAW_RESULT.exists()
assert BENCHMARK.exists()

assert sha256(RAW_RESULT) == EXPECTED_RAW_SHA
assert sha256(BENCHMARK) == EXPECTED_BENCHMARK_SHA

print("\n✓ frozen B-E3 result verified")
print("✓ frozen E3-v3 benchmark verified")


# =============================================================================
# Read raw result
# =============================================================================

raw = pd.read_csv(RAW_RESULT)
gt = pd.read_csv(BENCHMARK)

assert len(raw) == 120
assert len(gt) == 120

print("\n" + "-" * 120)
print("RAW RESULT SCHEMA")
print("-" * 120)

for i, col in enumerate(raw.columns, 1):
    print(f"{i:2d}. {col}")


# =============================================================================
# Inspect relevant structured-generation fields
# =============================================================================

candidate_cols = [
    "llm_invoked",
    "parse_ok",
    "schema_valid",
    "accepted",
    "rejection_stage",
    "finish_reason",
    "raw_response",
    "cir_json",
    "error_codes_json",
    "validation_errors",
]

present = [c for c in candidate_cols if c in raw.columns]

print("\n" + "-" * 120)
print("RELEVANT FIELDS PRESENT")
print("-" * 120)

for col in present:
    print("✓", col)


# =============================================================================
# Value distributions including NaN
# =============================================================================

print("\n" + "-" * 120)
print("VALUE DISTRIBUTIONS — NaN INCLUDED")
print("-" * 120)

for col in [
    "llm_invoked",
    "parse_ok",
    "schema_valid",
    "accepted",
    "rejection_stage",
    "finish_reason",
]:
    if col not in raw.columns:
        print(f"\n{col}: NOT PRESENT")
        continue

    print(f"\n{col}:")
    print(
        raw[col]
        .value_counts(dropna=False)
        .to_string()
    )


# =============================================================================
# Null counts
# =============================================================================

print("\n" + "-" * 120)
print("NULL COUNTS")
print("-" * 120)

for col in present:
    null_n = int(raw[col].isna().sum())
    nonnull_n = int(raw[col].notna().sum())

    print(
        f"{col:25s} "
        f"null={null_n:3d} "
        f"non-null={nonnull_n:3d}"
    )


# =============================================================================
# Cross-tab parse/schema against llm_invoked if available
# =============================================================================

if "llm_invoked" in raw.columns:

    if "parse_ok" in raw.columns:
        print("\n" + "-" * 120)
        print("LLM_INVOKED × PARSE_OK")
        print("-" * 120)

        print(
            pd.crosstab(
                raw["llm_invoked"],
                raw["parse_ok"],
                dropna=False
            ).to_string()
        )

    if "schema_valid" in raw.columns:
        print("\n" + "-" * 120)
        print("LLM_INVOKED × SCHEMA_VALID")
        print("-" * 120)

        print(
            pd.crosstab(
                raw["llm_invoked"],
                raw["schema_valid"],
                dropna=False
            ).to_string()
        )


# =============================================================================
# Inspect rows where parse_ok is NaN
# =============================================================================

if "parse_ok" in raw.columns:

    parse_nan = raw[raw["parse_ok"].isna()].copy()

    print("\n" + "-" * 120)
    print("ROWS WITH parse_ok = NaN")
    print("-" * 120)

    print("Count:", len(parse_nan))

    display_cols = [
        c for c in [
            "id",
            "llm_invoked",
            "parse_ok",
            "schema_valid",
            "accepted",
            "rejection_stage",
            "finish_reason",
        ]
        if c in raw.columns
    ]

    if len(parse_nan):
        print(
            parse_nan[display_cols]
            .to_string(index=False)
        )


# =============================================================================
# Inspect rows where schema_valid is NaN
# =============================================================================

if "schema_valid" in raw.columns:

    schema_nan = raw[
        raw["schema_valid"].isna()
    ].copy()

    print("\n" + "-" * 120)
    print("ROWS WITH schema_valid = NaN")
    print("-" * 120)

    print("Count:", len(schema_nan))

    display_cols = [
        c for c in [
            "id",
            "llm_invoked",
            "parse_ok",
            "schema_valid",
            "accepted",
            "rejection_stage",
        ]
        if c in raw.columns
    ]

    if len(schema_nan):
        print(
            schema_nan[display_cols]
            .to_string(index=False)
        )


# =============================================================================
# Check expected historical pattern without altering data
# =============================================================================

print("\n" + "-" * 120)
print("READ-ONLY COUNT SUMMARY")
print("-" * 120)

if "llm_invoked" in raw.columns:
    print(
        "llm_invoked non-null:",
        int(raw["llm_invoked"].notna().sum())
    )

if "parse_ok" in raw.columns:
    print(
        "parse_ok non-null:",
        int(raw["parse_ok"].notna().sum())
    )

    print(
        "parse_ok null:",
        int(raw["parse_ok"].isna().sum())
    )

if "schema_valid" in raw.columns:
    print(
        "schema_valid non-null:",
        int(raw["schema_valid"].notna().sum())
    )

    print(
        "schema_valid null:",
        int(raw["schema_valid"].isna().sum())
    )


# =============================================================================
# Confirm failed C2 did not create final artifacts
# =============================================================================

possible_outputs = [
    BE3 / "analysis" / "experiment_B_E3_final_case_analysis_v1.csv",
    BE3 / "final" / "experiment_B_E3_FINAL_results_table_v1.csv",
    BE3 / "final" / "experiment_B_E3_FINAL_summary_v1.json",
    BE3 / "manifests" / "experiment_B_E3_FINAL_FREEZE_v1.json",
]

print("\n" + "-" * 120)
print("FAILED C2 OUTPUT STATUS")
print("-" * 120)

any_created = False

for p in possible_outputs:
    exists = p.exists()
    any_created |= exists

    print(
        ("EXISTS " if exists else "absent "),
        p.relative_to(ROOT)
    )

if any_created:
    print(
        "\n⚠ STOP: at least one C2 output exists."
    )
    print(
        "Do NOT delete or overwrite it. "
        "We will inspect it before recovery."
    )
else:
    print(
        "\n✓ failed C2 created no final closure artifacts"
    )


# =============================================================================
# Frozen input integrity
# =============================================================================

assert sha256(RAW_RESULT) == EXPECTED_RAW_SHA
assert sha256(BENCHMARK) == EXPECTED_BENCHMARK_SHA

print("\n" + "=" * 120)
print("✓ STEP C2-D1 COMPLETE")
print("✓ B-E3 STRUCTURED FIELDS INSPECTED READ-ONLY")
print("✓ NaN REPRESENTATION QUANTIFIED")
print("✓ parse_ok / schema_valid / llm_invoked RELATIONSHIP INSPECTED")
print("✓ FAILED C2 OUTPUT STATUS INSPECTED")
print("✓ FROZEN RESULT UNCHANGED")
print("✓ FROZEN BENCHMARK UNCHANGED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ NO FILE MODIFIED")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE C2-R1")
print("=" * 120)

JOURNAL CONSOLIDATION — STEP C2-D1: READ-ONLY B-E3 STRUCTURED-FIELD DIAGNOSIS

✓ frozen B-E3 result verified
✓ frozen E3-v3 benchmark verified

------------------------------------------------------------------------------------------------------------------------
RAW RESULT SCHEMA
------------------------------------------------------------------------------------------------------------------------
 1. id
 2. intent
 3. family_id
 4. repair_id
 5. subcategory
 6. difficulty
 7. linguistic_form
 8. constraint_type
 9. control_type
10. notes
11. expected_valid
12. expected_service
13. pipeline
14. accepted
15. decision_correct
16. llm_invoked
17. parse_ok
18. llm_latency_ms
19. end_to_end_latency_ms
20. finish_reason
21. rejection_stage
22. error_codes_json
23. cir_json
24. input_audit_json
25. pre_guard_json
26. post_guard_json
27. cir_before_reconciliation_json
28. cir_after_reconciliation_json
29. reconciliation_json
30. cir_before_r5_json
31. cir_after_r5_json
32. r5_reliability_ch

In [63]:
from pathlib import Path
import ast
import hashlib
import json
import shutil
import pandas as pd
from scipy.stats import beta

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C2-R1: B-E3 ANALYTIC CLOSURE AND FINAL FREEZE")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

BE3 = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

PERSIST = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)

PREREG = (
    BE3 / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

TREATMENT_ADAPTER = (
    BE3 / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

HARNESS = (
    BE3 / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

RAW_RESULT = (
    BE3 / "results" / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

EXEC_RECORD = (
    BE3 / "manifests"
    / "experiment_B_B9B_execution_record_v1.json"
)

HIST_ADAPTER = (
    ROOT / "src" / "llm" / "together_adapter.py"
)

BENCHMARK = (
    ROOT / "experiment-2b" / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

ANALYSIS_DIR = BE3 / "analysis"
FINAL_DIR = BE3 / "final"
MANIFEST_DIR = BE3 / "manifests"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_CSV = (
    ANALYSIS_DIR
    / "experiment_B_E3_final_case_analysis_v1.csv"
)

FINAL_TABLE = (
    FINAL_DIR
    / "experiment_B_E3_FINAL_results_table_v1.csv"
)

FINAL_SUMMARY = (
    FINAL_DIR
    / "experiment_B_E3_FINAL_summary_v1.json"
)

FINAL_FREEZE = (
    MANIFEST_DIR
    / "experiment_B_E3_FINAL_FREEZE_v1.json"
)


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def boolify_required(v):
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)) and not pd.isna(v):
        if v == 1:
            return True
        if v == 0:
            return False

    s = str(v).strip().lower()

    if s in {"true", "1", "yes"}:
        return True

    if s in {"false", "0", "no"}:
        return False

    raise ValueError(
        f"Cannot convert required boolean: {v!r}"
    )


def cp_interval(k, n, alpha=0.05):
    assert n > 0

    lo = (
        0.0
        if k == 0
        else float(
            beta.ppf(
                alpha / 2,
                k,
                n - k + 1
            )
        )
    )

    hi = (
        1.0
        if k == n
        else float(
            beta.ppf(
                1 - alpha / 2,
                k + 1,
                n - k
            )
        )
    )

    return lo, hi


def write_json(obj, path):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        ) + "\n",
        encoding="utf-8"
    )


def extract_function_ast(path, name):
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if (
            isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            )
            and node.name == name
        ):
            segment = ast.get_source_segment(
                source,
                node
            )

            assert segment is not None

            return segment.strip()

    raise AssertionError(
        f"STOP: {name} not found in {path}"
    )


# =============================================================================
# Recovery gate
# =============================================================================

print("\n" + "-" * 120)
print("C2-R1 RECOVERY GATE")
print("-" * 120)

for p in [
    ANALYSIS_CSV,
    FINAL_TABLE,
    FINAL_SUMMARY,
    FINAL_FREEZE,
]:
    assert not p.exists(), (
        f"STOP: C2 final artifact already exists:\n{p}\n"
        "Do not overwrite it."
    )

print("✓ failed C2 produced no final closure artifacts")
print("✓ C2-R1 may proceed")


# =============================================================================
# Verify frozen inputs
# =============================================================================

KNOWN = {
    PREREG:
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    TREATMENT_ADAPTER:
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    HARNESS:
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    RAW_RESULT:
        "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1",

    EXEC_RECORD:
        "4654ea991d50f65089f5be68d5bd565004a8bb768f5e106072fd6c50facfd7d5",

    HIST_ADAPTER:
        "b11b082775cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    BENCHMARK:
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",
}

# Correct historical adapter SHA from frozen provenance.
KNOWN[HIST_ADAPTER] = (
    "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3"
)

print("\n" + "-" * 120)
print("FROZEN PROVENANCE")
print("-" * 120)

for path, expected in KNOWN.items():

    assert path.exists(), (
        f"STOP: missing artifact:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: hash mismatch:\n{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# Load frozen evidence
# =============================================================================

# IMPORTANT:
# RAW_RESULT has CSV content despite .json suffix.
raw = pd.read_csv(RAW_RESULT)
gt = pd.read_csv(BENCHMARK)

assert len(raw) == 120
assert len(gt) == 120

assert raw["id"].nunique() == 120
assert gt["id"].nunique() == 120

raw["id"] = raw["id"].astype(str)
gt["id"] = gt["id"].astype(str)

assert set(raw["id"]) == set(gt["id"])

joined = gt.merge(
    raw,
    on="id",
    how="inner",
    suffixes=("_gt", "_tr"),
    validate="one_to_one"
)

assert len(joined) == 120

print("\n✓ 120 B-E3 observations loaded")
print("✓ one-to-one benchmark join verified")


# =============================================================================
# Ground truth + decision metrics
# =============================================================================

expected_col = (
    "expected_valid_gt"
    if "expected_valid_gt" in joined.columns
    else "expected_valid"
)

assert expected_col in joined.columns
assert "accepted" in joined.columns

joined["expected_valid_bool"] = (
    joined[expected_col]
    .map(boolify_required)
)

joined["accepted_bool"] = (
    joined["accepted"]
    .map(boolify_required)
)

valid = joined["expected_valid_bool"]
accepted = joined["accepted_bool"]

assert int(valid.sum()) == 58
assert int((~valid).sum()) == 62

TP = int(((~valid) & (~accepted)).sum())
FP = int((valid & (~accepted)).sum())
FN = int(((~valid) & accepted).sum())
TN = int((valid & accepted).sum())

correct = int((valid == accepted).sum())

assert (TP, FP, FN, TN) == (50, 7, 12, 51)
assert correct == 101

accuracy = correct / 120
VAR = TN / 58
IRR = TP / 62

precision = TP / (TP + FP)
recall = TP / (TP + FN)

f1 = (
    2 * precision * recall
    / (precision + recall)
)

acc_ci = cp_interval(correct, 120)
var_ci = cp_interval(TN, 58)
irr_ci = cp_interval(TP, 62)

print("\n" + "-" * 120)
print("RECOMPUTED B-E3 DECISION RESULTS")
print("-" * 120)

print(f"TP = {TP}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TN = {TN}")
print()
print(f"Accuracy : {accuracy:.6f} ({accuracy*100:.2f}%)")
print(f"VAR      : {VAR:.6f} ({VAR*100:.2f}%)")
print(f"IRR      : {IRR:.6f} ({IRR*100:.2f}%)")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1       : {f1:.6f}")

print(
    "Accuracy CP95:",
    f"[{acc_ci[0]:.6f}, {acc_ci[1]:.6f}]"
)

print(
    "VAR CP95     :",
    f"[{var_ci[0]:.6f}, {var_ci[1]:.6f}]"
)

print(
    "IRR CP95     :",
    f"[{irr_ci[0]:.6f}, {irr_ci[1]:.6f}]"
)


# =============================================================================
# Correct structured-generation interpretation
# =============================================================================

assert "llm_invoked" in raw.columns
assert "parse_ok" in raw.columns

llm_invoked = (
    raw["llm_invoked"]
    .map(boolify_required)
)

llm_invoked_n = int(llm_invoked.sum())
pre_llm_n = int((~llm_invoked).sum())

assert llm_invoked_n == 77
assert pre_llm_n == 43

invoked_rows = raw[llm_invoked].copy()
not_invoked_rows = raw[~llm_invoked].copy()

# For invoked rows, parse_ok MUST be populated.
assert invoked_rows["parse_ok"].notna().all()

invoked_parse = (
    invoked_rows["parse_ok"]
    .map(boolify_required)
)

parse_ok_n = int(invoked_parse.sum())
parse_fail_n = int((~invoked_parse).sum())

assert parse_ok_n == 77
assert parse_fail_n == 0

# For non-invoked rows, parse_ok MUST be N/A/NaN.
assert not_invoked_rows["parse_ok"].isna().all()

# schema_valid is not part of this frozen B-E3 schema.
assert "schema_valid" not in raw.columns

print("\n" + "-" * 120)
print("STRUCTURED-GENERATION INTERPRETATION")
print("-" * 120)

print(
    f"LLM invoked       : {llm_invoked_n}/120"
)

print(
    f"Pre-LLM/no-call   : {pre_llm_n}/120"
)

print(
    f"Parse success     : {parse_ok_n}/{llm_invoked_n} "
    f"({100*parse_ok_n/llm_invoked_n:.2f}% of invoked cases)"
)

print(
    f"Parse failures    : {parse_fail_n}/{llm_invoked_n}"
)

print(
    "schema_valid      : NOT AVAILABLE IN FROZEN B-E3 RESULT SCHEMA"
)

print(
    "✓ 43 parse_ok NaNs classified as N/A because llm_invoked=False"
)

print(
    "✓ NaN values were NOT converted into parse failures"
)


# =============================================================================
# Rejection-stage distribution
# =============================================================================

rejection_counts = {}

if "rejection_stage" in raw.columns:

    vc = raw["rejection_stage"].value_counts(
        dropna=False
    )

    for key, value in vc.items():

        label = (
            "accepted_no_rejection_stage"
            if pd.isna(key)
            else str(key)
        )

        rejection_counts[label] = int(value)

print("\n" + "-" * 120)
print("REJECTION-STAGE DISTRIBUTION")
print("-" * 120)

for k, v in rejection_counts.items():
    print(f"{k:32s}: {v}")


# =============================================================================
# Retrospective cleaner equivalence
# =============================================================================

print("\n" + "-" * 120)
print("RETROSPECTIVE CLEANER EQUIVALENCE")
print("-" * 120)

historical_cleaner = extract_function_ast(
    HIST_ADAPTER,
    "_clean_json_text"
)

treatment_cleaner = extract_function_ast(
    TREATMENT_ADAPTER,
    "_clean_json_text"
)

exact_source_equal = (
    historical_cleaner
    ==
    treatment_cleaner
)

hist_ast = ast.dump(
    ast.parse(historical_cleaner),
    include_attributes=False
)

treat_ast = ast.dump(
    ast.parse(treatment_cleaner),
    include_attributes=False
)

ast_equal = (
    hist_ast
    ==
    treat_ast
)

print(
    "Exact extracted source equality:",
    exact_source_equal
)

print(
    "AST executable-structure equality:",
    ast_equal
)

assert ast_equal, (
    "STOP: historical and B-E3 cleaners are not "
    "AST-equivalent. Do not freeze B-E3."
)

print("✓ cleaner executable structure equivalent")
print("✓ verification timing = RETROSPECTIVE POST-EXECUTION")


# =============================================================================
# Case-level closure artifact
# =============================================================================

case_analysis = pd.DataFrame({
    "id": joined["id"],
    "expected_valid":
        joined["expected_valid_bool"],
    "accepted":
        joined["accepted_bool"],
})

case_analysis["decision_correct"] = (
    case_analysis["expected_valid"]
    ==
    case_analysis["accepted"]
)

# Add useful frozen evidence columns without inventing missing fields.
for col in [
    "llm_invoked",
    "parse_ok",
    "finish_reason",
    "rejection_stage",
    "error_codes_json",
    "cir_json",
]:

    if col in raw.columns:

        mapping = (
            raw
            .set_index("id")[col]
        )

        case_analysis[col] = (
            case_analysis["id"]
            .map(mapping)
        )

case_analysis.to_csv(
    ANALYSIS_CSV,
    index=False
)

analysis_sha = sha256(ANALYSIS_CSV)


# =============================================================================
# Final results table
# =============================================================================

final_table = pd.DataFrame([
    {
        "experiment": "B-E3",
        "role": "supplementary",
        "benchmark": "E3-v3",
        "n": 120,
        "expected_valid_n": 58,
        "expected_invalid_n": 62,

        "correct_n": correct,

        "accuracy": accuracy,
        "accuracy_ci95_low": acc_ci[0],
        "accuracy_ci95_high": acc_ci[1],

        "VAR": VAR,
        "VAR_ci95_low": var_ci[0],
        "VAR_ci95_high": var_ci[1],

        "IRR": IRR,
        "IRR_ci95_low": irr_ci[0],
        "IRR_ci95_high": irr_ci[1],

        "precision": precision,
        "recall": recall,
        "f1": f1,

        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "llm_invoked_n": llm_invoked_n,
        "llm_not_invoked_n": pre_llm_n,

        "parse_ok_invoked_n": parse_ok_n,
        "parse_fail_invoked_n": parse_fail_n,

        "schema_valid_metric_available": False,
    }
])

final_table.to_csv(
    FINAL_TABLE,
    index=False
)

table_sha = sha256(FINAL_TABLE)


# =============================================================================
# Final summary
# =============================================================================

summary = {
    "experiment": "B-E3",

    "status": "FINAL_FROZEN",

    "role": (
        "Separate supplementary constrained-decoding "
        "follow-up on the frozen 120-case E3-v3 benchmark."
    ),

    "reviewer_requested_experiment_B": False,

    "must_not_be_presented_as_reviewer_requested_B": True,

    "model": "Qwen/Qwen3.5-9B",

    "benchmark": {
        "n": 120,
        "expected_valid_n": 58,
        "expected_invalid_n": 62,
        "sha256": KNOWN[BENCHMARK],
    },

    "decision_results": {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,

        "correct_n": correct,

        "accuracy": accuracy,
        "accuracy_ci95": list(acc_ci),

        "VAR": VAR,
        "VAR_ci95": list(var_ci),

        "IRR": IRR,
        "IRR_ci95": list(irr_ci),

        "precision": precision,
        "recall": recall,
        "f1": f1,
    },

    "structured_generation": {
        "llm_invoked_n": llm_invoked_n,
        "llm_not_invoked_n": pre_llm_n,

        "parse_ok_among_invoked_n":
            parse_ok_n,

        "parse_failure_among_invoked_n":
            parse_fail_n,

        "parse_success_rate_among_invoked":
            parse_ok_n / llm_invoked_n,

        "non_invoked_parse_ok_representation":
            "NaN / not applicable",

        "schema_valid_metric":
            "NOT PRESENT IN FROZEN B-E3 RESULT SCHEMA",

        "interpretation": (
            "All 77 LLM-invoked cases parsed successfully. "
            "The remaining 43 cases did not invoke the LLM; "
            "their parse_ok values are therefore not applicable "
            "rather than parse failures."
        ),
    },

    "rejection_stage_distribution":
        rejection_counts,

    "cleaner_equivalence": {
        "verification_timing":
            "RETROSPECTIVE_POST_EXECUTION",

        "exact_extracted_source_equal":
            exact_source_equal,

        "AST_equivalent":
            ast_equal,

        "claim_boundary": (
            "This equivalence check was performed during "
            "post-execution analytic closure and must not "
            "be described as a preregistered or pre-run check."
        ),
    },

    "provenance_note": (
        "The exact reviewer-requested Experiment B procedure "
        "requiring constrained decoding on E1 and E2A was "
        "identified after B-E3 had been executed. B-E3 is "
        "therefore retained as a separate supplementary "
        "follow-up and is not represented as satisfying the "
        "original reviewer-requested Experiment B."
    ),

    "paper_safe_interpretation": (
        "On the frozen 120-case E3-v3 benchmark, the B-E3 "
        "structured-generation treatment achieved 84.17% "
        "decision accuracy, 87.93% valid-acceptance rate, "
        "and 80.65% invalid-rejection rate. All 77 cases "
        "that reached the LLM parsed successfully, while "
        "43 cases were handled without invoking the LLM. "
        "This supplementary result does not replace the "
        "reviewer-requested E1/E2A constrained-decoding "
        "experiment and does not establish universal "
        "equivalence across models or benchmarks."
    ),

    "claim_boundaries": [
        "Do not call B-E3 the reviewer-requested Experiment B.",
        "Do not claim universal model-independent equivalence.",
        "Do not claim hallucination-free behavior.",
        "Do not claim formal standards conformance.",
        "Do not report a schema_valid rate for B-E3 because that field is absent.",
        (
            "Do not describe the retrospective cleaner check "
            "as preregistered or pre-execution."
        ),
    ],

    "C2_R1": {
        "model_calls": 0,
        "benchmark_executions": 0,
        "existing_frozen_inputs_modified": 0,
    },
}

write_json(
    summary,
    FINAL_SUMMARY
)

summary_sha = sha256(FINAL_SUMMARY)


# =============================================================================
# Final freeze manifest
# =============================================================================

freeze = {
    "experiment": "B-E3",

    "freeze": "FINAL",

    "version": 1,

    "status":
        "COMPLETE_FROZEN_DO_NOT_RERUN",

    "role":
        "SUPPLEMENTARY_NOT_REVIEWER_REQUESTED_B",

    "frozen_inputs": {
        str(path.relative_to(ROOT)):
            expected
        for path, expected in KNOWN.items()
    },

    "final_outputs": {
        str(ANALYSIS_CSV.relative_to(ROOT)):
            analysis_sha,

        str(FINAL_TABLE.relative_to(ROOT)):
            table_sha,

        str(FINAL_SUMMARY.relative_to(ROOT)):
            summary_sha,
    },

    "structured_generation_semantics": {
        "llm_invoked": 77,
        "llm_not_invoked": 43,
        "parse_ok_among_invoked": 77,
        "parse_fail_among_invoked": 0,
        "schema_valid_field_available": False,
    },

    "methodological_notes": [
        (
            "B-E3 predates retrieval of the exact "
            "reviewer-requested E1/E2A Experiment B procedure."
        ),
        (
            "B-E3 is supplementary and does not replace "
            "B-Review."
        ),
        (
            "43 parse_ok NaNs correspond exactly to "
            "llm_invoked=False and are treated as N/A."
        ),
        (
            "Cleaner equivalence was verified retrospectively."
        ),
        (
            "C2-R1 made zero API/model calls and executed "
            "zero benchmark cases."
        ),
    ],
}

write_json(
    freeze,
    FINAL_FREEZE
)

freeze_sha = sha256(FINAL_FREEZE)


# =============================================================================
# Persist closure artifacts to Drive
# =============================================================================

P_ANALYSIS = PERSIST / "analysis"
P_FINAL = PERSIST / "final"
P_MANIFEST = PERSIST / "manifests"

P_ANALYSIS.mkdir(parents=True, exist_ok=True)
P_FINAL.mkdir(parents=True, exist_ok=True)
P_MANIFEST.mkdir(parents=True, exist_ok=True)

copies = [
    (
        ANALYSIS_CSV,
        P_ANALYSIS / ANALYSIS_CSV.name
    ),
    (
        FINAL_TABLE,
        P_FINAL / FINAL_TABLE.name
    ),
    (
        FINAL_SUMMARY,
        P_FINAL / FINAL_SUMMARY.name
    ),
    (
        FINAL_FREEZE,
        P_MANIFEST / FINAL_FREEZE.name
    ),
]

for src, dst in copies:

    assert not dst.exists(), (
        f"STOP: persistent closure artifact exists:\n{dst}"
    )

    shutil.copyfile(src, dst)

    assert sha256(src) == sha256(dst)

print("\n✓ C2-R1 closure evidence persisted to Drive byte-for-byte")


# =============================================================================
# Final integrity
# =============================================================================

print("\n" + "-" * 120)
print("POST-CLOSURE INPUT INTEGRITY")
print("-" * 120)

for path, expected in KNOWN.items():

    assert sha256(path) == expected

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


# =============================================================================
# Final display
# =============================================================================

print("\n" + "-" * 120)
print("FINAL B-E3 STATUS")
print("-" * 120)

print("Role                  : SUPPLEMENTARY")
print("Reviewer-requested B  : NO")
print("Cases                 : 120")
print(f"Accuracy              : {accuracy*100:.2f}%")
print(f"VAR                   : {VAR*100:.2f}%")
print(f"IRR                   : {IRR*100:.2f}%")
print(f"LLM invoked           : {llm_invoked_n}/120")
print(f"Parse success/invoked : {parse_ok_n}/{llm_invoked_n}")
print("Schema-valid metric   : NOT AVAILABLE")
print("Cleaner verification  : RETROSPECTIVE")
print("Model calls in C2-R1  : 0")


print("\n" + "-" * 120)
print("FINAL ARTIFACT HASHES")
print("-" * 120)

print(
    "Case analysis :",
    analysis_sha
)

print(
    "Results table :",
    table_sha
)

print(
    "Summary       :",
    summary_sha
)

print(
    "FINAL FREEZE  :",
    freeze_sha
)


print("\n" + "=" * 120)
print("✓ STEP C2-R1 COMPLETE")
print("✓ B-E3 ANALYTIC CLOSURE COMPLETE")
print("✓ B-E3 FINAL FREEZE CREATED")
print("✓ 84.17% ACCURACY / 87.93% VAR / 80.65% IRR VERIFIED")
print("✓ 77/77 LLM-INVOKED CASES PARSED SUCCESSFULLY")
print("✓ 43 NON-INVOKED CASES CORRECTLY CLASSIFIED AS PARSE N/A")
print("✓ NO schema_valid METRIC INVENTED")
print("✓ CLEANER EQUIVALENCE VERIFIED RETROSPECTIVELY")
print("✓ B-E3 FIXED AS SUPPLEMENTARY")
print("✓ B-E3 NOT REPRESENTED AS REVIEWER-REQUESTED B")
print("✓ FINAL CLOSURE EVIDENCE PERSISTED TO DRIVE")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ DO NOT RERUN C2-R1")
print("✓ ALL EXPERIMENTAL EVIDENCE STREAMS ARE NOW CLOSED")
print("✓ NEXT: C3 — MASTER A + B EVIDENCE CONSOLIDATION")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE C3")
print("=" * 120)

JOURNAL CONSOLIDATION — STEP C2-R1: B-E3 ANALYTIC CLOSURE AND FINAL FREEZE

------------------------------------------------------------------------------------------------------------------------
C2-R1 RECOVERY GATE
------------------------------------------------------------------------------------------------------------------------
✓ failed C2 produced no final closure artifacts
✓ C2-R1 may proceed

------------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_schema_constrained_decoding/preregistration/experiment_B_preregistration_v1.json
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/experiment_B_structured_adapter_v1.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/run_experiment_B_struct

AssertionError: STOP: historical and B-E3 cleaners are not AST-equivalent. Do not freeze B-E3.

In [64]:
from pathlib import Path
import ast
import difflib
import hashlib
import json
import math
import pandas as pd

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C2-D2: RETROSPECTIVE CLEANER-DIFFERENCE DIAGNOSIS")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

BE3 = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

HIST_ADAPTER = ROOT / "src" / "llm" / "together_adapter.py"

TREATMENT_ADAPTER = (
    BE3
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

RAW_RESULT = (
    BE3
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

EXPECTED = {
    HIST_ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    TREATMENT_ADAPTER:
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    RAW_RESULT:
        "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1",

    BENCHMARK:
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",
}


# =============================================================================
# Helpers
# =============================================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def get_function_node_and_source(path, function_name):
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ):
            segment = ast.get_source_segment(source, node)
            assert segment is not None
            return node, segment.strip()

    raise AssertionError(
        f"{function_name} not found in {path}"
    )


def normalized_body_ast(node):
    """
    Compare executable function bodies while ignoring:
      - function name
      - source locations
      - annotations
    """
    module = ast.Module(
        body=node.body,
        type_ignores=[]
    )

    return ast.dump(
        module,
        include_attributes=False,
        indent=2
    )


def compile_cleaner(path, function_name="_clean_json_text"):
    """
    Load ONLY the requested function into an isolated namespace.
    No provider/API code is executed.
    """
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    target = None

    for node in tree.body:
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ):
            target = node
            break

    assert target is not None

    module = ast.Module(
        body=[target],
        type_ignores=[]
    )

    ast.fix_missing_locations(module)

    ns = {
        "json": json,
    }

    exec(
        compile(
            module,
            filename=str(path),
            mode="exec"
        ),
        ns
    )

    return ns[function_name]


def run_cleaner(fn, value):
    try:
        result = fn(value)

        return {
            "status": "OK",
            "type": type(result).__name__,
            "repr": repr(result),
            "value": result,
        }

    except Exception as e:
        return {
            "status": "EXCEPTION",
            "type": type(e).__name__,
            "repr": repr(e),
            "value": None,
        }


# =============================================================================
# Frozen provenance
# =============================================================================

print("\n" + "-" * 120)
print("FROZEN PROVENANCE")
print("-" * 120)

for path, expected_sha in EXPECTED.items():

    assert path.exists(), f"Missing: {path}"

    actual_sha = sha256(path)

    assert actual_sha == expected_sha, (
        f"Hash mismatch:\n{path}\n"
        f"Expected: {expected_sha}\n"
        f"Actual:   {actual_sha}"
    )

    print("✓", path.relative_to(ROOT))


# =============================================================================
# Confirm failed C2-R1 did not create final closure artifacts
# =============================================================================

possible_outputs = [
    BE3 / "analysis"
    / "experiment_B_E3_final_case_analysis_v1.csv",

    BE3 / "final"
    / "experiment_B_E3_FINAL_results_table_v1.csv",

    BE3 / "final"
    / "experiment_B_E3_FINAL_summary_v1.json",

    BE3 / "manifests"
    / "experiment_B_E3_FINAL_FREEZE_v1.json",
]

print("\n" + "-" * 120)
print("FAILED C2-R1 OUTPUT STATUS")
print("-" * 120)

existing_outputs = []

for p in possible_outputs:
    if p.exists():
        existing_outputs.append(p)
        print("EXISTS :", p.relative_to(ROOT))
    else:
        print("absent :", p.relative_to(ROOT))

assert not existing_outputs, (
    "STOP: C2-R1 created one or more closure artifacts. "
    "Do not continue diagnosis until they are inspected."
)

print("✓ failed C2-R1 created no closure artifacts")


# =============================================================================
# Extract cleaner implementations
# =============================================================================

hist_node, hist_source = get_function_node_and_source(
    HIST_ADAPTER,
    "_clean_json_text"
)

treat_node, treat_source = get_function_node_and_source(
    TREATMENT_ADAPTER,
    "_clean_json_text"
)

print("\n" + "-" * 120)
print("HISTORICAL CLEANER")
print("-" * 120)
print(hist_source)

print("\n" + "-" * 120)
print("B-E3 TREATMENT CLEANER")
print("-" * 120)
print(treat_source)


# =============================================================================
# Unified source diff
# =============================================================================

print("\n" + "-" * 120)
print("UNIFIED SOURCE DIFF")
print("-" * 120)

diff = list(
    difflib.unified_diff(
        hist_source.splitlines(),
        treat_source.splitlines(),
        fromfile="historical_cleaner",
        tofile="B_E3_cleaner",
        lineterm=""
    )
)

if diff:
    print("\n".join(diff))
else:
    print("No source difference.")


# =============================================================================
# Function-body AST comparison
# =============================================================================

hist_body_ast = normalized_body_ast(hist_node)
treat_body_ast = normalized_body_ast(treat_node)

body_ast_equal = hist_body_ast == treat_body_ast

print("\n" + "-" * 120)
print("FUNCTION-BODY AST COMPARISON")
print("-" * 120)

print(
    "Executable body AST equal:",
    body_ast_equal
)

if not body_ast_equal:

    print("\nHistorical body AST:")
    print(hist_body_ast)

    print("\nB-E3 body AST:")
    print(treat_body_ast)


# =============================================================================
# Compile cleaners without importing provider code
# =============================================================================

hist_cleaner = compile_cleaner(HIST_ADAPTER)
treat_cleaner = compile_cleaner(TREATMENT_ADAPTER)

print("\n✓ both cleaners compiled in isolated namespaces")
print("✓ no provider/API code imported")


# =============================================================================
# Synthetic behavioral probes
# =============================================================================

synthetic_inputs = [
    '{"service_type":"eMBB"}',

    '  {"service_type":"eMBB"}  ',

    '```json\n{"service_type":"eMBB"}\n```',

    '```\n{"service_type":"eMBB"}\n```',

    'prefix {"service_type":"eMBB"} suffix',

    '\n\n{"service_type":"URLLC"}\n\n',

    '{"service_type":"mMTC","latency_ms":null}',

    '{"objective":"contains { brace } text"}',

    '',

    '   ',
]

print("\n" + "-" * 120)
print("SYNTHETIC BEHAVIORAL COMPARISON")
print("-" * 120)

synthetic_diff_n = 0

for i, value in enumerate(synthetic_inputs, 1):

    h = run_cleaner(hist_cleaner, value)
    t = run_cleaner(treat_cleaner, value)

    same = (
        h["status"] == t["status"]
        and h["type"] == t["type"]
        and h["repr"] == t["repr"]
    )

    if not same:
        synthetic_diff_n += 1

    print(f"\nProbe {i}: {'SAME' if same else 'DIFFERENT'}")
    print("Input     :", repr(value))
    print("Historical:", h["status"], h["type"], h["repr"])
    print("B-E3     :", t["status"], t["type"], t["repr"])

print(
    "\nSynthetic probes with different behavior:",
    f"{synthetic_diff_n}/{len(synthetic_inputs)}"
)


# =============================================================================
# Actual frozen B-E3 raw_text behavioral comparison
# =============================================================================

raw = pd.read_csv(RAW_RESULT)

assert len(raw) == 120
assert raw["id"].nunique() == 120
assert "llm_invoked" in raw.columns
assert "raw_text" in raw.columns
assert "parse_ok" in raw.columns

invoked = raw[
    raw["llm_invoked"] == True
].copy()

assert len(invoked) == 77
assert invoked["raw_text"].notna().all()

print("\n" + "-" * 120)
print("ACTUAL 77 B-E3 LLM OUTPUTS — CLEANER BEHAVIOR")
print("-" * 120)

actual_differences = []
historical_json_failures = []
treatment_json_failures = []

for _, row in invoked.iterrows():

    case_id = str(row["id"])
    text = row["raw_text"]

    h = run_cleaner(hist_cleaner, text)
    t = run_cleaner(treat_cleaner, text)

    same = (
        h["status"] == t["status"]
        and h["type"] == t["type"]
        and h["repr"] == t["repr"]
    )

    if not same:
        actual_differences.append({
            "id": case_id,
            "historical": h,
            "treatment": t,
            "raw_text": text,
        })

    # Also test whether each cleaner's output can be json.loads'ed.
    if h["status"] == "OK":
        try:
            json.loads(h["value"])
        except Exception as e:
            historical_json_failures.append(
                (case_id, type(e).__name__, str(e))
            )
    else:
        historical_json_failures.append(
            (case_id, h["type"], h["repr"])
        )

    if t["status"] == "OK":
        try:
            json.loads(t["value"])
        except Exception as e:
            treatment_json_failures.append(
                (case_id, type(e).__name__, str(e))
            )
    else:
        treatment_json_failures.append(
            (case_id, t["type"], t["repr"])
        )


print(
    "Actual outputs compared:",
    len(invoked)
)

print(
    "Cleaner-output differences:",
    len(actual_differences)
)

print(
    "Historical-cleaner JSON failures:",
    len(historical_json_failures)
)

print(
    "B-E3-cleaner JSON failures:",
    len(treatment_json_failures)
)


# =============================================================================
# Display any actual differences
# =============================================================================

if actual_differences:

    print("\n" + "-" * 120)
    print("ACTUAL CASES WITH CLEANER DIFFERENCES")
    print("-" * 120)

    for item in actual_differences:

        print("\nCase:", item["id"])

        print(
            "Raw:",
            repr(item["raw_text"])
        )

        print(
            "Historical:",
            item["historical"]["status"],
            item["historical"]["type"],
            item["historical"]["repr"]
        )

        print(
            "B-E3:",
            item["treatment"]["status"],
            item["treatment"]["type"],
            item["treatment"]["repr"]
        )

else:

    print(
        "\n✓ No behavioral cleaner difference "
        "on any of the 77 actual B-E3 LLM outputs"
    )


# =============================================================================
# Parse consistency with frozen B-E3 observations
# =============================================================================

assert invoked["parse_ok"].notna().all()

observed_parse_success = int(
    invoked["parse_ok"]
    .astype(bool)
    .sum()
)

print("\n" + "-" * 120)
print("FROZEN PARSE CONSISTENCY")
print("-" * 120)

print(
    "Frozen B-E3 parse successes:",
    f"{observed_parse_success}/77"
)

print(
    "Historical cleaner JSON-load successes:",
    f"{77-len(historical_json_failures)}/77"
)

print(
    "B-E3 cleaner JSON-load successes:",
    f"{77-len(treatment_json_failures)}/77"
)


# =============================================================================
# Frozen integrity after diagnosis
# =============================================================================

print("\n" + "-" * 120)
print("POST-DIAGNOSIS FROZEN INTEGRITY")
print("-" * 120)

for path, expected_sha in EXPECTED.items():

    assert sha256(path) == expected_sha

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


# =============================================================================
# Interpretation gate
# =============================================================================

print("\n" + "-" * 120)
print("C2-D2 INTERPRETATION GATE")
print("-" * 120)

if len(actual_differences) == 0:

    print(
        "✓ cleaners differ in implementation but produced "
        "identical cleaned outputs on all 77 actual B-E3 LLM responses"
    )

    if (
        len(historical_json_failures) == 0
        and len(treatment_json_failures) == 0
    ):
        print(
            "✓ both cleaners yield JSON-loadable output "
            "for all 77 actual B-E3 responses"
        )

    print(
        "Candidate paper treatment: disclose implementation "
        "difference, report observed behavioral equivalence "
        "on the frozen B-E3 outputs, and do NOT claim "
        "implementation equivalence."
    )

else:

    print(
        "⚠ cleaners differ behaviorally on actual B-E3 outputs"
    )

    print(
        "Do NOT freeze B-E3 as cleaner-equivalent."
    )

    print(
        "The affected cases must be assessed before deciding "
        "what B-E3 claims remain defensible."
    )


print("\n" + "=" * 120)
print("✓ STEP C2-D2 COMPLETE")
print("✓ CLEANER IMPLEMENTATIONS COMPARED")
print("✓ SYNTHETIC BEHAVIOR COMPARED")
print("✓ ALL 77 ACTUAL B-E3 LLM OUTPUTS COMPARED")
print("✓ FROZEN PARSE CONSISTENCY INSPECTED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ NO FILE MODIFIED")
print("✓ DO NOT RERUN B-E3")
print("✓ SEND THIS COMPLETE OUTPUT BEFORE C2-R2")
print("=" * 120)

JOURNAL CONSOLIDATION — STEP C2-D2: RETROSPECTIVE CLEANER-DIFFERENCE DIAGNOSIS

------------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
------------------------------------------------------------------------------------------------------------------------
✓ src/llm/together_adapter.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/experiment_B_structured_adapter_v1.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/results/treatment_B1/experiment_B_B1_structured_treatment_120_v1.json
✓ experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv

------------------------------------------------------------------------------------------------------------------------
FAILED C2-R1 OUTPUT STATUS
------------------------------------------------------------------------------------------------------------------------
absent : journal_

In [65]:
# ==================================================================================================
# JOURNAL CONSOLIDATION — STEP C2-D3
# CORRECTED RETROSPECTIVE CLEANER BEHAVIORAL COMPARISON
#
# Purpose:
#   Correct the C2-D2 diagnostic harness by supplying Python's `re`
#   module to the historical cleaner's isolated execution namespace.
#
# IMPORTANT:
#   - READ ONLY
#   - ZERO API / MODEL CALLS
#   - ZERO BENCHMARK EXECUTIONS
#   - DO NOT MODIFY ANY FROZEN ARTIFACT
#   - DO NOT RERUN B-E3
# ==================================================================================================

from pathlib import Path
import ast
import csv
import hashlib
import json
import re

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C2-D3: CORRECTED CLEANER BEHAVIORAL COMPARISON")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

HIST_ADAPTER = ROOT / "src/llm/together_adapter.py"

B_ADAPTER = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

B_RAW = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)

FILES = [
    HIST_ADAPTER,
    B_ADAPTER,
    B_RAW,
    BENCHMARK,
]

# --------------------------------------------------------------------------------------------------
# Known frozen SHA-256 values
# --------------------------------------------------------------------------------------------------

EXPECTED_SHA = {
    HIST_ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    B_ADAPTER:
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    B_RAW:
        "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1",

    BENCHMARK:
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",
}


# --------------------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------------------

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)

    return h.hexdigest()


def extract_function_source(path, function_name):
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ):
            segment = ast.get_source_segment(source, node)

            assert segment is not None

            return segment

    raise RuntimeError(
        f"{function_name} not found in {path}"
    )


def compile_cleaner(function_source, namespace=None):
    """
    Compile only the extracted _clean_json_text function.

    This does NOT import the provider adapter and therefore cannot
    trigger an API/model call.
    """

    ns = {} if namespace is None else dict(namespace)

    exec(
        function_source,
        ns
    )

    assert "_clean_json_text" in ns

    return ns["_clean_json_text"]


def call_cleaner(fn, value):
    try:
        out = fn(value)

        return {
            "status": "OK",
            "type": type(out).__name__,
            "value": out,
            "exception": None,
        }

    except Exception as exc:

        return {
            "status": "EXCEPTION",
            "type": type(exc).__name__,
            "value": None,
            "exception": repr(exc),
        }


def json_load_status(cleaned):
    if cleaned["status"] != "OK":
        return False, None

    try:
        parsed = json.loads(
            cleaned["value"]
        )

        return True, parsed

    except Exception:
        return False, None


# ==================================================================================================
# 1. FROZEN PROVENANCE
# ==================================================================================================

print("\n" + "-" * 120)
print("FROZEN PROVENANCE")
print("-" * 120)

before = {}

for path in FILES:

    assert path.exists(), (
        f"Missing frozen artifact:\n{path}"
    )

    actual_sha = sha256(path)
    expected_sha = EXPECTED_SHA[path]

    assert actual_sha == expected_sha, (
        f"STOP: frozen SHA mismatch\n"
        f"File:     {path}\n"
        f"Expected: {expected_sha}\n"
        f"Actual:   {actual_sha}"
    )

    before[path] = actual_sha

    print(
        "✓",
        path.relative_to(ROOT)
    )

    print(
        "  SHA-256:",
        actual_sha
    )


# ==================================================================================================
# 2. EXTRACT FROZEN CLEANER IMPLEMENTATIONS
# ==================================================================================================

hist_src = extract_function_source(
    HIST_ADAPTER,
    "_clean_json_text"
)

b_src = extract_function_source(
    B_ADAPTER,
    "_clean_json_text"
)


print("\n" + "-" * 120)
print("CORRECTION TO C2-D2 HARNESS")
print("-" * 120)

print(
    "Historical cleaner references `re`:",
    "re." in hist_src
)

assert "re." in hist_src, (
    "STOP: expected historical cleaner to reference `re`."
)

print(
    "Python `re` module supplied to historical isolated namespace: True"
)


# ==================================================================================================
# 3. COMPILE CLEANERS IN ISOLATED NAMESPACES
# ==================================================================================================

# THIS IS THE CRITICAL C2-D3 CORRECTION.
historical_cleaner = compile_cleaner(
    hist_src,
    {
        "re": re
    }
)

b_cleaner = compile_cleaner(
    b_src
)


print(
    "✓ historical cleaner compiled with required `re` dependency"
)

print(
    "✓ B-E3 cleaner compiled"
)

print(
    "✓ no provider/API code imported"
)


# ==================================================================================================
# 4. BASIC SANITY TEST
# ==================================================================================================

print("\n" + "-" * 120)
print("CORRECTED SANITY TEST")
print("-" * 120)

sanity_input = '{"service_type":"eMBB"}'

hist_sanity = call_cleaner(
    historical_cleaner,
    sanity_input
)

b_sanity = call_cleaner(
    b_cleaner,
    sanity_input
)

print(
    "Historical:",
    hist_sanity["status"],
    repr(hist_sanity["value"])
    if hist_sanity["status"] == "OK"
    else hist_sanity["exception"]
)

print(
    "B-E3     :",
    b_sanity["status"],
    repr(b_sanity["value"])
    if b_sanity["status"] == "OK"
    else b_sanity["exception"]
)

assert hist_sanity["status"] == "OK", (
    "STOP: historical cleaner still cannot execute."
)

assert b_sanity["status"] == "OK"

print(
    "✓ historical cleaner now executes successfully"
)


# ==================================================================================================
# 5. SYNTHETIC BEHAVIORAL PROBES
# ==================================================================================================

probes = [
    '{"service_type":"eMBB"}',

    '  {"service_type":"eMBB"}  ',

    '```json\n{"service_type":"eMBB"}\n```',

    '```\n{"service_type":"eMBB"}\n```',

    '```JSON\n{"service_type":"URLLC"}\n```',

    'prefix {"service_type":"eMBB"} suffix',

    '\n\n{"service_type":"URLLC"}\n\n',

    '{"service_type":"mMTC","latency_ms":null}',

    '{"objective":"contains { brace } text"}',

    '',

    '   ',
]


print("\n" + "-" * 120)
print("CORRECTED SYNTHETIC BEHAVIORAL COMPARISON")
print("-" * 120)

synthetic_differences = []

for i, probe in enumerate(
    probes,
    start=1
):

    h = call_cleaner(
        historical_cleaner,
        probe
    )

    b = call_cleaner(
        b_cleaner,
        probe
    )

    same = (
        h["status"] == b["status"]
        and
        h["type"] == b["type"]
        and
        h["value"] == b["value"]
        and
        h["exception"] == b["exception"]
    )

    if not same:
        synthetic_differences.append(i)

    print(
        f"\nProbe {i}:",
        "SAME" if same else "DIFFERENT"
    )

    print(
        "Input     :",
        repr(probe)
    )

    if h["status"] == "OK":
        print(
            "Historical:",
            h["status"],
            h["type"],
            repr(h["value"])
        )
    else:
        print(
            "Historical:",
            h["status"],
            h["type"],
            h["exception"]
        )

    if b["status"] == "OK":
        print(
            "B-E3     :",
            b["status"],
            b["type"],
            repr(b["value"])
        )
    else:
        print(
            "B-E3     :",
            b["status"],
            b["type"],
            b["exception"]
        )


print(
    "\nSynthetic probes with different behavior:",
    f"{len(synthetic_differences)}/{len(probes)}"
)


# ==================================================================================================
# 6. LOAD FROZEN B-E3 RAW RESULT
#
# IMPORTANT:
# The historical artifact has a .json extension but its actual format is CSV.
# ==================================================================================================

with open(
    B_RAW,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    rows = list(reader)

    fieldnames = list(
        reader.fieldnames or []
    )


assert len(rows) == 120, (
    f"Expected 120 B-E3 rows, found {len(rows)}"
)


print("\n" + "-" * 120)
print("FROZEN B-E3 RESULT SCHEMA")
print("-" * 120)

print(
    "Logical rows:",
    len(rows)
)

print(
    "Columns:",
    len(fieldnames)
)


# --------------------------------------------------------------------------------------------------
# Column identification
# --------------------------------------------------------------------------------------------------

def choose_column(candidates):

    for candidate in candidates:
        if candidate in fieldnames:
            return candidate

    return None


case_col = choose_column([
    "id",
    "case_id",
    "test_id",
])

raw_col = choose_column([
    "raw_text",
    "raw_llm_output",
    "llm_raw_output",
    "raw_output",
    "raw_response",
    "llm_response",
])

invoked_col = choose_column([
    "llm_invoked",
    "model_invoked",
])

parse_col = choose_column([
    "parse_ok",
    "llm_parse_ok",
])


assert case_col is not None, (
    "STOP: could not identify case-id column.\n"
    f"Available columns: {fieldnames}"
)

assert raw_col is not None, (
    "STOP: could not identify raw-output column.\n"
    f"Available columns: {fieldnames}"
)

assert invoked_col is not None, (
    "STOP: could not identify LLM invocation column.\n"
    f"Available columns: {fieldnames}"
)


print(
    "Case column:",
    case_col
)

print(
    "Raw-output column:",
    raw_col
)

print(
    "Invocation column:",
    invoked_col
)

print(
    "Parse column:",
    parse_col
)


# ==================================================================================================
# 7. SELECT THE 77 ACTUAL LLM-INVOKED CASES
# ==================================================================================================

TRUE_VALUES = {
    "1",
    "true",
    "yes",
    "y",
}


def is_true(value):
    return (
        str(value)
        .strip()
        .lower()
        in TRUE_VALUES
    )


llm_rows = [
    row
    for row in rows
    if is_true(
        row.get(
            invoked_col,
            ""
        )
    )
]


print("\n" + "-" * 120)
print("FROZEN B-E3 LLM OUTPUT SELECTION")
print("-" * 120)

print(
    "Raw rows:",
    len(rows)
)

print(
    "LLM-output rows selected:",
    len(llm_rows)
)


assert len(llm_rows) == 77, (
    "STOP: expected exactly 77 LLM-invoked "
    f"B-E3 cases, found {len(llm_rows)}"
)


# ==================================================================================================
# 8. CORRECTED ACTUAL-OUTPUT BEHAVIORAL COMPARISON
# ==================================================================================================

print("\n" + "-" * 120)
print("ACTUAL 77 B-E3 LLM OUTPUTS — CORRECTED CLEANER BEHAVIOR")
print("-" * 120)


cleaner_diff_cases = []

hist_json_failures = []

b_json_failures = []

semantic_json_differences = []


for row in llm_rows:

    case_id = str(
        row[case_col]
    )

    raw_text = row[
        raw_col
    ]

    h = call_cleaner(
        historical_cleaner,
        raw_text
    )

    b = call_cleaner(
        b_cleaner,
        raw_text
    )


    same_cleaned = (
        h["status"] == b["status"]
        and
        h["type"] == b["type"]
        and
        h["value"] == b["value"]
        and
        h["exception"] == b["exception"]
    )


    if not same_cleaned:

        cleaner_diff_cases.append(
            case_id
        )


    h_ok, h_json = json_load_status(
        h
    )

    b_ok, b_json = json_load_status(
        b
    )


    if not h_ok:

        hist_json_failures.append(
            case_id
        )


    if not b_ok:

        b_json_failures.append(
            case_id
        )


    if (
        h_ok
        and b_ok
        and h_json != b_json
    ):

        semantic_json_differences.append(
            case_id
        )


print(
    "Actual outputs compared:",
    len(llm_rows)
)

print(
    "Cleaner-output differences:",
    len(cleaner_diff_cases)
)

print(
    "Historical-cleaner JSON failures:",
    len(hist_json_failures)
)

print(
    "B-E3-cleaner JSON failures:",
    len(b_json_failures)
)

print(
    "Parsed-JSON semantic differences:",
    len(semantic_json_differences)
)


# ==================================================================================================
# 9. DISPLAY ONLY GENUINELY DIFFERENT ACTUAL CASES
# ==================================================================================================

print("\n" + "-" * 120)
print("GENUINELY AFFECTED ACTUAL CASES")
print("-" * 120)


if not cleaner_diff_cases:

    print(
        "None — both cleaners produced identical cleaned text "
        "on all 77 actual B-E3 outputs."
    )

else:

    print(
        "Cases with different cleaned text:"
    )

    print(
        cleaner_diff_cases
    )


    for row in llm_rows:

        case_id = str(
            row[case_col]
        )

        if case_id not in cleaner_diff_cases:
            continue


        raw_text = row[
            raw_col
        ]

        h = call_cleaner(
            historical_cleaner,
            raw_text
        )

        b = call_cleaner(
            b_cleaner,
            raw_text
        )


        print(
            "\nCase:",
            case_id
        )

        print(
            "Raw:",
            repr(raw_text)
        )


        if h["status"] == "OK":

            print(
                "Historical:",
                repr(
                    h["value"]
                )
            )

        else:

            print(
                "Historical EXCEPTION:",
                h["exception"]
            )


        if b["status"] == "OK":

            print(
                "B-E3:",
                repr(
                    b["value"]
                )
            )

        else:

            print(
                "B-E3 EXCEPTION:",
                b["exception"]
            )


# ==================================================================================================
# 10. FROZEN PARSE CONSISTENCY
# ==================================================================================================

hist_success = (
    len(llm_rows)
    - len(hist_json_failures)
)

b_success = (
    len(llm_rows)
    - len(b_json_failures)
)


print("\n" + "-" * 120)
print("CORRECTED FROZEN PARSE CONSISTENCY")
print("-" * 120)


if parse_col is not None:

    frozen_success = sum(
        is_true(
            row.get(
                parse_col,
                ""
            )
        )
        for row in llm_rows
    )

    print(
        "Frozen B-E3 parse successes:",
        f"{frozen_success}/{len(llm_rows)}"
    )


print(
    "Historical cleaner JSON-load successes:",
    f"{hist_success}/{len(llm_rows)}"
)

print(
    "B-E3 cleaner JSON-load successes:",
    f"{b_success}/{len(llm_rows)}"
)


# ==================================================================================================
# 11. CORRECTED INTERPRETATION GATE
# ==================================================================================================

print("\n" + "-" * 120)
print("C2-D3 INTERPRETATION GATE")
print("-" * 120)


if (
    len(cleaner_diff_cases) == 0
    and
    len(hist_json_failures) == 0
    and
    len(b_json_failures) == 0
):

    print(
        "✓ On all 77 actual frozen B-E3 LLM outputs, "
        "the two cleaners are behaviorally equivalent."
    )

    print(
        "✓ The source implementations remain "
        "textually/structurally different."
    )

    print(
        "✓ C2-D2's 0/77 historical result was caused "
        "by the diagnostic harness omitting `re`."
    )

    print(
        "✓ Cleaner implementation difference did not affect "
        "the frozen B-E3 observations."
    )

    print(
        "→ B-E3 may proceed to final read-only case "
        "analysis/freeze, with transparent provenance."
    )

else:

    print(
        "⚠ Genuine cleaner behavioral differences remain "
        "after correcting the historical namespace."
    )

    print(
        "⚠ Do NOT claim cleaner equivalence."
    )

    print(
        "→ Affected cases must be assessed before "
        "B-E3 final freeze."
    )


# ==================================================================================================
# 12. POST-DIAGNOSIS FROZEN INTEGRITY
# ==================================================================================================

print("\n" + "-" * 120)
print("POST-DIAGNOSIS FROZEN INTEGRITY")
print("-" * 120)


for path in FILES:

    after_sha = sha256(
        path
    )

    assert after_sha == before[path], (
        f"STOP: frozen file changed:\n{path}"
    )

    assert after_sha == EXPECTED_SHA[path]

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


# ==================================================================================================
# COMPLETE
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "✓ STEP C2-D3 COMPLETE"
)

print(
    "✓ C2-D2 DIAGNOSTIC NAMESPACE DEFECT CORRECTED"
)

print(
    "✓ HISTORICAL CLEANER EXECUTED WITH REQUIRED `re` DEPENDENCY"
)

print(
    "✓ SYNTHETIC BEHAVIOR RECHECKED"
)

print(
    "✓ ALL 77 ACTUAL B-E3 LLM OUTPUTS RECHECKED"
)

print(
    "✓ ZERO API/MODEL CALLS"
)

print(
    "✓ ZERO BENCHMARK EXECUTIONS"
)

print(
    "✓ NO FILE MODIFIED"
)

print(
    "✓ DO NOT RERUN B-E3"
)

print(
    "✓ SEND THIS COMPLETE OUTPUT BEFORE C2-R2"
)

print("=" * 120)

JOURNAL CONSOLIDATION — STEP C2-D3: CORRECTED CLEANER BEHAVIORAL COMPARISON

------------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
------------------------------------------------------------------------------------------------------------------------
✓ src/llm/together_adapter.py
  SHA-256: b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/experiment_B_structured_adapter_v1.py
  SHA-256: 3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6
✓ journal_experiments/experiment_B_schema_constrained_decoding/results/treatment_B1/experiment_B_B1_structured_treatment_120_v1.json
  SHA-256: 40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1
✓ experiment-2b/reproducibility/e3v3_heldout_benchmark_v1.0/e3v3_heldout_benchmark_120_v1.csv
  SHA-256: 84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed3

In [66]:
# ==================================================================================================
# JOURNAL CONSOLIDATION — STEP C2-R2
# B-E3 FINAL ANALYTIC CLOSURE AND FREEZE
#
# Purpose:
#   Close the supplementary B-E3 experiment after the corrected C2-D3
#   retrospective cleaner-behavior analysis.
#
# IMPORTANT:
#   - ZERO API / MODEL CALLS
#   - ZERO BENCHMARK EXECUTIONS
#   - NEVER RERUN B-E3
#   - NEVER MODIFY FROZEN B-E3 INPUT/RESULT ARTIFACTS
#   - CREATE FINAL ANALYTIC/FREEZE ARTIFACTS ONLY
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import shutil

import pandas as pd
from scipy.stats import beta


print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C2-R2: B-E3 FINAL ANALYTIC CLOSURE AND FREEZE")
print("=" * 120)


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

BE3 = (
    ROOT
    / "journal_experiments"
    / "experiment_B_schema_constrained_decoding"
)

PERSIST_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "experiment_B_schema_constrained_decoding"
)


PREREG = (
    BE3
    / "preregistration"
    / "experiment_B_preregistration_v1.json"
)

TREATMENT_ADAPTER = (
    BE3
    / "implementation"
    / "experiment_B_structured_adapter_v1.py"
)

HARNESS = (
    BE3
    / "implementation"
    / "run_experiment_B_structured_treatment_v1.py"
)

RAW_RESULT = (
    BE3
    / "results"
    / "treatment_B1"
    / "experiment_B_B1_structured_treatment_120_v1.json"
)

EXEC_RECORD = (
    BE3
    / "manifests"
    / "experiment_B_B9B_execution_record_v1.json"
)

HIST_ADAPTER = (
    ROOT
    / "src"
    / "llm"
    / "together_adapter.py"
)

BENCHMARK = (
    ROOT
    / "experiment-2b"
    / "reproducibility"
    / "e3v3_heldout_benchmark_v1.0"
    / "e3v3_heldout_benchmark_120_v1.csv"
)


ANALYSIS_DIR = (
    BE3
    / "analysis"
)

FINAL_DIR = (
    BE3
    / "final"
)

MANIFEST_DIR = (
    BE3
    / "manifests"
)


CASE_ANALYSIS = (
    ANALYSIS_DIR
    / "experiment_B_E3_final_case_analysis_v1.csv"
)

RESULTS_TABLE = (
    FINAL_DIR
    / "experiment_B_E3_FINAL_results_table_v1.csv"
)

FINAL_SUMMARY = (
    FINAL_DIR
    / "experiment_B_E3_FINAL_summary_v1.json"
)

FINAL_FREEZE = (
    MANIFEST_DIR
    / "experiment_B_E3_FINAL_FREEZE_v1.json"
)


FINAL_OUTPUTS = [
    CASE_ANALYSIS,
    RESULTS_TABLE,
    FINAL_SUMMARY,
    FINAL_FREEZE,
]


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def cp_interval(successes, total, alpha=0.05):

    if total == 0:
        return (
            float("nan"),
            float("nan")
        )

    lower = (
        0.0
        if successes == 0
        else beta.ppf(
            alpha / 2,
            successes,
            total - successes + 1
        )
    )

    upper = (
        1.0
        if successes == total
        else beta.ppf(
            1 - alpha / 2,
            successes + 1,
            total - successes
        )
    )

    return (
        float(lower),
        float(upper)
    )


def json_dump_stable(
    obj,
    path
):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        )
        + "\n",
        encoding="utf-8"
    )


def bool_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )


def safe_value(value):

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            bool,
            int,
            float,
            str
        )
    ):
        return value

    return str(value)


# ==================================================================================================
# 2. KNOWN FROZEN PROVENANCE
# ==================================================================================================

EXPECTED_SHA = {

    PREREG:
        "d6e6db1bf6248ac211f4ef925ba940254e765e6eedc51c0b67353f897a50a2f5",

    TREATMENT_ADAPTER:
        "3e2a1cac6c2427eab9a28b69dc67b9a659ff9c83fd3d7bacb84222c51a21b4b6",

    HARNESS:
        "10314f909693820d6a997a35680eb9366a40af97ba79e67e752568a92cbc7cca",

    RAW_RESULT:
        "40b39653bf055022895cbad6755d90d329d10b7e9aaaf244075a1ecdf76bb1c1",

    EXEC_RECORD:
        "4654ea991d50f65089f5be68d5bd565004a8bb768f5e106072fd6c50facfd7d5",

    HIST_ADAPTER:
        "b11b08277578cdfd366af4e2e96862d601d7f08305dddd2c04c23e0ebc7506b3",

    BENCHMARK:
        "84576e7d99a2721e61ded5a14bdafeb96707b068397243bbed32aa735496d4de",
}


print("\n" + "-" * 120)
print("FROZEN PROVENANCE")
print("-" * 120)


SOURCE_HASHES_BEFORE = {}


for path, expected in EXPECTED_SHA.items():

    assert path.exists(), (
        f"STOP: missing frozen artifact:\n{path}"
    )

    actual = sha256(
        path
    )

    assert actual == expected, (
        f"STOP: SHA mismatch\n"
        f"{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    SOURCE_HASHES_BEFORE[path] = actual

    print(
        "✓",
        path.relative_to(ROOT)
    )


# ==================================================================================================
# 3. RECOVERY / OUTPUT GATE
# ==================================================================================================

print("\n" + "-" * 120)
print("FINAL-CLOSURE OUTPUT GATE")
print("-" * 120)


for path in FINAL_OUTPUTS:

    print(
        ("EXISTS : " if path.exists() else "absent : "),
        path.relative_to(ROOT)
    )


assert not any(
    path.exists()
    for path in FINAL_OUTPUTS
), (
    "STOP: one or more B-E3 final closure artifacts already exist. "
    "Do not overwrite or rerun."
)


print(
    "✓ no prior B-E3 final closure artifacts exist"
)


# ==================================================================================================
# 4. LOAD FROZEN RESULT + BENCHMARK
#
# NOTE:
# RAW_RESULT has a .json suffix historically, but its actual content is CSV.
# ==================================================================================================

raw = pd.read_csv(
    RAW_RESULT
)

gt = pd.read_csv(
    BENCHMARK
)


assert len(raw) == 120
assert len(gt) == 120

assert raw["id"].nunique() == 120
assert gt["id"].nunique() == 120


print("\n" + "-" * 120)
print("FROZEN DATA LOAD")
print("-" * 120)

print(
    "B-E3 observations:",
    len(raw)
)

print(
    "Benchmark cases:",
    len(gt)
)


# ==================================================================================================
# 5. ONE-TO-ONE JOIN
# ==================================================================================================

joined = gt.merge(
    raw,
    on="id",
    how="inner",
    suffixes=(
        "_gt",
        "_tr"
    ),
    validate="one_to_one"
)


assert len(joined) == 120


print(
    "✓ one-to-one benchmark/result join verified"
)


# ==================================================================================================
# 6. RESOLVE EXPECTED + PREDICTED VALIDITY
# ==================================================================================================

if "expected_valid_gt" in joined.columns:
    expected_col = "expected_valid_gt"

elif "expected_valid" in joined.columns:
    expected_col = "expected_valid"

else:
    raise AssertionError(
        "STOP: expected_valid not found."
    )


if "accepted" not in joined.columns:
    raise AssertionError(
        "STOP: accepted column not found."
    )


expected_valid = bool_series(
    joined[expected_col]
)

predicted_valid = bool_series(
    joined["accepted"]
)


assert expected_valid.notna().all()
assert predicted_valid.notna().all()


# ==================================================================================================
# 7. RECOMPUTE CONFUSION MATRIX
#
# Convention retained from historical analysis:
#
# Positive = invalid request correctly rejected.
#
# TP = invalid + rejected
# FP = valid + rejected
# FN = invalid + accepted
# TN = valid + accepted
# ==================================================================================================

invalid = ~expected_valid
rejected = ~predicted_valid


TP = int(
    (invalid & rejected).sum()
)

FP = int(
    (expected_valid & rejected).sum()
)

FN = int(
    (invalid & predicted_valid).sum()
)

TN = int(
    (expected_valid & predicted_valid).sum()
)


assert (
    TP,
    FP,
    FN,
    TN
) == (
    50,
    7,
    12,
    51
), (
    "STOP: recomputed confusion matrix does not match frozen B-E3 result."
)


N = len(joined)

VALID_N = int(
    expected_valid.sum()
)

INVALID_N = int(
    invalid.sum()
)


CORRECT = TP + TN

ACCURACY = (
    CORRECT / N
)

VAR = (
    TN / VALID_N
)

IRR = (
    TP / INVALID_N
)

PRECISION = (
    TP / (TP + FP)
)

RECALL = (
    TP / (TP + FN)
)

F1 = (
    2 * PRECISION * RECALL
    / (PRECISION + RECALL)
)


ACC_CI = cp_interval(
    CORRECT,
    N
)

VAR_CI = cp_interval(
    TN,
    VALID_N
)

IRR_CI = cp_interval(
    TP,
    INVALID_N
)


print("\n" + "-" * 120)
print("FINAL RECOMPUTED B-E3 RESULTS")
print("-" * 120)

print(
    "TP =", TP,
    "FP =", FP,
    "FN =", FN,
    "TN =", TN
)

print(
    f"Accuracy : {ACCURACY:.6f} ({ACCURACY*100:.2f}%)"
)

print(
    f"VAR      : {VAR:.6f} ({VAR*100:.2f}%)"
)

print(
    f"IRR      : {IRR:.6f} ({IRR*100:.2f}%)"
)

print(
    f"Precision: {PRECISION:.6f}"
)

print(
    f"Recall   : {RECALL:.6f}"
)

print(
    f"F1       : {F1:.6f}"
)

print(
    f"Accuracy CP95: [{ACC_CI[0]:.6f}, {ACC_CI[1]:.6f}]"
)

print(
    f"VAR CP95     : [{VAR_CI[0]:.6f}, {VAR_CI[1]:.6f}]"
)

print(
    f"IRR CP95     : [{IRR_CI[0]:.6f}, {IRR_CI[1]:.6f}]"
)


# ==================================================================================================
# 8. LLM INVOCATION + PARSE INTERPRETATION
# ==================================================================================================

assert "llm_invoked" in joined.columns
assert "parse_ok" in joined.columns


llm_invoked = bool_series(
    joined["llm_invoked"]
)


assert llm_invoked.notna().all()


INVOKED_N = int(
    llm_invoked.sum()
)

NO_CALL_N = int(
    (~llm_invoked).sum()
)


parse_invoked_raw = joined.loc[
    llm_invoked,
    "parse_ok"
]


parse_invoked = bool_series(
    parse_invoked_raw
)


assert parse_invoked.notna().all()


PARSE_SUCCESS_N = int(
    parse_invoked.sum()
)

PARSE_FAILURE_N = (
    INVOKED_N
    - PARSE_SUCCESS_N
)


parse_no_call = joined.loc[
    ~llm_invoked,
    "parse_ok"
]


assert parse_no_call.isna().all(), (
    "STOP: expected parse_ok=N/A for all non-invoked cases."
)


assert INVOKED_N == 77
assert NO_CALL_N == 43
assert PARSE_SUCCESS_N == 77
assert PARSE_FAILURE_N == 0


assert "schema_valid" not in joined.columns, (
    "Unexpected schema_valid column found. "
    "Reassess before freezing."
)


print("\n" + "-" * 120)
print("FINAL STRUCTURED-GENERATION INTERPRETATION")
print("-" * 120)

print(
    f"LLM invoked     : {INVOKED_N}/{N}"
)

print(
    f"Pre-LLM/no-call : {NO_CALL_N}/{N}"
)

print(
    f"Parse success   : {PARSE_SUCCESS_N}/{INVOKED_N} "
    f"({PARSE_SUCCESS_N/INVOKED_N*100:.2f}% of invoked cases)"
)

print(
    f"Parse failures  : {PARSE_FAILURE_N}/{INVOKED_N}"
)

print(
    "schema_valid    : NOT AVAILABLE IN FROZEN B-E3 RESULT SCHEMA"
)


# ==================================================================================================
# 9. REJECTION-STAGE DISTRIBUTION
# ==================================================================================================

assert "rejection_stage" in joined.columns


rejection_stage = (
    joined["rejection_stage"]
    .fillna(
        "accepted_no_rejection_stage"
    )
)


stage_counts = {
    str(k): int(v)
    for k, v in
    rejection_stage
    .value_counts()
    .items()
}


EXPECTED_STAGE_COUNTS = {
    "accepted_no_rejection_stage": 63,
    "source_gate": 24,
    "constraint_consistency": 14,
    "cir_validation": 12,
    "r7_qualitative_source_gate": 5,
    "preservation": 2,
}


assert stage_counts == EXPECTED_STAGE_COUNTS, (
    "STOP: rejection-stage distribution changed."
)


print("\n" + "-" * 120)
print("FINAL REJECTION-STAGE DISTRIBUTION")
print("-" * 120)


for stage, count in EXPECTED_STAGE_COUNTS.items():

    print(
        f"{stage:31s}: {count}"
    )


# ==================================================================================================
# 10. C2-D3 RETROSPECTIVE CLEANER FINDING
#
# C2-D3 established:
#   - source implementations differ
#   - AST implementations differ
#   - corrected historical execution includes required `re`
#   - 0/77 actual cleaner-output differences
#   - 0/77 JSON failures for both cleaners
#   - 0/77 parsed-JSON semantic differences
#
# This is recorded transparently as POST-EXECUTION RETROSPECTIVE ANALYSIS.
# ==================================================================================================

CLEANER_FINDING = {
    "timing":
        "RETROSPECTIVE_POST_EXECUTION",

    "reason":
        (
            "B-E3 treatment cleaner was manually implemented rather than "
            "being verified as source-identical to the historical cleaner "
            "before benchmark execution."
        ),

    "source_implementation_equivalent":
        False,

    "ast_implementation_equivalent":
        False,

    "corrected_behavioral_analysis":
        {
            "actual_llm_outputs_compared": 77,
            "cleaned_text_differences": 0,
            "historical_cleaner_json_failures": 0,
            "b_e3_cleaner_json_failures": 0,
            "parsed_json_semantic_differences": 0,
        },

    "diagnostic_correction":
        (
            "An earlier retrospective diagnostic (C2-D2) omitted Python's "
            "`re` module from the isolated historical-cleaner namespace, "
            "causing diagnostic-only NameError failures. C2-D3 corrected "
            "the namespace and repeated the comparison."
        ),

    "interpretation":
        (
            "The cleaner implementations are not source/AST identical, "
            "but the corrected retrospective comparison found identical "
            "cleaned text and identical parsed JSON on all 77 actually "
            "observed frozen B-E3 LLM responses. Therefore no observed "
            "B-E3 case was affected by the cleaner implementation difference."
        ),

    "claim_limit":
        (
            "This is an observed behavioral-equivalence result for the 77 "
            "frozen B-E3 model responses only; it is not a universal proof "
            "that the two cleaner implementations are behaviorally equivalent "
            "for arbitrary strings."
        ),
}


print("\n" + "-" * 120)
print("RETROSPECTIVE CLEANER CLOSURE")
print("-" * 120)

print(
    "Source implementation equivalent : False"
)

print(
    "AST implementation equivalent    : False"
)

print(
    "Actual outputs compared           : 77"
)

print(
    "Actual cleaned-text differences   : 0"
)

print(
    "Historical JSON failures          : 0"
)

print(
    "B-E3 JSON failures                : 0"
)

print(
    "Parsed-JSON semantic differences  : 0"
)

print(
    "✓ cleaner difference did not affect any observed frozen B-E3 response"
)


# ==================================================================================================
# 11. BUILD CASE-LEVEL ANALYSIS
# ==================================================================================================

case_df = pd.DataFrame({
    "id":
        joined["id"].astype(str),

    "expected_valid":
        expected_valid.astype(bool),

    "predicted_valid":
        predicted_valid.astype(bool),

    "correct_decision":
        (
            expected_valid
            == predicted_valid
        ).astype(bool),

    "llm_invoked":
        llm_invoked.astype(bool),

    "parse_status":
        [
            (
                "PASS"
                if invoked and bool(parse_ok)
                else
                "FAIL"
                if invoked
                else
                "N/A"
            )
            for invoked, parse_ok in zip(
                llm_invoked,
                bool_series(joined["parse_ok"]).fillna(False)
            )
        ],

    "rejection_stage":
        rejection_stage.astype(str),
})


if "category_gt" in joined.columns:

    case_df["category"] = (
        joined["category_gt"]
        .astype(str)
    )

elif "category" in joined.columns:

    case_df["category"] = (
        joined["category"]
        .astype(str)
    )


assert len(case_df) == 120

assert int(
    case_df["correct_decision"].sum()
) == 101


# ==================================================================================================
# 12. BUILD FINAL RESULTS TABLE
# ==================================================================================================

results_df = pd.DataFrame([
    {
        "experiment":
            "B-E3 supplementary structured-generation treatment",

        "benchmark":
            "E3-v3 frozen held-out benchmark",

        "model":
            "Qwen/Qwen3.5-9B",

        "n":
            N,

        "valid_n":
            VALID_N,

        "invalid_n":
            INVALID_N,

        "tp_invalid_rejected":
            TP,

        "fp_valid_rejected":
            FP,

        "fn_invalid_accepted":
            FN,

        "tn_valid_accepted":
            TN,

        "accuracy":
            ACCURACY,

        "accuracy_pct":
            ACCURACY * 100,

        "accuracy_cp95_low":
            ACC_CI[0],

        "accuracy_cp95_high":
            ACC_CI[1],

        "valid_acceptance_rate":
            VAR,

        "valid_acceptance_pct":
            VAR * 100,

        "var_cp95_low":
            VAR_CI[0],

        "var_cp95_high":
            VAR_CI[1],

        "invalid_rejection_rate":
            IRR,

        "invalid_rejection_pct":
            IRR * 100,

        "irr_cp95_low":
            IRR_CI[0],

        "irr_cp95_high":
            IRR_CI[1],

        "precision_invalid_rejection":
            PRECISION,

        "recall_invalid_rejection":
            RECALL,

        "f1_invalid_rejection":
            F1,

        "llm_invoked":
            INVOKED_N,

        "pre_llm_no_call":
            NO_CALL_N,

        "parse_success_invoked":
            PARSE_SUCCESS_N,

        "parse_failure_invoked":
            PARSE_FAILURE_N,

        "parse_success_pct_of_invoked":
            (
                PARSE_SUCCESS_N
                / INVOKED_N
                * 100
            ),

        "schema_valid_metric":
            "NOT_AVAILABLE",

        "role":
            "SUPPLEMENTARY_FOLLOW_UP_NOT_REVIEWER_REQUESTED_B",
    }
])


# ==================================================================================================
# 13. CREATE OUTPUT DIRECTORIES
# ==================================================================================================

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 14. WRITE CASE ANALYSIS
# ==================================================================================================

case_df.to_csv(
    CASE_ANALYSIS,
    index=False
)


print("\n" + "-" * 120)
print("FINAL ARTIFACT CREATION")
print("-" * 120)

print(
    "✓ case analysis written"
)

print(
    " ",
    CASE_ANALYSIS.relative_to(ROOT)
)

print(
    "  SHA-256:",
    sha256(CASE_ANALYSIS)
)


# ==================================================================================================
# 15. WRITE FINAL RESULTS TABLE
# ==================================================================================================

results_df.to_csv(
    RESULTS_TABLE,
    index=False
)


print(
    "✓ final results table written"
)

print(
    " ",
    RESULTS_TABLE.relative_to(ROOT)
)

print(
    "  SHA-256:",
    sha256(RESULTS_TABLE)
)


# ==================================================================================================
# 16. FINAL SUMMARY
# ==================================================================================================

summary = {

    "experiment_id":
        "B-E3",

    "status":
        "FINAL_FROZEN",

    "role":
        "SUPPLEMENTARY_FOLLOW_UP_NOT_REVIEWER_REQUESTED_B",

    "reviewer_requested_B_satisfied_by":
        (
            "Separate B-Review experiment on frozen E1 (50 cases) "
            "and E2A (200 cases)."
        ),

    "benchmark":
        {
            "name":
                "E3-v3 frozen held-out benchmark",

            "n":
                N,

            "valid_n":
                VALID_N,

            "invalid_n":
                INVALID_N,

            "sha256":
                EXPECTED_SHA[BENCHMARK],
        },

    "model":
        "Qwen/Qwen3.5-9B",

    "decision_results":
        {
            "tp_invalid_rejected":
                TP,

            "fp_valid_rejected":
                FP,

            "fn_invalid_accepted":
                FN,

            "tn_valid_accepted":
                TN,

            "accuracy":
                ACCURACY,

            "accuracy_pct":
                ACCURACY * 100,

            "accuracy_cp95":
                list(ACC_CI),

            "valid_acceptance_rate":
                VAR,

            "valid_acceptance_pct":
                VAR * 100,

            "valid_acceptance_cp95":
                list(VAR_CI),

            "invalid_rejection_rate":
                IRR,

            "invalid_rejection_pct":
                IRR * 100,

            "invalid_rejection_cp95":
                list(IRR_CI),

            "precision":
                PRECISION,

            "recall":
                RECALL,

            "f1":
                F1,
        },

    "structured_generation":
        {
            "llm_invoked":
                INVOKED_N,

            "pre_llm_no_call":
                NO_CALL_N,

            "parse_success":
                PARSE_SUCCESS_N,

            "parse_failure":
                PARSE_FAILURE_N,

            "parse_success_pct_of_invoked":
                (
                    PARSE_SUCCESS_N
                    / INVOKED_N
                    * 100
                ),

            "schema_valid_metric":
                "NOT_AVAILABLE_IN_FROZEN_RESULT_SCHEMA",

            "parse_status_for_non_invoked_cases":
                "N/A",
        },

    "rejection_stage_distribution":
        EXPECTED_STAGE_COUNTS,

    "retrospective_cleaner_analysis":
        CLEANER_FINDING,

    "methodological_notes":
        [
            (
                "B-E3 is a supplementary E3-v3 structured-generation "
                "follow-up and must not be represented as the original "
                "reviewer-requested constrained-decoding Experiment B."
            ),

            (
                "The original reviewer-requested Experiment B is the "
                "separately frozen B-Review E1/E2A experiment."
            ),

            (
                "The treatment cleaner was not verified as source-identical "
                "to the historical cleaner before B-E3 execution."
            ),

            (
                "Corrected retrospective C2-D3 analysis found no cleaner-"
                "induced difference on any of the 77 actually observed "
                "B-E3 LLM responses."
            ),

            (
                "No schema_valid metric is reported because that field "
                "does not exist in the frozen B-E3 result schema."
            ),

            (
                "The 43 cases that did not invoke the LLM have parse "
                "status N/A and are not counted as parse failures."
            ),
        ],

    "claim_boundaries":
        [
            "Do not claim universal model independence.",
            "Do not claim hallucination-free behavior.",
            "Do not claim universal cleaner equivalence.",
            "Do not claim formal standards conformance.",
            "Do not claim zero risk.",
            (
                "Do not represent B-E3 as satisfying the original "
                "reviewer-requested Experiment B."
            ),
        ],

    "paper_safe_interpretation":
        (
            "On the frozen 120-case E3-v3 benchmark, the B-E3 "
            "structured-generation treatment achieved 84.17% decision "
            "accuracy, 87.93% valid-acceptance rate, and 80.65% "
            "invalid-rejection rate. All 77 cases that reached the LLM "
            "parsed successfully, while 43 cases were handled without "
            "invoking the LLM. This supplementary result does not replace "
            "the reviewer-requested E1/E2A constrained-decoding experiment "
            "and does not establish universal equivalence across models "
            "or benchmarks."
        ),

    "frozen_source_artifacts":
        {
            str(path.relative_to(ROOT)):
                expected
            for path, expected
            in EXPECTED_SHA.items()
        },
}


json_dump_stable(
    summary,
    FINAL_SUMMARY
)


print(
    "✓ final summary written"
)

print(
    " ",
    FINAL_SUMMARY.relative_to(ROOT)
)

print(
    "  SHA-256:",
    sha256(FINAL_SUMMARY)
)


# ==================================================================================================
# 17. BUILD FINAL FREEZE MANIFEST
# ==================================================================================================

freeze_manifest = {

    "freeze_id":
        "B-E3_FINAL_FREEZE_v1",

    "experiment":
        "B-E3",

    "status":
        "FINAL_FROZEN",

    "role":
        "SUPPLEMENTARY_FOLLOW_UP_NOT_REVIEWER_REQUESTED_B",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_execution":
        {
            "benchmark_cases_executed_in_C2_R2":
                0,

            "api_calls_in_C2_R2":
                0,

            "model_calls_in_C2_R2":
                0,

            "B_E3_rerun":
                False,
        },

    "frozen_inputs":
        {
            str(path.relative_to(ROOT)):
                expected
            for path, expected
            in EXPECTED_SHA.items()
        },

    "final_outputs":
        {
            str(CASE_ANALYSIS.relative_to(ROOT)):
                sha256(CASE_ANALYSIS),

            str(RESULTS_TABLE.relative_to(ROOT)):
                sha256(RESULTS_TABLE),

            str(FINAL_SUMMARY.relative_to(ROOT)):
                sha256(FINAL_SUMMARY),
        },

    "closure_provenance":
        {
            "C2_R1":
                (
                    "Stopped before final artifact creation because "
                    "source/AST cleaner equivalence assertion failed."
                ),

            "C2_D2":
                (
                    "Retrospective diagnostic identified cleaner source "
                    "differences but contained an isolated-namespace defect: "
                    "Python `re` was omitted for the historical cleaner."
                ),

            "C2_D3":
                (
                    "Corrected read-only diagnostic supplied `re`; "
                    "all 77 actual frozen B-E3 responses produced identical "
                    "cleaned text and identical parsed JSON under both cleaners."
                ),

            "C2_R2":
                (
                    "Final analytic closure and freeze only; no model/API "
                    "calls and no benchmark execution."
                ),
        },

    "cleaner_equivalence_scope":
        {
            "source_equivalence":
                False,

            "ast_equivalence":
                False,

            "observed_behavioral_equivalence":
                True,

            "observed_outputs_compared":
                77,

            "scope":
                (
                    "Observed frozen B-E3 responses only; "
                    "not a universal equivalence proof."
                ),
        },

    "finality":
        {
            "rerun_prohibited":
                True,

            "modify_frozen_inputs_prohibited":
                True,

            "future_use":
                (
                    "Read-only analysis, publication tables, "
                    "paper text, and reproducibility packaging."
                ),
        },
}


json_dump_stable(
    freeze_manifest,
    FINAL_FREEZE
)


print(
    "✓ FINAL FREEZE manifest written"
)

print(
    " ",
    FINAL_FREEZE.relative_to(ROOT)
)

print(
    "  SHA-256:",
    sha256(FINAL_FREEZE)
)


# ==================================================================================================
# 18. PERSIST FINAL ARTIFACTS TO DRIVE
# ==================================================================================================

print("\n" + "-" * 120)
print("PERSISTENCE TO GOOGLE DRIVE")
print("-" * 120)


PERSIST_MAP = {
    CASE_ANALYSIS:
        (
            PERSIST_ROOT
            / "analysis"
            / CASE_ANALYSIS.name
        ),

    RESULTS_TABLE:
        (
            PERSIST_ROOT
            / "final"
            / RESULTS_TABLE.name
        ),

    FINAL_SUMMARY:
        (
            PERSIST_ROOT
            / "final"
            / FINAL_SUMMARY.name
        ),

    FINAL_FREEZE:
        (
            PERSIST_ROOT
            / "manifests"
            / FINAL_FREEZE.name
        ),
}


for src, dst in PERSIST_MAP.items():

    dst.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    assert not dst.exists(), (
        f"STOP: persistent destination already exists:\n{dst}"
    )

    shutil.copy2(
        src,
        dst
    )

    assert sha256(src) == sha256(dst)

    print(
        "✓ persisted:",
        dst
    )

    print(
        "  SHA-256:",
        sha256(dst)
    )


# ==================================================================================================
# 19. VERIFY ALL FROZEN SOURCES REMAIN UNCHANGED
# ==================================================================================================

print("\n" + "-" * 120)
print("POST-CLOSURE FROZEN SOURCE INTEGRITY")
print("-" * 120)


for path, expected in EXPECTED_SHA.items():

    actual = sha256(
        path
    )

    assert actual == expected

    assert actual == SOURCE_HASHES_BEFORE[path]

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


# ==================================================================================================
# 20. VERIFY FINAL ARTIFACTS
# ==================================================================================================

print("\n" + "-" * 120)
print("FINAL B-E3 ARTIFACT HASHES")
print("-" * 120)


print(
    "Case analysis :",
    sha256(CASE_ANALYSIS)
)

print(
    "Results table :",
    sha256(RESULTS_TABLE)
)

print(
    "Final summary :",
    sha256(FINAL_SUMMARY)
)

print(
    "FINAL FREEZE  :",
    sha256(FINAL_FREEZE)
)


# ==================================================================================================
# 21. FINAL SCIENTIFIC SUMMARY
# ==================================================================================================

print("\n" + "-" * 120)
print("B-E3 FINAL SCIENTIFIC SUMMARY")
print("-" * 120)


print(
    "Benchmark       : E3-v3 frozen held-out benchmark"
)

print(
    "Cases           : 120 (58 valid / 62 invalid)"
)

print(
    "Model           : Qwen/Qwen3.5-9B"
)

print(
    "Accuracy        : 84.17%"
)

print(
    "Valid acceptance: 87.93%"
)

print(
    "Invalid rejection: 80.65%"
)

print(
    "LLM invoked     : 77/120"
)

print(
    "Parse success   : 77/77 invoked cases"
)

print(
    "Pre-LLM/no-call : 43/120"
)

print(
    "schema_valid    : not available in frozen B-E3 schema"
)

print(
    "Cleaner finding : implementations differ, but 0/77 "
    "observed behavioral differences after corrected retrospective analysis"
)

print(
    "Experiment role : supplementary B-E3 follow-up"
)

print(
    "Reviewer B role : satisfied separately by frozen B-Review E1/E2A experiment"
)


# ==================================================================================================
# COMPLETE
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "✓ STEP C2-R2 COMPLETE"
)

print(
    "✓ B-E3 ANALYTIC CLOSURE COMPLETE"
)

print(
    "✓ B-E3 FINAL FREEZE CREATED"
)

print(
    "✓ C2-D3 CLEANER FINDING RECORDED TRANSPARENTLY"
)

print(
    "✓ 77/77 INVOKED CASES PARSED SUCCESSFULLY"
)

print(
    "✓ 43 NON-INVOKED CASES RETAINED AS PARSE N/A"
)

print(
    "✓ NO schema_valid METRIC INVENTED"
)

print(
    "✓ ZERO API/MODEL CALLS"
)

print(
    "✓ ZERO BENCHMARK EXECUTIONS"
)

print(
    "✓ FROZEN SOURCE ARTIFACTS UNCHANGED"
)

print(
    "✓ FINAL ARTIFACTS PERSISTED BYTE-FOR-BYTE TO DRIVE"
)

print(
    "✓ DO NOT RERUN OR MODIFY B-E3"
)

print(
    "✓ SEND THIS COMPLETE OUTPUT BEFORE STEP C3"
)

print("=" * 120)

JOURNAL CONSOLIDATION — STEP C2-R2: B-E3 FINAL ANALYTIC CLOSURE AND FREEZE

------------------------------------------------------------------------------------------------------------------------
FROZEN PROVENANCE
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_B_schema_constrained_decoding/preregistration/experiment_B_preregistration_v1.json
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/experiment_B_structured_adapter_v1.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/implementation/run_experiment_B_structured_treatment_v1.py
✓ journal_experiments/experiment_B_schema_constrained_decoding/results/treatment_B1/experiment_B_B1_structured_treatment_120_v1.json
✓ journal_experiments/experiment_B_schema_constrained_decoding/manifests/experiment_B_B9B_execution_record_v1.json
✓ src/llm/together_adapter.py
✓ experiment-2b/reproducibility/

/tmp/ipykernel_3091/1226660349.py:880: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bool_series(joined["parse_ok"]).fillna(False)


✓ persisted: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/journal_experiments_persistent/experiment_B_schema_constrained_decoding/analysis/experiment_B_E3_final_case_analysis_v1.csv
  SHA-256: 6663345076357ffae72e95fb9dc71da701161feee9ae03b7c217ba39237c9e5c
✓ persisted: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/journal_experiments_persistent/experiment_B_schema_constrained_decoding/final/experiment_B_E3_FINAL_results_table_v1.csv
  SHA-256: 10e0599c30630c6bfa6523be546bff50ffea2489aa04c380355f5802c229d403
✓ persisted: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/journal_experiments_persistent/experiment_B_schema_constrained_decoding/final/experiment_B_E3_FINAL_summary_v1.json
  SHA-256: 89842933a3db43cdc447a350838a92ed32a5c1b40f129a84bc1433a4e5691bab
✓ persisted: /content/drive/MyDrive/PhD_Research/Deterministic_Intent_Translation/journal_experiments_persistent/experiment_B_schema_constrained_decoding/manifests/e

In [67]:
# ==================================================================================================
# JOURNAL CONSOLIDATION — STEP C3
# MASTER FROZEN-EVIDENCE CONSOLIDATION
#
# Purpose:
#   Create one authoritative journal-level evidence manifest covering:
#
#       Experiment A
#       B-Review (reviewer-requested Experiment B)
#       B-E3 (supplementary follow-up)
#
# IMPORTANT:
#   - ZERO API / MODEL CALLS
#   - ZERO BENCHMARK EXECUTIONS
#   - DO NOT MODIFY ANY EXPERIMENT
#   - DO NOT RECONSTRUCT LOST RAW EVIDENCE
#   - DO NOT CREATE PUBLICATION TABLES YET
#   - HASH/REFERENCE AUTHORITATIVE FINAL ARTIFACTS ONLY
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil


print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C3: MASTER FROZEN-EVIDENCE CONSOLIDATION")
print("=" * 120)


# ==================================================================================================
# 0. ROOTS
# ==================================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

JEXP = (
    ROOT
    / "journal_experiments"
)

MASTER = (
    JEXP
    / "journal_final_evidence"
)

MASTER_MANIFEST_DIR = (
    MASTER
    / "manifests"
)

PERSIST_MASTER = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "journal_final_evidence"
)

PERSIST_MANIFEST_DIR = (
    PERSIST_MASTER
    / "manifests"
)


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def stable_json(
    obj,
    path
):

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        )
        + "\n",
        encoding="utf-8"
    )


# ==================================================================================================
# 2. C1 INVENTORY
# ==================================================================================================

C1 = (
    MASTER
    / "inventory"
    / "C1_frozen_evidence_inventory_v1.json"
)

C1_SHA = (
    "550446bdd7f70481e0eb1a512af4e8d3fddb62eec7272f227d1c192514e7ced2"
)


# ==================================================================================================
# 3. EXPERIMENT A — AUTHORITATIVE FINAL FREEZE
# ==================================================================================================

A_ROOT = (
    JEXP
    / "experiment_A_model_sweep"
)

A_FREEZE = (
    A_ROOT
    / "manifests"
    / "experiment_A_FINAL_FREEZE_v1.json"
)

A_FREEZE_SHA = (
    "667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a"
)


# ==================================================================================================
# 4. B-REVIEW — AUTHORITATIVE FINAL ARTIFACTS
# ==================================================================================================

BR_ROOT = (
    JEXP
    / "experiment_B_review_requested_constrained_decoding"
)


# Locate BR13 artifacts by exact known names where possible,
# otherwise bounded recursive lookup is permitted only within BR_ROOT.

BR_RESULTS_NAME = (
    "experiment_B_review_FINAL_results_table_v1.csv"
)

BR_SUMMARY_NAME = (
    "experiment_B_review_FINAL_summary_v1.json"
)

BR_FREEZE_NAME = (
    "experiment_B_review_FINAL_FREEZE_v1.json"
)


def find_unique(root, filename):

    matches = list(
        root.rglob(filename)
    )

    assert len(matches) == 1, (
        f"STOP: expected exactly one {filename} under {root}; "
        f"found {len(matches)}:\n"
        + "\n".join(str(x) for x in matches)
    )

    return matches[0]


BR_RESULTS = find_unique(
    BR_ROOT,
    BR_RESULTS_NAME
)

BR_SUMMARY = find_unique(
    BR_ROOT,
    BR_SUMMARY_NAME
)

BR_FREEZE = find_unique(
    BR_ROOT,
    BR_FREEZE_NAME
)


BR_RESULTS_SHA = (
    "4d362b0b0862cf79b0acd189c843ce854a4ce2da64bfd5f95091ba98b1d3bf89"
)

BR_SUMMARY_SHA = (
    "a3a9539f8dade42a4f75ec4f28d4a9421937edbcb79d41091c3f7d374d22c8eb"
)

BR_FREEZE_SHA = (
    "63fa99c81477eb7d2c3136b67f72ac3d4673f7d081d7b332ef8d634c68faf3dd"
)


# ==================================================================================================
# 5. B-E3 — AUTHORITATIVE FINAL ARTIFACTS
# ==================================================================================================

BE3_ROOT = (
    JEXP
    / "experiment_B_schema_constrained_decoding"
)

BE3_CASE = (
    BE3_ROOT
    / "analysis"
    / "experiment_B_E3_final_case_analysis_v1.csv"
)

BE3_RESULTS = (
    BE3_ROOT
    / "final"
    / "experiment_B_E3_FINAL_results_table_v1.csv"
)

BE3_SUMMARY = (
    BE3_ROOT
    / "final"
    / "experiment_B_E3_FINAL_summary_v1.json"
)

BE3_FREEZE = (
    BE3_ROOT
    / "manifests"
    / "experiment_B_E3_FINAL_FREEZE_v1.json"
)


BE3_CASE_SHA = (
    "6663345076357ffae72e95fb9dc71da701161feee9ae03b7c217ba39237c9e5c"
)

BE3_RESULTS_SHA = (
    "10e0599c30630c6bfa6523be546bff50ffea2489aa04c380355f5802c229d403"
)

BE3_SUMMARY_SHA = (
    "89842933a3db43cdc447a350838a92ed32a5c1b40f129a84bc1433a4e5691bab"
)

BE3_FREEZE_SHA = (
    "e3e5dee967eeb233f1e583624d204b26c648040938715d41ff7cd4102d6d77bb"
)


# ==================================================================================================
# 6. VERIFY AUTHORITATIVE INPUTS
# ==================================================================================================

AUTHORITATIVE = {

    "C1_inventory": {
        "path": C1,
        "sha256": C1_SHA,
    },

    "experiment_A_final_freeze": {
        "path": A_FREEZE,
        "sha256": A_FREEZE_SHA,
    },

    "B_review_results": {
        "path": BR_RESULTS,
        "sha256": BR_RESULTS_SHA,
    },

    "B_review_summary": {
        "path": BR_SUMMARY,
        "sha256": BR_SUMMARY_SHA,
    },

    "B_review_final_freeze": {
        "path": BR_FREEZE,
        "sha256": BR_FREEZE_SHA,
    },

    "B_E3_case_analysis": {
        "path": BE3_CASE,
        "sha256": BE3_CASE_SHA,
    },

    "B_E3_results": {
        "path": BE3_RESULTS,
        "sha256": BE3_RESULTS_SHA,
    },

    "B_E3_summary": {
        "path": BE3_SUMMARY,
        "sha256": BE3_SUMMARY_SHA,
    },

    "B_E3_final_freeze": {
        "path": BE3_FREEZE,
        "sha256": BE3_FREEZE_SHA,
    },
}


print("\n" + "-" * 120)
print("AUTHORITATIVE FROZEN-EVIDENCE VERIFICATION")
print("-" * 120)


INPUT_HASHES_BEFORE = {}


for label, item in AUTHORITATIVE.items():

    path = item["path"]

    assert path.exists(), (
        f"STOP: missing authoritative artifact:\n{path}"
    )

    actual = sha256(
        path
    )

    expected = item[
        "sha256"
    ]

    assert actual == expected, (
        f"STOP: authoritative hash mismatch\n"
        f"Label:    {label}\n"
        f"Path:     {path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    INPUT_HASHES_BEFORE[
        path
    ] = actual

    print(
        f"✓ {label}"
    )

    print(
        " ",
        path.relative_to(ROOT)
    )

    print(
        "  SHA-256:",
        actual
    )


# ==================================================================================================
# 7. VERIFY EXPERIMENT ROLES
# ==================================================================================================

print("\n" + "-" * 120)
print("EVIDENCE-STREAM ROLE SEPARATION")
print("-" * 120)


with open(
    BR_FREEZE,
    "r",
    encoding="utf-8"
) as f:

    br_freeze_data = json.load(
        f
    )


with open(
    BE3_FREEZE,
    "r",
    encoding="utf-8"
) as f:

    be3_freeze_data = json.load(
        f
    )


# B-E3 must remain supplementary.
be3_text = json.dumps(
    be3_freeze_data
).upper()


assert (
    "SUPPLEMENTARY" in be3_text
), (
    "STOP: B-E3 final freeze does not clearly identify "
    "the experiment as supplementary."
)


print(
    "✓ Experiment A:"
)

print(
    "  model-sweep robustness / containment evidence"
)

print(
    "✓ B-Review:"
)

print(
    "  reviewer-requested constrained-decoding experiment "
    "on E1/E2A"
)

print(
    "✓ B-E3:"
)

print(
    "  supplementary structured-generation follow-up on E3-v3"
)

print(
    "✓ B-E3 is NOT reclassified as the reviewer-requested B experiment"
)


# ==================================================================================================
# 8. MASTER MANIFEST OUTPUT GATE
# ==================================================================================================

MASTER_MANIFEST = (
    MASTER_MANIFEST_DIR
    / "journal_master_frozen_evidence_manifest_v1.json"
)

PERSIST_MASTER_MANIFEST = (
    PERSIST_MANIFEST_DIR
    / MASTER_MANIFEST.name
)


print("\n" + "-" * 120)
print("MASTER OUTPUT GATE")
print("-" * 120)


print(
    "Working manifest exists:",
    MASTER_MANIFEST.exists()
)

print(
    "Persistent manifest exists:",
    PERSIST_MASTER_MANIFEST.exists()
)


assert not MASTER_MANIFEST.exists(), (
    "STOP: master manifest already exists. "
    "Do not overwrite."
)

assert not PERSIST_MASTER_MANIFEST.exists(), (
    "STOP: persistent master manifest already exists. "
    "Do not overwrite."
)


print(
    "✓ master manifest paths are unused"
)


# ==================================================================================================
# 9. BUILD MASTER EVIDENCE MANIFEST
# ==================================================================================================

manifest = {

    "manifest_id":
        "journal_master_frozen_evidence_manifest_v1",

    "status":
        "FROZEN_EVIDENCE_CONSOLIDATED",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "purpose":
        (
            "Authoritative journal-level consolidation of the frozen "
            "Experiment A, reviewer-requested B-Review, and supplementary "
            "B-E3 evidence streams. This manifest references existing "
            "authoritative artifacts and does not reconstruct or modify "
            "experimental evidence."
        ),

    "scientific_execution_during_C3":
        {
            "api_calls": 0,
            "model_calls": 0,
            "benchmark_executions": 0,
            "experiment_reruns": 0,
        },

    "evidence_streams":
        {

            "experiment_A": {
                "role":
                    (
                        "Multi-model robustness and deterministic "
                        "failure-containment evidence."
                    ),

                "status":
                    "FINAL_FROZEN",

                "final_freeze": {
                    "path":
                        str(
                            A_FREEZE.relative_to(ROOT)
                        ),

                    "sha256":
                        A_FREEZE_SHA,
                },

                "paper_scope":
                    (
                        "Six evaluable semantic-parser models on the "
                        "frozen 120-case E3-v3 benchmark."
                    ),

                "claim_boundary":
                    (
                        "Supports observed cross-model stability of the "
                        "tested deterministic containment boundary; does "
                        "not establish universal model independence."
                    ),
            },


            "B_review": {
                "role":
                    (
                        "Original reviewer-requested constrained-decoding "
                        "experiment."
                    ),

                "status":
                    "FINAL_FROZEN",

                "benchmarks":
                    [
                        "E1 frozen 50-case benchmark",
                        "E2A frozen 200-case benchmark",
                    ],

                "results": {
                    "path":
                        str(
                            BR_RESULTS.relative_to(ROOT)
                        ),

                    "sha256":
                        BR_RESULTS_SHA,
                },

                "summary": {
                    "path":
                        str(
                            BR_SUMMARY.relative_to(ROOT)
                        ),

                    "sha256":
                        BR_SUMMARY_SHA,
                },

                "final_freeze": {
                    "path":
                        str(
                            BR_FREEZE.relative_to(ROOT)
                        ),

                    "sha256":
                        BR_FREEZE_SHA,
                },

                "paper_scope":
                    (
                        "Tests whether provider-side structural constrained "
                        "decoding resolves malformed output while examining "
                        "whether semantic omission, mis-mapping, invalid-range "
                        "values, and unsupported requirements remain."
                    ),

                "claim_boundary":
                    (
                        "Provider-side structural decoding is distinguished "
                        "from downstream semantic validation; no universal "
                        "structured-output guarantee is claimed."
                    ),
            },


            "B_E3": {
                "role":
                    (
                        "Supplementary E3-v3 structured-generation "
                        "follow-up; not the reviewer-requested Experiment B."
                    ),

                "status":
                    "FINAL_FROZEN",

                "benchmark":
                    "E3-v3 frozen 120-case benchmark",

                "case_analysis": {
                    "path":
                        str(
                            BE3_CASE.relative_to(ROOT)
                        ),

                    "sha256":
                        BE3_CASE_SHA,
                },

                "results": {
                    "path":
                        str(
                            BE3_RESULTS.relative_to(ROOT)
                        ),

                    "sha256":
                        BE3_RESULTS_SHA,
                },

                "summary": {
                    "path":
                        str(
                            BE3_SUMMARY.relative_to(ROOT)
                        ),

                    "sha256":
                        BE3_SUMMARY_SHA,
                },

                "final_freeze": {
                    "path":
                        str(
                            BE3_FREEZE.relative_to(ROOT)
                        ),

                    "sha256":
                        BE3_FREEZE_SHA,
                },

                "cleaner_provenance":
                    {
                        "source_equivalent":
                            False,

                        "ast_equivalent":
                            False,

                        "observed_behavioral_equivalence":
                            True,

                        "observed_outputs_compared":
                            77,

                        "observed_cleaned_text_differences":
                            0,

                        "scope":
                            (
                                "Retrospective corrected behavioral "
                                "comparison on the 77 actually observed "
                                "frozen B-E3 LLM responses only."
                            ),
                    },

                "claim_boundary":
                    (
                        "Supplementary evidence only; does not replace "
                        "the frozen B-Review E1/E2A experiment and does "
                        "not establish universal cleaner or model equivalence."
                    ),
            },
        },

    "C1_inventory": {
        "path":
            str(
                C1.relative_to(ROOT)
            ),

        "sha256":
            C1_SHA,
    },

    "global_claim_boundaries":
        [
            "No hallucination-free claim.",
            "No universal model-independence claim.",
            "No universal cleaner-equivalence claim.",
            "No zero-risk claim.",
            "No formal 3GPP/O-RAN conformance claim.",
            (
                "No reconstruction or fabrication of raw evidence "
                "lost during earlier ephemeral Colab reset."
            ),
            (
                "B-E3 must remain explicitly supplementary and must "
                "not be represented as satisfying the original "
                "reviewer-requested Experiment B."
            ),
        ],

    "next_allowed_operations":
        [
            "Read-only publication-table derivation.",
            "Read-only paper-results drafting.",
            "Read-only reproducibility packaging.",
            "Final journal evidence freeze.",
        ],

    "prohibited_operations":
        [
            "Rerun Experiment A.",
            "Add new Experiment A model arms.",
            "Rerun B-Review E1 treatment.",
            "Rerun B-Review E2A treatment.",
            "Rerun B-E3.",
            "Modify frozen experimental results.",
            "Reconstruct lost raw evidence.",
            "Silently change preregistered procedures after outcomes.",
        ],
}


# ==================================================================================================
# 10. WRITE MASTER MANIFEST
# ==================================================================================================

MASTER_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


stable_json(
    manifest,
    MASTER_MANIFEST
)


MASTER_SHA = sha256(
    MASTER_MANIFEST
)


print("\n" + "-" * 120)
print("MASTER MANIFEST CREATED")
print("-" * 120)


print(
    MASTER_MANIFEST
)

print(
    "SHA-256:",
    MASTER_SHA
)


# ==================================================================================================
# 11. PERSIST BYTE-FOR-BYTE TO DRIVE
# ==================================================================================================

PERSIST_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


shutil.copy2(
    MASTER_MANIFEST,
    PERSIST_MASTER_MANIFEST
)


assert (
    sha256(MASTER_MANIFEST)
    ==
    sha256(PERSIST_MASTER_MANIFEST)
)


print("\n" + "-" * 120)
print("PERSISTENCE")
print("-" * 120)


print(
    "✓ persisted byte-for-byte:"
)

print(
    PERSIST_MASTER_MANIFEST
)

print(
    "SHA-256:",
    sha256(PERSIST_MASTER_MANIFEST)
)


# ==================================================================================================
# 12. POST-CONSOLIDATION SOURCE INTEGRITY
# ==================================================================================================

print("\n" + "-" * 120)
print("POST-CONSOLIDATION AUTHORITATIVE SOURCE INTEGRITY")
print("-" * 120)


for label, item in AUTHORITATIVE.items():

    path = item[
        "path"
    ]

    actual = sha256(
        path
    )

    assert (
        actual
        ==
        INPUT_HASHES_BEFORE[path]
    )

    assert (
        actual
        ==
        item["sha256"]
    )

    print(
        "✓ unchanged:",
        label
    )


# ==================================================================================================
# 13. FINAL MASTER-MANIFEST SELF-CHECK
# ==================================================================================================

with open(
    MASTER_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    check = json.load(
        f
    )


assert (
    check["status"]
    ==
    "FROZEN_EVIDENCE_CONSOLIDATED"
)

assert (
    check["evidence_streams"]["experiment_A"]["status"]
    ==
    "FINAL_FROZEN"
)

assert (
    check["evidence_streams"]["B_review"]["status"]
    ==
    "FINAL_FROZEN"
)

assert (
    check["evidence_streams"]["B_E3"]["status"]
    ==
    "FINAL_FROZEN"
)

assert (
    "Supplementary"
    in
    check["evidence_streams"]["B_E3"]["role"]
)


print("\n" + "-" * 120)
print("MASTER EVIDENCE STATUS")
print("-" * 120)


print(
    "Experiment A : FINAL_FROZEN"
)

print(
    "B-Review     : FINAL_FROZEN"
)

print(
    "B-E3         : FINAL_FROZEN / SUPPLEMENTARY"
)

print(
    "C1 inventory : VERIFIED"
)

print(
    "Master SHA   :",
    MASTER_SHA
)


# ==================================================================================================
# COMPLETE
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "✓ STEP C3 COMPLETE"
)

print(
    "✓ ALL THREE EVIDENCE STREAMS CONSOLIDATED"
)

print(
    "✓ EXPERIMENT A REMAINS FINAL FROZEN"
)

print(
    "✓ B-REVIEW REMAINS FINAL FROZEN"
)

print(
    "✓ B-E3 REMAINS FINAL FROZEN AND SUPPLEMENTARY"
)

print(
    "✓ NO EXPERIMENT WAS RERUN"
)

print(
    "✓ ZERO API/MODEL CALLS"
)

print(
    "✓ ZERO BENCHMARK EXECUTIONS"
)

print(
    "✓ AUTHORITATIVE ARTIFACTS UNCHANGED"
)

print(
    "✓ MASTER MANIFEST PERSISTED BYTE-FOR-BYTE TO DRIVE"
)

print(
    "✓ NEXT STEP AFTER REVIEW: C4 PUBLICATION TABLES"
)

print(
    "✓ SEND THIS COMPLETE OUTPUT BEFORE C4"
)

print("=" * 120)

JOURNAL CONSOLIDATION — STEP C3: MASTER FROZEN-EVIDENCE CONSOLIDATION

------------------------------------------------------------------------------------------------------------------------
AUTHORITATIVE FROZEN-EVIDENCE VERIFICATION
------------------------------------------------------------------------------------------------------------------------
✓ C1_inventory
  journal_experiments/journal_final_evidence/inventory/C1_frozen_evidence_inventory_v1.json
  SHA-256: 550446bdd7f70481e0eb1a512af4e8d3fddb62eec7272f227d1c192514e7ced2
✓ experiment_A_final_freeze
  journal_experiments/experiment_A_model_sweep/manifests/experiment_A_FINAL_FREEZE_v1.json
  SHA-256: 667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a
✓ B_review_results
  journal_experiments/experiment_B_review_requested_constrained_decoding/final/experiment_B_review_FINAL_results_table_v1.csv
  SHA-256: 4d362b0b0862cf79b0acd189c843ce854a4ce2da64bfd5f95091ba98b1d3bf89
✓ B_review_summary
  journal_experiments/expe

In [68]:
# ==================================================================================================
# JOURNAL CONSOLIDATION — STEP C4
# DERIVE PUBLICATION-READY TABLES FROM FROZEN EVIDENCE
#
# Produces:
#   T1 — Experiment A six-model robustness table
#   T2 — Reviewer-requested B-Review constrained-decoding results
#   T3 — E2A constrained vs historical CIR comparison
#   T4 — E2A paired correctness comparison
#   T5 — Unsupported-requirement forensic analysis
#   T6 — Supplementary B-E3 results
#
# IMPORTANT:
#   - READ FROZEN EVIDENCE ONLY
#   - ZERO API / MODEL CALLS
#   - ZERO BENCHMARK EXECUTIONS
#   - NO EXPERIMENT RERUN
#   - NO MANUAL RECONSTRUCTION OF LOST RAW DATA
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import pandas as pd

print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C4: PUBLICATION TABLES FROM FROZEN EVIDENCE")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")
JEXP = ROOT / "journal_experiments"

MASTER = JEXP / "journal_final_evidence"
TABLE_DIR = MASTER / "publication_tables"
MANIFEST_DIR = MASTER / "manifests"

PERSIST_ROOT = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "journal_final_evidence"
)

PERSIST_TABLE_DIR = PERSIST_ROOT / "publication_tables"
PERSIST_MANIFEST_DIR = PERSIST_ROOT / "manifests"


# ==================================================================================================
# Helpers
# ==================================================================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def stable_json(obj, path):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        ) + "\n",
        encoding="utf-8"
    )


def find_unique(root, filename):
    matches = list(root.rglob(filename))

    assert len(matches) == 1, (
        f"STOP: expected exactly one {filename} under {root}; "
        f"found {len(matches)}:\n"
        + "\n".join(str(x) for x in matches)
    )

    return matches[0]


# ==================================================================================================
# 1. Verify C3 master manifest
# ==================================================================================================

C3_MANIFEST = (
    MANIFEST_DIR
    / "journal_master_frozen_evidence_manifest_v1.json"
)

C3_SHA = (
    "80555895d6b3c3c877455f252c9924ad7c7d35d68ebcc8bed1f74f9622d1884c"
)

assert C3_MANIFEST.exists()
assert sha256(C3_MANIFEST) == C3_SHA

master = load_json(C3_MANIFEST)

assert master["status"] == "FROZEN_EVIDENCE_CONSOLIDATED"

print("\n" + "-" * 120)
print("C3 MASTER EVIDENCE GATE")
print("-" * 120)

print("✓ C3 master manifest verified")
print("  SHA-256:", C3_SHA)


# ==================================================================================================
# 2. Locate frozen evidence
# ==================================================================================================

A_ROOT = JEXP / "experiment_A_model_sweep"

BR_ROOT = (
    JEXP
    / "experiment_B_review_requested_constrained_decoding"
)

BE3_ROOT = (
    JEXP
    / "experiment_B_schema_constrained_decoding"
)


A_FREEZE = (
    A_ROOT
    / "manifests"
    / "experiment_A_FINAL_FREEZE_v1.json"
)

BR_RESULTS = (
    BR_ROOT
    / "final"
    / "experiment_B_review_FINAL_results_table_v1.csv"
)

BR_SUMMARY = (
    BR_ROOT
    / "final"
    / "experiment_B_review_FINAL_summary_v1.json"
)

BR_FREEZE = (
    BR_ROOT
    / "manifests"
    / "experiment_B_review_FINAL_FREEZE_v1.json"
)

BE3_RESULTS = (
    BE3_ROOT
    / "final"
    / "experiment_B_E3_FINAL_results_table_v1.csv"
)

BE3_SUMMARY = (
    BE3_ROOT
    / "final"
    / "experiment_B_E3_FINAL_summary_v1.json"
)

BE3_FREEZE = (
    BE3_ROOT
    / "manifests"
    / "experiment_B_E3_FINAL_FREEZE_v1.json"
)


KNOWN = {
    A_FREEZE:
        "667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a",

    BR_RESULTS:
        "4d362b0b0862cf79b0acd189c843ce854a4ce2da64bfd5f95091ba98b1d3bf89",

    BR_SUMMARY:
        "a3a9539f8dade42a4f75ec4f28d4a9421937edbcb79d41091c3f7d374d22c8eb",

    BR_FREEZE:
        "63fa99c81477eb7d2c3136b67f72ac3d4673f7d081d7b332ef8d634c68faf3dd",

    BE3_RESULTS:
        "10e0599c30630c6bfa6523be546bff50ffea2489aa04c380355f5802c229d403",

    BE3_SUMMARY:
        "89842933a3db43cdc447a350838a92ed32a5c1b40f129a84bc1433a4e5691bab",

    BE3_FREEZE:
        "e3e5dee967eeb233f1e583624d204b26c648040938715d41ff7cd4102d6d77bb",
}


print("\n" + "-" * 120)
print("FROZEN INPUT VERIFICATION")
print("-" * 120)

INPUT_BEFORE = {}

for path, expected in KNOWN.items():
    assert path.exists(), f"Missing frozen evidence: {path}"

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: frozen hash mismatch\n"
        f"{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    INPUT_BEFORE[path] = actual

    print("✓", path.relative_to(ROOT))


# ==================================================================================================
# 3. Output gate
# ==================================================================================================

T1 = TABLE_DIR / "Table_C4_T1_experiment_A_model_robustness.csv"
T2 = TABLE_DIR / "Table_C4_T2_B_review_constrained_results.csv"
T3 = TABLE_DIR / "Table_C4_T3_E2A_constrained_vs_historical_CIR.csv"
T4 = TABLE_DIR / "Table_C4_T4_E2A_paired_correctness.csv"
T5 = TABLE_DIR / "Table_C4_T5_unsupported_requirement_forensics.csv"
T6 = TABLE_DIR / "Table_C4_T6_B_E3_supplementary.csv"

C4_MANIFEST = (
    MANIFEST_DIR
    / "journal_C4_publication_tables_manifest_v1.json"
)

OUTPUTS = [T1, T2, T3, T4, T5, T6, C4_MANIFEST]

print("\n" + "-" * 120)
print("C4 OUTPUT GATE")
print("-" * 120)

for path in OUTPUTS:
    print(
        ("EXISTS : " if path.exists() else "absent : "),
        path.relative_to(ROOT)
    )

assert not any(p.exists() for p in OUTPUTS), (
    "STOP: one or more C4 outputs already exist. "
    "Do not overwrite."
)

print("✓ all C4 output paths unused")


# ==================================================================================================
# 4. Load frozen final summaries
# ==================================================================================================

a_freeze = load_json(A_FREEZE)
br_summary = load_json(BR_SUMMARY)
br_freeze = load_json(BR_FREEZE)
be3_summary = load_json(BE3_SUMMARY)

br_results = pd.read_csv(BR_RESULTS)
be3_results = pd.read_csv(BE3_RESULTS)


# ==================================================================================================
# 5. TABLE T1 — Experiment A
#
# We derive the six evaluable arms from the final A freeze.
# The freeze may store these under different nested labels, so first
# recursively locate dictionaries containing the expected model names.
# ==================================================================================================

EXPECTED_A_MODELS = [
    "Qwen/Qwen3.5-9B",
    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    "arize-ai/qwen-2-1.5b-instruct",
    "Prism-ML/Ternary-Bonsai-27B",
    "deepseek-ai/DeepSeek-V4.1-Flash",
    "openai/gpt-oss-120b",
]


def recursively_find_model_records(obj, found=None):
    if found is None:
        found = {}

    if isinstance(obj, dict):

        # Dictionary itself may be a model record.
        model_value = None

        for key in [
            "model",
            "model_id",
            "model_name",
        ]:
            if key in obj and isinstance(obj[key], str):
                model_value = obj[key]
                break

        if model_value in EXPECTED_A_MODELS:
            found.setdefault(model_value, []).append(obj)

        # Or model may be the dictionary key.
        for key, value in obj.items():

            if key in EXPECTED_A_MODELS and isinstance(value, dict):
                record = dict(value)
                record["_model_from_key"] = key
                found.setdefault(key, []).append(record)

            recursively_find_model_records(value, found)

    elif isinstance(obj, list):

        for value in obj:
            recursively_find_model_records(value, found)

    return found


a_records = recursively_find_model_records(a_freeze)


def metric_from_candidates(record, candidates):
    for key in candidates:
        if key in record and record[key] is not None:
            return record[key]

    # one-level nested search
    for value in record.values():
        if isinstance(value, dict):
            for key in candidates:
                if key in value and value[key] is not None:
                    return value[key]

    return None


A_EXPECTED = {
    "Qwen/Qwen3.5-9B":
        (84.17, 87.93, 80.65),

    "meta-llama/Llama-3.3-70B-Instruct-Turbo":
        (83.33, 82.76, 83.87),

    "arize-ai/qwen-2-1.5b-instruct":
        (60.00, 18.97, 98.39),

    "Prism-ML/Ternary-Bonsai-27B":
        (75.83, 70.69, 80.65),

    "deepseek-ai/DeepSeek-V4.1-Flash":
        (84.17, 87.93, 80.65),

    "openai/gpt-oss-120b":
        (74.17, 65.52, 82.26),
}


# Frozen A41 is the authority for status/provenance.
# Published numerical values are independently locked here against the
# already-frozen final A result values; they are NOT recomputed from lost raw data.

a_rows = []

for model in EXPECTED_A_MODELS:

    assert model in a_records, (
        f"STOP: model {model} not represented in A final freeze."
    )

    expected_acc, expected_var, expected_irr = A_EXPECTED[model]

    a_rows.append({
        "model": model,
        "accuracy_pct": expected_acc,
        "valid_acceptance_rate_pct": expected_var,
        "invalid_rejection_rate_pct": expected_irr,
        "containment_violations": 0,
        "evidence_status": "FINAL_FROZEN",
    })


t1 = pd.DataFrame(a_rows)

assert len(t1) == 6
assert set(t1["model"]) == set(EXPECTED_A_MODELS)
assert (t1["containment_violations"] == 0).all()

assert round(t1["accuracy_pct"].min(), 2) == 60.00
assert round(t1["accuracy_pct"].max(), 2) == 84.17

assert round(
    t1["valid_acceptance_rate_pct"].min(),
    2
) == 18.97

assert round(
    t1["valid_acceptance_rate_pct"].max(),
    2
) == 87.93

assert round(
    t1["invalid_rejection_rate_pct"].min(),
    2
) == 80.65

assert round(
    t1["invalid_rejection_rate_pct"].max(),
    2
) == 98.39


# ==================================================================================================
# 6. TABLE T2 — Reviewer-requested constrained decoding
# ==================================================================================================

# These values are locked against BR13 final frozen evidence.

t2 = pd.DataFrame([
    {
        "benchmark": "E1",
        "n": 50,
        "valid_n": 25,
        "invalid_n": 25,
        "configuration": "cir_constrained",
        "accuracy_pct": 94.00,
        "valid_acceptance_rate_pct": 100.00,
        "invalid_rejection_rate_pct": 88.00,
        "tp_invalid_rejected": 22,
        "fp_valid_rejected": 0,
        "fn_invalid_accepted": 3,
        "tn_valid_accepted": 25,
    },
    {
        "benchmark": "E2A",
        "n": 200,
        "valid_n": 100,
        "invalid_n": 100,
        "configuration": "cir_constrained",
        "accuracy_pct": 73.00,
        "valid_acceptance_rate_pct": 93.00,
        "invalid_rejection_rate_pct": 53.00,
        "tp_invalid_rejected": 53,
        "fp_valid_rejected": 7,
        "fn_invalid_accepted": 47,
        "tn_valid_accepted": 93,
    },
])

assert len(t2) == 2


# ==================================================================================================
# 7. TABLE T3 — E2A constrained vs historical CIR
# ==================================================================================================

t3 = pd.DataFrame([
    {
        "configuration": "Historical CIR",
        "accuracy_pct": 75.50,
        "valid_acceptance_rate_pct": 96.00,
        "invalid_rejection_rate_pct": 55.00,
        "accuracy_delta_vs_historical_pp": 0.00,
        "var_delta_vs_historical_pp": 0.00,
        "irr_delta_vs_historical_pp": 0.00,
    },
    {
        "configuration": "CIR constrained",
        "accuracy_pct": 73.00,
        "valid_acceptance_rate_pct": 93.00,
        "invalid_rejection_rate_pct": 53.00,
        "accuracy_delta_vs_historical_pp": -2.50,
        "var_delta_vs_historical_pp": -3.00,
        "irr_delta_vs_historical_pp": -2.00,
    },
])

assert len(t3) == 2


# ==================================================================================================
# 8. TABLE T4 — paired E2A correctness
# ==================================================================================================

t4 = pd.DataFrame([
    {
        "paired_category": "Both correct",
        "n": 144,
    },
    {
        "paired_category": "Constrained only correct",
        "n": 2,
    },
    {
        "paired_category": "Historical CIR only correct",
        "n": 7,
    },
    {
        "paired_category": "Both wrong",
        "n": 47,
    },
])

assert int(t4["n"].sum()) == 200

# McNemar exact p-value frozen by BR13.
T4_MCNEMAR_P = 0.1796875


# ==================================================================================================
# 9. TABLE T5 — unsupported-requirement forensic analysis
# ==================================================================================================

t5 = pd.DataFrame([
    {
        "forensic_outcome": "Dropped",
        "n": 17,
        "pct_of_unsupported_cases": round(17 / 24 * 100, 2),
    },
    {
        "forensic_outcome": "Mapped to supported field",
        "n": 3,
        "pct_of_unsupported_cases": round(3 / 24 * 100, 2),
    },
    {
        "forensic_outcome": "Preserved as free text",
        "n": 4,
        "pct_of_unsupported_cases": round(4 / 24 * 100, 2),
    },
    {
        "forensic_outcome": "Other / ambiguous",
        "n": 0,
        "pct_of_unsupported_cases": 0.00,
    },
])

assert int(t5["n"].sum()) == 24


# ==================================================================================================
# 10. TABLE T6 — supplementary B-E3
# ==================================================================================================

decision = be3_summary["decision_results"]
structured = be3_summary["structured_generation"]

t6 = pd.DataFrame([
    {
        "benchmark": "E3-v3",
        "n": 120,
        "valid_n": 58,
        "invalid_n": 62,
        "model": "Qwen/Qwen3.5-9B",
        "accuracy_pct": round(float(decision["accuracy_pct"]), 2),
        "valid_acceptance_rate_pct":
            round(float(decision["valid_acceptance_pct"]), 2),
        "invalid_rejection_rate_pct":
            round(float(decision["invalid_rejection_pct"]), 2),
        "llm_invoked": int(structured["llm_invoked"]),
        "pre_llm_no_call": int(structured["pre_llm_no_call"]),
        "parse_success": int(structured["parse_success"]),
        "parse_failure": int(structured["parse_failure"]),
        "schema_valid_metric": structured["schema_valid_metric"],
        "role": "SUPPLEMENTARY",
    }
])

assert float(t6.iloc[0]["accuracy_pct"]) == 84.17
assert float(t6.iloc[0]["valid_acceptance_rate_pct"]) == 87.93
assert float(t6.iloc[0]["invalid_rejection_rate_pct"]) == 80.65
assert int(t6.iloc[0]["llm_invoked"]) == 77
assert int(t6.iloc[0]["parse_success"]) == 77
assert int(t6.iloc[0]["parse_failure"]) == 0


# ==================================================================================================
# 11. Cross-check B-Review final evidence contains expected numerical values
#
# We do not silently trust manual publication transcription.
# Search the frozen BR13 table/summary serialization for the expected values.
# ==================================================================================================

br_blob = (
    BR_RESULTS.read_text(encoding="utf-8")
    +
    json.dumps(br_summary, sort_keys=True)
)

# Required final quantities must be represented in the frozen evidence.
required_tokens = [
    "0.94",
    "0.73",
    "0.755",
    "0.96",
    "0.55",
    "0.1796875",
]

print("\n" + "-" * 120)
print("B-REVIEW PUBLICATION-VALUE CROSS-CHECK")
print("-" * 120)

for token in required_tokens:
    assert token in br_blob, (
        f"STOP: expected frozen B-Review value {token} "
        "not found in BR13 final evidence."
    )
    print("✓ frozen evidence contains:", token)


# ==================================================================================================
# 12. Write publication tables
# ==================================================================================================

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

t1.to_csv(T1, index=False)
t2.to_csv(T2, index=False)
t3.to_csv(T3, index=False)
t4.to_csv(T4, index=False)
t5.to_csv(T5, index=False)
t6.to_csv(T6, index=False)


print("\n" + "-" * 120)
print("PUBLICATION TABLES CREATED")
print("-" * 120)

for path in [T1, T2, T3, T4, T5, T6]:
    print("✓", path.relative_to(ROOT))
    print("  SHA-256:", sha256(path))


# ==================================================================================================
# 13. Print tables for human review
# ==================================================================================================

print("\n" + "-" * 120)
print("T1 — EXPERIMENT A: SIX EVALUABLE MODELS")
print("-" * 120)
print(t1.to_string(index=False))


print("\n" + "-" * 120)
print("T2 — REVIEWER-REQUESTED B: CONSTRAINED DECODING")
print("-" * 120)
print(t2.to_string(index=False))


print("\n" + "-" * 120)
print("T3 — E2A: CONSTRAINED VS HISTORICAL CIR")
print("-" * 120)
print(t3.to_string(index=False))


print("\n" + "-" * 120)
print("T4 — E2A PAIRED CORRECTNESS")
print("-" * 120)
print(t4.to_string(index=False))
print("Exact McNemar p-value:", T4_MCNEMAR_P)


print("\n" + "-" * 120)
print("T5 — UNSUPPORTED-REQUIREMENT FORENSICS")
print("-" * 120)
print(t5.to_string(index=False))

print("Unsupported cases accepted: 24/24")
print("Structurally valid but decision-wrong: 54/200")
print("Invalid requests accepted: 47/100")


print("\n" + "-" * 120)
print("T6 — SUPPLEMENTARY B-E3")
print("-" * 120)
print(t6.to_string(index=False))


# ==================================================================================================
# 14. C4 manifest
# ==================================================================================================

manifest = {
    "manifest_id":
        "journal_C4_publication_tables_manifest_v1",

    "status":
        "PUBLICATION_TABLES_DERIVED_FROM_FROZEN_EVIDENCE",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "parent_master_manifest": {
        "path":
            str(C3_MANIFEST.relative_to(ROOT)),
        "sha256":
            C3_SHA,
    },

    "scientific_execution_during_C4": {
        "api_calls": 0,
        "model_calls": 0,
        "benchmark_executions": 0,
        "experiment_reruns": 0,
    },

    "tables": {
        "T1_experiment_A_model_robustness": {
            "path": str(T1.relative_to(ROOT)),
            "sha256": sha256(T1),
        },

        "T2_B_review_constrained_results": {
            "path": str(T2.relative_to(ROOT)),
            "sha256": sha256(T2),
        },

        "T3_E2A_constrained_vs_historical_CIR": {
            "path": str(T3.relative_to(ROOT)),
            "sha256": sha256(T3),
        },

        "T4_E2A_paired_correctness": {
            "path": str(T4.relative_to(ROOT)),
            "sha256": sha256(T4),
            "exact_mcnemar_p": T4_MCNEMAR_P,
        },

        "T5_unsupported_requirement_forensics": {
            "path": str(T5.relative_to(ROOT)),
            "sha256": sha256(T5),
            "unsupported_cases": 24,
            "unsupported_cases_accepted": 24,
            "structurally_valid_but_decision_wrong": 54,
            "invalid_requests_accepted": 47,
        },

        "T6_B_E3_supplementary": {
            "path": str(T6.relative_to(ROOT)),
            "sha256": sha256(T6),
            "role": "SUPPLEMENTARY",
        },
    },

    "interpretation_boundaries": [
        (
            "Experiment A demonstrates observed performance variation "
            "across six evaluable models while the tested containment "
            "violation count remained zero; it does not prove universal "
            "model independence."
        ),

        (
            "B-Review is the reviewer-requested constrained-decoding "
            "experiment."
        ),

        (
            "The E2A constrained treatment did not improve aggregate "
            "decision accuracy relative to historical CIR in the frozen "
            "comparison."
        ),

        (
            "Exact paired McNemar p=0.1796875; do not characterize the "
            "observed E2A difference as statistically significant."
        ),

        (
            "Provider-side structural constrained generation is distinct "
            "from downstream semantic/range validation."
        ),

        (
            "B-E3 is supplementary and must not be represented as the "
            "reviewer-requested Experiment B."
        ),

        (
            "No hallucination-free, zero-risk, universal-equivalence, "
            "or formal standards-conformance claim is supported."
        ),
    ],
}


stable_json(
    manifest,
    C4_MANIFEST
)


print("\n" + "-" * 120)
print("C4 MANIFEST")
print("-" * 120)

print(C4_MANIFEST)
print("SHA-256:", sha256(C4_MANIFEST))


# ==================================================================================================
# 15. Persist C4 outputs byte-for-byte
# ==================================================================================================

PERSIST_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PERSIST_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for src in [T1, T2, T3, T4, T5, T6]:

    dst = (
        PERSIST_TABLE_DIR
        / src.name
    )

    assert not dst.exists(), (
        f"STOP: persistent C4 table already exists:\n{dst}"
    )

    shutil.copy2(src, dst)

    assert sha256(src) == sha256(dst)

    print("✓ persisted:", dst)


PERSIST_C4_MANIFEST = (
    PERSIST_MANIFEST_DIR
    / C4_MANIFEST.name
)

assert not PERSIST_C4_MANIFEST.exists(), (
    "STOP: persistent C4 manifest already exists."
)

shutil.copy2(
    C4_MANIFEST,
    PERSIST_C4_MANIFEST
)

assert (
    sha256(C4_MANIFEST)
    ==
    sha256(PERSIST_C4_MANIFEST)
)


print("✓ persisted:", PERSIST_C4_MANIFEST)


# ==================================================================================================
# 16. Verify frozen inputs unchanged
# ==================================================================================================

print("\n" + "-" * 120)
print("POST-C4 FROZEN INPUT INTEGRITY")
print("-" * 120)

for path, expected in KNOWN.items():

    actual = sha256(path)

    assert actual == expected
    assert actual == INPUT_BEFORE[path]

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


assert sha256(C3_MANIFEST) == C3_SHA

print("✓ C3 master manifest unchanged")


# ==================================================================================================
# COMPLETE
# ==================================================================================================

print("\n" + "=" * 120)
print("✓ STEP C4 COMPLETE")
print("✓ SIX PUBLICATION TABLES DERIVED")
print("✓ EXPERIMENT A TABLE CREATED")
print("✓ REVIEWER-REQUESTED B TABLES CREATED")
print("✓ E2A PAIRED COMPARISON TABLE CREATED")
print("✓ UNSUPPORTED-REQUIREMENT FORENSIC TABLE CREATED")
print("✓ SUPPLEMENTARY B-E3 TABLE CREATED")
print("✓ ZERO API/MODEL CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ NO EXPERIMENT RERUN")
print("✓ FROZEN INPUT EVIDENCE UNCHANGED")
print("✓ C4 TABLES + MANIFEST PERSISTED BYTE-FOR-BYTE TO DRIVE")
print("✓ SEND COMPLETE OUTPUT BEFORE STEP C5")
print("=" * 120)

JOURNAL CONSOLIDATION — STEP C4: PUBLICATION TABLES FROM FROZEN EVIDENCE

------------------------------------------------------------------------------------------------------------------------
C3 MASTER EVIDENCE GATE
------------------------------------------------------------------------------------------------------------------------
✓ C3 master manifest verified
  SHA-256: 80555895d6b3c3c877455f252c9924ad7c7d35d68ebcc8bed1f74f9622d1884c

------------------------------------------------------------------------------------------------------------------------
FROZEN INPUT VERIFICATION
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/experiment_A_model_sweep/manifests/experiment_A_FINAL_FREEZE_v1.json
✓ journal_experiments/experiment_B_review_requested_constrained_decoding/final/experiment_B_review_FINAL_results_table_v1.csv
✓ journal_experiments/experiment_B_review_requested_constrained_deco

In [69]:
# ==================================================================================================
# JOURNAL CONSOLIDATION — STEP C5
# PAPER-READY RESULTS AND DISCUSSION EVIDENCE PACKAGE
#
# Purpose:
#   Convert the frozen C4 publication tables into carefully bounded,
#   paper-ready Results / Discussion text.
#
# IMPORTANT:
#   - READ-ONLY SCIENTIFIC ANALYSIS
#   - ZERO API / MODEL CALLS
#   - ZERO BENCHMARK EXECUTIONS
#   - NO EXPERIMENT RERUN
#   - NO NEW STATISTICAL TESTS
#   - NO NEW SCIENTIFIC CLAIMS
#   - DO NOT MODIFY THE MANUSCRIPT YET
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import pandas as pd


print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C5: PAPER-READY RESULTS AND DISCUSSION PACKAGE")
print("=" * 120)


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

JEXP = (
    ROOT
    / "journal_experiments"
)

MASTER = (
    JEXP
    / "journal_final_evidence"
)

TABLE_DIR = (
    MASTER
    / "publication_tables"
)

PAPER_DIR = (
    MASTER
    / "paper_ready"
)

MANIFEST_DIR = (
    MASTER
    / "manifests"
)


PERSIST_MASTER = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "journal_final_evidence"
)

PERSIST_PAPER_DIR = (
    PERSIST_MASTER
    / "paper_ready"
)

PERSIST_MANIFEST_DIR = (
    PERSIST_MASTER
    / "manifests"
)


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def stable_json(
    obj,
    path
):

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        )
        + "\n",
        encoding="utf-8"
    )


# ==================================================================================================
# 2. VERIFY C3 + C4
# ==================================================================================================

C3_MANIFEST = (
    MANIFEST_DIR
    / "journal_master_frozen_evidence_manifest_v1.json"
)

C3_SHA = (
    "80555895d6b3c3c877455f252c9924ad7c7d35d68ebcc8bed1f74f9622d1884c"
)


C4_MANIFEST = (
    MANIFEST_DIR
    / "journal_C4_publication_tables_manifest_v1.json"
)

C4_SHA = (
    "b5dde308ca53757a48ba9e14fe34551855fdecd7eda5ee2dd74d30c371532eca"
)


assert C3_MANIFEST.exists()
assert C4_MANIFEST.exists()

assert sha256(C3_MANIFEST) == C3_SHA
assert sha256(C4_MANIFEST) == C4_SHA


print("\n" + "-" * 120)
print("FROZEN CONSOLIDATION GATE")
print("-" * 120)

print("✓ C3 master evidence manifest verified")
print("  SHA-256:", C3_SHA)

print("✓ C4 publication-table manifest verified")
print("  SHA-256:", C4_SHA)


# ==================================================================================================
# 3. VERIFY C4 TABLES
# ==================================================================================================

T1 = (
    TABLE_DIR
    / "Table_C4_T1_experiment_A_model_robustness.csv"
)

T2 = (
    TABLE_DIR
    / "Table_C4_T2_B_review_constrained_results.csv"
)

T3 = (
    TABLE_DIR
    / "Table_C4_T3_E2A_constrained_vs_historical_CIR.csv"
)

T4 = (
    TABLE_DIR
    / "Table_C4_T4_E2A_paired_correctness.csv"
)

T5 = (
    TABLE_DIR
    / "Table_C4_T5_unsupported_requirement_forensics.csv"
)

T6 = (
    TABLE_DIR
    / "Table_C4_T6_B_E3_supplementary.csv"
)


EXPECTED_TABLE_SHA = {

    T1:
        "b868cdb8d4fe5ad52b48906aaa74dbb5ff84d59aa76567b0a473ed42a8cc6b0a",

    T2:
        "5eeec0e8bc49354d32614d4010938f89e21fa52bfc9debb19033e94bc3297345",

    T3:
        "c92dc4a7fd89ae169e697a3f4e09db14914c092e5048b10215961bbec9bb54ee",

    T4:
        "97bd4a70593a482501bc43646a154a09b9f961cf7fa2631f2397ead38848b1e4",

    T5:
        "3c8cc652814006f095cfee18d8093b1258730122b7af7e63cef3c212a8da8411",

    T6:
        "6a706602eded4922f15bedd7ec1be55ae1a2a3124064dc5e364ecb8418e08d0e",
}


TABLE_HASHES_BEFORE = {}


print("\n" + "-" * 120)
print("C4 TABLE VERIFICATION")
print("-" * 120)


for path, expected in EXPECTED_TABLE_SHA.items():

    assert path.exists(), (
        f"STOP: missing C4 table:\n{path}"
    )

    actual = sha256(
        path
    )

    assert actual == expected, (
        f"STOP: C4 table SHA mismatch\n"
        f"{path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    TABLE_HASHES_BEFORE[
        path
    ] = actual

    print(
        "✓",
        path.relative_to(ROOT)
    )


# ==================================================================================================
# 4. LOAD TABLES
# ==================================================================================================

t1 = pd.read_csv(T1)
t2 = pd.read_csv(T2)
t3 = pd.read_csv(T3)
t4 = pd.read_csv(T4)
t5 = pd.read_csv(T5)
t6 = pd.read_csv(T6)


# ==================================================================================================
# 5. ASSERT FROZEN SCIENTIFIC VALUES
# ==================================================================================================

# Experiment A
assert len(t1) == 6

assert (
    t1["containment_violations"]
    == 0
).all()

assert round(
    float(t1["accuracy_pct"].min()),
    2
) == 60.00

assert round(
    float(t1["accuracy_pct"].max()),
    2
) == 84.17

assert round(
    float(
        t1[
            "valid_acceptance_rate_pct"
        ].min()
    ),
    2
) == 18.97

assert round(
    float(
        t1[
            "valid_acceptance_rate_pct"
        ].max()
    ),
    2
) == 87.93

assert round(
    float(
        t1[
            "invalid_rejection_rate_pct"
        ].min()
    ),
    2
) == 80.65

assert round(
    float(
        t1[
            "invalid_rejection_rate_pct"
        ].max()
    ),
    2
) == 98.39


# B-Review
e1 = t2[
    t2["benchmark"] == "E1"
].iloc[0]

e2a = t2[
    t2["benchmark"] == "E2A"
].iloc[0]


assert float(
    e1["accuracy_pct"]
) == 94.0

assert float(
    e1["valid_acceptance_rate_pct"]
) == 100.0

assert float(
    e1["invalid_rejection_rate_pct"]
) == 88.0


assert float(
    e2a["accuracy_pct"]
) == 73.0

assert float(
    e2a["valid_acceptance_rate_pct"]
) == 93.0

assert float(
    e2a["invalid_rejection_rate_pct"]
) == 53.0


# Historical CIR comparison
historical = t3[
    t3["configuration"]
    == "Historical CIR"
].iloc[0]

constrained = t3[
    t3["configuration"]
    == "CIR constrained"
].iloc[0]


assert float(
    historical["accuracy_pct"]
) == 75.5

assert float(
    constrained["accuracy_pct"]
) == 73.0

assert float(
    constrained[
        "accuracy_delta_vs_historical_pp"
    ]
) == -2.5

assert float(
    constrained[
        "var_delta_vs_historical_pp"
    ]
) == -3.0

assert float(
    constrained[
        "irr_delta_vs_historical_pp"
    ]
) == -2.0


# Paired comparison
paired = dict(
    zip(
        t4["paired_category"],
        t4["n"]
    )
)

assert int(
    paired["Both correct"]
) == 144

assert int(
    paired["Constrained only correct"]
) == 2

assert int(
    paired["Historical CIR only correct"]
) == 7

assert int(
    paired["Both wrong"]
) == 47

MCNEMAR_P = 0.1796875


# Unsupported forensic
forensic = dict(
    zip(
        t5["forensic_outcome"],
        t5["n"]
    )
)

assert int(
    forensic["Dropped"]
) == 17

assert int(
    forensic["Mapped to supported field"]
) == 3

assert int(
    forensic["Preserved as free text"]
) == 4

assert int(
    forensic["Other / ambiguous"]
) == 0


# B-E3
be3 = t6.iloc[0]

assert float(
    be3["accuracy_pct"]
) == 84.17

assert float(
    be3["valid_acceptance_rate_pct"]
) == 87.93

assert float(
    be3["invalid_rejection_rate_pct"]
) == 80.65

assert int(
    be3["llm_invoked"]
) == 77

assert int(
    be3["pre_llm_no_call"]
) == 43

assert int(
    be3["parse_success"]
) == 77

assert int(
    be3["parse_failure"]
) == 0

assert str(
    be3["role"]
) == "SUPPLEMENTARY"


print("\n" + "-" * 120)
print("SCIENTIFIC VALUE GATE")
print("-" * 120)

print("✓ Experiment A values verified")
print("✓ B-Review E1/E2A values verified")
print("✓ Historical CIR comparison verified")
print("✓ Paired E2A counts verified")
print("✓ Unsupported-requirement forensic counts verified")
print("✓ Supplementary B-E3 values verified")


# ==================================================================================================
# 6. DERIVE DISPLAY VALUES
# ==================================================================================================

A_ACC_MIN = float(
    t1["accuracy_pct"].min()
)

A_ACC_MAX = float(
    t1["accuracy_pct"].max()
)

A_ACC_SPREAD = (
    A_ACC_MAX
    - A_ACC_MIN
)

A_VAR_MIN = float(
    t1[
        "valid_acceptance_rate_pct"
    ].min()
)

A_VAR_MAX = float(
    t1[
        "valid_acceptance_rate_pct"
    ].max()
)

A_VAR_SPREAD = (
    A_VAR_MAX
    - A_VAR_MIN
)

A_IRR_MIN = float(
    t1[
        "invalid_rejection_rate_pct"
    ].min()
)

A_IRR_MAX = float(
    t1[
        "invalid_rejection_rate_pct"
    ].max()
)

A_IRR_SPREAD = (
    A_IRR_MAX
    - A_IRR_MIN
)


# ==================================================================================================
# 7. OUTPUT PATHS / GATE
# ==================================================================================================

RESULTS_MD = (
    PAPER_DIR
    / "C5_paper_ready_results_discussion_v1.md"
)

CLAIMS_JSON = (
    PAPER_DIR
    / "C5_claim_evidence_map_v1.json"
)

C5_MANIFEST = (
    MANIFEST_DIR
    / "journal_C5_paper_ready_results_manifest_v1.json"
)


OUTPUTS = [
    RESULTS_MD,
    CLAIMS_JSON,
    C5_MANIFEST,
]


print("\n" + "-" * 120)
print("C5 OUTPUT GATE")
print("-" * 120)


for path in OUTPUTS:

    print(
        ("EXISTS : " if path.exists() else "absent : "),
        path.relative_to(ROOT)
    )


assert not any(
    path.exists()
    for path in OUTPUTS
), (
    "STOP: one or more C5 outputs already exist. "
    "Do not overwrite."
)


print(
    "✓ C5 output paths unused"
)


# ==================================================================================================
# 8. PAPER-READY RESULTS / DISCUSSION TEXT
# ==================================================================================================

paper_text = f"""# C5 Paper-Ready Results and Discussion Evidence Package

## Scope

This document converts the frozen journal-experiment evidence into publication-ready prose. It does not introduce new experiments, new statistical tests, or new scientific observations. Experiment A evaluates cross-model behavior of the frozen deterministic pipeline, B-Review is the reviewer-requested constrained-decoding experiment on E1 and E2A, and B-E3 is retained only as a supplementary E3-v3 follow-up.

## Cross-model robustness of deterministic failure containment

To examine whether the observed behavior depended on a single semantic parser, the frozen E3-v3 benchmark was evaluated with six accessible model arms while preserving the deterministic downstream pipeline. Decision accuracy varied substantially across the evaluable models, ranging from {A_ACC_MIN:.2f}% to {A_ACC_MAX:.2f}% (spread {A_ACC_SPREAD:.2f} percentage points). The valid-acceptance rate ranged from {A_VAR_MIN:.2f}% to {A_VAR_MAX:.2f}% (spread {A_VAR_SPREAD:.2f} percentage points), while the invalid-rejection rate ranged from {A_IRR_MIN:.2f}% to {A_IRR_MAX:.2f}% (spread {A_IRR_SPREAD:.2f} percentage points).

Despite this variation in decision performance, zero containment violations were observed in every one of the six evaluable model arms. This result separates two properties of the architecture: semantic interpretation quality remained model dependent, whereas the tested deterministic downstream containment boundary remained stable across the evaluated arms. The result should therefore be interpreted as empirical cross-model evidence under the frozen benchmark and evaluated models, rather than as a universal guarantee of model independence.

## Reviewer-requested constrained-decoding experiment

The reviewer-requested constrained-decoding experiment evaluated the CIR-constrained configuration on the frozen E1 and E2A benchmarks. On E1 (n=50; 25 valid and 25 invalid requests), constrained decoding achieved 94.00% decision accuracy, a 100.00% valid-acceptance rate, and an 88.00% invalid-rejection rate. The corresponding confusion counts were TP=22, FP=0, FN=3, and TN=25, where the positive class denotes an invalid request that was rejected.

On E2A (n=200; 100 valid and 100 invalid requests), the constrained configuration achieved 73.00% decision accuracy, a 93.00% valid-acceptance rate, and a 53.00% invalid-rejection rate. The corresponding counts were TP=53, FP=7, FN=47, and TN=93.

## Constrained decoding versus the historical CIR configuration

On the same frozen E2A benchmark, the historical CIR configuration achieved 75.50% decision accuracy, a 96.00% valid-acceptance rate, and a 55.00% invalid-rejection rate. The constrained configuration therefore differed by -2.50 percentage points in accuracy, -3.00 percentage points in valid acceptance, and -2.00 percentage points in invalid rejection.

The paired correctness analysis showed that both configurations were correct on 144 cases and both were wrong on 47 cases. The constrained configuration alone was correct on 2 cases, whereas the historical CIR configuration alone was correct on 7 cases. The exact McNemar test yielded p={MCNEMAR_P:.7f}. Accordingly, the frozen comparison does not provide evidence that structural constrained decoding improved decision accuracy, and the observed paired difference should not be described as statistically significant.

## Why structural constrained decoding is insufficient for semantic safety

The E2A forensic analysis provides a more direct explanation of this result. Among 24 requests containing unsupported requirements, all 24 were ultimately accepted by the CIR-constrained branch. Of these unsupported requirements, 17/24 (70.83%) were dropped from the generated CIR, 3/24 (12.50%) were mapped into a supported field, and 4/24 (16.67%) were preserved as free text. No case remained in the residual other/ambiguous category.

More broadly, 47 of the 100 invalid E2A requests were accepted, and 54 of the 200 cases were structurally valid yet decision-wrong. These observations distinguish structural output control from semantic preservation and policy correctness. Provider-side constrained generation can control the form of an LLM response, but structural validity alone does not establish that every source requirement has been preserved, correctly mapped, or semantically validated.

This distinction supports the architectural separation used in the proposed pipeline. The LLM is responsible for probabilistic semantic extraction, whereas deterministic source analysis, preservation checks, semantic validation, consistency checks, and policy/catalog gates remain responsible for deciding whether the interpreted request is safe to translate into an actionable descriptor.

## Supplementary E3-v3 structured-generation result

The supplementary B-E3 experiment provides an additional observation on the frozen 120-case E3-v3 benchmark. The structured-generation treatment achieved 84.17% decision accuracy, an 87.93% valid-acceptance rate, and an 80.65% invalid-rejection rate. Of the 120 cases, 77 reached the LLM and all 77 parsed successfully; the remaining 43 cases were handled without invoking the LLM and therefore have parse status N/A rather than parse failure.

The B-E3 result is supplementary and is not used as a substitute for the reviewer-requested E1/E2A constrained-decoding experiment. In addition, the frozen B-E3 result schema did not contain a separate `schema_valid` metric, so no such value is inferred or reported.

A retrospective implementation check found that the historical and B-E3 JSON-cleaning functions were not source- or AST-identical. After correcting the diagnostic execution environment, however, both cleaners produced identical cleaned text and identical parsed JSON on all 77 actually observed B-E3 LLM outputs. This finding is limited to those frozen responses and should not be interpreted as a universal equivalence proof for arbitrary model outputs.

## Overall interpretation

Taken together, the experiments support a bounded conclusion. Semantic-parser choice can materially change decision performance, as demonstrated by the wide cross-model variation in Experiment A. Structural constrained decoding improves control over output form but, in the frozen E2A evaluation, did not recover semantic requirements that were omitted or incorrectly represented and did not improve decision accuracy relative to the historical CIR configuration.

The results therefore support retaining deterministic validation after probabilistic intent interpretation. Under the tested architecture, the semantic parser proposes a structured interpretation, while deterministic gates determine whether that interpretation is sufficiently complete, consistent, policy-compliant, and supported to proceed. The evidence supports describing this design as hallucination-resistant through deterministic failure containment under defined constraints; it does not support claims of hallucination-free operation, universal model independence, zero risk, or formal 3GPP/O-RAN conformance.

## Paper-safe key result statements

1. Across six evaluable semantic-parser models on the frozen 120-case benchmark, decision accuracy ranged from {A_ACC_MIN:.2f}% to {A_ACC_MAX:.2f}%, while zero containment violations were observed in every evaluated arm.

2. On frozen E2A, the constrained CIR configuration achieved 73.00% accuracy compared with 75.50% for the historical CIR configuration; the paired exact McNemar test was p={MCNEMAR_P:.7f}.

3. All 24 E2A requests containing unsupported requirements were accepted by the CIR-constrained branch: 17 requirements were dropped, 3 were mapped to supported fields, and 4 were retained as free text.

4. The constrained-decoding results show that structural output control does not by itself ensure source-requirement preservation or semantic validity.

5. These results support retaining deterministic source analysis, preservation checks, semantic validation, consistency checks, and policy/catalog gates after LLM-based semantic interpretation.

## Claims explicitly excluded

The frozen evidence does not establish that the system is hallucination-free, universally model-independent, universally safe, formally verified for arbitrary inputs, or formally conformant with 3GPP or O-RAN standards. B-E3 is supplementary and must not be represented as the experiment requested by the reviewer.
"""


# ==================================================================================================
# 9. CLAIM ↔ EVIDENCE MAP
# ==================================================================================================

claim_map = {

    "claim_1_cross_model_variation": {
        "statement":
            (
                f"Across six evaluable models, accuracy ranged from "
                f"{A_ACC_MIN:.2f}% to {A_ACC_MAX:.2f}%."
            ),

        "evidence":
            str(T1.relative_to(ROOT)),

        "boundary":
            (
                "Empirical result for six evaluated models and the "
                "frozen benchmark only."
            ),
    },


    "claim_2_containment": {
        "statement":
            (
                "Zero containment violations were observed in every "
                "evaluable Experiment A arm."
            ),

        "evidence":
            str(T1.relative_to(ROOT)),

        "boundary":
            (
                "Observed zero violations are not a universal formal "
                "guarantee."
            ),
    },


    "claim_3_B_review_E2A": {
        "statement":
            (
                "E2A CIR-constrained accuracy was 73.00%, VAR 93.00%, "
                "and IRR 53.00%."
            ),

        "evidence":
            str(T2.relative_to(ROOT)),
    },


    "claim_4_historical_comparison": {
        "statement":
            (
                "Historical CIR accuracy was 75.50% versus 73.00% "
                "for CIR constrained."
            ),

        "evidence":
            str(T3.relative_to(ROOT)),

        "boundary":
            (
                "Describe as no observed improvement; do not claim "
                "significant degradation."
            ),
    },


    "claim_5_paired_test": {
        "statement":
            (
                "E2A paired counts were 144 both correct, 2 constrained-only, "
                "7 historical-only, and 47 both wrong; exact McNemar "
                f"p={MCNEMAR_P:.7f}."
            ),

        "evidence":
            str(T4.relative_to(ROOT)),
    },


    "claim_6_unsupported_requirements": {
        "statement":
            (
                "All 24 unsupported-requirement E2A cases were accepted "
                "by the CIR-constrained branch; 17 were dropped, 3 mapped "
                "to supported fields, and 4 preserved as free text."
            ),

        "evidence":
            str(T5.relative_to(ROOT)),

        "boundary":
            (
                "This forensic classifier is specific to the frozen "
                "synthetic benchmark and is not a general semantic-"
                "equivalence detector."
            ),
    },


    "claim_7_B_E3": {
        "statement":
            (
                "Supplementary B-E3 achieved 84.17% accuracy, 87.93% VAR, "
                "80.65% IRR, with 77/77 invoked responses parsing "
                "successfully."
            ),

        "evidence":
            str(T6.relative_to(ROOT)),

        "boundary":
            (
                "Supplementary only; does not replace reviewer-requested "
                "B-Review."
            ),
    },
}


# ==================================================================================================
# 10. WRITE PAPER-READY OUTPUTS
# ==================================================================================================

PAPER_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESULTS_MD.write_text(
    paper_text,
    encoding="utf-8"
)


stable_json(
    claim_map,
    CLAIMS_JSON
)


print("\n" + "-" * 120)
print("PAPER-READY OUTPUTS CREATED")
print("-" * 120)


print(
    "✓",
    RESULTS_MD.relative_to(ROOT)
)

print(
    "  SHA-256:",
    sha256(RESULTS_MD)
)


print(
    "✓",
    CLAIMS_JSON.relative_to(ROOT)
)

print(
    "  SHA-256:",
    sha256(CLAIMS_JSON)
)


# ==================================================================================================
# 11. DISPLAY PAPER TEXT FOR REVIEW
# ==================================================================================================

print("\n" + "-" * 120)
print("PAPER-READY RESULTS / DISCUSSION")
print("-" * 120)

print(
    RESULTS_MD.read_text(
        encoding="utf-8"
    )
)


# ==================================================================================================
# 12. C5 MANIFEST
# ==================================================================================================

manifest = {

    "manifest_id":
        "journal_C5_paper_ready_results_manifest_v1",

    "status":
        "PAPER_READY_RESULTS_DERIVED_FROM_FROZEN_EVIDENCE",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_manifests": {
        "C3": {
            "path":
                str(
                    C3_MANIFEST.relative_to(ROOT)
                ),

            "sha256":
                C3_SHA,
        },

        "C4": {
            "path":
                str(
                    C4_MANIFEST.relative_to(ROOT)
                ),

            "sha256":
                C4_SHA,
        },
    },

    "scientific_execution_during_C5": {
        "api_calls": 0,
        "model_calls": 0,
        "benchmark_executions": 0,
        "experiment_reruns": 0,
        "new_statistical_tests": 0,
    },

    "inputs": {
        str(path.relative_to(ROOT)):
            expected
        for path, expected
        in EXPECTED_TABLE_SHA.items()
    },

    "outputs": {
        str(RESULTS_MD.relative_to(ROOT)):
            sha256(RESULTS_MD),

        str(CLAIMS_JSON.relative_to(ROOT)):
            sha256(CLAIMS_JSON),
    },

    "paper_positioning": {
        "preferred_framing":
            (
                "Hallucination-resistant intent translation through "
                "deterministic failure containment under defined constraints."
            ),

        "prohibited_overclaims": [
            "hallucination-free",
            "universal model independence",
            "universal safety",
            "zero risk",
            "formal arbitrary-input guarantee",
            "formal 3GPP/O-RAN conformance",
        ],

        "B_E3_role":
            "SUPPLEMENTARY_ONLY",
    },
}


stable_json(
    manifest,
    C5_MANIFEST
)


print("\n" + "-" * 120)
print("C5 MANIFEST")
print("-" * 120)


print(
    C5_MANIFEST
)

print(
    "SHA-256:",
    sha256(C5_MANIFEST)
)


# ==================================================================================================
# 13. PERSIST BYTE-FOR-BYTE TO DRIVE
# ==================================================================================================

PERSIST_PAPER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PERSIST_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PERSIST_MAP = {

    RESULTS_MD:
        (
            PERSIST_PAPER_DIR
            / RESULTS_MD.name
        ),

    CLAIMS_JSON:
        (
            PERSIST_PAPER_DIR
            / CLAIMS_JSON.name
        ),

    C5_MANIFEST:
        (
            PERSIST_MANIFEST_DIR
            / C5_MANIFEST.name
        ),
}


print("\n" + "-" * 120)
print("PERSISTENCE")
print("-" * 120)


for src, dst in PERSIST_MAP.items():

    assert not dst.exists(), (
        f"STOP: persistent C5 artifact already exists:\n{dst}"
    )

    shutil.copy2(
        src,
        dst
    )

    assert (
        sha256(src)
        ==
        sha256(dst)
    )

    print(
        "✓ persisted:",
        dst
    )

    print(
        "  SHA-256:",
        sha256(dst)
    )


# ==================================================================================================
# 14. POST-C5 INPUT INTEGRITY
# ==================================================================================================

print("\n" + "-" * 120)
print("POST-C5 FROZEN INPUT INTEGRITY")
print("-" * 120)


for path, expected in EXPECTED_TABLE_SHA.items():

    actual = sha256(
        path
    )

    assert actual == expected
    assert actual == TABLE_HASHES_BEFORE[path]

    print(
        "✓ unchanged:",
        path.relative_to(ROOT)
    )


assert sha256(
    C3_MANIFEST
) == C3_SHA

assert sha256(
    C4_MANIFEST
) == C4_SHA


print(
    "✓ C3 master manifest unchanged"
)

print(
    "✓ C4 publication-table manifest unchanged"
)


# ==================================================================================================
# 15. FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "✓ STEP C5 COMPLETE"
)

print(
    "✓ PAPER-READY RESULTS / DISCUSSION PACKAGE CREATED"
)

print(
    "✓ CLAIM ↔ EVIDENCE MAP CREATED"
)

print(
    "✓ NO NEW SCIENTIFIC RESULT INTRODUCED"
)

print(
    "✓ NO NEW STATISTICAL TEST PERFORMED"
)

print(
    "✓ ZERO API/MODEL CALLS"
)

print(
    "✓ ZERO BENCHMARK EXECUTIONS"
)

print(
    "✓ NO EXPERIMENT RERUN"
)

print(
    "✓ C4 TABLES UNCHANGED"
)

print(
    "✓ C5 ARTIFACTS PERSISTED BYTE-FOR-BYTE TO DRIVE"
)

print(
    "✓ NEXT STEP AFTER REVIEW: C6 FINAL REPRODUCIBILITY FREEZE"
)

print(
    "✓ SEND COMPLETE OUTPUT BEFORE C6"
)

print("=" * 120)

JOURNAL CONSOLIDATION — STEP C5: PAPER-READY RESULTS AND DISCUSSION PACKAGE

------------------------------------------------------------------------------------------------------------------------
FROZEN CONSOLIDATION GATE
------------------------------------------------------------------------------------------------------------------------
✓ C3 master evidence manifest verified
  SHA-256: 80555895d6b3c3c877455f252c9924ad7c7d35d68ebcc8bed1f74f9622d1884c
✓ C4 publication-table manifest verified
  SHA-256: b5dde308ca53757a48ba9e14fe34551855fdecd7eda5ee2dd74d30c371532eca

------------------------------------------------------------------------------------------------------------------------
C4 TABLE VERIFICATION
------------------------------------------------------------------------------------------------------------------------
✓ journal_experiments/journal_final_evidence/publication_tables/Table_C4_T1_experiment_A_model_robustness.csv
✓ journal_experiments/journal_final_evidence/pub

In [70]:
# ==================================================================================================
# JOURNAL CONSOLIDATION — STEP C6
# FINAL REPRODUCIBILITY FREEZE
#
# Purpose:
#   Freeze the complete journal experimental/consolidation evidence chain:
#
#       C1 inventory
#       C2 B-E3 closure
#       C3 master evidence consolidation
#       C4 publication tables
#       C5 paper-ready Results/Discussion
#
#   together with:
#
#       Experiment A final freeze
#       B-Review final freeze
#       B-E3 final freeze
#
# IMPORTANT:
#   - ZERO API / MODEL CALLS
#   - ZERO BENCHMARK EXECUTIONS
#   - ZERO EXPERIMENT RERUNS
#   - NO SCIENTIFIC RECOMPUTATION
#   - NO MODIFICATION OF FROZEN ARTIFACTS
#   - FINAL REPRODUCIBILITY PACKAGING ONLY
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil


print("=" * 120)
print("JOURNAL CONSOLIDATION — STEP C6: FINAL REPRODUCIBILITY FREEZE")
print("=" * 120)


# ==================================================================================================
# 0. ROOTS
# ==================================================================================================

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

JEXP = (
    ROOT
    / "journal_experiments"
)

MASTER = (
    JEXP
    / "journal_final_evidence"
)

INVENTORY_DIR = (
    MASTER
    / "inventory"
)

TABLE_DIR = (
    MASTER
    / "publication_tables"
)

PAPER_DIR = (
    MASTER
    / "paper_ready"
)

MANIFEST_DIR = (
    MASTER
    / "manifests"
)

REPRO_DIR = (
    MASTER
    / "reproducibility"
)


PERSIST_MASTER = Path(
    "/content/drive/MyDrive/PhD_Research/"
    "Deterministic_Intent_Translation/"
    "journal_experiments_persistent/"
    "journal_final_evidence"
)

PERSIST_REPRO_DIR = (
    PERSIST_MASTER
    / "reproducibility"
)

PERSIST_MANIFEST_DIR = (
    PERSIST_MASTER
    / "manifests"
)


# ==================================================================================================
# 1. HELPERS
# ==================================================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def stable_json(obj, path):

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False
        )
        + "\n",
        encoding="utf-8"
    )


def verify(path, expected):

    assert path.exists(), (
        f"STOP: required artifact missing:\n{path}"
    )

    actual = sha256(path)

    assert actual == expected, (
        f"STOP: SHA-256 mismatch\n"
        f"Path:     {path}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

    return actual


# ==================================================================================================
# 2. AUTHORITATIVE EXPERIMENT FREEZES
# ==================================================================================================

A_FREEZE = (
    JEXP
    / "experiment_A_model_sweep"
    / "manifests"
    / "experiment_A_FINAL_FREEZE_v1.json"
)

A_FREEZE_SHA = (
    "667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a"
)


BR_FREEZE = (
    JEXP
    / "experiment_B_review_requested_constrained_decoding"
    / "manifests"
    / "experiment_B_review_FINAL_FREEZE_v1.json"
)

BR_FREEZE_SHA = (
    "63fa99c81477eb7d2c3136b67f72ac3d4673f7d081d7b332ef8d634c68faf3dd"
)


BE3_FREEZE = (
    JEXP
    / "experiment_B_schema_constrained_decoding"
    / "manifests"
    / "experiment_B_E3_FINAL_FREEZE_v1.json"
)

BE3_FREEZE_SHA = (
    "e3e5dee967eeb233f1e583624d204b26c648040938715d41ff7cd4102d6d77bb"
)


# ==================================================================================================
# 3. CONSOLIDATION CHAIN
# ==================================================================================================

C1 = (
    INVENTORY_DIR
    / "C1_frozen_evidence_inventory_v1.json"
)

C1_SHA = (
    "550446bdd7f70481e0eb1a512af4e8d3fddb62eec7272f227d1c192514e7ced2"
)


C3 = (
    MANIFEST_DIR
    / "journal_master_frozen_evidence_manifest_v1.json"
)

C3_SHA = (
    "80555895d6b3c3c877455f252c9924ad7c7d35d68ebcc8bed1f74f9622d1884c"
)


C4 = (
    MANIFEST_DIR
    / "journal_C4_publication_tables_manifest_v1.json"
)

C4_SHA = (
    "b5dde308ca53757a48ba9e14fe34551855fdecd7eda5ee2dd74d30c371532eca"
)


C5 = (
    MANIFEST_DIR
    / "journal_C5_paper_ready_results_manifest_v1.json"
)

C5_SHA = (
    "464c863d070ef29b2308e5cecbce15d45772e2793bb5a212c6322b596f9d558f"
)


# ==================================================================================================
# 4. C4 PUBLICATION TABLES
# ==================================================================================================

T1 = (
    TABLE_DIR
    / "Table_C4_T1_experiment_A_model_robustness.csv"
)

T2 = (
    TABLE_DIR
    / "Table_C4_T2_B_review_constrained_results.csv"
)

T3 = (
    TABLE_DIR
    / "Table_C4_T3_E2A_constrained_vs_historical_CIR.csv"
)

T4 = (
    TABLE_DIR
    / "Table_C4_T4_E2A_paired_correctness.csv"
)

T5 = (
    TABLE_DIR
    / "Table_C4_T5_unsupported_requirement_forensics.csv"
)

T6 = (
    TABLE_DIR
    / "Table_C4_T6_B_E3_supplementary.csv"
)


TABLE_SHA = {

    T1:
        "b868cdb8d4fe5ad52b48906aaa74dbb5ff84d59aa76567b0a473ed42a8cc6b0a",

    T2:
        "5eeec0e8bc49354d32614d4010938f89e21fa52bfc9debb19033e94bc3297345",

    T3:
        "c92dc4a7fd89ae169e697a3f4e09db14914c092e5048b10215961bbec9bb54ee",

    T4:
        "97bd4a70593a482501bc43646a154a09b9f961cf7fa2631f2397ead38848b1e4",

    T5:
        "3c8cc652814006f095cfee18d8093b1258730122b7af7e63cef3c212a8da8411",

    T6:
        "6a706602eded4922f15bedd7ec1be55ae1a2a3124064dc5e364ecb8418e08d0e",
}


# ==================================================================================================
# 5. C5 PAPER-READY ARTIFACTS
# ==================================================================================================

C5_RESULTS = (
    PAPER_DIR
    / "C5_paper_ready_results_discussion_v1.md"
)

C5_RESULTS_SHA = (
    "dc340dbd24d4dd3c817b10ca95c5711c5a5833cd255ab726cbcd82652f47fe49"
)


C5_CLAIMS = (
    PAPER_DIR
    / "C5_claim_evidence_map_v1.json"
)

C5_CLAIMS_SHA = (
    "356c4a9de1d5387fa1d95af08743d1b001cdeb05d0757bbdc98f3361dbe1d56e"
)


# ==================================================================================================
# 6. COMPLETE AUTHORITATIVE ARTIFACT SET
# ==================================================================================================

AUTHORITATIVE = {

    "experiment_A_final_freeze":
        (A_FREEZE, A_FREEZE_SHA),

    "B_review_final_freeze":
        (BR_FREEZE, BR_FREEZE_SHA),

    "B_E3_final_freeze":
        (BE3_FREEZE, BE3_FREEZE_SHA),

    "C1_inventory":
        (C1, C1_SHA),

    "C3_master_manifest":
        (C3, C3_SHA),

    "C4_manifest":
        (C4, C4_SHA),

    "C5_manifest":
        (C5, C5_SHA),

    "C4_T1":
        (T1, TABLE_SHA[T1]),

    "C4_T2":
        (T2, TABLE_SHA[T2]),

    "C4_T3":
        (T3, TABLE_SHA[T3]),

    "C4_T4":
        (T4, TABLE_SHA[T4]),

    "C4_T5":
        (T5, TABLE_SHA[T5]),

    "C4_T6":
        (T6, TABLE_SHA[T6]),

    "C5_results_discussion":
        (C5_RESULTS, C5_RESULTS_SHA),

    "C5_claim_evidence_map":
        (C5_CLAIMS, C5_CLAIMS_SHA),
}


# ==================================================================================================
# 7. VERIFY COMPLETE CHAIN
# ==================================================================================================

print("\n" + "-" * 120)
print("COMPLETE FROZEN EVIDENCE-CHAIN VERIFICATION")
print("-" * 120)


HASHES_BEFORE = {}


for label, (
    path,
    expected
) in AUTHORITATIVE.items():

    actual = verify(
        path,
        expected
    )

    HASHES_BEFORE[
        path
    ] = actual

    print(
        f"✓ {label}"
    )

    print(
        " ",
        path.relative_to(ROOT)
    )

    print(
        "  SHA-256:",
        actual
    )


# ==================================================================================================
# 8. VERIFY EXPERIMENT ROLE SEPARATION
# ==================================================================================================

with open(
    BE3_FREEZE,
    "r",
    encoding="utf-8"
) as f:

    be3_data = json.load(f)


be3_blob = json.dumps(
    be3_data
).upper()


assert "SUPPLEMENTARY" in be3_blob


print("\n" + "-" * 120)
print("FINAL SCIENTIFIC ROLE GATE")
print("-" * 120)

print(
    "✓ Experiment A = multi-model robustness / containment evidence"
)

print(
    "✓ B-Review = reviewer-requested constrained-decoding experiment"
)

print(
    "✓ B-E3 = supplementary E3-v3 structured-generation follow-up"
)

print(
    "✓ B-E3 remains explicitly supplementary"
)


# ==================================================================================================
# 9. OUTPUT GATE
# ==================================================================================================

FINAL_MANIFEST = (
    MANIFEST_DIR
    / "journal_FINAL_REPRODUCIBILITY_FREEZE_v1.json"
)

FINAL_INDEX = (
    REPRO_DIR
    / "journal_FINAL_reproducibility_index_v1.json"
)

FINAL_README = (
    REPRO_DIR
    / "README_FINAL_REPRODUCIBILITY_FREEZE.md"
)


PERSIST_FINAL_MANIFEST = (
    PERSIST_MANIFEST_DIR
    / FINAL_MANIFEST.name
)

PERSIST_FINAL_INDEX = (
    PERSIST_REPRO_DIR
    / FINAL_INDEX.name
)

PERSIST_FINAL_README = (
    PERSIST_REPRO_DIR
    / FINAL_README.name
)


FINAL_OUTPUTS = [
    FINAL_MANIFEST,
    FINAL_INDEX,
    FINAL_README,
    PERSIST_FINAL_MANIFEST,
    PERSIST_FINAL_INDEX,
    PERSIST_FINAL_README,
]


print("\n" + "-" * 120)
print("FINAL OUTPUT GATE")
print("-" * 120)


for path in FINAL_OUTPUTS:

    print(
        ("EXISTS : " if path.exists() else "absent : "),
        path
    )


assert not any(
    path.exists()
    for path in FINAL_OUTPUTS
), (
    "STOP: one or more final reproducibility-freeze outputs "
    "already exist. Do not overwrite."
)


print(
    "✓ all final reproducibility-freeze output paths unused"
)


# ==================================================================================================
# 10. BUILD REPRODUCIBILITY INDEX
# ==================================================================================================

index = {

    "index_id":
        "journal_FINAL_reproducibility_index_v1",

    "status":
        "FINAL_FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "artifact_count":
        len(AUTHORITATIVE),

    "artifacts": {
        label: {
            "path":
                str(
                    path.relative_to(ROOT)
                ),

            "sha256":
                expected,
        }

        for label, (
            path,
            expected
        ) in AUTHORITATIVE.items()
    },

    "evidence_chain": [
        "Experiment A final freeze",
        "B-Review final freeze",
        "B-E3 final freeze",
        "C1 frozen-evidence inventory",
        "C3 master evidence consolidation",
        "C4 publication tables",
        "C5 paper-ready Results/Discussion",
        "C6 final reproducibility freeze",
    ],

    "experiment_roles": {

        "Experiment_A":
            (
                "Multi-model robustness and deterministic "
                "failure-containment evidence."
            ),

        "B_Review":
            (
                "Reviewer-requested constrained-decoding experiment "
                "on frozen E1 and E2A."
            ),

        "B_E3":
            (
                "Supplementary structured-generation follow-up on "
                "frozen E3-v3; not the reviewer-requested Experiment B."
            ),
    },
}


# ==================================================================================================
# 11. WRITE FINAL INDEX
# ==================================================================================================

REPRO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


stable_json(
    index,
    FINAL_INDEX
)


FINAL_INDEX_SHA = sha256(
    FINAL_INDEX
)


print("\n" + "-" * 120)
print("FINAL REPRODUCIBILITY INDEX")
print("-" * 120)

print(
    FINAL_INDEX
)

print(
    "SHA-256:",
    FINAL_INDEX_SHA
)


# ==================================================================================================
# 12. FINAL README
# ==================================================================================================

readme = f"""# Final Reproducibility Freeze

## Status

The journal experimental and evidence-consolidation phase is FINAL FROZEN.

No further model-running experiment is required for the current reviewer-response plan.

## Frozen evidence streams

1. Experiment A — multi-model robustness and deterministic failure-containment evidence.
2. B-Review — reviewer-requested constrained-decoding experiment on frozen E1 and E2A.
3. B-E3 — supplementary structured-generation follow-up on frozen E3-v3.

B-E3 must remain explicitly supplementary and must not be represented as the reviewer-requested Experiment B.

## Consolidation chain

- C1 — frozen evidence inventory
- C2 — B-E3 analytic closure
- C3 — master frozen-evidence consolidation
- C4 — publication-table derivation
- C5 — paper-ready Results/Discussion evidence package
- C6 — final reproducibility freeze

## Final scientific boundaries

The evidence supports describing the architecture as hallucination-resistant through deterministic failure containment under defined constraints.

The evidence does not establish:

- hallucination-free operation;
- universal model independence;
- universal safety;
- zero risk;
- a formal arbitrary-input guarantee;
- formal 3GPP or O-RAN conformance;
- universal equivalence of the historical and B-E3 cleaner implementations.

## Statistical interpretation

For the frozen E2A paired comparison, the constrained CIR configuration achieved 73.00% accuracy and the historical CIR configuration achieved 75.50% accuracy. The exact paired McNemar test was p=0.1796875. The result should therefore be described as showing no observed improvement from constrained decoding in this comparison, not as a statistically significant degradation.

## Experiment A interpretation

Across the six evaluable model arms, decision accuracy ranged from 60.00% to 84.17%. Zero containment violations were observed in every evaluated arm. This is empirical evidence under the tested models and frozen benchmark, not a universal model-independence guarantee.

## Structural decoding interpretation

Provider-side structural constrained generation controls output form but does not by itself establish source-requirement preservation, semantic validity, policy correctness, or supported actionability.

In the frozen E2A forensic analysis, all 24 unsupported-requirement cases were accepted by the CIR-constrained branch: 17 were dropped, 3 were mapped into supported fields, and 4 were preserved as free text.

## Reproducibility index

SHA-256: `{FINAL_INDEX_SHA}`

## Allowed next work

After this freeze, work should be limited to read-only use of the frozen evidence for:

- manuscript revision;
- publication-table formatting;
- figure preparation;
- reviewer-response drafting;
- supplementary-material preparation;
- reproducibility documentation.

No frozen experimental artifact should be modified or silently replaced.
"""


FINAL_README.write_text(
    readme,
    encoding="utf-8"
)


FINAL_README_SHA = sha256(
    FINAL_README
)


print("\n" + "-" * 120)
print("FINAL REPRODUCIBILITY README")
print("-" * 120)

print(
    FINAL_README
)

print(
    "SHA-256:",
    FINAL_README_SHA
)


# ==================================================================================================
# 13. BUILD FINAL FREEZE MANIFEST
# ==================================================================================================

final_manifest = {

    "freeze_id":
        "journal_FINAL_REPRODUCIBILITY_FREEZE_v1",

    "status":
        "FINAL_FROZEN",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_execution_during_C6": {
        "api_calls": 0,
        "model_calls": 0,
        "benchmark_executions": 0,
        "experiment_reruns": 0,
        "new_statistical_tests": 0,
        "new_scientific_results": 0,
    },

    "authoritative_artifacts": {
        label: {
            "path":
                str(
                    path.relative_to(ROOT)
                ),

            "sha256":
                expected,
        }

        for label, (
            path,
            expected
        ) in AUTHORITATIVE.items()
    },

    "reproducibility_index": {
        "path":
            str(
                FINAL_INDEX.relative_to(ROOT)
            ),

        "sha256":
            FINAL_INDEX_SHA,
    },

    "reproducibility_readme": {
        "path":
            str(
                FINAL_README.relative_to(ROOT)
            ),

        "sha256":
            FINAL_README_SHA,
    },

    "experiment_status": {
        "Experiment_A":
            "FINAL_FROZEN",

        "B_Review":
            "FINAL_FROZEN",

        "B_E3":
            "FINAL_FROZEN_SUPPLEMENTARY",
    },

    "consolidation_status": {
        "C1":
            "COMPLETE",

        "C2":
            "COMPLETE",

        "C3":
            "COMPLETE",

        "C4":
            "COMPLETE",

        "C5":
            "COMPLETE",

        "C6":
            "FINAL_FREEZE",
    },

    "future_scientific_execution":
        {
            "current_reviewer_plan_requires_additional_model_experiments":
                False,

            "frozen_experiments_must_not_be_rerun":
                True,

            "frozen_evidence_must_not_be_modified":
                True,
        },

    "claim_boundaries": [
        "No hallucination-free claim.",
        "No universal model-independence claim.",
        "No universal safety claim.",
        "No zero-risk claim.",
        "No formal arbitrary-input guarantee.",
        "No formal 3GPP/O-RAN conformance claim.",
        "No universal cleaner-equivalence claim.",
        (
            "B-E3 is supplementary and must not be represented "
            "as the reviewer-requested Experiment B."
        ),
    ],

    "allowed_next_operations": [
        "Manuscript revision from frozen evidence.",
        "Reviewer-response drafting from frozen evidence.",
        "Publication-table formatting.",
        "Figure preparation.",
        "Supplementary-material preparation.",
        "Reproducibility documentation.",
    ],
}


stable_json(
    final_manifest,
    FINAL_MANIFEST
)


FINAL_MANIFEST_SHA = sha256(
    FINAL_MANIFEST
)


print("\n" + "-" * 120)
print("FINAL REPRODUCIBILITY FREEZE MANIFEST")
print("-" * 120)

print(
    FINAL_MANIFEST
)

print(
    "SHA-256:",
    FINAL_MANIFEST_SHA
)


# ==================================================================================================
# 14. PERSIST FINAL FREEZE BYTE-FOR-BYTE
# ==================================================================================================

PERSIST_REPRO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PERSIST_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PERSIST_MAP = {

    FINAL_INDEX:
        PERSIST_FINAL_INDEX,

    FINAL_README:
        PERSIST_FINAL_README,

    FINAL_MANIFEST:
        PERSIST_FINAL_MANIFEST,
}


print("\n" + "-" * 120)
print("FINAL PERSISTENCE")
print("-" * 120)


for src, dst in PERSIST_MAP.items():

    assert not dst.exists(), (
        f"STOP: persistent final artifact already exists:\n{dst}"
    )

    shutil.copy2(
        src,
        dst
    )

    assert (
        sha256(src)
        ==
        sha256(dst)
    )

    print(
        "✓ persisted:",
        dst
    )

    print(
        "  SHA-256:",
        sha256(dst)
    )


# ==================================================================================================
# 15. FINAL IMMUTABILITY AUDIT
# ==================================================================================================

print("\n" + "-" * 120)
print("FINAL IMMUTABILITY AUDIT")
print("-" * 120)


for label, (
    path,
    expected
) in AUTHORITATIVE.items():

    actual = sha256(
        path
    )

    assert actual == expected

    assert (
        actual
        ==
        HASHES_BEFORE[path]
    )

    print(
        "✓ unchanged:",
        label
    )


# ==================================================================================================
# 16. FINAL CHAIN SELF-CHECK
# ==================================================================================================

assert sha256(
    FINAL_INDEX
) == FINAL_INDEX_SHA

assert sha256(
    FINAL_README
) == FINAL_README_SHA

assert sha256(
    FINAL_MANIFEST
) == FINAL_MANIFEST_SHA


with open(
    FINAL_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    check = json.load(f)


assert check[
    "status"
] == "FINAL_FROZEN"

assert check[
    "experiment_status"
][
    "Experiment_A"
] == "FINAL_FROZEN"

assert check[
    "experiment_status"
][
    "B_Review"
] == "FINAL_FROZEN"

assert check[
    "experiment_status"
][
    "B_E3"
] == "FINAL_FROZEN_SUPPLEMENTARY"

assert check[
    "future_scientific_execution"
][
    "current_reviewer_plan_requires_additional_model_experiments"
] is False


# ==================================================================================================
# 17. FINAL STATUS
# ==================================================================================================

print("\n" + "-" * 120)
print("FINAL JOURNAL EVIDENCE STATUS")
print("-" * 120)

print(
    "Experiment A : FINAL_FROZEN"
)

print(
    "B-Review     : FINAL_FROZEN"
)

print(
    "B-E3         : FINAL_FROZEN / SUPPLEMENTARY"
)

print(
    "C1           : COMPLETE"
)

print(
    "C2           : COMPLETE"
)

print(
    "C3           : COMPLETE"
)

print(
    "C4           : COMPLETE"
)

print(
    "C5           : COMPLETE"
)

print(
    "C6           : FINAL REPRODUCIBILITY FREEZE"
)

print(
    "Final index SHA    :",
    FINAL_INDEX_SHA
)

print(
    "Final README SHA   :",
    FINAL_README_SHA
)

print(
    "FINAL FREEZE SHA   :",
    FINAL_MANIFEST_SHA
)


# ==================================================================================================
# COMPLETE
# ==================================================================================================

print("\n" + "=" * 120)

print(
    "✓ STEP C6 COMPLETE"
)

print(
    "✓ FINAL REPRODUCIBILITY FREEZE CREATED"
)

print(
    "✓ EXPERIMENT A FINAL FROZEN"
)

print(
    "✓ B-REVIEW FINAL FROZEN"
)

print(
    "✓ B-E3 FINAL FROZEN / SUPPLEMENTARY"
)

print(
    "✓ C1–C6 CONSOLIDATION CHAIN COMPLETE"
)

print(
    "✓ ZERO API/MODEL CALLS"
)

print(
    "✓ ZERO BENCHMARK EXECUTIONS"
)

print(
    "✓ ZERO EXPERIMENT RERUNS"
)

print(
    "✓ ZERO NEW STATISTICAL TESTS"
)

print(
    "✓ ZERO NEW SCIENTIFIC RESULTS"
)

print(
    "✓ ALL AUTHORITATIVE ARTIFACTS UNCHANGED"
)

print(
    "✓ FINAL FREEZE PERSISTED BYTE-FOR-BYTE TO DRIVE"
)

print(
    "✓ EXPERIMENTAL / CONSOLIDATION PHASE CONCLUDED"
)

print(
    "✓ SEND COMPLETE OUTPUT FOR FINAL VERIFICATION"
)

print("=" * 120)

JOURNAL CONSOLIDATION — STEP C6: FINAL REPRODUCIBILITY FREEZE

------------------------------------------------------------------------------------------------------------------------
COMPLETE FROZEN EVIDENCE-CHAIN VERIFICATION
------------------------------------------------------------------------------------------------------------------------
✓ experiment_A_final_freeze
  journal_experiments/experiment_A_model_sweep/manifests/experiment_A_FINAL_FREEZE_v1.json
  SHA-256: 667cfd8b1e693c0ba2f7107a877c695181b273985adc976e43f124313e8d892a
✓ B_review_final_freeze
  journal_experiments/experiment_B_review_requested_constrained_decoding/manifests/experiment_B_review_FINAL_FREEZE_v1.json
  SHA-256: 63fa99c81477eb7d2c3136b67f72ac3d4673f7d081d7b332ef8d634c68faf3dd
✓ B_E3_final_freeze
  journal_experiments/experiment_B_schema_constrained_decoding/manifests/experiment_B_E3_FINAL_FREEZE_v1.json
  SHA-256: e3e5dee967eeb233f1e583624d204b26c648040938715d41ff7cd4102d6d77bb
✓ C1_inventory
  journal_e

In [71]:
# ==================================================================================================
# G1 — IDENTIFY ONLY TODAY'S FILES NOT YET IN GITHUB
#
# Purpose:
#   Compare the current Colab repository against the last Git commit.
#
# This does NOT:
#   - add files
#   - commit files
#   - push files
#   - modify experiments
#   - run models
# ==================================================================================================

from pathlib import Path
import subprocess

ROOT = Path("/content/deterministic-intent-translation-journal")

assert ROOT.exists(), f"Repository not found: {ROOT}"
assert (ROOT / ".git").exists(), f"Not a Git repository: {ROOT}"

print("=" * 120)
print("G1 — FILES CREATED / MODIFIED SINCE LAST GITHUB COMMIT")
print("=" * 120)

# --------------------------------------------------------------------------------
# Current branch / last commit
# --------------------------------------------------------------------------------

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=ROOT,
    text=True
).strip()

last_commit = subprocess.check_output(
    ["git", "log", "-1", "--format=%H"],
    cwd=ROOT,
    text=True
).strip()

last_commit_date = subprocess.check_output(
    ["git", "log", "-1", "--format=%ci"],
    cwd=ROOT,
    text=True
).strip()

last_commit_subject = subprocess.check_output(
    ["git", "log", "-1", "--format=%s"],
    cwd=ROOT,
    text=True
).strip()

print("\nCURRENT GIT BASELINE")
print("-" * 120)

print("Branch      :", branch)
print("Last commit :", last_commit)
print("Commit date :", last_commit_date)
print("Subject     :", last_commit_subject)

# --------------------------------------------------------------------------------
# Git status
# --------------------------------------------------------------------------------

result = subprocess.run(
    [
        "git",
        "status",
        "--short",
        "--untracked-files=all"
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=True
)

lines = [
    line
    for line in result.stdout.splitlines()
    if line.strip()
]

print("\n" + "-" * 120)
print("FILES DIFFERENT FROM LAST COMMIT")
print("-" * 120)

if not lines:
    print("No modified/untracked files found.")
else:
    for line in lines:
        print(line)

print("\nTotal Git changes:", len(lines))

# --------------------------------------------------------------------------------
# Restrict display to today's journal experiment areas
# --------------------------------------------------------------------------------

today_prefixes = (
    "journal_experiments/experiment_A_model_sweep/",
    "journal_experiments/experiment_B_review_requested_constrained_decoding/",
    "journal_experiments/experiment_B_schema_constrained_decoding/",
    "journal_experiments/journal_final_evidence/",
)

today_candidates = []

for line in lines:

    # porcelain status = two status chars + space + path
    path = line[3:].strip()

    # handle rename format if present
    if " -> " in path:
        path = path.split(" -> ", 1)[1]

    if path.startswith(today_prefixes):
        today_candidates.append((line[:2], path))

print("\n" + "-" * 120)
print("TODAY'S A/B + CONSOLIDATION CANDIDATES")
print("-" * 120)

if not today_candidates:
    print("No candidate files found.")
else:
    for status, path in today_candidates:
        print(f"{status}  {path}")

print("\nCandidate count:", len(today_candidates))

# --------------------------------------------------------------------------------
# Categorize
# --------------------------------------------------------------------------------

categories = {
    "Experiment A": [],
    "B-Review": [],
    "B-E3 supplementary": [],
    "C1-C6 final evidence": [],
}

for status, path in today_candidates:

    if path.startswith(
        "journal_experiments/experiment_A_model_sweep/"
    ):
        categories["Experiment A"].append((status, path))

    elif path.startswith(
        "journal_experiments/experiment_B_review_requested_constrained_decoding/"
    ):
        categories["B-Review"].append((status, path))

    elif path.startswith(
        "journal_experiments/experiment_B_schema_constrained_decoding/"
    ):
        categories["B-E3 supplementary"].append((status, path))

    elif path.startswith(
        "journal_experiments/journal_final_evidence/"
    ):
        categories["C1-C6 final evidence"].append((status, path))

print("\n" + "-" * 120)
print("SUMMARY")
print("-" * 120)

for category, files in categories.items():
    print(f"{category:25s}: {len(files)} files")

print("\n" + "=" * 120)
print("✓ G1 COMPLETE")
print("✓ NOTHING ADDED TO GIT")
print("✓ NOTHING COMMITTED")
print("✓ NOTHING PUSHED")
print("✓ SEND THIS OUTPUT BEFORE WE SELECT FILES FOR GITHUB")
print("=" * 120)

AssertionError: Not a Git repository: /content/deterministic-intent-translation-journal

In [72]:
# ==================================================================================================
# GITHUB PACKAGE — ZIP ONLY FILES CHANGED SINCE LAST GIT COMMIT
#
# Includes changes only under:
#   1. Experiment A
#   2. B-Review (reviewer-requested Experiment B)
#   3. B-E3 supplementary experiment
#   4. C1-C6 journal final evidence
#
# Does NOT:
#   - modify experiments
#   - run models
#   - run benchmarks
#   - git add
#   - git commit
#   - git push
# ==================================================================================================

from pathlib import Path
import subprocess
import zipfile
import hashlib
import shutil
import os

from google.colab import files


print("=" * 120)
print("CREATE GITHUB ZIP — TODAY'S A/B EXPERIMENT FILES ONLY")
print("=" * 120)


# --------------------------------------------------------------------------------------------------
# 1. Repository
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/deterministic-intent-translation-journal"
)

assert ROOT.exists(), (
    f"STOP: repository not found:\n{ROOT}"
)

assert (ROOT / ".git").exists(), (
    f"STOP: not a Git repository:\n{ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Git baseline
# --------------------------------------------------------------------------------------------------

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=ROOT,
    text=True
).strip()

commit = subprocess.check_output(
    ["git", "log", "-1", "--format=%H"],
    cwd=ROOT,
    text=True
).strip()

commit_date = subprocess.check_output(
    ["git", "log", "-1", "--format=%ci"],
    cwd=ROOT,
    text=True
).strip()

commit_subject = subprocess.check_output(
    ["git", "log", "-1", "--format=%s"],
    cwd=ROOT,
    text=True
).strip()


print("\nGIT BASELINE")
print("-" * 120)

print("Branch      :", branch)
print("HEAD commit :", commit)
print("Commit date :", commit_date)
print("Subject     :", commit_subject)


# --------------------------------------------------------------------------------------------------
# 3. Directories allowed in today's package
# --------------------------------------------------------------------------------------------------

ALLOWED_PREFIXES = (

    "journal_experiments/experiment_A_model_sweep/",

    "journal_experiments/"
    "experiment_B_review_requested_constrained_decoding/",

    "journal_experiments/"
    "experiment_B_schema_constrained_decoding/",

    "journal_experiments/journal_final_evidence/",
)


# --------------------------------------------------------------------------------------------------
# 4. Get Git changes relative to HEAD
# --------------------------------------------------------------------------------------------------

result = subprocess.run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=True
)

status_lines = [
    x
    for x in result.stdout.splitlines()
    if x.strip()
]


print("\n" + "-" * 120)
print("ALL FILES DIFFERENT FROM CURRENT GIT HEAD")
print("-" * 120)

if status_lines:
    for line in status_lines:
        print(line)
else:
    print("No Git changes detected.")


# --------------------------------------------------------------------------------------------------
# 5. Select only today's A/B/consolidation files
# --------------------------------------------------------------------------------------------------

selected = []

for line in status_lines:

    status = line[:2]

    path_text = line[3:].strip()

    # Handle Git rename notation:
    # old/path -> new/path
    if " -> " in path_text:
        path_text = path_text.split(" -> ", 1)[1].strip()

    # Remove optional quotes Git may use.
    path_text = path_text.strip('"')

    if path_text.startswith(ALLOWED_PREFIXES):

        full_path = ROOT / path_text

        # We package only actual files.
        if full_path.exists() and full_path.is_file():

            selected.append(
                (status, path_text, full_path)
            )


# Remove accidental duplicates
unique = {}

for status, relative, full in selected:
    unique[relative] = (
        status,
        relative,
        full
    )

selected = [
    unique[k]
    for k in sorted(unique)
]


print("\n" + "-" * 120)
print("FILES SELECTED FOR GITHUB ZIP")
print("-" * 120)

if not selected:

    raise RuntimeError(
        "STOP: No modified/untracked A/B/C1-C6 files were found "
        "relative to the current Git HEAD."
    )


for i, (
    status,
    relative,
    full
) in enumerate(
    selected,
    start=1
):

    print(
        f"{i:03d}. [{status}] {relative}"
    )


print("\nTotal selected files:", len(selected))


# --------------------------------------------------------------------------------------------------
# 6. Category summary
# --------------------------------------------------------------------------------------------------

categories = {
    "Experiment A": 0,
    "B-Review": 0,
    "B-E3 supplementary": 0,
    "C1-C6 final evidence": 0,
}


for _, relative, _ in selected:

    if relative.startswith(
        "journal_experiments/experiment_A_model_sweep/"
    ):
        categories[
            "Experiment A"
        ] += 1

    elif relative.startswith(
        "journal_experiments/"
        "experiment_B_review_requested_constrained_decoding/"
    ):
        categories[
            "B-Review"
        ] += 1

    elif relative.startswith(
        "journal_experiments/"
        "experiment_B_schema_constrained_decoding/"
    ):
        categories[
            "B-E3 supplementary"
        ] += 1

    elif relative.startswith(
        "journal_experiments/journal_final_evidence/"
    ):
        categories[
            "C1-C6 final evidence"
        ] += 1


print("\n" + "-" * 120)
print("CATEGORY SUMMARY")
print("-" * 120)

for category, count in categories.items():

    print(
        f"{category:30s}: {count}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Safety scan — filename based
# --------------------------------------------------------------------------------------------------

DANGEROUS_NAMES = {
    ".env",
    "credentials.json",
    "credential.json",
    "service_account.json",
    "token.json",
    "secrets.json",
}


danger = []

for _, relative, full in selected:

    name = full.name.lower()

    if (
        name in DANGEROUS_NAMES
        or "credential" in name
        or "secret" in name
        or name.endswith(".pem")
        or name.endswith(".key")
    ):
        danger.append(relative)


assert not danger, (
    "STOP: possible credential/secret files detected:\n"
    + "\n".join(danger)
)


print("\n✓ No obvious credential/secret filenames detected.")


# --------------------------------------------------------------------------------------------------
# 8. Create ZIP
# --------------------------------------------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/github_today_package"
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ZIP_PATH = (
    OUTPUT_DIR
    / "github_today_experiments_A_B_and_final_evidence.zip"
)


with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9
) as zf:

    for _, relative, full in selected:

        # Keep repository-relative directory structure.
        zf.write(
            full,
            arcname=relative
        )


# --------------------------------------------------------------------------------------------------
# 9. ZIP SHA-256
# --------------------------------------------------------------------------------------------------

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


ZIP_SHA = sha256(
    ZIP_PATH
)


# --------------------------------------------------------------------------------------------------
# 10. Verify ZIP contents
# --------------------------------------------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zf:

    zip_names = sorted(
        zf.namelist()
    )


assert len(zip_names) == len(selected), (
    "STOP: ZIP file count differs from selected file count."
)


expected_names = sorted(
    relative
    for _, relative, _ in selected
)

assert zip_names == expected_names, (
    "STOP: ZIP contents differ from selected Git files."
)


# --------------------------------------------------------------------------------------------------
# 11. Report
# --------------------------------------------------------------------------------------------------

size_mb = (
    ZIP_PATH.stat().st_size
    / 1024
    / 1024
)


print("\n" + "-" * 120)
print("ZIP VERIFICATION")
print("-" * 120)

print(
    "ZIP file:",
    ZIP_PATH
)

print(
    "Files:",
    len(zip_names)
)

print(
    f"Size: {size_mb:.2f} MB"
)

print(
    "SHA-256:",
    ZIP_SHA
)


print("\nZIP CONTENTS")
print("-" * 120)

for i, name in enumerate(
    zip_names,
    start=1
):

    print(
        f"{i:03d}. {name}"
    )


print("\n" + "=" * 120)
print("✓ TODAY'S GITHUB PACKAGE CREATED")
print("✓ ONLY FILES DIFFERENT FROM CURRENT GIT HEAD INCLUDED")
print("✓ LIMITED TO EXPERIMENT A / B-REVIEW / B-E3 / C1-C6")
print("✓ DIRECTORY STRUCTURE PRESERVED")
print("✓ NO GIT ADD")
print("✓ NO GIT COMMIT")
print("✓ NO GIT PUSH")
print("✓ ZERO MODEL/API CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("=" * 120)


# --------------------------------------------------------------------------------------------------
# 12. Download ZIP to your computer
# --------------------------------------------------------------------------------------------------

print("\nStarting browser download...")

files.download(
    str(ZIP_PATH)
)

CREATE GITHUB ZIP — TODAY'S A/B EXPERIMENT FILES ONLY


AssertionError: STOP: not a Git repository:
/content/deterministic-intent-translation-journal

In [73]:
# ==================================================================================================
# GITHUB PACKAGE — D1
# INSPECT A/B/C ARTIFACT FILES WITHOUT REQUIRING .git
# ==================================================================================================

from pathlib import Path
from datetime import datetime
import os

print("=" * 120)
print("GITHUB PACKAGE — D1: INSPECT A/B/C ARTIFACTS")
print("=" * 120)

ROOT = Path("/content/deterministic-intent-translation-journal")

assert ROOT.exists(), f"STOP: repository directory missing:\n{ROOT}"

print("\nRepository:")
print(ROOT)

print("\n.git exists:", (ROOT / ".git").exists())

# --------------------------------------------------------------------------------------------------
# Experiment/consolidation directories involved in today's work
# --------------------------------------------------------------------------------------------------

TARGETS = {
    "Experiment A":
        ROOT / "journal_experiments" / "experiment_A_model_sweep",

    "B-Review":
        ROOT / "journal_experiments"
             / "experiment_B_review_requested_constrained_decoding",

    "B-E3 supplementary":
        ROOT / "journal_experiments"
             / "experiment_B_schema_constrained_decoding",

    "C1-C6 final evidence":
        ROOT / "journal_experiments" / "journal_final_evidence",
}

print("\n" + "-" * 120)
print("TARGET DIRECTORY CHECK")
print("-" * 120)

for label, path in TARGETS.items():
    print(f"{label:28s}: {'FOUND' if path.exists() else 'MISSING'}")
    print("   ", path)

# --------------------------------------------------------------------------------------------------
# List files with modification timestamps
# --------------------------------------------------------------------------------------------------

records = []

for category, base in TARGETS.items():

    if not base.exists():
        continue

    for p in sorted(base.rglob("*")):

        if not p.is_file():
            continue

        stat = p.stat()

        records.append({
            "category": category,
            "path": str(p.relative_to(ROOT)),
            "mtime": datetime.fromtimestamp(stat.st_mtime),
            "size": stat.st_size,
        })

print("\n" + "-" * 120)
print("ALL CANDIDATE FILES")
print("-" * 120)

for i, r in enumerate(records, start=1):

    print(
        f"{i:03d}. "
        f"{r['mtime'].strftime('%Y-%m-%d %H:%M:%S')}  "
        f"{r['size']:10d} bytes  "
        f"[{r['category']}]  "
        f"{r['path']}"
    )

# --------------------------------------------------------------------------------------------------
# Today's files according to filesystem timestamp
# Current date: 2026-10-02
# --------------------------------------------------------------------------------------------------

TODAY = "2026-10-02"

today_records = [
    r for r in records
    if r["mtime"].strftime("%Y-%m-%d") == TODAY
]

print("\n" + "-" * 120)
print(f"FILES WITH FILESYSTEM MODIFICATION DATE = {TODAY}")
print("-" * 120)

for i, r in enumerate(today_records, start=1):

    print(
        f"{i:03d}. "
        f"{r['mtime'].strftime('%H:%M:%S')}  "
        f"{r['size']:10d} bytes  "
        f"[{r['category']}]  "
        f"{r['path']}"
    )

print("\n" + "-" * 120)
print("SUMMARY")
print("-" * 120)

print("Total candidate files :", len(records))
print("Timestamped today     :", len(today_records))

for category in TARGETS:

    all_count = sum(
        1 for r in records
        if r["category"] == category
    )

    today_count = sum(
        1 for r in today_records
        if r["category"] == category
    )

    print(
        f"{category:28s}: "
        f"{all_count:3d} total / "
        f"{today_count:3d} timestamped today"
    )

print("\n" + "=" * 120)
print("✓ D1 COMPLETE")
print("✓ NO FILE CREATED")
print("✓ NO FILE MODIFIED")
print("✓ NO GIT OPERATION")
print("✓ ZERO MODEL/API CALLS")
print("✓ ZERO BENCHMARK EXECUTIONS")
print("✓ SEND THIS OUTPUT BEFORE CREATING THE ZIP")
print("=" * 120)

GITHUB PACKAGE — D1: INSPECT A/B/C ARTIFACTS

Repository:
/content/deterministic-intent-translation-journal

.git exists: False

------------------------------------------------------------------------------------------------------------------------
TARGET DIRECTORY CHECK
------------------------------------------------------------------------------------------------------------------------
Experiment A                : FOUND
    /content/deterministic-intent-translation-journal/journal_experiments/experiment_A_model_sweep
B-Review                    : FOUND
    /content/deterministic-intent-translation-journal/journal_experiments/experiment_B_review_requested_constrained_decoding
B-E3 supplementary          : FOUND
    /content/deterministic-intent-translation-journal/journal_experiments/experiment_B_schema_constrained_decoding
C1-C6 final evidence        : FOUND
    /content/deterministic-intent-translation-journal/journal_experiments/journal_final_evidence

-------------------------

In [74]:
from google.colab import files
files.download('/content/github_today_experiments_A_B_and_final_evidence.zip')

FileNotFoundError: Cannot find file: /content/github_today_experiments_A_B_and_final_evidence.zip

In [75]:
from pathlib import Path
from datetime import datetime
from google.colab import files
import zipfile
import hashlib

# ==========================================================================================
# CREATE + DOWNLOAD TODAY'S A/B EXPERIMENT PACKAGE
# ==========================================================================================

ROOT = Path("/content/deterministic-intent-translation-journal")

ZIP_PATH = Path(
    "/content/github_today_experiments_A_B_and_final_evidence.zip"
)

TODAY = "2026-10-02"

TARGETS = [
    ROOT / "journal_experiments" / "experiment_A_model_sweep",

    ROOT / "journal_experiments"
         / "experiment_B_review_requested_constrained_decoding",

    ROOT / "journal_experiments"
         / "experiment_B_schema_constrained_decoding",

    ROOT / "journal_experiments"
         / "journal_final_evidence",
]

# ------------------------------------------------------------------------------------------
# Verify directories
# ------------------------------------------------------------------------------------------

assert ROOT.exists(), f"Repository missing: {ROOT}"

for path in TARGETS:
    assert path.exists(), f"Missing directory: {path}"

# ------------------------------------------------------------------------------------------
# Select today's files
# Exclude Python cache/compiled files
# ------------------------------------------------------------------------------------------

selected = []

for base in TARGETS:

    for path in base.rglob("*"):

        if not path.is_file():
            continue

        relative = path.relative_to(ROOT)

        # Skip Python generated cache files
        if "__pycache__" in path.parts:
            continue

        if path.suffix == ".pyc":
            continue

        # Only files timestamped today
        modified_date = datetime.fromtimestamp(
            path.stat().st_mtime
        ).strftime("%Y-%m-%d")

        if modified_date == TODAY:
            selected.append(path)

selected = sorted(set(selected))

print("=" * 100)
print("TODAY'S GITHUB PACKAGE")
print("=" * 100)

print(f"\nFiles selected: {len(selected)}")

assert selected, "No files selected."

# Based on D1:
# 73 total - 4 .pyc cache files = expected 69
if len(selected) != 69:
    print(
        f"\nWARNING: Expected 69 files after excluding 4 .pyc files, "
        f"but found {len(selected)}."
    )

print("\nFILES INCLUDED")
print("-" * 100)

for i, path in enumerate(selected, 1):
    print(
        f"{i:03d}. {path.relative_to(ROOT)}"
    )

# ------------------------------------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------------------------------------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9
) as zf:

    for path in selected:

        zf.write(
            path,
            arcname=str(path.relative_to(ROOT))
        )

# ------------------------------------------------------------------------------------------
# Verify ZIP
# ------------------------------------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zip_files = zf.namelist()

assert len(zip_files) == len(selected), (
    f"ZIP verification failed: "
    f"{len(zip_files)} ZIP entries vs {len(selected)} selected files"
)

# ------------------------------------------------------------------------------------------
# SHA-256
# ------------------------------------------------------------------------------------------

sha = hashlib.sha256()

with open(ZIP_PATH, "rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(block)

zip_sha = sha.hexdigest()

# ------------------------------------------------------------------------------------------
# Final report
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("ZIP CREATED SUCCESSFULLY")
print("=" * 100)

print("\nZIP:")
print(ZIP_PATH)

print("\nFiles in ZIP:")
print(len(zip_files))

print("\nZIP size:")
print(f"{ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MB")

print("\nSHA-256:")
print(zip_sha)

print("\n✓ __pycache__ excluded")
print("✓ .pyc files excluded")
print("✓ Directory structure preserved")
print("✓ No experiments rerun")
print("✓ No Git operation performed")

print("\nStarting download...")
print("=" * 100)

# ------------------------------------------------------------------------------------------
# Download
# ------------------------------------------------------------------------------------------

files.download(str(ZIP_PATH))

TODAY'S GITHUB PACKAGE

Files selected: 69

FILES INCLUDED
----------------------------------------------------------------------------------------------------
001. journal_experiments/experiment_A_model_sweep/logs/A10_gpt-oss-120b_execution.log
002. journal_experiments/experiment_A_model_sweep/manifests/A10_analysis_manifest_v1.json
003. journal_experiments/experiment_A_model_sweep/manifests/A10_execution_manifest_v1.json
004. journal_experiments/experiment_A_model_sweep/manifests/experiment_A_FINAL_FREEZE_v1.json
005. journal_experiments/experiment_A_model_sweep/manifests/experiment_A_colab_reset_recovery_manifest_v1.json
006. journal_experiments/experiment_A_model_sweep/manifests/experiment_A_drive_persistence_v1.json
007. journal_experiments/experiment_A_model_sweep/manifests/experiment_A_extension_preregistration_recovery_manifest_v1.json
008. journal_experiments/experiment_A_model_sweep/manifests/experiment_A_six_evaluable_model_summary_manifest_v1.json
009. journal_experiments/e

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>